# 최신 모델 불러오기

In [2]:
!git clone https://github.com/pinpoint-capstone/pinpoint-capstone.git

Cloning into 'pinpoint-capstone'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 345 (delta 81), reused 152 (delta 43), pack-reused 154 (from 1)
Receiving objects: 100% (345/345), 39.37 MiB | 9.57 MiB/s, done.
Resolving deltas: 100% (147/147), done.


In [3]:
%cd /content/pinpoint-capstone

/content/pinpoint-capstone


In [4]:
!ls -lh models/moment_detr/

total 4.0K
-rw-r--r-- 1 root root 133 Oct 10 11:59 scratch_clip_align_best.ckpt


In [5]:
!apt-get update -qq && apt-get install -y git-lfs

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 3s (1,409 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [6]:
!git lfs pull

In [7]:
!ls -lh models/moment_detr/scratch_clip_align_best.ckpt

-rw-r--r-- 1 root root 49M Oct 10 12:00 models/moment_detr/scratch_clip_align_best.ckpt


# 실험영상 확인

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!ls -lh "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/"

total 21M
drwx------ 2 root root 4.0K Oct  9 15:42 AI_Backup
drwx------ 2 root root 4.0K Oct  9 06:10 clip_features
-rw------- 1 root root 1.4K Oct 10 06:23 detergent_evaluation.json
-rw------- 1 root root 3.0K Oct 10 06:17 fusion_baseline_evaluation.json
-rw------- 1 root root 5.0K Oct 10 06:08 fusion_experiment.py
-rw------- 1 root root 7.8K Oct 10 06:08 indexer_before_subtitle_integration.py
-rw------- 1 root root 2.2K Oct 10 06:08 kr_daiso_001_subtitles.json
-rw------- 1 root root 2.1K Oct 10 06:07 kr_kitchen_001_subtitles.json
-rw------- 1 root root 7.7K Oct 10 06:05 kr_tofu_001_subtitles.json
-rw------- 1 root root  699 Oct 10 06:23 pinpoint_ai_experiment_summary.json
-rw------- 1 root root 396K Oct 10 12:01 PinPoint_Korean_Video_Test.ipynb
drwx------ 2 root root 4.0K Oct  9 06:16 __pycache__
-rw------- 1 root root 1.3K Oct 10 06:19 rice_washing_evaluation.json
-rw------- 1 root root  28K Oct 10 06:08 run_moment_detr_inference_before_fusion.py
-rw------- 1 root root  11K Oct 10 0

In [10]:
from pathlib import Path
import unicodedata

video_dir = Path("/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test")

videos = {}

for path in video_dir.glob("*.mp4"):
    name = unicodedata.normalize("NFC", path.name)

    if "두부" in name:
        videos["tofu"] = path
    elif "주방" in name:
        videos["kitchen"] = path
    elif "다이소" in name:
        videos["daiso"] = path

for name, path in videos.items():
    print(name, "→", unicodedata.normalize("NFC", path.name))

print("확인된 영상:", len(videos), "개")

tofu → 두부요리영상.mp4
kitchen → 주방추천템영상.mp4
daiso → 다이소추천템영상.mp4
확인된 영상: 3 개


# 두부요리 영상

In [11]:
%pip install ffmpeg-python

In [12]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [13]:
import cv2

video_path = str(videos["tofu"])
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

print("영상:", "두부요리영상.mp4")
print("FPS:", fps)
print("영상 길이:", round(frames / fps, 2), "초")
print("해상도:", int(cap.get(3)), "x", int(cap.get(4)))
print("열기 성공:", cap.isOpened())

cap.release()

영상: 두부요리영상.mp4
FPS: 30.0
영상 길이: 165.0 초
해상도: 360 x 640
열기 성공: True


In [14]:
from pathlib import Path
import urllib.request

file_path = Path(
    "/content/pinpoint-capstone/ai/vendor/moment_detr/run_on_video/clip/bpe_simple_vocab_16e6.txt.gz"
)

url = "https://openaipublic.azureedge.net/clip/bpe_simple_vocab_16e6.txt.gz"

urllib.request.urlretrieve(url, file_path)

print("다운로드 완료:", file_path.exists())
print("파일 크기:", file_path.stat().st_size, "bytes")

다운로드 완료: True
파일 크기: 1356917 bytes


In [15]:
from ai.features.moment_detr_features import build_feature

print("build_feature 함수 불러오기 성공!")

build_feature 함수 불러오기 성공!


In [16]:
video_id = "kr_tofu_001"

feature_path = build_feature(
    video_path=str(videos["tofu"]),
    video_id=video_id
)

print("video_id:", video_id)
print("feature_path:", feature_path)

[AI] Extracting video feature: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/두부요리영상.mp4
[AI] Loading video CLIP feature extractor on cuda
Loading CLIP models


100%|████████████████████████████████████████| 354M/354M [00:01<00:00, 247MiB/s]


[AI] Feature saved: /content/pinpoint-capstone/data/features/kr_tofu_001_clip_features.pt
[AI] Feature shape: (83, 512)
video_id: kr_tofu_001
feature_path: /content/pinpoint-capstone/data/features/kr_tofu_001_clip_features.pt


In [17]:
from ai.features.run_moment_detr_inference import predict_moment_hybrid

video_duration = 165.0

result = predict_moment_hybrid(
    video_id=video_id,
    video_feature_path=feature_path,
    subtitle_json_path=None,
    query="cutting tofu with a knife",
    video_duration=video_duration
)

print(result)

[AI] Loading CLIP on cuda
Loading CLIP models
[AI] Loading Moment-DETR on cuda
[AI] Loading CLIP for visual refinement...
{'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 98.5, 'end_time': 118.9725, 'score': 0.9842}


In [18]:
from IPython.display import Video, display

display(Video(
    data=str(videos["tofu"]),
    embed=True,
    width=360
))

Output hidden; open in https://colab.research.google.com to view.

In [19]:
from ai.features.run_moment_detr_inference import predict_moments_hybrid

results = predict_moments_hybrid(
    video_id=video_id,
    video_feature_path=feature_path,
    subtitle_json_path=None,
    query="cutting tofu with a knife",
    video_duration=165.0,
    top_k=5
)

for i, result in enumerate(results, 1):
    print(f"{i}위:", result)

1위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 98.5, 'end_time': 118.9725, 'score': 0.9842, 'source': 'visual', 'subtitle_score': None}
2위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 84.5, 'end_time': 102.7907, 'score': 0.9608, 'source': 'visual', 'subtitle_score': None}
3위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 114.5, 'end_time': 135.9028, 'score': 0.7395, 'source': 'visual', 'subtitle_score': None}
4위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 38.5, 'end_time': 49.1084, 'score': 0.6759, 'source': 'visual', 'subtitle_score': None}
5위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 142.5, 'end_time': 163.8726, 'score': 0.4939, 'source': 'visual', 'subtitle_score': None}


In [20]:
!sed -n '1,100p' ai/preprocessing/transcribe.py

import json
from pathlib import Path

from faster_whisper import WhisperModel


audio_path = "data/processed/video_001/audio.wav"
output_path = Path("data/processed/video_001/transcript.json")

# 처음 테스트는 CPU에서도 부담이 덜한 small 모델 사용
model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("STT 시작...")

segments, info = model.transcribe(
    audio_path,
    beam_size=5
)

results = []

for segment in segments:
    item = {
        "start_time": round(segment.start, 2),
        "end_time": round(segment.end, 2),
        "text": segment.text.strip()
    }

    results.append(item)

    print(
        f"[{item['start_time']:.2f} ~ {item['end_time']:.2f}] "
        f"{item['text']}"
    )


with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print("STT 완료!")
print(f"감지 언어: {info.language}")
print(f"저장 위치: {output_path}")

In [21]:
%pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.1 MB/s eta 0:00:00


In [22]:
%pip install --force-reinstall "av==16.0.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 20.3 MB/s eta 0:00:00
  Attempting uninstall: av
    Found existing installation: av 19.0.1
    Uninstalling av-19.0.1:
      Successfully uninstalled av-19.0.1


In [23]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

segments, info = model.transcribe(
    str(videos["tofu"]),
    language="ko",
    beam_size=5
)

subtitles = [
    {
        "start_time": round(s.start, 2),
        "end_time": round(s.end, 2),
        "text": s.text.strip()
    }
    for s in segments
]

print("감지 언어:", info.language)
print("자막 개수:", len(subtitles))

for s in subtitles[:10]:
    print(s)

감지 언어: ko
자막 개수: 57
{'start_time': 0.0, 'end_time': 4.74, 'text': '여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면'}
{'start_time': 4.74, 'end_time': 8.7, 'text': '순식간에 정말 맛있는 두부 요리 하나가 완성돼요'}
{'start_time': 8.7, 'end_time': 12.88, 'text': '고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요'}
{'start_time': 12.88, 'end_time': 16.32, 'text': '먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요'}
{'start_time': 16.32, 'end_time': 19.02, 'text': '그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요'}
{'start_time': 19.02, 'end_time': 21.06, 'text': '두께는 젓가락 하나 정도면 딱 좋아요'}
{'start_time': 21.06, 'end_time': 23.72, 'text': '다 썰었으면 접시에 담아 잠시 놔둘게요'}
{'start_time': 23.72, 'end_time': 26.46, 'text': '제가 산두부는 한 모에 600번 정도 줬는데요'}
{'start_time': 26.46, 'end_time': 30.74, 'text': '여러분이 계신 곳에서는 두부 한 모에 얼마 정도 하나요?'}
{'start_time': 30.74, 'end_time': 34.14, 'text': '댓글로 가격도 한번 같이 이야기 해봐요 이제부터가 중요한데요'}


In [24]:
import json
from pathlib import Path

subtitle_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/kr_tofu_001_subtitles.json"
)

with open(subtitle_path, "w", encoding="utf-8") as f:
    json.dump(subtitles, f, ensure_ascii=False, indent=2)

print("자막 저장 완료:", subtitle_path)
print("자막 개수:", len(subtitles))

자막 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/kr_tofu_001_subtitles.json
자막 개수: 57


In [25]:
from ai.features.run_moment_detr_inference import predict_moment_hybrid

result = predict_moment_hybrid(
    video_id="kr_tofu_001",
    video_feature_path="/content/pinpoint-capstone/data/features/kr_tofu_001_clip_features.pt",
    subtitle_json_path=str(subtitle_path),
    query="cutting tofu with a knife",
    video_duration=165.0
)

print(result)

{'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 98.5, 'end_time': 118.9725, 'score': 0.9842}


In [26]:
result_ko = predict_moment_hybrid(
    video_id="kr_tofu_001",
    video_feature_path="/content/pinpoint-capstone/data/features/kr_tofu_001_clip_features.pt",
    subtitle_json_path=str(subtitle_path),
    query="두부를 칼로 자르는 장면",
    video_duration=165.0
)

print(result_ko)

{'video_id': 'kr_tofu_001', 'query': '두부를 칼로 자르는 장면', 'start_time': 144.04, 'end_time': 164.3203, 'score': 0.9902}


In [27]:
results_ko = predict_moments_hybrid(
    video_id="kr_tofu_001",
    video_feature_path=feature_path,
    subtitle_json_path=str(subtitle_path),
    query="두부를 칼로 자르는 장면",
    video_duration=165.0,
    top_k=5
)

for i, r in enumerate(results_ko, 1):
    print(f"{i}위:", r)

1위: {'video_id': 'kr_tofu_001', 'query': '두부를 칼로 자르는 장면', 'start_time': 144.04, 'end_time': 164.3203, 'score': 0.9902, 'source': 'subtitle', 'subtitle_score': 0.3597}
2위: {'video_id': 'kr_tofu_001', 'query': '두부를 칼로 자르는 장면', 'start_time': 106.5, 'end_time': 117.9744, 'score': 0.0087, 'source': 'visual', 'subtitle_score': 0.1628}
3위: {'video_id': 'kr_tofu_001', 'query': '두부를 칼로 자르는 장면', 'start_time': 32.64, 'end_time': 73.4991, 'score': 0.001, 'source': 'subtitle', 'subtitle_score': 0.3552}


In [28]:
for s in subtitles:
    if s["start_time"] < 30:
        print(
            f'{s["start_time"]:.2f} ~ {s["end_time"]:.2f}초:',
            s["text"]
        )

0.00 ~ 4.74초: 여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면
4.74 ~ 8.70초: 순식간에 정말 맛있는 두부 요리 하나가 완성돼요
8.70 ~ 12.88초: 고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요
12.88 ~ 16.32초: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요
16.32 ~ 19.02초: 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요
19.02 ~ 21.06초: 두께는 젓가락 하나 정도면 딱 좋아요
21.06 ~ 23.72초: 다 썰었으면 접시에 담아 잠시 놔둘게요
23.72 ~ 26.46초: 제가 산두부는 한 모에 600번 정도 줬는데요
26.46 ~ 30.74초: 여러분이 계신 곳에서는 두부 한 모에 얼마 정도 하나요?


In [29]:
!grep -n -E "def .*subtitle|TfidfVectorizer|subtitle_score|def predict_moments_hybrid" ai/features/run_moment_detr_inference.py

576:from sklearn.feature_extraction.text import TfidfVectorizer
605:def _search_subtitle_start(
671:    vectorizer = TfidfVectorizer(
1072:def _search_subtitle_start_in_window(
1137:    vectorizer = TfidfVectorizer(
1232:def predict_moments_hybrid(
1315:            subtitle_score = (
1338:            subtitle_score = (
1365:            "subtitle_score": (
1367:                if subtitle_score is None
1369:                    float(subtitle_score),


In [30]:
!sed -n '1072,1175p' ai/features/run_moment_detr_inference.py

def _search_subtitle_start_in_window(
    query,
    subtitle_json_path,
    window_start,
    window_end
):
    """
    전체 자막을 기준으로 TF-IDF similarity를 계산한 뒤,
    현재 Moment-DETR 후보와 겹치는 자막 중 최고점을 선택한다.

    Top-1 hybrid와 동일한 score scale을 유지하기 위한 방식.
    """

    subtitle_json_path = Path(
        subtitle_json_path
    )

    if not subtitle_json_path.exists():
        return None

    with open(
        subtitle_json_path,
        "r",
        encoding="utf-8"
    ) as f:
        subtitles = json.load(f)

    if not subtitles:
        return None

    windows = []

    # 전체 영상 자막에서 1~3줄 sliding window 생성
    for window_size in (1, 2, 3):

        for i in range(
            len(subtitles) - window_size + 1
        ):

            group = subtitles[
                i:i + window_size
            ]

            windows.append({
                "start": float(
                    group[0]["start_time"]
                ),
                "end": float(
                    group[-1]["end_tim

In [31]:
!sed -n '1175,1232p;1280,1375p' ai/features/run_moment_detr_inference.py

        return None

    best_idx = max(
        valid_indices,
        key=lambda i: scores[i]
    )

    best = windows[best_idx]

    return {
        "score": float(
            scores[best_idx]
        ),
        "start": best["start"],
        "end": best["end"],
        "text": best["text"]
    }


def _dedup_hybrid_starts(
    results,
    top_k=5,
    min_start_gap=8.0
):
    """
    Hybrid 보정 후 start_time이 너무 가까운
    결과들을 다시 제거한다.
    """

    selected = []

    for result in results:

        duplicated = False

        for kept in selected:

            gap = abs(
                result["start_time"]
                - kept["start_time"]
            )

            if gap < min_start_gap:
                duplicated = True
                break

        if not duplicated:
            selected.append(
                result
            )

        if len(selected) >= top_k:
            break

    return selected


def predict_moments_hybrid(
        )

        subtitle_result = 

In [32]:
from ai.features.run_moment_detr_inference import _search_subtitle_start_in_window

subtitle_test = _search_subtitle_start_in_window(
    query="두부를 칼로 자르는 장면",
    subtitle_json_path=str(subtitle_path),
    window_start=0,
    window_end=165
)

print(subtitle_test)

{'score': 0.35969509884354056, 'start': 145.54, 'end': 147.38, 'text': '두부를 이렇게 만들면 진짜 너무 맛있어요'}


In [33]:
for query in [
    "두부를 자르는 장면",
    "두부를 잘라주세요",
    "두부는 먼저 반으로 잘라주세요"
]:
    result = _search_subtitle_start_in_window(
        query=query,
        subtitle_json_path=str(subtitle_path),
        window_start=0,
        window_end=165
    )

    print("\n검색어:", query)
    print(result)


검색어: 두부를 자르는 장면
{'score': 0.35969509884354056, 'start': 145.54, 'end': 147.38, 'text': '두부를 이렇게 만들면 진짜 너무 맛있어요'}

검색어: 두부를 잘라주세요
{'score': 0.4753819122487197, 'start': 12.88, 'end': 16.32, 'text': '먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요'}

검색어: 두부는 먼저 반으로 잘라주세요
{'score': 0.7764978800281266, 'start': 12.88, 'end': 16.32, 'text': '먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요'}


In [34]:
%pip install sentence-transformers

In [35]:
from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda"
)

print("한국어 의미 검색 모델 로딩 완료!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

한국어 의미 검색 모델 로딩 완료!


In [36]:
from sentence_transformers import util

query = "두부를 칼로 자르는 장면"

texts = [s["text"] for s in subtitles]

query_embedding = semantic_model.encode(
    query,
    convert_to_tensor=True
)

subtitle_embeddings = semantic_model.encode(
    texts,
    convert_to_tensor=True
)

scores = util.cos_sim(
    query_embedding,
    subtitle_embeddings
)[0]

top_indices = scores.topk(5).indices.tolist()

for rank, idx in enumerate(top_indices, 1):
    s = subtitles[idx]
    print(
        f'{rank}위 | {s["start_time"]:.2f}~{s["end_time"]:.2f}초 '
        f'| 유사도 {scores[idx].item():.4f}'
    )
    print("자막:", s["text"])

1위 | 140.86~143.58초 | 유사도 0.6732
자막: 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 135.06~137.58초 | 유사도 0.6575
자막: 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요
3위 | 97.94~101.62초 | 유사도 0.6547
자막: 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요
4위 | 16.32~19.02초 | 유사도 0.6450
자막: 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요
5위 | 12.88~16.32초 | 유사도 0.6442
자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요


In [37]:
windows = []

for size in (1, 2, 3):
    for i in range(len(subtitles) - size + 1):
        group = subtitles[i:i + size]
        windows.append({
            "start": group[0]["start_time"],
            "end": group[-1]["end_time"],
            "text": " ".join(s["text"] for s in group)
        })

texts = [w["text"] for w in windows]

embeddings = semantic_model.encode(
    texts,
    convert_to_tensor=True
)

scores = util.cos_sim(
    query_embedding,
    embeddings
)[0]

for rank, idx in enumerate(scores.topk(5).indices.tolist(), 1):
    w = windows[idx]
    print(f'{rank}위 | {w["start"]:.2f}~{w["end"]:.2f}초 | 유사도 {scores[idx].item():.4f}')
    print(w["text"])
    print()

1위 | 140.86~143.58초 | 유사도 0.6732
이렇게 양념장을 끼얹은 두부가 완성됐어요

2위 | 135.06~137.58초 | 유사도 0.6575
이제 두부에 물기가 다 빠졌으니 접시에 담아주세요

3위 | 97.94~101.62초 | 유사도 0.6547
물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요

4위 | 137.58~143.58초 | 유사도 0.6458
그리고 만들어둔 양념장을 두부 위에 그대로 부어주세요 이렇게 양념장을 끼얹은 두부가 완성됐어요

5위 | 16.32~19.02초 | 유사도 0.6450
그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요



In [38]:
test_queries = [
    "두부를 칼로 자르는 장면",
    "두부를 자르는 장면",
    "두부를 써는 장면",
    "두부를 반으로 자르는 모습",
    "cutting tofu with a knife"
]

for q in test_queries:
    q_emb = semantic_model.encode(q, convert_to_tensor=True)
    sim = util.cos_sim(q_emb, embeddings)[0]

    idx = sim.argmax().item()
    w = windows[idx]

    print(f"\n검색어: {q}")
    print(f'1위: {w["start"]:.2f}~{w["end"]:.2f}초')
    print(f'유사도: {sim[idx].item():.4f}')
    print(f'자막: {w["text"]}')


검색어: 두부를 칼로 자르는 장면
1위: 140.86~143.58초
유사도: 0.6732
자막: 이렇게 양념장을 끼얹은 두부가 완성됐어요

검색어: 두부를 자르는 장면
1위: 78.78~81.42초
유사도: 0.6239
자막: 그중에 여러분께 도움이 되는 영상이 있을지도 몰라요

검색어: 두부를 써는 장면
1위: 78.78~81.42초
유사도: 0.8493
자막: 그중에 여러분께 도움이 되는 영상이 있을지도 몰라요

검색어: 두부를 반으로 자르는 모습
1위: 140.86~143.58초
유사도: 0.8728
자막: 이렇게 양념장을 끼얹은 두부가 완성됐어요

검색어: cutting tofu with a knife
1위: 12.88~21.06초
유사도: 0.5538
자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요 두께는 젓가락 하나 정도면 딱 좋아요


In [39]:
query_english = "cutting tofu with a knife"

q_emb = semantic_model.encode(
    query_english,
    convert_to_tensor=True
)

sim = util.cos_sim(q_emb, embeddings)[0]

for rank, idx in enumerate(sim.topk(5).indices.tolist(), 1):
    w = windows[idx]

    print(
        f'{rank}위 | {w["start"]:.2f}~{w["end"]:.2f}초 '
        f'| 유사도 {sim[idx].item():.4f}'
    )
    print(w["text"])
    print()

1위 | 12.88~21.06초 | 유사도 0.5538
먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요 두께는 젓가락 하나 정도면 딱 좋아요

2위 | 8.70~19.02초 | 유사도 0.5509
고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요

3위 | 8.70~16.32초 | 유사도 0.5087
고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요

4위 | 4.74~16.32초 | 유사도 0.5072
순식간에 정말 맛있는 두부 요리 하나가 완성돼요 고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요

5위 | 12.88~19.02초 | 유사도 0.4800
먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요



In [40]:
test_queries = [
    "cutting tofu",
    "slicing tofu into pieces",
    "a person cuts tofu in half"
]

for q in test_queries:
    q_emb = semantic_model.encode(q, convert_to_tensor=True)
    sim = util.cos_sim(q_emb, embeddings)[0]

    idx = sim.argmax().item()
    w = windows[idx]

    print(f"\n검색어: {q}")
    print(f'1위: {w["start"]:.2f}~{w["end"]:.2f}초')
    print(f'유사도: {sim[idx].item():.4f}')
    print(f'자막: {w["text"]}')


검색어: cutting tofu
1위: 8.70~19.02초
유사도: 0.5136
자막: 고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요

검색어: slicing tofu into pieces
1위: 8.70~19.02초
유사도: 0.5448
자막: 고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요

검색어: a person cuts tofu in half
1위: 12.88~21.06초
유사도: 0.4784
자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요 두께는 젓가락 하나 정도면 딱 좋아요


In [41]:
query = "cutting tofu with a knife"

# 1. 기존 Moment-DETR + Hybrid
hybrid = predict_moment_hybrid(
    video_id="kr_tofu_001",
    video_feature_path=feature_path,
    subtitle_json_path=str(subtitle_path),
    query=query,
    video_duration=165.0
)

# 2. 영상 전체 자막 의미 검색
q_emb = semantic_model.encode(query, convert_to_tensor=True)
sim = util.cos_sim(q_emb, embeddings)[0]
idx = sim.argmax().item()
best = windows[idx]

print("기존 Hybrid 결과:")
print(hybrid)

print("\n전체 자막 의미 검색 결과:")
print({
    "start_time": best["start"],
    "end_time": best["end"],
    "similarity": round(sim[idx].item(), 4),
    "text": best["text"]
})

기존 Hybrid 결과:
{'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 98.5, 'end_time': 118.9725, 'score': 0.9842}

전체 자막 의미 검색 결과:
{'start_time': 12.88, 'end_time': 21.06, 'similarity': 0.5538, 'text': '먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요 두께는 젓가락 하나 정도면 딱 좋아요'}


# 주방추천템 영상

In [42]:
kitchen_video_id = "kr_kitchen_001"

kitchen_feature_path = build_feature(
    video_path=str(videos["kitchen"]),
    video_id=kitchen_video_id
)

print("주방추천템 영상 특징 생성 완료!")
print(kitchen_feature_path)

[AI] Extracting video feature: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/주방추천템영상.mp4
[AI] Feature saved: /content/pinpoint-capstone/data/features/kr_kitchen_001_clip_features.pt
[AI] Feature shape: (19, 512)
주방추천템 영상 특징 생성 완료!
/content/pinpoint-capstone/data/features/kr_kitchen_001_clip_features.pt


In [43]:
kitchen_segments, kitchen_info = model.transcribe(
    str(videos["kitchen"]),
    language="ko",
    beam_size=5
)

kitchen_subtitles = [
    {
        "start_time": round(s.start, 2),
        "end_time": round(s.end, 2),
        "text": s.text.strip()
    }
    for s in kitchen_segments
]

print("감지 언어:", kitchen_info.language)
print("자막 개수:", len(kitchen_subtitles))

for s in kitchen_subtitles:
    print(s)

감지 언어: ko
자막 개수: 17
{'start_time': 0.0, 'end_time': 3.36, 'text': '진작 살걸 후회하는 앞머리 보구에 주방 추천템 세가지'}
{'start_time': 3.36, 'end_time': 5.96, 'text': '첫번째, 솔직히 이거 필요 없을거야 생각했는데'}
{'start_time': 5.96, 'end_time': 8.08, 'text': '이 가위로 소스 짜는거 진짜 편해요'}
{'start_time': 8.08, 'end_time': 11.32, 'text': '손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데'}
{'start_time': 11.32, 'end_time': 13.6, 'text': '가위 필요할땐 가위로 집게 필요할땐 집게로'}
{'start_time': 13.6, 'end_time': 15.96, 'text': '하나로 다 기능을 해서 솔직히 편하더라구요'}
{'start_time': 15.96, 'end_time': 18.2, 'text': '두번째, 냄비에 딱 맞는 체망이 있어서'}
{'start_time': 18.2, 'end_time': 19.76, 'text': '고리차 끓여서 건전해기 좋고'}
{'start_time': 19.76, 'end_time': 21.52, 'text': '물고가 있어서 한 방울도 흘리지 않아'}
{'start_time': 21.52, 'end_time': 23.32, 'text': '시금치 대치고 비빔면 끓이고'}
{'start_time': 23.32, 'end_time': 24.36, 'text': '튀김요리도 쉬운데'}
{'start_time': 24.36, 'end_time': 26.56, 'text': '최고 굽을 땐 316ti의 우염마제품'}
{'start_time': 26.56, 'end_time': 29.28, 'text': '중국제품 아니고 메이드인 코리아라 더 맘에 들어요'}
{'start_time': 29.28, 'en

In [44]:
kitchen_subtitle_path = video_dir / "kr_kitchen_001_subtitles.json"

with open(kitchen_subtitle_path, "w", encoding="utf-8") as f:
    json.dump(kitchen_subtitles, f, ensure_ascii=False, indent=2)

print("자막 저장 완료:", kitchen_subtitle_path)
print("자막 개수:", len(kitchen_subtitles))

자막 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/kr_kitchen_001_subtitles.json
자막 개수: 17


In [45]:
kitchen_query = "squeezing sauce with scissors"

kitchen_hybrid = predict_moment_hybrid(
    video_id=kitchen_video_id,
    video_feature_path=kitchen_feature_path,
    subtitle_json_path=str(kitchen_subtitle_path),
    query=kitchen_query,
    video_duration=37.13
)

print("기존 Hybrid 검색 결과:")
print(kitchen_hybrid)

기존 Hybrid 검색 결과:
{'video_id': 'kr_kitchen_001', 'query': 'squeezing sauce with scissors', 'start_time': 2.5, 'end_time': 10.2804, 'score': 0.8601}


In [46]:
kitchen_windows = []

for size in (1, 2, 3):
    for i in range(len(kitchen_subtitles) - size + 1):
        group = kitchen_subtitles[i:i + size]

        kitchen_windows.append({
            "start": group[0]["start_time"],
            "end": group[-1]["end_time"],
            "text": " ".join(s["text"] for s in group)
        })

kitchen_embeddings = semantic_model.encode(
    [w["text"] for w in kitchen_windows],
    convert_to_tensor=True
)

q_emb = semantic_model.encode(
    kitchen_query,
    convert_to_tensor=True
)

sim = util.cos_sim(q_emb, kitchen_embeddings)[0]

for rank, idx in enumerate(sim.topk(5).indices.tolist(), 1):
    w = kitchen_windows[idx]
    print(f'{rank}위 | {w["start"]:.2f}~{w["end"]:.2f}초 | 유사도 {sim[idx].item():.4f}')
    print(w["text"])
    print()

1위 | 21.52~26.56초 | 유사도 0.4822
시금치 대치고 비빔면 끓이고 튀김요리도 쉬운데 최고 굽을 땐 316ti의 우염마제품

2위 | 15.96~21.52초 | 유사도 0.4100
두번째, 냄비에 딱 맞는 체망이 있어서 고리차 끓여서 건전해기 좋고 물고가 있어서 한 방울도 흘리지 않아

3위 | 23.32~26.56초 | 유사도 0.3991
튀김요리도 쉬운데 최고 굽을 땐 316ti의 우염마제품

4위 | 19.76~23.32초 | 유사도 0.3909
물고가 있어서 한 방울도 흘리지 않아 시금치 대치고 비빔면 끓이고

5위 | 13.60~19.76초 | 유사도 0.3885
하나로 다 기능을 해서 솔직히 편하더라구요 두번째, 냄비에 딱 맞는 체망이 있어서 고리차 끓여서 건전해기 좋고



In [47]:
test_queries = [
    "가위로 소스를 짜는 장면",
    "가위로 소스 짜기",
    "소스를 짜내는 모습"
]

for q in test_queries:
    q_emb = semantic_model.encode(q, convert_to_tensor=True)
    sim = util.cos_sim(q_emb, kitchen_embeddings)[0]

    idx = sim.argmax().item()
    w = kitchen_windows[idx]

    print(f"\n검색어: {q}")
    print(f'1위: {w["start"]:.2f}~{w["end"]:.2f}초')
    print(f'유사도: {sim[idx].item():.4f}')
    print(f'자막: {w["text"]}')


검색어: 가위로 소스를 짜는 장면
1위: 5.96~8.08초
유사도: 0.8635
자막: 이 가위로 소스 짜는거 진짜 편해요

검색어: 가위로 소스 짜기
1위: 5.96~8.08초
유사도: 0.9805
자막: 이 가위로 소스 짜는거 진짜 편해요

검색어: 소스를 짜내는 모습
1위: 5.96~8.08초
유사도: 0.9709
자막: 이 가위로 소스 짜는거 진짜 편해요


In [48]:
queries = [
    "가위로 소스를 짜는 장면",
    "squeezing sauce with scissors"
]

for q in queries:
    q_emb = semantic_model.encode(q, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, kitchen_embeddings)[0]

    print(f"\n검색어: {q}")

    for rank, idx in enumerate(scores.topk(3).indices.tolist(), 1):
        w = kitchen_windows[idx]

        print(
            f'{rank}위 | {w["start"]:.2f}~{w["end"]:.2f}초 '
            f'| 유사도 {scores[idx].item():.4f}'
        )
        print(w["text"])


검색어: 가위로 소스를 짜는 장면
1위 | 5.96~8.08초 | 유사도 0.8635
이 가위로 소스 짜는거 진짜 편해요
2위 | 33.52~34.80초 | 유사도 0.8609
물 바로 버리기도 좋아요
3위 | 18.20~19.76초 | 유사도 0.8493
고리차 끓여서 건전해기 좋고

검색어: squeezing sauce with scissors
1위 | 21.52~26.56초 | 유사도 0.4822
시금치 대치고 비빔면 끓이고 튀김요리도 쉬운데 최고 굽을 땐 316ti의 우염마제품
2위 | 15.96~21.52초 | 유사도 0.4100
두번째, 냄비에 딱 맞는 체망이 있어서 고리차 끓여서 건전해기 좋고 물고가 있어서 한 방울도 흘리지 않아
3위 | 23.32~26.56초 | 유사도 0.3991
튀김요리도 쉬운데 최고 굽을 땐 316ti의 우염마제품


# 다이소추천템 영상

In [49]:
daiso_video_id = "kr_daiso_001"

daiso_feature_path = build_feature(
    video_path=str(videos["daiso"]),
    video_id=daiso_video_id
)

print("다이소 영상 특징 생성 완료!")
print(daiso_feature_path)

[AI] Extracting video feature: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/다이소추천템영상.mp4
[AI] Feature saved: /content/pinpoint-capstone/data/features/kr_daiso_001_clip_features.pt
[AI] Feature shape: (16, 512)
다이소 영상 특징 생성 완료!
/content/pinpoint-capstone/data/features/kr_daiso_001_clip_features.pt


In [50]:
daiso_segments, daiso_info = model.transcribe(
    str(videos["daiso"]),
    language="ko",
    beam_size=5
)

daiso_subtitles = [
    {
        "start_time": round(s.start, 2),
        "end_time": round(s.end, 2),
        "text": s.text.strip()
    }
    for s in daiso_segments
]

print("감지 언어:", daiso_info.language)
print("자막 개수:", len(daiso_subtitles))

for s in daiso_subtitles:
    print(s)

감지 언어: ko
자막 개수: 19
{'start_time': 0.0, 'end_time': 2.08, 'text': '팅으로 잘 썼던 다이소 상반기 결산템디'}
{'start_time': 2.08, 'end_time': 3.08, 'text': '빠르게 소개해 드릴게요'}
{'start_time': 3.08, 'end_time': 4.68, 'text': '여행 갈 때 무조건 챙기는 파우치'}
{'start_time': 4.68, 'end_time': 7.28, 'text': '화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데'}
{'start_time': 7.28, 'end_time': 8.4, 'text': '이제 이거 하나는 끝이에요'}
{'start_time': 8.4, 'end_time': 10.0, 'text': '고기 먹을 때 필수인 양념세트'}
{'start_time': 10.0, 'end_time': 11.36, 'text': '참기름, 소금, 쌈장'}
{'start_time': 11.36, 'end_time': 13.72, 'text': '쭈왕까지 수량씩 들어있어서 먹기 너무 편해요'}
{'start_time': 13.72, 'end_time': 15.16, 'text': '빨리 편하게 해주는 시트 세제'}
{'start_time': 15.16, 'end_time': 17.08, 'text': '톡 뜯어 여행 갈 때 들고 가기도 좋고'}
{'start_time': 17.08, 'end_time': 19.08, 'text': '섬유인지 안 넣어도 향기가 대박이에요'}
{'start_time': 19.08, 'end_time': 20.6, 'text': '피랑 진짜 비슷한 젤리판'}
{'start_time': 20.6, 'end_time': 22.68, 'text': '다이소 립 중에는 제일 잘 쓰게 되더라고요'}
{'start_time': 22.68, 'end_time': 24.48, 'text': '탱글티즈랑 비슷한 비슷한 요거 두 개'}
{

In [51]:
daiso_subtitle_path = video_dir / "kr_daiso_001_subtitles.json"

with open(daiso_subtitle_path, "w", encoding="utf-8") as f:
    json.dump(daiso_subtitles, f, ensure_ascii=False, indent=2)

print("자막 저장 완료:", daiso_subtitle_path)
print("자막 개수:", len(daiso_subtitles))

자막 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/kr_daiso_001_subtitles.json
자막 개수: 19


In [52]:
daiso_query = "a travel makeup pouch"

daiso_hybrid = predict_moment_hybrid(
    video_id=daiso_video_id,
    video_feature_path=daiso_feature_path,
    subtitle_json_path=str(daiso_subtitle_path),
    query=daiso_query,
    video_duration=32.07
)

print("기존 Hybrid 검색 결과:")
print(daiso_hybrid)

기존 Hybrid 검색 결과:
{'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 16.5, 'end_time': 24.4924, 'score': 0.9286}


In [53]:
daiso_windows = []

for size in (1, 2, 3):
    for i in range(len(daiso_subtitles) - size + 1):
        group = daiso_subtitles[i:i + size]

        daiso_windows.append({
            "start": group[0]["start_time"],
            "end": group[-1]["end_time"],
            "text": " ".join(s["text"] for s in group)
        })

daiso_embeddings = semantic_model.encode(
    [w["text"] for w in daiso_windows],
    convert_to_tensor=True
)

q_emb = semantic_model.encode(
    daiso_query,
    convert_to_tensor=True
)

sim = util.cos_sim(q_emb, daiso_embeddings)[0]

for rank, idx in enumerate(sim.topk(5).indices.tolist(), 1):
    w = daiso_windows[idx]
    print(f'{rank}위 | {w["start"]:.2f}~{w["end"]:.2f}초 | 유사도 {sim[idx].item():.4f}')
    print(w["text"])
    print()

1위 | 3.08~7.28초 | 유사도 0.6482
여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

2위 | 2.08~7.28초 | 유사도 0.6459
빠르게 소개해 드릴게요 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

3위 | 3.08~8.40초 | 유사도 0.6129
여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데 이제 이거 하나는 끝이에요

4위 | 4.68~10.00초 | 유사도 0.5149
화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데 이제 이거 하나는 끝이에요 고기 먹을 때 필수인 양념세트

5위 | 4.68~7.28초 | 유사도 0.5128
화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데



In [54]:
test_queries = [
    "여행용 화장품 파우치를 보여주는 장면",
    "여행 갈 때 사용하는 파우치",
    "화장품을 파우치에 넣는 장면"
]

for q in test_queries:
    q_emb = semantic_model.encode(q, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, daiso_embeddings)[0]

    idx = scores.argmax().item()
    w = daiso_windows[idx]

    print(f"\n검색어: {q}")
    print(f'1위: {w["start"]:.2f}~{w["end"]:.2f}초')
    print(f'유사도: {scores[idx].item():.4f}')
    print(f'자막: {w["text"]}')


검색어: 여행용 화장품 파우치를 보여주는 장면
1위: 2.08~7.28초
유사도: 0.6598
자막: 빠르게 소개해 드릴게요 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

검색어: 여행 갈 때 사용하는 파우치
1위: 3.08~4.68초
유사도: 0.8527
자막: 여행 갈 때 무조건 챙기는 파우치

검색어: 화장품을 파우치에 넣는 장면
1위: 4.68~7.28초
유사도: 0.6747
자막: 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데


In [55]:
!sed -n '1232,1295p' ai/features/run_moment_detr_inference.py

def predict_moments_hybrid(
    video_id,
    video_feature_path,
    query,
    subtitle_json_path=None,
    video_duration=None,
    top_k=5,
    iou_threshold=0.5,
    min_start_gap=8.0
):
    """
    서비스용 Top-K Hybrid inference.

    1. Moment-DETR에서 여러 후보 구간 생성
    2. Temporal NMS로 중복 후보 제거
    3. 각 후보 구간 내부에서 subtitle 검색
    4. subtitle confidence가 충분하면
       subtitle start 사용
    5. 부족하면 visual CLIP refinement
    6. 최종 start_time 기준 중복 제거
    """

    # 후보를 조금 넉넉하게 받아온 뒤
    # 최종 hybrid 보정 후 top_k 선택
    candidate_k = max(
        top_k * 3,
        top_k
    )

    visual_candidates = predict_moments(
        video_id=video_id,
        video_feature_path=video_feature_path,
        query=query,
        video_duration=video_duration,
        top_k=candidate_k,
        iou_threshold=iou_threshold
    )

    hybrid_results = []

    for candidate in visual_candidates:

        visual_start = float(
            candidate["start_time"]
        )

        visual_end = float(
      

In [56]:
from ai.features.run_moment_detr_inference import predict_moments

visual_results = predict_moments(
    video_id=daiso_video_id,
    video_feature_path=daiso_feature_path,
    query="a travel makeup pouch",
    video_duration=32.07,
    top_k=5
)

for i, result in enumerate(visual_results, 1):
    print(f"{i}위:", result)

1위: {'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 18.0663, 'end_time': 24.4924, 'score': 0.9286}
2위: {'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 27.3306, 'end_time': 29.2973, 'score': 0.8823}
3위: {'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 9.3605, 'end_time': 16.2451, 'score': 0.0028}


In [57]:
# 기존 Moment-DETR 검색 후보
combined_candidates = [
    {
        **r,
        "source": "visual",
        "visual_score": r["score"],
        "subtitle_score": None
    }
    for r in visual_results
]

# 전체 자막 의미 검색 상위 5개 후보
top_indices = sim.topk(5).indices.tolist()

for idx in top_indices:
    w = daiso_windows[idx]

    combined_candidates.append({
        "video_id": daiso_video_id,
        "query": daiso_query,
        "start_time": w["start"],
        "end_time": w["end"],
        "score": None,
        "source": "subtitle",
        "visual_score": None,
        "subtitle_score": round(sim[idx].item(), 4)
    })

for i, r in enumerate(combined_candidates, 1):
    print(f'{i}번 | {r["source"]} | '
          f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
          f'영상점수={r["visual_score"]} | 자막점수={r["subtitle_score"]}')

1번 | visual | 18.07~24.49초 | 영상점수=0.9286 | 자막점수=None
2번 | visual | 27.33~29.30초 | 영상점수=0.8823 | 자막점수=None
3번 | visual | 9.36~16.25초 | 영상점수=0.0028 | 자막점수=None
4번 | subtitle | 3.08~7.28초 | 영상점수=None | 자막점수=0.6482
5번 | subtitle | 2.08~7.28초 | 영상점수=None | 자막점수=0.6459
6번 | subtitle | 3.08~8.40초 | 영상점수=None | 자막점수=0.6129
7번 | subtitle | 4.68~10.00초 | 영상점수=None | 자막점수=0.5149
8번 | subtitle | 4.68~7.28초 | 영상점수=None | 자막점수=0.5128


In [58]:
def temporal_iou(a, b):
    intersection = max(
        0,
        min(a["end_time"], b["end_time"])
        - max(a["start_time"], b["start_time"])
    )

    union = (
        max(a["end_time"], b["end_time"])
        - min(a["start_time"], b["start_time"])
    )

    return intersection / union if union > 0 else 0


# 각 검색 방식 내부에서 높은 점수 순으로 정렬
visual = sorted(
    [r for r in combined_candidates if r["source"] == "visual"],
    key=lambda r: r["visual_score"],
    reverse=True
)

subtitle = sorted(
    [r for r in combined_candidates if r["source"] == "subtitle"],
    key=lambda r: r["subtitle_score"],
    reverse=True
)

# 방식별 순위를 번갈아 배치 (점수 직접 비교 X)
ordered = []
for i in range(max(len(visual), len(subtitle))):
    if i < len(visual):
        ordered.append(visual[i])
    if i < len(subtitle):
        ordered.append(subtitle[i])

# 겹치는 후보 제거
deduplicated = []

for candidate in ordered:
    if any(
        temporal_iou(candidate, kept) >= 0.5
        for kept in deduplicated
    ):
        continue

    deduplicated.append(candidate)

for i, r in enumerate(deduplicated, 1):
    print(
        f'{i}번 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초'
    )

1번 | visual | 18.07~24.49초
2번 | subtitle | 3.08~7.28초
3번 | visual | 27.33~29.30초
4번 | visual | 9.36~16.25초
5번 | subtitle | 4.68~10.00초


In [59]:
for r in deduplicated:
    matching = [
        i for i, w in enumerate(daiso_windows)
        if w["start"] < r["end_time"]
        and w["end"] > r["start_time"]
    ]

    r["rerank_score"] = (
        max(float(sim[i].item()) for i in matching)
        if matching else 0.0
    )

reranked = sorted(
    deduplicated,
    key=lambda r: r["rerank_score"],
    reverse=True
)

for rank, r in enumerate(reranked, 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'재순위 점수={r["rerank_score"]:.4f}'
    )

1위 | subtitle | 3.08~7.28초 | 재순위 점수=0.6482
2위 | subtitle | 4.68~10.00초 | 재순위 점수=0.6482
3위 | visual | 9.36~16.25초 | 재순위 점수=0.5149
4위 | visual | 18.07~24.49초 | 재순위 점수=0.4113
5위 | visual | 27.33~29.30초 | 재순위 점수=0.4113


In [60]:
final_results = []

for candidate in reranked:
    is_duplicate = any(
        temporal_iou(candidate, kept) >= 0.5
        or abs(candidate["start_time"] - kept["start_time"]) < 5.0
        for kept in final_results
    )

    if not is_duplicate:
        final_results.append(candidate)

for rank, r in enumerate(final_results, 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'재순위 점수={r["rerank_score"]:.4f}'
    )

1위 | subtitle | 3.08~7.28초 | 재순위 점수=0.6482
2위 | visual | 9.36~16.25초 | 재순위 점수=0.5149
3위 | visual | 18.07~24.49초 | 재순위 점수=0.4113
4위 | visual | 27.33~29.30초 | 재순위 점수=0.4113


In [61]:
tofu_query = "cutting tofu with a knife"

tofu_visual_results = predict_moments(
    video_id="kr_tofu_001",
    video_feature_path=feature_path,
    query=tofu_query,
    video_duration=165.0,
    top_k=5
)

for i, r in enumerate(tofu_visual_results, 1):
    print(f"{i}위:", r)

1위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 101.7673, 'end_time': 118.9725, 'score': 0.9842}
2위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 86.5279, 'end_time': 102.7907, 'score': 0.9608}
3위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 116.1929, 'end_time': 135.9028, 'score': 0.7395}
4위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 30.5973, 'end_time': 49.1084, 'score': 0.6759}
5위: {'video_id': 'kr_tofu_001', 'query': 'cutting tofu with a knife', 'start_time': 145.5858, 'end_time': 163.8726, 'score': 0.4939}


In [62]:
# 1. 두부 영상 자막 의미 유사도 계산
tofu_q_emb = semantic_model.encode(
    tofu_query, convert_to_tensor=True
)
tofu_sim = util.cos_sim(tofu_q_emb, embeddings)[0]

# 2. 영상 검색 후보
tofu_combined = [
    {**r, "source": "visual"}
    for r in tofu_visual_results
]

# 3. 자막 검색 상위 5개 후보 추가
for idx in tofu_sim.topk(5).indices.tolist():
    w = windows[idx]

    tofu_combined.append({
        "video_id": "kr_tofu_001",
        "query": tofu_query,
        "start_time": w["start"],
        "end_time": w["end"],
        "source": "subtitle"
    })

# 4. 모든 후보를 자막 의미 유사도로 재평가
for r in tofu_combined:
    matching = [
        i for i, w in enumerate(windows)
        if w["start"] < r["end_time"]
        and w["end"] > r["start_time"]
    ]

    r["rerank_score"] = (
        max(float(tofu_sim[i].item()) for i in matching)
        if matching else 0.0
    )

# 5. 재순위화 + 중복 제거
tofu_ranked = sorted(
    tofu_combined,
    key=lambda r: r["rerank_score"],
    reverse=True
)

tofu_final = []

for candidate in tofu_ranked:
    if any(
        temporal_iou(candidate, kept) >= 0.5
        or abs(candidate["start_time"] - kept["start_time"]) < 5.0
        for kept in tofu_final
    ):
        continue

    tofu_final.append(candidate)

for rank, r in enumerate(tofu_final[:5], 1):
    start = r["start_time"]
    end = r["end_time"]
    source = r["source"]
    score = r["rerank_score"]

    print(
        f"{rank}위 | {source} | "
        f"{start:.2f} ~ {end:.2f}초 | "
        f"점수={score:.4f}"
    )

1위 | subtitle | 12.88 ~ 21.06초 | 점수=0.5538
2위 | subtitle | 4.74 ~ 16.32초 | 점수=0.5538
3위 | visual | 30.60 ~ 49.11초 | 점수=0.4746
4위 | visual | 116.19 ~ 135.90초 | 점수=0.4488
5위 | visual | 145.59 ~ 163.87초 | 점수=0.4403


In [63]:
# 주방 영상의 기존 검색 후보
kitchen_visual_results = predict_moments(
    video_id=kitchen_video_id,
    video_feature_path=kitchen_feature_path,
    query=kitchen_query,
    video_duration=37.13,
    top_k=5
)

# 영어 검색어 기준 자막 의미 유사도
kitchen_q_emb = semantic_model.encode(
    kitchen_query, convert_to_tensor=True
)
kitchen_sim = util.cos_sim(
    kitchen_q_emb, kitchen_embeddings
)[0]

# 영상 + 자막 후보 통합
kitchen_combined = [
    {**r, "source": "visual"}
    for r in kitchen_visual_results
]

for idx in kitchen_sim.topk(5).indices.tolist():
    w = kitchen_windows[idx]
    kitchen_combined.append({
        "video_id": kitchen_video_id,
        "query": kitchen_query,
        "start_time": w["start"],
        "end_time": w["end"],
        "source": "subtitle"
    })

# 동일한 기준으로 재순위화
for r in kitchen_combined:
    matching = [
        i for i, w in enumerate(kitchen_windows)
        if w["start"] < r["end_time"]
        and w["end"] > r["start_time"]
    ]
    r["rerank_score"] = (
        max(float(kitchen_sim[i].item()) for i in matching)
        if matching else 0.0
    )

kitchen_ranked = sorted(
    kitchen_combined,
    key=lambda r: r["rerank_score"],
    reverse=True
)

# 중복 제거
kitchen_final = []

for candidate in kitchen_ranked:
    if any(
        temporal_iou(candidate, kept) >= 0.5
        or abs(candidate["start_time"] - kept["start_time"]) < 5.0
        for kept in kitchen_final
    ):
        continue
    kitchen_final.append(candidate)

for rank, r in enumerate(kitchen_final[:5], 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'점수={r["rerank_score"]:.4f}'
    )

1위 | visual | 23.40~31.72초 | 점수=0.4822
2위 | visual | 13.93~20.53초 | 점수=0.4100
3위 | visual | 4.50~10.28초 | 점수=0.3673


In [64]:
for visual in kitchen_visual_results:
    matches = []

    for subtitle in kitchen_combined:
        if subtitle["source"] != "subtitle":
            continue

        overlap = temporal_iou(visual, subtitle)

        if overlap > 0:
            matches.append(overlap)

    agreement = max(matches, default=0.0)

    print(
        f'영상 후보 {visual["start_time"]:.2f}~'
        f'{visual["end_time"]:.2f}초 | '
        f'영상-자막 최대 IoU={agreement:.4f}'
    )

영상 후보 4.50~10.28초 | 영상-자막 최대 IoU=0.0000
영상 후보 23.40~31.72초 | 영상-자막 최대 IoU=0.3761
영상 후보 13.93~20.53초 | 영상-자막 최대 IoU=0.8409


In [65]:
korean_query = "가위로 소스를 짜는 장면"

ko_emb = semantic_model.encode(
    korean_query,
    convert_to_tensor=True
)

ko_scores = util.cos_sim(
    ko_emb,
    kitchen_embeddings
)[0]

ko_top = ko_scores.topk(5).indices.tolist()

print("한국어 자막 검색 상위 5개")

for idx in ko_top:
    w = kitchen_windows[idx]

    print(
        f'{w["start"]:.2f}~{w["end"]:.2f}초 | '
        f'유사도={ko_scores[idx].item():.4f}'
    )

print("\n영상 후보와 한국어 자막 후보의 일치도")

for v in kitchen_visual_results:
    overlaps = [
        temporal_iou(
            v,
            {
                "start_time": kitchen_windows[i]["start"],
                "end_time": kitchen_windows[i]["end"]
            }
        )
        for i in ko_top
    ]

    print(
        f'{v["start_time"]:.2f}~{v["end_time"]:.2f}초 | '
        f'최대 IoU={max(overlaps):.4f}'
    )

한국어 자막 검색 상위 5개
5.96~8.08초 | 유사도=0.8635
33.52~34.80초 | 유사도=0.8609
18.20~19.76초 | 유사도=0.8493
21.52~23.32초 | 유사도=0.8420
23.32~24.36초 | 유사도=0.8406

영상 후보와 한국어 자막 후보의 일치도
4.50~10.28초 | 최대 IoU=0.3668
23.40~31.72초 | 최대 IoU=0.1142
13.93~20.53초 | 최대 IoU=0.2364


- 두부: 기존 Moment-DETR은 실패했지만, 영어 자막 의미 검색으로 정답 발견

- 주방추천템: 기존 Moment-DETR이 관련 구간을 찾았고, 한국어 자막 검색도 정답 발견

- 다이소추천템: 기존 Moment-DETR은 실패했지만, 한국어·영어 자막 검색으로 정답 발견

---

따라서 기존 Moment-DETR을 유지하면서 독립적인 자막 의미 검색을 추가하는 방향은 유망해.

하지만 언어별 유사도를 그대로 비교하거나 IoU만으로 순위를 결정하는 방법은 아직 검증되지 않았어.

# 재순위화 테스트

In [66]:
# 한국어 자막 후보 추가
kitchen_ko_candidates = [
    {**r, "source": "visual"}
    for r in kitchen_visual_results
]

for idx in ko_top:
    w = kitchen_windows[idx]

    kitchen_ko_candidates.append({
        "start_time": w["start"],
        "end_time": w["end"],
        "source": "subtitle"
    })

# 한국어 자막 유사도로 재평가
for r in kitchen_ko_candidates:
    matching = [
        i for i, w in enumerate(kitchen_windows)
        if w["start"] < r["end_time"]
        and w["end"] > r["start_time"]
    ]

    r["rerank_score"] = (
        max(float(ko_scores[i].item()) for i in matching)
        if matching else 0.0
    )

# 정렬 및 중복 제거
ranked = sorted(
    kitchen_ko_candidates,
    key=lambda r: r["rerank_score"],
    reverse=True
)

kitchen_ko_final = []

for candidate in ranked:
    if any(
        temporal_iou(candidate, kept) >= 0.5
        or abs(candidate["start_time"] - kept["start_time"]) < 5.0
        for kept in kitchen_ko_final
    ):
        continue

    kitchen_ko_final.append(candidate)

for rank, r in enumerate(kitchen_ko_final[:5], 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'점수={r["rerank_score"]:.4f}'
    )

1위 | visual | 4.50~10.28초 | 점수=0.8635
2위 | subtitle | 33.52~34.80초 | 점수=0.8609
3위 | visual | 13.93~20.53초 | 점수=0.8493
4위 | subtitle | 21.52~23.32초 | 점수=0.8420


In [67]:
from pathlib import Path
import shutil

source = Path("ai/features/run_moment_detr_inference.py")
backup = Path("ai/features/run_moment_detr_inference_before_fusion.py")

shutil.copy2(source, backup)

print("백업 완료:", backup)
print("원본 크기:", source.stat().st_size)
print("백업 크기:", backup.stat().st_size)

백업 완료: ai/features/run_moment_detr_inference_before_fusion.py
원본 크기: 28490
백업 크기: 28490


In [68]:
from sentence_transformers import util

def get_semantic_subtitle_candidates(
    query,
    subtitles,
    model,
    top_k=5
):
    windows = []

    for size in (1, 2, 3):
        for i in range(len(subtitles) - size + 1):
            group = subtitles[i:i + size]

            windows.append({
                "start_time": group[0]["start_time"],
                "end_time": group[-1]["end_time"],
                "text": " ".join(s["text"] for s in group)
            })

    embeddings = model.encode(
        [w["text"] for w in windows],
        convert_to_tensor=True
    )

    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )[0]

    candidates = []

    for idx in scores.topk(min(top_k, len(windows))).indices.tolist():
        w = windows[idx]

        candidates.append({
            **w,
            "score": round(scores[idx].item(), 4),
            "source": "subtitle"
        })

    return candidates

print("자막 후보 생성 함수 준비 완료!")

자막 후보 생성 함수 준비 완료!


In [69]:
def get_multilingual_subtitle_candidates(
    korean_query,
    english_query,
    subtitles,
    model,
    top_k=5
):
    candidates = []

    for language, query in [
        ("ko", korean_query),
        ("en", english_query)
    ]:
        results = get_semantic_subtitle_candidates(
            query=query,
            subtitles=subtitles,
            model=model,
            top_k=top_k
        )

        for r in results:
            candidates.append({
                **r,
                "language": language,
                "query_used": query
            })

    return candidates


print("한국어·영어 자막 후보 생성 함수 준비 완료!")

한국어·영어 자막 후보 생성 함수 준비 완료!


In [70]:
daiso_multilingual = get_multilingual_subtitle_candidates(
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    subtitles=daiso_subtitles,
    model=semantic_model,
    top_k=3
)

for r in daiso_multilingual:
    print(
        f'{r["language"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'유사도={r["score"]:.4f}'
    )
    print("자막:", r["text"])
    print()

ko | 2.08~7.28초 | 유사도=0.6598
자막: 빠르게 소개해 드릴게요 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

ko | 3.08~7.28초 | 유사도=0.6566
자막: 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

ko | 3.08~8.40초 | 유사도=0.6424
자막: 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데 이제 이거 하나는 끝이에요

en | 3.08~7.28초 | 유사도=0.6482
자막: 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

en | 2.08~7.28초 | 유사도=0.6459
자막: 빠르게 소개해 드릴게요 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데

en | 3.08~8.40초 | 유사도=0.6129
자막: 여행 갈 때 무조건 챙기는 파우치 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데 이제 이거 하나는 끝이에요



In [71]:
def deduplicate_multilingual_candidates(
    candidates,
    iou_threshold=0.5,
    start_gap=5.0
):
    # 언어 간 점수를 비교하지 않고
    # 원래 후보 순서를 유지
    selected = []

    for candidate in candidates:
        duplicate = any(
            temporal_iou(candidate, kept) >= iou_threshold
            or (
                abs(
                    candidate["start_time"]
                    - kept["start_time"]
                ) < start_gap
            )
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

    return selected


daiso_unique = deduplicate_multilingual_candidates(
    daiso_multilingual
)

print("중복 제거 전:", len(daiso_multilingual))
print("중복 제거 후:", len(daiso_unique))

for r in daiso_unique:
    print(
        f'{r["language"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'유사도={r["score"]:.4f}'
    )

중복 제거 전: 6
중복 제거 후: 1
ko | 2.08~7.28초 | 유사도=0.6598


In [72]:
tofu_multilingual = get_multilingual_subtitle_candidates(
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    subtitles=subtitles,
    model=semantic_model,
    top_k=3
)

tofu_unique = deduplicate_multilingual_candidates(
    tofu_multilingual
)

print("중복 제거 전:", len(tofu_multilingual))
print("중복 제거 후:", len(tofu_unique))

for r in tofu_unique:
    print(
        f'{r["language"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'유사도={r["score"]:.4f}'
    )
    print("자막:", r["text"])
    print()

중복 제거 전: 6
중복 제거 후: 4
ko | 140.86~143.58초 | 유사도=0.6732
자막: 이렇게 양념장을 끼얹은 두부가 완성됐어요

ko | 135.06~137.58초 | 유사도=0.6575
자막: 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요

ko | 97.94~101.62초 | 유사도=0.6547
자막: 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요

en | 12.88~21.06초 | 유사도=0.5538
자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요 두께는 젓가락 하나 정도면 딱 좋아요



In [73]:
ko_emb = semantic_model.encode(
    "두부를 칼로 자르는 장면",
    convert_to_tensor=True
)

en_emb = semantic_model.encode(
    "cutting tofu with a knife",
    convert_to_tensor=True
)

ko_scores = util.cos_sim(ko_emb, embeddings)[0]
en_scores = util.cos_sim(en_emb, embeddings)[0]

for r in tofu_unique:
    matching = [
        i for i, w in enumerate(windows)
        if w["start"] < r["end_time"]
        and w["end"] > r["start_time"]
    ]

    ko_score = max(ko_scores[i].item() for i in matching)
    en_score = max(en_scores[i].item() for i in matching)

    print(
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'한국어={ko_score:.4f} | 영어={en_score:.4f}'
    )

140.86~143.58초 | 한국어=0.6732 | 영어=0.4403
135.06~137.58초 | 한국어=0.6575 | 영어=0.4488
97.94~101.62초 | 한국어=0.6547 | 영어=0.4293
12.88~21.06초 | 한국어=0.6450 | 영어=0.5538


In [74]:
# 후보별 한국어·영어 점수 수집
comparison = []

for r in tofu_unique:
    matching = [
        i for i, w in enumerate(windows)
        if w["start"] < r["end_time"]
        and w["end"] > r["start_time"]
    ]

    comparison.append({
        "start": r["start_time"],
        "end": r["end_time"],
        "ko": max(float(ko_scores[i]) for i in matching),
        "en": max(float(en_scores[i]) for i in matching)
    })

# 각 언어별 순위 계산
ko_sorted = sorted(comparison, key=lambda x: x["ko"], reverse=True)
en_sorted = sorted(comparison, key=lambda x: x["en"], reverse=True)

for r in comparison:
    r["ko_rank"] = ko_sorted.index(r) + 1
    r["en_rank"] = en_sorted.index(r) + 1
    r["best_rank"] = min(r["ko_rank"], r["en_rank"])

for r in sorted(comparison, key=lambda x: x["best_rank"]):
    print(
        f'{r["start"]:.2f}~{r["end"]:.2f}초 | '
        f'한국어 {r["ko_rank"]}위 | '
        f'영어 {r["en_rank"]}위'
    )

140.86~143.58초 | 한국어 1위 | 영어 3위
12.88~21.06초 | 한국어 4위 | 영어 1위
135.06~137.58초 | 한국어 2위 | 영어 2위
97.94~101.62초 | 한국어 3위 | 영어 4위


In [75]:
ko_max = max(r["ko"] for r in comparison)
en_max = max(r["en"] for r in comparison)

for r in comparison:
    ko_relative = r["ko"] / ko_max
    en_relative = r["en"] / en_max

    print(
        f'{r["start"]:.2f}~{r["end"]:.2f}초 | '
        f'한국어 상대점수={ko_relative:.4f} | '
        f'영어 상대점수={en_relative:.4f}'
    )

140.86~143.58초 | 한국어 상대점수=1.0000 | 영어 상대점수=0.7951
135.06~137.58초 | 한국어 상대점수=0.9767 | 영어 상대점수=0.8104
97.94~101.62초 | 한국어 상대점수=0.9725 | 영어 상대점수=0.7752
12.88~21.06초 | 한국어 상대점수=0.9581 | 영어 상대점수=1.0000


In [76]:
for language, scores in [
    ("한국어", ko_scores),
    ("영어", en_scores)
]:
    values = sorted(
        [r[language == "영어" and "en" or "ko"] for r in comparison],
        reverse=True
    )

    print(f"\n{language}")
    print(f"1위 점수: {values[0]:.4f}")
    print(f"2위 점수: {values[1]:.4f}")
    print(f"점수 차이: {values[0] - values[1]:.4f}")


한국어
1위 점수: 0.6732
2위 점수: 0.6575
점수 차이: 0.0157

영어
1위 점수: 0.5538
2위 점수: 0.4488
점수 차이: 0.1050


In [77]:
queries = {
    "한국어": "가위로 소스를 짜는 장면",
    "영어": "squeezing sauce with scissors"
}

for language, query in queries.items():
    q_emb = semantic_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, kitchen_embeddings)[0]

    candidates = []

    for i, w in enumerate(kitchen_windows):
        candidates.append({
            "start_time": w["start"],
            "end_time": w["end"],
            "score": float(scores[i].item())
        })

    candidates.sort(key=lambda x: x["score"], reverse=True)

    unique = deduplicate_multilingual_candidates(candidates)

    print("\n언어:", language)

    for rank, r in enumerate(unique[:3], 1):
        start = r["start_time"]
        end = r["end_time"]
        score = r["score"]

        print(f"{rank}위: {start:.2f} ~ {end:.2f}초 | 점수={score:.4f}")

    if len(unique) >= 2:
        gap = unique[0]["score"] - unique[1]["score"]
        print(f"1위와 2위 점수 차이: {gap:.4f}")


언어: 한국어
1위: 5.96 ~ 8.08초 | 점수=0.8635
2위: 33.52 ~ 34.80초 | 점수=0.8609
3위: 18.20 ~ 19.76초 | 점수=0.8493
1위와 2위 점수 차이: 0.0026

언어: 영어
1위: 21.52 ~ 26.56초 | 점수=0.4822
2위: 15.96 ~ 21.52초 | 점수=0.4100
3위: 29.28 ~ 33.52초 | 점수=0.3876
1위와 2위 점수 차이: 0.0721


- 영어 검색의 점수 차이가 훨씬 큰데도 오답이라는 것
- 두부 영상에서는 점수 차이가 큰 영어 검색이 정확했지만, 주방추천템 영상에서는 반대였어.
- 따라서 1위와 2위의 점수 차이로 한국어·영어 중 더 정확한 결과를 선택하는 방식은 신뢰하기 어려워.

---

- 지금까지 확인한 문제는 명확해.
  - Moment-DETR만 사용하면 놓치는 장면이 있음.
  - 자막 검색만 사용해도 엉뚱한 장면이 나올 수 있음.
  - 언어별 점수, 순위, 점수 차이만으로는 정답을 안정적으로 구별하기 어려움.

- 그래서 이제는 점수 공식을 계속 바꾸기보다 후보 구간을 실제 영상 화면과 검색어가 얼마나 일치하는지 다시 검증하는 방식을 실험하는 게 좋아.
- 기존에 사용하는 CLIP을 활용할 수 있으므로, 새로운 모델을 학습하기 전에 검증해 볼 가치가 있어.

# CLIP 재검증 함수 확인

In [78]:
!grep -n -E "^def _refine_visual_start|^def .*clip|^def .*similarity" ai/features/run_moment_detr_inference.py

588:def get_refine_clip_model():
793:def _refine_visual_start(


In [79]:
!sed -n '580,610p;793,875p' ai/features/run_moment_detr_inference.py


SUBTITLE_THRESHOLD = 0.35
START_OFFSET = 1.5
VISUAL_TOP_K = 8

_REFINE_CLIP_MODEL = None


def get_refine_clip_model():
    global _REFINE_CLIP_MODEL

    if _REFINE_CLIP_MODEL is None:
        print("[AI] Loading CLIP for visual refinement...")

        _REFINE_CLIP_MODEL, _ = clip.load(
            "ViT-B/32",
            device=DEVICE,
            jit=False
        )

        _REFINE_CLIP_MODEL.eval()

    return _REFINE_CLIP_MODEL


def _search_subtitle_start(
    query,
    subtitle_json_path
):
    """
    인접 자막 1~3개를 합친 window에 대해
def _refine_visual_start(
    query,
    video_feature_path,
    span_start,
    span_end
):
    """
    Moment-DETR 후보 내부의 2초 CLIP bin을 검색한 후
    smoothing + 연속 cluster 기반으로 시작 시점 보정.
    """

    video_feats = torch.load(
        video_feature_path,
        map_location=DEVICE
    ).float()

    video_feats = F.normalize(
        video_feats,
        dim=-1,
        eps=1e-6
    )

    query_feat = _encode_refine_query(
        query
    )

    star

In [80]:
!grep -n -A 35 "^def _encode_refine_query" ai/features/run_moment_detr_inference.py

710:def _encode_refine_query(query):
711-
712-    clip_model = get_refine_clip_model()
713-
714-    tokens = clip.tokenize(
715-        [query]
716-    ).to(DEVICE)
717-
718-    output = clip_model.encode_text(
719-        tokens
720-    )
721-
722-    query_feat = output[
723-        "pooler_output"
724-    ].float()
725-
726-    query_feat = F.normalize(
727-        query_feat,
728-        dim=-1,
729-        eps=1e-6
730-    )
731-
732-    return query_feat[0]
733-
734-
735-def _smooth_scores(scores):
736-    """
737-    자기 자신 + 앞뒤 1개 bin의 평균.
738-    """
739-
740-    smoothed = torch.zeros_like(
741-        scores
742-    )
743-
744-    for i in range(len(scores)):
745-


In [81]:
import torch
import torch.nn.functional as F
from ai.features.run_moment_detr_inference import _encode_refine_query

def get_visual_candidate_score(query, feature_path, start, end):
    features = torch.load(
        feature_path,
        map_location="cpu"
    ).float()

    features = F.normalize(features, dim=-1)

    query_feature = _encode_refine_query(query).detach().cpu()
    query_feature = F.normalize(query_feature, dim=-1)

    start_idx = max(0, int(start // 2))
    end_idx = min(len(features), int(end // 2) + 1)

    if end_idx <= start_idx:
        return 0.0

    similarities = features[start_idx:end_idx] @ query_feature

    return float(similarities.mean().item())

print("CLIP 후보 평가 함수 준비 완료!")

CLIP 후보 평가 함수 준비 완료!


In [82]:
query = "a travel makeup pouch"

test_candidates = [
    ("정답 후보 (자막)", 3.08, 7.28),
    ("오답 후보 (Visual 1위)", 18.07, 24.49),
    ("오답 후보 (Visual 2위)", 27.33, 29.30),
    ("오답 후보 (Visual 3위)", 9.36, 16.25)
]

for name, start, end in test_candidates:
    score = get_visual_candidate_score(
        query=query,
        feature_path=daiso_feature_path,
        start=start,
        end=end
    )

    print(f"{name}: {score:.4f}")

정답 후보 (자막): 0.2382
오답 후보 (Visual 1위): 0.2215
오답 후보 (Visual 2위): 0.1960
오답 후보 (Visual 3위): 0.2057


In [83]:
query = "cutting tofu with a knife"

test_candidates = [
    ("정답 후보 (자막)", 12.88, 21.06),
    ("오답 후보 (Visual 1위)", 101.77, 118.97),
    ("오답 후보 (Visual 2위)", 86.53, 102.79),
    ("오답 후보 (Visual 3위)", 116.19, 135.90),
    ("오답 후보 (Visual 4위)", 30.60, 49.11),
    ("오답 후보 (Visual 5위)", 145.59, 163.87)
]

for name, start, end in test_candidates:
    score = get_visual_candidate_score(
        query=query,
        feature_path=feature_path,
        start=start,
        end=end
    )

    print(f"{name}: {score:.4f}")

정답 후보 (자막): 0.3113
오답 후보 (Visual 1위): 0.2776
오답 후보 (Visual 2위): 0.2917
오답 후보 (Visual 3위): 0.2613
오답 후보 (Visual 4위): 0.3193
오답 후보 (Visual 5위): 0.2865


In [84]:
query = "squeezing sauce with scissors"

test_candidates = [
    ("정답 후보 (자막)", 5.96, 8.08),
    ("정답 후보 (Visual)", 4.50, 10.28),
    ("오답 후보 (Visual 2)", 23.40, 31.72),
    ("오답 후보 (Visual 3)", 13.93, 20.53)
]

for name, start, end in test_candidates:
    score = get_visual_candidate_score(
        query=query,
        feature_path=kitchen_feature_path,
        start=start,
        end=end
    )

    print(f"{name}: {score:.4f}")

정답 후보 (자막): 0.2961
정답 후보 (Visual): 0.2971
오답 후보 (Visual 2): 0.2765
오답 후보 (Visual 3): 0.2974


In [85]:
from pathlib import Path
import shutil

feature_backup_dir = video_dir / "clip_features"
feature_backup_dir.mkdir(exist_ok=True)

for video_id in [
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_daiso_001"
]:
    source = Path(
        f"/content/pinpoint-capstone/data/features/{video_id}_clip_features.pt"
    )
    destination = feature_backup_dir / source.name

    shutil.copy2(source, destination)
    print("백업 완료:", destination)

백업 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/clip_features/kr_tofu_001_clip_features.pt
백업 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/clip_features/kr_kitchen_001_clip_features.pt
백업 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/clip_features/kr_daiso_001_clip_features.pt


In [86]:
from pathlib import Path
import shutil

source = Path(
    "/content/pinpoint-capstone/ai/features/run_moment_detr_inference_before_fusion.py"
)

destination = video_dir / source.name

shutil.copy2(source, destination)

print("코드 백업 완료:", destination)
print("파일 크기:", destination.stat().st_size)

코드 백업 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/run_moment_detr_inference_before_fusion.py
파일 크기: 28490


In [87]:
!sed -n '1340,1405p' ai/features/run_moment_detr_inference.py

                if subtitle_result is None
                else subtitle_result["score"]
            )

        if video_duration is not None:

            final_start = min(
                final_start,
                float(video_duration)
            )

        hybrid_results.append({
            "video_id": video_id,
            "query": query,
            "start_time": round(
                float(final_start),
                4
            ),
            "end_time": candidate[
                "end_time"
            ],
            "score": candidate[
                "score"
            ],
            "source": source,
            "subtitle_score": (
                None
                if subtitle_score is None
                else round(
                    float(subtitle_score),
                    4
                )
            )
        })

    # Moment-DETR score 높은 순서 유지
    hybrid_results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # Hybrid 보정 후

In [88]:
def make_fusion_result(
    video_id,
    query,
    candidate
):
    source = candidate["source"]

    return {
        "video_id": video_id,
        "query": query,
        "start_time": round(float(candidate["start_time"]), 4),
        "end_time": round(float(candidate["end_time"]), 4),
        "score": candidate.get("score") if source == "visual" else None,
        "source": source,
        "visual_score": (
            candidate.get("score") if source == "visual" else None
        ),
        "subtitle_score": (
            candidate.get("score") if source == "subtitle" else None
        ),
        "language": candidate.get("language")
    }

print("통합 검색 결과 형식 준비 완료!")

통합 검색 결과 형식 준비 완료!


In [89]:
def predict_moments_fusion(
    video_id, feature_path, subtitles,
    korean_query, english_query,
    video_duration, top_k=5
):
    visual = predict_moments(
        video_id=video_id,
        video_feature_path=feature_path,
        query=english_query,
        video_duration=video_duration,
        top_k=top_k
    )

    visual = [
        dict(r, source="visual")
        for r in visual
    ]

    subtitle = get_multilingual_subtitle_candidates(
        korean_query=korean_query,
        english_query=english_query,
        subtitles=subtitles,
        model=semantic_model,
        top_k=top_k
    )

    subtitle = deduplicate_multilingual_candidates(subtitle)

    # 영상 후보와 자막 후보를 번갈아 배치
    ordered = []

    for i in range(max(len(visual), len(subtitle))):
        if i < len(visual):
            ordered.append(visual[i])
        if i < len(subtitle):
            ordered.append(subtitle[i])

    # 중복 후보 제거
    selected = []

    for candidate in ordered:
        duplicate = any(
            temporal_iou(candidate, kept) >= 0.5
            or abs(candidate["start_time"] - kept["start_time"]) < 5
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

        if len(selected) >= top_k:
            break

    return [
        make_fusion_result(video_id, korean_query, r)
        for r in selected
    ]


print("PinPoint 통합 검색 함수 준비 완료!")

PinPoint 통합 검색 함수 준비 완료!


In [90]:
daiso_fusion_results = predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path=daiso_feature_path,
    subtitles=daiso_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    video_duration=32.07,
    top_k=5
)

for rank, r in enumerate(daiso_fusion_results, 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f} ~ {r["end_time"]:.2f}초'
    )
    print(
        "영상 점수:", r["visual_score"],
        "| 자막 점수:", r["subtitle_score"]
    )

1위 | visual | 18.07 ~ 24.49초
영상 점수: 0.9286 | 자막 점수: None
2위 | subtitle | 2.08 ~ 7.28초
영상 점수: None | 자막 점수: 0.6598
3위 | visual | 27.33 ~ 29.30초
영상 점수: 0.8823 | 자막 점수: None
4위 | visual | 9.36 ~ 16.25초
영상 점수: 0.0028 | 자막 점수: None


In [91]:
tofu_fusion_results = predict_moments_fusion(
    video_id="kr_tofu_001",
    feature_path=feature_path,
    subtitles=subtitles,
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    video_duration=165.0,
    top_k=5
)

for rank, r in enumerate(tofu_fusion_results, 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f} ~ {r["end_time"]:.2f}초'
    )
    print(
        "영상 점수:", r["visual_score"],
        "| 자막 점수:", r["subtitle_score"],
        "| 언어:", r["language"]
    )

1위 | visual | 101.77 ~ 118.97초
영상 점수: 0.9842 | 자막 점수: None | 언어: None
2위 | subtitle | 140.86 ~ 143.58초
영상 점수: None | 자막 점수: 0.6732 | 언어: ko
3위 | visual | 86.53 ~ 102.79초
영상 점수: 0.9608 | 자막 점수: None | 언어: None
4위 | subtitle | 135.06 ~ 137.58초
영상 점수: None | 자막 점수: 0.6575 | 언어: ko
5위 | visual | 116.19 ~ 135.90초
영상 점수: 0.7395 | 자막 점수: None | 언어: None


In [92]:
tofu_subtitle_candidates = get_multilingual_subtitle_candidates(
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    subtitles=subtitles,
    model=semantic_model,
    top_k=5
)

tofu_subtitle_unique = deduplicate_multilingual_candidates(
    tofu_subtitle_candidates
)

for rank, r in enumerate(tofu_subtitle_unique, 1):
    print(
        f'{rank}위 | {r["language"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초'
    )

1위 | ko | 140.86~143.58초
2위 | ko | 135.06~137.58초
3위 | ko | 97.94~101.62초
4위 | ko | 16.32~19.02초
5위 | en | 8.70~19.02초


In [93]:
def predict_moments_fusion(
    video_id, feature_path, subtitles,
    korean_query, english_query,
    video_duration, top_k=5
):
    visual = [
        dict(r, source="visual")
        for r in predict_moments(
            video_id=video_id,
            video_feature_path=feature_path,
            query=english_query,
            video_duration=video_duration,
            top_k=top_k
        )
    ]

    subtitle = get_multilingual_subtitle_candidates(
        korean_query=korean_query,
        english_query=english_query,
        subtitles=subtitles,
        model=semantic_model,
        top_k=top_k
    )

    ko = [r for r in subtitle if r["language"] == "ko"]
    en = [r for r in subtitle if r["language"] == "en"]

    # 첫 후보부터 세 검색 방식의 순서를 번갈아 사용
    ordered = []
    for i in range(max(len(visual), len(ko), len(en))):
        for group in (visual, ko, en):
            if i < len(group):
                ordered.append(group[i])

    selected = []

    for candidate in ordered:
        duplicate = any(
            temporal_iou(candidate, kept) >= 0.5
            or abs(candidate["start_time"] - kept["start_time"]) < 5.0
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

        if len(selected) >= top_k:
            break

    return [
        make_fusion_result(video_id, korean_query, r)
        for r in selected
    ]

print("언어별 후보 교차 배치 함수 준비 완료!")

언어별 후보 교차 배치 함수 준비 완료!


In [94]:
tofu_fusion_results = predict_moments_fusion(
    video_id="kr_tofu_001",
    feature_path=feature_path,
    subtitles=subtitles,
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    video_duration=165.0,
    top_k=5
)

for rank, r in enumerate(tofu_fusion_results, 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'언어={r["language"]}'
    )

1위 | visual | 101.77~118.97초 | 언어=None
2위 | subtitle | 140.86~143.58초 | 언어=ko
3위 | subtitle | 12.88~21.06초 | 언어=en
4위 | visual | 86.53~102.79초 | 언어=None
5위 | subtitle | 135.06~137.58초 | 언어=ko


In [95]:
kitchen_fusion_results = predict_moments_fusion(
    video_id=kitchen_video_id,
    feature_path=kitchen_feature_path,
    subtitles=kitchen_subtitles,
    korean_query="가위로 소스를 짜는 장면",
    english_query="squeezing sauce with scissors",
    video_duration=37.13,
    top_k=5
)

for rank, r in enumerate(kitchen_fusion_results, 1):
    start = r["start_time"]
    end = r["end_time"]
    source = r["source"]
    language = r["language"]

    print(
        f"{rank}위 | {source} | "
        f"{start:.2f}~{end:.2f}초 | 언어={language}"
    )

1위 | visual | 4.50~10.28초 | 언어=None
2위 | subtitle | 21.52~26.56초 | 언어=en
3위 | subtitle | 33.52~34.80초 | 언어=ko
4위 | subtitle | 15.96~21.52초 | 언어=en


In [96]:
daiso_fusion_results = predict_moments_fusion(
    video_id=daiso_video_id,
    feature_path=daiso_feature_path,
    subtitles=daiso_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    video_duration=32.07,
    top_k=5
)

for rank, r in enumerate(daiso_fusion_results, 1):
    print(
        f'{rank}위 | {r["source"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'언어={r["language"]}'
    )

1위 | visual | 18.07~24.49초 | 언어=None
2위 | subtitle | 2.08~7.28초 | 언어=ko
3위 | visual | 27.33~29.30초 | 언어=None
4위 | visual | 9.36~16.25초 | 언어=None


In [97]:
import inspect
from pathlib import Path

functions_to_save = [
    get_semantic_subtitle_candidates,
    get_multilingual_subtitle_candidates,
    deduplicate_multilingual_candidates,
    make_fusion_result,
    predict_moments_fusion
]

output_path = video_dir / "fusion_experiment.py"

header = """\
from sentence_transformers import util
from ai.features.run_moment_detr_inference import predict_moments

# temporal_iou는 아래에 별도로 정의
"""

source_code = header + "\n\n" + inspect.getsource(temporal_iou)

for func in functions_to_save:
    source_code += "\n\n" + inspect.getsource(func)

output_path.write_text(source_code, encoding="utf-8")

print("통합 검색 실험 코드 저장 완료:", output_path)
print("파일 크기:", output_path.stat().st_size)

통합 검색 실험 코드 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/fusion_experiment.py
파일 크기: 4791


In [98]:
!grep -n -E "semantic_model|predict_moments|temporal_iou|^def |^import |^from " "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/fusion_experiment.py"

1:from sentence_transformers import util
2:from ai.features.run_moment_detr_inference import predict_moments
4:# temporal_iou는 아래에 별도로 정의
7:def temporal_iou(a, b):
22:def get_semantic_subtitle_candidates(
69:def get_multilingual_subtitle_candidates(
99:def deduplicate_multilingual_candidates(
110:            temporal_iou(candidate, kept) >= iou_threshold
126:def make_fusion_result(
150:def predict_moments_fusion(
157:        for r in predict_moments(
170:        model=semantic_model,
188:            temporal_iou(candidate, kept) >= 0.5


In [99]:
from pathlib import Path

fusion_path = video_dir / "fusion_experiment.py"
code = fusion_path.read_text(encoding="utf-8")

old = "        model=semantic_model,"

new = "        model=get_semantic_model(),"

assert old in code, "수정할 코드를 찾지 못했습니다."

code = code.replace(old, new)

model_loader = '''
from functools import lru_cache
from sentence_transformers import SentenceTransformer

@lru_cache(maxsize=1)
def get_semantic_model():
    return SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
'''

code = model_loader + "\n" + code
fusion_path.write_text(code, encoding="utf-8")

print("의미 검색 모델 로딩 함수 추가 완료!")
print("semantic_model 직접 참조 남음:", "model=semantic_model" in code)

의미 검색 모델 로딩 함수 추가 완료!
semantic_model 직접 참조 남음: False


In [100]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "pinpoint_fusion",
    str(fusion_path)
)

fusion_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fusion_module)

print("통합 검색 모듈 불러오기 성공!")
print(
    "의미 검색 모델 로딩 함수:",
    hasattr(fusion_module, "get_semantic_model")
)
print(
    "통합 검색 함수:",
    hasattr(fusion_module, "predict_moments_fusion")
)

통합 검색 모듈 불러오기 성공!
의미 검색 모델 로딩 함수: True
통합 검색 함수: True


In [101]:
result = fusion_module.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path=daiso_feature_path,
    subtitles=daiso_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    video_duration=32.07,
    top_k=5
)

for rank, r in enumerate(result, 1):
    print(
        f"{rank}위 | {r['source']} | "
        f"{r['start_time']:.2f}~{r['end_time']:.2f}초 | "
        f"언어={r['language']}"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

1위 | visual | 18.07~24.49초 | 언어=None
2위 | subtitle | 2.08~7.28초 | 언어=ko
3위 | visual | 27.33~29.30초 | 언어=None
4위 | visual | 9.36~16.25초 | 언어=None


In [102]:
from pathlib import Path
import shutil

source = video_dir / "fusion_experiment.py"

destination = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

shutil.copy2(source, destination)

print("프로젝트 파일 추가 완료:", destination)
print("파일 크기:", destination.stat().st_size)

프로젝트 파일 추가 완료: /content/pinpoint-capstone/ai/features/fusion_search.py
파일 크기: 5042


In [103]:
import py_compile

py_compile.compile(
    "ai/features/fusion_search.py",
    doraise=True
)

print("통합 검색 모듈 문법 검사 통과!")

통합 검색 모듈 문법 검사 통과!


In [104]:
import inspect
from ai.features.fusion_search import predict_moments_fusion

print("함수 입력:")
print(inspect.signature(predict_moments_fusion))

print("\n실제 검색 결과:")
for r in result[:2]:
    print(r)

함수 입력:
(video_id, feature_path, subtitles, korean_query, english_query, video_duration, top_k=5)

실제 검색 결과:
{'video_id': 'kr_daiso_001', 'query': '여행용 화장품 파우치를 보여주는 장면', 'start_time': 18.0663, 'end_time': 24.4924, 'score': 0.9286, 'source': 'visual', 'visual_score': 0.9286, 'subtitle_score': None, 'language': None}
{'video_id': 'kr_daiso_001', 'query': '여행용 화장품 파우치를 보여주는 장면', 'start_time': 2.08, 'end_time': 7.28, 'score': None, 'source': 'subtitle', 'visual_score': None, 'subtitle_score': 0.6598, 'language': 'ko'}


In [105]:
!grep -RInE 'predict_moment|predict_moments|/api/search|@router.*search' backend --include="*.py" | head -60

In [106]:
!find backend -maxdepth 3 -type f | head -60

backend/.gitkeep


In [107]:
!git branch -r

  origin/HEAD -> origin/main
  origin/feature/backend
  origin/feature/frontend
  origin/main


In [108]:
!git ls-tree -r --name-only origin/feature/backend | grep -E '(^backend/|^api/|^app/|^server/)' | head -60

backend/.env.example
backend/.gitkeep
backend/main.py
backend/requirements.txt
backend/services/acquire.py
backend/services/indexer.py
backend/services/search_api.py
backend/services/youtube_search.py


In [109]:
!git show origin/feature/backend:backend/services/search_api.py

import re

import requests
from fastapi import APIRouter, BackgroundTasks, Depends, Header, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

try:
    from . import indexer
    from .youtube_search import fetch_video_meta, search_youtube
except ImportError:
    import indexer
    from youtube_search import fetch_video_meta, search_youtube

YOUTUBE_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")
MAX_DURATION_SEC = 30 * 60  # 임시값. 팀 합의 후 조정


class SearchRequest(BaseModel):
    youtube_video_id: str
    query: str
    top_k: int = 5

class MultiSearchRequest(BaseModel):
    query: str
    youtube_video_ids: list[str] | None = None
    top_k_per_video: int = 3
    max_videos: int = 10
    auto_analyze: bool = False      # true면 미분석 영상의 분석을 백그라운드로 시작
    max_new_analyses: int = 3       # 한 번의 요청으로 새로 시작할 최대 영상 수 (최대 5)

def _err(status: int, code: str, message: str) -> JSONResponse:
    return JSONResponse(status_code=status,
                        conten

In [110]:
!git show origin/feature/backend:backend/services/indexer.py | grep -n -E "subtitle|transcript|feature_path|video_duration|duration|USE_REAL_AI"

19:USE_REAL_AI = os.getenv("USE_REAL_AI", "0") == "1"
20:MODEL_VERSION = "moment-detr-clip" if USE_REAL_AI else "stub"
28:    return bool(feat) and (not USE_REAL_AI or feat.get("model_version") == MODEL_VERSION)
46:    if USE_REAL_AI:
154:        .select("video_id, feature_path, model_version, id")
160:    fmap = {f["video_id"]: f["feature_path"] for f in feats if _usable(f)}
161:    return [{**r, "feature_path": fmap[r["video_id"]]} for r in rows if r["video_id"] in fmap]
176:        "feature_path": feat["feature_path"] if feat else None,
188:        return {"state": "READY", "video_id": vid, "feature_path": feat["feature_path"]}
207:        feature_path = _to_stored_path(build_feature(str(video_path), youtube_video_id))
211:            "feature_path": feature_path,


In [111]:
!git show origin/feature/backend:backend/services/indexer.py | sed -n '190,245p'

    if row["status"] == "PROCESSING":
        if not reclaim_stale(vid):
            return {"state": "PROCESSING", "video_id": vid}

    if row["status"] == "INDEXED":  # INDEXED인데 feature가 없는 비정상 상태
        set_status(vid, "PENDING")

    if claim(vid):
        return {"state": "STARTED", "video_id": vid}
    return {"state": "PROCESSING", "video_id": vid}


def run_indexing(video_id: str, youtube_video_id: str) -> None:
    """다운로드 → feature 생성 → 저장 → mp4 삭제. 오래 걸려서 백그라운드로 실행합니다."""
    video_path = None
    try:
        video_path = download_video(youtube_video_id, out_dir=str(RAW_DIR))
        feature_path = _to_stored_path(build_feature(str(video_path), youtube_video_id))
        supabase.table("video_features").delete().eq("video_id", video_id).execute()
        supabase.table("video_features").insert({
            "video_id": video_id,
            "feature_path": feature_path,
            "model_version": MODEL_VERSION,
        }).execute()
        supabase.table("videos").upda

In [112]:
!find ai -type f -name "*.py" | grep -E "transcri|whisper|audio|subtitle"

ai/preprocessing/transcribe.py
ai/preprocessing/parse_subtitle.py
ai/preprocessing/align_transcript.py
ai/preprocessing/extract_audio.py


In [113]:
!cat ai/preprocessing/extract_audio.py ai/preprocessing/transcribe.py

import subprocess
from pathlib import Path

video_path = "data/raw/test_audio.mp4"

output_dir = Path("data/processed/video_001")
output_dir.mkdir(parents=True, exist_ok=True)

audio_path = output_dir / "audio.wav"

command = [
    "ffmpeg",
    "-y",
    "-i", video_path,
    "-vn",
    "-acodec", "pcm_s16le",
    "-ar", "16000",
    "-ac", "1",
    str(audio_path)
]

print("오디오 추출 시작...")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("오디오 추출 완료!")
    print(f"저장 위치: {audio_path}")
else:
    print("오디오 추출 실패")
    print(result.stderr)import json
from pathlib import Path

from faster_whisper import WhisperModel


audio_path = "data/processed/video_001/audio.wav"
output_path = Path("data/processed/video_001/transcript.json")

# 처음 테스트는 CPU에서도 부담이 덜한 small 모델 사용
model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("STT 시작...")

segments, info = model.transcribe(
    audio_path,
    be

In [114]:
from pathlib import Path

code = '''
import json
from pathlib import Path
from functools import lru_cache

from faster_whisper import WhisperModel


@lru_cache(maxsize=1)
def get_whisper_model():
    return WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )


def transcribe_video(video_path, video_id):
    output_path = (
        Path("data/processed")
        / video_id
        / "subtitles.json"
    )
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    segments, info = get_whisper_model().transcribe(
        str(video_path),
        beam_size=5
    )

    subtitles = [
        {
            "start_time": round(s.start, 2),
            "end_time": round(s.end, 2),
            "text": s.text.strip()
        }
        for s in segments
    ]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            subtitles,
            f,
            ensure_ascii=False,
            indent=2
        )

    return str(output_path)
'''

path = Path("ai/preprocessing/transcribe_service.py")
path.write_text(code.strip() + "\n", encoding="utf-8")

print("서비스용 자막 생성 파일 작성 완료:", path)

서비스용 자막 생성 파일 작성 완료: ai/preprocessing/transcribe_service.py


In [115]:
from ai.preprocessing.transcribe_service import transcribe_video
import json

test_subtitle_path = transcribe_video(
    video_path=str(videos["daiso"]),
    video_id="kr_daiso_001"
)

with open(test_subtitle_path, "r", encoding="utf-8") as f:
    test_subtitles = json.load(f)

print("자막 생성 완료:", test_subtitle_path)
print("자막 개수:", len(test_subtitles))
print("첫 번째 자막:", test_subtitles[0] if test_subtitles else "없음")

자막 생성 완료: data/processed/kr_daiso_001/subtitles.json
자막 개수: 19
첫 번째 자막: {'start_time': 0.0, 'end_time': 2.08, 'text': '팅으로 잘 썼던 다이소 상반기 결산템디'}


In [116]:
import py_compile

py_compile.compile(
    "ai/preprocessing/transcribe_service.py",
    doraise=True
)

print("서비스용 자막 생성 코드 문법 검사 통과!")

서비스용 자막 생성 코드 문법 검사 통과!


In [117]:
!git show origin/feature/backend:backend/services/indexer.py | sed -n '1,85p'

import os
import threading
from datetime import datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client

try:
    from .acquire import download_video
except ImportError:
    from acquire import download_video

ROOT = Path(__file__).resolve().parents[2]
RAW_DIR = ROOT / "data" / "raw"
FEATURE_DIR = ROOT / "data" / "features"

load_dotenv(ROOT / "backend" / ".env")
USE_REAL_AI = os.getenv("USE_REAL_AI", "0") == "1"
MODEL_VERSION = "moment-detr-clip" if USE_REAL_AI else "stub"
_AI_LOCK = threading.Lock()  # GPU를 한 번에 하나만 쓰게 함

supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])


def _usable(feat) -> bool:
    """진짜 모드는 같은 모델 버전만, 가짜 모드는 이미 분석된 기록을 그대로 인정."""
    return bool(feat) and (not USE_REAL_AI or feat.get("model_version") == MODEL_VERSION)


def reclaim_stale(video_id: str, minutes: int = 15) -> bool:
    """서버가 꺼지는 바람에 멈춘 PROCESSING을 PENDING으로 되돌립니다."""
    cutoff = (datetime.no

In [118]:
from pathlib import Path

project_root = Path("/content/pinpoint-capstone")

def to_stored_path(path):
    return Path(path).resolve().relative_to(project_root).as_posix()

subtitle_stored_path = to_stored_path(test_subtitle_path)

print("자막 실제 경로:", Path(test_subtitle_path).resolve())
print("DB 저장용 경로:", subtitle_stored_path)
print("파일 존재 여부:", (project_root / subtitle_stored_path).exists())

자막 실제 경로: /content/pinpoint-capstone/data/processed/kr_daiso_001/subtitles.json
DB 저장용 경로: data/processed/kr_daiso_001/subtitles.json
파일 존재 여부: True


In [119]:
import subprocess
from pathlib import Path

target = Path("backend/services/indexer.py")
target.parent.mkdir(parents=True, exist_ok=True)

code = subprocess.check_output(
    ["git", "show", "origin/feature/backend:backend/services/indexer.py"],
    text=True
)

target.write_text(code, encoding="utf-8")

print("백엔드 파일 준비 완료:", target)
print("파일 크기:", target.stat().st_size)

백엔드 파일 준비 완료: backend/services/indexer.py
파일 크기: 7976


In [120]:
import shutil
from pathlib import Path

source = Path("backend/services/indexer.py")
backup = video_dir / "indexer_before_subtitle_integration.py"

shutil.copy2(source, backup)

print("백엔드 원본 백업 완료:", backup)
print("파일 크기:", backup.stat().st_size)

백엔드 원본 백업 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/indexer_before_subtitle_integration.py
파일 크기: 7976


In [121]:
from pathlib import Path

path = Path("backend/services/indexer.py")
code = path.read_text(encoding="utf-8")

old = '        feature_path = _to_stored_path(build_feature(str(video_path), youtube_video_id))'

new = '''        feature_path = _to_stored_path(build_feature(str(video_path), youtube_video_id))

        # Whisper 자막 생성 (실패해도 기존 영상 검색 유지)
        try:
            from ai.preprocessing.transcribe_service import transcribe_video

            subtitle_path = transcribe_video(
                video_path=str(video_path),
                video_id=youtube_video_id
            )
            print(f"[AI] Subtitle saved: {subtitle_path}")

        except Exception as subtitle_error:
            print(f"[AI] Subtitle generation failed: {subtitle_error}")'''

assert code.count(old) == 1, "수정할 위치를 정확히 찾지 못했습니다."

code = code.replace(old, new)
path.write_text(code, encoding="utf-8")

print("백엔드 Whisper 자막 생성 단계 추가 완료!")

백엔드 Whisper 자막 생성 단계 추가 완료!


In [122]:
import py_compile

py_compile.compile(
    "backend/services/indexer.py",
    doraise=True
)

print("백엔드 자막 연동 코드 문법 검사 통과!")

백엔드 자막 연동 코드 문법 검사 통과!


In [123]:
!sed -n '200,245p' backend/services/indexer.py



def run_indexing(video_id: str, youtube_video_id: str) -> None:
    """다운로드 → feature 생성 → 저장 → mp4 삭제. 오래 걸려서 백그라운드로 실행합니다."""
    video_path = None
    try:
        video_path = download_video(youtube_video_id, out_dir=str(RAW_DIR))
        feature_path = _to_stored_path(build_feature(str(video_path), youtube_video_id))

        # Whisper 자막 생성 (실패해도 기존 영상 검색 유지)
        try:
            from ai.preprocessing.transcribe_service import transcribe_video

            subtitle_path = transcribe_video(
                video_path=str(video_path),
                video_id=youtube_video_id
            )
            print(f"[AI] Subtitle saved: {subtitle_path}")

        except Exception as subtitle_error:
            print(f"[AI] Subtitle generation failed: {subtitle_error}")
        supabase.table("video_features").delete().eq("video_id", video_id).execute()
        supabase.table("video_features").insert({
            "video_id": video_id,
            "feature_path": feature_path,
      

In [124]:
import inspect
from ai.features.run_moment_detr_inference import predict_moments_hybrid

print(inspect.signature(predict_moments_hybrid))

(video_id, video_feature_path, query, subtitle_json_path=None, video_duration=None, top_k=5, iou_threshold=0.5, min_start_gap=8.0)


In [125]:
from pathlib import Path

path = Path("ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

# 기존 import 변경
old_import = "from ai.features.run_moment_detr_inference import predict_moments"
new_import = "from ai.features.run_moment_detr_inference import predict_moments_hybrid"

# 기존 영상 검색 호출만 변경
old_call = """for r in predict_moments(
            video_id=video_id,
            video_feature_path=feature_path,
            query=english_query,
            video_duration=video_duration,
            top_k=top_k
        )"""

new_call = """for r in predict_moments_hybrid(
            video_id=video_id,
            video_feature_path=feature_path,
            query=english_query,
            subtitle_json_path=subtitle_json_path,
            video_duration=video_duration,
            top_k=top_k
        )"""

assert old_import in code, "import 위치를 찾지 못했습니다."
assert old_call in code, "Hybrid 호출 변경 위치를 찾지 못했습니다."

code = code.replace(old_import, new_import)
code = code.replace(old_call, new_call)

# 함수 인자에 자막 JSON 경로 추가
old_args = "video_duration, top_k=5\n):"
new_args = "video_duration, subtitle_json_path=None, top_k=5\n):"

assert code.count(old_args) == 1, "함수 인자 변경 위치를 확인해 주세요."
code = code.replace(old_args, new_args)

path.write_text(code, encoding="utf-8")

print("기존 Hybrid 보존형 통합 검색 코드 수정 완료!")

기존 Hybrid 보존형 통합 검색 코드 수정 완료!


# 수정한 통합 검색 함수 실행 테스트

In [126]:
import importlib
import ai.features.fusion_search as fusion_search

importlib.reload(fusion_search)

results = fusion_search.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path=daiso_feature_path,
    subtitles=daiso_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    video_duration=32.07,
    subtitle_json_path=str(daiso_subtitle_path),
    top_k=5
)

for rank, r in enumerate(results, 1):
    print(
        f"{rank}위 | {r['source']} | "
        f"{r['start_time']:.2f}~{r['end_time']:.2f}초 | "
        f"언어={r['language']}"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

1위 | visual | 16.50~24.49초 | 언어=None
2위 | subtitle | 2.08~7.28초 | 언어=ko
3위 | visual | 24.50~29.30초 | 언어=None


In [127]:
from ai.features.run_moment_detr_inference import predict_moments_hybrid

hybrid_results = predict_moments_hybrid(
    video_id="kr_daiso_001",
    video_feature_path=daiso_feature_path,
    query="a travel makeup pouch",
    subtitle_json_path=str(daiso_subtitle_path),
    video_duration=32.07,
    top_k=5
)

for rank, r in enumerate(hybrid_results, 1):
    print(f"{rank}위:", r)

1위: {'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 16.5, 'end_time': 24.4924, 'score': 0.9286, 'source': 'visual', 'subtitle_score': 0.0}
2위: {'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 24.5, 'end_time': 29.2973, 'score': 0.8823, 'source': 'visual', 'subtitle_score': 0.0}
3위: {'video_id': 'kr_daiso_001', 'query': 'a travel makeup pouch', 'start_time': 6.5, 'end_time': 16.2451, 'score': 0.0028, 'source': 'visual', 'subtitle_score': 0.0}


In [128]:
from pathlib import Path

path = Path("ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

old = 'dict(r, source="visual")'
new = 'dict(r, source="hybrid", hybrid_source=r.get("source"))'

assert code.count(old) == 1, "수정 위치를 확인해 주세요."

code = code.replace(old, new)
path.write_text(code, encoding="utf-8")

print("Hybrid 원본 source 정보 보존 완료!")

Hybrid 원본 source 정보 보존 완료!


In [129]:
!sed -n '125,165p' ai/features/fusion_search.py

                ) < start_gap
            )
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

    return selected


def make_fusion_result(
    video_id,
    query,
    candidate
):
    source = candidate["source"]

    return {
        "video_id": video_id,
        "query": query,
        "start_time": round(float(candidate["start_time"]), 4),
        "end_time": round(float(candidate["end_time"]), 4),
        "score": candidate.get("score") if source == "visual" else None,
        "source": source,
        "visual_score": (
            candidate.get("score") if source == "visual" else None
        ),
        "subtitle_score": (
            candidate.get("score") if source == "subtitle" else None
        ),
        "language": candidate.get("language")
    }


def predict_moments_fusion(
    video_id, feature_path, subtitles,
    korean_query, english_query,
    video_duration, subtitle_json_path=None, top_k=5
):
    visual =

In [130]:
from pathlib import Path

path = Path("ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

start = code.index("def make_fusion_result(")
end = code.index("\ndef predict_moments_fusion(", start)

new_function = '''def make_fusion_result(video_id, query, candidate):
    source = candidate["source"]
    is_hybrid = source in ("hybrid", "visual")

    return {
        "video_id": video_id,
        "query": query,
        "start_time": round(float(candidate["start_time"]), 4),
        "end_time": round(float(candidate["end_time"]), 4),
        "score": candidate.get("score") if is_hybrid else None,
        "source": source,
        "hybrid_source": candidate.get("hybrid_source"),
        "visual_score": candidate.get("score") if is_hybrid else None,
        "subtitle_score": (
            candidate.get("subtitle_score")
            if is_hybrid
            else candidate.get("score")
        ),
        "language": candidate.get("language")
    }

'''

code = code[:start] + new_function + code[end:]
path.write_text(code, encoding="utf-8")

print("통합 검색 결과 변환 함수 수정 완료!")

통합 검색 결과 변환 함수 수정 완료!


In [131]:
import importlib
import ai.features.fusion_search as fusion_search

importlib.reload(fusion_search)

results = fusion_search.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path=daiso_feature_path,
    subtitles=daiso_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    video_duration=32.07,
    subtitle_json_path=str(daiso_subtitle_path),
    top_k=5
)

for rank, r in enumerate(results, 1):
    print(f"\n{rank}위")
    print("구간:", r["start_time"], "~", r["end_time"])
    print("source:", r["source"])
    print("hybrid_source:", r["hybrid_source"])
    print("score:", r["score"])
    print("visual_score:", r["visual_score"])
    print("subtitle_score:", r["subtitle_score"])
    print("language:", r["language"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


1위
구간: 16.5 ~ 24.4924
source: hybrid
hybrid_source: visual
score: 0.9286
visual_score: 0.9286
subtitle_score: 0.0
language: None

2위
구간: 2.08 ~ 7.28
source: subtitle
hybrid_source: None
score: None
visual_score: None
subtitle_score: 0.6598
language: ko

3위
구간: 24.5 ~ 29.2973
source: hybrid
hybrid_source: visual
score: 0.8823
visual_score: 0.8823
subtitle_score: 0.0
language: None


In [132]:
from pathlib import Path

def resolve_subtitle_path(video_id):
    path = (
        Path("/content/pinpoint-capstone/data/processed")
        / video_id
        / "subtitles.json"
    )
    return str(path) if path.is_file() else None

print("다이소 자막:", resolve_subtitle_path("kr_daiso_001"))
print("없는 자막:", resolve_subtitle_path("nonexistent_video"))

다이소 자막: /content/pinpoint-capstone/data/processed/kr_daiso_001/subtitles.json
없는 자막: None


In [133]:
from pathlib import Path

path = Path("backend/services/indexer.py")
code = path.read_text(encoding="utf-8")

new_function = '''
def resolve_subtitle_path(youtube_video_id: str):
    """자막이 있으면 절대경로를 반환하고, 없으면 None을 반환합니다."""
    path = ROOT / "data" / "processed" / youtube_video_id / "subtitles.json"
    return str(path) if path.is_file() else None


'''

anchor = "def get_video(youtube_video_id: str)"

assert code.count(anchor) == 1, "함수 삽입 위치를 확인해 주세요."
assert "def resolve_subtitle_path(" not in code, "이미 추가된 함수입니다."

code = code.replace(anchor, new_function + anchor)
path.write_text(code, encoding="utf-8")

print("백엔드 자막 경로 조회 함수 추가 완료!")

백엔드 자막 경로 조회 함수 추가 완료!


In [134]:
import subprocess
from pathlib import Path

target = Path("backend/services/search_api.py")
target.parent.mkdir(parents=True, exist_ok=True)

code = subprocess.check_output(
    ["git", "show", "origin/feature/backend:backend/services/search_api.py"],
    text=True
)

target.write_text(code, encoding="utf-8")

print("검색 API 파일 준비 완료:", target)
print("파일 크기:", target.stat().st_size)

검색 API 파일 준비 완료: backend/services/search_api.py
파일 크기: 10700


In [135]:
import shutil
from pathlib import Path

source = Path("backend/services/search_api.py")
backup = video_dir / "search_api_before_fusion.py"

shutil.copy2(source, backup)

print("검색 API 원본 백업 완료:", backup)
print("원본 크기:", source.stat().st_size)
print("백업 크기:", backup.stat().st_size)

검색 API 원본 백업 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/search_api_before_fusion.py
원본 크기: 10700
백업 크기: 10700


In [136]:
!sed -n '1,75p' backend/services/search_api.py

import re

import requests
from fastapi import APIRouter, BackgroundTasks, Depends, Header, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

try:
    from . import indexer
    from .youtube_search import fetch_video_meta, search_youtube
except ImportError:
    import indexer
    from youtube_search import fetch_video_meta, search_youtube

YOUTUBE_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")
MAX_DURATION_SEC = 30 * 60  # 임시값. 팀 합의 후 조정


class SearchRequest(BaseModel):
    youtube_video_id: str
    query: str
    top_k: int = 5

class MultiSearchRequest(BaseModel):
    query: str
    youtube_video_ids: list[str] | None = None
    top_k_per_video: int = 3
    max_videos: int = 10
    auto_analyze: bool = False      # true면 미분석 영상의 분석을 백그라운드로 시작
    max_new_analyses: int = 3       # 한 번의 요청으로 새로 시작할 최대 영상 수 (최대 5)

def _err(status: int, code: str, message: str) -> JSONResponse:
    return JSONResponse(status_code=status,
                        conten

In [137]:
%pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00


In [1]:
%pip install "transformers==4.57.6" "sentencepiece"

In [2]:
from transformers import pipeline

ko_en_translator = pipeline(
    task="translation",
    model="Helsinki-NLP/opus-mt-ko-en",
    device=0
)

print("한국어 → 영어 번역 모델 로딩 완료!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

...

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


한국어 → 영어 번역 모델 로딩 완료!


In [3]:
test_queries = [
    "두부를 칼로 자르는 장면",
    "가위로 소스를 짜는 장면",
    "여행용 화장품 파우치를 보여주는 장면"
]

for query in test_queries:
    result = ko_en_translator(query, max_length=128)
    translated = result[0]["translation_text"]

    print("한국어:", query)
    print("영어:", translated)
    print()

한국어: 두부를 칼로 자르는 장면
영어: Cutting tofu with a knife

한국어: 가위로 소스를 짜는 장면
영어: Sweeping sauces

한국어: 여행용 화장품 파우치를 보여주는 장면
영어: View of a traveling cosmetics patch.



In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nllb_model_name = "facebook/nllb-200-distilled-600M"

nllb_tokenizer = AutoTokenizer.from_pretrained(
    nllb_model_name,
    src_lang="kor_Hang"
)

nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    nllb_model_name
).to("cuda")

nllb_model.eval()

print("NLLB 한국어 → 영어 번역 모델 로딩 완료!")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

NLLB 한국어 → 영어 번역 모델 로딩 완료!


In [5]:
import torch

test_queries = [
    "두부를 칼로 자르는 장면",
    "가위로 소스를 짜는 장면",
    "여행용 화장품 파우치를 보여주는 장면"
]

for query in test_queries:
    inputs = nllb_tokenizer(
        query,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = nllb_model.generate(
            **inputs,
            forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=128
        )

    translated = nllb_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("한국어:", query)
    print("영어:", translated)
    print()

한국어: 두부를 칼로 자르는 장면
영어: The scene of a knife cutting through both of them.

한국어: 가위로 소스를 짜는 장면
영어: The scene of the sauce being baked.

한국어: 여행용 화장품 파우치를 보여주는 장면
영어: A scene of a traveling cosmetics fountain.



In [6]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

m2m_name = "facebook/m2m100_418M"

m2m_tokenizer = M2M100Tokenizer.from_pretrained(m2m_name)
m2m_model = M2M100ForConditionalGeneration.from_pretrained(m2m_name).to("cuda")

m2m_model.eval()

print("M2M100 번역 모델 로딩 완료!")

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

M2M100 번역 모델 로딩 완료!


In [7]:
import torch

test_queries = [
    "두부를 칼로 자르는 장면",
    "가위로 소스를 짜는 장면",
    "여행용 화장품 파우치를 보여주는 장면"
]

m2m_tokenizer.src_lang = "ko"

for query in test_queries:
    inputs = m2m_tokenizer(
        query,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = m2m_model.generate(
            **inputs,
            forced_bos_token_id=m2m_tokenizer.get_lang_id("en"),
            max_new_tokens=128
        )

    translated = m2m_tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    print("한국어:", query)
    print("영어:", translated)
    print()

한국어: 두부를 칼로 자르는 장면
영어: Cut the two with a knife.

한국어: 가위로 소스를 짜는 장면
영어: When it comes to the sauce.

한국어: 여행용 화장품 파우치를 보여주는 장면
영어: Showing a travel package.



# 검색어 분리 구현

In [8]:
!grep -n -A 85 "^def predict_moments_fusion" ai/features/fusion_search.py

grep: ai/features/fusion_search.py: No such file or directory


In [9]:
!grep -n -A 75 "^def get_multilingual_subtitle_candidates" ai/features/fusion_search.py

grep: ai/features/fusion_search.py: No such file or directory


In [10]:
import json
from pathlib import Path

# 상대경로 대신 실제 전체 경로 사용
path = Path(
    "/content/pinpoint-capstone/data/processed/kr_daiso_001/subtitles.json"
)

print("현재 작업 폴더:", Path.cwd())
print("자막 파일 존재:", path.exists())

if path.exists():
    data = json.loads(path.read_text(encoding="utf-8"))

    print("데이터 타입:", type(data).__name__)
    print(
        "첫 번째 자막:",
        data[0] if isinstance(data, list) else str(data)[:500]
    )
else:
    print("자막 파일을 찾지 못했습니다.")

현재 작업 폴더: /content
자막 파일 존재: True
데이터 타입: list
첫 번째 자막: {'start_time': 0.0, 'end_time': 2.08, 'text': '팅으로 잘 썼던 다이소 상반기 결산템디'}


In [11]:
from pathlib import Path
import shutil
import json

# Google Drive에 저장된 기존 자막
source = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/kr_daiso_001_subtitles.json"
)

# 기존 코드에서 사용하는 Colab 경로
target = Path(
    "/content/pinpoint-capstone/"
    "data/processed/kr_daiso_001/subtitles.json"
)

print("Google Drive 자막 존재:", source.exists())
print("프로젝트 폴더 존재:", target.parents[2].exists())

if source.exists() and target.parents[2].exists():
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)

    data = json.loads(target.read_text(encoding="utf-8"))

    print("\n✅ 다이소 자막 복구 완료!")
    print("복구 경로:", target)
    print("데이터 타입:", type(data).__name__)
    print("자막 개수:", len(data))
    print("첫 번째 자막:", data[0] if isinstance(data, list) else str(data)[:300])
else:
    print("\n경로 확인이 필요합니다. 출력 결과를 보내주세요.")

Google Drive 자막 존재: True
프로젝트 폴더 존재: True

✅ 다이소 자막 복구 완료!
복구 경로: /content/pinpoint-capstone/data/processed/kr_daiso_001/subtitles.json
데이터 타입: list
자막 개수: 19
첫 번째 자막: {'start_time': 0.0, 'end_time': 2.08, 'text': '팅으로 잘 썼던 다이소 상반기 결산템디'}


In [12]:
%pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993335 sha256=f71a6556ba518715ba60116df41e2d4e20ae90f91ffe6ec3ebfd2fb4efa62658
  Stored in directory: /root/.cache/pip/wheels/eb/87/25/2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect


In [13]:
from pathlib import Path

video_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

print("영상 데이터 폴더:", video_dir)
print("폴더 존재 여부:", video_dir.exists())

for video_id in ["kr_tofu_001", "kr_kitchen_001", "kr_daiso_001"]:
    path = video_dir / f"{video_id}_subtitles.json"
    print(video_id, "자막 존재:", path.exists())

영상 데이터 폴더: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test
폴더 존재 여부: True
kr_tofu_001 자막 존재: True
kr_kitchen_001 자막 존재: True
kr_daiso_001 자막 존재: True


In [14]:
import json
from pathlib import Path
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 0

video_ids = [
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_daiso_001"
]

for video_id in video_ids:
    path = Path("data/processed") / video_id / "subtitles.json"

    if not path.exists():
        path = video_dir / f"{video_id}_subtitles.json"

    if not path.exists():
        print(f"{video_id}: 자막 파일 없음")
        continue

    subtitles = json.loads(path.read_text(encoding="utf-8"))

    full_text = " ".join(
        item.get("text", "") for item in subtitles
    ).strip()

    try:
        language = detect(full_text) if full_text else "unknown"
    except LangDetectException:
        language = "unknown"

    print(f"{video_id}: {language}")

kr_tofu_001: ko
kr_kitchen_001: ko
kr_daiso_001: ko


In [15]:
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 0

def select_subtitle_query(subtitles, korean_query, english_query):
    full_text = " ".join(
        str(item.get("text", ""))
        for item in subtitles
    ).strip()

    if not full_text:
        return "unknown", None

    try:
        language = detect(full_text)
    except LangDetectException:
        return "unknown", None

    if language == "ko":
        return "ko", korean_query

    if language == "en":
        return "en", english_query

    return language, None


print(
    select_subtitle_query(
        subtitles=data,
        korean_query="여행용 화장품 파우치를 보여주는 장면",
        english_query="showing a travel makeup pouch"
    )
)

('ko', '여행용 화장품 파우치를 보여주는 장면')


In [17]:
from pathlib import Path

repo = Path("/content/pinpoint-capstone")
fusion_file = repo / "ai/features/fusion_search.py"

print("현재 작업 폴더:", Path.cwd())
print("프로젝트 폴더 존재:", repo.exists())
print("Fusion 파일 존재:", fusion_file.exists())

if repo.exists():
    %cd /content/pinpoint-capstone

현재 작업 폴더: /content
프로젝트 폴더 존재: True
Fusion 파일 존재: True
/content/pinpoint-capstone


In [18]:
from pathlib import Path

path = Path("/content/pinpoint-capstone/ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

new_function = '''
def select_subtitle_query(subtitles, korean_query, english_query):
    from langdetect import detect, DetectorFactory, LangDetectException

    DetectorFactory.seed = 0

    full_text = " ".join(
        str(item.get("text", ""))
        for item in subtitles
    ).strip()

    if not full_text:
        return "unknown", None

    try:
        language = detect(full_text)
    except LangDetectException:
        return "unknown", None

    if language == "ko":
        return "ko", korean_query

    if language == "en":
        return "en", english_query

    return language, None


'''

anchor = "def get_multilingual_subtitle_candidates("

assert code.count(anchor) == 1, "함수 삽입 위치를 확인해 주세요."
assert "def select_subtitle_query(" not in code, "이미 함수가 추가되어 있습니다."

# 기존 코드 백업
backup_path = path.with_suffix(".py.bak")
backup_path.write_text(code, encoding="utf-8")

# 새 함수 추가
code = code.replace(anchor, new_function + anchor, 1)

# 문법 검사 후 저장
compile(code, str(path), "exec")
path.write_text(code, encoding="utf-8")

print("✅ 자막 언어별 검색어 선택 함수 추가 완료!")
print("✅ 기존 코드 백업:", backup_path)

✅ 자막 언어별 검색어 선택 함수 추가 완료!
✅ 기존 코드 백업: /content/pinpoint-capstone/ai/features/fusion_search.py.bak


In [19]:
from pathlib import Path
import os

project_dir = Path("/content/pinpoint-capstone")
fusion_path = project_dir / "ai/features/fusion_search.py"

print("현재 작업 폴더:", Path.cwd())
print("프로젝트 폴더 존재:", project_dir.exists())
print("Fusion 파일 존재:", fusion_path.exists())

if project_dir.exists():
    os.chdir(project_dir)
    print("변경된 작업 폴더:", Path.cwd())

현재 작업 폴더: /content/pinpoint-capstone
프로젝트 폴더 존재: True
Fusion 파일 존재: True
변경된 작업 폴더: /content/pinpoint-capstone


In [20]:
from pathlib import Path

path = Path("ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

old = '''    subtitle = get_multilingual_subtitle_candidates(
        korean_query=korean_query,
        english_query=english_query,
        subtitles=subtitles,
        model=get_semantic_model(),
        top_k=top_k
    )
'''

new = '''    language, subtitle_query = select_subtitle_query(
        subtitles=subtitles,
        korean_query=korean_query,
        english_query=english_query
    )

    subtitle = []
    if subtitle_query is not None:
        subtitle = get_semantic_subtitle_candidates(
            query=subtitle_query,
            subtitles=subtitles,
            model=get_semantic_model(),
            top_k=top_k
        )
        subtitle = [
            dict(r, language=language, query_used=subtitle_query)
            for r in subtitle
        ]
'''

assert code.count(old) == 1, "교체할 코드 위치를 확인해 주세요."
code = code.replace(old, new, 1)

# 기존 ko/en 후보 분리 로직은 유지하므로 수정하지 않음
compile(code, str(path), "exec")
path.write_text(code, encoding="utf-8")

print("Fusion 자막 언어별 검색어 연결 완료!")

Fusion 자막 언어별 검색어 연결 완료!


In [21]:
import importlib
import json
from pathlib import Path
from ai.features import fusion_search

importlib.reload(fusion_search)

test_subtitles = json.loads(
    Path("data/processed/kr_daiso_001/subtitles.json").read_text(
        encoding="utf-8"
    )
)

print(
    "자막 검색 언어 및 검색어:",
    fusion_search.select_subtitle_query(
        subtitles=test_subtitles,
        korean_query="여행용 화장품 파우치를 보여주는 장면",
        english_query="showing a travel makeup pouch"
    )
)

results = fusion_search.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path="data/features/kr_daiso_001_clip_features.pt",
    subtitles=test_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="showing a travel makeup pouch",
    video_duration=32.07,
    subtitle_json_path="data/processed/kr_daiso_001/subtitles.json",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(
        i,
        result["source"],
        result["language"],
        result["start_time"],
        result["end_time"],
        "자막 점수:", result["subtitle_score"]
    )

자막 검색 언어 및 검색어: ('ko', '여행용 화장품 파우치를 보여주는 장면')
[AI] Loading CLIP on cuda
Loading CLIP models
[AI] Loading Moment-DETR on cuda
[AI] Loading CLIP for visual refinement...
1 hybrid None 14.5 25.0135 자막 점수: 0.0
2 subtitle ko 2.08 7.28 자막 점수: 0.6598
3 hybrid None 24.5 28.4991 자막 점수: 0.0


In [22]:
english_subtitles = [
    {
        "start_time": 0.0,
        "end_time": 5.0,
        "text": "Today I will show you my favorite travel makeup pouch."
    },
    {
        "start_time": 5.0,
        "end_time": 10.0,
        "text": "This small cosmetic bag is useful when traveling."
    }
]

language, query = fusion_search.select_subtitle_query(
    subtitles=english_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="showing a travel makeup pouch"
)

print("감지된 자막 언어:", language)
print("선택된 검색어:", query)

assert language == "en"
assert query == "showing a travel makeup pouch"

print("영어 자막 검색어 선택 테스트 통과!")

감지된 자막 언어: en
선택된 검색어: showing a travel makeup pouch
영어 자막 검색어 선택 테스트 통과!


In [23]:
import json
from ai.features import fusion_search

language, query = fusion_search.select_subtitle_query(
    subtitles=[],
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife"
)

print("자막 언어:", language)
print("선택된 자막 검색어:", query)

assert language == "unknown"
assert query is None

results = fusion_search.predict_moments_fusion(
    video_id="kr_tofu_001",
    feature_path="data/features/kr_tofu_001_clip_features.pt",
    subtitles=[],
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    video_duration=165.0,
    subtitle_json_path=None,
    top_k=3
)

for i, result in enumerate(results, 1):
    print(i, result["source"], result["start_time"], result["end_time"])

assert results, "검색 결과가 없습니다."
assert all(r["source"] == "hybrid" for r in results)

print("자막 없는 영상의 Hybrid 검색 테스트 통과!")

자막 언어: unknown
선택된 자막 검색어: None
1 hybrid 98.5 118.9725
2 hybrid 84.5 102.7907
3 hybrid 114.5 135.9028
자막 없는 영상의 Hybrid 검색 테스트 통과!


In [24]:
import json

for i, result in enumerate(results, 1):
    print(f"\n결과 {i}")
    print(json.dumps(result, ensure_ascii=False, indent=2))

print("\n현재 score 타입:")
for result in results:
    print(result["source"], type(result["score"]).__name__)


결과 1
{
  "video_id": "kr_tofu_001",
  "query": "두부를 칼로 자르는 장면",
  "start_time": 98.5,
  "end_time": 118.9725,
  "score": 0.9842,
  "source": "hybrid",
  "hybrid_source": "visual",
  "visual_score": 0.9842,
  "subtitle_score": null,
  "language": null
}

결과 2
{
  "video_id": "kr_tofu_001",
  "query": "두부를 칼로 자르는 장면",
  "start_time": 84.5,
  "end_time": 102.7907,
  "score": 0.9608,
  "source": "hybrid",
  "hybrid_source": "visual",
  "visual_score": 0.9608,
  "subtitle_score": null,
  "language": null
}

결과 3
{
  "video_id": "kr_tofu_001",
  "query": "두부를 칼로 자르는 장면",
  "start_time": 114.5,
  "end_time": 135.9028,
  "score": 0.7395,
  "source": "hybrid",
  "hybrid_source": "visual",
  "visual_score": 0.7395,
  "subtitle_score": null,
  "language": null
}

현재 score 타입:
hybrid float
hybrid float
hybrid float


In [25]:
daiso_results = fusion_search.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path="data/features/kr_daiso_001_clip_features.pt",
    subtitles=test_subtitles,
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="showing a travel makeup pouch",
    video_duration=32.07,
    subtitle_json_path="data/processed/kr_daiso_001/subtitles.json",
    top_k=5
)

for i, result in enumerate(daiso_results, 1):
    print(
        f"{i}. source={result['source']}, "
        f"score={result['score']}, "
        f"subtitle_score={result['subtitle_score']}, "
        f"구간={result['start_time']}~{result['end_time']}초"
    )

1. source=hybrid, score=0.8893, subtitle_score=0.0, 구간=14.5~25.0135초
2. source=subtitle, score=None, subtitle_score=0.6598, 구간=2.08~7.28초
3. source=hybrid, score=0.8883, subtitle_score=0.0, 구간=24.5~28.4991초


In [26]:
from pathlib import Path

path = Path("backend/services/search_api.py")
lines = path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, 1):
    if "predict_moments(" in line or '["score"]' in line or "best_score" in line:
        start = max(1, i - 5)
        end = min(len(lines), i + 8)

        print(f"\n--- {start}~{end}줄 ---")
        for j in range(start, end + 1):
            print(f"{j}: {lines[j-1]}")


--- 39~52줄 ---
39:     return _err(503 if quota else 502,
40:                 "YOUTUBE_QUOTA" if quota else "YOUTUBE_ERROR",
41:                 "유튜브 API 호출에 실패했습니다.")
42: 
43: 
44: def predict_moments(video_id: str, feature_path: str, query: str, top_k: int = 5):
45:     if indexer.USE_REAL_AI:
46:         from ai.features.run_moment_detr_inference import predict_moments_hybrid
47:         with indexer._AI_LOCK:
48:             return predict_moments_hybrid(
49:                 video_id=video_id, video_feature_path=feature_path,
50:                 query=query, top_k=top_k,
51:             )
52:     return [

--- 159~172줄 ---
159:                 "youtube_video_id": yid,
160:                 "message": "영상을 분석 중입니다.",
161:             })
162: 
163:         feature = indexer.resolve_path(r["feature_path"])
164:         raw = predict_moments(yid, str(feature), query, top_k)
165:         results = [
166:             {"segment_id": f"{yid}-{i}", "start_time": m["start_time"],
167:       

In [27]:
def convert_fusion_results_for_api(fusion_results):
    converted = []

    for i, result in enumerate(fusion_results, start=1):
        score = result.get("score")

        # Hybrid 점수가 없는 자막 검색 결과는
        # 임의 점수를 만들지 않고 None으로 유지
        if score is not None:
            score = float(score)

        converted.append({
            "segment_id": f"{result['video_id']}-{i}",
            "start_time": result["start_time"],
            "end_time": result["end_time"],
            "score": score,
            "source": result["source"],
            "subtitle_score": result.get("subtitle_score"),
        })

    return converted


api_results = convert_fusion_results_for_api(daiso_results)

# 숫자 점수가 있는 후보만 영상별 최고 점수 계산에 사용
numeric_scores = [
    r["score"]
    for r in api_results
    if r["score"] is not None
]

best_score = max(numeric_scores, default=None)

for result in api_results:
    print(result)

print("\n영상 최고 Hybrid 점수:", best_score)

{'segment_id': 'kr_daiso_001-1', 'start_time': 14.5, 'end_time': 25.0135, 'score': 0.8893, 'source': 'hybrid', 'subtitle_score': 0.0}
{'segment_id': 'kr_daiso_001-2', 'start_time': 2.08, 'end_time': 7.28, 'score': None, 'source': 'subtitle', 'subtitle_score': 0.6598}
{'segment_id': 'kr_daiso_001-3', 'start_time': 24.5, 'end_time': 28.4991, 'score': 0.8883, 'source': 'hybrid', 'subtitle_score': 0.0}

영상 최고 Hybrid 점수: 0.8893


In [28]:
from pathlib import Path

path = Path("backend/services/search_api.py")
lines = path.read_text(encoding="utf-8").splitlines()

for start, end in [(135, 190), (195, 255)]:
    print(f"\n===== {start}~{end}줄 =====")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i}: {lines[i-1]}")


===== 135~190줄 =====
135:         if row is None:
136:             try:
137:                 info = fetch_video_meta(yid)
138:             except requests.HTTPError as e:
139:                 return _youtube_error(e)
140:             if not info:
141:                 return _err(404, "VIDEO_NOT_FOUND", "유튜브에서 영상을 찾을 수 없습니다.")
142:             if info["duration"] and info["duration"] > MAX_DURATION_SEC:
143:                 return _err(422, "VIDEO_TOO_LONG",
144:                             f"{MAX_DURATION_SEC // 60}분 이하 영상만 분석할 수 있습니다.")
145:             title = info["title"]
146:             meta = {"channel_name": info["channel_name"],
147:                     "published_at": info["published_at"],
148:                     "thumbnail_url": info["thumbnail_url"]}
149:         else:
150:             title = row["title"]
151: 
152:         r = indexer.prepare(yid, title, **meta)
153: 
154:         if r["state"] == "STARTED":
155:             background_tasks.add_task(indexer.run_indexin

In [29]:
import subprocess

result = subprocess.run(
    [
        "git", "grep", "-n", "-E",
        "best_score|segment\\.score|moment\\.score|\\.score",
        "origin/feature/frontend",
        "--", "frontend"
    ],
    capture_output=True,
    text=True
)

print(result.stdout[:12000] or "검색된 코드가 없습니다.")
print("종료 코드:", result.returncode)

origin/feature/frontend:frontend/src/components/LibraryView.tsx:112:            관련도 {Math.round(segment.score * 100)}%
origin/feature/frontend:frontend/src/components/SearchResultsList.tsx:145:                        <span className="text-[11px] font-mono font-bold text-blue-700 bg-blue-50 px-2 py-0.5 rounded-md whitespace-nowrap border border-blue-200/70" title={`관련도 점수: ${item.score}`}>
origin/feature/frontend:frontend/src/components/SearchResultsList.tsx:146:                          Score {item.score}
origin/feature/frontend:frontend/src/services/searchApi.ts:28:      results: [...resp.results].sort((a, b) => b.score - a.score)
origin/feature/frontend:frontend/src/services/searchApi.ts:38:          results: [...resp.results].sort((a, b) => b.score - a.score)
origin/feature/frontend:frontend/src/services/searchApi.ts:46:          results: [...resp.results].sort((a, b) => b.score - a.score)
origin/feature/frontend:frontend/src/services/searchApi.ts:54:          results: [...resp.resu

# AI 검색 정확도 개선

In [30]:
import json
from pathlib import Path

tofu_subtitles = json.loads(
    (video_dir / "kr_tofu_001_subtitles.json").read_text(
        encoding="utf-8"
    )
)

tofu_results = fusion_search.predict_moments_fusion(
    video_id="kr_tofu_001",
    feature_path="data/features/kr_tofu_001_clip_features.pt",
    subtitles=tofu_subtitles,
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    video_duration=165.0,
    subtitle_json_path=str(
        video_dir / "kr_tofu_001_subtitles.json"
    ),
    top_k=5
)

for i, r in enumerate(tofu_results, 1):
    print(
        f"{i}. {r['source']} | "
        f"{r['start_time']}~{r['end_time']}초 | "
        f"visual={r['visual_score']} | "
        f"subtitle={r['subtitle_score']}"
    )

1. hybrid | 98.5~118.9725초 | visual=0.9842 | subtitle=0.0
2. subtitle | 140.86~143.58초 | visual=None | subtitle=0.6732
3. hybrid | 84.5~102.7907초 | visual=0.9608 | subtitle=0.0
4. subtitle | 135.06~137.58초 | visual=None | subtitle=0.6575
5. hybrid | 114.5~135.9028초 | visual=0.7395 | subtitle=0.0


In [31]:
queries = {
    "한국어": "두부를 칼로 자르는 장면",
    "영어": "cutting tofu with a knife"
}

for language, query in queries.items():
    candidates = fusion_search.get_semantic_subtitle_candidates(
        query=query,
        subtitles=tofu_subtitles,
        model=fusion_search.get_semantic_model(),
        top_k=5
    )

    print(f"\n===== {language} 검색 결과 =====")

    for i, r in enumerate(candidates, 1):
        print(
            f"{i}. {r['start_time']}~{r['end_time']}초 "
            f"| 유사도={r['score']:.4f}"
        )


===== 한국어 검색 결과 =====
1. 140.86~143.58초 | 유사도=0.6732
2. 135.06~137.58초 | 유사도=0.6575
3. 97.94~101.62초 | 유사도=0.6547
4. 137.58~143.58초 | 유사도=0.6458
5. 16.32~19.02초 | 유사도=0.6450

===== 영어 검색 결과 =====
1. 12.88~21.06초 | 유사도=0.5538
2. 8.7~19.02초 | 유사도=0.5509
3. 8.7~16.32초 | 유사도=0.5087
4. 4.74~16.32초 | 유사도=0.5072
5. 12.88~19.02초 | 유사도=0.4800


In [32]:
ko_candidates = fusion_search.get_semantic_subtitle_candidates(
    query="두부를 칼로 자르는 장면",
    subtitles=tofu_subtitles,
    model=fusion_search.get_semantic_model(),
    top_k=5
)

en_candidates = fusion_search.get_semantic_subtitle_candidates(
    query="cutting tofu with a knife",
    subtitles=tofu_subtitles,
    model=fusion_search.get_semantic_model(),
    top_k=5
)

# 두 언어의 순위를 번갈아 사용
combined = []

for i in range(max(len(ko_candidates), len(en_candidates))):
    if i < len(ko_candidates):
        combined.append(dict(ko_candidates[i], language="ko"))

    if i < len(en_candidates):
        combined.append(dict(en_candidates[i], language="en"))

# 시간 구간 중복 제거
selected = fusion_search.deduplicate_multilingual_candidates(combined)

print("===== 한국어 + 영어 의미 검색 통합 =====")

for i, r in enumerate(selected[:5], 1):
    print(
        f"{i}. {r['language']} | "
        f"{r['start_time']}~{r['end_time']}초 | "
        f"유사도={r['score']:.4f}"
    )

===== 한국어 + 영어 의미 검색 통합 =====
1. ko | 140.86~143.58초 | 유사도=0.6732
2. en | 12.88~21.06초 | 유사도=0.5538
3. ko | 135.06~137.58초 | 유사도=0.6575
4. ko | 97.94~101.62초 | 유사도=0.6547
5. en | 4.74~16.32초 | 유사도=0.5072


In [33]:
import json

tofu_subtitles = json.loads(
    (video_dir / "kr_tofu_001_subtitles.json").read_text(encoding="utf-8")
)

kitchen_subtitles = json.loads(
    (video_dir / "kr_kitchen_001_subtitles.json").read_text(encoding="utf-8")
)

daiso_subtitles = json.loads(
    (video_dir / "kr_daiso_001_subtitles.json").read_text(encoding="utf-8")
)

print("두부 자막:", len(tofu_subtitles), "개")
print("주방 자막:", len(kitchen_subtitles), "개")
print("다이소 자막:", len(daiso_subtitles), "개")

두부 자막: 57 개
주방 자막: 17 개
다이소 자막: 19 개


In [34]:
import importlib
from ai.features import fusion_search

importlib.reload(fusion_search)

print("Fusion 검색 코드 로딩 완료!")
print(
    "한국어·영어 의미 검색 함수:",
    hasattr(fusion_search, "get_multilingual_subtitle_candidates")
)

Fusion 검색 코드 로딩 완료!
한국어·영어 의미 검색 함수: True


In [35]:
test_cases = [
    (
        "주방",
        kitchen_subtitles,
        "가위로 소스를 짜는 장면",
        "squeezing sauce with scissors"
    ),
    (
        "다이소",
        daiso_subtitles,
        "여행용 화장품 파우치를 보여주는 장면",
        "showing a travel makeup pouch"
    )
]

for name, subs, query_ko, query_en in test_cases:
    print(f"\n===== {name} 영상 =====")

    for language, query in [("ko", query_ko), ("en", query_en)]:
        candidates = fusion_search.get_semantic_subtitle_candidates(
            query=query,
            subtitles=subs,
            model=fusion_search.get_semantic_model(),
            top_k=3
        )

        print(f"\n{language} 검색:")

        for i, r in enumerate(candidates, 1):
            print(
                f"{i}. {r['start_time']}~{r['end_time']}초 "
                f"| 유사도={r['score']:.4f}"
            )


===== 주방 영상 =====

ko 검색:
1. 5.96~8.08초 | 유사도=0.8635
2. 33.52~34.8초 | 유사도=0.8609
3. 18.2~19.76초 | 유사도=0.8493

en 검색:
1. 21.52~26.56초 | 유사도=0.4822
2. 15.96~21.52초 | 유사도=0.4100
3. 23.32~26.56초 | 유사도=0.3991

===== 다이소 영상 =====

ko 검색:
1. 2.08~7.28초 | 유사도=0.6598
2. 3.08~7.28초 | 유사도=0.6566
3. 3.08~8.4초 | 유사도=0.6424

en 검색:
1. 3.08~7.28초 | 유사도=0.6301
2. 2.08~7.28초 | 유사도=0.6273
3. 3.08~8.4초 | 유사도=0.5871


In [36]:
combined = fusion_search.get_multilingual_subtitle_candidates(
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    subtitles=tofu_subtitles,
    model=fusion_search.get_semantic_model(),
    top_k=3
)

for language in ("ko", "en"):
    print(f"\n===== {language} 후보 =====")

    candidates = [
        r for r in combined
        if r["language"] == language
    ]

    for i, r in enumerate(candidates, 1):
        print(
            f"{i}. {r['start_time']}~{r['end_time']}초 "
            f"| 유사도={r['score']:.4f}"
        )

assert any(r["language"] == "ko" for r in combined)
assert any(r["language"] == "en" for r in combined)

print("\n한국어·영어 후보 생성 테스트 통과!")


===== ko 후보 =====
1. 140.86~143.58초 | 유사도=0.6732
2. 135.06~137.58초 | 유사도=0.6575
3. 97.94~101.62초 | 유사도=0.6547

===== en 후보 =====
1. 12.88~21.06초 | 유사도=0.5538
2. 8.7~19.02초 | 유사도=0.5509
3. 8.7~16.32초 | 유사도=0.5087

한국어·영어 후보 생성 테스트 통과!


In [37]:
from pathlib import Path

path = Path("ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

old = '''    language, subtitle_query = select_subtitle_query(
        subtitles=subtitles,
        korean_query=korean_query,
        english_query=english_query
    )

    subtitle = []
    if subtitle_query is not None:
        subtitle = get_semantic_subtitle_candidates(
            query=subtitle_query,
            subtitles=subtitles,
            model=get_semantic_model(),
            top_k=top_k
        )
        subtitle = [
            dict(r, language=language, query_used=subtitle_query)
            for r in subtitle
        ]
'''

new = '''    # 다국어 의미 검색: 한국어·영어 검색어 모두 사용
    subtitle = []
    if subtitles:
        subtitle = get_multilingual_subtitle_candidates(
            korean_query=korean_query,
            english_query=english_query,
            subtitles=subtitles,
            model=get_semantic_model(),
            top_k=top_k
        )
'''

assert code.count(old) == 1, "수정할 코드 위치를 확인해 주세요."

code = code.replace(old, new, 1)

compile(code, str(path), "exec")
path.write_text(code, encoding="utf-8")

print("Fusion 한국어·영어 의미 검색 연결 완료!")

Fusion 한국어·영어 의미 검색 연결 완료!


In [38]:
import importlib
from ai.features import fusion_search

# 수정된 코드 다시 불러오기
importlib.reload(fusion_search)

tofu_results = fusion_search.predict_moments_fusion(
    video_id="kr_tofu_001",
    feature_path="data/features/kr_tofu_001_clip_features.pt",
    subtitles=tofu_subtitles,
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    video_duration=165.0,
    subtitle_json_path=str(video_dir / "kr_tofu_001_subtitles.json"),
    top_k=5
)

print("===== 두부 영상 Fusion 검색 결과 =====")

for i, r in enumerate(tofu_results, 1):
    print(
        f"{i}. {r['source']} ({r['language']}) | "
        f"{r['start_time']}~{r['end_time']}초 | "
        f"visual={r['visual_score']} | "
        f"subtitle={r['subtitle_score']}"
    )

===== 두부 영상 Fusion 검색 결과 =====
1. hybrid (None) | 98.5~118.9725초 | visual=0.9842 | subtitle=0.0
2. subtitle (ko) | 140.86~143.58초 | visual=None | subtitle=0.6732
3. subtitle (en) | 12.88~21.06초 | visual=None | subtitle=0.5538
4. hybrid (None) | 84.5~102.7907초 | visual=0.9608 | subtitle=0.0
5. subtitle (ko) | 135.06~137.58초 | visual=None | subtitle=0.6575


In [39]:
test_cases = [
    (
        "주방",
        "kr_kitchen_001",
        kitchen_subtitles,
        "가위로 소스를 짜는 장면",
        "squeezing sauce with scissors",
        37.13
    ),
    (
        "다이소",
        "kr_daiso_001",
        daiso_subtitles,
        "여행용 화장품 파우치를 보여주는 장면",
        "showing a travel makeup pouch",
        32.07
    )
]

for name, video_id, subs, query_ko, query_en, duration in test_cases:
    results = fusion_search.predict_moments_fusion(
        video_id=video_id,
        feature_path=f"data/features/{video_id}_clip_features.pt",
        subtitles=subs,
        korean_query=query_ko,
        english_query=query_en,
        video_duration=duration,
        subtitle_json_path=str(video_dir / f"{video_id}_subtitles.json"),
        top_k=5
    )

    print(f"\n===== {name} 영상 Fusion 검색 결과 =====")

    for i, r in enumerate(results, 1):
        print(
            f"{i}. {r['source']} ({r['language']}) | "
            f"{r['start_time']}~{r['end_time']}초 | "
            f"visual={r['visual_score']} | "
            f"subtitle={r['subtitle_score']}"
        )


===== 주방 영상 Fusion 검색 결과 =====
1. hybrid (None) | 2.5~10.2804초 | visual=0.8601 | subtitle=0.0
2. subtitle (en) | 21.52~26.56초 | visual=None | subtitle=0.4822
3. subtitle (ko) | 33.52~34.8초 | visual=None | subtitle=0.8609
4. subtitle (en) | 15.96~21.52초 | visual=None | subtitle=0.41
5. hybrid (None) | 10.5~20.5314초 | visual=0.0139 | subtitle=0.0

===== 다이소 영상 Fusion 검색 결과 =====
1. hybrid (None) | 14.5~25.0135초 | visual=0.8893 | subtitle=0.0
2. subtitle (ko) | 2.08~7.28초 | visual=None | subtitle=0.6598
3. hybrid (None) | 24.5~28.4991초 | visual=0.8883 | subtitle=0.0


In [40]:
kitchen_hybrid = {
    "start_time": 2.5,
    "end_time": 10.2804
}

kitchen_subtitle = {
    "start_time": 5.96,
    "end_time": 8.08
}

iou = fusion_search.temporal_iou(
    kitchen_hybrid,
    kitchen_subtitle
)

start_gap = abs(
    kitchen_hybrid["start_time"]
    - kitchen_subtitle["start_time"]
)

print("Hybrid 구간:", kitchen_hybrid)
print("자막 구간:", kitchen_subtitle)
print("시간 구간 IoU:", round(iou, 4))
print("시작 시점 차이:", round(start_gap, 4), "초")

print(
    "현재 중복 제거 조건 충족:",
    iou >= 0.5 or start_gap < 5.0
)

Hybrid 구간: {'start_time': 2.5, 'end_time': 10.2804}
자막 구간: {'start_time': 5.96, 'end_time': 8.08}
시간 구간 IoU: 0.2725
시작 시점 차이: 3.46 초
현재 중복 제거 조건 충족: True


In [41]:
print("===== 주방 영상 중복 판단 비교 =====")

print(
    "기존 방식:",
    iou >= 0.5 or start_gap < 5.0
)

print(
    "IoU만 사용하는 방식:",
    iou >= 0.5
)

print(
    "IoU 방식에서 두 후보 모두 유지:",
    iou < 0.5
)

===== 주방 영상 중복 판단 비교 =====
기존 방식: True
IoU만 사용하는 방식: False
IoU 방식에서 두 후보 모두 유지: True


In [42]:
from ai.features import fusion_search

def test_iou_only_fusion(name, visual_results, ko_results, en_results, top_k=5):
    ordered = []

    for i in range(max(len(visual_results), len(ko_results), len(en_results))):
        for group in (visual_results, ko_results, en_results):
            if i < len(group):
                ordered.append(group[i])

    selected = []

    for candidate in ordered:
        duplicate = any(
            fusion_search.temporal_iou(candidate, kept) >= 0.5
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

        if len(selected) >= top_k:
            break

    print(f"\n===== {name}: IoU 기준만 적용 =====")
    for i, r in enumerate(selected, 1):
        print(
            f"{i}. {r['source']} ({r.get('language')}) | "
            f"{r['start_time']}~{r['end_time']}초"
        )


# 먼저 주방 영상 후보부터 준비
kitchen_visual = [
    dict(r, source="hybrid", hybrid_source=r.get("source"))
    for r in fusion_search.predict_moments_hybrid(
        video_id="kr_kitchen_001",
        video_feature_path="data/features/kr_kitchen_001_clip_features.pt",
        query="squeezing sauce with scissors",
        subtitle_json_path=str(video_dir / "kr_kitchen_001_subtitles.json"),
        video_duration=37.13,
        top_k=5
    )
]

kitchen_all_subtitle = fusion_search.get_multilingual_subtitle_candidates(
    korean_query="가위로 소스를 짜는 장면",
    english_query="squeezing sauce with scissors",
    subtitles=kitchen_subtitles,
    model=fusion_search.get_semantic_model(),
    top_k=5
)

kitchen_ko = [r for r in kitchen_all_subtitle if r["language"] == "ko"]
kitchen_en = [r for r in kitchen_all_subtitle if r["language"] == "en"]

test_iou_only_fusion(
    "주방",
    kitchen_visual,
    kitchen_ko,
    kitchen_en
)


===== 주방: IoU 기준만 적용 =====
1. hybrid (None) | 2.5~10.2804초
2. subtitle (ko) | 5.96~8.08초
3. subtitle (en) | 21.52~26.56초
4. hybrid (None) | 20.5~31.7216초
5. subtitle (ko) | 33.52~34.8초


In [43]:
remaining_cases = [
    (
        "두부", "kr_tofu_001", tofu_subtitles,
        "두부를 칼로 자르는 장면",
        "cutting tofu with a knife", 165.0
    ),
    (
        "다이소", "kr_daiso_001", daiso_subtitles,
        "여행용 화장품 파우치를 보여주는 장면",
        "showing a travel makeup pouch", 32.07
    )
]

for name, video_id, subs, query_ko, query_en, duration in remaining_cases:
    visual = [
        dict(r, source="hybrid", hybrid_source=r.get("source"))
        for r in fusion_search.predict_moments_hybrid(
            video_id=video_id,
            video_feature_path=f"data/features/{video_id}_clip_features.pt",
            query=query_en,
            subtitle_json_path=str(video_dir / f"{video_id}_subtitles.json"),
            video_duration=duration,
            top_k=5
        )
    ]

    subtitle = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=query_ko,
        english_query=query_en,
        subtitles=subs,
        model=fusion_search.get_semantic_model(),
        top_k=5
    )

    ko = [r for r in subtitle if r["language"] == "ko"]
    en = [r for r in subtitle if r["language"] == "en"]

    test_iou_only_fusion(name, visual, ko, en)


===== 두부: IoU 기준만 적용 =====
1. hybrid (None) | 98.5~118.9725초
2. subtitle (ko) | 140.86~143.58초
3. subtitle (en) | 12.88~21.06초
4. hybrid (None) | 84.5~102.7907초
5. subtitle (ko) | 135.06~137.58초

===== 다이소: IoU 기준만 적용 =====
1. hybrid (None) | 14.5~25.0135초
2. subtitle (ko) | 2.08~7.28초
3. hybrid (None) | 24.5~28.4991초
4. subtitle (en) | 4.68~10.0초


In [44]:
from pathlib import Path

path = Path("ai/features/fusion_search.py")
code = path.read_text(encoding="utf-8")

old = '''        duplicate = any(
            temporal_iou(candidate, kept) >= 0.5
            or abs(candidate["start_time"] - kept["start_time"]) < 5.0
            for kept in selected
        )
'''

new = '''        # 실험: IoU 기준으로만 중복 후보 제거
        duplicate = any(
            temporal_iou(candidate, kept) >= 0.5
            for kept in selected
        )
'''

assert code.count(old) == 1, "수정할 코드 위치를 확인해 주세요."

code = code.replace(old, new, 1)

compile(code, str(path), "exec")
path.write_text(code, encoding="utf-8")

print("Fusion 중복 제거 기준을 IoU 0.5로 변경 완료!")

Fusion 중복 제거 기준을 IoU 0.5로 변경 완료!


In [45]:
import importlib
from ai.features import fusion_search

# 수정한 코드 다시 불러오기
importlib.reload(fusion_search)

# 주방 영상으로 실제 검색
kitchen_results = fusion_search.predict_moments_fusion(
    video_id="kr_kitchen_001",
    feature_path="data/features/kr_kitchen_001_clip_features.pt",
    subtitles=kitchen_subtitles,
    korean_query="가위로 소스를 짜는 장면",
    english_query="squeezing sauce with scissors",
    video_duration=37.13,
    subtitle_json_path=str(video_dir / "kr_kitchen_001_subtitles.json"),
    top_k=5
)

print("===== 수정 후 주방 영상 Fusion 결과 =====")

for i, r in enumerate(kitchen_results, 1):
    print(
        f"{i}. {r['source']} ({r['language']}) | "
        f"{r['start_time']}~{r['end_time']}초"
    )

===== 수정 후 주방 영상 Fusion 결과 =====
1. hybrid (None) | 2.5~10.2804초
2. subtitle (ko) | 5.96~8.08초
3. subtitle (en) | 21.52~26.56초
4. hybrid (None) | 20.5~31.7216초
5. subtitle (ko) | 33.52~34.8초


In [46]:
test_results = {
    "두부": tofu_results,
    "주방": kitchen_results,
    "다이소": daiso_results
}

for name, results in test_results.items():
    print(f"\n===== {name} 영상 =====")

    hybrid = [r for r in results if r["source"] == "hybrid"]
    subtitle = [r for r in results if r["source"] == "subtitle"]

    for h in hybrid:
        for s in subtitle:
            iou = fusion_search.temporal_iou(h, s)

            if iou > 0:
                print(
                    f"Hybrid {h['start_time']}~{h['end_time']}초 "
                    f"↔ 자막 {s['start_time']}~{s['end_time']}초 "
                    f"| IoU={iou:.4f}"
                )


===== 두부 영상 =====

===== 주방 영상 =====
Hybrid 2.5~10.2804초 ↔ 자막 5.96~8.08초 | IoU=0.2725
Hybrid 20.5~31.7216초 ↔ 자막 21.52~26.56초 | IoU=0.4491

===== 다이소 영상 =====


In [47]:
ground_truth = {
    "두부": {
        "start_time": 12.0,
        "end_time": 23.0
    },
    "주방": {
        "start_time": 5.96,
        "end_time": 11.32
    },
    "다이소": {
        "start_time": 3.08,
        "end_time": 8.4
    }
}

for name, gt in ground_truth.items():
    print(
        f"{name}: "
        f"{gt['start_time']}~{gt['end_time']}초"
    )

두부: 12.0~23.0초
주방: 5.96~11.32초
다이소: 3.08~8.4초


In [48]:
test_cases = [
    ("두부", "kr_tofu_001", tofu_subtitles,
     "두부를 칼로 자르는 장면",
     "cutting tofu with a knife", 165.0),

    ("주방", "kr_kitchen_001", kitchen_subtitles,
     "가위로 소스를 짜는 장면",
     "squeezing sauce with scissors", 37.13),

    ("다이소", "kr_daiso_001", daiso_subtitles,
     "여행용 화장품 파우치를 보여주는 장면",
     "showing a travel makeup pouch", 32.07)
]

evaluation_results = {}

for name, video_id, subs, query_ko, query_en, duration in test_cases:

    predictions = fusion_search.predict_moments_fusion(
        video_id=video_id,
        feature_path=f"data/features/{video_id}_clip_features.pt",
        subtitles=subs,
        korean_query=query_ko,
        english_query=query_en,
        video_duration=duration,
        subtitle_json_path=str(
            video_dir / f"{video_id}_subtitles.json"
        ),
        top_k=5
    )

    gt = ground_truth[name]

    evaluated = []

    for rank, pred in enumerate(predictions, 1):
        iou = fusion_search.temporal_iou(pred, gt)

        evaluated.append({
            "rank": rank,
            "source": pred["source"],
            "language": pred["language"],
            "start_time": pred["start_time"],
            "end_time": pred["end_time"],
            "iou": iou
        })

    evaluation_results[name] = evaluated

    print(f"\n===== {name} 영상 =====")

    for r in evaluated:
        print(
            f"{r['rank']}위 | "
            f"{r['source']} ({r['language']}) | "
            f"{r['start_time']}~{r['end_time']}초 | "
            f"IoU={r['iou']:.4f}"
        )

    hits = [r for r in evaluated if r["iou"] >= 0.5]

    if hits:
        print("IoU 0.5 이상 첫 순위:", hits[0]["rank"])
    else:
        print("상위 5개에서 IoU 0.5 이상 후보 없음")


===== 두부 영상 =====
1위 | hybrid (None) | 98.5~118.9725초 | IoU=0.0000
2위 | subtitle (ko) | 140.86~143.58초 | IoU=0.0000
3위 | subtitle (en) | 12.88~21.06초 | IoU=0.7436
4위 | hybrid (None) | 84.5~102.7907초 | IoU=0.0000
5위 | subtitle (ko) | 135.06~137.58초 | IoU=0.0000
IoU 0.5 이상 첫 순위: 3

===== 주방 영상 =====
1위 | hybrid (None) | 2.5~10.2804초 | IoU=0.4898
2위 | subtitle (ko) | 5.96~8.08초 | IoU=0.3955
3위 | subtitle (en) | 21.52~26.56초 | IoU=0.0000
4위 | hybrid (None) | 20.5~31.7216초 | IoU=0.0000
5위 | subtitle (ko) | 33.52~34.8초 | IoU=0.0000
상위 5개에서 IoU 0.5 이상 후보 없음

===== 다이소 영상 =====
1위 | hybrid (None) | 14.5~25.0135초 | IoU=0.0000
2위 | subtitle (ko) | 2.08~7.28초 | IoU=0.6646
3위 | hybrid (None) | 24.5~28.4991초 | IoU=0.0000
4위 | subtitle (en) | 4.68~10.0초 | IoU=0.5376
IoU 0.5 이상 첫 순위: 2


# Hybrid와 자막 의미 검색의 성능 분리 평가

In [49]:
print("===== 검색 방식별 최고 IoU 비교 =====")

for name, evaluated in evaluation_results.items():
    hybrid_ious = [
        r["iou"] for r in evaluated
        if r["source"] == "hybrid"
    ]

    subtitle_ious = [
        r["iou"] for r in evaluated
        if r["source"] == "subtitle"
    ]

    best_hybrid = max(hybrid_ious, default=0.0)
    best_subtitle = max(subtitle_ious, default=0.0)

    print(f"\n{name}")
    print(f"Hybrid 최고 IoU: {best_hybrid:.4f}")
    print(f"자막 의미 검색 최고 IoU: {best_subtitle:.4f}")

    if best_hybrid > best_subtitle:
        print("→ Hybrid가 더 정확")
    elif best_subtitle > best_hybrid:
        print("→ 자막 의미 검색이 더 정확")
    else:
        print("→ 동일")

===== 검색 방식별 최고 IoU 비교 =====

두부
Hybrid 최고 IoU: 0.0000
자막 의미 검색 최고 IoU: 0.7436
→ 자막 의미 검색이 더 정확

주방
Hybrid 최고 IoU: 0.4898
자막 의미 검색 최고 IoU: 0.3955
→ Hybrid가 더 정확

다이소
Hybrid 최고 IoU: 0.0000
자막 의미 검색 최고 IoU: 0.6646
→ 자막 의미 검색이 더 정확


In [50]:
print("===== 원본 Hybrid vs 자막 검색 1위 비교 =====")

for name, video_id, subs, query_ko, query_en, duration in test_cases:

    gt = ground_truth[name]

    hybrid = fusion_search.predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=f"data/features/{video_id}_clip_features.pt",
        query=query_en,
        subtitle_json_path=str(video_dir / f"{video_id}_subtitles.json"),
        video_duration=duration,
        top_k=1
    )

    subtitle = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=query_ko,
        english_query=query_en,
        subtitles=subs,
        model=fusion_search.get_semantic_model(),
        top_k=1
    )

    print(f"\n{name}")

    if hybrid:
        h = hybrid[0]
        print(
            f"Hybrid 1위: {h['start_time']}~{h['end_time']}초 "
            f"| IoU={fusion_search.temporal_iou(h, gt):.4f}"
        )

    for s in subtitle:
        print(
            f"자막 {s['language']} 1위: "
            f"{s['start_time']}~{s['end_time']}초 "
            f"| IoU={fusion_search.temporal_iou(s, gt):.4f}"
        )

===== 원본 Hybrid vs 자막 검색 1위 비교 =====

두부
Hybrid 1위: 98.5~118.9725초 | IoU=0.0000
자막 ko 1위: 140.86~143.58초 | IoU=0.0000
자막 en 1위: 12.88~21.06초 | IoU=0.7436

주방
Hybrid 1위: 2.5~10.2804초 | IoU=0.4898
자막 ko 1위: 5.96~8.08초 | IoU=0.3955
자막 en 1위: 21.52~26.56초 | IoU=0.0000

다이소
Hybrid 1위: 14.5~25.0135초 | IoU=0.0000
자막 ko 1위: 2.08~7.28초 | IoU=0.6646
자막 en 1위: 3.08~7.28초 | IoU=0.7895


In [51]:
from pathlib import Path

files = [
    Path("ai/features/fusion_search.py"),
    Path("ai/features/run_moment_detr_inference.py")
]

keywords = [
    "cosine_similarity",
    "clip_similarity",
    "encode_text",
    "text_features",
    "visual_score"
]

for path in files:
    print(f"\n===== {path} =====")

    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines, 1):
        if any(keyword in line for keyword in keywords):
            print(f"{i}: {line.strip()}")


===== ai/features/fusion_search.py =====
176: "visual_score": candidate.get("score") if is_hybrid else None,

===== ai/features/run_moment_detr_inference.py =====
202: query_feats = feature_extractor.encode_text(
456: feature_extractor.encode_text(
577: from sklearn.metrics.pairwise import cosine_similarity
688: scores = cosine_similarity(
718: output = clip_model.encode_text(
1156: scores = cosine_similarity(


In [52]:
from pathlib import Path

path = Path("ai/features/run_moment_detr_inference.py")
lines = path.read_text(encoding="utf-8").splitlines()

for start, end in [(665, 735), (1135, 1170)]:
    print(f"\n===== {start}~{end}줄 =====")

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i}: {lines[i-1]}")


===== 665~735줄 =====
665: 
666:     corpus = [
667:         x["text"]
668:         for x in windows
669:     ]
670: 
671:     vectorizer = TfidfVectorizer(
672:         lowercase=True,
673:         analyzer="char_wb",
674:         ngram_range=(3, 5),
675:         min_df=1
676:     )
677: 
678:     subtitle_matrix = (
679:         vectorizer.fit_transform(
680:             corpus
681:         )
682:     )
683: 
684:     query_vector = vectorizer.transform(
685:         [query]
686:     )
687: 
688:     scores = cosine_similarity(
689:         query_vector,
690:         subtitle_matrix
691:     )[0]
692: 
693:     best_idx = int(
694:         scores.argmax()
695:     )
696: 
697:     best = windows[best_idx]
698: 
699:     return {
700:         "score": float(
701:             scores[best_idx]
702:         ),
703:         "start": best["start"],
704:         "end": best["end"],
705:         "text": best["text"]
706:     }
707: 
708: 
709: @torch.no_grad()
710: def _encode_refine_query(q

In [53]:
from pathlib import Path

path = Path("ai/features/run_moment_detr_inference.py")
lines = path.read_text(encoding="utf-8").splitlines()

for start, end in [(709, 815), (815, 900)]:
    print(f"\n===== {start}~{end}줄 =====")

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i}: {lines[i-1]}")


===== 709~815줄 =====
709: @torch.no_grad()
710: def _encode_refine_query(query):
711: 
712:     clip_model = get_refine_clip_model()
713: 
714:     tokens = clip.tokenize(
715:         [query]
716:     ).to(DEVICE)
717: 
718:     output = clip_model.encode_text(
719:         tokens
720:     )
721: 
722:     query_feat = output[
723:         "pooler_output"
724:     ].float()
725: 
726:     query_feat = F.normalize(
727:         query_feat,
728:         dim=-1,
729:         eps=1e-6
730:     )
731: 
732:     return query_feat[0]
733: 
734: 
735: def _smooth_scores(scores):
736:     """
737:     자기 자신 + 앞뒤 1개 bin의 평균.
738:     """
739: 
740:     smoothed = torch.zeros_like(
741:         scores
742:     )
743: 
744:     for i in range(len(scores)):
745: 
746:         left = max(
747:             0,
748:             i - 1
749:         )
750: 
751:         right = min(
752:             len(scores),
753:             i + 2
754:         )
755: 
756:         smoothed[i] = scores[
757:         

In [54]:
import torch
import torch.nn.functional as F
from ai.features import run_moment_detr_inference as inference

video_feats = torch.load(
    "data/features/kr_tofu_001_clip_features.pt",
    map_location="cpu"
).float()

query_feat = inference._encode_refine_query(
    "cutting tofu with a knife"
).detach().cpu().float()

print("영상 특징 크기:", tuple(video_feats.shape))
print("검색어 특징 크기:", tuple(query_feat.shape))

print(
    "영상 특징 평균 길이:",
    round(F.normalize(video_feats, dim=-1).norm(dim=-1).mean().item(), 4)
)

print("검색어 특징 길이:", round(query_feat.norm().item(), 4))

assert video_feats.shape[-1] == query_feat.shape[-1]

print("CLIP 영상·검색어 특징 차원 일치!")

영상 특징 크기: (83, 512)
검색어 특징 크기: (512,)
영상 특징 평균 길이: 1.0
검색어 특징 길이: 1.0
CLIP 영상·검색어 특징 차원 일치!


In [55]:
import torch
import torch.nn.functional as F

video_feats = F.normalize(video_feats, dim=-1)
query_feat = F.normalize(query_feat, dim=-1)

def get_clip_similarity(start_time, end_time):
    # 2초 간격 영상 특징에서 해당 구간의 인덱스 계산
    start_idx = max(0, int(start_time // 2))
    end_idx = min(
        len(video_feats),
        int(end_time // 2) + 1
    )

    if end_idx <= start_idx:
        return None

    # 구간 안의 프레임별 유사도
    scores = video_feats[start_idx:end_idx] @ query_feat

    return {
        "mean": round(scores.mean().item(), 4),
        "max": round(scores.max().item(), 4)
    }


test_spans = [
    ("Hybrid 1위 (잘못된 구간)", 98.5, 118.9725),
    ("한국어 자막 1위 (잘못된 구간)", 140.86, 143.58),
    ("영어 자막 1위 (실제 두부 자르기)", 12.88, 21.06)
]

for name, start, end in test_spans:
    print(name, get_clip_similarity(start, end))

Hybrid 1위 (잘못된 구간) {'mean': 0.279, 'max': 0.2935}
한국어 자막 1위 (잘못된 구간) {'mean': 0.2968, 'max': 0.3157}
영어 자막 1위 (실제 두부 자르기) {'mean': 0.3113, 'max': 0.3369}


In [56]:
import torch
import torch.nn.functional as F
from ai.features import run_moment_detr_inference as inference

for name, video_id, subs, query_ko, query_en, duration in test_cases:
    feats = torch.load(
        f"data/features/{video_id}_clip_features.pt",
        map_location="cpu"
    ).float()

    feats = F.normalize(feats, dim=-1)

    text_feat = inference._encode_refine_query(
        query_en
    ).detach().cpu().float()

    text_feat = F.normalize(text_feat, dim=-1)

    ranked = []

    for r in evaluation_results[name]:
        start_idx = max(0, int(r["start_time"] // 2))
        end_idx = min(len(feats), int(r["end_time"] // 2) + 1)

        if end_idx <= start_idx:
            continue

        clip_score = (
            feats[start_idx:end_idx] @ text_feat
        ).mean().item()

        ranked.append({
            **r,
            "clip_score": clip_score
        })

    ranked.sort(key=lambda x: x["clip_score"], reverse=True)

    print(f"\n===== {name} CLIP 재정렬 결과 =====")

    for i, r in enumerate(ranked, 1):
        print(
            f"{i}위 | {r['source']} ({r['language']}) | "
            f"{r['start_time']}~{r['end_time']}초 | "
            f"CLIP={r['clip_score']:.4f} | "
            f"정답 IoU={r['iou']:.4f}"
        )


===== 두부 CLIP 재정렬 결과 =====
1위 | subtitle (en) | 12.88~21.06초 | CLIP=0.3113 | 정답 IoU=0.7436
2위 | subtitle (ko) | 140.86~143.58초 | CLIP=0.2968 | 정답 IoU=0.0000
3위 | subtitle (ko) | 135.06~137.58초 | CLIP=0.2961 | 정답 IoU=0.0000
4위 | hybrid (None) | 84.5~102.7907초 | CLIP=0.2933 | 정답 IoU=0.0000
5위 | hybrid (None) | 98.5~118.9725초 | CLIP=0.2790 | 정답 IoU=0.0000

===== 주방 CLIP 재정렬 결과 =====
1위 | subtitle (ko) | 33.52~34.8초 | CLIP=0.3026 | 정답 IoU=0.0000
2위 | subtitle (ko) | 5.96~8.08초 | CLIP=0.2961 | 정답 IoU=0.3955
3위 | hybrid (None) | 2.5~10.2804초 | CLIP=0.2955 | 정답 IoU=0.4898
4위 | hybrid (None) | 20.5~31.7216초 | CLIP=0.2797 | 정답 IoU=0.0000
5위 | subtitle (en) | 21.52~26.56초 | CLIP=0.2784 | 정답 IoU=0.0000

===== 다이소 CLIP 재정렬 결과 =====
1위 | subtitle (en) | 4.68~10.0초 | CLIP=0.2667 | 정답 IoU=0.5376
2위 | subtitle (ko) | 2.08~7.28초 | CLIP=0.2626 | 정답 IoU=0.6646
3위 | hybrid (None) | 14.5~25.0135초 | CLIP=0.2437 | 정답 IoU=0.0000
4위 | hybrid (None) | 24.5~28.4991초 | CLIP=0.2254 | 정답 IoU=0.0000


In [57]:
def rrf_score(ranks, k=60):
    return sum(
        1.0 / (k + rank)
        for rank in ranks
        if rank is not None
    )

examples = {
    "두 검색에서 모두 1위": [1, 1],
    "한 검색에서만 1위": [1, None],
    "두 검색에서 모두 3위": [3, 3],
    "각각 1위와 5위": [1, 5]
}

print("===== RRF 계산 테스트 =====")

for name, ranks in examples.items():
    print(f"{name}: {rrf_score(ranks):.6f}")

===== RRF 계산 테스트 =====
두 검색에서 모두 1위: 0.032787
한 검색에서만 1위: 0.016393
두 검색에서 모두 3위: 0.031746
각각 1위와 5위: 0.031778


In [58]:
def compare_temporal_overlap(a, b):
    intersection = max(
        0.0,
        min(a["end_time"], b["end_time"])
        - max(a["start_time"], b["start_time"])
    )

    duration_a = a["end_time"] - a["start_time"]
    duration_b = b["end_time"] - b["start_time"]

    union = duration_a + duration_b - intersection

    iou = intersection / union if union > 0 else 0.0

    shorter = min(duration_a, duration_b)
    overlap_shorter = intersection / shorter if shorter > 0 else 0.0

    return iou, overlap_shorter


hybrid_span = {
    "start_time": 2.5,
    "end_time": 10.2804
}

subtitle_span = {
    "start_time": 5.96,
    "end_time": 8.08
}

iou, overlap_shorter = compare_temporal_overlap(
    hybrid_span,
    subtitle_span
)

print("시간 구간 IoU:", round(iou, 4))
print("짧은 구간 기준 겹침 비율:", round(overlap_shorter, 4))

시간 구간 IoU: 0.2725
짧은 구간 기준 겹침 비율: 1.0


In [59]:
for name in ["두부", "주방", "다이소"]:
    results = evaluation_results[name]

    hybrid = [r for r in results if r["source"] == "hybrid"]
    subtitle = [r for r in results if r["source"] == "subtitle"]

    print(f"\n===== {name} =====")

    found = False

    for h in hybrid:
        for s in subtitle:
            iou, overlap = compare_temporal_overlap(h, s)

            if overlap >= 0.8:
                found = True
                print(
                    f"Hybrid {h['start_time']}~{h['end_time']}초"
                    f" ↔ 자막 {s['start_time']}~{s['end_time']}초"
                    f" | IoU={iou:.4f}"
                    f" | 짧은 구간 겹침={overlap:.4f}"
                )

    if not found:
        print("짧은 구간 기준 80% 이상 겹치는 후보 없음")


===== 두부 =====
짧은 구간 기준 80% 이상 겹치는 후보 없음

===== 주방 =====
Hybrid 2.5~10.2804초 ↔ 자막 5.96~8.08초 | IoU=0.2725 | 짧은 구간 겹침=1.0000
Hybrid 20.5~31.7216초 ↔ 자막 21.52~26.56초 | IoU=0.4491 | 짧은 구간 겹침=1.0000

===== 다이소 =====
짧은 구간 기준 80% 이상 겹치는 후보 없음


In [60]:
def experiment_overlap_rrf(results, threshold=0.8, k=60):
    ranked = []

    for candidate in results:
        supporting_ranks = []

        for other in results:
            if candidate["source"] == other["source"]:
                continue

            _, overlap = compare_temporal_overlap(candidate, other)

            if overlap >= threshold:
                supporting_ranks.append(other["rank"])

        score = rrf_score(
            [candidate["rank"]] + supporting_ranks,
            k=k
        )

        ranked.append({
            **candidate,
            "rrf_score": score,
            "supporting_ranks": supporting_ranks
        })

    return sorted(
        ranked,
        key=lambda r: r["rrf_score"],
        reverse=True
    )


for name in ["두부", "주방", "다이소"]:
    ranked = experiment_overlap_rrf(evaluation_results[name])

    print(f"\n===== {name}: RRF 재정렬 =====")

    for i, r in enumerate(ranked, 1):
        print(
            f"{i}위 | {r['source']} ({r['language']}) | "
            f"{r['start_time']}~{r['end_time']}초 | "
            f"RRF={r['rrf_score']:.6f} | "
            f"정답 IoU={r['iou']:.4f}"
        )


===== 두부: RRF 재정렬 =====
1위 | hybrid (None) | 98.5~118.9725초 | RRF=0.016393 | 정답 IoU=0.0000
2위 | subtitle (ko) | 140.86~143.58초 | RRF=0.016129 | 정답 IoU=0.0000
3위 | subtitle (en) | 12.88~21.06초 | RRF=0.015873 | 정답 IoU=0.7436
4위 | hybrid (None) | 84.5~102.7907초 | RRF=0.015625 | 정답 IoU=0.0000
5위 | subtitle (ko) | 135.06~137.58초 | RRF=0.015385 | 정답 IoU=0.0000

===== 주방: RRF 재정렬 =====
1위 | hybrid (None) | 2.5~10.2804초 | RRF=0.032522 | 정답 IoU=0.4898
2위 | subtitle (ko) | 5.96~8.08초 | RRF=0.032522 | 정답 IoU=0.3955
3위 | subtitle (en) | 21.52~26.56초 | RRF=0.031498 | 정답 IoU=0.0000
4위 | hybrid (None) | 20.5~31.7216초 | RRF=0.031498 | 정답 IoU=0.0000
5위 | subtitle (ko) | 33.52~34.8초 | RRF=0.015385 | 정답 IoU=0.0000

===== 다이소: RRF 재정렬 =====
1위 | hybrid (None) | 14.5~25.0135초 | RRF=0.016393 | 정답 IoU=0.0000
2위 | subtitle (ko) | 2.08~7.28초 | RRF=0.016129 | 정답 IoU=0.6646
3위 | hybrid (None) | 24.5~28.4991초 | RRF=0.015873 | 정답 IoU=0.0000
4위 | subtitle (en) | 4.68~10.0초 | RRF=0.015625 | 정답 IoU=0.5376


In [61]:
for name, video_id, subs, query_ko, query_en, duration in test_cases:

    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=query_ko,
        english_query=query_en,
        subtitles=subs,
        model=fusion_search.get_semantic_model(),
        top_k=3
    )

    ko = [r for r in candidates if r["language"] == "ko"]
    en = [r for r in candidates if r["language"] == "en"]

    print(f"\n===== {name} 영상 =====")

    for i, k in enumerate(ko, 1):
        for j, e in enumerate(en, 1):
            iou, overlap = compare_temporal_overlap(k, e)

            if overlap >= 0.8:
                print(
                    f"한국어 {i}위 {k['start_time']}~{k['end_time']}초 "
                    f"↔ 영어 {j}위 {e['start_time']}~{e['end_time']}초 "
                    f"| IoU={iou:.4f}, 포함률={overlap:.4f}"
                )


===== 두부 영상 =====

===== 주방 영상 =====
한국어 3위 18.2~19.76초 ↔ 영어 2위 15.96~21.52초 | IoU=0.2806, 포함률=1.0000

===== 다이소 영상 =====
한국어 1위 2.08~7.28초 ↔ 영어 1위 3.08~7.28초 | IoU=0.8077, 포함률=1.0000
한국어 1위 2.08~7.28초 ↔ 영어 2위 2.08~7.28초 | IoU=1.0000, 포함률=1.0000
한국어 1위 2.08~7.28초 ↔ 영어 3위 3.08~8.4초 | IoU=0.6646, 포함률=0.8077
한국어 2위 3.08~7.28초 ↔ 영어 1위 3.08~7.28초 | IoU=1.0000, 포함률=1.0000
한국어 2위 3.08~7.28초 ↔ 영어 2위 2.08~7.28초 | IoU=0.8077, 포함률=1.0000
한국어 2위 3.08~7.28초 ↔ 영어 3위 3.08~8.4초 | IoU=0.7895, 포함률=1.0000
한국어 3위 3.08~8.4초 ↔ 영어 1위 3.08~7.28초 | IoU=0.7895, 포함률=1.0000
한국어 3위 3.08~8.4초 ↔ 영어 2위 2.08~7.28초 | IoU=0.6646, 포함률=0.8077
한국어 3위 3.08~8.4초 ↔ 영어 3위 3.08~8.4초 | IoU=1.0000, 포함률=1.0000


In [62]:
import json
from pathlib import Path

save_path = video_dir / "fusion_baseline_evaluation.json"

report = {
    "description": "PinPoint Fusion 검색 예비 평가",
    "evaluation_note": "3개 한국어 영상, 수동 영어 query, 임시 정답 구간",
    "deduplication": "temporal IoU >= 0.5",
    "ranking": "hybrid, Korean semantic, English semantic interleaving",
    "ground_truth": ground_truth,
    "results": evaluation_results
}

save_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("평가 결과 저장 완료:", save_path)
print("평가 영상 수:", len(report["results"]))
print("파일 크기:", save_path.stat().st_size, "bytes")

평가 결과 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/fusion_baseline_evaluation.json
평가 영상 수: 3
파일 크기: 2999 bytes


# 기존 학습 모델 확인하기

In [63]:
from pathlib import Path

model_path = Path(
    "/content/pinpoint-capstone/models/moment_detr/scratch_clip_align_best.ckpt"
)

if model_path.exists():
    print("기존 학습 모델 확인 완료!")
    print("파일:", model_path)
    print("크기:", round(model_path.stat().st_size / (1024**2), 2), "MB")
else:
    print("현재 Colab에서 기존 학습 모델을 찾을 수 없습니다.")

기존 학습 모델 확인 완료!
파일: /content/pinpoint-capstone/models/moment_detr/scratch_clip_align_best.ckpt
크기: 48.88 MB


In [64]:
import shutil
import hashlib
from pathlib import Path

backup_dir = video_dir / "AI_Backup"
backup_dir.mkdir(parents=True, exist_ok=True)

backup_path = backup_dir / "scratch_clip_align_best_ORIGINAL.ckpt"

# 원본과 백업 파일의 SHA-256 계산
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if backup_path.exists():
    print("기존 백업 파일이 있어 덮어쓰지 않았습니다.")
else:
    shutil.copy2(model_path, backup_path)
    print("Google Drive 모델 백업 완료!")

original_hash = file_sha256(model_path)
backup_hash = file_sha256(backup_path)

print("원본:", model_path)
print("백업:", backup_path)
print("원본 SHA-256:", original_hash)
print("백업 SHA-256:", backup_hash)
print("파일 일치:", original_hash == backup_hash)

assert original_hash == backup_hash, "원본과 백업 파일이 다릅니다!"

기존 백업 파일이 있어 덮어쓰지 않았습니다.
원본: /content/pinpoint-capstone/models/moment_detr/scratch_clip_align_best.ckpt
백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/scratch_clip_align_best_ORIGINAL.ckpt
원본 SHA-256: e4ae27433dd080811d40e6e58ab72ed58c41da9f83fb1fb82eee4f677857eee6
백업 SHA-256: e4ae27433dd080811d40e6e58ab72ed58c41da9f83fb1fb82eee4f677857eee6
파일 일치: True


In [65]:
import shutil
import hashlib
from pathlib import Path

source = Path("/content/pinpoint-capstone/ai/features/fusion_search.py")
backup = backup_dir / "fusion_search_20261010.py"

assert source.is_file(), "Fusion 검색 코드가 없습니다."

if backup.exists():
    print("기존 백업이 있어 덮어쓰지 않았습니다.")
else:
    shutil.copy2(source, backup)
    print("Fusion 검색 코드 백업 완료!")

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("원본:", source)
print("백업:", backup)
print("원본과 백업 일치:", sha256(source) == sha256(backup))

assert sha256(source) == sha256(backup), "백업 파일이 원본과 다릅니다!"

기존 백업이 있어 덮어쓰지 않았습니다.
원본: /content/pinpoint-capstone/ai/features/fusion_search.py
백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/fusion_search_20261010.py
원본과 백업 일치: True


# Fusion 후보별 자막 내용 분석

In [66]:
import json

# Google Drive에 저장한 이전 평가 결과 불러오기
report = json.loads(
    (video_dir / "fusion_baseline_evaluation.json").read_text(
        encoding="utf-8"
    )
)

subtitle_data = {
    "두부": tofu_subtitles,
    "주방": kitchen_subtitles,
    "다이소": daiso_subtitles
}

def get_subtitle_text(subtitles, start, end):
    texts = [
        s["text"]
        for s in subtitles
        if s["start_time"] < end
        and s["end_time"] > start
    ]
    return " ".join(texts)

for name, predictions in report["results"].items():
    print(f"\n===== {name} 영상 =====")

    for r in predictions:
        text = get_subtitle_text(
            subtitle_data[name],
            r["start_time"],
            r["end_time"]
        )

        print(
            f"\n{r['rank']}위 | {r['source']} ({r['language']}) "
            f"| {r['start_time']}~{r['end_time']}초 "
            f"| 정답 IoU={r['iou']:.4f}"
        )
        print("자막:", text[:250] if text else "(해당 구간 자막 없음)")


===== 두부 영상 =====

1위 | hybrid (None) | 98.5~118.9725초 | 정답 IoU=0.0000
자막: 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요 여기서 다진 마늘이 핵심이라 꼭 넉넉하게 넣어주세요 송송선 대파도 조금 넣어서 향과 맛을 살려주고 고수도도 넣어주세요 매콤한 걸 좋아하시면 청양고추도 넣으시면 돼요 재료를 다 넣었으면 뜨거운 기름 한 숟가락을 부어서 향을 확 끓어 올려주세요 두부 맛을 좌우하는 건 바로 이 양념이에요 만드는 방법은 정말 간단해요

2위 | subtitle (ko) | 140.86~143.58초 | 정답 IoU=0.0000
자막: 이렇게 양념장을 끼얹은 두부가 완성됐어요

3위 | subtitle (en) | 12.88~21.06초 | 정답 IoU=0.7436
자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요 두께는 젓가락 하나 정도면 딱 좋아요

4위 | hybrid (None) | 84.5~102.7907초 | 정답 IoU=0.0000
자막: 그리고 이 영상을 보고 계신 모든 분들 늘 건강하시고 자녀분들도 아무 탈 없이 평안하고 온 가족이 행복하시길 진심으로 바랄게요 항상 응원해주셔서 감사합니다 두부가 다 익었으면 건져서 남아있는 물기를 꼭 충분히 빼주세요 그래야 양념이 훨씬 잘 빼고 더 맛있어요 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요 여기서 다진 마늘이 핵심이라 꼭 넉넉하게 넣어주세요

5위 | subtitle (ko) | 135.06~137.58초 | 정답 IoU=0.0000
자막: 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요

===== 주방 영상 =====

1위 | hybrid (None) | 2.5~10.2804초 | 정답 IoU=0.4898
자막: 진작 살걸 후회하는 앞머리 보구에 주방 추천템 세가지 첫번째, 솔직히 이거 필요 없을거야 생각했는데 이 가위로 소스 짜는거 진짜 편해요 손

In [67]:
from sentence_transformers import util

semantic_model = fusion_search.get_semantic_model()

queries = {
    "두부": "두부를 칼로 자르는 장면",
    "주방": "가위로 소스를 짜는 장면",
    "다이소": "여행용 화장품 파우치를 보여주는 장면"
}

for name, predictions in report["results"].items():
    query = queries[name]
    subtitles = subtitle_data[name]

    texts = [
        get_subtitle_text(
            subtitles,
            r["start_time"],
            r["end_time"]
        )
        for r in predictions
    ]

    query_embedding = semantic_model.encode(
        query,
        convert_to_tensor=True
    )

    text_embeddings = semantic_model.encode(
        texts,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        query_embedding,
        text_embeddings
    )[0]

    print(f"\n===== {name} =====")

    for r, score in zip(predictions, scores):
        print(
            f"{r['rank']}위 | {r['source']} | "
            f"의미 유사도={score.item():.4f} | "
            f"정답 IoU={r['iou']:.4f}"
        )


===== 두부 =====
1위 | hybrid | 의미 유사도=0.2146 | 정답 IoU=0.0000
2위 | subtitle | 의미 유사도=0.6732 | 정답 IoU=0.0000
3위 | subtitle | 의미 유사도=0.4849 | 정답 IoU=0.7436
4위 | hybrid | 의미 유사도=0.2986 | 정답 IoU=0.0000
5위 | subtitle | 의미 유사도=0.6575 | 정답 IoU=0.0000

===== 주방 =====
1위 | hybrid | 의미 유사도=0.2734 | 정답 IoU=0.4898
2위 | subtitle | 의미 유사도=0.8635 | 정답 IoU=0.3955
3위 | subtitle | 의미 유사도=0.3760 | 정답 IoU=0.0000
4위 | hybrid | 의미 유사도=0.5034 | 정답 IoU=0.0000
5위 | subtitle | 의미 유사도=0.8609 | 정답 IoU=0.0000

===== 다이소 =====
1위 | hybrid | 의미 유사도=0.4201 | 정답 IoU=0.0000
2위 | subtitle | 의미 유사도=0.6598 | 정답 IoU=0.6646
3위 | hybrid | 의미 유사도=0.3944 | 정답 IoU=0.0000
4위 | subtitle | 의미 유사도=0.4903 | 정답 IoU=0.5376


In [68]:
query_variants = [
    "두부를 칼로 자르는 장면",
    "칼로 두부를 써는 장면",
    "두부를 반으로 자르는 장면",
    "두부를 큼직하게 써는 장면"
]

print("===== 두부 영상 검색어 표현 비교 =====")

for query in query_variants:
    candidates = fusion_search.get_semantic_subtitle_candidates(
        query=query,
        subtitles=tofu_subtitles,
        model=fusion_search.get_semantic_model(),
        top_k=3
    )

    print(f"\n검색어: {query}")

    for i, r in enumerate(candidates, 1):
        iou = fusion_search.temporal_iou(
            r,
            ground_truth["두부"]
        )

        print(
            f"{i}위 | {r['start_time']}~{r['end_time']}초 | "
            f"유사도={r['score']:.4f} | IoU={iou:.4f}"
        )

===== 두부 영상 검색어 표현 비교 =====

검색어: 두부를 칼로 자르는 장면
1위 | 140.86~143.58초 | 유사도=0.6732 | IoU=0.0000
2위 | 135.06~137.58초 | 유사도=0.6575 | IoU=0.0000
3위 | 97.94~101.62초 | 유사도=0.6547 | IoU=0.0000

검색어: 칼로 두부를 써는 장면
1위 | 140.86~143.58초 | 유사도=0.7354 | IoU=0.0000
2위 | 137.58~143.58초 | 유사도=0.7202 | IoU=0.0000
3위 | 135.06~137.58초 | 유사도=0.7143 | IoU=0.0000

검색어: 두부를 반으로 자르는 장면
1위 | 140.86~143.58초 | 유사도=0.7112 | IoU=0.0000
2위 | 137.58~143.58초 | 유사도=0.6816 | IoU=0.0000
3위 | 16.32~19.02초 | 유사도=0.6813 | IoU=0.2455

검색어: 두부를 큼직하게 써는 장면
1위 | 16.32~19.02초 | 유사도=0.9264 | IoU=0.2455
2위 | 130.26~131.98초 | 유사도=0.9249 | IoU=0.0000
3위 | 116.02~117.9초 | 유사도=0.9248 | IoU=0.0000


In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

query = "두부를 칼로 자르는 장면"

texts = [s["text"] for s in tofu_subtitles]

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5)
)

matrix = vectorizer.fit_transform(texts + [query])
scores = cosine_similarity(matrix[-1], matrix[:-1])[0]

top_indices = scores.argsort()[::-1][:5]

print("===== 두부 영상 TF-IDF 검색 결과 =====")

for rank, idx in enumerate(top_indices, 1):
    s = tofu_subtitles[idx]

    iou = fusion_search.temporal_iou(
        s, ground_truth["두부"]
    )

    print(
        f"{rank}위 | {s['start_time']}~{s['end_time']}초 | "
        f"TF-IDF={scores[idx]:.4f} | IoU={iou:.4f}"
    )
    print("자막:", s["text"])

===== 두부 영상 TF-IDF 검색 결과 =====
1위 | 145.54~147.38초 | TF-IDF=0.1476 | IoU=0.0000
자막: 두부를 이렇게 만들면 진짜 너무 맛있어요
2위 | 50.9~53.26초 | TF-IDF=0.1447 | IoU=0.0000
자막: 두부를 익힐 때는 먼저 물을 끓인 다음
3위 | 155.7~157.5초 | TF-IDF=0.1288 | IoU=0.0000
자막: 두부를 저처럼 이렇게 만들어보세요
4위 | 36.98~40.86초 | TF-IDF=0.1224 | IoU=0.0000
자막: 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요
5위 | 0.0~4.74초 | TF-IDF=0.1161 | IoU=0.0000
자막: 여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면


In [70]:
query_ko = "두부 양념을 준비하는 장면"
query_en = "preparing seasoning sauce for tofu"

for language, query in [("ko", query_ko), ("en", query_en)]:
    candidates = fusion_search.get_semantic_subtitle_candidates(
        query=query,
        subtitles=tofu_subtitles,
        model=fusion_search.get_semantic_model(),
        top_k=3
    )

    print(f"\n===== {language} 검색 =====")

    for i, r in enumerate(candidates, 1):
        print(
            f"{i}위 | {r['start_time']}~{r['end_time']}초 "
            f"| 유사도={r['score']:.4f}"
        )


===== ko 검색 =====
1위 | 140.86~143.58초 | 유사도=0.7338
2위 | 157.5~158.74초 | 유사도=0.7228
3위 | 130.26~131.98초 | 유사도=0.7161

===== en 검색 =====
1위 | 108.82~117.9초 | 유사도=0.5530
2위 | 108.82~116.02초 | 유사도=0.5423
3위 | 111.26~119.62초 | 유사도=0.5285


In [71]:
start_time = 108.82
end_time = 117.90

print("===== 영어 검색 1위 구간의 자막 =====")

for s in tofu_subtitles:
    if s["start_time"] < end_time and s["end_time"] > start_time:
        print(
            f"{s['start_time']}~{s['end_time']}초:",
            s["text"]
        )

===== 영어 검색 1위 구간의 자막 =====
108.82~111.26초: 매콤한 걸 좋아하시면 청양고추도 넣으시면 돼요
111.26~116.02초: 재료를 다 넣었으면 뜨거운 기름 한 숟가락을 부어서 향을 확 끓어 올려주세요
116.02~117.9초: 두부 맛을 좌우하는 건 바로 이 양념이에요


In [72]:
print("===== 두부 영상 90~125초 자막 =====")

for s in tofu_subtitles:
    if s["start_time"] < 125 and s["end_time"] > 90:
        print(
            f"{s['start_time']:.2f}~{s['end_time']:.2f}초 | "
            f"{s['text']}"
        )

===== 두부 영상 90~125초 자막 =====
89.58~91.66초 | 항상 응원해주셔서 감사합니다
91.66~95.86초 | 두부가 다 익었으면 건져서 남아있는 물기를 꼭 충분히 빼주세요
95.86~97.94초 | 그래야 양념이 훨씬 잘 빼고 더 맛있어요
97.94~101.62초 | 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요
101.62~104.54초 | 여기서 다진 마늘이 핵심이라 꼭 넉넉하게 넣어주세요
104.54~108.82초 | 송송선 대파도 조금 넣어서 향과 맛을 살려주고 고수도도 넣어주세요
108.82~111.26초 | 매콤한 걸 좋아하시면 청양고추도 넣으시면 돼요
111.26~116.02초 | 재료를 다 넣었으면 뜨거운 기름 한 숟가락을 부어서 향을 확 끓어 올려주세요
116.02~117.90초 | 두부 맛을 좌우하는 건 바로 이 양념이에요
117.90~119.62초 | 만드는 방법은 정말 간단해요
119.62~121.06초 | 먼저 소금을 조금 넣어주세요
121.06~122.94초 | 그다음 설탕 작은 숟가락 하나
122.94~124.30초 | 미원 작은 숟가락 하나를 넣고
124.30~126.58초 | 굴소스도 조금 짜 넣어서 감칠맛을 더해주세요


In [73]:
from pathlib import Path

video_files = [
    p for p in video_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in (".mp4", ".mov", ".mkv", ".webm")
]

print("영상 파일 개수:", len(video_files))

for i, path in enumerate(video_files, 1):
    print(f"{i}. {path.name}")
    print("경로:", path)

영상 파일 개수: 4
1. 두부요리영상.mp4
경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/두부요리영상.mp4
2. 주방추천템영상.mp4
경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/주방추천템영상.mp4
3. 다이소추천템영상.mp4
경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/다이소추천템영상.mp4
4. 스쿼트자세설명영상.mp4
경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/스쿼트자세설명영상.mp4


In [75]:
from pathlib import Path

video_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

print("===== Google Drive 영상 파일 확인 =====")
print("폴더 존재:", video_dir.exists())

if video_dir.exists():
    for path in sorted(video_dir.iterdir()):
        if path.suffix.lower() in [".mp4", ".mov", ".mkv"]:
            print(path.name)

print("\n✅ 영상 파일 확인 완료!")

===== Google Drive 영상 파일 확인 =====
폴더 존재: True
다이소추천템영상.mp4
두부요리영상.mp4
스쿼트자세설명영상.mp4
주방추천템영상.mp4

✅ 영상 파일 확인 완료!


In [76]:
from pathlib import Path
from IPython.display import Video, display
import unicodedata

video_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

# 한글 파일명의 유니코드 표기 차이까지 처리
video_files = {
    unicodedata.normalize("NFC", p.stem): p
    for p in video_dir.glob("*.mp4")
}

videos = {
    "tofu": video_files["두부요리영상"],
    "kitchen": video_files["주방추천템영상"],
    "daiso": video_files["다이소추천템영상"],
    "squat": video_files["스쿼트자세설명영상"]
}

print("===== 영상 경로 복구 완료 =====")

for name, path in videos.items():
    print(f"{name}: {path.exists()} | {path.name}")

print("\n===== 두부 영상 재생 =====")

display(
    Video(
        str(videos["tofu"]),
        embed=True
    )
)

Output hidden; open in https://colab.research.google.com to view.

In [77]:
from IPython.display import Video, display

print("두부 영상 경로:", videos["tofu"])
print("파일 존재:", videos["tofu"].is_file())

display(
    Video(
        str(videos["tofu"]),
        embed=True
    )
)

Output hidden; open in https://colab.research.google.com to view.

In [78]:
from pathlib import Path
import unicodedata

video_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

videos = {}

for path in video_dir.glob("*.mp4"):
    name = unicodedata.normalize("NFC", path.name)

    if "두부" in name:
        videos["tofu"] = path
    elif "주방" in name:
        videos["kitchen"] = path
    elif "다이소" in name:
        videos["daiso"] = path

print("===== 영상 경로 복구 결과 =====")

for key in ["tofu", "kitchen", "daiso"]:
    path = videos.get(key)
    print(f"{key}: {path}")
    print("파일 존재:", path.is_file() if path else False)

===== 영상 경로 복구 결과 =====
tofu: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/두부요리영상.mp4
파일 존재: True
kitchen: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/주방추천템영상.mp4
파일 존재: True
daiso: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/다이소추천템영상.mp4
파일 존재: True


In [79]:
# 두부 영상의 두 번째 검색 테스트 정답
seasoning_ground_truth = {
    "video_id": "kr_tofu_001",
    "query_ko": "두부 양념을 준비하는 장면",
    "query_en": "preparing seasoning sauce for tofu",
    "start_time": 95.0,
    "end_time": 135.0
}

print("===== 새로운 정답 구간 =====")
print("영상:", seasoning_ground_truth["video_id"])
print("한국어 검색어:", seasoning_ground_truth["query_ko"])
print("영어 검색어:", seasoning_ground_truth["query_en"])
print(
    "정답 구간:",
    seasoning_ground_truth["start_time"],
    "~",
    seasoning_ground_truth["end_time"],
    "초"
)

===== 새로운 정답 구간 =====
영상: kr_tofu_001
한국어 검색어: 두부 양념을 준비하는 장면
영어 검색어: preparing seasoning sauce for tofu
정답 구간: 95.0 ~ 135.0 초


In [80]:
import json

query_ko = seasoning_ground_truth["query_ko"]
query_en = seasoning_ground_truth["query_en"]

# 기존 Moment-DETR Hybrid 검색
hybrid_results = fusion_search.predict_moments_hybrid(
    video_id="kr_tofu_001",
    video_feature_path="data/features/kr_tofu_001_clip_features.pt",
    query=query_en,
    subtitle_json_path=str(video_dir / "kr_tofu_001_subtitles.json"),
    video_duration=165.0,
    top_k=5
)

# 우리가 만든 Fusion 검색
fusion_results = fusion_search.predict_moments_fusion(
    video_id="kr_tofu_001",
    feature_path="data/features/kr_tofu_001_clip_features.pt",
    subtitles=tofu_subtitles,
    korean_query=query_ko,
    english_query=query_en,
    video_duration=165.0,
    subtitle_json_path=str(video_dir / "kr_tofu_001_subtitles.json"),
    top_k=5
)

print("===== 기존 Hybrid =====")
for i, r in enumerate(hybrid_results, 1):
    iou = fusion_search.temporal_iou(
        r, seasoning_ground_truth
    )
    print(
        f"{i}위 | {r['start_time']}~{r['end_time']}초 "
        f"| IoU={iou:.4f}"
    )

print("\n===== Fusion =====")
for i, r in enumerate(fusion_results, 1):
    iou = fusion_search.temporal_iou(
        r, seasoning_ground_truth
    )
    print(
        f"{i}위 | {r['source']} ({r['language']}) "
        f"| {r['start_time']}~{r['end_time']}초 "
        f"| IoU={iou:.4f}"
    )

===== 기존 Hybrid =====
1위 | 98.5~119.1695초 | IoU=0.5167
2위 | 134.5~151.3339초 | IoU=0.0089
3위 | 40.5~98.2857초 | IoU=0.0348
4위 | 80.5~96.8931초 | IoU=0.0347
5위 | 156.5~165.0초 | IoU=0.0000

===== Fusion =====
1위 | hybrid (None) | 98.5~119.1695초 | IoU=0.5167
2위 | subtitle (ko) | 140.86~143.58초 | IoU=0.0000
3위 | subtitle (en) | 108.82~117.9초 | IoU=0.2270
4위 | hybrid (None) | 134.5~151.3339초 | IoU=0.0089
5위 | subtitle (ko) | 157.5~158.74초 | IoU=0.0000


In [81]:
import json
from pathlib import Path

new_evaluation = {
    "ground_truth": seasoning_ground_truth,
    "hybrid_results": [
        {
            "rank": i,
            "start_time": r["start_time"],
            "end_time": r["end_time"],
            "iou": fusion_search.temporal_iou(r, seasoning_ground_truth)
        }
        for i, r in enumerate(hybrid_results, 1)
    ],
    "fusion_results": [
        {
            "rank": i,
            "source": r["source"],
            "language": r["language"],
            "start_time": r["start_time"],
            "end_time": r["end_time"],
            "iou": fusion_search.temporal_iou(r, seasoning_ground_truth)
        }
        for i, r in enumerate(fusion_results, 1)
    ]
}

save_path = video_dir / "seasoning_evaluation.json"

save_path.write_text(
    json.dumps(new_evaluation, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("양념 준비 장면 평가 저장 완료:", save_path)
print("Hybrid 후보:", len(new_evaluation["hybrid_results"]))
print("Fusion 후보:", len(new_evaluation["fusion_results"]))

양념 준비 장면 평가 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/seasoning_evaluation.json
Hybrid 후보: 5
Fusion 후보: 5


In [82]:
print("===== 주방 영상 전체 자막 =====")

for s in kitchen_subtitles:
    print(
        f"{s['start_time']:.2f}~{s['end_time']:.2f}초 | "
        f"{s['text']}"
    )

===== 주방 영상 전체 자막 =====
0.00~3.36초 | 진작 살걸 후회하는 앞머리 보구에 주방 추천템 세가지
3.36~5.96초 | 첫번째, 솔직히 이거 필요 없을거야 생각했는데
5.96~8.08초 | 이 가위로 소스 짜는거 진짜 편해요
8.08~11.32초 | 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데
11.32~13.60초 | 가위 필요할땐 가위로 집게 필요할땐 집게로
13.60~15.96초 | 하나로 다 기능을 해서 솔직히 편하더라구요
15.96~18.20초 | 두번째, 냄비에 딱 맞는 체망이 있어서
18.20~19.76초 | 고리차 끓여서 건전해기 좋고
19.76~21.52초 | 물고가 있어서 한 방울도 흘리지 않아
21.52~23.32초 | 시금치 대치고 비빔면 끓이고
23.32~24.36초 | 튀김요리도 쉬운데
24.36~26.56초 | 최고 굽을 땐 316ti의 우염마제품
26.56~29.28초 | 중국제품 아니고 메이드인 코리아라 더 맘에 들어요
29.28~31.72초 | 세번째, 이런 물구멍이 있어서 쌀 씻으면
31.72~33.52초 | 한 톨도 흘리지 않고 과일 씻어서
33.52~34.80초 | 물 바로 버리기도 좋아요
34.80~37.08초 | 각도 조절이 되서 바로 쌀 불리기도 오케이


In [83]:
from IPython.display import Video, display

display(
    Video(
        str(videos["kitchen"]),
        embed=True
    )
)

Output hidden; open in https://colab.research.google.com to view.

In [84]:
rice_ground_truth = {
    "video_id": "kr_kitchen_001",
    "query_ko": "쌀을 씻는 장면",
    "query_en": "washing rice",
    "start_time": 29.0,
    "end_time": 32.0
}

print("===== 쌀 씻는 장면 정답 =====")
print("영상:", rice_ground_truth["video_id"])
print("한국어 검색어:", rice_ground_truth["query_ko"])
print("영어 검색어:", rice_ground_truth["query_en"])
print(
    "정답 구간:",
    rice_ground_truth["start_time"],
    "~",
    rice_ground_truth["end_time"],
    "초"
)

===== 쌀 씻는 장면 정답 =====
영상: kr_kitchen_001
한국어 검색어: 쌀을 씻는 장면
영어 검색어: washing rice
정답 구간: 29.0 ~ 32.0 초


In [85]:
query_ko = rice_ground_truth["query_ko"]
query_en = rice_ground_truth["query_en"]

# ① 기존 Moment-DETR Hybrid
rice_hybrid_results = fusion_search.predict_moments_hybrid(
    video_id="kr_kitchen_001",
    video_feature_path="data/features/kr_kitchen_001_clip_features.pt",
    query=query_en,
    subtitle_json_path=str(video_dir / "kr_kitchen_001_subtitles.json"),
    video_duration=37.13,
    top_k=5
)

# ② 우리가 만든 Fusion 검색
rice_fusion_results = fusion_search.predict_moments_fusion(
    video_id="kr_kitchen_001",
    feature_path="data/features/kr_kitchen_001_clip_features.pt",
    subtitles=kitchen_subtitles,
    korean_query=query_ko,
    english_query=query_en,
    video_duration=37.13,
    subtitle_json_path=str(video_dir / "kr_kitchen_001_subtitles.json"),
    top_k=5
)

print("===== 기존 Hybrid =====")

for i, r in enumerate(rice_hybrid_results, 1):
    iou = fusion_search.temporal_iou(r, rice_ground_truth)

    print(
        f"{i}위 | {r['start_time']}~{r['end_time']}초 "
        f"| IoU={iou:.4f}"
    )

print("\n===== Fusion =====")

for i, r in enumerate(rice_fusion_results, 1):
    iou = fusion_search.temporal_iou(r, rice_ground_truth)

    print(
        f"{i}위 | {r['source']} ({r['language']}) "
        f"| {r['start_time']}~{r['end_time']}초 "
        f"| IoU={iou:.4f}"
    )

===== 기존 Hybrid =====
1위 | 22.5~36.4752초 | IoU=0.2147
2위 | 6.5~14.4555초 | IoU=0.0000

===== Fusion =====
1위 | hybrid (None) | 22.5~36.4752초 | IoU=0.2147
2위 | subtitle (ko) | 29.28~31.72초 | IoU=0.8133
3위 | hybrid (None) | 6.5~14.4555초 | IoU=0.0000
4위 | subtitle (ko) | 26.56~31.72초 | IoU=0.5000
5위 | subtitle (en) | 29.28~34.8초 | IoU=0.4690


In [86]:
import json
from pathlib import Path

rice_evaluation = {
    "ground_truth": rice_ground_truth,
    "hybrid_results": [
        {
            "rank": i,
            "start_time": r["start_time"],
            "end_time": r["end_time"],
            "iou": fusion_search.temporal_iou(r, rice_ground_truth)
        }
        for i, r in enumerate(rice_hybrid_results, 1)
    ],
    "fusion_results": [
        {
            "rank": i,
            "source": r["source"],
            "language": r["language"],
            "start_time": r["start_time"],
            "end_time": r["end_time"],
            "iou": fusion_search.temporal_iou(r, rice_ground_truth)
        }
        for i, r in enumerate(rice_fusion_results, 1)
    ]
}

save_path = video_dir / "rice_washing_evaluation.json"

save_path.write_text(
    json.dumps(rice_evaluation, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("쌀 씻기 평가 저장 완료:", save_path)
print("Hybrid 후보:", len(rice_evaluation["hybrid_results"]))
print("Fusion 후보:", len(rice_evaluation["fusion_results"]))

쌀 씻기 평가 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/rice_washing_evaluation.json
Hybrid 후보: 2
Fusion 후보: 5


In [87]:
print("===== 다이소 영상 전체 자막 =====")

for s in daiso_subtitles:
    print(
        f"{s['start_time']:.2f}~{s['end_time']:.2f}초 | "
        f"{s['text']}"
    )

===== 다이소 영상 전체 자막 =====
0.00~2.08초 | 팅으로 잘 썼던 다이소 상반기 결산템디
2.08~3.08초 | 빠르게 소개해 드릴게요
3.08~4.68초 | 여행 갈 때 무조건 챙기는 파우치
4.68~7.28초 | 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데
7.28~8.40초 | 이제 이거 하나는 끝이에요
8.40~10.00초 | 고기 먹을 때 필수인 양념세트
10.00~11.36초 | 참기름, 소금, 쌈장
11.36~13.72초 | 쭈왕까지 수량씩 들어있어서 먹기 너무 편해요
13.72~15.16초 | 빨리 편하게 해주는 시트 세제
15.16~17.08초 | 톡 뜯어 여행 갈 때 들고 가기도 좋고
17.08~19.08초 | 섬유인지 안 넣어도 향기가 대박이에요
19.08~20.60초 | 피랑 진짜 비슷한 젤리판
20.60~22.68초 | 다이소 립 중에는 제일 잘 쓰게 되더라고요
22.68~24.48초 | 탱글티즈랑 비슷한 비슷한 요거 두 개
24.48~26.80초 | 현상모듣게 잘 빗겨서 여행 갈 때 꼭 챙겨가요
26.80~29.00초 | 집에 두면 한층 더 귀여워지는 인테리어템
29.00~30.08초 | 저소음이라 더 좋고
30.08~31.08초 | 어디도 더 잘 어울려서
31.08~32.08초 | 이건 진짜 잘 썬데


In [88]:
from IPython.display import Video, display

display(
    Video(
        str(videos["daiso"]),
        embed=True
    )
)

Output hidden; open in https://colab.research.google.com to view.

In [89]:
detergent_ground_truth = {
    "video_id": "kr_daiso_001",
    "query_ko": "여행용 시트 세제를 보여주는 장면",
    "query_en": "showing travel laundry detergent sheets",
    "start_time": 13.0,
    "end_time": 18.0
}

print("===== 시트 세제 장면 정답 =====")
print("영상:", detergent_ground_truth["video_id"])
print("한국어 검색어:", detergent_ground_truth["query_ko"])
print("영어 검색어:", detergent_ground_truth["query_en"])
print(
    "정답 구간:",
    detergent_ground_truth["start_time"],
    "~",
    detergent_ground_truth["end_time"],
    "초"
)

===== 시트 세제 장면 정답 =====
영상: kr_daiso_001
한국어 검색어: 여행용 시트 세제를 보여주는 장면
영어 검색어: showing travel laundry detergent sheets
정답 구간: 13.0 ~ 18.0 초


In [90]:
query_ko = detergent_ground_truth["query_ko"]
query_en = detergent_ground_truth["query_en"]

# ① 기존 Moment-DETR Hybrid 검색
detergent_hybrid_results = fusion_search.predict_moments_hybrid(
    video_id="kr_daiso_001",
    video_feature_path="data/features/kr_daiso_001_clip_features.pt",
    query=query_en,
    subtitle_json_path=str(video_dir / "kr_daiso_001_subtitles.json"),
    video_duration=32.07,
    top_k=5
)

# ② 우리가 만든 Fusion 검색
detergent_fusion_results = fusion_search.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path="data/features/kr_daiso_001_clip_features.pt",
    subtitles=daiso_subtitles,
    korean_query=query_ko,
    english_query=query_en,
    video_duration=32.07,
    subtitle_json_path=str(video_dir / "kr_daiso_001_subtitles.json"),
    top_k=5
)

print("===== 기존 Hybrid =====")

for i, r in enumerate(detergent_hybrid_results, 1):
    iou = fusion_search.temporal_iou(r, detergent_ground_truth)

    print(
        f"{i}위 | {r['start_time']}~{r['end_time']}초 "
        f"| IoU={iou:.4f}"
    )

print("\n===== Fusion =====")

for i, r in enumerate(detergent_fusion_results, 1):
    iou = fusion_search.temporal_iou(r, detergent_ground_truth)

    print(
        f"{i}위 | {r['source']} ({r['language']}) "
        f"| {r['start_time']}~{r['end_time']}초 "
        f"| IoU={iou:.4f}"
    )

===== 기존 Hybrid =====
1위 | 14.5~24.5127초 | IoU=0.3040
2위 | 22.5~29.1065초 | IoU=0.0000
3위 | 2.5~14.7196초 | IoU=0.1109

===== Fusion =====
1위 | hybrid (None) | 14.5~24.5127초 | IoU=0.3040
2위 | subtitle (ko) | 3.08~4.68초 | IoU=0.0000
3위 | subtitle (en) | 22.68~29.0초 | IoU=0.0000
4위 | subtitle (ko) | 3.08~8.4초 | IoU=0.0000
5위 | subtitle (en) | 20.6~24.48초 | IoU=0.0000


In [91]:
print("===== 한국어 검색어 기준 자막 후보 =====")

ko_candidates = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=detergent_ground_truth["query_ko"],
    english_query=detergent_ground_truth["query_en"],
    subtitles=daiso_subtitles,
    model=fusion_search.get_semantic_model(),
    top_k=10
)

for i, r in enumerate(ko_candidates, 1):
    print(
        f"{i}위 | {r['start_time']:.2f}~{r['end_time']:.2f}초 "
        f"| 언어={r.get('language')} "
        f"| 유사도={r.get('subtitle_score', r.get('score'))}"
    )

===== 한국어 검색어 기준 자막 후보 =====
1위 | 3.08~4.68초 | 언어=ko | 유사도=0.423
2위 | 3.08~8.40초 | 언어=ko | 유사도=0.4087
3위 | 3.08~7.28초 | 언어=ko | 유사도=0.3922
4위 | 2.08~7.28초 | 언어=ko | 유사도=0.3864
5위 | 2.08~4.68초 | 언어=ko | 유사도=0.3735
6위 | 0.00~4.68초 | 언어=ko | 유사도=0.3626
7위 | 22.68~29.00초 | 언어=ko | 유사도=0.3473
8위 | 24.48~29.00초 | 언어=ko | 유사도=0.343
9위 | 15.16~19.08초 | 언어=ko | 유사도=0.3397
10위 | 22.68~24.48초 | 언어=ko | 유사도=0.3273
11위 | 22.68~29.00초 | 언어=en | 유사도=0.3968
12위 | 20.60~24.48초 | 언어=en | 유사도=0.3878
13위 | 3.08~7.28초 | 언어=en | 유사도=0.3604
14위 | 2.08~7.28초 | 언어=en | 유사도=0.3589
15위 | 20.60~26.80초 | 언어=en | 유사도=0.3503
16위 | 3.08~8.40초 | 언어=en | 유사도=0.3338
17위 | 19.08~24.48초 | 언어=en | 유사도=0.3309
18위 | 22.68~26.80초 | 언어=en | 유사도=0.2766
19위 | 4.68~10.00초 | 언어=en | 유사도=0.2751
20위 | 24.48~29.00초 | 언어=en | 유사도=0.2566


In [92]:
print("===== 검색어 변경 후 한국어 자막 후보 =====")

for i, r in enumerate(ko_candidates, 1):
    if r.get("language") != "ko":
        continue

    iou = fusion_search.temporal_iou(
        r, detergent_ground_truth
    )

    score = r.get("subtitle_score")
    if score is None:
        score = r.get("score")

    print(
        f"{i}위 | {r['start_time']:.2f}~{r['end_time']:.2f}초 "
        f"| 유사도={score} "
        f"| IoU={iou:.4f}"
    )

===== 검색어 변경 후 한국어 자막 후보 =====
1위 | 3.08~4.68초 | 유사도=0.423 | IoU=0.0000
2위 | 3.08~8.40초 | 유사도=0.4087 | IoU=0.0000
3위 | 3.08~7.28초 | 유사도=0.3922 | IoU=0.0000
4위 | 2.08~7.28초 | 유사도=0.3864 | IoU=0.0000
5위 | 2.08~4.68초 | 유사도=0.3735 | IoU=0.0000
6위 | 0.00~4.68초 | 유사도=0.3626 | IoU=0.0000
7위 | 22.68~29.00초 | 유사도=0.3473 | IoU=0.0000
8위 | 24.48~29.00초 | 유사도=0.343 | IoU=0.0000
9위 | 15.16~19.08초 | 유사도=0.3397 | IoU=0.4671
10위 | 22.68~24.48초 | 유사도=0.3273 | IoU=0.0000


In [93]:
print("===== 시트 세제 관련 자막 후보 확인 =====")

for label, candidates in [
    ("기존 검색어", ko_candidates)
]:
    print(f"\n[{label}]")

    for i, r in enumerate(candidates, 1):
        if r.get("language") != "ko":
            continue

        start = r["start_time"]
        end = r["end_time"]

        # 실제 정답 구간(13~18초)과 겹치는 후보
        if start < 18.0 and end > 13.0:
            score = r.get("subtitle_score")
            if score is None:
                score = r.get("score")

            iou = fusion_search.temporal_iou(
                r, detergent_ground_truth
            )

            print(
                f"{i}위 | {start:.2f}~{end:.2f}초 "
                f"| 유사도={score} | IoU={iou:.4f}"
            )

===== 시트 세제 관련 자막 후보 확인 =====

[기존 검색어]
9위 | 15.16~19.08초 | 유사도=0.3397 | IoU=0.4671


In [94]:
from sentence_transformers import util

model = fusion_search.get_semantic_model()

queries = [
    "여행용 시트 세제를 보여주는 장면",
    "시트 세제를 보여주는 장면"
]

captions = [
    "빨리 편하게 해주는 시트 세제",
    "여행 갈 때 무조건 챙기는 파우치"
]

query_embeddings = model.encode(
    queries,
    convert_to_tensor=True
)

caption_embeddings = model.encode(
    captions,
    convert_to_tensor=True
)

similarities = util.cos_sim(
    query_embeddings,
    caption_embeddings
)

for i, query in enumerate(queries):
    print(f"\n검색어: {query}")

    for j, caption in enumerate(captions):
        print(
            f"자막: {caption}"
            f" | 유사도: {similarities[i][j].item():.4f}"
        )


검색어: 여행용 시트 세제를 보여주는 장면
자막: 빨리 편하게 해주는 시트 세제 | 유사도: 0.3213
자막: 여행 갈 때 무조건 챙기는 파우치 | 유사도: 0.4230

검색어: 시트 세제를 보여주는 장면
자막: 빨리 편하게 해주는 시트 세제 | 유사도: 0.3238
자막: 여행 갈 때 무조건 챙기는 파우치 | 유사도: 0.3592


In [95]:
query = "시트 세제를 보여주는 장면"

captions = [
    "빨리 편하게 해주는 시트 세제",
    "여행 갈 때 무조건 챙기는 파우치"
]

query_emb = model.encode(
    query,
    convert_to_tensor=True
)

caption_embs = model.encode(
    captions,
    convert_to_tensor=True
)

semantic_scores = util.cos_sim(
    query_emb,
    caption_embs
)[0]

# 이번에는 핵심 제품명 '시트 세제' 일치 여부만 실험
keyword = "시트 세제"

print("===== 핵심 단어 일치 가점 실험 =====")

for caption, semantic_score in zip(captions, semantic_scores):
    semantic_score = semantic_score.item()

    keyword_bonus = 0.15 if keyword in caption else 0.0
    final_score = semantic_score + keyword_bonus

    print(
        f"자막: {caption}\n"
        f"의미 유사도: {semantic_score:.4f}\n"
        f"단어 일치 가점: {keyword_bonus:.2f}\n"
        f"최종 점수: {final_score:.4f}\n"
    )

===== 핵심 단어 일치 가점 실험 =====
자막: 빨리 편하게 해주는 시트 세제
의미 유사도: 0.3238
단어 일치 가점: 0.15
최종 점수: 0.4738

자막: 여행 갈 때 무조건 챙기는 파우치
의미 유사도: 0.3592
단어 일치 가점: 0.00
최종 점수: 0.3592



In [96]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

query = "시트 세제를 보여주는 장면"

captions = [
    "빨리 편하게 해주는 시트 세제",
    "여행 갈 때 무조건 챙기는 파우치"
]

# 한국어 형태소 분석기 없이 글자 단위로 일치 정도 계산
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)

vectors = vectorizer.fit_transform([query] + captions)

keyword_scores = cosine_similarity(
    vectors[0:1],
    vectors[1:]
)[0]

semantic_scores = util.cos_sim(
    model.encode(query, convert_to_tensor=True),
    model.encode(captions, convert_to_tensor=True)
)[0]

print("===== 의미 유사도 + 자동 글자 일치 실험 =====")

for caption, sem, keyword_score in zip(
    captions,
    semantic_scores,
    keyword_scores
):
    sem = sem.item()
    combined = sem + 0.15 * keyword_score

    print(f"\n자막: {caption}")
    print(f"의미 유사도: {sem:.4f}")
    print(f"글자 일치 점수: {keyword_score:.4f}")
    print(f"결합 점수: {combined:.4f}")

===== 의미 유사도 + 자동 글자 일치 실험 =====

자막: 빨리 편하게 해주는 시트 세제
의미 유사도: 0.3238
글자 일치 점수: 0.1999
결합 점수: 0.3538

자막: 여행 갈 때 무조건 챙기는 파우치
의미 유사도: 0.3592
글자 일치 점수: 0.0091
결합 점수: 0.3606


In [97]:
print("===== 글자 일치 가중치별 순위 비교 =====")

for weight in [0.0, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]:
    combined_scores = [
        semantic_scores[i].item() + weight * keyword_scores[i]
        for i in range(len(captions))
    ]

    winner_idx = max(
        range(len(captions)),
        key=lambda i: combined_scores[i]
    )

    print(
        f"가중치 {weight:.2f} | "
        f"시트 세제={combined_scores[0]:.4f} | "
        f"파우치={combined_scores[1]:.4f} | "
        f"1위={captions[winner_idx]}"
    )

===== 글자 일치 가중치별 순위 비교 =====
가중치 0.00 | 시트 세제=0.3238 | 파우치=0.3592 | 1위=여행 갈 때 무조건 챙기는 파우치
가중치 0.10 | 시트 세제=0.3438 | 파우치=0.3601 | 1위=여행 갈 때 무조건 챙기는 파우치
가중치 0.15 | 시트 세제=0.3538 | 파우치=0.3606 | 1위=여행 갈 때 무조건 챙기는 파우치
가중치 0.20 | 시트 세제=0.3638 | 파우치=0.3610 | 1위=빨리 편하게 해주는 시트 세제
가중치 0.30 | 시트 세제=0.3838 | 파우치=0.3619 | 1위=빨리 편하게 해주는 시트 세제
가중치 0.50 | 시트 세제=0.4238 | 파우치=0.3638 | 1위=빨리 편하게 해주는 시트 세제
가중치 1.00 | 시트 세제=0.5237 | 파우치=0.3683 | 1위=빨리 편하게 해주는 시트 세제


In [98]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import util
import numpy as np

query = detergent_ground_truth["query_ko"]

texts = [s["text"] for s in daiso_subtitles]

# 의미 유사도
model = fusion_search.get_semantic_model()

semantic_scores = util.cos_sim(
    model.encode(query, convert_to_tensor=True),
    model.encode(texts, convert_to_tensor=True)
)[0].cpu().numpy()

# 글자 단위 TF-IDF
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)

vectors = vectorizer.fit_transform([query] + texts)

keyword_scores = cosine_similarity(
    vectors[0:1],
    vectors[1:]
)[0]

print("===== 다이소 전체 자막 가중치 비교 =====")

for weight in [0.0, 0.15, 0.2, 0.3, 0.5, 1.0]:
    combined_scores = semantic_scores + weight * keyword_scores

    top_indices = np.argsort(combined_scores)[::-1][:3]

    print(f"\n[가중치 {weight}]")

    for rank, idx in enumerate(top_indices, 1):
        s = daiso_subtitles[idx]

        iou = fusion_search.temporal_iou(
            {
                "start_time": s["start_time"],
                "end_time": s["end_time"]
            },
            detergent_ground_truth
        )

        print(
            f"{rank}위 | {s['start_time']:.2f}~{s['end_time']:.2f}초 "
            f"| 점수={combined_scores[idx]:.4f} "
            f"| IoU={iou:.4f} "
            f"| {s['text']}"
        )

===== 다이소 전체 자막 가중치 비교 =====

[가중치 0.0]
1위 | 3.08~4.68초 | 점수=0.4230 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치
2위 | 22.68~24.48초 | 점수=0.3273 | IoU=0.0000 | 탱글티즈랑 비슷한 비슷한 요거 두 개
3위 | 20.60~22.68초 | 점수=0.3216 | IoU=0.0000 | 다이소 립 중에는 제일 잘 쓰게 되더라고요

[가중치 0.15]
1위 | 3.08~4.68초 | 점수=0.4263 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치
2위 | 13.72~15.16초 | 점수=0.3628 | IoU=0.2880 | 빨리 편하게 해주는 시트 세제
3위 | 22.68~24.48초 | 점수=0.3273 | IoU=0.0000 | 탱글티즈랑 비슷한 비슷한 요거 두 개

[가중치 0.2]
1위 | 3.08~4.68초 | 점수=0.4274 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치
2위 | 13.72~15.16초 | 점수=0.3766 | IoU=0.2880 | 빨리 편하게 해주는 시트 세제
3위 | 22.68~24.48초 | 점수=0.3273 | IoU=0.0000 | 탱글티즈랑 비슷한 비슷한 요거 두 개

[가중치 0.3]
1위 | 3.08~4.68초 | 점수=0.4296 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치
2위 | 13.72~15.16초 | 점수=0.4043 | IoU=0.2880 | 빨리 편하게 해주는 시트 세제
3위 | 22.68~24.48초 | 점수=0.3273 | IoU=0.0000 | 탱글티즈랑 비슷한 비슷한 요거 두 개

[가중치 0.5]
1위 | 13.72~15.16초 | 점수=0.4597 | IoU=0.2880 | 빨리 편하게 해주는 시트 세제
2위 | 3.08~4.68초 | 점수=0.4341 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치
3위 | 22.68~24.48초 | 점수=0

In [99]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import util

query = "두부를 칼로 자르는 장면"
texts = [s["text"] for s in tofu_subtitles]

model = fusion_search.get_semantic_model()

# ① 의미 유사도
semantic_scores = util.cos_sim(
    model.encode(query, convert_to_tensor=True),
    model.encode(texts, convert_to_tensor=True)
)[0].cpu().numpy()

# ② 글자 일치 점수
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)

vectors = vectorizer.fit_transform([query] + texts)

keyword_scores = cosine_similarity(
    vectors[0:1],
    vectors[1:]
)[0]

ground_truth = {
    "start_time": 12.0,
    "end_time": 23.0
}

print("===== 두부 자르기: 전체 자막 비교 =====")

for weight in [0.0, 0.2, 0.5, 1.0]:
    combined = semantic_scores + weight * keyword_scores
    top_indices = np.argsort(combined)[::-1][:3]

    print(f"\n[가중치 {weight}]")

    for rank, idx in enumerate(top_indices, 1):
        s = tofu_subtitles[idx]

        iou = fusion_search.temporal_iou(
            {
                "start_time": s["start_time"],
                "end_time": s["end_time"]
            },
            ground_truth
        )

        print(
            f"{rank}위 | {s['start_time']:.2f}~{s['end_time']:.2f}초 "
            f"| IoU={iou:.4f} "
            f"| {s['text']}"
        )

===== 두부 자르기: 전체 자막 비교 =====

[가중치 0.0]
1위 | 140.86~143.58초 | IoU=0.0000 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 135.06~137.58초 | IoU=0.0000 | 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요
3위 | 97.94~101.62초 | IoU=0.0000 | 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요

[가중치 0.2]
1위 | 140.86~143.58초 | IoU=0.0000 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 135.06~137.58초 | IoU=0.0000 | 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요
3위 | 97.94~101.62초 | IoU=0.0000 | 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요

[가중치 0.5]
1위 | 140.86~143.58초 | IoU=0.0000 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 97.94~101.62초 | IoU=0.0000 | 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요
3위 | 135.06~137.58초 | IoU=0.0000 | 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요

[가중치 1.0]
1위 | 140.86~143.58초 | IoU=0.0000 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 155.70~157.50초 | IoU=0.0000 | 두부를 저처럼 이렇게 만들어보세요
3위 | 12.88~16.32초 | IoU=0.3127 | 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요


In [100]:
query = "두부를 칼로 자르는 장면"

captions = [
    "먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요",
    "이렇게 양념장을 끼얹은 두부가 완성됐어요",
    "물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요"
]

model = fusion_search.get_semantic_model()

# 의미 유사도
sem = util.cos_sim(
    model.encode(query, convert_to_tensor=True),
    model.encode(captions, convert_to_tensor=True)
)[0].cpu().numpy()

# 글자 일치 점수
vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)

X = vec.fit_transform([query] + captions)
lex = cosine_similarity(X[0:1], X[1:])[0]

print("===== 두부 자르기: 자막별 점수 비교 =====")

for caption, s, l in zip(captions, sem, lex):
    print(f"\n자막: {caption}")
    print(f"의미 유사도: {s:.4f}")
    print(f"글자 일치: {l:.4f}")
    print(f"결합 점수(가중치 0.5): {s + 0.5*l:.4f}")

===== 두부 자르기: 자막별 점수 비교 =====

자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요
의미 유사도: 0.6442
글자 일치: 0.0355
결합 점수(가중치 0.5): 0.6619

자막: 이렇게 양념장을 끼얹은 두부가 완성됐어요
의미 유사도: 0.6732
글자 일치: 0.0066
결합 점수(가중치 0.5): 0.6765

자막: 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요
의미 유사도: 0.6547
글자 일치: 0.0221
결합 점수(가중치 0.5): 0.6658


In [101]:
from sentence_transformers import util

queries = [
    "두부를 칼로 자르는 장면",
    "두부를 반으로 자르는 장면",
    "두부를 잘라주세요",
    "두부를 써는 장면"
]

captions = [
    "먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요",
    "이렇게 양념장을 끼얹은 두부가 완성됐어요",
    "물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요"
]

model = fusion_search.get_semantic_model()

scores = util.cos_sim(
    model.encode(queries, convert_to_tensor=True),
    model.encode(captions, convert_to_tensor=True)
)

print("===== 두부 자르기 검색어 표현 비교 =====")

for i, query in enumerate(queries):
    values = scores[i].cpu().numpy()

    print(f"\n검색어: {query}")
    print(f"자르기 자막: {values[0]:.4f}")
    print(f"두부 완성 자막: {values[1]:.4f}")
    print(f"양념 준비 자막: {values[2]:.4f}")
    print("세 자막 중 1위:", captions[values.argmax()])

===== 두부 자르기 검색어 표현 비교 =====

검색어: 두부를 칼로 자르는 장면
자르기 자막: 0.6442
두부 완성 자막: 0.6732
양념 준비 자막: 0.6547
세 자막 중 1위: 이렇게 양념장을 끼얹은 두부가 완성됐어요

검색어: 두부를 반으로 자르는 장면
자르기 자막: 0.6758
두부 완성 자막: 0.7112
양념 준비 자막: 0.6761
세 자막 중 1위: 이렇게 양념장을 끼얹은 두부가 완성됐어요

검색어: 두부를 잘라주세요
자르기 자막: 0.8077
두부 완성 자막: 0.9177
양념 준비 자막: 0.9644
세 자막 중 1위: 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요

검색어: 두부를 써는 장면
자르기 자막: 0.6838
두부 완성 자막: 0.8142
양념 준비 자막: 0.8082
세 자막 중 1위: 이렇게 양념장을 끼얹은 두부가 완성됐어요


In [102]:
model = fusion_search.get_semantic_model()

print("===== 의미 검색 모델 확인 =====")
print("모델:", type(model).__name__)
print("임베딩 차원:", model.get_sentence_embedding_dimension())
print("모델 설정:", model)

===== 의미 검색 모델 확인 =====
모델: SentenceTransformer
임베딩 차원: 384
모델 설정: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)


/tmp/ipykernel_8812/2315749398.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("임베딩 차원:", model.get_sentence_embedding_dimension())


In [103]:
from sentence_transformers import SentenceTransformer, util

# 기존 모델을 교체하지 않고 비교용 모델만 별도로 로드
e5_model = SentenceTransformer("intfloat/multilingual-e5-small")

query = "query: 두부를 칼로 자르는 장면"

captions = [
    "먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요",
    "이렇게 양념장을 끼얹은 두부가 완성됐어요",
    "물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요"
]

passages = ["passage: " + caption for caption in captions]

query_embedding = e5_model.encode(
    query, convert_to_tensor=True, normalize_embeddings=True
)

passage_embeddings = e5_model.encode(
    passages, convert_to_tensor=True, normalize_embeddings=True
)

scores = util.cos_sim(query_embedding, passage_embeddings)[0]

print("===== E5 두부 자르기 검색 =====")

for caption, score in zip(captions, scores):
    print(f"{score.item():.4f} | {caption}")

print("\n1위:", captions[scores.argmax().item()])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

===== E5 두부 자르기 검색 =====
0.8706 | 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요
0.8785 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
0.8522 | 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요

1위: 이렇게 양념장을 끼얹은 두부가 완성됐어요


In [104]:
import numpy as np
from sentence_transformers import util

query = "두부를 칼로 자르는 장면"
texts = [s["text"] for s in tofu_subtitles]

minilm_model = fusion_search.get_semantic_model()

# 기존 MiniLM
minilm_scores = util.cos_sim(
    minilm_model.encode(query, convert_to_tensor=True),
    minilm_model.encode(texts, convert_to_tensor=True)
)[0].cpu().numpy()

# 새 E5
e5_scores = util.cos_sim(
    e5_model.encode(
        "query: " + query,
        convert_to_tensor=True,
        normalize_embeddings=True
    ),
    e5_model.encode(
        ["passage: " + text for text in texts],
        convert_to_tensor=True,
        normalize_embeddings=True
    )
)[0].cpu().numpy()

ground_truth = {
    "start_time": 12.0,
    "end_time": 23.0
}

for name, scores in [
    ("기존 MiniLM", minilm_scores),
    ("새 E5", e5_scores)
]:
    print(f"\n===== {name} 전체 자막 검색 =====")

    for rank, idx in enumerate(np.argsort(scores)[::-1][:5], 1):
        s = tofu_subtitles[idx]

        iou = fusion_search.temporal_iou(
            {
                "start_time": s["start_time"],
                "end_time": s["end_time"]
            },
            ground_truth
        )

        print(
            f"{rank}위 | {s['start_time']:.2f}~{s['end_time']:.2f}초 "
            f"| 유사도={scores[idx]:.4f} "
            f"| IoU={iou:.4f} "
            f"| {s['text']}"
        )


===== 기존 MiniLM 전체 자막 검색 =====
1위 | 140.86~143.58초 | 유사도=0.6732 | IoU=0.0000 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 135.06~137.58초 | 유사도=0.6575 | IoU=0.0000 | 이제 두부에 물기가 다 빠졌으니 접시에 담아주세요
3위 | 97.94~101.62초 | 유사도=0.6547 | IoU=0.0000 | 물기를 빼는 동안 두부 위에 끼얹을 양념을 준비해 볼게요
4위 | 16.32~19.02초 | 유사도=0.6450 | IoU=0.2455 | 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요
5위 | 12.88~16.32초 | 유사도=0.6442 | IoU=0.3127 | 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요

===== 새 E5 전체 자막 검색 =====
1위 | 140.86~143.58초 | 유사도=0.8785 | IoU=0.0000 | 이렇게 양념장을 끼얹은 두부가 완성됐어요
2위 | 155.70~157.50초 | 유사도=0.8764 | IoU=0.0000 | 두부를 저처럼 이렇게 만들어보세요
3위 | 145.54~147.38초 | 유사도=0.8757 | IoU=0.0000 | 두부를 이렇게 만들면 진짜 너무 맛있어요
4위 | 36.98~40.86초 | 유사도=0.8727 | IoU=0.0000 | 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요
5위 | 137.58~140.86초 | 유사도=0.8726 | IoU=0.0000 | 그리고 만들어둔 양념장을 두부 위에 그대로 부어주세요


In [105]:
import numpy as np
from sentence_transformers import util

query_en = "cutting tofu with a knife"
texts = [s["text"] for s in tofu_subtitles]

# E5: 영어 검색어 + 한국어 자막
scores = util.cos_sim(
    e5_model.encode(
        "query: " + query_en,
        convert_to_tensor=True,
        normalize_embeddings=True
    ),
    e5_model.encode(
        ["passage: " + text for text in texts],
        convert_to_tensor=True,
        normalize_embeddings=True
    )
)[0].cpu().numpy()

ground_truth = {
    "start_time": 12.0,
    "end_time": 23.0
}

print("===== E5 영어 검색어 + 한국어 자막 =====")

for rank, idx in enumerate(np.argsort(scores)[::-1][:5], 1):
    s = tofu_subtitles[idx]

    iou = fusion_search.temporal_iou(
        {
            "start_time": s["start_time"],
            "end_time": s["end_time"]
        },
        ground_truth
    )

    print(
        f"{rank}위 | {s['start_time']:.2f}~{s['end_time']:.2f}초 "
        f"| 유사도={scores[idx]:.4f} "
        f"| IoU={iou:.4f} "
        f"| {s['text']}"
    )

===== E5 영어 검색어 + 한국어 자막 =====
1위 | 128.26~130.26초 | 유사도=0.8122 | IoU=0.0000 | 그리고 참기름도 한 숟가락 넣어주세요
2위 | 111.26~116.02초 | 유사도=0.8120 | IoU=0.0000 | 재료를 다 넣었으면 뜨거운 기름 한 숟가락을 부어서 향을 확 끓어 올려주세요
3위 | 53.26~56.50초 | 유사도=0.8057 | IoU=0.0000 | 불을 끄고 뚜껑을 덮어 1분 동안 그대로 두세요
4위 | 137.58~140.86초 | 유사도=0.8021 | IoU=0.0000 | 그리고 만들어둔 양념장을 두부 위에 그대로 부어주세요
5위 | 117.90~119.62초 | 유사도=0.8014 | IoU=0.0000 | 만드는 방법은 정말 간단해요


In [106]:
import numpy as np
from sentence_transformers import util

query_en = "cutting tofu with a knife"
texts = [s["text"] for s in tofu_subtitles]

minilm_model = fusion_search.get_semantic_model()

scores = util.cos_sim(
    minilm_model.encode(
        query_en,
        convert_to_tensor=True
    ),
    minilm_model.encode(
        texts,
        convert_to_tensor=True
    )
)[0].cpu().numpy()

ground_truth = {
    "start_time": 12.0,
    "end_time": 23.0
}

print("===== MiniLM 영어 검색어 + 한국어 자막 =====")

for rank, idx in enumerate(np.argsort(scores)[::-1][:5], 1):
    s = tofu_subtitles[idx]

    iou = fusion_search.temporal_iou(
        {
            "start_time": s["start_time"],
            "end_time": s["end_time"]
        },
        ground_truth
    )

    print(
        f"{rank}위 | {s['start_time']:.2f}~{s['end_time']:.2f}초 "
        f"| 유사도={scores[idx]:.4f} "
        f"| IoU={iou:.4f} "
        f"| {s['text']}"
    )

===== MiniLM 영어 검색어 + 한국어 자막 =====
1위 | 12.88~16.32초 | 유사도=0.3877 | IoU=0.3127 | 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요
2위 | 0.00~4.74초 | 유사도=0.3252 | IoU=0.0000 | 여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면
3위 | 158.74~160.62초 | 유사도=0.3188 | IoU=0.0000 | 생선이나 고기 요리보다 더 맛있어요
4위 | 131.98~135.06초 | 유사도=0.3143 | IoU=0.0000 | 이 양념장은 집에서 어떤 무침에 활용해도 정말 맛있어요
5위 | 111.26~116.02초 | 유사도=0.2939 | IoU=0.0000 | 재료를 다 넣었으면 뜨거운 기름 한 숟가락을 부어서 향을 확 끓어 올려주세요


# MiniLM 한국어 vs 영어 비교

In [107]:
import numpy as np
from sentence_transformers import util

queries = {
    "한국어": "여행용 시트 세제를 보여주는 장면",
    "영어": "showing travel laundry detergent sheets"
}

texts = [s["text"] for s in daiso_subtitles]
model = fusion_search.get_semantic_model()

caption_embeddings = model.encode(
    texts,
    convert_to_tensor=True
)

print("===== MiniLM 한국어 vs 영어: 시트 세제 =====")

for language, query in queries.items():

    scores = util.cos_sim(
        model.encode(query, convert_to_tensor=True),
        caption_embeddings
    )[0].cpu().numpy()

    print(f"\n[{language} 검색어]")

    for rank, idx in enumerate(np.argsort(scores)[::-1][:5], 1):

        s = daiso_subtitles[idx]

        iou = fusion_search.temporal_iou(
            {
                "start_time": s["start_time"],
                "end_time": s["end_time"]
            },
            detergent_ground_truth
        )

        print(
            f"{rank}위 | {s['start_time']:.2f}~{s['end_time']:.2f}초 "
            f"| 유사도={scores[idx]:.4f} "
            f"| IoU={iou:.4f} "
            f"| {s['text']}"
        )

===== MiniLM 한국어 vs 영어: 시트 세제 =====

[한국어 검색어]
1위 | 3.08~4.68초 | 유사도=0.4230 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치
2위 | 22.68~24.48초 | 유사도=0.3273 | IoU=0.0000 | 탱글티즈랑 비슷한 비슷한 요거 두 개
3위 | 20.60~22.68초 | 유사도=0.3216 | IoU=0.0000 | 다이소 립 중에는 제일 잘 쓰게 되더라고요
4위 | 13.72~15.16초 | 유사도=0.3213 | IoU=0.2880 | 빨리 편하게 해주는 시트 세제
5위 | 26.80~29.00초 | 유사도=0.3188 | IoU=0.0000 | 집에 두면 한층 더 귀여워지는 인테리어템

[영어 검색어]
1위 | 4.68~7.28초 | 유사도=0.2520 | IoU=0.0000 | 화장품 진짜 많아서 파우치 두세 개씩 챙겼었는데
2위 | 22.68~24.48초 | 유사도=0.2277 | IoU=0.0000 | 탱글티즈랑 비슷한 비슷한 요거 두 개
3위 | 13.72~15.16초 | 유사도=0.1938 | IoU=0.2880 | 빨리 편하게 해주는 시트 세제
4위 | 10.00~11.36초 | 유사도=0.1668 | IoU=0.0000 | 참기름, 소금, 쌈장
5위 | 3.08~4.68초 | 유사도=0.1587 | IoU=0.0000 | 여행 갈 때 무조건 챙기는 파우치


In [108]:
import json

# 기존 평가 파일에서 검색 사례 확인
baseline_path = video_dir / "fusion_baseline_evaluation.json"

with open(baseline_path, "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

print("===== 기존 평가 데이터 구조 =====")
print("자료형:", type(baseline_data).__name__)

if isinstance(baseline_data, dict):
    print("최상위 항목:", list(baseline_data.keys()))
else:
    print("평가 사례 수:", len(baseline_data))
    print("첫 번째 사례:", baseline_data[0])

print("\n===== 추가한 정답 3개 =====")
for gt in [
    seasoning_ground_truth,
    rice_ground_truth,
    detergent_ground_truth
]:
    print(gt)

===== 기존 평가 데이터 구조 =====
자료형: dict
최상위 항목: ['description', 'evaluation_note', 'deduplication', 'ranking', 'ground_truth', 'results']

===== 추가한 정답 3개 =====
{'video_id': 'kr_tofu_001', 'query_ko': '두부 양념을 준비하는 장면', 'query_en': 'preparing seasoning sauce for tofu', 'start_time': 95.0, 'end_time': 135.0}
{'video_id': 'kr_kitchen_001', 'query_ko': '쌀을 씻는 장면', 'query_en': 'washing rice', 'start_time': 29.0, 'end_time': 32.0}
{'video_id': 'kr_daiso_001', 'query_ko': '여행용 시트 세제를 보여주는 장면', 'query_en': 'showing travel laundry detergent sheets', 'start_time': 13.0, 'end_time': 18.0}


In [109]:
print("===== 기존 3개 검색 사례 정답 =====")

original_gt = baseline_data["ground_truth"]

print("자료형:", type(original_gt).__name__)

if isinstance(original_gt, dict):
    for key, value in original_gt.items():
        print(f"\n{key}")
        print(value)

elif isinstance(original_gt, list):
    for i, item in enumerate(original_gt, 1):
        print(f"\n{i}번째 검색 사례")
        print(item)

===== 기존 3개 검색 사례 정답 =====
자료형: dict

두부
{'start_time': 12.0, 'end_time': 23.0}

주방
{'start_time': 5.96, 'end_time': 11.32}

다이소
{'start_time': 3.08, 'end_time': 8.4}


In [110]:
import numpy as np
from sentence_transformers import util

# 기존 3개 + 추가 3개 검색 사례
test_cases = [
    {
        "name": "두부 자르기",
        "query_ko": "두부를 칼로 자르는 장면",
        "query_en": "cutting tofu with a knife",
        "subtitles": tofu_subtitles,
        "ground_truth": baseline_data["ground_truth"]["두부"]
    },
    {
        "name": "두부 양념 준비",
        "query_ko": seasoning_ground_truth["query_ko"],
        "query_en": seasoning_ground_truth["query_en"],
        "subtitles": tofu_subtitles,
        "ground_truth": seasoning_ground_truth
    },
    {
        "name": "가위로 소스 짜기",
        "query_ko": "가위로 소스를 짜는 장면",
        "query_en": "squeezing sauce with scissors",
        "subtitles": kitchen_subtitles,
        "ground_truth": baseline_data["ground_truth"]["주방"]
    },
    {
        "name": "쌀 씻기",
        "query_ko": rice_ground_truth["query_ko"],
        "query_en": rice_ground_truth["query_en"],
        "subtitles": kitchen_subtitles,
        "ground_truth": rice_ground_truth
    },
    {
        "name": "다이소 파우치",
        "query_ko": "여행용 화장품 파우치를 보여주는 장면",
        "query_en": "showing a travel makeup pouch",
        "subtitles": daiso_subtitles,
        "ground_truth": baseline_data["ground_truth"]["다이소"]
    },
    {
        "name": "다이소 시트 세제",
        "query_ko": detergent_ground_truth["query_ko"],
        "query_en": detergent_ground_truth["query_en"],
        "subtitles": daiso_subtitles,
        "ground_truth": detergent_ground_truth
    }
]

model = fusion_search.get_semantic_model()

print("===== MiniLM 한국어 vs 영어 6개 사례 평가 =====")

for case in test_cases:
    texts = [s["text"] for s in case["subtitles"]]

    print(f"\n[{case['name']}]")

    for language, query in [
        ("한국어", case["query_ko"]),
        ("영어", case["query_en"])
    ]:
        scores = util.cos_sim(
            model.encode(query, convert_to_tensor=True),
            model.encode(texts, convert_to_tensor=True)
        )[0].cpu().numpy()

        ranked_indices = np.argsort(scores)[::-1]

        # 정답과 조금이라도 겹치는 자막의 최초 순위
        correct_rank = None
        best_iou_top5 = 0.0

        for rank, idx in enumerate(ranked_indices, 1):
            s = case["subtitles"][idx]

            iou = fusion_search.temporal_iou(
                {
                    "start_time": s["start_time"],
                    "end_time": s["end_time"]
                },
                case["ground_truth"]
            )

            if iou > 0 and correct_rank is None:
                correct_rank = rank

            if rank <= 5:
                best_iou_top5 = max(best_iou_top5, iou)

        print(
            f"{language} | "
            f"정답 겹침 최초 순위={correct_rank} | "
            f"Top-5 최고 IoU={best_iou_top5:.4f}"
        )

===== MiniLM 한국어 vs 영어 6개 사례 평가 =====

[두부 자르기]
한국어 | 정답 겹침 최초 순위=4 | Top-5 최고 IoU=0.3127
영어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.3127

[두부 양념 준비]
한국어 | 정답 겹침 최초 순위=3 | Top-5 최고 IoU=0.0430
영어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.1190

[가위로 소스 짜기]
한국어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.3955
영어 | 정답 겹침 최초 순위=6 | Top-5 최고 IoU=0.0000

[쌀 씻기]
한국어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.8133
영어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.8133

[다이소 파우치]
한국어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.4887
영어 | 정답 겹침 최초 순위=1 | Top-5 최고 IoU=0.4887

[다이소 시트 세제]
한국어 | 정답 겹침 최초 순위=4 | Top-5 최고 IoU=0.2880
영어 | 정답 겹침 최초 순위=3 | Top-5 최고 IoU=0.2880


- 현재로서는 기존 MiniLM을 유지하고, 한국어와 영어 검색 후보를 함께 활용하는 방향이 가장 합리적이야.

---

- 다만 다음 세 가지를 구분해야 해.
  - 후보 검색: 한국어와 영어를 함께 사용하면 어느 한쪽이 놓친 구간을 확보할 가능성이 있음.

  - 최종 순위: 두 언어의 후보를 합쳤다고 정확한 장면이 자동으로 1위가 되지는 않음.

  - 구간 정확도: 자막 한 문장은 실제 행동보다 짧을 수 있어. 따라서 자막 검색의 IoU만으로 Moment-DETR 전체 성능을 판단하면 안 됨.

---

- 그리고 이번 평가에서 '정답 순위'는 정답과 조금이라도 겹치는 자막이 처음 나타난 순위라는 점도 중요해. 정답을 충분히 정확하게 찾았다는 뜻은 아니야.

# 한국어·영어 후보를 합쳤을 때 실제로 개선되는지 확인

In [111]:
import numpy as np
from sentence_transformers import util

model = fusion_search.get_semantic_model()

print("===== 한국어 + 영어 순위 결합 평가 =====")

for case in test_cases:
    subtitles = case["subtitles"]
    texts = [s["text"] for s in subtitles]

    caption_embs = model.encode(
        texts,
        convert_to_tensor=True
    )

    rankings = []

    for query in [case["query_ko"], case["query_en"]]:
        scores = util.cos_sim(
            model.encode(query, convert_to_tensor=True),
            caption_embs
        )[0].cpu().numpy()

        rankings.append(np.argsort(scores)[::-1].tolist())

    # 한국어 1위 → 영어 1위 → 한국어 2위 → 영어 2위 ...
    merged = []
    seen = set()

    for ko_idx, en_idx in zip(*rankings):
        for idx in [ko_idx, en_idx]:
            if idx not in seen:
                merged.append(idx)
                seen.add(idx)

    top5_ious = []

    for idx in merged[:5]:
        s = subtitles[idx]

        iou = fusion_search.temporal_iou(
            {
                "start_time": s["start_time"],
                "end_time": s["end_time"]
            },
            case["ground_truth"]
        )

        top5_ious.append(iou)

    first_correct = next(
        (
            i + 1
            for i, idx in enumerate(merged)
            if fusion_search.temporal_iou(
                subtitles[idx],
                case["ground_truth"]
            ) > 0
        ),
        None
    )

    print(
        f"{case['name']} | "
        f"최초 정답 겹침 순위={first_correct} | "
        f"Top-5 최고 IoU={max(top5_ious):.4f}"
    )

===== 한국어 + 영어 순위 결합 평가 =====
두부 자르기 | 최초 정답 겹침 순위=2 | Top-5 최고 IoU=0.3127
두부 양념 준비 | 최초 정답 겹침 순위=2 | Top-5 최고 IoU=0.1190
가위로 소스 짜기 | 최초 정답 겹침 순위=1 | Top-5 최고 IoU=0.3955
쌀 씻기 | 최초 정답 겹침 순위=1 | Top-5 최고 IoU=0.8133
다이소 파우치 | 최초 정답 겹침 순위=1 | Top-5 최고 IoU=0.4887
다이소 시트 세제 | 최초 정답 겹침 순위=5 | Top-5 최고 IoU=0.2880


- 좋은 점: 한국어만 사용할 때 놓칠 수 있는 두부 자르기와 양념 준비 장면을 영어 후보 덕분에 2위에서 찾았어.
- 아쉬운 점: 시트 세제는 영어만 사용하면 3위였는데, 두 언어를 번갈아 합치면서 오히려 5위로 밀렸어.

=> 따라서 현재의 단순 교차 결합 방식이 항상 최선은 아니야.

---

<지금까지 비교한 방법>
- 기존 Moment-DETR Hybrid
- MiniLM 한국어 자막 검색
- MiniLM 영어 자막 검색
- E5 다국어 자막 검색
- 의미 유사도 + 글자 일치 가중치
- 한국어·영어 후보 교차 결합

---

- 현재로서는 기존 Moment-DETR과 MiniLM을 유지하면서, 후보 순위를 결정하는 방법을 개선하는 방향이 가장 합리적이야.
- 특히 중요한 건 한국어와 영어 중 어느 언어가 더 정확한지 미리 알 수 없다는 점이야.

- 그래서 다음 실험은 단순히 번갈아 배치하는 방식이 아니라, 각 언어에서 높은 순위를 얻은 자막을 함께 고려하는 RRF(Reciprocal Rank Fusion) 방식으로 비교해 보자.

- RRF는 유사도 점수를 직접 더하지 않고 각 검색 방식에서의 순위를 이용하는 방법이야. 언어별 유사도 점수의 크기가 달라도 결합할 수 있다는 장점이 있어.
- 다만 이전에 영상 구간 후보에 RRF를 적용했을 때 개선이 제한적이었던 것처럼, 이번에도 성능 향상을 보장하지는 않아.

# 6개 사례에서 RRF 평가

In [112]:
import numpy as np
from sentence_transformers import util

model = fusion_search.get_semantic_model()

print("===== MiniLM 한국어 + 영어 RRF 결합 평가 =====")

for case in test_cases:
    subtitles = case["subtitles"]
    texts = [s["text"] for s in subtitles]

    caption_embeddings = model.encode(
        texts,
        convert_to_tensor=True
    )

    rank_arrays = []

    for query in [case["query_ko"], case["query_en"]]:
        scores = util.cos_sim(
            model.encode(query, convert_to_tensor=True),
            caption_embeddings
        )[0].cpu().numpy()

        order = np.argsort(scores)[::-1]

        ranks = np.empty(len(order), dtype=int)
        ranks[order] = np.arange(1, len(order) + 1)

        rank_arrays.append(ranks)

    # RRF: 각 언어의 순위를 이용해 점수 계산
    k = 60

    rrf_scores = (
        1 / (k + rank_arrays[0])
        + 1 / (k + rank_arrays[1])
    )

    order = np.argsort(rrf_scores)[::-1]

    top5_ious = []

    for idx in order[:5]:
        s = subtitles[idx]

        iou = fusion_search.temporal_iou(
            s,
            case["ground_truth"]
        )

        top5_ious.append(iou)

    first_correct = next(
        (
            rank
            for rank, idx in enumerate(order, 1)
            if fusion_search.temporal_iou(
                subtitles[idx],
                case["ground_truth"]
            ) > 0
        ),
        None
    )

    print(
        f"{case['name']} | "
        f"최초 정답 겹침 순위={first_correct} | "
        f"Top-5 최고 IoU={max(top5_ious):.4f}"
    )

===== MiniLM 한국어 + 영어 RRF 결합 평가 =====
두부 자르기 | 최초 정답 겹침 순위=1 | Top-5 최고 IoU=0.3127
두부 양념 준비 | 최초 정답 겹침 순위=3 | Top-5 최고 IoU=0.0570
가위로 소스 짜기 | 최초 정답 겹침 순위=4 | Top-5 최고 IoU=0.3955
쌀 씻기 | 최초 정답 겹침 순위=1 | Top-5 최고 IoU=0.8133
다이소 파우치 | 최초 정답 겹침 순위=1 | Top-5 최고 IoU=0.4887
다이소 시트 세제 | 최초 정답 겹침 순위=3 | Top-5 최고 IoU=0.2880


- RRF가 개선한 경우도 있지만, 특히 가위로 소스 짜기에서는 한국어 단독 1위였던 정답이 4위로 밀렸어.
- 또한 두부 양념 준비에서도 영어 단독은 Top-5 최고 IoU가 0.1190이었는데, RRF는 0.0570이었어.
- 따라서 RRF를 현재 Fusion에 바로 적용하는 건 추천하지 않아.

---

- Moment-DETR은 유지: 영상 자체에서 관련 구간을 예측하는 역할
- 기존 MiniLM은 유지: 한국어·영어 자막에서 추가 후보를 찾는 역할
- E5는 보류: 지금까지의 테스트에서 교체할 근거가 없음
- RRF와 글자 일치 가중치는 보류: 일부 사례에만 효과가 있고 일관된 개선이 확인되지 않음

---

- 그리고 현재 가장 중요한 문제는 정확한 검색 후보를 찾았을 때 그 후보를 어떻게 1위로 올릴 것인가야.
- 다만 이번 비교는 단일 자막 검색 실험이고, 기존 Fusion은 자막 여러 개를 묶고 Moment-DETR 결과까지 결합하므로 둘을 동일한 성능으로 해석하면 안 돼.

In [113]:
import json

evaluation_files = [
    video_dir / "fusion_baseline_evaluation.json",
    video_dir / "seasoning_evaluation.json",
    video_dir / "rice_washing_evaluation.json",
    video_dir / "detergent_evaluation.json"
]

for path in evaluation_files:
    print(
        path.name,
        "| 존재 여부:",
        path.exists()
    )

fusion_baseline_evaluation.json | 존재 여부: True
seasoning_evaluation.json | 존재 여부: True
rice_washing_evaluation.json | 존재 여부: True
detergent_evaluation.json | 존재 여부: True


In [114]:
import json

detergent_evaluation = {
    "ground_truth": detergent_ground_truth,

    "hybrid_results": [
        {
            "rank": i,
            "start_time": r["start_time"],
            "end_time": r["end_time"],
            "iou": fusion_search.temporal_iou(
                r, detergent_ground_truth
            )
        }
        for i, r in enumerate(detergent_hybrid_results, 1)
    ],

    "fusion_results": [
        {
            "rank": i,
            "source": r["source"],
            "language": r.get("language"),
            "start_time": r["start_time"],
            "end_time": r["end_time"],
            "iou": fusion_search.temporal_iou(
                r, detergent_ground_truth
            )
        }
        for i, r in enumerate(detergent_fusion_results, 1)
    ]
}

save_path = video_dir / "detergent_evaluation.json"

save_path.write_text(
    json.dumps(
        detergent_evaluation,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("시트 세제 평가 저장 완료:", save_path)
print("Hybrid 후보:", len(detergent_evaluation["hybrid_results"]))
print("Fusion 후보:", len(detergent_evaluation["fusion_results"]))
print("파일 존재 여부:", save_path.exists())

시트 세제 평가 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/detergent_evaluation.json
Hybrid 후보: 3
Fusion 후보: 5
파일 존재 여부: True


In [115]:
print("===== 기존 3개 사례 평가 결과 구조 =====")

results = baseline_data["results"]

print("자료형:", type(results).__name__)

if isinstance(results, dict):
    print("항목:", list(results.keys()))

    for key, value in results.items():
        print(f"\n[{key}]")
        print("자료형:", type(value).__name__)

        if isinstance(value, dict):
            print("내부 항목:", list(value.keys()))
            for subkey, subvalue in value.items():
                if isinstance(subvalue, list):
                    print(f"  {subkey}: 리스트 {len(subvalue)}개")
                    if subvalue:
                        print("  첫 번째 결과:", subvalue[0])
                else:
                    print(f"  {subkey}:", subvalue)

        elif isinstance(value, list):
            print("결과 개수:", len(value))
            if value:
                print("첫 번째 결과:", value[0])

elif isinstance(results, list):
    print("결과 개수:", len(results))
    if results:
        print("첫 번째 결과:", results[0])

===== 기존 3개 사례 평가 결과 구조 =====
자료형: dict
항목: ['두부', '주방', '다이소']

[두부]
자료형: list
결과 개수: 5
첫 번째 결과: {'rank': 1, 'source': 'hybrid', 'language': None, 'start_time': 98.5, 'end_time': 118.9725, 'iou': 0.0}

[주방]
자료형: list
결과 개수: 5
첫 번째 결과: {'rank': 1, 'source': 'hybrid', 'language': None, 'start_time': 2.5, 'end_time': 10.2804, 'iou': 0.48984126984126986}

[다이소]
자료형: list
결과 개수: 4
첫 번째 결과: {'rank': 1, 'source': 'hybrid', 'language': None, 'start_time': 14.5, 'end_time': 25.0135, 'iou': 0.0}


In [116]:
import json
import pandas as pd

# ① 기존 3개 + 추가 3개 평가 결과 불러오기
cases = []

for name in ["두부", "주방", "다이소"]:
    cases.append({
        "name": name,
        "fusion": baseline_data["results"][name]
    })

for name, filename in [
    ("두부 양념 준비", "seasoning_evaluation.json"),
    ("쌀 씻기", "rice_washing_evaluation.json"),
    ("시트 세제", "detergent_evaluation.json")
]:
    with open(video_dir / filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    cases.append({
        "name": name,
        "hybrid": data["hybrid_results"],
        "fusion": data["fusion_results"]
    })

# ② 성능 비교
rows = []

for case in cases:
    fusion = case["fusion"]

    # 기존 3개 사례에는 별도 Hybrid 결과가 없으므로
    # Fusion에 포함된 Hybrid 후보만 추출
    hybrid = case.get("hybrid")

    if hybrid is None:
        hybrid = [
            r for r in fusion
            if r.get("source") == "hybrid"
        ]

    rows.append({
        "검색 사례": case["name"],
        "Hybrid Top-1": round(hybrid[0]["iou"], 4) if hybrid else None,
        "Fusion Top-1": round(fusion[0]["iou"], 4) if fusion else None,
        "Hybrid 최고 IoU": round(max(r["iou"] for r in hybrid), 4) if hybrid else None,
        "Fusion 최고 IoU": round(max(r["iou"] for r in fusion), 4) if fusion else None
    })

df = pd.DataFrame(rows)

print("===== Hybrid vs Fusion 종합 평가 =====")
print(df.to_string(index=False))

print("\n===== 평균 성능 =====")
print(df.iloc[:, 1:].mean().round(4))

===== Hybrid vs Fusion 종합 평가 =====
   검색 사례  Hybrid Top-1  Fusion Top-1  Hybrid 최고 IoU  Fusion 최고 IoU
      두부        0.0000        0.0000         0.0000         0.7436
      주방        0.4898        0.4898         0.4898         0.4898
     다이소        0.0000        0.0000         0.0000         0.6646
두부 양념 준비        0.5167        0.5167         0.5167         0.5167
    쌀 씻기        0.2147        0.2147         0.2147         0.8133
   시트 세제        0.3040        0.3040         0.3040         0.3040

===== 평균 성능 =====
Hybrid Top-1     0.2542
Fusion Top-1     0.2542
Hybrid 최고 IoU    0.2542
Fusion 최고 IoU    0.5887
dtype: float64


# 현재 Fusion이 어떻게 순위를 정하는지 확인

In [117]:
import inspect

print("===== 현재 Fusion 함수 코드 =====")

source = inspect.getsource(
    fusion_search.predict_moments_fusion
)

print(source)

===== 현재 Fusion 함수 코드 =====
def predict_moments_fusion(
    video_id, feature_path, subtitles,
    korean_query, english_query,
    video_duration, subtitle_json_path=None, top_k=5
):
    visual = [
        dict(r, source="hybrid", hybrid_source=r.get("source"))
        for r in predict_moments_hybrid(
            video_id=video_id,
            video_feature_path=feature_path,
            query=english_query,
            subtitle_json_path=subtitle_json_path,
            video_duration=video_duration,
            top_k=top_k
        )
    ]

    # 다국어 의미 검색: 한국어·영어 검색어 모두 사용
    subtitle = []
    if subtitles:
        subtitle = get_multilingual_subtitle_candidates(
            korean_query=korean_query,
            english_query=english_query,
            subtitles=subtitles,
            model=get_semantic_model(),
            top_k=top_k
        )

    ko = [r for r in subtitle if r["language"] == "ko"]
    en = [r for r in subtitle if r["language"] == "en"]

    # 첫 후보부터 세 검색 방식의 순서

In [118]:
print("===== 시트 세제 Fusion 후보 상세 정보 =====")

for i, r in enumerate(detergent_fusion_results, 1):
    print(f"\n[{i}위]")
    print("출처:", r.get("source"))
    print("언어:", r.get("language"))
    print("시작:", r.get("start_time"))
    print("종료:", r.get("end_time"))
    print("score:", r.get("score"))
    print("subtitle_score:", r.get("subtitle_score"))
    print("hybrid_source:", r.get("hybrid_source"))
    print("전체 항목:", list(r.keys()))

===== 시트 세제 Fusion 후보 상세 정보 =====

[1위]
출처: hybrid
언어: None
시작: 14.5
종료: 24.5127
score: 0.9745
subtitle_score: 0.0
hybrid_source: visual
전체 항목: ['video_id', 'query', 'start_time', 'end_time', 'score', 'source', 'hybrid_source', 'visual_score', 'subtitle_score', 'language']

[2위]
출처: subtitle
언어: ko
시작: 3.08
종료: 4.68
score: None
subtitle_score: 0.423
hybrid_source: None
전체 항목: ['video_id', 'query', 'start_time', 'end_time', 'score', 'source', 'hybrid_source', 'visual_score', 'subtitle_score', 'language']

[3위]
출처: subtitle
언어: en
시작: 22.68
종료: 29.0
score: None
subtitle_score: 0.3968
hybrid_source: None
전체 항목: ['video_id', 'query', 'start_time', 'end_time', 'score', 'source', 'hybrid_source', 'visual_score', 'subtitle_score', 'language']

[4위]
출처: subtitle
언어: ko
시작: 3.08
종료: 8.4
score: None
subtitle_score: 0.4087
hybrid_source: None
전체 항목: ['video_id', 'query', 'start_time', 'end_time', 'score', 'source', 'hybrid_source', 'visual_score', 'subtitle_score', 'language']

[5위]
출처: subtitle


In [119]:
print("===== 시트 세제: Hybrid와 자막 후보 시간 겹침 =====")

hybrid_candidates = [
    r for r in detergent_fusion_results
    if r["source"] == "hybrid"
]

subtitle_candidates = [
    r for r in detergent_fusion_results
    if r["source"] == "subtitle"
]

for h in hybrid_candidates:
    print(
        f"\nHybrid: {h['start_time']:.2f}~{h['end_time']:.2f}초"
    )

    for s in subtitle_candidates:
        overlap = fusion_search.temporal_iou(h, s)

        print(
            f"  자막({s['language']}) "
            f"{s['start_time']:.2f}~{s['end_time']:.2f}초 "
            f"| 후보 간 IoU={overlap:.4f}"
        )

===== 시트 세제: Hybrid와 자막 후보 시간 겹침 =====

Hybrid: 14.50~24.51초
  자막(ko) 3.08~4.68초 | 후보 간 IoU=0.0000
  자막(en) 22.68~29.00초 | 후보 간 IoU=0.1264
  자막(ko) 3.08~8.40초 | 후보 간 IoU=0.0000
  자막(en) 20.60~24.48초 | 후보 간 IoU=0.3875


- Moment-DETR: 기존 학습 모델 유지
- MiniLM: 한국어·영어 자막 검색용으로 유지
- E5: 교체 보류
- RRF·글자 일치 가중치: 일관된 개선이 확인되지 않아 보류
- 시간 겹침 기반 재정렬: 아직 채택할 근거 부족

---

- 따라서 지금은 새로운 가중치를 계속 시도하기보다는, 검색 후보가 왜 누락되는지부터 확인하는 게 더 중요해.
- 특히 시트 세제 정답 자막은 한국어 의미 검색에서 9위에 있었지. 그런데 Fusion은 한국어 후보를 일부만 가져와 상위 5개 결과로 만들고 있어.

In [120]:
import inspect

print("===== 자막 후보 생성 함수 =====")

print(
    inspect.getsource(
        fusion_search.get_multilingual_subtitle_candidates
    )
)

===== 자막 후보 생성 함수 =====
def get_multilingual_subtitle_candidates(
    korean_query,
    english_query,
    subtitles,
    model,
    top_k=5
):
    candidates = []

    for language, query in [
        ("ko", korean_query),
        ("en", english_query)
    ]:
        results = get_semantic_subtitle_candidates(
            query=query,
            subtitles=subtitles,
            model=model,
            top_k=top_k
        )

        for r in results:
            candidates.append({
                **r,
                "language": language,
                "query_used": query
            })

    return candidates



In [121]:
print("===== 시트 세제: 후보 수에 따른 정답 포함 여부 =====")

for k in [5, 10]:
    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=detergent_ground_truth["query_ko"],
        english_query=detergent_ground_truth["query_en"],
        subtitles=daiso_subtitles,
        model=fusion_search.get_semantic_model(),
        top_k=k
    )

    print(f"\n[언어별 Top-{k}]")

    for language in ["ko", "en"]:
        group = [r for r in candidates if r["language"] == language]

        matches = [
            (i, r)
            for i, r in enumerate(group, 1)
            if fusion_search.temporal_iou(
                r, detergent_ground_truth
            ) > 0
        ]

        print(
            f"{language} | 후보 {len(group)}개 | "
            f"정답과 겹치는 후보 {len(matches)}개"
        )

        for rank, r in matches:
            print(
                f"  {rank}위 | "
                f"{r['start_time']:.2f}~{r['end_time']:.2f}초 "
                f"| IoU={fusion_search.temporal_iou(r, detergent_ground_truth):.4f}"
            )

===== 시트 세제: 후보 수에 따른 정답 포함 여부 =====

[언어별 Top-5]
ko | 후보 5개 | 정답과 겹치는 후보 0개
en | 후보 5개 | 정답과 겹치는 후보 0개

[언어별 Top-10]
ko | 후보 10개 | 정답과 겹치는 후보 1개
  9위 | 15.16~19.08초 | IoU=0.4671
en | 후보 10개 | 정답과 겹치는 후보 0개


In [122]:
all_candidates = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=detergent_ground_truth["query_ko"],
    english_query=detergent_ground_truth["query_en"],
    subtitles=daiso_subtitles,
    model=fusion_search.get_semantic_model(),
    top_k=100
)

print("===== 시트 세제 전체 후보 순위 분석 =====")

for language in ["ko", "en"]:
    group = [r for r in all_candidates if r["language"] == language]

    overlapping = []

    for rank, r in enumerate(group, 1):
        iou = fusion_search.temporal_iou(r, detergent_ground_truth)

        if iou > 0:
            overlapping.append((rank, r, iou))

    print(f"\n[{language}] 전체 후보 {len(group)}개")
    print("정답과 겹치는 후보 수:", len(overlapping))

    for rank, r, iou in overlapping[:10]:
        print(
            f"{rank}위 | "
            f"{r['start_time']:.2f}~{r['end_time']:.2f}초 "
            f"| IoU={iou:.4f}"
        )

===== 시트 세제 전체 후보 순위 분석 =====

[ko] 전체 후보 54개
정답과 겹치는 후보 수: 15
9위 | 15.16~19.08초 | IoU=0.4671
14위 | 13.72~15.16초 | IoU=0.2880
22위 | 15.16~17.08초 | IoU=0.3840
23위 | 17.08~19.08초 | IoU=0.1513
24위 | 13.72~17.08초 | IoU=0.6720
27위 | 13.72~19.08초 | IoU=0.7039
34위 | 15.16~20.60초 | IoU=0.3737
38위 | 17.08~20.60초 | IoU=0.1211
41위 | 11.36~17.08초 | IoU=0.6145
44위 | 17.08~22.68초 | IoU=0.0950

[en] 전체 후보 54개
정답과 겹치는 후보 수: 15
15위 | 13.72~19.08초 | IoU=0.7039
16위 | 13.72~15.16초 | IoU=0.2880
20위 | 11.36~15.16초 | IoU=0.3253
23위 | 17.08~19.08초 | IoU=0.1513
24위 | 15.16~19.08초 | IoU=0.4671
25위 | 10.00~15.16초 | IoU=0.2700
26위 | 17.08~22.68초 | IoU=0.0950
29위 | 15.16~20.60초 | IoU=0.3737
30위 | 17.08~20.60초 | IoU=0.1211
31위 | 8.40~13.72초 | IoU=0.0750


In [123]:
print("===== 시트 세제 관련 자막 원문 =====")

for s in daiso_subtitles:
    if s["start_time"] < 20 and s["end_time"] > 12:
        print(
            f"{s['start_time']:.2f}~{s['end_time']:.2f}초 | "
            f"{s['text']}"
        )

print("\n===== 실제 정답 구간 =====")
print(
    detergent_ground_truth["start_time"],
    "~",
    detergent_ground_truth["end_time"],
    "초"
)

===== 시트 세제 관련 자막 원문 =====
11.36~13.72초 | 쭈왕까지 수량씩 들어있어서 먹기 너무 편해요
13.72~15.16초 | 빨리 편하게 해주는 시트 세제
15.16~17.08초 | 톡 뜯어 여행 갈 때 들고 가기도 좋고
17.08~19.08초 | 섬유인지 안 넣어도 향기가 대박이에요
19.08~20.60초 | 피랑 진짜 비슷한 젤리판

===== 실제 정답 구간 =====
13.0 ~ 18.0 초


In [124]:
from sentence_transformers import util

model = fusion_search.get_semantic_model()

query_ko = "여행용 시트 세제를 보여주는 장면"
query_en = "showing travel laundry detergent sheets"

captions = {
    "제품명만": "빨리 편하게 해주는 시트 세제",
    "여행 설명만": "톡 뜯어 여행 갈 때 들고 가기도 좋고",
    "제품명 + 여행 설명": (
        "빨리 편하게 해주는 시트 세제 "
        "톡 뜯어 여행 갈 때 들고 가기도 좋고"
    ),
    "제품명 + 전체 설명": (
        "빨리 편하게 해주는 시트 세제 "
        "톡 뜯어 여행 갈 때 들고 가기도 좋고 "
        "섬유인지 안 넣어도 향기가 대박이에요"
    ),
    "오답 파우치": "여행 갈 때 무조건 챙기는 파우치"
}

texts = list(captions.values())

caption_embeddings = model.encode(
    texts,
    convert_to_tensor=True
)

print("===== 자막 연결에 따른 의미 유사도 비교 =====")

for language, query in [
    ("한국어", query_ko),
    ("영어", query_en)
]:
    scores = util.cos_sim(
        model.encode(query, convert_to_tensor=True),
        caption_embeddings
    )[0]

    print(f"\n[{language} 검색어]")

    for name, score in zip(captions.keys(), scores):
        print(f"{name}: {score.item():.4f}")

===== 자막 연결에 따른 의미 유사도 비교 =====

[한국어 검색어]
제품명만: 0.3213
여행 설명만: 0.3086
제품명 + 여행 설명: 0.3006
제품명 + 전체 설명: 0.2986
오답 파우치: 0.4230

[영어 검색어]
제품명만: 0.1938
여행 설명만: 0.0545
제품명 + 여행 설명: 0.0680
제품명 + 전체 설명: 0.1957
오답 파우치: 0.1587


In [125]:
import json
from pathlib import Path

summary = {
    "project": "PinPoint",
    "evaluation_type": "Korean video retrieval preliminary evaluation",
    "video_count": 3,
    "query_count": 6,
    "baseline": {
        "model": "Moment-DETR Hybrid",
        "mean_top1_iou": 0.2542,
        "mean_best_iou_reference": 0.2542
    },
    "fusion": {
        "model": "Moment-DETR + MiniLM KO/EN subtitle retrieval",
        "mean_top1_iou": 0.2542,
        "mean_best_iou_top5": 0.5887
    },
    "limitations": [
        "Ground truth timestamps are approximate.",
        "Only three videos and six queries were evaluated.",
        "Hybrid best-IoU reference is not a fully independent evaluation.",
        "No improvement in Top-1 IoU was observed.",
        "English queries were manually provided."
    ]
}

save_path = video_dir / "pinpoint_ai_experiment_summary.json"

save_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("AI 실험 요약 저장 완료:", save_path)
print("파일 존재 여부:", save_path.exists())

AI 실험 요약 저장 완료: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/pinpoint_ai_experiment_summary.json
파일 존재 여부: True


# AI_HUB_한국어 텍스트-비디오-사운드 데이터

In [126]:
import tarfile
from pathlib import Path

# AI-Hub 한국어 학습 데이터 위치
training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

# 한글 파일명 인코딩 문제를 피하기 위해 자동 탐색
tar_files = list(training_dir.glob("*.tar"))

print("===== AI-Hub 데이터 확인 =====")
print("TAR 파일 개수:", len(tar_files))

for tar_path in tar_files:
    print("\n파일명:", tar_path.name)
    print("크기:", round(tar_path.stat().st_size / 1024**2, 2), "MB")

    with tarfile.open(tar_path, "r:*") as tar:
        members = [m for m in tar.getmembers() if m.isfile()]

        json_files = [
            m for m in members
            if m.name.lower().endswith(".json")
        ]

        print("전체 파일 개수:", len(members))
        print("JSON 파일 개수:", len(json_files))

        print("\n===== 처음 10개 파일 =====")
        for member in members[:10]:
            print(member.name)

===== AI-Hub 데이터 확인 =====
TAR 파일 개수: 2

파일명: TL_12.음식요리.tar
크기: 16.6 MB
전체 파일 개수: 1
JSON 파일 개수: 0

===== 처음 10개 파일 =====
017.한국어_텍스트-비디오-사운드_데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_12._음식요리.zip.part0

파일명: VL_12.음식요리.tar
크기: 2.04 MB
전체 파일 개수: 1
JSON 파일 개수: 0

===== 처음 10개 파일 =====
017.한국어_텍스트-비디오-사운드_데이터/3.개방데이터/1.데이터/Validation/02.라벨링데이터/VL_12._음식요리.zip.part0


In [127]:
import tarfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

tar_path = next(training_dir.glob("*.tar"))

with tarfile.open(tar_path, "r:*") as tar:
    files = [m for m in tar.getmembers() if m.isfile()]

    for member in files:
        print("===== TAR 내부 파일 정보 =====")
        print("파일명:", Path(member.name).name)
        print("크기(MB):", round(member.size / 1024**2, 2))

        # 압축을 풀지 않고 파일 시작 부분만 확인
        f = tar.extractfile(member)
        header = f.read(8)

        print("파일 시작 바이트:", header)
        print("ZIP 형식으로 시작:", header.startswith(b"PK"))

===== TAR 내부 파일 정보 =====
파일명: TL_12._음식요리.zip.part0
크기(MB): 16.6
파일 시작 바이트: b'PK\x03\x04\x14\x00\x08\x08'
ZIP 형식으로 시작: True


In [128]:
import tarfile
import zipfile
import io
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

tar_path = next(training_dir.glob("*.tar"))

with tarfile.open(tar_path, "r:*") as tar:
    member = next(m for m in tar.getmembers() if m.isfile())

    # TAR 내부의 ZIP 데이터를 메모리로 읽기
    zip_bytes = tar.extractfile(member).read()

print("ZIP 데이터 크기:", round(len(zip_bytes) / 1024**2, 2), "MB")

try:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        names = z.namelist()
        json_names = [
            name for name in names
            if name.lower().endswith(".json")
        ]

        print("✅ ZIP 파일 정상적으로 열림!")
        print("전체 내부 파일 수:", len(names))
        print("JSON 파일 수:", len(json_names))

        print("\n===== 처음 10개 파일 =====")
        for name in names[:10]:
            print(name)

except zipfile.BadZipFile:
    print("⚠️ ZIP 파일을 완전히 열 수 없습니다.")
    print("분할 압축의 다른 부분이 필요한지 확인해야 합니다.")

ZIP 데이터 크기: 16.6 MB
✅ ZIP 파일 정상적으로 열림!
전체 내부 파일 수: 17899
JSON 파일 수: 17899

===== 처음 10개 파일 =====
/MYL_00019.json
/MYL_00010.json
/MYL_00007.json
/MYL_00028.json
/MYL_00002.json
/MYL_00001.json
/MYL_00026.json
/MYL_00008.json
/MYL_00014.json
/MYL_00021.json


In [129]:
import json
import io
import tarfile
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

tar_path = next(training_dir.glob("*.tar"))

with tarfile.open(tar_path, "r:*") as tar:
    member = next(m for m in tar.getmembers() if m.isfile())
    zip_bytes = tar.extractfile(member).read()

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    json_name = next(
        name for name in z.namelist()
        if name.lower().endswith(".json")
    )

    data = json.loads(z.read(json_name).decode("utf-8-sig"))

print("===== 확인한 JSON 파일 =====")
print(json_name)

print("\n===== 최상위 항목 =====")
print(list(data.keys()))

print("\n===== 실제 데이터 예시 =====")
print(json.dumps(data, ensure_ascii=False, indent=2)[:5000])

===== 확인한 JSON 파일 =====
/MYL_00019.json

===== 최상위 항목 =====
['metadata', 'video', 'summary', 'image']

===== 실제 데이터 예시 =====
{
  "metadata": {
    "filename": "MYL_00019",
    "category": "음식/요리",
    "sub_category": 2,
    "sound": [
      2
    ],
    "quality": "HD",
    "length": 27.894,
    "format": "MP4",
    "date": "2023-07-19",
    "license": "개인 방송",
    "agreement_score": 2
  },
  "video": {
    "speakers_info": [
      {
        "speaker_id": "1",
        "age": "20대",
        "gender": "남성"
      }
    ],
    "term": [
      {
        "speaker_id": "1",
        "transcription": "아 그리고 버섯류 없는 것도 너무 아쉬워.",
        "sound_type": [
          2
        ],
        "start": 0.0,
        "end": 5.497
      },
      {
        "speaker_id": "1",
        "transcription": "음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다.",
        "sound_type": [
          2
        ],
        "start": 5.507,
        "end": 12.184
      },
      {
        "speaker_id": "1",
        "transcription": "음 기본 오리지

In [130]:
import json
import io
import tarfile
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

tar_path = next(training_dir.glob("*.tar"))

stats = {
    "total_json": 0,
    "with_transcription": 0,
    "without_transcription": 0,
    "total_terms": 0,
    "valid_timed_terms": 0,
    "with_summary": 0
}

with tarfile.open(tar_path, "r:*") as tar:
    member = next(m for m in tar.getmembers() if m.isfile())
    zip_bytes = tar.extractfile(member).read()

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    json_names = [
        name for name in z.namelist()
        if name.lower().endswith(".json")
    ]

    for name in json_names:
        try:
            data = json.loads(z.read(name).decode("utf-8-sig"))
        except (UnicodeError, json.JSONDecodeError):
            continue

        stats["total_json"] += 1

        terms = data.get("video", {}).get("term", [])
        if not isinstance(terms, list):
            terms = []

        valid_texts = [
            term for term in terms
            if isinstance(term, dict)
            and str(term.get("transcription") or "").strip()
        ]

        if valid_texts:
            stats["with_transcription"] += 1
        else:
            stats["without_transcription"] += 1

        stats["total_terms"] += len(valid_texts)

        for term in valid_texts:
            start = term.get("start")
            end = term.get("end")

            if (
                isinstance(start, (int, float))
                and isinstance(end, (int, float))
                and 0 <= start < end
            ):
                stats["valid_timed_terms"] += 1

        if str(data.get("summary") or "").strip():
            stats["with_summary"] += 1

print("===== AI-Hub 전체 자막 데이터 통계 =====")
for key, value in stats.items():
    print(f"{key}: {value:,}")

if stats["total_json"]:
    print(
        "자막 보유 영상 비율:",
        f"{stats['with_transcription'] / stats['total_json'] * 100:.2f}%"
    )

===== AI-Hub 전체 자막 데이터 통계 =====
total_json: 17,899
with_transcription: 17,899
without_transcription: 0
total_terms: 100,690
valid_timed_terms: 100,688
with_summary: 17,899
자막 보유 영상 비율: 100.00%


In [131]:
import json
import io
import tarfile
import zipfile
import statistics
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

tar_path = next(training_dir.glob("*.tar"))

texts = []

with tarfile.open(tar_path, "r:*") as tar:
    member = next(m for m in tar.getmembers() if m.isfile())
    zip_bytes = tar.extractfile(member).read()

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        try:
            data = json.loads(z.read(name).decode("utf-8-sig"))
        except (UnicodeError, json.JSONDecodeError):
            continue

        for term in data.get("video", {}).get("term", []):
            if not isinstance(term, dict):
                continue

            text = str(term.get("transcription") or "").strip()

            if text:
                texts.append(text)

lengths = [len(text) for text in texts]

print("===== 한국어 발화문 길이 분석 =====")
print("전체 발화문:", len(texts))
print("평균 글자 수:", round(statistics.mean(lengths), 2))
print("중앙값 글자 수:", statistics.median(lengths))

for threshold in [5, 10, 20, 30]:
    count = sum(len(text) >= threshold for text in texts)
    print(f"{threshold}글자 이상: {count:,}개 ({count / len(texts) * 100:.2f}%)")

print("\n===== 실제 발화문 예시 =====")
for text in texts[:10]:
    print("-", text)

===== 한국어 발화문 길이 분석 =====
전체 발화문: 100690
평균 글자 수: 28.98
중앙값 글자 수: 24.0
5글자 이상: 98,346개 (97.67%)
10글자 이상: 90,988개 (90.36%)
20글자 이상: 62,439개 (62.01%)
30글자 이상: 37,895개 (37.64%)

===== 실제 발화문 예시 =====
- 아 그리고 버섯류 없는 것도 너무 아쉬워.
- 음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다.
- 음 기본 오리지널보다 마라맛이 싹 배어있네.
- 아 이거 마라탕이라 생각했어.
- 떡을 너무 안 먹었다.
- 떡볶이지.
- 음 어 근데 먹다 보니까 맵다.
- 음 매워요.
- 이거나 한 입 먹고 끝내야 했다.
- 야 이거 기름 그냥 그거다 그거 행주에 그거다.


In [132]:
import json
import io
import zipfile
import random

random.seed(42)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    json_names = [
        name for name in z.namelist()
        if name.lower().endswith(".json")
    ]

    sampled_names = random.sample(json_names, 8)

    for i, name in enumerate(sampled_names, 1):
        data = json.loads(z.read(name).decode("utf-8-sig"))

        summary = data.get("summary", "")
        terms = data.get("video", {}).get("term", [])

        print(f"\n===== 영상 {i}: {name} =====")
        print("요약:", summary)
        print("발화 구간 수:", len(terms))

        for term in terms[:3]:
            print(
                f"  [{term.get('start')}~{term.get('end')}초]",
                term.get("transcription", "")
            )


===== 영상 1: /MYL_03691.json =====
요약: 그릇에 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼을 넣고 살짝 섞어주면 드레싱소스는 완성이 된다.
발화 구간 수: 5
  [0.0~21.804초] 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼 이 블루베리 잼이 들어가믄 또 기가 막히게 맛있거든요.
  [21.814~27.03초] 이렇게 인자 섞어 주면은 드레싱은 완성이 되는 겁니다.
  [27.04~30.825초] 이 블루베리가 톡톡 터지면서 달콤하거든요.

===== 영상 2: /MYL_00843.json =====
요약: 밥 한 공기에 참기름과 식초를 넣어 양념을 한다.
발화 구간 수: 7
  [0.0~2.75초] 힘을 주면서 동그랗게 인자 해 줍니다.
  [2.76~5.929초] 이렇게 잠시 둔 다음에 인자 밥에다가 양념을 하겠습니다.
  [5.939~7.776초] 오늘은 인자 밥 한 공기 준비했는데요.

===== 영상 3: /MYL_09092.json =====
요약: 남성은 커피창고에서 7월에 출시될 블랜드인 코소롱 블랜드를 두 번째로 소개하고 있다.
발화 구간 수: 3
  [0.0~2.05초] 이런 생각도 해봅니다.
  [2.05~6.426초] 지금 이거 먹어보니까 2탄, 3탄이 좀 더 기대가 됩니다.
  [6.43~18.066초] 두 번째로 소개해드릴 원두 블랜딩 원두인데요. 이 블랜딩 이름이 코소롱 블랜드입니다. 자 이 블랜드 같은 경우는 커피창고에서 7월에 출시될 블랜드라고 해요. 근데

===== 영상 4: /MYL_08118.json =====
요약: 그게 있으면 커피 찌꺼기나 먼지들 빨아먹기가 굉장히 편한데 물론 얘의 원래 용도는 차량용 청소기 같은 느낌이라서 차에서 쓰셔도 전혀 문제는 없다.
발화 구간 수: 6
  [0.0~4.551초] 그게 있으면 커피 찌꺼기나 먼지들 빨아먹기가 굉장히 편합니다.
  [4.561~8.131초] 솔로 쓸 때도 있지만 요런 청소기 하나 있으면 굉장히 편하죠.
  [8

In [133]:
import json
import io
import zipfile

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    sample_names = [
        "/MYL_03691.json",
        "/MYL_00843.json",
        "/MYL_04593.json"
    ]

    for name in sample_names:
        data = json.loads(z.read(name).decode("utf-8-sig"))

        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])
        full_transcript = " ".join(
            str(term.get("transcription") or "").strip()
            for term in terms
            if isinstance(term, dict)
        ).strip()

        print(f"\n===== {name} =====")
        print("[요약문]")
        print(summary)

        print("\n[전체 발화문]")
        print(full_transcript)

        print("\n요약문 길이:", len(summary))
        print("전체 발화문 길이:", len(full_transcript))


===== /MYL_03691.json =====
[요약문]
그릇에 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼을 넣고 살짝 섞어주면 드레싱소스는 완성이 된다.

[전체 발화문]
식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼 이 블루베리 잼이 들어가믄 또 기가 막히게 맛있거든요. 이렇게 인자 섞어 주면은 드레싱은 완성이 되는 겁니다. 이 블루베리가 톡톡 터지면서 달콤하거든요. 이렇게 살짝만 섞어 줘야 됩니다. 오래 섞는 거 아닙니다.

요약문 길이: 60
전체 발화문 길이: 148

===== /MYL_00843.json =====
[요약문]
밥 한 공기에 참기름과 식초를 넣어 양념을 한다.

[전체 발화문]
힘을 주면서 동그랗게 인자 해 줍니다. 이렇게 잠시 둔 다음에 인자 밥에다가 양념을 하겠습니다. 오늘은 인자 밥 한 공기 준비했는데요. 220그램입니다. 여기에다가 참기름 반 스푼 정도만 넣겠습니다. 식초는 쬐끔만 넣습니다. 식초도

요약문 길이: 27
전체 발화문 길이: 129

===== /MYL_04593.json =====
[요약문]
박을 절여놓으면 식감도 좋아지고 간이 배서 훨씬 맛있어지기 때문에 집간장에 절이는 게 좋고 국간장을 사용해도 되지만 그럼 색깔이 시커멓게 되버린다.

[전체 발화문]
20분 절이도록 하겠습니다. 절이는 이유는 이 박이 굉장히 연하거든요 지금 근데 절여놓으면은 식감도 좋아지고 간이 배겨갖고 훨씬 맛있거든요 그리고 집간장이 없으신 분들은 국간장을 사용해도 되는데 국간장을 사용하게 되면은 시커멓게 돼부러요. 국간장을 적게 사용하고 소금 좀 더 넣으시면 되거든요.

요약문 길이: 82
전체 발화문 길이: 164


In [134]:
import json
import io
import zipfile
from sentence_transformers import util

# 기존 Fusion에서 사용하던 MiniLM (추가 학습 전)
model = fusion_search.get_semantic_model()

sample_name = "/MYL_00843.json"

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    data = json.loads(z.read(sample_name).decode("utf-8-sig"))

query = data["summary"]

terms = [
    term for term in data["video"]["term"]
    if isinstance(term, dict)
    and str(term.get("transcription") or "").strip()
]

texts = [term["transcription"] for term in terms]

query_embedding = model.encode(query, convert_to_tensor=True)
text_embeddings = model.encode(texts, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, text_embeddings)[0]

ranked = sorted(
    zip(terms, scores.tolist()),
    key=lambda x: x[1],
    reverse=True
)

print("===== 기존 MiniLM 검색 결과 =====")
print("검색어:", query)

for i, (term, score) in enumerate(ranked[:5], 1):
    print(
        f"\n{i}위 | 유사도: {score:.4f}"
        f"\n구간: {term['start']:.2f}~{term['end']:.2f}초"
        f"\n자막: {term['transcription']}"
    )

===== 기존 MiniLM 검색 결과 =====
검색어: 밥 한 공기에 참기름과 식초를 넣어 양념을 한다.

1위 | 유사도: 0.5979
구간: 5.94~7.78초
자막: 오늘은 인자 밥 한 공기 준비했는데요.

2위 | 유사도: 0.4361
구간: 12.50~16.81초
자막: 식초는 쬐끔만 넣습니다.

3위 | 유사도: 0.4252
구간: 9.23~12.49초
자막: 여기에다가 참기름 반 스푼 정도만 넣겠습니다.

4위 | 유사도: 0.3788
구간: 16.82~18.53초
자막: 식초도

5위 | 유사도: 0.3199
구간: 2.76~5.93초
자막: 이렇게 잠시 둔 다음에 인자 밥에다가 양념을 하겠습니다.


In [135]:
from sentence_transformers import util

# 기존 단일 발화 + 연속된 발화 2개를 합친 후보 생성
candidates = []

for i, term in enumerate(terms):
    candidates.append({
        "start": term["start"],
        "end": term["end"],
        "text": term["transcription"],
        "type": "단일 자막"
    })

    if i + 1 < len(terms):
        next_term = terms[i + 1]

        candidates.append({
            "start": term["start"],
            "end": next_term["end"],
            "text": term["transcription"] + " " + next_term["transcription"],
            "type": "자막 2개 결합"
        })

candidate_texts = [c["text"] for c in candidates]

embeddings = model.encode(candidate_texts, convert_to_tensor=True)
scores = util.cos_sim(query_embedding, embeddings)[0].tolist()

ranked_candidates = sorted(
    zip(candidates, scores),
    key=lambda x: x[1],
    reverse=True
)

print("===== 자막 결합 후 MiniLM 검색 결과 =====")
print("검색어:", query)

for i, (candidate, score) in enumerate(ranked_candidates[:7], 1):
    print(
        f"\n{i}위 | {candidate['type']}"
        f"\n유사도: {score:.4f}"
        f"\n구간: {candidate['start']:.2f}~{candidate['end']:.2f}초"
        f"\n자막: {candidate['text']}"
    )

===== 자막 결합 후 MiniLM 검색 결과 =====
검색어: 밥 한 공기에 참기름과 식초를 넣어 양념을 한다.

1위 | 자막 2개 결합
유사도: 0.6388
구간: 9.23~16.81초
자막: 여기에다가 참기름 반 스푼 정도만 넣겠습니다. 식초는 쬐끔만 넣습니다.

2위 | 단일 자막
유사도: 0.5979
구간: 5.94~7.78초
자막: 오늘은 인자 밥 한 공기 준비했는데요.

3위 | 자막 2개 결합
유사도: 0.5071
구간: 2.76~7.78초
자막: 이렇게 잠시 둔 다음에 인자 밥에다가 양념을 하겠습니다. 오늘은 인자 밥 한 공기 준비했는데요.

4위 | 단일 자막
유사도: 0.4361
구간: 12.50~16.81초
자막: 식초는 쬐끔만 넣습니다.

5위 | 자막 2개 결합
유사도: 0.4351
구간: 5.94~9.22초
자막: 오늘은 인자 밥 한 공기 준비했는데요. 220그램입니다.

6위 | 단일 자막
유사도: 0.4252
구간: 9.23~12.49초
자막: 여기에다가 참기름 반 스푼 정도만 넣겠습니다.

7위 | 자막 2개 결합
유사도: 0.4140
구간: 7.79~12.49초
자막: 220그램입니다. 여기에다가 참기름 반 스푼 정도만 넣겠습니다.


In [136]:
import json
import io
import zipfile

sample_pairs = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    json_names = [
        name for name in z.namelist()
        if name.lower().endswith(".json")
    ]

    for name in json_names[:5]:
        data = json.loads(z.read(name).decode("utf-8-sig"))

        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])
        transcript = " ".join(
            str(term.get("transcription") or "").strip()
            for term in terms
            if isinstance(term, dict)
        ).strip()

        if summary and transcript:
            sample_pairs.append({
                "video_id": data.get("metadata", {}).get("filename"),
                "query": summary,
                "positive": transcript
            })

print("===== MiniLM 학습 쌍 샘플 =====")
print("생성된 샘플:", len(sample_pairs))

for pair in sample_pairs[:3]:
    print("\n영상 ID:", pair["video_id"])
    print("Query:", pair["query"])
    print("Positive:", pair["positive"][:300])

===== MiniLM 학습 쌍 샘플 =====
생성된 샘플: 5

영상 ID: MYL_00019
Query: 떡볶이에 버섯류가 없는 것이 아쉽고 분모자는 엽떡보다 마라 떡볶이에 들어간 것이 더 맛있었고 먹다 보니까 매웠습니다.
Positive: 아 그리고 버섯류 없는 것도 너무 아쉬워. 음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다. 음 기본 오리지널보다 마라맛이 싹 배어있네. 아 이거 마라탕이라 생각했어. 떡을 너무 안 먹었다. 떡볶이지. 음 어 근데 먹다 보니까 맵다. 음 매워요.

영상 ID: MYL_00010
Query: 이 떡볶이는 마라 떡볶이인데 엽기 떡볶이에서 신제품이 나왔다고 해서 사 와봤습니다.
Positive: 이거나 한 입 먹고 끝내야 했다. 야 이거 기름 그냥 그거다 그거 행주에 그거다. 야 이거 그냥 보이세요? 쭉 안 보이네. 쭉 헤엑 아니 안 먹을래. 자 유튜브 친구들 안녕하세요? @이름1입니다. 자 오늘 같은 경우에는요. 엽기 떡볶이에 신제품이 나왔어요. 엽떡을 제가 그렇게 좋아하진 않는데 이거는 먹어봐야겠다 싶어 가지고 사 왔거든요. 이게 지금 마라 떡볶이입니다.

영상 ID: MYL_00007
Query: 이 파스타는 처음 먹었을 때부터 우유를 넣고 만들어서 그런지 더 찐득찐득하고 맛이 있었고 파스타를 좋아하는 분들은 이게 인스턴트지만 맛있다고 느끼실 거라고 제가 자부할 수 있고 추천합니다.
Positive: 네 다 먹었습니다. 아 진짜 맛있어요. 그때 처음했을 때도 우유를 넣어 가지고 먹었는데 우유를 넣어서 더 찐득찐득하고 맛있는 거 같아요. 면 음식의 맛은 제가 잘 안다고 자부할 수 있습니다. 네 파스타 좋아하는 분들은 이거 진짜 인스턴트지만 그래도 맛있다고 느끼실 거예요. 이거 추천할게요. 네 오늘은 또 첫 먹방 영상을 찍어봤는데요.


In [137]:
import json
import io
import zipfile
import statistics

transcript_lengths = []
summary_lengths = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    json_names = [
        name for name in z.namelist()
        if name.lower().endswith(".json")
    ]

    for name in json_names:
        data = json.loads(z.read(name).decode("utf-8-sig"))

        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])
        transcript = " ".join(
            str(term.get("transcription") or "").strip()
            for term in terms
            if isinstance(term, dict)
        ).strip()

        if summary and transcript:
            summary_lengths.append(len(summary))
            transcript_lengths.append(len(transcript))

print("===== 전체 학습 쌍 길이 분석 =====")
print("분석한 학습 쌍:", len(transcript_lengths))
print("요약문 평균 글자 수:", round(statistics.mean(summary_lengths), 2))
print("전체 발화문 평균 글자 수:", round(statistics.mean(transcript_lengths), 2))
print("전체 발화문 중앙값:", statistics.median(transcript_lengths))
print("전체 발화문 최대 길이:", max(transcript_lengths))

for threshold in [128, 256, 512, 1000]:
    count = sum(length > threshold for length in transcript_lengths)
    print(
        f"{threshold}글자 초과: {count:,}개 "
        f"({count / len(transcript_lengths) * 100:.2f}%)"
    )

===== 전체 학습 쌍 길이 분석 =====
분석한 학습 쌍: 17899
요약문 평균 글자 수: 56.49
전체 발화문 평균 글자 수: 167.67
전체 발화문 중앙값: 152
전체 발화문 최대 길이: 2065
128글자 초과: 13,527개 (75.57%)
256글자 초과: 1,322개 (7.39%)
512글자 초과: 98개 (0.55%)
1000글자 초과: 6개 (0.03%)


In [138]:
import json
import io
import zipfile
import numpy as np

# 기존 Fusion에서 사용 중인 MiniLM
model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer

print("===== MiniLM 입력 길이 설정 =====")
print("모델 최대 입력 토큰:", model.max_seq_length)

token_lengths = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))

        terms = data.get("video", {}).get("term", [])
        transcript = " ".join(
            str(term.get("transcription") or "").strip()
            for term in terms
            if isinstance(term, dict)
        ).strip()

        if not transcript:
            continue

        # 잘라내지 않은 원래 토큰 수 확인
        tokens = tokenizer.encode(
            transcript,
            add_special_tokens=True,
            truncation=False
        )

        token_lengths.append(len(tokens))

lengths = np.array(token_lengths)
limit = model.max_seq_length

print("\n===== 전체 발화문 토큰 길이 분석 =====")
print("분석한 영상 수:", len(lengths))
print("평균 토큰 수:", round(float(lengths.mean()), 2))
print("중앙값 토큰 수:", round(float(np.median(lengths)), 2))
print("최대 토큰 수:", int(lengths.max()))

over_limit = int(np.sum(lengths > limit))

print(
    f"\n모델 입력 제한 초과: {over_limit:,}개 "
    f"({over_limit / len(lengths) * 100:.2f}%)"
)

Token indices sequence length is longer than the specified maximum sequence length for this model (338 > 128). Running this sequence through the model will result in indexing errors


===== MiniLM 입력 길이 설정 =====
모델 최대 입력 토큰: 128

===== 전체 발화문 토큰 길이 분석 =====
분석한 영상 수: 17899
평균 토큰 수: 95.94
중앙값 토큰 수: 87.0
최대 토큰 수: 1058

모델 입력 제한 초과: 2,076개 (11.60%)


In [139]:
import json
import io
import zipfile
import numpy as np

model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

pair_lengths = []
pair_count = 0

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))

        terms = [
            t for t in data.get("video", {}).get("term", [])
            if isinstance(t, dict)
            and str(t.get("transcription") or "").strip()
        ]

        for i in range(len(terms) - 1):
            text = (
                terms[i]["transcription"].strip()
                + " "
                + terms[i + 1]["transcription"].strip()
            )

            # 경고 없이 토큰 길이 확인
            tokens = tokenizer(
                text,
                add_special_tokens=True,
                truncation=False,
                return_attention_mask=False,
                verbose=False
            )["input_ids"]

            pair_lengths.append(len(tokens))
            pair_count += 1

lengths = np.array(pair_lengths)
over_limit = int(np.sum(lengths > limit))

print("===== 연속 자막 2개 결합 토큰 분석 =====")
print("전체 결합 후보:", pair_count)
print("평균 토큰 수:", round(float(lengths.mean()), 2))
print("중앙값:", float(np.median(lengths)))
print("최대:", int(lengths.max()))
print(
    f"128토큰 초과: {over_limit:,}개 "
    f"({over_limit / pair_count * 100:.2f}%)"
)
print(
    f"128토큰 이내: {pair_count - over_limit:,}개 "
    f"({(pair_count - over_limit) / pair_count * 100:.2f}%)"
)

===== 연속 자막 2개 결합 토큰 분석 =====
전체 결합 후보: 82791
평균 토큰 수: 34.66
중앙값: 31.0
최대: 177
128토큰 초과: 36개 (0.04%)
128토큰 이내: 82,755개 (99.96%)


In [140]:
import json
import io
import zipfile
from sentence_transformers import util

model = fusion_search.get_semantic_model()

sample_name = "/MYL_03691.json"

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    data = json.loads(z.read(sample_name).decode("utf-8-sig"))

summary = data["summary"]

terms = [
    t for t in data["video"]["term"]
    if isinstance(t, dict)
    and str(t.get("transcription") or "").strip()
]

candidates = []

for i in range(len(terms) - 1):
    first = terms[i]
    second = terms[i + 1]

    candidates.append({
        "start": first["start"],
        "end": second["end"],
        "text": first["transcription"] + " " + second["transcription"]
    })

summary_embedding = model.encode(
    summary, convert_to_tensor=True
)

candidate_embeddings = model.encode(
    [c["text"] for c in candidates],
    convert_to_tensor=True
)

scores = util.cos_sim(
    summary_embedding, candidate_embeddings
)[0].tolist()

ranked = sorted(
    zip(candidates, scores),
    key=lambda x: x[1],
    reverse=True
)

print("===== 요약문과 결합 자막 유사도 =====")
print("요약문:", summary)

for i, (candidate, score) in enumerate(ranked, 1):
    print(
        f"\n{i}위 | 유사도={score:.4f}"
        f"\n구간: {candidate['start']:.2f}~{candidate['end']:.2f}초"
        f"\n자막: {candidate['text']}"
    )

===== 요약문과 결합 자막 유사도 =====
요약문: 그릇에 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼을 넣고 살짝 섞어주면 드레싱소스는 완성이 된다.

1위 | 유사도=0.8610
구간: 0.00~27.03초
자막: 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼 이 블루베리 잼이 들어가믄 또 기가 막히게 맛있거든요. 이렇게 인자 섞어 주면은 드레싱은 완성이 되는 겁니다.

2위 | 유사도=0.6423
구간: 21.81~30.82초
자막: 이렇게 인자 섞어 주면은 드레싱은 완성이 되는 겁니다. 이 블루베리가 톡톡 터지면서 달콤하거든요.

3위 | 유사도=0.5690
구간: 27.04~33.01초
자막: 이 블루베리가 톡톡 터지면서 달콤하거든요. 이렇게 살짝만 섞어 줘야 됩니다.

4위 | 유사도=0.5446
구간: 30.84~34.90초
자막: 이렇게 살짝만 섞어 줘야 됩니다. 오래 섞는 거 아닙니다.


In [141]:
import json
import io
import zipfile
from sentence_transformers import util

model = fusion_search.get_semantic_model()

sample_names = [
    "/MYL_03691.json",
    "/MYL_00843.json",
    "/MYL_04593.json",
    "/MYL_00019.json"
]

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in sample_names:
        data = json.loads(z.read(name).decode("utf-8-sig"))

        summary = str(data.get("summary") or "").strip()

        terms = [
            t for t in data.get("video", {}).get("term", [])
            if isinstance(t, dict)
            and str(t.get("transcription") or "").strip()
        ]

        candidates = [
            terms[i]["transcription"].strip()
            + " "
            + terms[i + 1]["transcription"].strip()
            for i in range(len(terms) - 1)
        ]

        if not summary or len(candidates) < 2:
            continue

        embeddings = model.encode(
            [summary] + candidates,
            convert_to_tensor=True
        )

        scores = util.cos_sim(
            embeddings[0:1],
            embeddings[1:]
        )[0].tolist()

        ranked = sorted(
            zip(candidates, scores),
            key=lambda x: x[1],
            reverse=True
        )

        first_score = ranked[0][1]
        second_score = ranked[1][1]

        print(f"\n===== {name} =====")
        print("요약:", summary)
        print("1위 유사도:", round(first_score, 4))
        print("2위 유사도:", round(second_score, 4))
        print("점수 차이:", round(first_score - second_score, 4))
        print("1위 자막:", ranked[0][0][:150])


===== /MYL_03691.json =====
요약: 그릇에 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼을 넣고 살짝 섞어주면 드레싱소스는 완성이 된다.
1위 유사도: 0.861
2위 유사도: 0.6423
점수 차이: 0.2187
1위 자막: 식초 2스푼, 토마토소스 3스푼, 블루베리 잼 3스푼 이 블루베리 잼이 들어가믄 또 기가 막히게 맛있거든요. 이렇게 인자 섞어 주면은 드레싱은 완성이 되는 겁니다.

===== /MYL_00843.json =====
요약: 밥 한 공기에 참기름과 식초를 넣어 양념을 한다.
1위 유사도: 0.6388
2위 유사도: 0.5071
점수 차이: 0.1317
1위 자막: 여기에다가 참기름 반 스푼 정도만 넣겠습니다. 식초는 쬐끔만 넣습니다.

===== /MYL_04593.json =====
요약: 박을 절여놓으면 식감도 좋아지고 간이 배서 훨씬 맛있어지기 때문에 집간장에 절이는 게 좋고 국간장을 사용해도 되지만 그럼 색깔이 시커멓게 되버린다.
1위 유사도: 0.77
2위 유사도: 0.7293
점수 차이: 0.0407
1위 자막: 20분 절이도록 하겠습니다. 절이는 이유는 이 박이 굉장히 연하거든요 지금 근데 절여놓으면은 식감도 좋아지고 간이 배겨갖고 훨씬 맛있거든요

===== /MYL_00019.json =====
요약: 떡볶이에 버섯류가 없는 것이 아쉽고 분모자는 엽떡보다 마라 떡볶이에 들어간 것이 더 맛있었고 먹다 보니까 매웠습니다.
1위 유사도: 0.9458
2위 유사도: 0.8744
점수 차이: 0.0713
1위 자막: 아 그리고 버섯류 없는 것도 너무 아쉬워. 음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다.


In [142]:
import json
import io
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

output_path = training_dir / "korean_subtitle_candidates.jsonl"

records = []
model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))

        video_id = data.get("metadata", {}).get("filename")
        summary = str(data.get("summary") or "").strip()

        terms = [
            t for t in data.get("video", {}).get("term", [])
            if isinstance(t, dict)
            and str(t.get("transcription") or "").strip()
            and isinstance(t.get("start"), (int, float))
            and isinstance(t.get("end"), (int, float))
            and 0 <= t["start"] < t["end"]
        ]

        for i in range(len(terms) - 1):
            first, second = terms[i], terms[i + 1]

            text = (
                first["transcription"].strip()
                + " "
                + second["transcription"].strip()
            )

            token_count = len(
                tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=False,
                    verbose=False
                )["input_ids"]
            )

            if token_count > limit:
                continue

            records.append({
                "video_id": video_id,
                "start_time": first["start"],
                "end_time": second["end"],
                "text": text,
                "summary": summary,
                "token_count": token_count
            })

with output_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("===== 한국어 자막 후보 데이터셋 생성 완료 =====")
print("저장 경로:", output_path)
print("전체 후보 수:", len(records))
print("파일 크기(MB):", round(output_path.stat().st_size / 1024**2, 2))
print("\n첫 번째 데이터:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

===== 한국어 자막 후보 데이터셋 생성 완료 =====
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/korean_subtitle_candidates.jsonl
전체 후보 수: 82753
파일 크기(MB): 31.02

첫 번째 데이터:
{
  "video_id": "MYL_00019",
  "start_time": 0.0,
  "end_time": 12.184,
  "text": "아 그리고 버섯류 없는 것도 너무 아쉬워. 음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다.",
  "summary": "떡볶이에 버섯류가 없는 것이 아쉽고 분모자는 엽떡보다 마라 떡볶이에 들어간 것이 더 맛있었고 먹다 보니까 매웠습니다.",
  "token_count": 40
}


# 학습·검증·테스트 데이터 분리
- 같은 영상에서 나온 자막이 학습 데이터와 테스트 데이터에 동시에 들어가면, 모델이 이미 본 영상 내용으로 평가받게 될 수 있어.
- 그래서 자막 단위가 아니라 video_id 단위로 분리할 거야.
이번에는 80:10:10 비율로 나누자.

In [143]:
import tarfile
import zipfile
import io
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

# 한글 파일명 인코딩 문제를 피하기 위해 VL_12로 탐색
val_tar_files = list(training_dir.glob("VL_12*.tar"))

print("===== Validation 데이터 확인 =====")
print("찾은 TAR 파일 수:", len(val_tar_files))

if len(val_tar_files) == 1:
    val_tar_path = val_tar_files[0]

    print("파일명:", val_tar_path.name)
    print("파일 크기(MB):", round(val_tar_path.stat().st_size / 1024**2, 2))

    with tarfile.open(val_tar_path, "r:*") as tar:
        members = [m for m in tar.getmembers() if m.isfile()]

        print("\nTAR 내부 파일 수:", len(members))

        for member in members:
            print("내부 파일명:", Path(member.name).name)

        if len(members) == 1:
            val_zip_bytes = tar.extractfile(members[0]).read()

    if len(members) == 1:
        try:
            with zipfile.ZipFile(io.BytesIO(val_zip_bytes)) as z:
                json_names = [
                    name for name in z.namelist()
                    if name.lower().endswith(".json")
                ]

                print("\n✅ Validation ZIP 정상적으로 열림!")
                print("JSON 파일 개수:", len(json_names))
                print("첫 번째 JSON:", json_names[0] if json_names else "없음")

        except zipfile.BadZipFile:
            print("ZIP을 열 수 없습니다. 분할 파일 여부를 확인해야 합니다.")
else:
    print("Validation TAR 파일 경로를 확인해 주세요.")

===== Validation 데이터 확인 =====
찾은 TAR 파일 수: 1
파일명: VL_12.음식요리.tar
파일 크기(MB): 2.04

TAR 내부 파일 수: 1
내부 파일명: VL_12._음식요리.zip.part0

✅ Validation ZIP 정상적으로 열림!
JSON 파일 개수: 2237
첫 번째 JSON: /MYL_18130.json


In [144]:
import io
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

# Training JSON의 영상 ID
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    train_ids = {
        Path(name).stem
        for name in z.namelist()
        if name.lower().endswith(".json")
    }

# Validation JSON의 영상 ID
with zipfile.ZipFile(io.BytesIO(val_zip_bytes)) as z:
    val_ids = {
        Path(name).stem
        for name in z.namelist()
        if name.lower().endswith(".json")
    }

overlap = train_ids & val_ids

print("===== Training / Validation 영상 ID 검사 =====")
print("Training 영상 수:", len(train_ids))
print("Validation 영상 수:", len(val_ids))
print("중복 영상 ID 수:", len(overlap))

if overlap:
    print("중복 ID 예시:", sorted(overlap)[:10])
else:
    print("✅ Training과 Validation의 영상 ID가 중복되지 않습니다.")

===== Training / Validation 영상 ID 검사 =====
Training 영상 수: 17899
Validation 영상 수: 2237
중복 영상 ID 수: 0
✅ Training과 Validation의 영상 ID가 중복되지 않습니다.


In [145]:
import json
import io
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

output_path = training_dir / "korean_subtitle_candidates_val_official.jsonl"

model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

records = []
total_json = 0

with zipfile.ZipFile(io.BytesIO(val_zip_bytes)) as z:
    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))
        total_json += 1

        video_id = data.get("metadata", {}).get("filename")
        summary = str(data.get("summary") or "").strip()

        terms = [
            t for t in data.get("video", {}).get("term", [])
            if isinstance(t, dict)
            and str(t.get("transcription") or "").strip()
            and isinstance(t.get("start"), (int, float))
            and isinstance(t.get("end"), (int, float))
            and 0 <= t["start"] < t["end"]
        ]

        for i in range(len(terms) - 1):
            first, second = terms[i], terms[i + 1]

            text = (
                first["transcription"].strip()
                + " "
                + second["transcription"].strip()
            )

            token_count = len(
                tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=False,
                    verbose=False
                )["input_ids"]
            )

            if token_count > limit:
                continue

            records.append({
                "video_id": video_id,
                "start_time": first["start"],
                "end_time": second["end"],
                "text": text,
                "summary": summary,
                "token_count": token_count
            })

with output_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("===== 공식 Validation 자막 후보 생성 완료 =====")
print("원본 JSON 수:", total_json)
print("생성된 자막 후보 수:", len(records))
print("고유 영상 수:", len({r["video_id"] for r in records}))
print("저장 경로:", output_path)
print("파일 크기(MB):", round(output_path.stat().st_size / 1024**2, 2))

if records:
    print("\n===== 첫 번째 후보 =====")
    print(json.dumps(records[0], ensure_ascii=False, indent=2))

===== 공식 Validation 자막 후보 생성 완료 =====
원본 JSON 수: 2237
생성된 자막 후보 수: 10120
고유 영상 수: 2202
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/korean_subtitle_candidates_val_official.jsonl
파일 크기(MB): 3.57

===== 첫 번째 후보 =====
{
  "video_id": "MYL_18130",
  "start_time": 0.0,
  "end_time": 5.12,
  "text": "선생님 제가 살을 빼려고 그러는데요. 그래서 오늘은 어떤 운동을 하면 될까요?",
  "summary": "한 여성이 살을 빼기 위해 등 운동을 한 뒤 양념치킨을 못 먹어도 닭가슴살로 대체할 수 있으며 조리를 해보겠다고 얘기하고 있다.",
  "token_count": 20
}


In [146]:
import json
import io
import zipfile

# 공식 Training과 Validation 원본 ZIP
datasets = {
    "TRAIN": zip_bytes,
    "VALIDATION": val_zip_bytes
}

print("===== 공식 데이터셋 경계 확인 =====")

for split, data_bytes in datasets.items():
    with zipfile.ZipFile(io.BytesIO(data_bytes)) as z:
        names = sorted(
            [name for name in z.namelist()
             if name.lower().endswith(".json")]
        )

        selected = names[-5:] if split == "TRAIN" else names[:5]

        print(f"\n===== {split} =====")
        print("영상 ID 범위:", names[0], "~", names[-1])

        for name in selected:
            data = json.loads(
                z.read(name).decode("utf-8-sig")
            )

            metadata = data.get("metadata", {})

            print("\n영상:", metadata.get("filename"))
            print("날짜:", metadata.get("date"))
            print("세부 카테고리:", metadata.get("sub_category"))
            print("요약:", data.get("summary", ""))

===== 공식 데이터셋 경계 확인 =====

===== TRAIN =====
영상 ID 범위: /MYL_00001.json ~ /MYL_18126.json

영상: MYL_18122
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여성이 LA식 꽃갈비는 해동한 다음 해동된 갈비를 예열된 프라이팬에 올린 뒤에 소스를 반 정도 붓고 중약불에 8분간 익혀서 먹으면 된다고 얘기하고 있다.

영상: MYL_18123
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여성이 남은 LA갈비 양념 소스에 갈빗살을 썰어 넣어 밥을 볶으면 갈비 볶음밥으로 먹을 수 있고 대구식 매운 돼지 갈비찜은 해동한 다음 예열된 냄비에 올린 뒤 중약불로 10분간 익혀 먹으면 된다고 얘기하고 있다.

영상: MYL_18124
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여성이 남은 매운 양념 소스와 밥, 김가루, 참기름을 넣고 볶으면 후식 볶음밥이 되며 조리한 LA꽃갈비와 대구식 매운 대지 갈비찜을 가족들이랑 맛있게 먹어본다고 얘기하고 있다.

영상: MYL_18125
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여성이 곧 추석이여서 명절에 맛은 갈비를 좋을 것 같아 갈비 신상품 2종을 준비했으며 LA식 꽃갈비와 대구식 매운 갈비찜을 먹어보겠다고 얘기하고 있다.

영상: MYL_18126
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여성이 애기들에게는 안 매운 LA식 갈비를 줬고 본인도 안 매운 것을 먼저 먹어보도록 하겠다고 얘기하고 있다.

===== VALIDATION =====
영상 ID 범위: /MYL_18127.json ~ /MYL_20385.json

영상: MYL_18127
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여성이 가족들과 함께 갈비를 먹고 있고 매운 거는 맵기만 한 것이 아니라 은은한 단맛이 난다고 얘기하고 있다.

영상: MYL_18128
날짜: 2023-09-12
세부 카테고리: 1
요약: 한 여

In [147]:
import json
import io
import zipfile

train_zip = zipfile.ZipFile(io.BytesIO(zip_bytes))
val_zip = zipfile.ZipFile(io.BytesIO(val_zip_bytes))

print("===== Training / Validation 경계 상세 검사 =====")

for number in range(18120, 18141):
    video_id = f"MYL_{number:05d}"
    filename = f"/{video_id}.json"

    if filename in train_zip.namelist():
        z = train_zip
        split = "TRAIN"
    elif filename in val_zip.namelist():
        z = val_zip
        split = "VALIDATION"
    else:
        continue

    data = json.loads(z.read(filename).decode("utf-8-sig"))
    metadata = data.get("metadata", {})

    print(
        f"{video_id} | {split:10s} "
        f"| 날짜={metadata.get('date')} "
        f"| 세부카테고리={metadata.get('sub_category')}"
    )

train_zip.close()
val_zip.close()

===== Training / Validation 경계 상세 검사 =====
MYL_18120 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18121 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18122 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18123 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18124 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18125 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18126 | TRAIN      | 날짜=2023-09-12 | 세부카테고리=1
MYL_18127 | VALIDATION | 날짜=2023-09-12 | 세부카테고리=1
MYL_18128 | VALIDATION | 날짜=2023-09-12 | 세부카테고리=1
MYL_18129 | VALIDATION | 날짜=2023-03-18 | 세부카테고리=9
MYL_18130 | VALIDATION | 날짜=2023-03-18 | 세부카테고리=9
MYL_18131 | VALIDATION | 날짜=2023-03-18 | 세부카테고리=9
MYL_18134 | VALIDATION | 날짜=2023-01-19 | 세부카테고리=8
MYL_18135 | VALIDATION | 날짜=2023-01-19 | 세부카테고리=8
MYL_18136 | VALIDATION | 날짜=2022-12-15 | 세부카테고리=1
MYL_18137 | VALIDATION | 날짜=2022-12-15 | 세부카테고리=1
MYL_18138 | VALIDATION | 날짜=2022-12-15 | 세부카테고리=1
MYL_18139 | VALIDATION | 날짜=2022-12-15 | 세부카테고리=1
MYL_18140 | VALIDATION | 날짜=2022-12-15 | 세부카테고리=1


In [148]:
import json
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

input_path = training_dir / "korean_subtitle_candidates_val_official.jsonl"
output_path = training_dir / "korean_subtitle_candidates_val_clean.jsonl"

# 원본 영상 중복 가능성이 높은 Validation 클립
exclude_ids = {
    "MYL_18127",
    "MYL_18128"
}

total = 0
kept = 0
excluded = 0
remaining_ids = set()

with input_path.open("r", encoding="utf-8") as fin, \
     output_path.open("w", encoding="utf-8") as fout:

    for line in fin:
        record = json.loads(line)
        total += 1

        if record["video_id"] in exclude_ids:
            excluded += 1
            continue

        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        kept += 1
        remaining_ids.add(record["video_id"])

print("===== Validation 정제 결과 =====")
print("기존 자막 후보:", total)
print("제외된 자막 후보:", excluded)
print("남은 자막 후보:", kept)
print("남은 영상 수:", len(remaining_ids))
print("저장 경로:", output_path)

===== Validation 정제 결과 =====
기존 자막 후보: 10120
제외된 자막 후보: 21
남은 자막 후보: 10099
남은 영상 수: 2200
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/korean_subtitle_candidates_val_clean.jsonl


In [149]:
import json
from pathlib import Path
from collections import Counter

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

for split, filename in [
    ("TRAIN", "korean_subtitle_candidates.jsonl"),
    ("VALIDATION", "korean_subtitle_candidates_val_clean.jsonl")
]:
    path = training_dir / filename
    counts = Counter()

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            counts[record["video_id"]] += 1

    values = sorted(counts.values())

    print(f"\n===== {split} =====")
    print("영상 수:", len(values))
    print("전체 자막 후보 수:", sum(values))
    print("영상당 평균 후보 수:", round(sum(values) / len(values), 2))
    print("영상당 중앙값:", values[len(values) // 2])
    print("최소 후보 수:", min(values))
    print("최대 후보 수:", max(values))
    print("후보 1개인 영상:", sum(v == 1 for v in values))
    print("후보 10개 이상인 영상:", sum(v >= 10 for v in values))


===== TRAIN =====
영상 수: 17770
전체 자막 후보 수: 82753
영상당 평균 후보 수: 4.66
영상당 중앙값: 4
최소 후보 수: 1
최대 후보 수: 57
후보 1개인 영상: 1178
후보 10개 이상인 영상: 1011

===== VALIDATION =====
영상 수: 2200
전체 자막 후보 수: 10099
영상당 평균 후보 수: 4.59
영상당 중앙값: 4
최소 후보 수: 1
최대 후보 수: 30
후보 1개인 영상: 283
후보 10개 이상인 영상: 180


In [150]:
import json
import random
from pathlib import Path
from collections import defaultdict
from sentence_transformers import util

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

train_path = training_dir / "korean_subtitle_candidates.jsonl"

grouped = defaultdict(list)

with train_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        grouped[record["video_id"]].append(record)

# 재현 가능한 무작위 샘플
rng = random.Random(42)
sample_ids = rng.sample(sorted(grouped.keys()), 10)

model = fusion_search.get_semantic_model()

print("===== MiniLM 학습용 Positive 후보 10개 =====")

for video_id in sample_ids:
    candidates = grouped[video_id]
    summary = candidates[0]["summary"]
    texts = [record["text"] for record in candidates]

    embeddings = model.encode(
        [summary] + texts,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        embeddings[:1],
        embeddings[1:]
    )[0].tolist()

    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    best = candidates[best_idx]

    print(f"\n===== {video_id} =====")
    print("Query:", summary)
    print("Positive 후보:", best["text"])
    print(
        "시간:",
        f"{best['start_time']:.2f}~{best['end_time']:.2f}초"
    )
    print("유사도:", round(scores[best_idx], 4))

===== MiniLM 학습용 Positive 후보 10개 =====

===== MYL_03683 =====
Query: 마늘 3개와 새송이버섯 1개 썰어 팬에 넣고 불을 약간 높여서 계속 볶아 준다.
Positive 후보: 이렇게 해서 한번 살짝 볶아 주면서 다시 놔두고 마늘을 썰어 줍니다. 새송이버섯 1개 사용합니다.
시간: 0.00~13.03초
유사도: 0.8177

===== MYL_00848 =====
Query: 칼에 참기름을 살짝 발라 김밥을 잘라주면 맛있는 계란김밥이 완성된다.
Positive 후보: 자 이렇게 해 갖고 인자 그냥 먹기 좋은 크기로 썰면 됩니다. 드디어 계란김밥이 맛있게 완성되었습니다.
시간: 7.28~19.48초
유사도: 0.7872

===== MYL_09102 =====
Query: 남성이 세 번째로 소개해드릴 원두는 브라질 디카페인 원두라고 한다.
Positive 후보: 한번 구매해 보시기를 추천드립니다. 세 번째로 소개해드릴 원두 브라질 디카페인 원두입니다.
시간: 0.00~7.28초
유사도: 0.883

===== MYL_08109 =====
Query: 약간 리볼버 권총 같이 생겼는데 뒤집어보면 쓰리봄버 로고랑 미끄럼 방지 요런 게 붙어 있다.
Positive 후보: 뭔가 약간 리볼버 권총 같이 생기기도 했죠? 거꾸로 뒤집어보면 요렇게 쓰리봄버 로고랑 미끄럼 방지 요런 게 붙어 있어요.
시간: 10.77~20.29초
유사도: 0.9051

===== MYL_07392 =====
Query: 진하고 달콤한 육수를 면에 몽땅 부어 먹으면 되고 면도 색이 진하고 보기 좋았다.
Positive 후보: 그런 육수보다 조금 더 진하고 좀 달콤한 그런 맛이 있는 육수였습니다. 소바 면발이 상당히 색이 진하고 보기가 좋은데요.
시간: 0.00~9.59초
유사도: 0.7452

===== MYL_04607 =====
Query: 꽈리고추 150그램을 끓은 물에 소금 약간 풀고 1분 정도 데쳐서 사용한다.
Positive 후보: 매운

In [151]:
import json
import io
import zipfile
from collections import Counter

model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

stats = Counter()
examples = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))

        summary = str(data.get("summary") or "").strip()
        terms = data.get("video", {}).get("term", [])

        transcript = " ".join(
            str(t.get("transcription") or "").strip()
            for t in terms
            if isinstance(t, dict)
        ).strip()

        if not summary or not transcript:
            continue

        stats["total"] += 1

        lengths = [
            len(tokenizer(
                text,
                add_special_tokens=True,
                truncation=False,
                verbose=False
            )["input_ids"])
            for text in (summary, transcript)
        ]

        if max(lengths) <= limit:
            stats["valid"] += 1

            if len(examples) < 3:
                examples.append({
                    "video_id": data["metadata"]["filename"],
                    "summary_tokens": lengths[0],
                    "transcript_tokens": lengths[1]
                })
        else:
            stats["over_limit"] += 1

print("===== MiniLM 학습 쌍 길이 검사 =====")
print("전체 영상:", stats["total"])
print("양쪽 모두 128토큰 이내:", stats["valid"])
print("길이 초과:", stats["over_limit"])
print("사용 가능 비율:", round(stats["valid"] / stats["total"] * 100, 2), "%")

print("\n===== 샘플 =====")
for example in examples:
    print(example)

===== MiniLM 학습 쌍 길이 검사 =====
전체 영상: 17899
양쪽 모두 128토큰 이내: 15823
길이 초과: 2076
사용 가능 비율: 88.4 %

===== 샘플 =====
{'video_id': 'MYL_00019', 'summary_tokens': 44, 'transcript_tokens': 97}
{'video_id': 'MYL_00010', 'summary_tokens': 33, 'transcript_tokens': 123}
{'video_id': 'MYL_00007', 'summary_tokens': 60, 'transcript_tokens': 97}


In [152]:
import json
import io
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

output_path = training_dir / "minilm_train_pairs.jsonl"

model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

count = 0

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z, \
     output_path.open("w", encoding="utf-8") as f:

    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))

        video_id = data.get("metadata", {}).get("filename")
        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])
        transcript = " ".join(
            str(t.get("transcription") or "").strip()
            for t in terms
            if isinstance(t, dict)
        ).strip()

        if not summary or not transcript:
            continue

        summary_tokens = len(tokenizer(
            summary, truncation=False, verbose=False
        )["input_ids"])

        transcript_tokens = len(tokenizer(
            transcript, truncation=False, verbose=False
        )["input_ids"])

        if summary_tokens > limit or transcript_tokens > limit:
            continue

        record = {
            "video_id": video_id,
            "query": summary,
            "positive": transcript
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        count += 1

print("===== MiniLM Training 데이터 생성 완료 =====")
print("학습 쌍 수:", count)
print("저장 경로:", output_path)
print("파일 크기(MB):", round(output_path.stat().st_size / 1024**2, 2))

with output_path.open("r", encoding="utf-8") as f:
    print("\n첫 번째 학습 쌍:")
    print(json.dumps(json.loads(f.readline()), ensure_ascii=False, indent=2))

===== MiniLM Training 데이터 생성 완료 =====
학습 쌍 수: 15823
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_train_pairs.jsonl
파일 크기(MB): 8.3

첫 번째 학습 쌍:
{
  "video_id": "MYL_00019",
  "query": "떡볶이에 버섯류가 없는 것이 아쉽고 분모자는 엽떡보다 마라 떡볶이에 들어간 것이 더 맛있었고 먹다 보니까 매웠습니다.",
  "positive": "아 그리고 버섯류 없는 것도 너무 아쉬워. 음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다. 음 기본 오리지널보다 마라맛이 싹 배어있네. 아 이거 마라탕이라 생각했어. 떡을 너무 안 먹었다. 떡볶이지. 음 어 근데 먹다 보니까 맵다. 음 매워요."
}


In [153]:
import json
import io
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

output_path = training_dir / "minilm_val_pairs.jsonl"

# Training과 동일한 토큰 길이 기준
model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

# Training과 같은 원본에서 분리됐을 가능성이 높은 클립
exclude_ids = {"MYL_18127", "MYL_18128"}

stats = {
    "total": 0,
    "excluded": 0,
    "over_limit": 0,
    "saved": 0
}

with zipfile.ZipFile(io.BytesIO(val_zip_bytes)) as z, \
     output_path.open("w", encoding="utf-8") as f:

    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        stats["total"] += 1

        data = json.loads(z.read(name).decode("utf-8-sig"))
        video_id = data.get("metadata", {}).get("filename")

        if video_id in exclude_ids:
            stats["excluded"] += 1
            continue

        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])
        transcript = " ".join(
            str(t.get("transcription") or "").strip()
            for t in terms
            if isinstance(t, dict)
        ).strip()

        if not summary or not transcript:
            continue

        summary_tokens = len(tokenizer(
            summary, truncation=False, verbose=False
        )["input_ids"])

        transcript_tokens = len(tokenizer(
            transcript, truncation=False, verbose=False
        )["input_ids"])

        if summary_tokens > limit or transcript_tokens > limit:
            stats["over_limit"] += 1
            continue

        record = {
            "video_id": video_id,
            "query": summary,
            "positive": transcript
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        stats["saved"] += 1

print("===== MiniLM Validation 데이터 생성 완료 =====")
print("전체 Validation JSON:", stats["total"])
print("중복 의심 클립 제외:", stats["excluded"])
print("128토큰 초과 제외:", stats["over_limit"])
print("최종 Validation 학습 쌍:", stats["saved"])
print("저장 경로:", output_path)
print("파일 크기(MB):", round(output_path.stat().st_size / 1024**2, 2))

with output_path.open("r", encoding="utf-8") as f:
    print("\n첫 번째 Validation 쌍:")
    print(json.dumps(json.loads(f.readline()), ensure_ascii=False, indent=2))

===== MiniLM Validation 데이터 생성 완료 =====
전체 Validation JSON: 2237
중복 의심 클립 제외: 2
128토큰 초과 제외: 287
최종 Validation 학습 쌍: 1948
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_val_pairs.jsonl
파일 크기(MB): 0.92

첫 번째 Validation 쌍:
{
  "video_id": "MYL_18130",
  "query": "한 여성이 살을 빼기 위해 등 운동을 한 뒤 양념치킨을 못 먹어도 닭가슴살로 대체할 수 있으며 조리를 해보겠다고 얘기하고 있다.",
  "positive": "선생님 제가 살을 빼려고 그러는데요. 그래서 오늘은 어떤 운동을 하면 될까요? 어 살을 빼셔야 되니까 네. 그러면 등 운동부터 할게요. 네 알겠습니다. 등 운동 한 번 해볼게요. 네. 양념치킨을 못 먹어도 이렇게 닭가슴살로 대체할 수 있어요. 조리 방법을 한 번 볼게요. 냉장 해동 시 약 1분 냉동 상태 시 약 2분에서 3분이네요. 한 번 조리를 해보도록 하겠습니다."
}


In [154]:
import json
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

train_path = training_dir / "minilm_train_pairs.jsonl"
val_path = training_dir / "minilm_val_pairs.jsonl"

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

train_pairs = load_jsonl(train_path)
val_pairs = load_jsonl(val_path)

print("===== MiniLM Fine-tuning 데이터 확인 =====")
print("Training 쌍:", len(train_pairs))
print("Validation 쌍:", len(val_pairs))

train_ids = {p["video_id"] for p in train_pairs}
val_ids = {p["video_id"] for p in val_pairs}

overlap = train_ids & val_ids

print("Training/Validation 중복 영상 ID:", len(overlap))

assert len(overlap) == 0, "영상 ID가 중복됩니다!"
assert all(
    p.get("query") and p.get("positive")
    for p in train_pairs + val_pairs
), "빈 학습 문장이 있습니다!"

print("✅ 학습 데이터 기본 검사 통과!")

===== MiniLM Fine-tuning 데이터 확인 =====
Training 쌍: 15823
Validation 쌍: 1948
Training/Validation 중복 영상 ID: 0
✅ 학습 데이터 기본 검사 통과!


In [155]:
import json
import random
import numpy as np
from sentence_transformers import util
from pathlib import Path

# 추가 학습 전 기존 MiniLM
model = fusion_search.get_semantic_model()

# 동일한 샘플을 재현할 수 있도록 고정
rng = random.Random(42)
eval_pairs = rng.sample(val_pairs, 100)

queries = [p["query"] for p in eval_pairs]
documents = [p["positive"] for p in eval_pairs]

# 기존 MiniLM 임베딩 생성
query_embeddings = model.encode(
    queries,
    convert_to_tensor=True,
    show_progress_bar=True
)

doc_embeddings = model.encode(
    documents,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 100개 요약문 × 100개 발화문 유사도
scores = util.cos_sim(query_embeddings, doc_embeddings)

ranks = []

for i in range(len(eval_pairs)):
    ranked_indices = scores[i].argsort(descending=True).tolist()

    # 같은 인덱스의 요약문-발화문을 정답 쌍으로 간주
    rank = ranked_indices.index(i) + 1
    ranks.append(rank)

recall_at_1 = np.mean([rank <= 1 for rank in ranks])
recall_at_5 = np.mean([rank <= 5 for rank in ranks])
mrr = np.mean([1 / rank for rank in ranks])

print("\n===== 기존 MiniLM Baseline 평가 =====")
print("평가 쌍:", len(eval_pairs))
print(f"Recall@1: {recall_at_1:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"MRR: {mrr:.4f}")

# 평가 결과를 Google Drive에 저장
baseline_result = {
    "model": "paraphrase-multilingual-MiniLM-L12-v2",
    "evaluation": "AI-Hub Korean summary-to-transcript retrieval",
    "sample_size": len(eval_pairs),
    "random_seed": 42,
    "recall_at_1": float(recall_at_1),
    "recall_at_5": float(recall_at_5),
    "mrr": float(mrr)
}

save_path = training_dir / "minilm_baseline_evaluation.json"

save_path.write_text(
    json.dumps(baseline_result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n✅ Baseline 평가 저장 완료:", save_path)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


===== 기존 MiniLM Baseline 평가 =====
평가 쌍: 100
Recall@1: 0.6700
Recall@5: 0.8400
MRR: 0.7546

✅ Baseline 평가 저장 완료: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_baseline_evaluation.json


In [156]:
import torch
import transformers
import sentence_transformers
import accelerate

print("===== MiniLM Fine-tuning 환경 확인 =====")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence-Transformers:", sentence_transformers.__version__)
print("Accelerate:", accelerate.__version__)

print("\n===== GPU 확인 =====")
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
    print(
        "GPU 메모리(GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        )
    )
else:
    print("현재 GPU가 연결되지 않았습니다.")

===== MiniLM Fine-tuning 환경 확인 =====
PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Sentence-Transformers: 5.7.0
Accelerate: 1.15.0

===== GPU 확인 =====
CUDA 사용 가능: True
GPU 이름: Tesla T4
GPU 메모리(GB): 14.56


In [157]:
from pathlib import Path
import torch

# 현재 Fusion에서 사용 중인 원본 MiniLM
base_model = fusion_search.get_semantic_model()

# 추가 학습한 모델을 별도로 저장할 폴더
output_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned"
)

print("===== MiniLM Fine-tuning 설정 =====")

print("기존 모델:", base_model)
print("최대 입력 토큰:", base_model.max_seq_length)
print("모델 사용 장치:", base_model.device)

print("\n===== 학습 데이터 =====")
print("Training:", len(train_pairs), "쌍")
print("Validation:", len(val_pairs), "쌍")

print("\n===== 저장 경로 =====")
print(output_dir)

print("\n===== GPU 메모리 =====")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("사용 가능:", round(free / 1024**3, 2), "GB")
    print("전체:", round(total / 1024**3, 2), "GB")

print("\n✅ 학습 설정 확인 완료")

===== MiniLM Fine-tuning 설정 =====
기존 모델: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
최대 입력 토큰: 128
모델 사용 장치: cuda:0

===== 학습 데이터 =====
Training: 15823 쌍
Validation: 1948 쌍

===== 저장 경로 =====
/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned

===== GPU 메모리 =====
사용 가능: 8.31 GB
전체: 14.56 GB

✅ 학습 설정 확인 완료


In [158]:
from collections import Counter

print("===== MiniLM 학습 데이터 중복 검사 =====")

# 기존에 불러온 train_pairs 사용
queries = [p["query"].strip() for p in train_pairs]
positives = [p["positive"].strip() for p in train_pairs]

query_counts = Counter(queries)
positive_counts = Counter(positives)

exact_pairs = [
    (p["query"].strip(), p["positive"].strip())
    for p in train_pairs
]
pair_counts = Counter(exact_pairs)

print("전체 학습 쌍:", len(train_pairs))
print("고유 Query 수:", len(query_counts))
print("고유 Positive 수:", len(positive_counts))

print(
    "완전히 동일한 학습 쌍 중복:",
    len(train_pairs) - len(pair_counts)
)
print(
    "동일 Query 반복:",
    len(queries) - len(query_counts)
)
print(
    "동일 Positive 반복:",
    len(positives) - len(positive_counts)
)

print("\n===== 반복되는 Query 예시 =====")

duplicates = [
    (query, count)
    for query, count in query_counts.items()
    if count > 1
]

duplicates.sort(key=lambda x: x[1], reverse=True)

for query, count in duplicates[:5]:
    print(f"\n반복 횟수: {count}")
    print("Query:", query[:200])

===== MiniLM 학습 데이터 중복 검사 =====
전체 학습 쌍: 15823
고유 Query 수: 15573
고유 Positive 수: 15823
완전히 동일한 학습 쌍 중복: 0
동일 Query 반복: 250
동일 Positive 반복: 0

===== 반복되는 Query 예시 =====

반복 횟수: 49
Query: 한 남성이 요리를 하면서 설명하고 있다.

반복 횟수: 26
Query: 한 남성이 포스기에 대해서 얘기하고 있다.

반복 횟수: 24
Query: 한 남성이 그라인더에 대해서 얘기하고 있다.

반복 횟수: 16
Query: 그는 요리하는 방법에 대해서 설명하고 있다.

반복 횟수: 15
Query: 한 남성이 커피 내리는 방식에 대해서 설명하고 있다.


In [159]:
import json
from pathlib import Path
from collections import Counter

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

output_path = training_dir / "minilm_train_pairs_clean.jsonl"

# 기존 train_pairs 사용
query_counts = Counter(
    p["query"].strip()
    for p in train_pairs
)

# Query가 전체 학습 데이터에서 정확히 1번만 등장한 쌍
clean_pairs = [
    p for p in train_pairs
    if query_counts[p["query"].strip()] == 1
]

with output_path.open("w", encoding="utf-8") as f:
    for pair in clean_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print("===== MiniLM Training 중복 Query 정제 =====")
print("기존 학습 쌍:", len(train_pairs))
print("정제 후 학습 쌍:", len(clean_pairs))
print("제외된 학습 쌍:", len(train_pairs) - len(clean_pairs))

print("\n고유 Query 수:", len({
    p["query"].strip() for p in clean_pairs
}))

print("저장 경로:", output_path)
print("파일 크기(MB):", round(output_path.stat().st_size / 1024**2, 2))

===== MiniLM Training 중복 Query 정제 =====
기존 학습 쌍: 15823
정제 후 학습 쌍: 15502
제외된 학습 쌍: 321

고유 Query 수: 15502
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_train_pairs_clean.jsonl
파일 크기(MB): 8.15


In [160]:
import json
from pathlib import Path
from datasets import Dataset

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

train_path = training_dir / "minilm_train_pairs_clean.jsonl"
val_path = training_dir / "minilm_val_pairs.jsonl"

def load_pairs(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

train_clean = load_pairs(train_path)
val_data = load_pairs(val_path)

# Sentence-Transformers 학습에 사용할 두 텍스트
train_dataset = Dataset.from_list([
    {
        "anchor": p["query"],
        "positive": p["positive"]
    }
    for p in train_clean
])

val_dataset = Dataset.from_list([
    {
        "anchor": p["query"],
        "positive": p["positive"]
    }
    for p in val_data
])

print("===== MiniLM Fine-tuning 데이터 로드 =====")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("학습 데이터 열:", train_dataset.column_names)

print("\n===== 첫 번째 학습 예시 =====")
print("Anchor:", train_dataset[0]["anchor"])
print("Positive:", train_dataset[0]["positive"])

assert len(train_dataset) == 15502
assert len(val_dataset) == 1948

print("\n✅ Fine-tuning 데이터 로드 완료!")

===== MiniLM Fine-tuning 데이터 로드 =====
Training: 15502
Validation: 1948
학습 데이터 열: ['anchor', 'positive']

===== 첫 번째 학습 예시 =====
Anchor: 떡볶이에 버섯류가 없는 것이 아쉽고 분모자는 엽떡보다 마라 떡볶이에 들어간 것이 더 맛있었고 먹다 보니까 매웠습니다.
Positive: 아 그리고 버섯류 없는 것도 너무 아쉬워. 음 분모자는 엽떡에 들어가는 것보다 마라 떡볶이에 들어간 게 더 맛있다. 음 기본 오리지널보다 마라맛이 싹 배어있네. 아 이거 마라탕이라 생각했어. 떡을 너무 안 먹었다. 떡볶이지. 음 어 근데 먹다 보니까 맵다. 음 매워요.

✅ Fine-tuning 데이터 로드 완료!


In [161]:
from sentence_transformers import SentenceTransformer
from pathlib import Path
import torch

# 원본 MiniLM을 새로운 객체로 불러오기
finetune_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda"
)

# 기존 Fusion과 동일한 입력 길이 유지
finetune_model.max_seq_length = 128

# 학습 결과 저장 위치 (기존 모델과 별도)
finetune_output_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned"
)

print("===== Fine-tuning용 MiniLM 확인 =====")
print("모델:", finetune_model)
print("입력 토큰 제한:", finetune_model.max_seq_length)
print("사용 장치:", finetune_model.device)
print("저장 예정 경로:", finetune_output_dir)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("남은 GPU 메모리:", round(free / 1024**3, 2), "GB")

print("\n✅ Fine-tuning용 MiniLM 별도 로드 완료!")

===== Fine-tuning용 MiniLM 확인 =====
모델: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
입력 토큰 제한: 128
사용 장치: cuda:0
저장 예정 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned
남은 GPU 메모리: 7.93 GB

✅ Fine-tuning용 MiniLM 별도 로드 완료!


In [162]:
import torch
from sentence_transformers import losses
from sentence_transformers.training_args import (
    SentenceTransformerTrainingArguments,
    BatchSamplers
)

# Positive 쌍을 이용하는 대조학습 Loss
train_loss = losses.MultipleNegativesRankingLoss(
    model=finetune_model
)

# 첫 번째 Fine-tuning 실험 설정
training_args = SentenceTransformerTrainingArguments(
    output_dir=str(finetune_output_dir),

    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    warmup_ratio=0.1,

    fp16=torch.cuda.is_available(),
    bf16=False,

    batch_sampler=BatchSamplers.NO_DUPLICATES,

    save_strategy="no",
    eval_strategy="no",

    logging_steps=100,
    report_to="none",

    seed=42,
    data_seed=42
)

print("===== MiniLM Fine-tuning 설정 =====")
print("Loss: MultipleNegativesRankingLoss")
print("Epoch:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Batch sampler:", training_args.batch_sampler)
print("모델 저장 경로:", training_args.output_dir)

print("\n✅ Fine-tuning 설정 완료!")

===== MiniLM Fine-tuning 설정 =====
Loss: MultipleNegativesRankingLoss
Epoch: 1
Batch size: 8
Gradient accumulation: 2
Learning rate: 2e-05
FP16: True
Batch sampler: BatchSamplers.NO_DUPLICATES
모델 저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned

✅ Fine-tuning 설정 완료!


/tmp/ipykernel_8812/1258967228.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses
/tmp/ipykernel_8812/1258967228.py:3: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import (


In [163]:
import torch
from sentence_transformers import losses
from sentence_transformers.training_args import (
    SentenceTransformerTrainingArguments,
    BatchSamplers
)

# MiniLM 대조학습 Loss
train_loss = losses.MultipleNegativesRankingLoss(
    model=finetune_model
)

# PinPoint MiniLM 1차 Fine-tuning 설정
training_args = SentenceTransformerTrainingArguments(
    output_dir=str(finetune_output_dir),

    # 첫 번째 실험은 1 epoch
    num_train_epochs=1,

    # Tesla T4 메모리를 고려한 배치 설정
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    # 사전학습 모델 추가 학습
    learning_rate=2e-5,
    warmup_ratio=0.1,

    # GPU 메모리 절약
    fp16=True,
    bf16=False,

    # 배치 내 동일 텍스트 중복 방지
    batch_sampler=BatchSamplers.NO_DUPLICATES,

    # 학습 결과 저장
    save_strategy="epoch",
    save_total_limit=1,

    # 평가는 학습 종료 후 별도로 진행
    eval_strategy="no",

    logging_steps=100,
    report_to="none",

    seed=42,
    data_seed=42
)

print("===== PinPoint MiniLM 최종 학습 설정 =====")
print("Loss:", type(train_loss).__name__)
print("Epoch:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("Warmup ratio:", training_args.warmup_ratio)
print("FP16:", training_args.fp16)
print("Batch sampler:", training_args.batch_sampler)
print("Checkpoint 저장:", training_args.save_strategy)
print("출력 폴더:", training_args.output_dir)

print("\n✅ 최종 학습 설정 준비 완료!")

===== PinPoint MiniLM 최종 학습 설정 =====
Loss: MultipleNegativesRankingLoss
Epoch: 1
Batch size: 8
Gradient accumulation: 2
Learning rate: 2e-05
Warmup ratio: 0.1
FP16: True
Batch sampler: BatchSamplers.NO_DUPLICATES
Checkpoint 저장: SaveStrategy.EPOCH
출력 폴더: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned

✅ 최종 학습 설정 준비 완료!


In [164]:
from sentence_transformers import SentenceTransformerTrainer

# Fine-tuning 학습 관리자 생성
trainer = SentenceTransformerTrainer(
    model=finetune_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss
)

print("===== PinPoint MiniLM Trainer 확인 =====")
print("모델:", type(trainer.model).__name__)
print("Loss:", type(train_loss).__name__)
print("Training 데이터:", len(trainer.train_dataset))
print("Epoch:", trainer.args.num_train_epochs)
print("Batch size:", trainer.args.per_device_train_batch_size)
print("저장 경로:", trainer.args.output_dir)

print("\n✅ Trainer 생성 완료!")
print("아직 실제 학습은 시작하지 않았습니다.")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

===== PinPoint MiniLM Trainer 확인 =====
모델: SentenceTransformer
Loss: MultipleNegativesRankingLoss
Training 데이터: 15502
Epoch: 1
Batch size: 8
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned

✅ Trainer 생성 완료!
아직 실제 학습은 시작하지 않았습니다.


# MiniLM Fine-tuning 시작

In [165]:
import time

print("===== PinPoint MiniLM Fine-tuning 시작 =====")

start_time = time.time()

# 실제 MiniLM 추가 학습
train_result = trainer.train()

elapsed = time.time() - start_time

print("\n===== MiniLM Fine-tuning 완료 =====")
print("학습 소요 시간(분):", round(elapsed / 60, 2))
print("Training Loss:", train_result.training_loss)
print("총 학습 Step:", train_result.global_step)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


===== PinPoint MiniLM Fine-tuning 시작 =====


Step,Training Loss
100,0.181200
200,0.109200
300,0.085000
400,0.067900
500,0.047100
600,0.042600
700,0.038100
800,0.061700
900,0.037200



===== MiniLM Fine-tuning 완료 =====
학습 소요 시간(분): 6.51
Training Loss: 0.07302766835357383
총 학습 Step: 969


In [166]:
from pathlib import Path

# 학습된 MiniLM 저장 경로
final_model_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned/final_model"
)

# Fine-tuning된 모델 저장
trainer.model.save_pretrained(str(final_model_dir))

print("===== MiniLM Fine-tuning 모델 저장 =====")
print("저장 경로:", final_model_dir)
print("폴더 존재:", final_model_dir.exists())

print("\n===== 저장된 파일 =====")
for path in sorted(final_model_dir.iterdir()):
    print(path.name)

print("\n✅ Fine-tuning 모델 저장 완료!")

===== MiniLM Fine-tuning 모델 저장 =====
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model
폴더 존재: True

===== 저장된 파일 =====
1_Pooling
README.md
config.json
config_sentence_transformers.json
model.safetensors
modules.json
sentence_bert_config.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json

✅ Fine-tuning 모델 저장 완료!


# 기존 MiniLM vs 학습된 MiniLM 성능 비교

In [167]:
import json
import random
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer, util

# 기존 MiniLM (학습 전)
original_model = fusion_search.get_semantic_model()

# 새로 Fine-tuning한 MiniLM
trained_model = SentenceTransformer(
    str(final_model_dir),
    device="cuda"
)

# 학습 전 Baseline 평가와 동일한 100쌍
rng = random.Random(42)
eval_pairs = rng.sample(val_pairs, 100)

queries = [p["query"] for p in eval_pairs]
documents = [p["positive"] for p in eval_pairs]

def evaluate_minilm(model, queries, documents):
    query_embeddings = model.encode(
        queries,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    doc_embeddings = model.encode(
        documents,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(query_embeddings, doc_embeddings)

    ranks = []

    for i in range(len(queries)):
        ranked_indices = scores[i].argsort(descending=True).tolist()
        ranks.append(ranked_indices.index(i) + 1)

    return {
        "recall_at_1": float(np.mean([r == 1 for r in ranks])),
        "recall_at_5": float(np.mean([r <= 5 for r in ranks])),
        "mrr": float(np.mean([1 / r for r in ranks]))
    }

print("===== 기존 MiniLM 평가 =====")
original_result = evaluate_minilm(
    original_model, queries, documents
)
print(original_result)

print("\n===== Fine-tuning MiniLM 평가 =====")
trained_result = evaluate_minilm(
    trained_model, queries, documents
)
print(trained_result)

print("\n===== 성능 변화 =====")

for metric in ["recall_at_1", "recall_at_5", "mrr"]:
    before = original_result[metric]
    after = trained_result[metric]

    print(
        f"{metric}: {before:.4f} → {after:.4f} "
        f"(변화: {after - before:+.4f})"
    )

# 평가 결과를 Google Drive에 저장
result_path = training_dir / "minilm_finetuning_comparison.json"

result_path.write_text(
    json.dumps({
        "evaluation": "AI-Hub Korean summary-to-transcript retrieval",
        "sample_size": 100,
        "random_seed": 42,
        "original": original_result,
        "finetuned": trained_result
    }, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n✅ 성능 비교 결과 저장 완료:", result_path)

The tokenizer you are loading from '/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


===== 기존 MiniLM 평가 =====
{'recall_at_1': 0.67, 'recall_at_5': 0.84, 'mrr': 0.7545881026774643}

===== Fine-tuning MiniLM 평가 =====
{'recall_at_1': 0.85, 'recall_at_5': 0.93, 'mrr': 0.8812919998277252}

===== 성능 변화 =====
recall_at_1: 0.6700 → 0.8500 (변화: +0.1800)
recall_at_5: 0.8400 → 0.9300 (변화: +0.0900)
mrr: 0.7546 → 0.8813 (변화: +0.1267)

✅ 성능 비교 결과 저장 완료: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuning_comparison.json


In [168]:
# 기존 모델과 Fine-tuning 모델의 tokenizer 비교

original_tokenizer = original_model.tokenizer
trained_tokenizer = trained_model.tokenizer

sample_texts = [
    "두부를 칼로 자르는 장면",
    "밥 한 공기에 참기름과 식초를 넣어 양념을 한다.",
    "여기에다가 참기름 반 스푼 정도만 넣겠습니다."
]

print("===== Tokenizer 기본 정보 =====")
print("기존:", type(original_tokenizer).__name__)
print("학습 후:", type(trained_tokenizer).__name__)
print("기존 vocab 크기:", len(original_tokenizer))
print("학습 후 vocab 크기:", len(trained_tokenizer))

print("\n===== Special Token IDs =====")
print("기존:", original_tokenizer.special_tokens_map)
print("학습 후:", trained_tokenizer.special_tokens_map)

print("\n===== 실제 토큰화 비교 =====")

for text in sample_texts:
    before = original_tokenizer.encode(text)
    after = trained_tokenizer.encode(text)

    print("\n문장:", text)
    print("기존 토큰 수:", len(before))
    print("학습 후 토큰 수:", len(after))
    print("토큰 ID 동일:", before == after)
    print("기존 앞 15개:", before[:15])
    print("학습 후 앞 15개:", after[:15])

===== Tokenizer 기본 정보 =====
기존: PreTrainedTokenizerFast
학습 후: PreTrainedTokenizerFast
기존 vocab 크기: 250002
학습 후 vocab 크기: 250002

===== Special Token IDs =====
기존: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
학습 후: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}

===== 실제 토큰화 비교 =====

문장: 두부를 칼로 자르는 장면
기존 토큰 수: 11
학습 후 토큰 수: 11
토큰 ID 동일: True
기존 앞 15개: [0, 11385, 158721, 6, 76716, 1083, 8511, 8495, 769, 191545, 2]
학습 후 앞 15개: [0, 11385, 158721, 6, 76716, 1083, 8511, 8495, 769, 191545, 2]

문장: 밥 한 공기에 참기름과 식초를 넣어 양념을 한다.
기존 토큰 수: 20
학습 후 토큰 수: 20
토큰 ID 동일: True
기존 앞 15개: [0, 6, 79144, 3103, 206782, 480, 42297, 1503, 28913, 1291, 34788, 27222, 688, 201049, 16837]
학습 후 앞 15개: [0, 6, 79144, 3103, 206782, 480, 42297, 1503, 28913, 1291, 34788, 27222, 688, 201049, 16837]

문장: 여기에다가 참기름 반

In [169]:
import json

# 공식 Validation 전체 1,948쌍
full_queries = [p["query"] for p in val_pairs]
full_documents = [p["positive"] for p in val_pairs]

print("===== 전체 Validation 평가 =====")
print("평가 데이터:", len(val_pairs), "쌍")

# 기존 MiniLM
print("\n[기존 MiniLM]")
full_original = evaluate_minilm(
    original_model,
    full_queries,
    full_documents
)
print(full_original)

# Fine-tuning MiniLM
print("\n[Fine-tuning MiniLM]")
full_trained = evaluate_minilm(
    trained_model,
    full_queries,
    full_documents
)
print(full_trained)

print("\n===== 전체 Validation 성능 변화 =====")

for metric in ["recall_at_1", "recall_at_5", "mrr"]:
    before = full_original[metric]
    after = full_trained[metric]

    print(
        f"{metric}: {before:.4f} → {after:.4f} "
        f"(차이: {after - before:+.4f})"
    )

# Google Drive에 결과 저장
full_result_path = training_dir / "minilm_full_validation_comparison.json"

full_result_path.write_text(
    json.dumps({
        "evaluation": "AI-Hub full validation summary-to-transcript retrieval",
        "sample_size": len(val_pairs),
        "original": full_original,
        "finetuned": full_trained
    }, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n✅ 전체 Validation 평가 결과 저장 완료!")
print("경로:", full_result_path)

===== 전체 Validation 평가 =====
평가 데이터: 1948 쌍

[기존 MiniLM]
{'recall_at_1': 0.5102669404517454, 'recall_at_5': 0.6555441478439425, 'mrr': 0.577302482579806}

[Fine-tuning MiniLM]
{'recall_at_1': 0.7515400410677618, 'recall_at_5': 0.858829568788501, 'mrr': 0.7996197008399655}

===== 전체 Validation 성능 변화 =====
recall_at_1: 0.5103 → 0.7515 (차이: +0.2413)
recall_at_5: 0.6555 → 0.8588 (차이: +0.2033)
mrr: 0.5773 → 0.7996 (차이: +0.2223)

✅ 전체 Validation 평가 결과 저장 완료!
경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_full_validation_comparison.json


In [170]:
from sentence_transformers import util

# 기존 다이소 영상 자막
subtitles = daiso_subtitles

query = "여행용 시트 세제를 보여주는 장면"

# 기존 MiniLM과 Fine-tuning MiniLM
models = {
    "기존 MiniLM": original_model,
    "Fine-tuning MiniLM": trained_model
}

texts = [s["text"] for s in subtitles]

print("===== 다이소 한국어 자막 검색 비교 =====")
print("검색어:", query)

for model_name, model in models.items():

    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    subtitle_embeddings = model.encode(
        texts,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        query_embedding,
        subtitle_embeddings
    )[0].tolist()

    ranked = sorted(
        zip(subtitles, scores),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\n===== {model_name} =====")

    for i, (subtitle, score) in enumerate(ranked[:5], 1):
        print(
            f"{i}위 | "
            f"{subtitle['start_time']:.2f}~"
            f"{subtitle['end_time']:.2f}초 "
            f"| 유사도={score:.4f}"
        )
        print("자막:", subtitle["text"])

===== 다이소 한국어 자막 검색 비교 =====
검색어: 여행용 시트 세제를 보여주는 장면

===== 기존 MiniLM =====
1위 | 3.08~4.68초 | 유사도=0.4230
자막: 여행 갈 때 무조건 챙기는 파우치
2위 | 22.68~24.48초 | 유사도=0.3273
자막: 탱글티즈랑 비슷한 비슷한 요거 두 개
3위 | 20.60~22.68초 | 유사도=0.3216
자막: 다이소 립 중에는 제일 잘 쓰게 되더라고요
4위 | 13.72~15.16초 | 유사도=0.3213
자막: 빨리 편하게 해주는 시트 세제
5위 | 26.80~29.00초 | 유사도=0.3188
자막: 집에 두면 한층 더 귀여워지는 인테리어템

===== Fine-tuning MiniLM =====
1위 | 13.72~15.16초 | 유사도=0.5068
자막: 빨리 편하게 해주는 시트 세제
2위 | 3.08~4.68초 | 유사도=0.2796
자막: 여행 갈 때 무조건 챙기는 파우치
3위 | 24.48~26.80초 | 유사도=0.2318
자막: 현상모듣게 잘 빗겨서 여행 갈 때 꼭 챙겨가요
4위 | 20.60~22.68초 | 유사도=0.2291
자막: 다이소 립 중에는 제일 잘 쓰게 되더라고요
5위 | 31.08~32.08초 | 유사도=0.1926
자막: 이건 진짜 잘 썬데


# 기존 Fusion 코드의 MiniLM 연결 방식 확인

In [171]:
from pathlib import Path

fusion_path = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

code = fusion_path.read_text(encoding="utf-8")
lines = code.splitlines()

keywords = [
    "def get_semantic_model",
    "def get_multilingual_subtitle_candidates",
    "def predict_moments_fusion",
    "get_semantic_model(",
    "model.encode("
]

print("===== Fusion 코드의 MiniLM 연결 위치 =====")

for i, line in enumerate(lines, 1):
    if any(keyword in line for keyword in keywords):
        print(f"{i:4d}: {line.strip()}")

print("\nFusion 파일 존재:", fusion_path.exists())
print("전체 코드 줄 수:", len(lines))

===== Fusion 코드의 MiniLM 연결 위치 =====
   6: def get_semantic_model():
  50: embeddings = model.encode(
  55: query_embedding = model.encode(
 107: def get_multilingual_subtitle_candidates(
 186: def predict_moments_fusion(
 210: model=get_semantic_model(),

Fusion 파일 존재: True
전체 코드 줄 수: 242


In [172]:
from pathlib import Path

fusion_path = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

lines = fusion_path.read_text(encoding="utf-8").splitlines()

print("===== 1. MiniLM 로드 함수 =====")
for i in range(0, min(25, len(lines))):
    print(f"{i+1:3d}: {lines[i]}")

print("\n===== 2. Fusion 검색 함수 =====")
for i in range(185, min(242, len(lines))):
    print(f"{i+1:3d}: {lines[i]}")

===== 1. MiniLM 로드 함수 =====
  1: 
  2: from functools import lru_cache
  3: from sentence_transformers import SentenceTransformer
  4: 
  5: @lru_cache(maxsize=1)
  6: def get_semantic_model():
  7:     return SentenceTransformer(
  8:         "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
  9:     )
 10: 
 11: from sentence_transformers import util
 12: from ai.features.run_moment_detr_inference import predict_moments_hybrid
 13: 
 14: # temporal_iou는 아래에 별도로 정의
 15: 
 16: 
 17: def temporal_iou(a, b):
 18:     intersection = max(
 19:         0,
 20:         min(a["end_time"], b["end_time"])
 21:         - max(a["start_time"], b["start_time"])
 22:     )
 23: 
 24:     union = (
 25:         max(a["end_time"], b["end_time"])

===== 2. Fusion 검색 함수 =====
186: def predict_moments_fusion(
187:     video_id, feature_path, subtitles,
188:     korean_query, english_query,
189:     video_duration, subtitle_json_path=None, top_k=5
190: ):
191:     visual = [
192:         dict(

In [173]:
from unittest.mock import patch

print("===== Fine-tuning MiniLM Fusion 테스트 =====")

with patch.object(
    fusion_search,
    "get_semantic_model",
    return_value=trained_model
):
    improved_fusion_results = fusion_search.predict_moments_fusion(
        video_id="kr_daiso_001",
        feature_path="data/features/kr_daiso_001_clip_features.pt",
        subtitles=daiso_subtitles,
        korean_query="여행용 시트 세제를 보여주는 장면",
        english_query="showing travel laundry detergent sheets",
        video_duration=32.07,
        subtitle_json_path="data/processed/kr_daiso_001/subtitles.json",
        top_k=5
    )

print("\n===== 개선 Fusion Top-5 =====")

for i, result in enumerate(improved_fusion_results, 1):
    print(
        f"{i}위 | "
        f"{result['start_time']:.2f}~"
        f"{result['end_time']:.2f}초 "
        f"| source={result.get('source')}"
    )

print("\n✅ 기존 Fusion 파일 수정 없이 테스트 완료!")

===== Fine-tuning MiniLM Fusion 테스트 =====

===== 개선 Fusion Top-5 =====
1위 | 14.50~24.51초 | source=hybrid
2위 | 13.72~15.16초 | source=subtitle
3위 | 22.68~26.80초 | source=subtitle
4위 | 13.72~17.08초 | source=subtitle
5위 | 2.50~14.72초 | source=hybrid

✅ 기존 Fusion 파일 수정 없이 테스트 완료!


In [174]:
print("===== 기존 MiniLM Fusion 테스트 =====")

original_fusion_results = fusion_search.predict_moments_fusion(
    video_id="kr_daiso_001",
    feature_path="data/features/kr_daiso_001_clip_features.pt",
    subtitles=daiso_subtitles,
    korean_query="여행용 시트 세제를 보여주는 장면",
    english_query="showing travel laundry detergent sheets",
    video_duration=32.07,
    subtitle_json_path="data/processed/kr_daiso_001/subtitles.json",
    top_k=5
)

ground_truth = {
    "start_time": 13.0,
    "end_time": 18.0
}

print("\n===== 기존 Fusion vs 개선 Fusion =====")

for label, results in [
    ("기존 Fusion", original_fusion_results),
    ("Fine-tuning Fusion", improved_fusion_results)
]:
    print(f"\n[{label}]")

    ious = []

    for i, result in enumerate(results, 1):
        iou = fusion_search.temporal_iou(result, ground_truth)
        ious.append(iou)

        print(
            f"{i}위 | "
            f"{result['start_time']:.2f}~"
            f"{result['end_time']:.2f}초 "
            f"| source={result.get('source')} "
            f"| IoU={iou:.4f}"
        )

    print("Top-1 IoU:", round(ious[0], 4))
    print("Top-5 최고 IoU:", round(max(ious), 4))

===== 기존 MiniLM Fusion 테스트 =====

===== 기존 Fusion vs 개선 Fusion =====

[기존 Fusion]
1위 | 14.50~24.51초 | source=hybrid | IoU=0.3040
2위 | 3.08~4.68초 | source=subtitle | IoU=0.0000
3위 | 22.68~29.00초 | source=subtitle | IoU=0.0000
4위 | 3.08~8.40초 | source=subtitle | IoU=0.0000
5위 | 20.60~24.48초 | source=subtitle | IoU=0.0000
Top-1 IoU: 0.304
Top-5 최고 IoU: 0.304

[Fine-tuning Fusion]
1위 | 14.50~24.51초 | source=hybrid | IoU=0.3040
2위 | 13.72~15.16초 | source=subtitle | IoU=0.2880
3위 | 22.68~26.80초 | source=subtitle | IoU=0.0000
4위 | 13.72~17.08초 | source=subtitle | IoU=0.6720
5위 | 2.50~14.72초 | source=hybrid | IoU=0.1109
Top-1 IoU: 0.304
Top-5 최고 IoU: 0.672


In [175]:
from unittest.mock import patch

# 다이소 파우치 검색어
korean_query = "여행용 화장품 파우치를 보여주는 장면"
english_query = "showing a travel makeup pouch"

# 기존 평가에서 사용한 임시 정답 구간
ground_truth = {
    "start_time": 3.08,
    "end_time": 8.40
}

search_args = {
    "video_id": "kr_daiso_001",
    "feature_path": "data/features/kr_daiso_001_clip_features.pt",
    "subtitles": daiso_subtitles,
    "korean_query": korean_query,
    "english_query": english_query,
    "video_duration": 32.07,
    "subtitle_json_path": "data/processed/kr_daiso_001/subtitles.json",
    "top_k": 5
}

print("===== 다이소 파우치 검색 비교 =====")

for label, model in [
    ("기존 Fusion", original_model),
    ("Fine-tuning Fusion", trained_model)
]:
    with patch.object(
        fusion_search,
        "get_semantic_model",
        return_value=model
    ):
        results = fusion_search.predict_moments_fusion(**search_args)

    ious = [
        fusion_search.temporal_iou(r, ground_truth)
        for r in results
    ]

    print(f"\n[{label}]")

    for i, (result, iou) in enumerate(zip(results, ious), 1):
        print(
            f"{i}위 | "
            f"{result['start_time']:.2f}~"
            f"{result['end_time']:.2f}초 "
            f"| source={result.get('source')} "
            f"| IoU={iou:.4f}"
        )

    print("Top-1 IoU:", round(ious[0], 4))
    print("Top-5 최고 IoU:", round(max(ious), 4))

===== 다이소 파우치 검색 비교 =====

[기존 Fusion]
1위 | 14.50~25.01초 | source=hybrid | IoU=0.0000
2위 | 2.08~7.28초 | source=subtitle | IoU=0.6646
3위 | 24.50~28.50초 | source=hybrid | IoU=0.0000
4위 | 4.68~10.00초 | source=subtitle | IoU=0.5376
Top-1 IoU: 0.0
Top-5 최고 IoU: 0.6646

[Fine-tuning Fusion]
1위 | 14.50~25.01초 | source=hybrid | IoU=0.0000
2위 | 2.08~7.28초 | source=subtitle | IoU=0.6646
3위 | 24.50~28.50초 | source=hybrid | IoU=0.0000
4위 | 4.68~8.40초 | source=subtitle | IoU=0.6992
Top-1 IoU: 0.0
Top-5 최고 IoU: 0.6992


In [176]:
print("===== 기존 한국어 영상 테스트 관련 변수 확인 =====")

keywords = [
    "tofu", "kitchen", "daiso",
    "test_cases", "ground_truth",
    "eval_cases", "benchmark"
]

matches = []

for name, value in globals().copy().items():
    if name.startswith("_"):
        continue

    if any(keyword in name.lower() for keyword in keywords):
        matches.append((name, type(value).__name__))

for name, value_type in sorted(matches):
    print(f"{name}: {value_type}")

print("\n✅ 변수 확인 완료")

===== 기존 한국어 영상 테스트 관련 변수 확인 =====
daiso_results: list
daiso_subtitles: list
detergent_ground_truth: dict
ground_truth: dict
kitchen_all_subtitle: list
kitchen_en: list
kitchen_hybrid: dict
kitchen_ko: list
kitchen_results: list
kitchen_subtitle: dict
kitchen_subtitles: list
kitchen_visual: list
rice_ground_truth: dict
seasoning_ground_truth: dict
test_cases: list
tofu_results: list
tofu_subtitles: list

✅ 변수 확인 완료


In [177]:
from pprint import pprint

print("===== 기존 PinPoint 테스트 케이스 확인 =====")
print("test_cases 자료형:", type(test_cases).__name__)
print("테스트 케이스 개수:", len(test_cases))

for i, case in enumerate(test_cases, 1):
    print(f"\n===== 테스트 케이스 {i} =====")
    pprint(case, sort_dicts=False, depth=4, width=120)

print("\n✅ 테스트 설정 확인 완료!")

===== 기존 PinPoint 테스트 케이스 확인 =====
test_cases 자료형: list
테스트 케이스 개수: 6

===== 테스트 케이스 1 =====
{'name': '두부 자르기',
 'query_ko': '두부를 칼로 자르는 장면',
 'query_en': 'cutting tofu with a knife',
 'subtitles': [{'start_time': 0.0, 'end_time': 4.74, 'text': '여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면'},
               {'start_time': 4.74, 'end_time': 8.7, 'text': '순식간에 정말 맛있는 두부 요리 하나가 완성돼요'},
               {'start_time': 8.7, 'end_time': 12.88, 'text': '고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요'},
               {'start_time': 12.88, 'end_time': 16.32, 'text': '먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요'},
               {'start_time': 16.32, 'end_time': 19.02, 'text': '그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요'},
               {'start_time': 19.02, 'end_time': 21.06, 'text': '두께는 젓가락 하나 정도면 딱 좋아요'},
               {'start_time': 21.06, 'end_time': 23.72, 'text': '다 썰었으면 접시에 담아 잠시 놔둘게요'},
               {'start_time': 23.72, 'end_time': 26.46, 'text': '제가 산두부는 한 모에 600번 정도 줬는데요'},
               {'start_time': 26.46, 'end_time': 

In [178]:
from pathlib import Path

print("===== PinPoint 한국어 영상 Feature 파일 확인 =====")

video_ids = [
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_daiso_001"
]

for video_id in video_ids:
    feature_path = Path(
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    subtitle_path = Path(
        f"/content/pinpoint-capstone/data/processed/"
        f"{video_id}/subtitles.json"
    )

    print(f"\n영상: {video_id}")
    print("Feature 파일:", feature_path.exists())
    print("자막 JSON:", subtitle_path.exists())

    if feature_path.exists():
        print("Feature 경로:", feature_path)
    if subtitle_path.exists():
        print("자막 경로:", subtitle_path)

print("\n✅ 파일 확인 완료!")

===== PinPoint 한국어 영상 Feature 파일 확인 =====

영상: kr_tofu_001
Feature 파일: True
자막 JSON: False
Feature 경로: /content/pinpoint-capstone/data/features/kr_tofu_001_clip_features.pt

영상: kr_kitchen_001
Feature 파일: True
자막 JSON: False
Feature 경로: /content/pinpoint-capstone/data/features/kr_kitchen_001_clip_features.pt

영상: kr_daiso_001
Feature 파일: True
자막 JSON: True
Feature 경로: /content/pinpoint-capstone/data/features/kr_daiso_001_clip_features.pt
자막 경로: /content/pinpoint-capstone/data/processed/kr_daiso_001/subtitles.json

✅ 파일 확인 완료!


# 두부·주방 Hybrid 실행 가능 여부 확인

In [179]:
from pathlib import Path
from ai.features.run_moment_detr_inference import predict_moments_hybrid

print("===== 두부·주방 Hybrid 검색 확인 =====")

test_videos = [
    {
        "video_id": "kr_tofu_001",
        "query": "cutting tofu with a knife",
        "duration": 164.98
    },
    {
        "video_id": "kr_kitchen_001",
        "query": "squeezing sauce with scissors",
        "duration": 37.08
    }
]

for video in test_videos:
    video_id = video["video_id"]

    feature_path = (
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    print(f"\n===== {video_id} =====")

    try:
        results = predict_moments_hybrid(
            video_id=video_id,
            video_feature_path=feature_path,
            query=video["query"],
            subtitle_json_path=None,
            video_duration=video["duration"],
            top_k=5
        )

        print("실행 성공!")
        print("반환 후보 수:", len(results))

        for i, result in enumerate(results[:3], 1):
            print(
                f"{i}위:",
                round(result["start_time"], 2),
                "~",
                round(result["end_time"], 2),
                "초"
            )

    except Exception as e:
        print("실행 오류:", type(e).__name__)
        print("오류 내용:", str(e)[:500])

print("\n✅ Hybrid 실행 확인 종료")

===== 두부·주방 Hybrid 검색 확인 =====

===== kr_tofu_001 =====
실행 성공!
반환 후보 수: 5
1위: 98.5 ~ 118.96 초
2위: 84.5 ~ 102.78 초
3위: 114.5 ~ 135.89 초

===== kr_kitchen_001 =====
실행 성공!
반환 후보 수: 3
1위: 2.5 ~ 10.27 초
2위: 20.5 ~ 31.68 초
3위: 10.5 ~ 20.5 초

✅ Hybrid 실행 확인 종료


In [180]:
from pathlib import Path
from unittest.mock import patch
import numpy as np
import json

# 영상별 설정
video_config = {
    "두부": {
        "video_id": "kr_tofu_001",
        "duration": 164.98
    },
    "주방": {
        "video_id": "kr_kitchen_001",
        "duration": 37.08
    },
    "다이소": {
        "video_id": "kr_daiso_001",
        "duration": 32.08
    }
}

# 기존 모델과 Fine-tuning 모델
models = {
    "기존 Fusion": original_model,
    "Fine-tuning Fusion": trained_model
}

all_results = []

print("===== PinPoint 한국어 영상 6개 검색어 비교 =====")

for case in test_cases:

    name = case["name"]

    if name.startswith("두부"):
        config = video_config["두부"]
    elif name in ("가위로 소스 짜기", "쌀 씻기"):
        config = video_config["주방"]
    else:
        config = video_config["다이소"]

    video_id = config["video_id"]

    feature_path = (
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        f"/content/pinpoint-capstone/data/processed/"
        f"{video_id}/subtitles.json"
    )

    if not Path(subtitle_path).exists():
        subtitle_path = None

    gt = case["ground_truth"]

    print(f"\n[{name}]")

    for model_name, model in models.items():

        with patch.object(
            fusion_search,
            "get_semantic_model",
            return_value=model
        ):
            results = fusion_search.predict_moments_fusion(
                video_id=video_id,
                feature_path=feature_path,
                subtitles=case["subtitles"],
                korean_query=case["query_ko"],
                english_query=case["query_en"],
                video_duration=config["duration"],
                subtitle_json_path=subtitle_path,
                top_k=5
            )

        ious = [
            fusion_search.temporal_iou(r, gt)
            for r in results
        ]

        top1 = ious[0] if ious else 0.0
        top5 = max(ious) if ious else 0.0

        all_results.append({
            "name": name,
            "model": model_name,
            "top1_iou": top1,
            "top5_best_iou": top5,
            "recall_at_1": int(top1 >= 0.5),
            "recall_at_5": int(top5 >= 0.5),
            "candidates": results
        })

        print(
            f"  {model_name}: "
            f"Top-1 IoU={top1:.4f}, "
            f"Top-5 최고 IoU={top5:.4f}"
        )

print("\n===== 전체 평균 성능 =====")

summary = {}

for model_name in models:

    rows = [
        r for r in all_results
        if r["model"] == model_name
    ]

    summary[model_name] = {
        "mean_top1_iou": float(np.mean([
            r["top1_iou"] for r in rows
        ])),
        "mean_top5_best_iou": float(np.mean([
            r["top5_best_iou"] for r in rows
        ])),
        "recall_at_1_iou05": float(np.mean([
            r["recall_at_1"] for r in rows
        ])),
        "recall_at_5_iou05": float(np.mean([
            r["recall_at_5"] for r in rows
        ]))
    }

    print(f"\n[{model_name}]")
    for metric, value in summary[model_name].items():
        print(f"{metric}: {value:.4f}")

# 평가 결과 저장
output_path = (
    training_dir /
    "minilm_fusion_6queries_comparison.json"
)

output_path.write

===== PinPoint 한국어 영상 6개 검색어 비교 =====

[두부 자르기]
  기존 Fusion: Top-1 IoU=0.0000, Top-5 최고 IoU=0.7436
  Fine-tuning Fusion: Top-1 IoU=0.0000, Top-5 최고 IoU=0.3127

[두부 양념 준비]
  기존 Fusion: Top-1 IoU=0.5164, Top-5 최고 IoU=0.5164
  Fine-tuning Fusion: Top-1 IoU=0.5164, Top-5 최고 IoU=0.5164

[가위로 소스 짜기]
  기존 Fusion: Top-1 IoU=0.4883, Top-5 최고 IoU=0.4883
  Fine-tuning Fusion: Top-1 IoU=0.4883, Top-5 최고 IoU=0.4883

[쌀 씻기]
  기존 Fusion: Top-1 IoU=0.2154, Top-5 최고 IoU=0.8133
  Fine-tuning Fusion: Top-1 IoU=0.2154, Top-5 최고 IoU=0.8133

[다이소 파우치]
  기존 Fusion: Top-1 IoU=0.0000, Top-5 최고 IoU=0.6646
  Fine-tuning Fusion: Top-1 IoU=0.0000, Top-5 최고 IoU=0.6992

[다이소 시트 세제]
  기존 Fusion: Top-1 IoU=0.3038, Top-5 최고 IoU=0.3038
  Fine-tuning Fusion: Top-1 IoU=0.3038, Top-5 최고 IoU=0.6720

===== 전체 평균 성능 =====

[기존 Fusion]
mean_top1_iou: 0.2540
mean_top5_best_iou: 0.5883
recall_at_1_iou05: 0.1667
recall_at_5_iou05: 0.6667

[Fine-tuning Fusion]
mean_top1_iou: 0.2540
mean_top5_best_iou: 0.5837
recall_at_1_iou05: 0.1

AttributeError: 'PosixPath' object has no attribute 'write'

In [181]:
import json
from pathlib import Path

output_path = (
    training_dir /
    "minilm_fusion_6queries_comparison.json"
)

# 이미 계산된 결과 저장
output_path.write_text(
    json.dumps({
        "test_count": len(test_cases),
        "summary": summary,
        "results": all_results
    }, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("===== Fusion 평가 결과 저장 확인 =====")
print("저장 성공:", output_path.exists())
print("저장 경로:", output_path)
print("파일 크기:", output_path.stat().st_size, "bytes")

print("\n✅ 한국어 영상 6개 검색어 평가 결과 저장 완료!")

===== Fusion 평가 결과 저장 확인 =====
저장 성공: True
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_fusion_6queries_comparison.json
파일 크기: 24518 bytes

✅ 한국어 영상 6개 검색어 평가 결과 저장 완료!


In [182]:
print("===== 두부 자르기 Fusion 후보 비교 =====")

gt = test_cases[0]["ground_truth"]

for model_name in ["기존 Fusion", "Fine-tuning Fusion"]:

    row = next(
        r for r in all_results
        if r["name"] == "두부 자르기"
        and r["model"] == model_name
    )

    print(f"\n===== {model_name} =====")

    for i, result in enumerate(row["candidates"], 1):

        iou = fusion_search.temporal_iou(result, gt)

        print(
            f"{i}위 | "
            f"{result['start_time']:.2f}~"
            f"{result['end_time']:.2f}초 "
            f"| source={result.get('source')} "
            f"| IoU={iou:.4f}"
        )

        # 자막 후보의 텍스트 및 언어 확인
        for key in ["language", "text", "transcript", "score"]:
            if key in result:
                print(f"   {key}: {str(result[key])[:150]}")

print("\n✅ 두부 자르기 후보 비교 완료!")

===== 두부 자르기 Fusion 후보 비교 =====

===== 기존 Fusion =====
1위 | 98.50~118.96초 | source=hybrid | IoU=0.0000
   language: None
   score: 0.9842
2위 | 140.86~143.58초 | source=subtitle | IoU=0.0000
   language: ko
   score: None
3위 | 12.88~21.06초 | source=subtitle | IoU=0.7436
   language: en
   score: None
4위 | 84.50~102.78초 | source=hybrid | IoU=0.0000
   language: None
   score: 0.9608
5위 | 135.06~137.58초 | source=subtitle | IoU=0.0000
   language: ko
   score: None

===== Fine-tuning Fusion =====
1위 | 98.50~118.96초 | source=hybrid | IoU=0.0000
   language: None
   score: 0.9842
2위 | 4.74~8.70초 | source=subtitle | IoU=0.0000
   language: ko
   score: None
3위 | 12.88~16.32초 | source=subtitle | IoU=0.3127
   language: en
   score: None
4위 | 84.50~102.78초 | source=hybrid | IoU=0.0000
   language: None
   score: 0.9608
5위 | 23.72~26.46초 | source=subtitle | IoU=0.0000
   language: ko
   score: None

✅ 두부 자르기 후보 비교 완료!


In [183]:
print("===== 두부 자르기 MiniLM 원본 후보 비교 =====")

case = test_cases[0]
gt = case["ground_truth"]

for label, model in [
    ("기존 MiniLM", original_model),
    ("Fine-tuning MiniLM", trained_model)
]:

    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=case["subtitles"],
        model=model,
        top_k=10
    )

    print(f"\n===== {label} =====")
    print("생성된 후보 수:", len(candidates))

    for i, candidate in enumerate(candidates, 1):
        iou = fusion_search.temporal_iou(candidate, gt)

        print(
            f"{i}위 | "
            f"{candidate['start_time']:.2f}~"
            f"{candidate['end_time']:.2f}초 "
            f"| language={candidate.get('language')} "
            f"| IoU={iou:.4f}"
        )

        # 후보에 점수가 있으면 함께 출력
        for key in ["similarity", "semantic_score", "score"]:
            if candidate.get(key) is not None:
                print(f"   {key}: {candidate[key]}")

===== 두부 자르기 MiniLM 원본 후보 비교 =====

===== 기존 MiniLM =====
생성된 후보 수: 20
1위 | 140.86~143.58초 | language=ko | IoU=0.0000
   score: 0.6732
2위 | 135.06~137.58초 | language=ko | IoU=0.0000
   score: 0.6575
3위 | 97.94~101.62초 | language=ko | IoU=0.0000
   score: 0.6547
4위 | 137.58~143.58초 | language=ko | IoU=0.0000
   score: 0.6458
5위 | 16.32~19.02초 | language=ko | IoU=0.2455
   score: 0.645
6위 | 12.88~16.32초 | language=ko | IoU=0.3127
   score: 0.6442
7위 | 137.58~140.86초 | language=ko | IoU=0.0000
   score: 0.6435
8위 | 135.06~140.86초 | language=ko | IoU=0.0000
   score: 0.6338
9위 | 78.78~81.42초 | language=ko | IoU=0.0000
   score: 0.6326
10위 | 21.06~23.72초 | language=ko | IoU=0.1655
   score: 0.6256
11위 | 12.88~21.06초 | language=en | IoU=0.7436
   score: 0.5538
12위 | 8.70~19.02초 | language=en | IoU=0.4909
   score: 0.5509
13위 | 8.70~16.32초 | language=en | IoU=0.3021
   score: 0.5087
14위 | 4.74~16.32초 | language=en | IoU=0.2366
   score: 0.5072
15위 | 12.88~19.02초 | language=en | IoU=0.5582
   

In [184]:
from pathlib import Path

fusion_path = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

lines = fusion_path.read_text(encoding="utf-8").splitlines()

print("===== 자막 후보 생성 및 점수 계산 코드 =====")

for i in range(28, min(186, len(lines))):
    print(f"{i+1:3d}: {lines[i]}")

===== 자막 후보 생성 및 점수 계산 코드 =====
 29:     return intersection / union if union > 0 else 0
 30: 
 31: 
 32: def get_semantic_subtitle_candidates(
 33:     query,
 34:     subtitles,
 35:     model,
 36:     top_k=5
 37: ):
 38:     windows = []
 39: 
 40:     for size in (1, 2, 3):
 41:         for i in range(len(subtitles) - size + 1):
 42:             group = subtitles[i:i + size]
 43: 
 44:             windows.append({
 45:                 "start_time": group[0]["start_time"],
 46:                 "end_time": group[-1]["end_time"],
 47:                 "text": " ".join(s["text"] for s in group)
 48:             })
 49: 
 50:     embeddings = model.encode(
 51:         [w["text"] for w in windows],
 52:         convert_to_tensor=True
 53:     )
 54: 
 55:     query_embedding = model.encode(
 56:         query,
 57:         convert_to_tensor=True
 58:     )
 59: 
 60:     scores = util.cos_sim(
 61:         query_embedding,
 62:         embeddings
 63:     )[0]
 64: 
 65:     candidates

In [185]:
case = test_cases[0]

# 기존 Hybrid 후보
hybrid_candidates = predict_moments_hybrid(
    video_id="kr_tofu_001",
    video_feature_path=(
        "/content/pinpoint-capstone/data/features/"
        "kr_tofu_001_clip_features.pt"
    ),
    query=case["query_en"],
    subtitle_json_path=None,
    video_duration=164.98,
    top_k=5
)

# Fine-tuning MiniLM 후보
subtitle_candidates = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=case["query_ko"],
    english_query=case["query_en"],
    subtitles=case["subtitles"],
    model=trained_model,
    top_k=5
)

print("===== Hybrid vs Fine-tuning MiniLM 후보 겹침 =====")

for i, hybrid in enumerate(hybrid_candidates, 1):
    overlaps = []

    for subtitle in subtitle_candidates:
        iou = fusion_search.temporal_iou(hybrid, subtitle)

        if iou >= 0.3:
            overlaps.append({
                "start": subtitle["start_time"],
                "end": subtitle["end_time"],
                "language": subtitle["language"],
                "iou": round(iou, 4)
            })

    print(
        f"\nHybrid {i}위: "
        f"{hybrid['start_time']:.2f}~"
        f"{hybrid['end_time']:.2f}초"
    )
    print("IoU 0.3 이상 겹치는 자막 후보:", overlaps)

print("\n✅ 후보 중복 확인 완료!")

===== Hybrid vs Fine-tuning MiniLM 후보 겹침 =====

Hybrid 1위: 98.50~118.96초
IoU 0.3 이상 겹치는 자막 후보: []

Hybrid 2위: 84.50~102.78초
IoU 0.3 이상 겹치는 자막 후보: []

Hybrid 3위: 114.50~135.89초
IoU 0.3 이상 겹치는 자막 후보: []

Hybrid 4위: 38.50~49.10초
IoU 0.3 이상 겹치는 자막 후보: []

Hybrid 5위: 142.50~163.85초
IoU 0.3 이상 겹치는 자막 후보: []

✅ 후보 중복 확인 완료!


# 한국어 vs 영어 MiniLM 검색 성능 비교

In [186]:
import numpy as np

print("===== Fine-tuning MiniLM 언어별 검색 평가 =====")

language_results = {"ko": [], "en": []}

for case in test_cases:
    print(f"\n[{case['name']}]")

    for language, query in [
        ("ko", case["query_ko"]),
        ("en", case["query_en"])
    ]:
        candidates = fusion_search.get_semantic_subtitle_candidates(
            query=query,
            subtitles=case["subtitles"],
            model=trained_model,
            top_k=5
        )

        ious = [
            fusion_search.temporal_iou(
                candidate,
                case["ground_truth"]
            )
            for candidate in candidates
        ]

        top1 = ious[0] if ious else 0.0
        top5 = max(ious) if ious else 0.0

        language_results[language].append({
            "name": case["name"],
            "top1_iou": top1,
            "top5_best_iou": top5
        })

        print(
            f"  {language}: "
            f"Top-1 IoU={top1:.4f}, "
            f"Top-5 최고 IoU={top5:.4f}"
        )

print("\n===== 언어별 평균 =====")

for language, rows in language_results.items():
    mean_top1 = np.mean([r["top1_iou"] for r in rows])
    mean_top5 = np.mean([r["top5_best_iou"] for r in rows])

    print(
        f"{language}: "
        f"평균 Top-1 IoU={mean_top1:.4f}, "
        f"평균 Top-5 최고 IoU={mean_top5:.4f}"
    )

print("\n✅ 언어별 검색 평가 완료!")

===== Fine-tuning MiniLM 언어별 검색 평가 =====

[두부 자르기]
  ko: Top-1 IoU=0.0000, Top-5 최고 IoU=0.1342
  en: Top-1 IoU=0.3127, Top-5 최고 IoU=0.5582

[두부 양념 준비]
  ko: Top-1 IoU=0.0000, Top-5 최고 IoU=0.0000
  en: Top-1 IoU=0.1190, Top-5 최고 IoU=0.1800

[가위로 소스 짜기]
  ko: Top-1 IoU=0.3955, Top-5 최고 IoU=1.0000
  en: Top-1 IoU=0.0000, Top-5 최고 IoU=0.0000

[쌀 씻기]
  ko: Top-1 IoU=0.8133, Top-5 최고 IoU=0.8133
  en: Top-1 IoU=0.8133, Top-5 최고 IoU=0.8133

[다이소 파우치]
  ko: Top-1 IoU=0.6646, Top-5 최고 IoU=1.0000
  en: Top-1 IoU=1.0000, Top-5 최고 IoU=1.0000

[다이소 시트 세제]
  ko: Top-1 IoU=0.2880, Top-5 최고 IoU=0.7039
  en: Top-1 IoU=0.0000, Top-5 최고 IoU=0.0000

===== 언어별 평균 =====
ko: 평균 Top-1 IoU=0.3602, 평균 Top-5 최고 IoU=0.6086
en: 평균 Top-1 IoU=0.3742, 평균 Top-5 최고 IoU=0.4253

✅ 언어별 검색 평가 완료!


In [187]:
import numpy as np

print("===== 한국어 + 영어 MiniLM 후보 통합 평가 =====")

combined_results = []

for case in test_cases:
    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    ious = [
        fusion_search.temporal_iou(candidate, case["ground_truth"])
        for candidate in candidates
    ]

    best_index = int(np.argmax(ious))
    best_iou = ious[best_index]
    best_candidate = candidates[best_index]

    combined_results.append(best_iou)

    print(f"\n[{case['name']}]")
    print("후보 수:", len(candidates))
    print("후보 10개 최고 IoU:", round(best_iou, 4))
    print(
        "최고 후보:",
        f"{best_candidate['start_time']:.2f}~"
        f"{best_candidate['end_time']:.2f}초",
        "| 언어:", best_candidate["language"]
    )

print("\n===== 전체 평균 =====")
print("평균 최고 IoU:", round(float(np.mean(combined_results)), 4))
print(
    "Recall@10 (IoU≥0.5):",
    round(float(np.mean(np.array(combined_results) >= 0.5)), 4)
)

print("\n✅ 한국어·영어 후보 통합 평가 완료!")

===== 한국어 + 영어 MiniLM 후보 통합 평가 =====

[두부 자르기]
후보 수: 10
후보 10개 최고 IoU: 0.5582
최고 후보: 12.88~19.02초 | 언어: en

[두부 양념 준비]
후보 수: 10
후보 10개 최고 IoU: 0.18
최고 후보: 108.82~116.02초 | 언어: en

[가위로 소스 짜기]
후보 수: 10
후보 10개 최고 IoU: 1.0
최고 후보: 5.96~11.32초 | 언어: ko

[쌀 씻기]
후보 수: 10
후보 10개 최고 IoU: 0.8133
최고 후보: 29.28~31.72초 | 언어: ko

[다이소 파우치]
후보 수: 10
후보 10개 최고 IoU: 1.0
최고 후보: 3.08~8.40초 | 언어: ko

[다이소 시트 세제]
후보 수: 10
후보 10개 최고 IoU: 0.7039
최고 후보: 13.72~19.08초 | 언어: ko

===== 전체 평균 =====
평균 최고 IoU: 0.7092
Recall@10 (IoU≥0.5): 0.8333

✅ 한국어·영어 후보 통합 평가 완료!


In [188]:
print("===== MiniLM 정답 후보의 Fusion 포함 여부 =====")

for case in test_cases:

    # Fine-tuning MiniLM 원본 후보 10개
    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    # 정답과 가장 많이 겹치는 후보
    best_candidate = max(
        candidates,
        key=lambda r: fusion_search.temporal_iou(
            r, case["ground_truth"]
        )
    )

    best_iou = fusion_search.temporal_iou(
        best_candidate, case["ground_truth"]
    )

    # 이전에 저장한 Fine-tuning Fusion 결과
    fusion_row = next(
        r for r in all_results
        if r["name"] == case["name"]
        and r["model"] == "Fine-tuning Fusion"
    )

    final_candidates = fusion_row["candidates"]

    # 원본 최고 후보가 최종 결과에 그대로 포함됐는지
    included = any(
        abs(r["start_time"] - best_candidate["start_time"]) < 0.01
        and abs(r["end_time"] - best_candidate["end_time"]) < 0.01
        for r in final_candidates
    )

    print(f"\n[{case['name']}]")
    print(
        "MiniLM 최고 후보:",
        f"{best_candidate['start_time']:.2f}~"
        f"{best_candidate['end_time']:.2f}초"
    )
    print("원본 최고 IoU:", round(best_iou, 4))
    print("Fusion 포함 여부:", "✅ 포함" if included else "❌ 제외")
    print(
        "Fusion 최고 IoU:",
        round(fusion_row["top5_best_iou"], 4)
    )

print("\n✅ 후보 누락 분석 완료!")

===== MiniLM 정답 후보의 Fusion 포함 여부 =====

[두부 자르기]
MiniLM 최고 후보: 12.88~19.02초
원본 최고 IoU: 0.5582
Fusion 포함 여부: ❌ 제외
Fusion 최고 IoU: 0.3127

[두부 양념 준비]
MiniLM 최고 후보: 108.82~116.02초
원본 최고 IoU: 0.18
Fusion 포함 여부: ❌ 제외
Fusion 최고 IoU: 0.5164

[가위로 소스 짜기]
MiniLM 최고 후보: 5.96~11.32초
원본 최고 IoU: 1.0
Fusion 포함 여부: ❌ 제외
Fusion 최고 IoU: 0.4883

[쌀 씻기]
MiniLM 최고 후보: 29.28~31.72초
원본 최고 IoU: 0.8133
Fusion 포함 여부: ✅ 포함
Fusion 최고 IoU: 0.8133

[다이소 파우치]
MiniLM 최고 후보: 3.08~8.40초
원본 최고 IoU: 1.0
Fusion 포함 여부: ❌ 제외
Fusion 최고 IoU: 0.6992

[다이소 시트 세제]
MiniLM 최고 후보: 13.72~19.08초
원본 최고 IoU: 0.7039
Fusion 포함 여부: ❌ 제외
Fusion 최고 IoU: 0.672

✅ 후보 누락 분석 완료!


In [189]:
print("===== 가위로 소스 짜기 후보 누락 원인 분석 =====")

case = test_cases[2]
gt = case["ground_truth"]

# Hybrid 후보
visual = [
    dict(r, source="hybrid", hybrid_source=r.get("source"))
    for r in predict_moments_hybrid(
        video_id="kr_kitchen_001",
        video_feature_path=(
            "/content/pinpoint-capstone/data/features/"
            "kr_kitchen_001_clip_features.pt"
        ),
        query=case["query_en"],
        subtitle_json_path=None,
        video_duration=37.08,
        top_k=5
    )
]

# Fine-tuning MiniLM 후보
subtitle = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=case["query_ko"],
    english_query=case["query_en"],
    subtitles=case["subtitles"],
    model=trained_model,
    top_k=5
)

ko = [r for r in subtitle if r["language"] == "ko"]
en = [r for r in subtitle if r["language"] == "en"]

# 기존 Fusion과 동일한 순서 구성
ordered = []

for i in range(max(len(visual), len(ko), len(en))):
    for group in (visual, ko, en):
        if i < len(group):
            ordered.append(group[i])

selected = []

print("\n===== 후보 선택 과정 =====")

for i, candidate in enumerate(ordered, 1):

    iou_gt = fusion_search.temporal_iou(candidate, gt)

    duplicates = [
        kept for kept in selected
        if fusion_search.temporal_iou(candidate, kept) >= 0.5
    ]

    if len(selected) >= 5:
        reason = "Top-5 제한"
    elif duplicates:
        reason = "중복 제거"
    else:
        reason = "선택"
        selected.append(candidate)

    print(
        f"{i:2d} | "
        f"{candidate['start_time']:.2f}~"
        f"{candidate['end_time']:.2f}초 "
        f"| {candidate.get('source')} "
        f"| {candidate.get('language')} "
        f"| 정답 IoU={iou_gt:.4f} "
        f"| {reason}"
    )

    if reason == "중복 제거":
        for kept in duplicates:
            overlap = fusion_search.temporal_iou(candidate, kept)
            print(
                f"     ↳ 기존 선택 후보 "
                f"{kept['start_time']:.2f}~"
                f"{kept['end_time']:.2f}초와 "
                f"IoU={overlap:.4f}"
            )

print("\n✅ 후보 누락 원인 분석 완료!")

===== 가위로 소스 짜기 후보 누락 원인 분석 =====

===== 후보 선택 과정 =====
 1 | 2.50~10.27초 | hybrid | None | 정답 IoU=0.4883 | 선택
 2 | 5.96~8.08초 | subtitle | ko | 정답 IoU=0.3955 | 선택
 3 | 23.32~26.56초 | subtitle | en | 정답 IoU=0.0000 | 선택
 4 | 20.50~31.68초 | hybrid | None | 정답 IoU=0.0000 | 선택
 5 | 3.36~8.08초 | subtitle | ko | 정답 IoU=0.2663 | 중복 제거
     ↳ 기존 선택 후보 2.50~10.27초와 IoU=0.6077
 6 | 24.36~26.56초 | subtitle | en | 정답 IoU=0.0000 | 중복 제거
     ↳ 기존 선택 후보 23.32~26.56초와 IoU=0.6790
 7 | 10.50~20.50초 | hybrid | None | 정답 IoU=0.0564 | 선택
 8 | 33.52~34.80초 | subtitle | ko | 정답 IoU=0.0000 | Top-5 제한
 9 | 19.76~21.52초 | subtitle | en | 정답 IoU=0.0000 | Top-5 제한
10 | 5.96~11.32초 | subtitle | ko | 정답 IoU=1.0000 | Top-5 제한
11 | 24.36~29.28초 | subtitle | en | 정답 IoU=0.0000 | Top-5 제한
12 | 23.32~24.36초 | subtitle | ko | 정답 IoU=0.0000 | Top-5 제한
13 | 23.32~29.28초 | subtitle | en | 정답 IoU=0.0000 | Top-5 제한

✅ 후보 누락 원인 분석 완료!


In [190]:
print("===== 정답 후보와 기존 선택 후보의 중복 분석 =====")

# 앞 단계에서 생성된 ordered, selected 변수 사용
target = next(
    r for r in ordered
    if abs(r["start_time"] - 5.96) < 0.01
    and abs(r["end_time"] - 11.32) < 0.01
    and r.get("language") == "ko"
)

print("분석 대상:", target["start_time"], "~", target["end_time"])

for i, candidate in enumerate(selected, 1):
    overlap = fusion_search.temporal_iou(target, candidate)

    print(
        f"{i}위 | "
        f"{candidate['start_time']:.2f}~"
        f"{candidate['end_time']:.2f}초 "
        f"| source={candidate.get('source')} "
        f"| 후보 간 IoU={overlap:.4f} "
        f"| 중복 판정={overlap >= 0.5}"
    )

print("\n✅ 중복 분석 완료!")

===== 정답 후보와 기존 선택 후보의 중복 분석 =====
분석 대상: 5.96 ~ 11.32
1위 | 2.50~10.27초 | source=hybrid | 후보 간 IoU=0.4883 | 중복 판정=False
2위 | 5.96~8.08초 | source=subtitle | 후보 간 IoU=0.3955 | 중복 판정=False
3위 | 23.32~26.56초 | source=subtitle | 후보 간 IoU=0.0000 | 중복 판정=False
4위 | 20.50~31.68초 | source=hybrid | 후보 간 IoU=0.0000 | 중복 판정=False
5위 | 10.50~20.50초 | source=hybrid | 후보 간 IoU=0.0564 | 중복 판정=False

✅ 중복 분석 완료!


# MiniLM 후보 수 확대 실험

In [191]:
import numpy as np

print("===== Fine-tuning MiniLM Top-10 후보 확대 실험 =====")

expanded_results = []

for case in test_cases:

    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=10
    )

    ious = [
        fusion_search.temporal_iou(
            candidate,
            case["ground_truth"]
        )
        for candidate in candidates
    ]

    best_index = int(np.argmax(ious))
    best_candidate = candidates[best_index]
    best_iou = ious[best_index]

    expanded_results.append(best_iou)

    print(f"\n[{case['name']}]")
    print("후보 수:", len(candidates))
    print("최고 IoU:", round(best_iou, 4))
    print(
        "최고 후보:",
        f"{best_candidate['start_time']:.2f}~"
        f"{best_candidate['end_time']:.2f}초",
        "| 언어:", best_candidate["language"]
    )

print("\n===== 전체 평균 =====")
print("평균 최고 IoU:", round(float(np.mean(expanded_results)), 4))
print(
    "Recall@20 (IoU≥0.5):",
    round(float(np.mean(np.array(expanded_results) >= 0.5)), 4)
)

print("\n✅ 후보 확대 실험 완료!")

===== Fine-tuning MiniLM Top-10 후보 확대 실험 =====

[두부 자르기]
후보 수: 20
최고 IoU: 0.5582
최고 후보: 12.88~19.02초 | 언어: en

[두부 양념 준비]
후보 수: 20
최고 IoU: 0.209
최고 후보: 111.26~119.62초 | 언어: en

[가위로 소스 짜기]
후보 수: 20
최고 IoU: 1.0
최고 후보: 5.96~11.32초 | 언어: ko

[쌀 씻기]
후보 수: 20
최고 IoU: 0.8133
최고 후보: 29.28~31.72초 | 언어: ko

[다이소 파우치]
후보 수: 20
최고 IoU: 1.0
최고 후보: 3.08~8.40초 | 언어: ko

[다이소 시트 세제]
후보 수: 20
최고 IoU: 0.7039
최고 후보: 13.72~19.08초 | 언어: ko

===== 전체 평균 =====
평균 최고 IoU: 0.7141
Recall@20 (IoU≥0.5): 0.8333

✅ 후보 확대 실험 완료!


In [192]:
def rank_by_consensus(hybrid, ko, en):
    """
    Hybrid / 한국어 MiniLM / 영어 MiniLM 후보를
    순위 및 시간 구간 겹침 기준으로 재정렬한다.

    정답 구간(ground truth)은 사용하지 않는다.
    """

    groups = {
        "hybrid": hybrid,
        "ko": ko,
        "en": en
    }

    candidates = []

    # 각 검색 방식 내부 순위 점수
    for source, group in groups.items():
        for rank, candidate in enumerate(group, 1):
            item = dict(candidate)

            item["rank_source"] = source
            item["original_rank"] = rank

            # 순위가 높을수록 높은 기본 점수
            item["rank_score"] = 1.0 / rank

            candidates.append(item)

    # 서로 다른 검색 방식의 후보가 겹치면 가점
    for candidate in candidates:
        support = 0.0

        for source, group in groups.items():

            if source == candidate["rank_source"]:
                continue

            best_overlap = max(
                (
                    fusion_search.temporal_iou(
                        candidate, other
                    )
                    for other in group
                ),
                default=0.0
            )

            support += best_overlap

        candidate["consensus_score"] = (
            candidate["rank_score"] + support
        )

    return sorted(
        candidates,
        key=lambda x: x["consensus_score"],
        reverse=True
    )


print("✅ 후보 간 합의 점수 함수 생성 완료!")
print("기존 Fusion 파일은 수정하지 않았습니다.")

✅ 후보 간 합의 점수 함수 생성 완료!
기존 Fusion 파일은 수정하지 않았습니다.


In [193]:
import numpy as np
from pathlib import Path

print("===== PinPoint Consensus Fusion 평가 =====")

consensus_results = []

for case in test_cases:

    name = case["name"]

    # 기존 영상 설정 재사용
    if name.startswith("두부"):
        config = video_config["두부"]
    elif name in ("가위로 소스 짜기", "쌀 씻기"):
        config = video_config["주방"]
    else:
        config = video_config["다이소"]

    video_id = config["video_id"]

    feature_path = (
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        f"/content/pinpoint-capstone/data/processed/"
        f"{video_id}/subtitles.json"
    )

    if not Path(subtitle_path).exists():
        subtitle_path = None

    # 1. 기존 Hybrid 후보
    hybrid = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=config["duration"],
        top_k=5
    )

    # 2. Fine-tuning MiniLM 자막 후보
    subtitle = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    ko = [r for r in subtitle if r["language"] == "ko"]
    en = [r for r in subtitle if r["language"] == "en"]

    # 3. 후보 합의 점수로 재정렬
    ranked = rank_by_consensus(hybrid, ko, en)

    # 4. 기존 Fusion과 동일한 중복 제거 기준
    selected = []

    for candidate in ranked:
        duplicate = any(
            fusion_search.temporal_iou(candidate, kept) >= 0.5
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

        if len(selected) >= 5:
            break

    # 5. 정답 구간과 비교
    ious = [
        fusion_search.temporal_iou(candidate, case["ground_truth"])
        for candidate in selected
    ]

    top1 = ious[0] if ious else 0.0
    top5 = max(ious) if ious else 0.0

    consensus_results.append({
        "name": name,
        "top1_iou": top1,
        "top5_best_iou": top5,
        "recall_at_1": int(top1 >= 0.5),
        "recall_at_5": int(top5 >= 0.5),
        "candidates": selected
    })

    print(f"\n[{name}]")
    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Top-5 최고 IoU: {top5:.4f}")
    print(
        "1위 후보:",
        f"{selected[0]['start_time']:.2f}~"
        f"{selected[0]['end_time']:.2f}초"
        if selected else "없음"
    )

print("\n===== 전체 평균 성능 비교 =====")

consensus_summary = {
    "mean_top1_iou": float(np.mean([
        r["top1_iou"] for r in consensus_results
    ])),
    "mean_top5_best_iou": float(np.mean([
        r["top5_best_iou"] for r in consensus_results
    ])),
    "recall_at_1_iou05": float(np.mean([
        r["recall_at_1"] for r in consensus_results
    ])),
    "recall_at_5_iou05": float(np.mean([
        r["recall_at_5"] for r in consensus_results
    ]))
}

for model_name in ["기존 Fusion", "Fine-tuning Fusion"]:
    print(f"\n[{model_name}]")
    for metric, value in summary[model_name].items():
        print(f"{metric}: {value:.4f}")

print("\n[Consensus Fusion]")
for metric, value in consensus_summary.items():
    print(f"{metric}: {value:.4f}")

print("\n✅ Consensus Fusion 평가 완료!")

===== PinPoint Consensus Fusion 평가 =====

[두부 자르기]
Top-1 IoU: 0.0000
Top-5 최고 IoU: 0.3127
1위 후보: 98.50~118.96초

[두부 양념 준비]
Top-1 IoU: 0.0000
Top-5 최고 IoU: 0.5164
1위 후보: 137.58~143.58초

[가위로 소스 짜기]
Top-1 IoU: 0.0000
Top-5 최고 IoU: 1.0000
1위 후보: 23.32~26.56초

[쌀 씻기]
Top-1 IoU: 0.8133
Top-5 최고 IoU: 0.8133
1위 후보: 29.28~31.72초

[다이소 파우치]
Top-1 IoU: 0.6646
Top-5 최고 IoU: 0.6992
1위 후보: 2.08~7.28초

[다이소 시트 세제]
Top-1 IoU: 0.3038
Top-5 최고 IoU: 0.7039
1위 후보: 14.50~24.52초

===== 전체 평균 성능 비교 =====

[기존 Fusion]
mean_top1_iou: 0.2540
mean_top5_best_iou: 0.5883
recall_at_1_iou05: 0.1667
recall_at_5_iou05: 0.6667

[Fine-tuning Fusion]
mean_top1_iou: 0.2540
mean_top5_best_iou: 0.5837
recall_at_1_iou05: 0.1667
recall_at_5_iou05: 0.6667

[Consensus Fusion]
mean_top1_iou: 0.2970
mean_top5_best_iou: 0.6743
recall_at_1_iou05: 0.3333
recall_at_5_iou05: 0.8333

✅ Consensus Fusion 평가 완료!


In [194]:
print("===== 가위로 소스 짜기 Consensus 점수 분석 =====")

case = test_cases[2]

hybrid = predict_moments_hybrid(
    video_id="kr_kitchen_001",
    video_feature_path=(
        "/content/pinpoint-capstone/data/features/"
        "kr_kitchen_001_clip_features.pt"
    ),
    query=case["query_en"],
    subtitle_json_path=None,
    video_duration=37.08,
    top_k=5
)

subtitle = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=case["query_ko"],
    english_query=case["query_en"],
    subtitles=case["subtitles"],
    model=trained_model,
    top_k=5
)

ko = [r for r in subtitle if r["language"] == "ko"]
en = [r for r in subtitle if r["language"] == "en"]

ranked = rank_by_consensus(hybrid, ko, en)

for i, candidate in enumerate(ranked[:10], 1):

    iou = fusion_search.temporal_iou(
        candidate,
        case["ground_truth"]
    )

    print(
        f"{i:2d}위 | "
        f"{candidate['start_time']:.2f}~"
        f"{candidate['end_time']:.2f}초"
        f" | 방식={candidate['rank_source']}"
        f" | 원래 순위={candidate['original_rank']}"
        f" | 기본 점수={candidate['rank_score']:.4f}"
        f" | 최종 점수={candidate['consensus_score']:.4f}"
        f" | 정답 IoU={iou:.4f}"
    )

print("\n✅ Consensus 점수 분석 완료!")

===== 가위로 소스 짜기 Consensus 점수 분석 =====
 1위 | 23.32~26.56초 | 방식=en | 원래 순위=1 | 기본 점수=1.0000 | 최종 점수=1.6108 | 정답 IoU=0.0000
 2위 | 2.50~10.27초 | 방식=hybrid | 원래 순위=1 | 기본 점수=1.0000 | 최종 점수=1.6077 | 정답 IoU=0.4883
 3위 | 5.96~8.08초 | 방식=ko | 원래 순위=1 | 기본 점수=1.0000 | 최종 점수=1.2730 | 정답 IoU=0.3955
 4위 | 20.50~31.68초 | 방식=hybrid | 원래 순위=2 | 기본 점수=0.5000 | 최종 점수=1.1262 | 정답 IoU=0.0000
 5위 | 3.36~8.08초 | 방식=ko | 원래 순위=2 | 기본 점수=0.5000 | 최종 점수=1.1077 | 정답 IoU=0.2663
 6위 | 23.32~29.28초 | 방식=en | 원래 순위=5 | 기본 점수=0.2000 | 최종 점수=0.9076 | 정답 IoU=0.0000
 7위 | 5.96~11.32초 | 방식=ko | 원래 순위=4 | 기본 점수=0.2500 | 최종 점수=0.7383 | 정답 IoU=1.0000
 8위 | 24.36~26.56초 | 방식=en | 원래 순위=2 | 기본 점수=0.5000 | 최종 점수=0.6968 | 정답 IoU=0.0000
 9위 | 24.36~29.28초 | 방식=en | 원래 순위=4 | 기본 점수=0.2500 | 최종 점수=0.6901 | 정답 IoU=0.0000
10위 | 23.32~24.36초 | 방식=ko | 원래 순위=5 | 기본 점수=0.2000 | 최종 점수=0.6140 | 정답 IoU=0.0000

✅ Consensus 점수 분석 완료!


In [195]:
print("===== Consensus 후보별 점수 정보 확인 =====")

# 이전 단계에서 계산한 가위로 소스 짜기 후보 사용
for i, candidate in enumerate(ranked[:10], 1):

    print(f"\n{i}위 | {candidate['rank_source']}")

    print(
        "구간:",
        f"{candidate['start_time']:.2f}~"
        f"{candidate['end_time']:.2f}초"
    )

    print("원본 순위:", candidate["original_rank"])
    print("원본 모델 점수:", candidate.get("score"))
    print("기본 순위 점수:", candidate["rank_score"])
    print("Consensus 점수:", candidate["consensus_score"])

    if candidate.get("text"):
        print("자막:", candidate["text"][:150])

print("\n✅ 후보 점수 구조 확인 완료!")

===== Consensus 후보별 점수 정보 확인 =====

1위 | en
구간: 23.32~26.56초
원본 순위: 1
원본 모델 점수: 0.232
기본 순위 점수: 1.0
Consensus 점수: 1.6108193908961423
자막: 튀김요리도 쉬운데 최고 굽을 땐 316ti의 우염마제품

2위 | hybrid
구간: 2.50~10.27초
원본 순위: 1
원본 모델 점수: 0.8601
기본 순위 점수: 1.0
Consensus 점수: 1.607738363484195

3위 | ko
구간: 5.96~8.08초
원본 순위: 1
원본 모델 점수: 0.5815
기본 순위 점수: 1.0
Consensus 점수: 1.2729672310564604
자막: 이 가위로 소스 짜는거 진짜 편해요

4위 | hybrid
구간: 20.50~31.68초
원본 순위: 2
원본 모델 점수: 0.6314
기본 순위 점수: 0.5
Consensus 점수: 1.1261796777858288

5위 | ko
구간: 3.36~8.08초
원본 순위: 2
원본 모델 점수: 0.4244
기본 순위 점수: 0.5
Consensus 점수: 1.107738363484195
자막: 첫번째, 솔직히 이거 필요 없을거야 생각했는데 이 가위로 소스 짜는거 진짜 편해요

6위 | en
구간: 23.32~29.28초
원본 순위: 5
원본 모델 점수: 0.1612
기본 순위 점수: 0.2
Consensus 점수: 0.9076439128100933
자막: 튀김요리도 쉬운데 최고 굽을 땐 316ti의 우염마제품 중국제품 아니고 메이드인 코리아라 더 맘에 들어요

7위 | ko
구간: 5.96~11.32초
원본 순위: 4
원본 모델 점수: 0.3682
기본 순위 점수: 0.25
Consensus 점수: 0.7382653061224491
자막: 이 가위로 소스 짜는거 진짜 편해요 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데

8위 | en
구간: 24.36~26.56초
원본 순위: 2
원본 모델 점수

In [196]:
print("===== MiniLM 언어별 검색 신뢰도 분석 =====")

for case in test_cases:
    print(f"\n[{case['name']}]")

    for language, query in [
        ("ko", case["query_ko"]),
        ("en", case["query_en"])
    ]:
        candidates = fusion_search.get_semantic_subtitle_candidates(
            query=query,
            subtitles=case["subtitles"],
            model=trained_model,
            top_k=5
        )

        first = candidates[0]
        second = candidates[1]

        score_gap = first["score"] - second["score"]

        top1_iou = fusion_search.temporal_iou(
            first,
            case["ground_truth"]
        )

        print(
            f"  {language} | "
            f"1위 점수={first['score']:.4f} | "
            f"2위 점수={second['score']:.4f} | "
            f"점수 차이={score_gap:.4f} | "
            f"1위 IoU={top1_iou:.4f}"
        )

print("\n✅ 언어별 검색 점수 차이 분석 완료!")

===== MiniLM 언어별 검색 신뢰도 분석 =====

[두부 자르기]
  ko | 1위 점수=0.3667 | 2위 점수=0.3510 | 점수 차이=0.0157 | 1위 IoU=0.0000
  en | 1위 점수=0.3233 | 2위 점수=0.3229 | 점수 차이=0.0004 | 1위 IoU=0.3127

[두부 양념 준비]
  ko | 1위 점수=0.7083 | 2위 점수=0.6990 | 점수 차이=0.0093 | 1위 IoU=0.0000
  en | 1위 점수=0.2509 | 2위 점수=0.2393 | 점수 차이=0.0116 | 1위 IoU=0.1190

[가위로 소스 짜기]
  ko | 1위 점수=0.5815 | 2위 점수=0.4244 | 점수 차이=0.1571 | 1위 IoU=0.3955
  en | 1위 점수=0.2320 | 2위 점수=0.2248 | 점수 차이=0.0072 | 1위 IoU=0.0000

[쌀 씻기]
  ko | 1위 점수=0.7528 | 2위 점수=0.6392 | 점수 차이=0.1136 | 1위 IoU=0.8133
  en | 1위 점수=0.7476 | 2위 점수=0.6461 | 점수 차이=0.1015 | 1위 IoU=0.8133

[다이소 파우치]
  ko | 1위 점수=0.6242 | 2위 점수=0.6156 | 점수 차이=0.0086 | 1위 IoU=0.6646
  en | 1위 점수=0.4255 | 2위 점수=0.4004 | 점수 차이=0.0251 | 1위 IoU=1.0000

[다이소 시트 세제]
  ko | 1위 점수=0.5068 | 2위 점수=0.4634 | 점수 차이=0.0434 | 1위 IoU=0.2880
  en | 1위 점수=0.3398 | 2위 점수=0.2921 | 점수 차이=0.0477 | 1위 IoU=0.0000

✅ 언어별 검색 점수 차이 분석 완료!


In [197]:
print("===== MiniLM 후보 길이와 IoU 비교 =====")

for case in test_cases:

    candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    gt = case["ground_truth"]

    # 한국어·영어 후보 각각의 1위
    top_ko = next(r for r in candidates if r["language"] == "ko")
    top_en = next(r for r in candidates if r["language"] == "en")

    # 전체 10개 중 정답과 가장 잘 겹치는 후보 (진단용)
    best = max(
        candidates,
        key=lambda r: fusion_search.temporal_iou(r, gt)
    )

    print(f"\n[{case['name']}]")

    for label, candidate in [
        ("한국어 1위", top_ko),
        ("영어 1위", top_en),
        ("최고 IoU 후보", best)
    ]:
        duration = candidate["end_time"] - candidate["start_time"]
        iou = fusion_search.temporal_iou(candidate, gt)

        print(
            f"  {label}: "
            f"{candidate['start_time']:.2f}~"
            f"{candidate['end_time']:.2f}초 "
            f"| 길이={duration:.2f}초 "
            f"| IoU={iou:.4f}"
        )

print("\n✅ 후보 길이 분석 완료!")

===== MiniLM 후보 길이와 IoU 비교 =====

[두부 자르기]
  한국어 1위: 4.74~8.70초 | 길이=3.96초 | IoU=0.0000
  영어 1위: 12.88~16.32초 | 길이=3.44초 | IoU=0.3127
  최고 IoU 후보: 12.88~19.02초 | 길이=6.14초 | IoU=0.5582

[두부 양념 준비]
  한국어 1위: 137.58~143.58초 | 길이=6.00초 | IoU=0.0000
  영어 1위: 111.26~116.02초 | 길이=4.76초 | IoU=0.1190
  최고 IoU 후보: 108.82~116.02초 | 길이=7.20초 | IoU=0.1800

[가위로 소스 짜기]
  한국어 1위: 5.96~8.08초 | 길이=2.12초 | IoU=0.3955
  영어 1위: 23.32~26.56초 | 길이=3.24초 | IoU=0.0000
  최고 IoU 후보: 5.96~11.32초 | 길이=5.36초 | IoU=1.0000

[쌀 씻기]
  한국어 1위: 29.28~31.72초 | 길이=2.44초 | IoU=0.8133
  영어 1위: 29.28~31.72초 | 길이=2.44초 | IoU=0.8133
  최고 IoU 후보: 29.28~31.72초 | 길이=2.44초 | IoU=0.8133

[다이소 파우치]
  한국어 1위: 2.08~7.28초 | 길이=5.20초 | IoU=0.6646
  영어 1위: 3.08~8.40초 | 길이=5.32초 | IoU=1.0000
  최고 IoU 후보: 3.08~8.40초 | 길이=5.32초 | IoU=1.0000

[다이소 시트 세제]
  한국어 1위: 13.72~15.16초 | 길이=1.44초 | IoU=0.2880
  영어 1위: 22.68~26.80초 | 길이=4.12초 | IoU=0.0000
  최고 IoU 후보: 13.72~19.08초 | 길이=5.36초 | IoU=0.7039

✅ 후보 길이 분석 완료!


# 자막 묶음 크기별 성능 비교

In [198]:
import numpy as np
from sentence_transformers import util

print("===== MiniLM 자막 묶음 크기별 평가 =====")

window_size_results = {1: [], 2: [], 3: []}

for case in test_cases:
    print(f"\n[{case['name']}]")

    subtitles = case["subtitles"]
    gt = case["ground_truth"]

    for size in (1, 2, 3):

        windows = []

        for i in range(len(subtitles) - size + 1):
            group = subtitles[i:i + size]

            windows.append({
                "start_time": group[0]["start_time"],
                "end_time": group[-1]["end_time"],
                "text": " ".join(s["text"] for s in group)
            })

        # 동일한 후보 집합을 한국어·영어 검색에 사용
        doc_embeddings = trained_model.encode(
            [w["text"] for w in windows],
            convert_to_tensor=True,
            show_progress_bar=False
        )

        selected_candidates = []

        for query in (case["query_ko"], case["query_en"]):

            query_embedding = trained_model.encode(
                query,
                convert_to_tensor=True,
                show_progress_bar=False
            )

            scores = util.cos_sim(
                query_embedding,
                doc_embeddings
            )[0]

            indices = scores.topk(
                min(5, len(windows))
            ).indices.tolist()

            selected_candidates.extend(
                windows[idx] for idx in indices
            )

        # 한국어 Top-5 + 영어 Top-5의 최고 IoU
        best_iou = max(
            fusion_search.temporal_iou(candidate, gt)
            for candidate in selected_candidates
        )

        window_size_results[size].append(best_iou)

        print(
            f"  자막 {size}개 묶음: "
            f"후보 최고 IoU={best_iou:.4f}"
        )

print("\n===== 묶음 크기별 평균 =====")

for size, values in window_size_results.items():
    print(
        f"자막 {size}개: "
        f"평균 최고 IoU={np.mean(values):.4f}, "
        f"Recall@10(IoU≥0.5)={np.mean(np.array(values) >= 0.5):.4f}"
    )

print("\n✅ 자막 묶음 크기별 평가 완료!")

===== MiniLM 자막 묶음 크기별 평가 =====

[두부 자르기]
  자막 1개 묶음: 후보 최고 IoU=0.3127
  자막 2개 묶음: 후보 최고 IoU=0.5582
  자막 3개 묶음: 후보 최고 IoU=0.4909

[두부 양념 준비]
  자막 1개 묶음: 후보 최고 IoU=0.1190
  자막 2개 묶음: 후보 최고 IoU=0.1800
  자막 3개 묶음: 후보 최고 IoU=0.2270

[가위로 소스 짜기]
  자막 1개 묶음: 후보 최고 IoU=0.3955
  자막 2개 묶음: 후보 최고 IoU=1.0000
  자막 3개 묶음: 후보 최고 IoU=0.7016

[쌀 씻기]
  자막 1개 묶음: 후보 최고 IoU=0.8133
  자막 2개 묶음: 후보 최고 IoU=0.6018
  자막 3개 묶음: 후보 최고 IoU=0.4690

[다이소 파우치]
  자막 1개 묶음: 후보 최고 IoU=0.4887
  자막 2개 묶음: 후보 최고 IoU=0.7895
  자막 3개 묶음: 후보 최고 IoU=1.0000

[다이소 시트 세제]
  자막 1개 묶음: 후보 최고 IoU=0.2880
  자막 2개 묶음: 후보 최고 IoU=0.6720
  자막 3개 묶음: 후보 최고 IoU=0.7039

===== 묶음 크기별 평균 =====
자막 1개: 평균 최고 IoU=0.4029, Recall@10(IoU≥0.5)=0.1667
자막 2개: 평균 최고 IoU=0.6336, Recall@10(IoU≥0.5)=0.8333
자막 3개: 평균 최고 IoU=0.5987, Recall@10(IoU≥0.5)=0.5000

✅ 자막 묶음 크기별 평가 완료!


In [199]:
from sentence_transformers import util

case = test_cases[2]
subtitles = case["subtitles"]
query = case["query_ko"]
gt = case["ground_truth"]

windows = []

for size in (1, 2, 3):
    for i in range(len(subtitles) - size + 1):
        group = subtitles[i:i + size]

        windows.append({
            "start_time": group[0]["start_time"],
            "end_time": group[-1]["end_time"],
            "text": " ".join(s["text"] for s in group),
            "window_size": size
        })

query_emb = trained_model.encode(
    query, convert_to_tensor=True
)

window_embs = trained_model.encode(
    [w["text"] for w in windows],
    convert_to_tensor=True
)

scores = util.cos_sim(query_emb, window_embs)[0]

ranked_indices = scores.argsort(descending=True).tolist()

print("===== 가위로 소스 짜기 한국어 후보 Top-15 =====")

for rank, idx in enumerate(ranked_indices[:15], 1):
    w = windows[idx]

    iou = fusion_search.temporal_iou(w, gt)

    print(
        f"{rank:2d}위 | "
        f"자막 {w['window_size']}개 | "
        f"{w['start_time']:.2f}~{w['end_time']:.2f}초 "
        f"| 유사도={scores[idx].item():.4f} "
        f"| IoU={iou:.4f}"
    )

    print("    자막:", w["text"][:120])

print("\n✅ 자막 묶음별 검색 점수 확인 완료!")

===== 가위로 소스 짜기 한국어 후보 Top-15 =====
 1위 | 자막 1개 | 5.96~8.08초 | 유사도=0.5815 | IoU=0.3955
    자막: 이 가위로 소스 짜는거 진짜 편해요
 2위 | 자막 2개 | 3.36~8.08초 | 유사도=0.4244 | IoU=0.2663
    자막: 첫번째, 솔직히 이거 필요 없을거야 생각했는데 이 가위로 소스 짜는거 진짜 편해요
 3위 | 자막 1개 | 33.52~34.80초 | 유사도=0.3893 | IoU=0.0000
    자막: 물 바로 버리기도 좋아요
 4위 | 자막 2개 | 5.96~11.32초 | 유사도=0.3682 | IoU=1.0000
    자막: 이 가위로 소스 짜는거 진짜 편해요 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데
 5위 | 자막 1개 | 23.32~24.36초 | 유사도=0.3335 | IoU=0.0000
    자막: 튀김요리도 쉬운데
 6위 | 자막 1개 | 11.32~13.60초 | 유사도=0.3298 | IoU=0.0000
    자막: 가위 필요할땐 가위로 집게 필요할땐 집게로
 7위 | 자막 1개 | 21.52~23.32초 | 유사도=0.3256 | IoU=0.0000
    자막: 시금치 대치고 비빔면 끓이고
 8위 | 자막 1개 | 8.08~11.32초 | 유사도=0.3050 | IoU=0.6045
    자막: 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데
 9위 | 자막 1개 | 31.72~33.52초 | 유사도=0.2805 | IoU=0.0000
    자막: 한 톨도 흘리지 않고 과일 씻어서
10위 | 자막 1개 | 18.20~19.76초 | 유사도=0.2638 | IoU=0.0000
    자막: 고리차 끓여서 건전해기 좋고
11위 | 자막 3개 | 5.96~13.60초 | 유사도=0.2586 | IoU=0.7016
    자막: 이 가위로 소스 짜는거 진짜 편해요 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜

# 핵심 자막과 인접 자막의 의미 관련성 확인

In [200]:
from sentence_transformers import util

case = test_cases[2]

query = case["query_ko"]
subtitles = case["subtitles"]

# MiniLM 한국어 검색 1위 자막
anchor_index = 2  # 5.96~8.08초

# 앞뒤 자막 확인
neighbor_indices = [
    i for i in range(anchor_index - 2, anchor_index + 3)
    if 0 <= i < len(subtitles)
]

texts = [subtitles[i]["text"] for i in neighbor_indices]

query_embedding = trained_model.encode(
    query,
    convert_to_tensor=True
)

text_embeddings = trained_model.encode(
    texts,
    convert_to_tensor=True
)

scores = util.cos_sim(
    query_embedding,
    text_embeddings
)[0].tolist()

print("===== Anchor 주변 자막 의미 관련성 =====")
print("검색어:", query)

for idx, score in zip(neighbor_indices, scores):
    subtitle = subtitles[idx]

    marker = "⭐ Anchor" if idx == anchor_index else "  주변"

    print(
        f"{marker} | "
        f"{subtitle['start_time']:.2f}~"
        f"{subtitle['end_time']:.2f}초 "
        f"| 유사도={score:.4f}"
    )
    print("   자막:", subtitle["text"])

print("\n✅ 인접 자막 관련성 분석 완료!")

===== Anchor 주변 자막 의미 관련성 =====
검색어: 가위로 소스를 짜는 장면
  주변 | 0.00~3.36초 | 유사도=0.0388
   자막: 진작 살걸 후회하는 앞머리 보구에 주방 추천템 세가지
  주변 | 3.36~5.96초 | 유사도=0.2408
   자막: 첫번째, 솔직히 이거 필요 없을거야 생각했는데
⭐ Anchor | 5.96~8.08초 | 유사도=0.5815
   자막: 이 가위로 소스 짜는거 진짜 편해요
  주변 | 8.08~11.32초 | 유사도=0.3050
   자막: 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데
  주변 | 11.32~13.60초 | 유사도=0.3298
   자막: 가위 필요할땐 가위로 집게 필요할땐 집게로

✅ 인접 자막 관련성 분석 완료!


In [201]:
from sentence_transformers import util

case = test_cases[2]

query = case["query_ko"]
subtitles = case["subtitles"]

anchor_index = 2

neighbor_indices = [
    i for i in range(anchor_index - 2, anchor_index + 4)
    if 0 <= i < len(subtitles)
]

texts = [subtitles[i]["text"] for i in neighbor_indices]

# 검색어, Anchor, 주변 자막 임베딩
query_emb = trained_model.encode(
    query, convert_to_tensor=True
)

anchor_emb = trained_model.encode(
    subtitles[anchor_index]["text"],
    convert_to_tensor=True
)

neighbor_embs = trained_model.encode(
    texts, convert_to_tensor=True
)

query_scores = util.cos_sim(
    query_emb, neighbor_embs
)[0].tolist()

anchor_scores = util.cos_sim(
    anchor_emb, neighbor_embs
)[0].tolist()

print("===== Query 유사도 vs Anchor 유사도 =====")

for idx, query_score, anchor_score in zip(
    neighbor_indices, query_scores, anchor_scores
):
    s = subtitles[idx]

    print(
        f"{s['start_time']:.2f}~{s['end_time']:.2f}초"
        f" | Query={query_score:.4f}"
        f" | Anchor={anchor_score:.4f}"
    )
    print("자막:", s["text"])

print("\n✅ Anchor 관련성 비교 완료!")

===== Query 유사도 vs Anchor 유사도 =====
0.00~3.36초 | Query=0.0388 | Anchor=0.1607
자막: 진작 살걸 후회하는 앞머리 보구에 주방 추천템 세가지
3.36~5.96초 | Query=0.2408 | Anchor=0.5902
자막: 첫번째, 솔직히 이거 필요 없을거야 생각했는데
5.96~8.08초 | Query=0.5815 | Anchor=1.0000
자막: 이 가위로 소스 짜는거 진짜 편해요
8.08~11.32초 | Query=0.3050 | Anchor=0.5002
자막: 손에 묻지도 않고 한 방울도 남기지 않고 깨끗하게 짜내는데
11.32~13.60초 | Query=0.3298 | Anchor=0.6536
자막: 가위 필요할땐 가위로 집게 필요할땐 집게로
13.60~15.96초 | Query=0.2304 | Anchor=0.7142
자막: 하나로 다 기능을 해서 솔직히 편하더라구요

✅ Anchor 관련성 비교 완료!


# MiniLM 1위와 Hybrid 후보 간 시간적 겹침 분석

In [202]:
from pathlib import Path

print("===== MiniLM Anchor와 Hybrid 후보 겹침 분석 =====")

for case in test_cases:

    name = case["name"]

    if name.startswith("두부"):
        config = video_config["두부"]
    elif name in ("가위로 소스 짜기", "쌀 씻기"):
        config = video_config["주방"]
    else:
        config = video_config["다이소"]

    video_id = config["video_id"]

    feature_path = (
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        f"/content/pinpoint-capstone/data/processed/"
        f"{video_id}/subtitles.json"
    )

    if not Path(subtitle_path).exists():
        subtitle_path = None

    # Hybrid 후보
    hybrid_candidates = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=config["duration"],
        top_k=5
    )

    # 한국어 MiniLM 1위 후보 (Anchor)
    subtitle_candidates = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_ko"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=1
    )

    anchor = subtitle_candidates[0]

    overlaps = [
        fusion_search.temporal_iou(anchor, hybrid)
        for hybrid in hybrid_candidates
    ]

    best_overlap = max(overlaps)
    best_hybrid_idx = overlaps.index(best_overlap)
    best_hybrid = hybrid_candidates[best_hybrid_idx]

    print(f"\n[{name}]")

    print(
        "MiniLM Anchor:",
        f"{anchor['start_time']:.2f}~"
        f"{anchor['end_time']:.2f}초"
    )

    print(
        "가장 겹치는 Hybrid:",
        f"{best_hybrid['start_time']:.2f}~"
        f"{best_hybrid['end_time']:.2f}초"
    )

    print("후보 간 최고 IoU:", round(best_overlap, 4))

    print(
        "Anchor 정답 IoU:",
        round(
            fusion_search.temporal_iou(
                anchor, case["ground_truth"]
            ), 4
        )
    )

print("\n✅ Anchor-Hybrid 겹침 분석 완료!")

===== MiniLM Anchor와 Hybrid 후보 겹침 분석 =====

[두부 자르기]
MiniLM Anchor: 4.74~8.70초
가장 겹치는 Hybrid: 98.50~118.96초
후보 간 최고 IoU: 0.0
Anchor 정답 IoU: 0.0

[두부 양념 준비]
MiniLM Anchor: 137.58~143.58초
가장 겹치는 Hybrid: 134.50~151.32초
후보 간 최고 IoU: 0.3568
Anchor 정답 IoU: 0.0

[가위로 소스 짜기]
MiniLM Anchor: 5.96~8.08초
가장 겹치는 Hybrid: 2.50~10.27초
후보 간 최고 IoU: 0.273
Anchor 정답 IoU: 0.3955

[쌀 씻기]
MiniLM Anchor: 29.28~31.72초
가장 겹치는 Hybrid: 22.50~36.43초
후보 간 최고 IoU: 0.1752
Anchor 정답 IoU: 0.8133

[다이소 파우치]
MiniLM Anchor: 2.08~7.28초
가장 겹치는 Hybrid: 14.50~25.02초
후보 간 최고 IoU: 0.0
Anchor 정답 IoU: 0.6646

[다이소 시트 세제]
MiniLM Anchor: 13.72~15.16초
가장 겹치는 Hybrid: 2.50~14.72초
후보 간 최고 IoU: 0.0793
Anchor 정답 IoU: 0.288

✅ Anchor-Hybrid 겹침 분석 완료!


# 새로운 평가 검색어 준비하기

In [203]:
# 새로운 평가용 검색어 및 임시 정답 구간
# 기존 영상의 자막 시간대를 기준으로 작성한 후보이며
# 실제 영상 확인 후 정답 구간을 확정해야 함

new_test_cases = [
    {
        "name": "두부 데치기",
        "video_id": "kr_tofu_001",
        "query_ko": "두부를 끓는 물에 데치는 장면",
        "query_en": "blanching tofu in hot water",
        "ground_truth": {
            "start_time": 36.98,
            "end_time": 44.62
        }
    },
    {
        "name": "주방 체망 소개",
        "video_id": "kr_kitchen_001",
        "query_ko": "냄비에 사용하는 체망을 소개하는 장면",
        "query_en": "introducing a strainer for a pot",
        "ground_truth": {
            "start_time": 15.96,
            "end_time": 21.52
        }
    },
    {
        "name": "다이소 헤어브러시",
        "video_id": "kr_daiso_001",
        "query_ko": "여행용 헤어브러시를 소개하는 장면",
        "query_en": "showing a travel hairbrush",
        "ground_truth": {
            "start_time": 22.68,
            "end_time": 26.80
        }
    }
]

print("===== 새로운 평가 검색어 확인 =====")

for case in new_test_cases:
    gt = case["ground_truth"]

    print(f"\n[{case['name']}]")
    print("영상:", case["video_id"])
    print("한국어 검색어:", case["query_ko"])
    print("영어 검색어:", case["query_en"])
    print(
        "임시 정답:",
        f"{gt['start_time']:.2f}~{gt['end_time']:.2f}초"
    )

print("\n✅ 새로운 평가 검색어 3개 준비 완료!")
print("※ 실제 영상 확인 전에는 정답 구간을 확정하지 않습니다.")

===== 새로운 평가 검색어 확인 =====

[두부 데치기]
영상: kr_tofu_001
한국어 검색어: 두부를 끓는 물에 데치는 장면
영어 검색어: blanching tofu in hot water
임시 정답: 36.98~44.62초

[주방 체망 소개]
영상: kr_kitchen_001
한국어 검색어: 냄비에 사용하는 체망을 소개하는 장면
영어 검색어: introducing a strainer for a pot
임시 정답: 15.96~21.52초

[다이소 헤어브러시]
영상: kr_daiso_001
한국어 검색어: 여행용 헤어브러시를 소개하는 장면
영어 검색어: showing a travel hairbrush
임시 정답: 22.68~26.80초

✅ 새로운 평가 검색어 3개 준비 완료!
※ 실제 영상 확인 전에는 정답 구간을 확정하지 않습니다.


In [204]:
# 실제 영상 확인 결과를 정답 구간에 반영
corrected_ground_truth = {
    "두부 데치기": (38.0, 61.0),
    "주방 체망 소개": (16.0, 28.0),
    "다이소 헤어브러시": (22.0, 26.0)
}

for case in new_test_cases:
    start, end = corrected_ground_truth[case["name"]]

    case["ground_truth"] = {
        "start_time": start,
        "end_time": end
    }

print("===== PinPoint 추가 평가 정답 구간 확정 =====")

for case in new_test_cases:
    gt = case["ground_truth"]

    print(
        f"{case['name']} | "
        f"{gt['start_time']:.2f}~"
        f"{gt['end_time']:.2f}초"
    )

print("\n✅ 실제 영상 확인 기준으로 정답 구간 수정 완료!")

===== PinPoint 추가 평가 정답 구간 확정 =====
두부 데치기 | 38.00~61.00초
주방 체망 소개 | 16.00~28.00초
다이소 헤어브러시 | 22.00~26.00초

✅ 실제 영상 확인 기준으로 정답 구간 수정 완료!


In [205]:
from pathlib import Path
from unittest.mock import patch
import numpy as np
import json

print("===== PinPoint 추가 검색어 3개 Fusion 비교 =====")

new_eval_results = []

models = {
    "기존 Fusion": original_model,
    "Fine-tuning Fusion": trained_model
}

# 영상별 자막
subtitle_map = {
    "kr_tofu_001": tofu_subtitles,
    "kr_kitchen_001": kitchen_subtitles,
    "kr_daiso_001": daiso_subtitles
}

# 영상별 길이
duration_map = {
    "kr_tofu_001": 164.98,
    "kr_kitchen_001": 37.08,
    "kr_daiso_001": 32.08
}

for case in new_test_cases:

    video_id = case["video_id"]
    subtitles = subtitle_map[video_id]
    duration = duration_map[video_id]
    gt = case["ground_truth"]

    feature_path = (
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        f"/content/pinpoint-capstone/data/processed/"
        f"{video_id}/subtitles.json"
    )

    if not Path(subtitle_path).exists():
        subtitle_path = None

    print(f"\n[{case['name']}]")

    # 1. 기존 Fusion / Fine-tuning Fusion
    for model_name, model in models.items():

        with patch.object(
            fusion_search,
            "get_semantic_model",
            return_value=model
        ):
            results = fusion_search.predict_moments_fusion(
                video_id=video_id,
                feature_path=feature_path,
                subtitles=subtitles,
                korean_query=case["query_ko"],
                english_query=case["query_en"],
                video_duration=duration,
                subtitle_json_path=subtitle_path,
                top_k=5
            )

        ious = [
            fusion_search.temporal_iou(r, gt)
            for r in results
        ]

        top1 = ious[0] if ious else 0.0
        top5 = max(ious) if ious else 0.0

        new_eval_results.append({
            "name": case["name"],
            "model": model_name,
            "top1_iou": top1,
            "top5_best_iou": top5,
            "recall_at_1": int(top1 >= 0.5),
            "recall_at_5": int(top5 >= 0.5)
        })

        print(
            f"  {model_name}: "
            f"Top-1={top1:.4f}, "
            f"Top-5={top5:.4f}"
        )

    # 2. Consensus Fusion
    hybrid = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=duration,
        top_k=5
    )

    subtitle_candidates = (
        fusion_search.get_multilingual_subtitle_candidates(
            korean_query=case["query_ko"],
            english_query=case["query_en"],
            subtitles=subtitles,
            model=trained_model,
            top_k=5
        )
    )

    ko = [
        r for r in subtitle_candidates
        if r["language"] == "ko"
    ]
    en = [
        r for r in subtitle_candidates
        if r["language"] == "en"
    ]

    ranked = rank_by_consensus(hybrid, ko, en)

    selected = []

    for candidate in ranked:

        duplicate = any(
            fusion_search.temporal_iou(candidate, kept) >= 0.5
            for kept in selected
        )

        if not duplicate:
            selected.append(candidate)

        if len(selected) >= 5:
            break

    ious = [
        fusion_search.temporal_iou(r, gt)
        for r in selected
    ]

    top1 = ious[0] if ious else 0.0
    top5 = max(ious) if ious else 0.0

    new_eval_results.append({
        "name": case["name"],
        "model": "Consensus Fusion",
        "top1_iou": top1,
        "top5_best_iou": top5,
        "recall_at_1": int(top1 >= 0.5),
        "recall_at_5": int(top5 >= 0.5)
    })

    print(
        f"  Consensus Fusion: "
        f"Top-1={top1:.4f}, "
        f"Top-5={top5:.4f}"
    )

print("\n===== 추가 검색어 3개 평균 성능 =====")

new_eval_summary = {}

for model_name in [
    "기존 Fusion",
    "Fine-tuning Fusion",
    "Consensus Fusion"
]:
    rows = [
        r for r in new_eval_results
        if r["model"] == model_name
    ]

    new_eval_summary[model_name] = {
        "mean_top1_iou": float(np.mean([
            r["top1_iou"] for r in rows
        ])),
        "mean_top5_best_iou": float(np.mean([
            r["top5_best_iou"] for r in rows
        ])),
        "recall_at_1_iou05": float(np.mean([
            r["recall_at_1"] for r in rows
        ])),
        "recall_at_5_iou05": float(np.mean([
            r["recall_at_5"] for r in rows
        ]))
    }

    print(f"\n[{model_name}]")
    for metric, value in new_eval_summary[model_name].items():
        print(f"{metric}: {value:.4f}")

# Google Drive에 결과 저장
output_path = (
    training_dir /
    "fusion_new_3queries_comparison.json"
)

output_path.write_text(
    json.dumps({
        "test_count": len(new_test_cases),
        "ground_truth_source": "Manual approximate video review",
        "summary": new_eval_summary,
        "results": new_eval_results
    }, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n✅ 추가 검색어 3개 평가 및 저장 완료!")
print("저장 경로:", output_path)

===== PinPoint 추가 검색어 3개 Fusion 비교 =====

[두부 데치기]
  기존 Fusion: Top-1=0.0000, Top-5=0.1065
  Fine-tuning Fusion: Top-1=0.0000, Top-5=0.1191
  Consensus Fusion: Top-1=0.1026, Top-5=0.3589

[주방 체망 소개]
  기존 Fusion: Top-1=0.2609, Top-5=0.4585
  Fine-tuning Fusion: Top-1=0.2609, Top-5=0.2609
  Consensus Fusion: Top-1=0.1319, Top-5=0.2609

[다이소 헤어브러시]
  기존 Fusion: Top-1=0.3003, Top-5=0.3003
  Fine-tuning Fusion: Top-1=0.3003, Top-5=0.3167
  Consensus Fusion: Top-1=0.0000, Top-5=0.3167

===== 추가 검색어 3개 평균 성능 =====

[기존 Fusion]
mean_top1_iou: 0.1871
mean_top5_best_iou: 0.2884
recall_at_1_iou05: 0.0000
recall_at_5_iou05: 0.0000

[Fine-tuning Fusion]
mean_top1_iou: 0.1871
mean_top5_best_iou: 0.2322
recall_at_1_iou05: 0.0000
recall_at_5_iou05: 0.0000

[Consensus Fusion]
mean_top1_iou: 0.0782
mean_top5_best_iou: 0.3122
recall_at_1_iou05: 0.0000
recall_at_5_iou05: 0.0000

✅ 추가 검색어 3개 평가 및 저장 완료!
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/fusion_new_3queries_comparison.j

In [206]:
print("===== 두부 데치기 후보 상세 분석 =====")

case = new_test_cases[0]
gt = case["ground_truth"]

# 1. Hybrid 후보
hybrid = predict_moments_hybrid(
    video_id=case["video_id"],
    video_feature_path=(
        "/content/pinpoint-capstone/data/features/"
        "kr_tofu_001_clip_features.pt"
    ),
    query=case["query_en"],
    subtitle_json_path=None,
    video_duration=164.98,
    top_k=5
)

# 2. Fine-tuning MiniLM 후보
subtitle = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=case["query_ko"],
    english_query=case["query_en"],
    subtitles=tofu_subtitles,
    model=trained_model,
    top_k=10
)

# 검색 방식별 후보
groups = {
    "Hybrid": hybrid,
    "MiniLM 한국어": [
        r for r in subtitle if r["language"] == "ko"
    ],
    "MiniLM 영어": [
        r for r in subtitle if r["language"] == "en"
    ]
}

for group_name, candidates in groups.items():
    print(f"\n===== {group_name} =====")

    for i, candidate in enumerate(candidates, 1):
        iou = fusion_search.temporal_iou(candidate, gt)

        print(
            f"{i}위 | "
            f"{candidate['start_time']:.2f}~"
            f"{candidate['end_time']:.2f}초 "
            f"| IoU={iou:.4f}"
            f" | score={candidate.get('score')}"
        )

        if candidate.get("text"):
            print("   자막:", candidate["text"][:120])

print("\n✅ 두부 데치기 후보 분석 완료!")

===== 두부 데치기 후보 상세 분석 =====

===== Hybrid =====
1위 | 104.50~119.50초 | IoU=0.0000 | score=0.9886
2위 | 86.50~104.12초 | IoU=0.0000 | score=0.9672
3위 | 134.50~163.85초 | IoU=0.0000 | score=0.941
4위 | 38.50~96.79초 | IoU=0.3827 | score=0.8439
5위 | 158.50~164.98초 | IoU=0.0000 | score=0.2768

===== MiniLM 한국어 =====
1위 | 36.98~40.86초 | IoU=0.1191 | score=0.6995
   자막: 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요
2위 | 50.90~53.26초 | IoU=0.1026 | score=0.6883
   자막: 두부를 익힐 때는 먼저 물을 끓인 다음
3위 | 34.14~40.86초 | IoU=0.1065 | score=0.6092
   자막: 두부를 어떻게 손질하느냐에 따라 훨씬 더 맛있어져요 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요
4위 | 36.98~44.62초 | IoU=0.2756 | score=0.5114
   자막: 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요 이렇게 데쳐주면 두부 특유의 콩비린내를 확실하게 잡을 수 있어요
5위 | 36.98~46.62초 | IoU=0.3589 | score=0.4833
   자막: 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요 이렇게 데쳐주면 두부 특유의 콩비린내를 확실하게 잡을 수 있어요 여기에 소금 두 숟가락을 넣어주세요
6위 | 34.14~44.62초 | IoU=0.2465 | score=0.4815
   자막: 두부를 어떻게 손질하느냐에 따라 훨씬 더 맛있어져요 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요 이렇게 데쳐주면 두부 특유의 콩비린내를 확실하게 잡을 수 있어요
7위 | 3

In [207]:
print("===== 두부 데치기 MiniLM 시간 분포 분석 =====")

case = new_test_cases[0]

candidates = fusion_search.get_multilingual_subtitle_candidates(
    korean_query=case["query_ko"],
    english_query=case["query_en"],
    subtitles=tofu_subtitles,
    model=trained_model,
    top_k=10
)

# 후보를 시작 시간순으로 정렬
candidates_by_time = sorted(
    candidates,
    key=lambda r: r["start_time"]
)

for c in candidates_by_time:
    print(
        f"{c['start_time']:6.2f}~{c['end_time']:6.2f}초"
        f" | {c['language']}"
        f" | 유사도={c['score']:.4f}"
    )

print("\n===== 정답 구간과 겹치는 후보 =====")

gt = case["ground_truth"]

related = [
    c for c in candidates
    if fusion_search.temporal_iou(c, gt) > 0
]

print("전체 후보:", len(candidates))
print("정답과 겹치는 후보:", len(related))

if related:
    print(
        "겹치는 후보의 전체 시간 범위:",
        round(min(c["start_time"] for c in related), 2),
        "~",
        round(max(c["end_time"] for c in related), 2),
        "초"
    )

print("\n✅ 시간 분포 분석 완료!")

===== 두부 데치기 MiniLM 시간 분포 분석 =====
  0.00~  8.70초 | ko | 유사도=0.4448
 30.74~ 40.86초 | ko | 유사도=0.4720
 30.74~ 40.86초 | en | 유사도=0.2053
 34.14~ 40.86초 | ko | 유사도=0.6092
 34.14~ 44.62초 | ko | 유사도=0.4815
 34.14~ 40.86초 | en | 유사도=0.2358
 36.98~ 40.86초 | ko | 유사도=0.6995
 36.98~ 44.62초 | ko | 유사도=0.5114
 36.98~ 46.62초 | ko | 유사도=0.4833
 36.98~ 40.86초 | en | 유사도=0.2992
 50.90~ 53.26초 | ko | 유사도=0.6883
 50.90~ 56.50초 | ko | 유사도=0.4652
 50.90~ 53.26초 | en | 유사도=0.3160
 50.90~ 56.50초 | en | 유사도=0.2031
 50.90~ 62.54초 | en | 유사도=0.2007
104.54~116.02초 | en | 유사도=0.2044
111.26~116.02초 | en | 유사도=0.2769
111.26~117.90초 | en | 유사도=0.1914
111.26~119.62초 | en | 유사도=0.1874
135.06~137.58초 | ko | 유사도=0.4397

===== 정답 구간과 겹치는 후보 =====
전체 후보: 20
정답과 겹치는 후보: 14
겹치는 후보의 전체 시간 범위: 30.74 ~ 62.54 초

✅ 시간 분포 분석 완료!


In [208]:
from sentence_transformers import util

case = new_test_cases[0]

query = case["query_ko"]
subtitles = tofu_subtitles

query_emb = trained_model.encode(
    query,
    convert_to_tensor=True,
    show_progress_bar=False
)

subtitle_embs = trained_model.encode(
    [s["text"] for s in subtitles],
    convert_to_tensor=True,
    show_progress_bar=False
)

scores = util.cos_sim(
    query_emb,
    subtitle_embs
)[0].tolist()

print("===== 두부 데치기 자막별 의미 점수 =====")

# 정답 부근을 포함한 30~70초 자막 확인
for subtitle, score in zip(subtitles, scores):

    if subtitle["end_time"] < 30 or subtitle["start_time"] > 70:
        continue

    print(
        f"{subtitle['start_time']:6.2f}~"
        f"{subtitle['end_time']:6.2f}초"
        f" | 유사도={score:.4f}"
    )
    print("   자막:", subtitle["text"])

print("\n✅ 자막별 의미 점수 분석 완료!")

===== 두부 데치기 자막별 의미 점수 =====
 26.46~ 30.74초 | 유사도=0.2622
   자막: 여러분이 계신 곳에서는 두부 한 모에 얼마 정도 하나요?
 30.74~ 34.14초 | 유사도=-0.0238
   자막: 댓글로 가격도 한번 같이 이야기 해봐요 이제부터가 중요한데요
 34.14~ 36.98초 | 유사도=0.3876
   자막: 두부를 어떻게 손질하느냐에 따라 훨씬 더 맛있어져요
 36.98~ 40.86초 | 유사도=0.6995
   자막: 두부를 다 썰었으면 물이 끓을 때 넣어서 한번 데쳐주세요
 40.86~ 44.62초 | 유사도=0.1879
   자막: 이렇게 데쳐주면 두부 특유의 콩비린내를 확실하게 잡을 수 있어요
 44.62~ 46.62초 | 유사도=0.2193
   자막: 여기에 소금 두 숟가락을 넣어주세요
 46.62~ 50.90초 | 유사도=0.1113
   자막: 두부는 원래 간이 잘 배지 않기 때문에 소금을 넣어서 미리 밑간을 해주는 거예요
 50.90~ 53.26초 | 유사도=0.6883
   자막: 두부를 익힐 때는 먼저 물을 끓인 다음
 53.26~ 56.50초 | 유사도=0.0969
   자막: 불을 끄고 뚜껑을 덮어 1분 동안 그대로 두세요
 56.50~ 62.54초 | 유사도=0.2012
   자막: 두부 건면까지 열이 충분히 전달됐을 때 식감이 가장 좋고 정말 부드럽고 맛있어요
 62.54~ 65.90초 | 유사도=0.0929
   자막: 제 프로필에는 벌써 수십 가지 요리 영상이 올라와 있는데요
 65.90~ 71.22초 | 유사도=0.1252
   자막: 하나하나 자세하게 알려드리고 있어서 그대로 따라하시면 실패할 일이 없어요

✅ 자막별 의미 점수 분석 완료!


In [209]:
print("===== 추가 검색어 평가 결과 저장 정보 확인 =====")

print("new_eval_results 개수:", len(new_eval_results))

for row in new_eval_results:
    print(
        row["name"],
        "|",
        row["model"],
        "| 저장된 항목:",
        list(row.keys())
    )

print("\n✅ 저장 정보 확인 완료!")

===== 추가 검색어 평가 결과 저장 정보 확인 =====
new_eval_results 개수: 9
두부 데치기 | 기존 Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
두부 데치기 | Fine-tuning Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
두부 데치기 | Consensus Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
주방 체망 소개 | 기존 Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
주방 체망 소개 | Fine-tuning Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
주방 체망 소개 | Consensus Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
다이소 헤어브러시 | 기존 Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
다이소 헤어브러시 | Fine-tuning Fusion | 저장된 항목: ['name', 'model', 'top1_iou', 'top5_best_iou', 'recall_at_1', 'recall_at_5']
다이소 헤어브러시 | Consensus Fusion | 저장된 항목: ['n

In [210]:
from pathlib import Path
from unittest.mock import patch
import numpy as np
import json

print("===== PinPoint 추가 검색어 Top-1 시작 시간 평가 =====")

start_time_results = []

for case in new_test_cases:
    video_id = case["video_id"]
    subtitles = subtitle_map[video_id]
    duration = duration_map[video_id]
    gt = case["ground_truth"]

    feature_path = (
        f"/content/pinpoint-capstone/data/features/"
        f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        f"/content/pinpoint-capstone/data/processed/"
        f"{video_id}/subtitles.json"
    )

    if not Path(subtitle_path).exists():
        subtitle_path = None

    print(f"\n[{case['name']}]")
    print("정답 시작:", gt["start_time"], "초")

    for model_name, model in [
        ("기존 Fusion", original_model),
        ("Fine-tuning Fusion", trained_model)
    ]:
        with patch.object(
            fusion_search,
            "get_semantic_model",
            return_value=model
        ):
            results = fusion_search.predict_moments_fusion(
                video_id=video_id,
                feature_path=feature_path,
                subtitles=subtitles,
                korean_query=case["query_ko"],
                english_query=case["query_en"],
                video_duration=duration,
                subtitle_json_path=subtitle_path,
                top_k=5
            )

        if results:
            top1 = results[0]
            predicted_start = float(top1["start_time"])
            error = abs(predicted_start - gt["start_time"])
            iou = fusion_search.temporal_iou(top1, gt)
        else:
            predicted_start, error, iou = None, None, 0.0

        start_time_results.append({
            "name": case["name"],
            "model": model_name,
            "predicted_start": predicted_start,
            "ground_truth_start": gt["start_time"],
            "start_error_seconds": error,
            "top1_iou": iou
        })

        print(
            f"  {model_name}: "
            f"시작={predicted_start}, "
            f"오차={error}, "
            f"IoU={iou:.4f}"
        )

    # Consensus Fusion
    hybrid = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=duration,
        top_k=5
    )

    subtitle_candidates = fusion_search.get_multilingual_subtitle_candidates(
        korean_query=case["query_ko"],
        english_query=case["query_en"],
        subtitles=subtitles,
        model=trained_model,
        top_k=5
    )

    ko = [r for r in subtitle_candidates if r["language"] == "ko"]
    en = [r for r in subtitle_candidates if r["language"] == "en"]

    ranked = rank_by_consensus(hybrid, ko, en)

    # Consensus 1위는 재정렬된 첫 번째 후보
    top1 = ranked[0] if ranked else None

    if top1:
        predicted_start = float(top1["start_time"])
        error = abs(predicted_start - gt["start_time"])
        iou = fusion_search.temporal_iou(top1, gt)
    else:
        predicted_start, error, iou = None, None, 0.0

    start_time_results.append({
        "name": case["name"],
        "model": "Consensus Fusion",
        "predicted_start": predicted_start,
        "ground_truth_start": gt["start_time"],
        "start_error_seconds": error,
        "top1_iou": iou
    })

    print(
        f"  Consensus Fusion: "
        f"시작={predicted_start}, "
        f"오차={error}, "
        f"IoU={iou:.4f}"
    )

print("\n===== 모델별 시작 시간 평균 절대오차 =====")

for model_name in [
    "기존 Fusion",
    "Fine-tuning Fusion",
    "Consensus Fusion"
]:
    errors = [
        r["start_error_seconds"]
        for r in start_time_results
        if r["model"] == model_name
        and r["start_error_seconds"] is not None
    ]

    print(
        f"{model_name}: "
        f"MAE={np.mean(errors):.2f}초"
        if errors else f"{model_name}: 평가 불가"
    )

output_path = training_dir / "fusion_start_time_comparison.json"

output_path.write_text(
    json.dumps(start_time_results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n✅ 시작 시간 평가 완료!")
print("저장 경로:", output_path)

===== PinPoint 추가 검색어 Top-1 시작 시간 평가 =====

[두부 데치기]
정답 시작: 38.0 초
  기존 Fusion: 시작=104.5, 오차=66.5, IoU=0.0000
  Fine-tuning Fusion: 시작=104.5, 오차=66.5, IoU=0.0000
  Consensus Fusion: 시작=50.9, 오차=12.899999999999999, IoU=0.1026

[주방 체망 소개]
정답 시작: 16.0 초
  기존 Fusion: 시작=22.5, 오차=6.5, IoU=0.2609
  Fine-tuning Fusion: 시작=22.5, 오차=6.5, IoU=0.2609
  Consensus Fusion: 시작=11.32, 오차=4.68, IoU=0.1319

[다이소 헤어브러시]
정답 시작: 22.0 초
  기존 Fusion: 시작=14.5, 오차=7.5, IoU=0.3003
  Fine-tuning Fusion: 시작=14.5, 오차=7.5, IoU=0.3003
  Consensus Fusion: 시작=15.16, 오차=6.84, IoU=0.0000

===== 모델별 시작 시간 평균 절대오차 =====
기존 Fusion: MAE=26.83초
Fine-tuning Fusion: MAE=26.83초
Consensus Fusion: MAE=8.14초

✅ 시작 시간 평가 완료!
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/fusion_start_time_comparison.json


In [211]:
import numpy as np

print("===== PinPoint 시작 시간 허용 오차 평가 =====")

for model_name in [
    "기존 Fusion",
    "Fine-tuning Fusion",
    "Consensus Fusion"
]:
    rows = [
        r for r in start_time_results
        if r["model"] == model_name
        and r["start_error_seconds"] is not None
    ]

    errors = np.array([
        r["start_error_seconds"] for r in rows
    ])

    print(f"\n[{model_name}]")
    print("평균 시작 오차:", round(float(np.mean(errors)), 2), "초")

    for threshold in [3, 5, 10]:
        success = float(np.mean(errors <= threshold))
        print(
            f"시작 오차 {threshold}초 이내:",
            f"{success:.2%}",
            f"({int(np.sum(errors <= threshold))}/{len(errors)})"
        )

print("\n✅ 시작 시간 허용 오차 평가 완료!")

===== PinPoint 시작 시간 허용 오차 평가 =====

[기존 Fusion]
평균 시작 오차: 26.83 초
시작 오차 3초 이내: 0.00% (0/3)
시작 오차 5초 이내: 0.00% (0/3)
시작 오차 10초 이내: 66.67% (2/3)

[Fine-tuning Fusion]
평균 시작 오차: 26.83 초
시작 오차 3초 이내: 0.00% (0/3)
시작 오차 5초 이내: 0.00% (0/3)
시작 오차 10초 이내: 66.67% (2/3)

[Consensus Fusion]
평균 시작 오차: 8.14 초
시작 오차 3초 이내: 0.00% (0/3)
시작 오차 5초 이내: 33.33% (1/3)
시작 오차 10초 이내: 66.67% (2/3)

✅ 시작 시간 허용 오차 평가 완료!


In [212]:
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

required_paths = {
    "Fine-tuning MiniLM": training_dir / "minilm_finetuned/final_model",
    "전체 Validation 평가": training_dir / "minilm_full_validation_comparison.json",
    "기존 검색어 6개 평가": training_dir / "minilm_fusion_6queries_comparison.json",
    "추가 검색어 3개 평가": training_dir / "fusion_new_3queries_comparison.json",
    "시작 시간 평가": training_dir / "fusion_start_time_comparison.json"
}

print("===== PinPoint AI 실험 결과 백업 확인 =====")

for name, path in required_paths.items():
    status = "✅ 저장됨" if path.exists() else "❌ 없음"
    print(f"{status} | {name}")
    print("   ", path)

print("\n✅ 백업 확인 완료!")

===== PinPoint AI 실험 결과 백업 확인 =====
✅ 저장됨 | Fine-tuning MiniLM
    /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model
✅ 저장됨 | 전체 Validation 평가
    /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_full_validation_comparison.json
✅ 저장됨 | 기존 검색어 6개 평가
    /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_fusion_6queries_comparison.json
✅ 저장됨 | 추가 검색어 3개 평가
    /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/fusion_new_3queries_comparison.json
✅ 저장됨 | 시작 시간 평가
    /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/fusion_start_time_comparison.json

✅ 백업 확인 완료!


# 스쿼트자세설명 영상 테스트

In [213]:
from pathlib import Path
import cv2

video_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/스쿼트자세설명영상.mp4"
)

print("===== 스쿼트 영상 파일 확인 =====")

print("파일 존재:", video_path.exists())

if video_path.exists():
    print("파일 크기(MB):", round(
        video_path.stat().st_size / (1024 ** 2), 2
    ))

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print("영상 열기 성공:", cap.isOpened())
    print("FPS:", fps)
    print("총 프레임:", int(frame_count))
    print("해상도:", f"{width}×{height}")
    print("영상 길이(초):", round(frame_count / fps, 2) if fps > 0 else "확인 불가")

    cap.release()

print("\n✅ 영상 기본 정보 확인 완료!")

===== 스쿼트 영상 파일 확인 =====
파일 존재: True
파일 크기(MB): 3.08
영상 열기 성공: True
FPS: 30.0
총 프레임: 1260
해상도: 360×640
영상 길이(초): 42.0

✅ 영상 기본 정보 확인 완료!


In [214]:
%pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 39.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [215]:
import whisper
import torch

print("✅ Whisper 불러오기 성공!")
print("GPU 사용 가능:", torch.cuda.is_available())

✅ Whisper 불러오기 성공!
GPU 사용 가능: True


In [216]:
import whisper
import json
from pathlib import Path

print("===== 스쿼트 영상 Whisper 자막 생성 =====")

# 기존에 확인한 스쿼트 영상
video_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/스쿼트자세설명영상.mp4"
)

# Whisper Small 모델 로드
whisper_model = whisper.load_model("small", device="cuda")

# 한국어 음성을 자막으로 변환
result = whisper_model.transcribe(
    str(video_path),
    language="ko",
    task="transcribe",
    fp16=True,
    verbose=False
)

# 기존 Fusion과 호환되는 자막 형식
squat_subtitles = [
    {
        "start_time": round(float(s["start"]), 2),
        "end_time": round(float(s["end"]), 2),
        "text": s["text"].strip()
    }
    for s in result["segments"]
    if s["text"].strip()
]

# Google Drive에 저장
squat_subtitle_path = (
    video_path.parent / "squat_subtitles.json"
)

squat_subtitle_path.write_text(
    json.dumps(
        squat_subtitles,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\n===== 생성된 한국어 자막 =====")

for s in squat_subtitles:
    print(
        f"{s['start_time']:.2f}~{s['end_time']:.2f}초 | "
        f"{s['text']}"
    )

print("\n자막 개수:", len(squat_subtitles))
print("저장 경로:", squat_subtitle_path)
print("\n✅ 스쿼트 영상 자막 생성 완료!")

===== 스쿼트 영상 Whisper 자막 생성 =====


100%|███████████████████████████████████████| 461M/461M [00:10<00:00, 44.4MiB/s]
100%|██████████| 4204/4204 [00:06<00:00, 662.71frames/s]



===== 생성된 한국어 자막 =====
0.00~3.66초 | 오늘은 올바른 스쿼트 자세에 대해 배워볼 거야.
3.66~8.52초 | 먼저, 발을 어깨 넙이 정도로 벌려줘, 발은 살짝 바깥으로 벌려줘.
8.52~11.04초 | 맨몸이라면 발을 11자로 놓아줘도 돼.
11.04~13.76초 | 그대로 엉덩이를 무릎이 치까지 내려주면 돼.
13.76~17.72초 | 헬린이라면 몸통이 일직선을 유지할 수 있는 정도까지만 앉아줘.
17.72~21.84초 | 앉을 때 무릎이 아프다면 통증이 없는 가동범위까지 앉으면 돼.
21.84~27.14초 | 뒷꿈치, 발가락, 발랄이 모두 바닥에 붙어있는 상태로 동작해줘.
27.14~30.02초 | 이때, 발의 뒷꿈치는 떨어지면 안 돼.
30.02~33.54초 | 무릎이 살짝 앞으로 나오면서 엉덩이를 앉아주면 돼.
33.54~36.30초 | 체중은 항상 발바닥 중앙에 오게 해줘.
36.30~39.54초 | 무릎의 방향이 악꿈치 방향을 향하게 해줘야 해.
39.54~41.98초 | 거울을 보며 자세를 점검하면서 해줘.

자막 개수: 12
저장 경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/squat_subtitles.json

✅ 스쿼트 영상 자막 생성 완료!


In [217]:
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

print("===== 기존 CLIP Feature 추출 코드 찾기 =====")

keywords = [
    "clip",
    "feature",
    "extract",
    "video"
]

for path in sorted(repo.rglob("*.py")):
    if any(word in path.name.lower() for word in keywords):
        if ".venv" not in path.parts:
            print(path.relative_to(repo))

print("\n✅ 코드 파일 확인 완료!")

===== 기존 CLIP Feature 추출 코드 찾기 =====
ai/features/check_features.py
ai/features/extract_query_features.py
ai/features/moment_detr_features.py
ai/preprocessing/build_video_data.py
ai/preprocessing/extract_audio.py
ai/preprocessing/extract_frames.py
ai/preprocessing/segment_video.py
ai/preprocessing/video_metadata.py
ai/vendor/moment_detr/run_on_video/clip/clip.py

✅ 코드 파일 확인 완료!


In [218]:
from pathlib import Path

feature_file = Path(
    "/content/pinpoint-capstone/ai/features/moment_detr_features.py"
)

print("===== Moment-DETR Feature 코드 확인 =====")

lines = feature_file.read_text(encoding="utf-8").splitlines()

print("전체 코드 줄 수:", len(lines))

for i, line in enumerate(lines[:220], 1):
    print(f"{i:3d}: {line}")

if len(lines) > 220:
    print("\n※ 나머지 코드가 있어 추가 확인이 필요합니다.")

print("\n✅ Feature 추출 코드 확인 완료!")

===== Moment-DETR Feature 코드 확인 =====
전체 코드 줄 수: 388
  1: from pathlib import Path
  2: import sys
  3: import math
  4: import subprocess
  5: 
  6: import numpy as np
  7: import torch
  8: 
  9: 
 10: # ==================================================
 11: # Paths
 12: # ==================================================
 13: 
 14: PROJECT_ROOT = Path(__file__).resolve().parents[2]
 15: 
 16: MOMENT_DETR_ROOT = (
 17:     PROJECT_ROOT
 18:     / "ai"
 19:     / "vendor"
 20:     / "moment_detr"
 21: )
 22: 
 23: if str(MOMENT_DETR_ROOT) not in sys.path:
 24:     sys.path.append(str(MOMENT_DETR_ROOT))
 25: 
 26: 
 27: from run_on_video.data_utils import ClipFeatureExtractor
 28: 
 29: 
 30: # ==================================================
 31: # Config
 32: # ==================================================
 33: 
 34: CLIP_LEN = 2.0
 35: DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
 36: 
 37: DEFAULT_OUTPUT_DIR = (
 38:     PROJECT_ROOT
 39:     / "data"
 40:     /

In [219]:
from pathlib import Path

feature_file = Path(
    "/content/pinpoint-capstone/ai/features/moment_detr_features.py"
)

lines = feature_file.read_text(encoding="utf-8").splitlines()

print("===== Moment-DETR Feature 코드 나머지 부분 =====")

for i in range(220, len(lines)):
    print(f"{i+1:3d}: {lines[i]}")

print("\n✅ Feature 생성·저장 코드 확인 완료!")

===== Moment-DETR Feature 코드 나머지 부분 =====
221:             .clip_extractor
222:             .encode_image(batch)
223:         )
224: 
225:         features.append(
226:             output
227:         )
228: 
229:     return torch.cat(
230:         features,
231:         dim=0
232:     )
233: 
234: 
235: # ==================================================
236: # Service function
237: # ==================================================
238: 
239: def build_feature(
240:     video_path,
241:     video_id,
242:     output_dir=None,
243:     overwrite=False
244: ):
245:     """
246:     MP4 영상에서 Moment-DETR용 CLIP feature를 생성한다.
247: 
248:     Args:
249:         video_path:
250:             입력 mp4 파일 경로
251: 
252:         video_id:
253:             영상 식별자.
254:             결과 파일명에 사용한다.
255: 
256:         output_dir:
257:             feature 저장 디렉터리.
258:             기본값: data/features
259: 
260:         overwrite:
261:             False이면 이미 feature가 존재할 경우
262:             다시 생성하지 않고 기존

In [220]:
%cd /content/pinpoint-capstone

# GitHub 최신 커밋 정보만 가져오기 (파일 변경 없음)
!git fetch origin main

print("\n===== 현재 Colab Git 상태 =====")
!git status --short --branch

print("\n===== Colab 현재 커밋 =====")
!git log -1 --format="%h %s"

print("\n===== GitHub main 최신 커밋 =====")
!git log -1 origin/main --format="%h %s"

print("\n===== Colab과 GitHub 커밋 차이 =====")
!git rev-list --left-right --count HEAD...origin/main

/content/pinpoint-capstone
From https://github.com/pinpoint-capstone/pinpoint-capstone
 * branch            main       -> FETCH_HEAD

===== 현재 Colab Git 상태 =====
## main...origin/main
?? ai/features/fusion_search.py
?? ai/features/fusion_search.py.bak
?? ai/features/run_moment_detr_inference_before_fusion.py
?? ai/preprocessing/transcribe_service.py
?? backend/services/

===== Colab 현재 커밋 =====
14b3eb9 fix: include CLIP tokenizer vocabulary

===== GitHub main 최신 커밋 =====
14b3eb9 fix: include CLIP tokenizer vocabulary

===== Colab과 GitHub 커밋 차이 =====
0	0


In [221]:
import sys
import torch
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from ai.features.moment_detr_features import build_feature

# 새 스쿼트 영상
video_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/스쿼트자세설명영상.mp4"
)

video_id = "kr_squat_001"

# 기존 영상들과 동일한 CLIP Feature 추출
squat_feature_path = build_feature(
    video_path=str(video_path),
    video_id=video_id,
    overwrite=False
)

# 결과 확인
features = torch.load(
    squat_feature_path,
    map_location="cpu",
    weights_only=True
)

print("\n===== 스쿼트 영상 Feature 확인 =====")
print("Feature 경로:", squat_feature_path)
print("Feature Shape:", tuple(features.shape))
print("데이터 타입:", features.dtype)
print("NaN 포함 여부:", torch.isnan(features).any().item())

print("\n✅ 스쿼트 영상 CLIP Feature 생성 완료!")

[AI] Extracting video feature: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/스쿼트자세설명영상.mp4
[AI] Loading video CLIP feature extractor on cuda
Loading CLIP models
[AI] Feature saved: /content/pinpoint-capstone/data/features/kr_squat_001_clip_features.pt
[AI] Feature shape: (21, 512)

===== 스쿼트 영상 Feature 확인 =====
Feature 경로: /content/pinpoint-capstone/data/features/kr_squat_001_clip_features.pt
Feature Shape: (21, 512)
데이터 타입: torch.float32
NaN 포함 여부: False

✅ 스쿼트 영상 CLIP Feature 생성 완료!


In [222]:
import json
from pathlib import Path

# 새 영상 평가 정보
squat_test_cases = [
    {
        "video_id": "kr_squat_001",
        "korean_query": "발을 어깨너비로 벌리는 장면",
        "english_query": "spreading feet shoulder-width apart",
        "gt_start": 3.0,
        "gt_end": 7.0
    },
    {
        "video_id": "kr_squat_001",
        "korean_query": "엉덩이를 낮춰 스쿼트하는 장면",
        "english_query": "lowering the hips into a squat",
        "gt_start": 11.0,
        "gt_end": 21.0
    },
    {
        "video_id": "kr_squat_001",
        "korean_query": "뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면",
        "english_query": "explaining how to keep the heels on the ground",
        "gt_start": 22.0,
        "gt_end": 30.0
    }
]

# Google Drive에 정답 데이터 저장
gt_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/squat_ground_truth.json"
)

gt_path.write_text(
    json.dumps(squat_test_cases, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("===== 스쿼트 영상 정답 데이터 =====")

for i, case in enumerate(squat_test_cases, 1):
    print(f"\n검색 {i}")
    print("한국어:", case["korean_query"])
    print("영어:", case["english_query"])
    print(f"정답 구간: {case['gt_start']}~{case['gt_end']}초")

print("\n저장 경로:", gt_path)
print("✅ 평가 데이터 저장 완료!")

===== 스쿼트 영상 정답 데이터 =====

검색 1
한국어: 발을 어깨너비로 벌리는 장면
영어: spreading feet shoulder-width apart
정답 구간: 3.0~7.0초

검색 2
한국어: 엉덩이를 낮춰 스쿼트하는 장면
영어: lowering the hips into a squat
정답 구간: 11.0~21.0초

검색 3
한국어: 뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면
영어: explaining how to keep the heels on the ground
정답 구간: 22.0~30.0초

저장 경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/squat_ground_truth.json
✅ 평가 데이터 저장 완료!


In [223]:
import sys
import json
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from ai.features.run_moment_detr_inference import predict_moments_hybrid

# 저장된 정답 데이터 불러오기
gt_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/squat_ground_truth.json"
)

squat_test_cases = json.loads(gt_path.read_text(encoding="utf-8"))

# 영상 정보
squat_feature_path = str(
    repo / "data/features/kr_squat_001_clip_features.pt"
)
squat_subtitle_path = str(
    Path(
        "/content/drive/MyDrive/pinpoint_capstone/"
        "02_Korean_Video_Test/squat_subtitles.json"
    )
)

def calculate_iou(pred_start, pred_end, gt_start, gt_end):
    intersection = max(
        0.0,
        min(pred_end, gt_end) - max(pred_start, gt_start)
    )
    union = max(pred_end, gt_end) - min(pred_start, gt_start)
    return intersection / union if union > 0 else 0.0

squat_hybrid_results = []

for i, case in enumerate(squat_test_cases, 1):
    print(f"\n===== 검색 {i}: {case['korean_query']} =====")
    print(f"정답: {case['gt_start']}~{case['gt_end']}초")

    results = predict_moments_hybrid(
        video_id="kr_squat_001",
        video_feature_path=squat_feature_path,
        query=case["english_query"],
        subtitle_json_path=squat_subtitle_path,
        video_duration=42.0,
        top_k=5
    )

    squat_hybrid_results.append(results)

    print("반환 타입:", type(results).__name__)
    print("검색 결과:")
    print(results)

print("\n✅ 스쿼트 영상 Hybrid 검색 완료!")


===== 검색 1: 발을 어깨너비로 벌리는 장면 =====
정답: 3.0~7.0초
반환 타입: list
검색 결과:
[{'video_id': 'kr_squat_001', 'query': 'spreading feet shoulder-width apart', 'start_time': 10.5, 'end_time': 19.8222, 'score': 0.814, 'source': 'visual', 'subtitle_score': 0.0}, {'video_id': 'kr_squat_001', 'query': 'spreading feet shoulder-width apart', 'start_time': 20.5, 'end_time': 30.5083, 'score': 0.5893, 'source': 'visual', 'subtitle_score': 0.0}, {'video_id': 'kr_squat_001', 'query': 'spreading feet shoulder-width apart', 'start_time': 0.5, 'end_time': 9.5745, 'score': 0.0851, 'source': 'visual', 'subtitle_score': 0.0}, {'video_id': 'kr_squat_001', 'query': 'spreading feet shoulder-width apart', 'start_time': 34.5, 'end_time': 40.7119, 'score': 0.002, 'source': 'visual', 'subtitle_score': 0.0}]

===== 검색 2: 엉덩이를 낮춰 스쿼트하는 장면 =====
정답: 11.0~21.0초
반환 타입: list
검색 결과:
[{'video_id': 'kr_squat_001', 'query': 'lowering the hips into a squat', 'start_time': 0.0, 'end_time': 5.7956, 'score': 0.9684, 'source': 'visual', '

In [224]:
import sys
import json
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from ai.features import fusion_search

# 저장된 스쿼트 영상 데이터
test_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

cases = json.loads(
    (test_dir / "squat_ground_truth.json").read_text(encoding="utf-8")
)

subtitles = json.loads(
    (test_dir / "squat_subtitles.json").read_text(encoding="utf-8")
)

feature_path = str(
    repo / "data/features/kr_squat_001_clip_features.pt"
)

def temporal_iou(pred, gt):
    start = max(pred["start_time"], gt["gt_start"])
    end = min(pred["end_time"], gt["gt_end"])

    intersection = max(0.0, end - start)
    union = max(
        pred["end_time"], gt["gt_end"]
    ) - min(
        pred["start_time"], gt["gt_start"]
    )

    return intersection / union if union > 0 else 0.0

squat_fusion_results = []

for i, case in enumerate(cases, 1):
    print(f"\n===== 검색 {i}: {case['korean_query']} =====")
    print(f"정답: {case['gt_start']}~{case['gt_end']}초")

    predictions = fusion_search.predict_moments_fusion(
        video_id="kr_squat_001",
        feature_path=feature_path,
        subtitles=subtitles,
        korean_query=case["korean_query"],
        english_query=case["english_query"],
        video_duration=42.0,
        subtitle_json_path=str(test_dir / "squat_subtitles.json"),
        top_k=5
    )

    squat_fusion_results.append(predictions)

    for rank, pred in enumerate(predictions, 1):
        iou = temporal_iou(pred, case)

        print(
            f"{rank}위: "
            f"{pred['start_time']:.2f}~{pred['end_time']:.2f}초 "
            f"| IoU={iou:.4f}"
        )

    top1 = temporal_iou(predictions[0], case) if predictions else 0
    best5 = max(
        [temporal_iou(pred, case) for pred in predictions],
        default=0
    )

    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

print("\n✅ 기존 Fusion 검색 평가 완료!")


===== 검색 1: 발을 어깨너비로 벌리는 장면 =====
정답: 3.0~7.0초
1위: 10.50~19.82초 | IoU=0.0000
2위: 3.66~8.52초 | IoU=0.6051
3위: 3.66~13.76초 | IoU=0.3104
4위: 20.50~30.51초 | IoU=0.0000
5위: 34.50~40.71초 | IoU=0.0000
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.6051

===== 검색 2: 엉덩이를 낮춰 스쿼트하는 장면 =====
정답: 11.0~21.0초
1위: 0.00~5.80초 | IoU=0.0000
2위: 11.04~13.76초 | IoU=0.2720
3위: 30.02~33.54초 | IoU=0.0000
4위: 22.50~30.86초 | IoU=0.0000
5위: 3.66~8.52초 | IoU=0.0000
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.2720

===== 검색 3: 뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면 =====
정답: 22.0~30.0초
1위: 32.50~41.89초 | IoU=0.0000
2위: 11.04~13.76초 | IoU=0.0000
3위: 21.84~33.54초 | IoU=0.6838
4위: 6.50~19.49초 | IoU=0.0000
5위: 21.84~27.14초 | IoU=0.6299
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.6838

✅ 기존 Fusion 검색 평가 완료!


In [225]:
import sys
import json
from pathlib import Path
from unittest.mock import patch
from sentence_transformers import SentenceTransformer

repo = Path("/content/pinpoint-capstone")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from ai.features import fusion_search

test_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

model_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned/final_model"
)

cases = json.loads(
    (test_dir / "squat_ground_truth.json").read_text(encoding="utf-8")
)

subtitles = json.loads(
    (test_dir / "squat_subtitles.json").read_text(encoding="utf-8")
)

feature_path = str(
    repo / "data/features/kr_squat_001_clip_features.pt"
)

assert model_dir.exists(), "Fine-tuning MiniLM 모델 경로 확인 필요"

# 기존에 학습한 MiniLM 로드
trained_model = SentenceTransformer(str(model_dir), device="cuda")

def temporal_iou(pred, gt):
    intersection = max(
        0.0,
        min(pred["end_time"], gt["gt_end"])
        - max(pred["start_time"], gt["gt_start"])
    )
    union = (
        max(pred["end_time"], gt["gt_end"])
        - min(pred["start_time"], gt["gt_start"])
    )
    return intersection / union if union > 0 else 0.0

squat_finetuned_results = []

for i, case in enumerate(cases, 1):
    print(f"\n===== 검색 {i}: {case['korean_query']} =====")
    print(f"정답: {case['gt_start']}~{case['gt_end']}초")

    # 이번 실행에서만 Fine-tuning MiniLM 사용
    with patch.object(
        fusion_search,
        "get_semantic_model",
        return_value=trained_model
    ):
        predictions = fusion_search.predict_moments_fusion(
            video_id="kr_squat_001",
            feature_path=feature_path,
            subtitles=subtitles,
            korean_query=case["korean_query"],
            english_query=case["english_query"],
            video_duration=42.0,
            subtitle_json_path=str(test_dir / "squat_subtitles.json"),
            top_k=5
        )

    squat_finetuned_results.append(predictions)

    for rank, pred in enumerate(predictions, 1):
        iou = temporal_iou(pred, case)
        print(
            f"{rank}위: "
            f"{pred['start_time']:.2f}~{pred['end_time']:.2f}초 "
            f"| IoU={iou:.4f}"
        )

    top1 = temporal_iou(predictions[0], case) if predictions else 0
    best5 = max(
        [temporal_iou(pred, case) for pred in predictions],
        default=0.0
    )

    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

print("\n✅ Fine-tuning MiniLM Fusion 평가 완료!")

The tokenizer you are loading from '/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



===== 검색 1: 발을 어깨너비로 벌리는 장면 =====
정답: 3.0~7.0초
1위: 10.50~19.82초 | IoU=0.0000
2위: 3.66~8.52초 | IoU=0.6051
3위: 20.50~30.51초 | IoU=0.0000
4위: 27.14~30.02초 | IoU=0.0000
5위: 3.66~13.76초 | IoU=0.3104
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.6051

===== 검색 2: 엉덩이를 낮춰 스쿼트하는 장면 =====
정답: 11.0~21.0초
1위: 0.00~5.80초 | IoU=0.0000
2위: 30.02~33.54초 | IoU=0.0000
3위: 22.50~30.86초 | IoU=0.0000
4위: 11.04~13.76초 | IoU=0.2720
5위: 32.50~40.36초 | IoU=0.0000
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.2720

===== 검색 3: 뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면 =====
정답: 22.0~30.0초
1위: 32.50~41.89초 | IoU=0.0000
2위: 27.14~33.54초 | IoU=0.2478
3위: 30.02~36.30초 | IoU=0.0000
4위: 6.50~19.49초 | IoU=0.0000
5위: 33.54~36.30초 | IoU=0.0000
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.2478

✅ Fine-tuning MiniLM Fusion 평가 완료!


In [226]:
print("===== Consensus Fusion 함수 확인 =====")

if "rank_by_consensus" in globals():
    print("✅ rank_by_consensus 함수가 현재 Colab에 존재합니다.")
else:
    print("❌ rank_by_consensus 함수가 없습니다.")
    print("기존 함수 코드를 다시 불러와야 합니다.")

print("\n===== 기존 평가 결과 변수 확인 =====")

for name in [
    "squat_hybrid_results",
    "squat_fusion_results",
    "squat_finetuned_results",
    "trained_model"
]:
    print(f"{name}: {'✅ 있음' if name in globals() else '❌ 없음'}")

===== Consensus Fusion 함수 확인 =====
✅ rank_by_consensus 함수가 현재 Colab에 존재합니다.

===== 기존 평가 결과 변수 확인 =====
squat_hybrid_results: ✅ 있음
squat_fusion_results: ✅ 있음
squat_finetuned_results: ✅ 있음
trained_model: ✅ 있음


In [227]:
import inspect

print("===== Consensus Fusion 함수 구조 =====")

print("함수 입력값:")
print(inspect.signature(rank_by_consensus))

print("\n===== 함수 코드 =====")
print(inspect.getsource(rank_by_consensus))

print("\n✅ Consensus Fusion 함수 확인 완료!")

===== Consensus Fusion 함수 구조 =====
함수 입력값:
(hybrid, ko, en)

===== 함수 코드 =====
def rank_by_consensus(hybrid, ko, en):
    """
    Hybrid / 한국어 MiniLM / 영어 MiniLM 후보를
    순위 및 시간 구간 겹침 기준으로 재정렬한다.

    정답 구간(ground truth)은 사용하지 않는다.
    """

    groups = {
        "hybrid": hybrid,
        "ko": ko,
        "en": en
    }

    candidates = []

    # 각 검색 방식 내부 순위 점수
    for source, group in groups.items():
        for rank, candidate in enumerate(group, 1):
            item = dict(candidate)

            item["rank_source"] = source
            item["original_rank"] = rank

            # 순위가 높을수록 높은 기본 점수
            item["rank_score"] = 1.0 / rank

            candidates.append(item)

    # 서로 다른 검색 방식의 후보가 겹치면 가점
    for candidate in candidates:
        support = 0.0

        for source, group in groups.items():

            if source == candidate["rank_source"]:
                continue

            best_overlap = max(
                (
                    fusion_search.temporal_iou(

In [228]:
import inspect
from ai.features import fusion_search

print("===== Fusion 후보 생성 함수 확인 =====")

for name in [
    "get_semantic_subtitle_candidates",
    "get_multilingual_subtitle_candidates"
]:
    func = getattr(fusion_search, name, None)

    if func is None:
        print(f"\n❌ {name}: 함수 없음")
    else:
        print(f"\n✅ {name}")
        print("입력값:", inspect.signature(func))
        print(inspect.getsource(func))

print("\n✅ 후보 생성 함수 확인 완료!")

===== Fusion 후보 생성 함수 확인 =====

✅ get_semantic_subtitle_candidates
입력값: (query, subtitles, model, top_k=5)
def get_semantic_subtitle_candidates(
    query,
    subtitles,
    model,
    top_k=5
):
    windows = []

    for size in (1, 2, 3):
        for i in range(len(subtitles) - size + 1):
            group = subtitles[i:i + size]

            windows.append({
                "start_time": group[0]["start_time"],
                "end_time": group[-1]["end_time"],
                "text": " ".join(s["text"] for s in group)
            })

    embeddings = model.encode(
        [w["text"] for w in windows],
        convert_to_tensor=True
    )

    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )[0]

    candidates = []

    for idx in scores.topk(min(top_k, len(windows))).indices.tolist():
        w = windows[idx]

        candidates.append({
            **w,
            "sc

In [229]:
from ai.features import fusion_search
import json
from pathlib import Path

test_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test"
)

cases = json.loads(
    (test_dir / "squat_ground_truth.json").read_text(encoding="utf-8")
)

subtitles = json.loads(
    (test_dir / "squat_subtitles.json").read_text(encoding="utf-8")
)

# 98단계의 Hybrid 결과와 100단계의 Fine-tuned MiniLM 사용
assert len(squat_hybrid_results) == len(cases) == 3

squat_consensus_results = []

def squat_iou(pred, gt):
    intersection = max(
        0.0,
        min(pred["end_time"], gt["gt_end"])
        - max(pred["start_time"], gt["gt_start"])
    )
    union = (
        max(pred["end_time"], gt["gt_end"])
        - min(pred["start_time"], gt["gt_start"])
    )
    return intersection / union if union > 0 else 0.0

for i, case in enumerate(cases):
    # 기존 Hybrid 후보 그대로 사용
    hybrid = squat_hybrid_results[i]

    # 추가 학습 MiniLM으로 한국어·영어 후보 각각 생성
    ko = fusion_search.get_semantic_subtitle_candidates(
        query=case["korean_query"],
        subtitles=subtitles,
        model=trained_model,
        top_k=5
    )

    en = fusion_search.get_semantic_subtitle_candidates(
        query=case["english_query"],
        subtitles=subtitles,
        model=trained_model,
        top_k=5
    )

    # 기존 Consensus 함수로 재정렬
    ranked = rank_by_consensus(hybrid, ko, en)

    # 기존 실험과 동일하게 IoU 0.5 이상 중복 후보 제거
    predictions = []

    for candidate in ranked:
        if all(
            fusion_search.temporal_iou(candidate, selected) < 0.5
            for selected in predictions
        ):
            predictions.append(candidate)

        if len(predictions) == 5:
            break

    squat_consensus_results.append(predictions)

    print(f"\n===== 검색 {i+1}: {case['korean_query']} =====")
    print(f"정답: {case['gt_start']}~{case['gt_end']}초")

    for rank, pred in enumerate(predictions, 1):
        iou = squat_iou(pred, case)

        print(
            f"{rank}위: "
            f"{pred['start_time']:.2f}~{pred['end_time']:.2f}초 "
            f"| 출처={pred['rank_source']} "
            f"| IoU={iou:.4f} "
            f"| Consensus={pred['consensus_score']:.4f}"
        )

    top1 = squat_iou(predictions[0], case) if predictions else 0.0
    best5 = max(
        [squat_iou(p, case) for p in predictions],
        default=0.0
    )

    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

print("\n✅ 스쿼트 영상 Consensus Fusion 평가 완료!")


===== 검색 1: 발을 어깨너비로 벌리는 장면 =====
정답: 3.0~7.0초
1위: 3.66~8.52초 | 출처=ko | IoU=0.6051 | Consensus=2.5356
2위: 20.50~30.51초 | 출처=hybrid | IoU=0.0000 | Consensus=1.5305
3위: 3.66~13.76초 | 출처=en | IoU=0.3104 | Consensus=1.5101
4위: 10.50~19.82초 | 출처=hybrid | IoU=0.0000 | Consensus=1.2351
5위: 27.14~30.02초 | 출처=ko | IoU=0.0000 | Consensus=1.1022
Top-1 IoU: 0.6051
Best Top-5 IoU: 0.6051

===== 검색 2: 엉덩이를 낮춰 스쿼트하는 장면 =====
정답: 11.0~21.0초
1위: 30.02~33.54초 | 출처=ko | IoU=0.0000 | Consensus=2.1006
2위: 0.00~5.80초 | 출처=hybrid | IoU=0.0000 | Consensus=1.8354
3위: 11.04~13.76초 | 출처=ko | IoU=0.2720 | Consensus=1.6687
4위: 3.66~13.76초 | 출처=en | IoU=0.1592 | Consensus=0.9415
5위: 22.50~30.86초 | 출처=hybrid | IoU=0.0000 | Consensus=0.9126
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.2720

===== 검색 3: 뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면 =====
정답: 22.0~30.0초
1위: 27.14~33.54초 | 출처=ko | IoU=0.2478 | Consensus=2.0705
2위: 30.02~36.30초 | 출처=en | IoU=0.0000 | Consensus=2.0058
3위: 32.50~41.89초 | 출처=hybrid | IoU=0.0000 | Consensus=1.5779
4위: 

In [230]:
import json
from pathlib import Path

save_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

save_dir.mkdir(parents=True, exist_ok=True)

methods = {
    "hybrid": squat_hybrid_results,
    "original_fusion": squat_fusion_results,
    "finetuned_fusion": squat_finetuned_results,
    "consensus_fusion": squat_consensus_results
}

summary = {}
saved_results = {}

for method_name, all_predictions in methods.items():
    top1_ious = []
    best5_ious = []
    details = []

    for case, predictions in zip(squat_test_cases, all_predictions):
        ious = [
            temporal_iou(p, case)
            for p in predictions
        ]

        top1 = ious[0] if ious else 0.0
        best5 = max(ious, default=0.0)

        top1_ious.append(top1)
        best5_ious.append(best5)

        details.append({
            "query": case["korean_query"],
            "ground_truth": [
                case["gt_start"],
                case["gt_end"]
            ],
            "top1_iou": round(top1, 4),
            "best5_iou": round(best5, 4),
            "predictions": predictions
        })

    summary[method_name] = {
        "mean_top1_iou": round(
            sum(top1_ious) / len(top1_ious), 4
        ),
        "mean_best5_iou": round(
            sum(best5_ious) / len(best5_ious), 4
        ),
        "r1_at_05": sum(x >= 0.5 for x in top1_ious),
        "r5_at_05": sum(x >= 0.5 for x in best5_ious),
        "total_queries": len(top1_ious)
    }

    saved_results[method_name] = details

output = {
    "video_id": "kr_squat_001",
    "video_duration": 42.0,
    "summary": summary,
    "results": saved_results
}

save_path = save_dir / "squat_4methods_comparison.json"

save_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("===== 모델별 성능 요약 =====")

for name, metrics in summary.items():
    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

print("\n저장 경로:", save_path)
print("✅ 스쿼트 영상 4가지 모델 평가 결과 백업 완료!")

===== 모델별 성능 요약 =====

hybrid
  mean_top1_iou: 0.0
  mean_best5_iou: 0.3473
  r1_at_05: 0
  r5_at_05: 1
  total_queries: 3

original_fusion
  mean_top1_iou: 0.0
  mean_best5_iou: 0.5203
  r1_at_05: 0
  r5_at_05: 2
  total_queries: 3

finetuned_fusion
  mean_top1_iou: 0.0
  mean_best5_iou: 0.375
  r1_at_05: 0
  r5_at_05: 1
  total_queries: 3

consensus_fusion
  mean_top1_iou: 0.2843
  mean_best5_iou: 0.6184
  r1_at_05: 1
  r5_at_05: 2
  total_queries: 3

저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/squat_4methods_comparison.json
✅ 스쿼트 영상 4가지 모델 평가 결과 백업 완료!


In [231]:
from ai.features import fusion_search

print("===== 검색 방식별 후보 비교 =====")

for i, case in enumerate(squat_test_cases):
    print(f"\n{'='*60}")
    print(f"검색 {i+1}: {case['korean_query']}")
    print(f"정답: {case['gt_start']}~{case['gt_end']}초")

    hybrid = squat_hybrid_results[i]

    ko = fusion_search.get_semantic_subtitle_candidates(
        query=case["korean_query"],
        subtitles=squat_subtitles,
        model=trained_model,
        top_k=5
    )

    en = fusion_search.get_semantic_subtitle_candidates(
        query=case["english_query"],
        subtitles=squat_subtitles,
        model=trained_model,
        top_k=5
    )

    for name, predictions in [
        ("Hybrid", hybrid),
        ("한국어 MiniLM", ko),
        ("영어 MiniLM", en)
    ]:
        print(f"\n[{name}]")

        for rank, pred in enumerate(predictions, 1):
            iou = temporal_iou(pred, case)

            print(
                f"{rank}위 | "
                f"{pred['start_time']:.2f}~{pred['end_time']:.2f}초 "
                f"| IoU={iou:.4f} "
                f"| 점수={pred['score']:.4f}"
            )

print("\n✅ 검색 방식별 후보 분석 완료!")

===== 검색 방식별 후보 비교 =====

검색 1: 발을 어깨너비로 벌리는 장면
정답: 3.0~7.0초

[Hybrid]
1위 | 10.50~19.82초 | IoU=0.0000 | 점수=0.8140
2위 | 20.50~30.51초 | IoU=0.0000 | 점수=0.5893
3위 | 0.50~9.57초 | IoU=0.4408 | 점수=0.0851
4위 | 34.50~40.71초 | IoU=0.0000 | 점수=0.0020

[한국어 MiniLM]
1위 | 3.66~8.52초 | IoU=0.6051 | 점수=0.5743
2위 | 27.14~30.02초 | IoU=0.0000 | 점수=0.5102
3위 | 3.66~11.04초 | IoU=0.4154 | 점수=0.4994
4위 | 21.84~30.02초 | IoU=0.0000 | 점수=0.4573
5위 | 0.00~8.52초 | IoU=0.4695 | 점수=0.4552

[영어 MiniLM]
1위 | 3.66~8.52초 | IoU=0.6051 | 점수=0.5235
2위 | 3.66~11.04초 | IoU=0.4154 | 점수=0.4589
3위 | 3.66~13.76초 | IoU=0.3104 | 점수=0.4376
4위 | 27.14~36.30초 | IoU=0.0000 | 점수=0.4275
5위 | 33.54~39.54초 | IoU=0.0000 | 점수=0.4232

검색 2: 엉덩이를 낮춰 스쿼트하는 장면
정답: 11.0~21.0초

[Hybrid]
1위 | 0.00~5.80초 | IoU=0.0000 | 점수=0.9684
2위 | 22.50~30.86초 | IoU=0.0000 | 점수=0.9532
3위 | 32.50~40.36초 | IoU=0.0000 | 점수=0.9251
4위 | 12.50~18.51초 | IoU=0.6011 | 점수=0.8874

[한국어 MiniLM]
1위 | 30.02~33.54초 | IoU=0.0000 | 점수=0.5092
2위 | 11.04~13.76초 | IoU=0.2720 | 점수

In [232]:
from ai.features import fusion_search

print("===== 한국어 자막 우선 검색 실험 =====")

squat_korean_first_results = []

for i, case in enumerate(squat_test_cases):

    hybrid = squat_hybrid_results[i]

    ko = fusion_search.get_semantic_subtitle_candidates(
        query=case["korean_query"],
        subtitles=squat_subtitles,
        model=trained_model,
        top_k=5
    )

    en = fusion_search.get_semantic_subtitle_candidates(
        query=case["english_query"],
        subtitles=squat_subtitles,
        model=trained_model,
        top_k=5
    )

    # 정답 구간을 사용하지 않는 고정된 후보 순서
    ordered = []

    for rank in range(5):
        for group in [ko, hybrid, en]:
            if rank < len(group):
                ordered.append(group[rank])

    # 기존과 동일한 중복 제거
    predictions = []

    for candidate in ordered:
        if all(
            fusion_search.temporal_iou(candidate, selected) < 0.5
            for selected in predictions
        ):
            predictions.append(candidate)

        if len(predictions) == 5:
            break

    squat_korean_first_results.append(predictions)

    top1 = temporal_iou(predictions[0], case) if predictions else 0.0
    best5 = max(
        [temporal_iou(p, case) for p in predictions],
        default=0.0
    )

    print(f"\n검색 {i+1}: {case['korean_query']}")
    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

print("\n✅ 한국어 자막 우선 검색 실험 완료!")

===== 한국어 자막 우선 검색 실험 =====

검색 1: 발을 어깨너비로 벌리는 장면
Top-1 IoU: 0.6051
Best Top-5 IoU: 0.6051

검색 2: 엉덩이를 낮춰 스쿼트하는 장면
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.2720

검색 3: 뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면
Top-1 IoU: 0.2478
Best Top-5 IoU: 0.2478

✅ 한국어 자막 우선 검색 실험 완료!


In [233]:
import json
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

files = [
    "minilm_fusion_6queries_comparison.json",
    "fusion_new_3queries_comparison.json",
    "squat_4methods_comparison.json"
]

print("===== 기존 영상 + 스쿼트 실험 결과 확인 =====")

for filename in files:
    path = training_dir / filename

    print(f"\n{'='*60}")
    print("파일:", filename)
    print("존재 여부:", path.exists())

    if not path.exists():
        continue

    data = json.loads(path.read_text(encoding="utf-8"))

    print("최상위 항목:", list(data.keys()))

    # 저장된 결과의 구조를 먼저 확인
    for key, value in data.items():
        print(
            f"  {key}: "
            f"{type(value).__name__}"
        )

        if isinstance(value, dict):
            print("    하위 항목:", list(value.keys())[:10])

        elif isinstance(value, list):
            print("    항목 개수:", len(value))

print("\n✅ 기존 실험 결과 구조 확인 완료!")

===== 기존 영상 + 스쿼트 실험 결과 확인 =====

파일: minilm_fusion_6queries_comparison.json
존재 여부: True
최상위 항목: ['test_count', 'summary', 'results']
  test_count: int
  summary: dict
    하위 항목: ['기존 Fusion', 'Fine-tuning Fusion']
  results: list
    항목 개수: 12

파일: fusion_new_3queries_comparison.json
존재 여부: True
최상위 항목: ['test_count', 'ground_truth_source', 'summary', 'results']
  test_count: int
  ground_truth_source: str
  summary: dict
    하위 항목: ['기존 Fusion', 'Fine-tuning Fusion', 'Consensus Fusion']
  results: list
    항목 개수: 9

파일: squat_4methods_comparison.json
존재 여부: True
최상위 항목: ['video_id', 'video_duration', 'summary', 'results']
  video_id: str
  video_duration: float
  summary: dict
    하위 항목: ['hybrid', 'original_fusion', 'finetuned_fusion', 'consensus_fusion']
  results: dict
    하위 항목: ['hybrid', 'original_fusion', 'finetuned_fusion', 'consensus_fusion']

✅ 기존 실험 결과 구조 확인 완료!


In [234]:
import json
from pathlib import Path
from pprint import pprint

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

files = {
    "기존 영상 6개 검색어": "minilm_fusion_6queries_comparison.json",
    "기존 영상 추가 3개 검색어": "fusion_new_3queries_comparison.json",
    "새 스쿼트 영상 3개 검색어": "squat_4methods_comparison.json"
}

print("===== 전체 실험 성능 요약 =====")

for experiment, filename in files.items():
    path = training_dir / filename

    data = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n{'=' * 60}")
    print("실험:", experiment)
    print("파일:", filename)

    print("\n[모델별 저장된 성능 지표]")

    for model_name, metrics in data["summary"].items():
        print(f"\n▶ {model_name}")
        pprint(metrics, sort_dicts=False)

print("\n✅ 전체 실험 성능 요약 확인 완료!")

===== 전체 실험 성능 요약 =====

실험: 기존 영상 6개 검색어
파일: minilm_fusion_6queries_comparison.json

[모델별 저장된 성능 지표]

▶ 기존 Fusion
{'mean_top1_iou': 0.2539795296562724,
 'mean_top5_best_iou': 0.5883301663005404,
 'recall_at_1_iou05': 0.16666666666666666,
 'recall_at_5_iou05': 0.6666666666666666}

▶ Fine-tuning Fusion
{'mean_top1_iou': 0.2539795296562724,
 'mean_top5_best_iou': 0.5836585887473009,
 'recall_at_1_iou05': 0.16666666666666666,
 'recall_at_5_iou05': 0.6666666666666666}

실험: 기존 영상 추가 3개 검색어
파일: fusion_new_3queries_comparison.json

[모델별 저장된 성능 지표]

▶ 기존 Fusion
{'mean_top1_iou': 0.1870646759240543,
 'mean_top5_best_iou': 0.2884093519728609,
 'recall_at_1_iou05': 0.0,
 'recall_at_5_iou05': 0.0}

▶ Fine-tuning Fusion
{'mean_top1_iou': 0.1870646759240543,
 'mean_top5_best_iou': 0.2322166257887001,
 'recall_at_1_iou05': 0.0,
 'recall_at_5_iou05': 0.0}

▶ Consensus Fusion
{'mean_top1_iou': 0.07816772668821463,
 'mean_top5_best_iou': 0.3121500146313312,
 'recall_at_1_iou05': 0.0,
 'recall_at_5_iou05

In [235]:
import json
from pathlib import Path
from pprint import pprint

path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/"
    "minilm_fusion_6queries_comparison.json"
)

data = json.loads(path.read_text(encoding="utf-8"))

print("===== 기존 6개 검색어 평가 데이터 확인 =====")

print("\n전체 결과 개수:", len(data["results"]))

print("\n첫 번째 결과 예시:")
pprint(data["results"][0], sort_dicts=False)

print("\n===== Colab 기존 변수 확인 =====")

for name in [
    "test_cases",
    "subtitle_map",
    "duration_map",
    "video_config"
]:
    print(
        f"{name}: "
        f"{'✅ 있음' if name in globals() else '❌ 없음'}"
    )

print("\n✅ 평가 데이터 구조 확인 완료!")

===== 기존 6개 검색어 평가 데이터 확인 =====

전체 결과 개수: 12

첫 번째 결과 예시:
{'name': '두부 자르기',
 'model': '기존 Fusion',
 'top1_iou': 0.0,
 'top5_best_iou': 0.7436363636363634,
 'recall_at_1': 0,
 'recall_at_5': 1,
 'candidates': [{'video_id': 'kr_tofu_001',
                 'query': '두부를 칼로 자르는 장면',
                 'start_time': 98.5,
                 'end_time': 118.9581,
                 'score': 0.9842,
                 'source': 'hybrid',
                 'hybrid_source': 'visual',
                 'visual_score': 0.9842,
                 'subtitle_score': None,
                 'language': None},
                {'video_id': 'kr_tofu_001',
                 'query': '두부를 칼로 자르는 장면',
                 'start_time': 140.86,
                 'end_time': 143.58,
                 'score': None,
                 'source': 'subtitle',
                 'hybrid_source': None,
                 'visual_score': None,
                 'subtitle_score': 0.6732,
                 'language': 'ko'},
                {

# 기존 6개 검색어 Consensus 평가 준비

In [236]:
print("===== 기존 6개 검색어 평가 데이터 확인 =====")

print("\n[1] test_cases")
print("개수:", len(test_cases))
for i, case in enumerate(test_cases):
    print(f"{i+1}. {case}")

print("\n[2] subtitle_map")
for key, value in subtitle_map.items():
    print(
        f"{key}: "
        f"타입={type(value).__name__}, "
        f"개수={len(value) if hasattr(value, '__len__') else '확인 불가'}"
    )

print("\n[3] duration_map")
print(duration_map)

print("\n[4] video_config")
print(video_config)

print("\n✅ 기존 평가 데이터 확인 완료!")

===== 기존 6개 검색어 평가 데이터 확인 =====

[1] test_cases
개수: 6
1. {'name': '두부 자르기', 'query_ko': '두부를 칼로 자르는 장면', 'query_en': 'cutting tofu with a knife', 'subtitles': [{'start_time': 0.0, 'end_time': 4.74, 'text': '여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면'}, {'start_time': 4.74, 'end_time': 8.7, 'text': '순식간에 정말 맛있는 두부 요리 하나가 완성돼요'}, {'start_time': 8.7, 'end_time': 12.88, 'text': '고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요'}, {'start_time': 12.88, 'end_time': 16.32, 'text': '먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요'}, {'start_time': 16.32, 'end_time': 19.02, 'text': '그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요'}, {'start_time': 19.02, 'end_time': 21.06, 'text': '두께는 젓가락 하나 정도면 딱 좋아요'}, {'start_time': 21.06, 'end_time': 23.72, 'text': '다 썰었으면 접시에 담아 잠시 놔둘게요'}, {'start_time': 23.72, 'end_time': 26.46, 'text': '제가 산두부는 한 모에 600번 정도 줬는데요'}, {'start_time': 26.46, 'end_time': 30.74, 'text': '여러분이 계신 곳에서는 두부 한 모에 얼마 정도 하나요?'}, {'start_time': 30.74, 'end_time': 34.14, 'text': '댓글로 가격도 한번 같이 이야기 해봐요 이제부터가 중요한데요'}, {'start_time': 34.14,

In [237]:
from pathlib import Path
from ai.features import fusion_search
from ai.features.run_moment_detr_inference import predict_moments_hybrid

repo = Path("/content/pinpoint-capstone")

# 기존 6개 검색어의 영상 ID
video_ids = [
    "kr_tofu_001",
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_kitchen_001",
    "kr_daiso_001",
    "kr_daiso_001"
]

consensus_6results = []

for i, case in enumerate(test_cases):
    video_id = video_ids[i]

    feature_path = str(
        repo / "data/features" / f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        str(repo / "data/processed/kr_daiso_001/subtitles.json")
        if video_id == "kr_daiso_001"
        else None
    )

    # 1. 기존 Hybrid 후보 생성
    hybrid = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=duration_map[video_id],
        top_k=5
    )

    # 2. Fine-tuning MiniLM 후보 생성
    ko = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_ko"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    en = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    # 3. 기존 Consensus 순위 계산
    ranked = rank_by_consensus(hybrid, ko, en)

    # 4. 기존 실험과 동일하게 중복 구간 제거
    predictions = []

    for candidate in ranked:
        if all(
            fusion_search.temporal_iou(candidate, selected) < 0.5
            for selected in predictions
        ):
            predictions.append(candidate)

        if len(predictions) == 5:
            break

    # 5. 정답 구간 평가
    gt = case["ground_truth"]

    gt_start = gt.get("start_time", gt.get("gt_start"))
    gt_end = gt.get("end_time", gt.get("gt_end"))

    def calc_iou(pred):
        intersection = max(
            0.0,
            min(pred["end_time"], gt_end)
            - max(pred["start_time"], gt_start)
        )

        union = (
            max(pred["end_time"], gt_end)
            - min(pred["start_time"], gt_start)
        )

        return intersection / union if union > 0 else 0.0

    top1 = calc_iou(predictions[0]) if predictions else 0.0
    best5 = max([calc_iou(p) for p in predictions], default=0.0)

    consensus_6results.append({
        "name": case["name"],
        "video_id": video_id,
        "top1_iou": top1,
        "best5_iou": best5,
        "predictions": predictions
    })

    print(f"\n검색 {i+1}: {case['name']}")
    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

# 전체 성능
n = len(consensus_6results)

print("\n===== 기존 6개 검색어 Consensus 성능 =====")

print(
    "평균 Top-1 IoU:",
    round(sum(r["top1_iou"] for r in consensus_6results) / n, 4)
)

print(
    "평균 Best Top-5 IoU:",
    round(sum(r["best5_iou"] for r in consensus_6results) / n, 4)
)

print(
    "R1@0.5:",
    sum(r["top1_iou"] >= 0.5 for r in consensus_6results),
    "/", n
)

print(
    "R5@0.5:",
    sum(r["best5_iou"] >= 0.5 for r in consensus_6results),
    "/", n
)

print("\n✅ 기존 6개 검색어 Consensus 평가 완료!")


검색 1: 두부 자르기
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.3127

검색 2: 두부 양념 준비
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.5164

검색 3: 가위로 소스 짜기
Top-1 IoU: 0.0000
Best Top-5 IoU: 1.0000

검색 4: 쌀 씻기
Top-1 IoU: 0.8133
Best Top-5 IoU: 0.8133

검색 5: 다이소 파우치
Top-1 IoU: 0.6646
Best Top-5 IoU: 0.6992

검색 6: 다이소 시트 세제
Top-1 IoU: 0.3038
Best Top-5 IoU: 0.7039

===== 기존 6개 검색어 Consensus 성능 =====
평균 Top-1 IoU: 0.297
평균 Best Top-5 IoU: 0.6743
R1@0.5: 2 / 6
R5@0.5: 5 / 6

✅ 기존 6개 검색어 Consensus 평가 완료!


In [238]:
import json
from pathlib import Path

save_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/"
    "consensus_6queries_comparison.json"
)

n = len(consensus_6results)

summary = {
    "mean_top1_iou": sum(
        r["top1_iou"] for r in consensus_6results
    ) / n,
    "mean_best5_iou": sum(
        r["best5_iou"] for r in consensus_6results
    ) / n,
    "r1_at_05": sum(
        r["top1_iou"] >= 0.5 for r in consensus_6results
    ),
    "r5_at_05": sum(
        r["best5_iou"] >= 0.5 for r in consensus_6results
    ),
    "total_queries": n
}

output = {
    "method": "Consensus Fusion (Fine-tuned MiniLM)",
    "summary": summary,
    "results": consensus_6results
}

save_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("===== Consensus 평가 결과 저장 =====")
print("저장 경로:", save_path)
print("저장된 검색어:", n)
print("성능 요약:", summary)
print("\n✅ 기존 6개 Consensus 평가 결과 백업 완료!")

===== Consensus 평가 결과 저장 =====
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/consensus_6queries_comparison.json
저장된 검색어: 6
성능 요약: {'mean_top1_iou': 0.29695030467407163, 'mean_best5_iou': 0.6742722657970684, 'r1_at_05': 2, 'r5_at_05': 5, 'total_queries': 6}

✅ 기존 6개 Consensus 평가 결과 백업 완료!


In [239]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/consensus_6queries_comparison.json"
)

data = json.loads(path.read_text(encoding="utf-8"))

print("===== Consensus Top-1 실패 사례 분석 =====")

for result in data["results"]:

    if result["top1_iou"] >= 0.5:
        continue

    print(f"\n{'='*55}")
    print("검색어:", result["name"])
    print("Top-1 IoU:", round(result["top1_iou"], 4))
    print("Best Top-5 IoU:", round(result["best5_iou"], 4))

    for rank, pred in enumerate(result["predictions"], 1):
        print(
            f"{rank}위 | "
            f"{pred['start_time']:.2f}~{pred['end_time']:.2f}초 "
            f"| 출처={pred.get('rank_source', 'unknown')} "
            f"| Consensus={pred.get('consensus_score', 0):.4f}"
        )

print("\n✅ 실패 사례 분석 완료!")

===== Consensus Top-1 실패 사례 분석 =====

검색어: 두부 자르기
Top-1 IoU: 0.0
Best Top-5 IoU: 0.3127
1위 | 98.50~118.96초 | 출처=hybrid | Consensus=1.0000
2위 | 4.74~8.70초 | 출처=ko | Consensus=1.0000
3위 | 12.88~16.32초 | 출처=en | Consensus=1.0000
4위 | 21.06~26.46초 | 출처=ko | Consensus=0.8259
5위 | 21.06~23.72초 | 출처=en | Consensus=0.7426

검색어: 두부 양념 준비
Top-1 IoU: 0.0
Best Top-5 IoU: 0.5164
1위 | 137.58~143.58초 | 출처=ko | Consensus=1.3568
2위 | 98.50~119.16초 | 출처=hybrid | Consensus=1.3486
3위 | 111.26~116.02초 | 출처=en | Consensus=1.2305
4위 | 134.50~151.32초 | 출처=hybrid | Consensus=1.0067
5위 | 160.62~164.98초 | 출처=en | Consensus=0.8475

검색어: 가위로 소스 짜기
Top-1 IoU: 0.0
Best Top-5 IoU: 1.0
1위 | 23.32~26.56초 | 출처=en | Consensus=1.6108
2위 | 2.50~10.27초 | 출처=hybrid | Consensus=1.6077
3위 | 5.96~8.08초 | 출처=ko | Consensus=1.2730
4위 | 20.50~31.68초 | 출처=hybrid | Consensus=1.1262
5위 | 5.96~11.32초 | 출처=ko | Consensus=0.7383

검색어: 다이소 시트 세제
Top-1 IoU: 0.3038
Best Top-5 IoU: 0.7039
1위 | 14.50~24.52초 | 출처=hybrid | Consensus=1.7428
2위 

In [240]:
from ai.features import fusion_search
from ai.features.run_moment_detr_inference import predict_moments_hybrid
from pathlib import Path

# 가위로 소스 짜기 (기존 test_cases의 세 번째 검색어)
case = test_cases[2]
video_id = "kr_kitchen_001"

feature_path = str(
    Path("/content/pinpoint-capstone/data/features")
    / f"{video_id}_clip_features.pt"
)

hybrid = predict_moments_hybrid(
    video_id=video_id,
    video_feature_path=feature_path,
    query=case["query_en"],
    subtitle_json_path=None,
    video_duration=duration_map[video_id],
    top_k=5
)

ko = fusion_search.get_semantic_subtitle_candidates(
    query=case["query_ko"],
    subtitles=case["subtitles"],
    model=trained_model,
    top_k=5
)

en = fusion_search.get_semantic_subtitle_candidates(
    query=case["query_en"],
    subtitles=case["subtitles"],
    model=trained_model,
    top_k=5
)

ranked = rank_by_consensus(hybrid, ko, en)

print("===== 가위로 소스 짜기: Consensus 점수 분석 =====")

for p in ranked[:10]:
    source = p["rank_source"]

    overlaps = {}
    for name, group in [
        ("hybrid", hybrid),
        ("ko", ko),
        ("en", en)
    ]:
        if name == source:
            continue

        overlaps[name] = max(
            (
                fusion_search.temporal_iou(p, other)
                for other in group
            ),
            default=0.0
        )

    print(
        f"{p['start_time']:.2f}~{p['end_time']:.2f}초"
        f" | 출처={source}"
        f" | 원래 순위={p['original_rank']}"
        f" | 순위 점수={p['rank_score']:.3f}"
        f" | 타 모델 겹침={overlaps}"
        f" | 최종 점수={p['consensus_score']:.4f}"
    )

print("\n✅ Consensus 점수 상세 분석 완료!")

===== 가위로 소스 짜기: Consensus 점수 분석 =====
23.32~26.56초 | 출처=en | 원래 순위=1 | 순위 점수=1.000 | 타 모델 겹침={'hybrid': 0.2898317365751549, 'ko': 0.3209876543209875} | 최종 점수=1.6108
2.50~10.27초 | 출처=hybrid | 원래 순위=1 | 순위 점수=1.000 | 타 모델 겹침={'ko': 0.607738363484195, 'en': 0.0} | 최종 점수=1.6077
5.96~8.08초 | 출처=ko | 원래 순위=1 | 순위 점수=1.000 | 타 모델 겹침={'hybrid': 0.2729672310564604, 'en': 0.0} | 최종 점수=1.2730
20.50~31.68초 | 출처=hybrid | 원래 순위=2 | 순위 점수=0.500 | 타 모델 겹침={'ko': 0.09303240927103734, 'en': 0.5331472685147914} | 최종 점수=1.1262
3.36~8.08초 | 출처=ko | 원래 순위=2 | 순위 점수=0.500 | 타 모델 겹침={'hybrid': 0.607738363484195, 'en': 0.0} | 최종 점수=1.1077
23.32~29.28초 | 출처=en | 원래 순위=5 | 순위 점수=0.200 | 타 모델 겹침={'hybrid': 0.5331472685147914, 'ko': 0.17449664429530184} | 최종 점수=0.9076
5.96~11.32초 | 출처=ko | 원래 순위=4 | 순위 점수=0.250 | 타 모델 겹침={'hybrid': 0.48826530612244906, 'en': 0.0} | 최종 점수=0.7383
24.36~26.56초 | 출처=en | 원래 순위=2 | 순위 점수=0.500 | 타 모델 겹침={'hybrid': 0.19679932730411753, 'ko': 0.0} | 최종 점수=0.6968
24.36~29.28초 | 출처=en | 원

In [241]:
from ai.features import fusion_search

def rank_by_consensus_v2(hybrid, ko, en):
    """
    Consensus V2 실험:
    다른 검색 방식의 후보와 IoU 0.5 이상일 때만
    겹침 점수를 추가한다.

    정답 구간은 순위 결정에 사용하지 않는다.
    """

    groups = {
        "hybrid": hybrid,
        "ko": ko,
        "en": en
    }

    candidates = []

    for source, group in groups.items():
        for rank, candidate in enumerate(group, 1):
            item = dict(candidate)

            item["rank_source"] = source
            item["original_rank"] = rank
            item["rank_score"] = 1.0 / rank

            support = 0.0

            for other_source, other_group in groups.items():
                if other_source == source:
                    continue

                best_overlap = max(
                    (
                        fusion_search.temporal_iou(item, other)
                        for other in other_group
                    ),
                    default=0.0
                )

                # V2: 충분히 겹치는 경우에만 가점
                if best_overlap >= 0.5:
                    support += best_overlap

            item["consensus_score"] = (
                item["rank_score"] + support
            )

            candidates.append(item)

    return sorted(
        candidates,
        key=lambda x: x["consensus_score"],
        reverse=True
    )


print("✅ Consensus V2 함수 생성 완료!")

# 방금 115단계에서 생성한 후보를 재사용
ranked_v2 = rank_by_consensus_v2(hybrid, ko, en)

print("\n===== 가위로 소스 짜기: V2 순위 =====")

for rank, pred in enumerate(ranked_v2[:10], 1):
    print(
        f"{rank}위 | "
        f"{pred['start_time']:.2f}~{pred['end_time']:.2f}초 "
        f"| 출처={pred['rank_source']} "
        f"| 점수={pred['consensus_score']:.4f}"
    )

print("\n✅ Consensus V2 첫 번째 실험 완료!")

✅ Consensus V2 함수 생성 완료!

===== 가위로 소스 짜기: V2 순위 =====
1위 | 2.50~10.27초 | 출처=hybrid | 점수=1.6077
2위 | 3.36~8.08초 | 출처=ko | 점수=1.1077
3위 | 20.50~31.68초 | 출처=hybrid | 점수=1.0331
4위 | 5.96~8.08초 | 출처=ko | 점수=1.0000
5위 | 23.32~26.56초 | 출처=en | 점수=1.0000
6위 | 23.32~29.28초 | 출처=en | 점수=0.7331
7위 | 24.36~26.56초 | 출처=en | 점수=0.5000
8위 | 10.50~20.50초 | 출처=hybrid | 점수=0.3333
9위 | 33.52~34.80초 | 출처=ko | 점수=0.3333
10위 | 19.76~21.52초 | 출처=en | 점수=0.3333

✅ Consensus V2 첫 번째 실험 완료!


# Consensus V2 전체 6개 검색어 평가

In [242]:
from pathlib import Path
from ai.features import fusion_search
from ai.features.run_moment_detr_inference import predict_moments_hybrid

repo = Path("/content/pinpoint-capstone")

video_ids = [
    "kr_tofu_001",
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_kitchen_001",
    "kr_daiso_001",
    "kr_daiso_001"
]

consensus_v2_6results = []

for i, case in enumerate(test_cases):
    video_id = video_ids[i]

    feature_path = str(
        repo / "data/features" / f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        str(repo / "data/processed/kr_daiso_001/subtitles.json")
        if video_id == "kr_daiso_001"
        else None
    )

    # 기존 Hybrid
    hybrid_candidates = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=duration_map[video_id],
        top_k=5
    )

    # Fine-tuned MiniLM
    ko_candidates = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_ko"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    en_candidates = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_en"],
        subtitles=case["subtitles"],
        model=trained_model,
        top_k=5
    )

    # V2 재정렬
    ranked = rank_by_consensus_v2(
        hybrid_candidates,
        ko_candidates,
        en_candidates
    )

    # 기존 평가와 동일한 중복 후보 제거
    predictions = []

    for candidate in ranked:
        if all(
            fusion_search.temporal_iou(candidate, selected) < 0.5
            for selected in predictions
        ):
            predictions.append(candidate)

        if len(predictions) == 5:
            break

    gt = case["ground_truth"]
    gt_start = gt["start_time"]
    gt_end = gt["end_time"]

    def calculate_iou(pred):
        intersection = max(
            0.0,
            min(pred["end_time"], gt_end)
            - max(pred["start_time"], gt_start)
        )
        union = (
            max(pred["end_time"], gt_end)
            - min(pred["start_time"], gt_start)
        )
        return intersection / union if union > 0 else 0.0

    top1 = calculate_iou(predictions[0]) if predictions else 0.0
    best5 = max(
        [calculate_iou(p) for p in predictions],
        default=0.0
    )

    consensus_v2_6results.append({
        "name": case["name"],
        "video_id": video_id,
        "top1_iou": top1,
        "best5_iou": best5,
        "predictions": predictions
    })

    print(f"\n검색 {i+1}: {case['name']}")
    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

n = len(consensus_v2_6results)

print("\n===== Consensus V2 6개 검색어 성능 =====")
print(
    "평균 Top-1 IoU:",
    round(sum(r["top1_iou"] for r in consensus_v2_6results) / n, 4)
)
print(
    "평균 Best Top-5 IoU:",
    round(sum(r["best5_iou"] for r in consensus_v2_6results) / n, 4)
)
print(
    "R1@0.5:",
    sum(r["top1_iou"] >= 0.5 for r in consensus_v2_6results),
    "/", n
)
print(
    "R5@0.5:",
    sum(r["best5_iou"] >= 0.5 for r in consensus_v2_6results),
    "/", n
)

print("\n✅ Consensus V2 6개 검색어 평가 완료!")


검색 1: 두부 자르기
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.3127

검색 2: 두부 양념 준비
Top-1 IoU: 0.0089
Best Top-5 IoU: 0.5164

검색 3: 가위로 소스 짜기
Top-1 IoU: 0.4883
Best Top-5 IoU: 0.4883

검색 4: 쌀 씻기
Top-1 IoU: 0.8133
Best Top-5 IoU: 0.8133

검색 5: 다이소 파우치
Top-1 IoU: 0.6646
Best Top-5 IoU: 0.6992

검색 6: 다이소 시트 세제
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.6720

===== Consensus V2 6개 검색어 성능 =====
평균 Top-1 IoU: 0.3292
평균 Best Top-5 IoU: 0.5837
R1@0.5: 2 / 6
R5@0.5: 4 / 6

✅ Consensus V2 6개 검색어 평가 완료!


In [243]:
from ai.features import fusion_search

squat_v2_results = []

for i, case in enumerate(squat_test_cases):

    hybrid_candidates = squat_hybrid_results[i]

    ko_candidates = fusion_search.get_semantic_subtitle_candidates(
        query=case["korean_query"],
        subtitles=squat_subtitles,
        model=trained_model,
        top_k=5
    )

    en_candidates = fusion_search.get_semantic_subtitle_candidates(
        query=case["english_query"],
        subtitles=squat_subtitles,
        model=trained_model,
        top_k=5
    )

    ranked = rank_by_consensus_v2(
        hybrid_candidates,
        ko_candidates,
        en_candidates
    )

    predictions = []

    for candidate in ranked:
        if all(
            fusion_search.temporal_iou(candidate, selected) < 0.5
            for selected in predictions
        ):
            predictions.append(candidate)

        if len(predictions) == 5:
            break

    squat_v2_results.append(predictions)

    top1 = temporal_iou(predictions[0], case) if predictions else 0.0
    best5 = max(
        [temporal_iou(p, case) for p in predictions],
        default=0.0
    )

    print(f"\n검색 {i+1}: {case['korean_query']}")
    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

print("\n===== 스쿼트 Consensus V2 성능 =====")

top1_scores = [
    temporal_iou(preds[0], case) if preds else 0.0
    for preds, case in zip(squat_v2_results, squat_test_cases)
]

best5_scores = [
    max([temporal_iou(p, case) for p in preds], default=0.0)
    for preds, case in zip(squat_v2_results, squat_test_cases)
]

print("평균 Top-1 IoU:", round(sum(top1_scores) / len(top1_scores), 4))
print("평균 Best Top-5 IoU:", round(sum(best5_scores) / len(best5_scores), 4))
print("R1@0.5:", sum(x >= 0.5 for x in top1_scores), "/3")
print("R5@0.5:", sum(x >= 0.5 for x in best5_scores), "/3")

print("\n✅ 스쿼트 영상 Consensus V2 평가 완료!")


검색 1: 발을 어깨너비로 벌리는 장면
Top-1 IoU: 0.6051
Best Top-5 IoU: 0.6051

검색 2: 엉덩이를 낮춰 스쿼트하는 장면
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.2720

검색 3: 뒤꿈치가 바닥에서 떨어지지 않게 설명하는 장면
Top-1 IoU: 0.2478
Best Top-5 IoU: 0.9780

===== 스쿼트 Consensus V2 성능 =====
평균 Top-1 IoU: 0.2843
평균 Best Top-5 IoU: 0.6184
R1@0.5: 1 /3
R5@0.5: 2 /3

✅ 스쿼트 영상 Consensus V2 평가 완료!


In [244]:
print("===== 추가 3개 검색어 데이터 확인 =====")

if "new_test_cases" in globals():
    print("✅ new_test_cases 존재")
    print("검색어 개수:", len(new_test_cases))

    for i, case in enumerate(new_test_cases, 1):
        print(f"\n검색 {i}")
        print("이름:", case.get("name", "없음"))
        print("한국어 검색어:", case.get("query_ko", "없음"))
        print("영어 검색어:", case.get("query_en", "없음"))
        print("정답:", case.get("ground_truth", "없음"))
else:
    print("❌ new_test_cases 변수가 없습니다.")

print("\n===== 필요한 변수 확인 =====")

for name in [
    "trained_model",
    "rank_by_consensus_v2",
    "subtitle_map",
    "duration_map"
]:
    print(
        f"{name}: "
        f"{'✅ 있음' if name in globals() else '❌ 없음'}"
    )

print("\n✅ 평가 준비 상태 확인 완료!")

===== 추가 3개 검색어 데이터 확인 =====
✅ new_test_cases 존재
검색어 개수: 3

검색 1
이름: 두부 데치기
한국어 검색어: 두부를 끓는 물에 데치는 장면
영어 검색어: blanching tofu in hot water
정답: {'start_time': 38.0, 'end_time': 61.0}

검색 2
이름: 주방 체망 소개
한국어 검색어: 냄비에 사용하는 체망을 소개하는 장면
영어 검색어: introducing a strainer for a pot
정답: {'start_time': 16.0, 'end_time': 28.0}

검색 3
이름: 다이소 헤어브러시
한국어 검색어: 여행용 헤어브러시를 소개하는 장면
영어 검색어: showing a travel hairbrush
정답: {'start_time': 22.0, 'end_time': 26.0}

===== 필요한 변수 확인 =====
trained_model: ✅ 있음
rank_by_consensus_v2: ✅ 있음
subtitle_map: ✅ 있음
duration_map: ✅ 있음

✅ 평가 준비 상태 확인 완료!


In [246]:
from pathlib import Path
from ai.features import fusion_search
from ai.features.run_moment_detr_inference import predict_moments_hybrid

repo = Path("/content/pinpoint-capstone")

video_ids = [
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_daiso_001"
]

consensus_v2_new3_results = []

for i, case in enumerate(new_test_cases):
    video_id = video_ids[i]

    feature_path = str(
        repo / "data/features" / f"{video_id}_clip_features.pt"
    )

    subtitle_path = (
        str(repo / "data/processed/kr_daiso_001/subtitles.json")
        if video_id == "kr_daiso_001"
        else None
    )

    # 1. 기존 Hybrid 후보
    hybrid = predict_moments_hybrid(
        video_id=video_id,
        video_feature_path=feature_path,
        query=case["query_en"],
        subtitle_json_path=subtitle_path,
        video_duration=duration_map[video_id],
        top_k=5
    )

    # 2. Fine-tuned MiniLM 후보
    ko = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_ko"],
        subtitles=subtitle_map[video_id],
        model=trained_model,
        top_k=5
    )

    en = fusion_search.get_semantic_subtitle_candidates(
        query=case["query_en"],
        subtitles=subtitle_map[video_id],
        model=trained_model,
        top_k=5
    )

    # 3. Consensus V2 재정렬
    ranked = rank_by_consensus_v2(hybrid, ko, en)

    # 4. 중복 구간 제거 후 상위 5개 선택
    predictions = []

    for candidate in ranked:
        if all(
            fusion_search.temporal_iou(candidate, selected) < 0.5
            for selected in predictions
        ):
            predictions.append(candidate)

        if len(predictions) == 5:
            break

    # 5. 정답 구간과 비교
    gt = case["ground_truth"]
    gt_start = gt["start_time"]
    gt_end = gt["end_time"]

    def calculate_iou(pred):
        intersection = max(
            0.0,
            min(pred["end_time"], gt_end)
            - max(pred["start_time"], gt_start)
        )

        union = (
            max(pred["end_time"], gt_end)
            - min(pred["start_time"], gt_start)
        )

        return intersection / union if union > 0 else 0.0

    top1 = calculate_iou(predictions[0]) if predictions else 0.0
    best5 = max(
        [calculate_iou(p) for p in predictions],
        default=0.0
    )

    consensus_v2_new3_results.append({
        "name": case["name"],
        "video_id": video_id,
        "top1_iou": top1,
        "best5_iou": best5,
        "predictions": predictions
    })

    print(f"\n검색 {i+1}: {case['name']}")
    print(f"Top-1 IoU: {top1:.4f}")
    print(f"Best Top-5 IoU: {best5:.4f}")

n = len(consensus_v2_new3_results)

print("\n===== 추가 3개 검색어 Consensus V2 성능 =====")

print(
    "평균 Top-1 IoU:",
    round(sum(r["top1_iou"] for r in consensus_v2_new3_results) / n, 4)
)

print(
    "평균 Best Top-5 IoU:",
    round(sum(r["best5_iou"] for r in consensus_v2_new3_results) / n, 4)
)

print(
    "R1@0.5:",
    sum(r["top1_iou"] >= 0.5 for r in consensus_v2_new3_results),
    "/", n
)

print(
    "R5@0.5:",
    sum(r["best5_iou"] >= 0.5 for r in consensus_v2_new3_results),
    "/", n
)

print("\n✅ 추가 3개 검색어 Consensus V2 평가 완료!")


검색 1: 두부 데치기
Top-1 IoU: 0.1191
Best Top-5 IoU: 0.1191

검색 2: 주방 체망 소개
Top-1 IoU: 0.1319
Best Top-5 IoU: 0.2609

검색 3: 다이소 헤어브러시
Top-1 IoU: 0.0000
Best Top-5 IoU: 0.3167

===== 추가 3개 검색어 Consensus V2 성능 =====
평균 Top-1 IoU: 0.0837
평균 Best Top-5 IoU: 0.2322
R1@0.5: 0 / 3
R5@0.5: 0 / 3

✅ 추가 3개 검색어 Consensus V2 평가 완료!


In [247]:
import json
from pathlib import Path

save_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

output = {
    "method": "Consensus V2 (minimum overlap IoU 0.5)",
    "experiment_note": "Exploratory evaluation on 4 previously examined videos",
    "results": {
        "original_6_queries": consensus_v2_6results,
        "additional_3_queries": consensus_v2_new3_results,
        "squat_3_queries": []
    }
}

# 스쿼트 결과와 정답 구간 함께 저장
for case, predictions in zip(squat_test_cases, squat_v2_results):
    ious = [temporal_iou(p, case) for p in predictions]

    output["results"]["squat_3_queries"].append({
        "name": case["korean_query"],
        "video_id": case["video_id"],
        "ground_truth": {
            "start_time": case["gt_start"],
            "end_time": case["gt_end"]
        },
        "top1_iou": ious[0] if ious else 0.0,
        "best5_iou": max(ious, default=0.0),
        "predictions": predictions
    })

save_path = save_dir / "consensus_v2_12queries_comparison.json"

save_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("===== Consensus V2 평가 결과 백업 =====")
print("저장 경로:", save_path)
print(
    "저장된 검색어:",
    sum(len(v) for v in output["results"].values())
)
print("✅ V2 평가 결과 백업 완료!")

===== Consensus V2 평가 결과 백업 =====
저장 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/consensus_v2_12queries_comparison.json
저장된 검색어: 12
✅ V2 평가 결과 백업 완료!


# Google Drive 파일과 현재 AI 코드 전체 확인하기

In [248]:
from pathlib import Path
import json
import subprocess

drive_root = Path("/content/drive/MyDrive/pinpoint_capstone")
repo = Path("/content/pinpoint-capstone")

folders = [
    drive_root / "02_Korean_Video_Test",
    drive_root / "03_Korean_Training_Data"
]

print("=" * 65)
print("1. GOOGLE DRIVE 파일 전체 목록")
print("=" * 65)

for folder in folders:
    print(f"\n📁 {folder.name}")
    print("폴더 존재:", folder.exists())

    if not folder.exists():
        continue

    files = sorted(
        [p for p in folder.rglob("*") if p.is_file()]
    )

    print("전체 파일 개수:", len(files))

    for p in files:
        relative = p.relative_to(folder)
        size_mb = p.stat().st_size / (1024 ** 2)

        print(f"\n  📄 {relative}")
        print(f"     크기: {size_mb:.2f} MB")

        # JSON은 내용 전체 대신 구조만 확인
        if p.suffix.lower() == ".json" and size_mb < 30:
            try:
                data = json.loads(
                    p.read_text(encoding="utf-8")
                )

                print("     JSON 타입:", type(data).__name__)

                if isinstance(data, dict):
                    print("     최상위 키:", list(data.keys())[:20])

                    if isinstance(data.get("summary"), dict):
                        print(
                            "     성능 요약 항목:",
                            list(data["summary"].keys())[:15]
                        )

                elif isinstance(data, list):
                    print("     항목 개수:", len(data))

            except Exception as e:
                print("     JSON 확인 실패:", str(e)[:100])

print("\n" + "=" * 65)
print("2. 현재 GitHub 프로젝트 AI 코드")
print("=" * 65)

ai_dir = repo / "ai"

if ai_dir.exists():
    for p in sorted(ai_dir.rglob("*.py")):
        if "vendor" in p.parts or ".venv" in p.parts:
            continue

        print(p.relative_to(repo))

print("\n" + "=" * 65)
print("3. Git 추적 상태")
print("=" * 65)

if (repo / ".git").exists():
    result = subprocess.run(
        ["git", "status", "--short"],
        cwd=repo,
        capture_output=True,
        text=True
    )
    print(result.stdout or "변경 사항 없음")
else:
    print("Git 저장소를 찾을 수 없습니다.")

print("\n✅ 전체 파일 및 코드 목록 확인 완료!")

1. GOOGLE DRIVE 파일 전체 목록

📁 02_Korean_Video_Test
폴더 존재: True
전체 파일 개수: 25

  📄 AI_Backup/fusion_search_20261010.py
     크기: 0.01 MB

  📄 AI_Backup/scratch_clip_align_best_ORIGINAL.ckpt
     크기: 48.88 MB

  📄 PinPoint_Korean_Video_Test.ipynb
     크기: 1.32 MB

  📄 __pycache__/fusion_experiment.cpython-313.pyc
     크기: 0.01 MB

  📄 clip_features/kr_daiso_001_clip_features.pt
     크기: 0.03 MB

  📄 clip_features/kr_kitchen_001_clip_features.pt
     크기: 0.04 MB

  📄 clip_features/kr_tofu_001_clip_features.pt
     크기: 0.16 MB

  📄 detergent_evaluation.json
     크기: 0.00 MB
     JSON 타입: dict
     최상위 키: ['ground_truth', 'hybrid_results', 'fusion_results']

  📄 fusion_baseline_evaluation.json
     크기: 0.00 MB
     JSON 타입: dict
     최상위 키: ['description', 'evaluation_note', 'deduplication', 'ranking', 'ground_truth', 'results']

  📄 fusion_experiment.py
     크기: 0.00 MB

  📄 indexer_before_subtitle_integration.py
     크기: 0.01 MB

  📄 kr_daiso_001_subtitles.json
     크기: 0.00 MB
     JSON 타입: 

In [249]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")
drive = Path("/content/drive/MyDrive/pinpoint_capstone")

files_to_check = {
    "현재 Fusion 코드":
        repo / "ai/features/fusion_search.py",

    "Fusion Drive 백업":
        drive / "02_Korean_Video_Test/AI_Backup/fusion_search_20261010.py",

    "Fusion 실험 코드":
        drive / "02_Korean_Video_Test/fusion_experiment.py",

    "Whisper 서비스 코드":
        repo / "ai/preprocessing/transcribe_service.py",

    "기존 Whisper 코드":
        repo / "ai/preprocessing/transcribe.py",

    "기존 Moment-DETR 추론 코드":
        repo / "ai/features/run_moment_detr_inference.py",
}

print("===== 업로드 후보 코드 구조 확인 =====")

for label, path in files_to_check.items():
    print(f"\n{'='*55}")
    print(label)
    print("경로:", path)
    print("파일 존재:", path.exists())

    if not path.exists():
        continue

    code = path.read_text(encoding="utf-8")
    print("코드 줄 수:", len(code.splitlines()))
    print("파일 크기:", len(code.encode("utf-8")), "bytes")

    try:
        tree = ast.parse(code)

        functions = [
            node.name
            for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        ]

        classes = [
            node.name
            for node in tree.body
            if isinstance(node, ast.ClassDef)
        ]

        print("함수:", functions)
        print("클래스:", classes)

    except SyntaxError as e:
        print("문법 확인 오류:", e)

print("\n===== Consensus 코드 저장 여부 =====")

for name, path in files_to_check.items():
    if not path.exists():
        continue

    content = path.read_text(encoding="utf-8")

    print(
        f"{name}: "
        f"V1={'rank_by_consensus(' in content}, "
        f"V2={'rank_by_consensus_v2(' in content}, "
        f"언어선택={'select_subtitle_query(' in content}"
    )

print("\n===== backend/services 파일 목록 =====")

backend_services = repo / "backend/services"

if backend_services.exists():
    for p in sorted(backend_services.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts:
            print(p.relative_to(repo))

print("\n✅ GitHub 업로드 후보 코드 확인 완료!")

===== 업로드 후보 코드 구조 확인 =====

현재 Fusion 코드
경로: /content/pinpoint-capstone/ai/features/fusion_search.py
파일 존재: True
코드 줄 수: 242
파일 크기: 5965 bytes
함수: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_multilingual_subtitle_candidates', 'deduplicate_multilingual_candidates', 'make_fusion_result', 'predict_moments_fusion']
클래스: []

Fusion Drive 백업
경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/fusion_search_20261010.py
파일 존재: True
코드 줄 수: 242
파일 크기: 5965 bytes
함수: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_multilingual_subtitle_candidates', 'deduplicate_multilingual_candidates', 'make_fusion_result', 'predict_moments_fusion']
클래스: []

Fusion 실험 코드
경로: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/fusion_experiment.py
파일 존재: True
코드 줄 수: 212
파일 크기: 5042 bytes
함수: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidat

In [250]:
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True
    )
    print(f"\n$ git {' '.join(args)}")
    print(result.stdout.strip() or "(출력 없음)")
    if result.returncode != 0:
        print("오류:", result.stderr.strip())

print("===== 1. 프로젝트 확인 =====")
print("프로젝트 존재:", repo.is_dir())
print("Git 저장소 존재:", (repo / ".git").exists())

if repo.is_dir() and (repo / ".git").exists():
    print("\n===== 2. 현재 브랜치 =====")
    git("branch", "--show-current")

    print("\n===== 3. GitHub 연결 주소 =====")
    git("remote", "-v")

    print("\n===== 4. 변경·추가된 파일 =====")
    git("status", "--short")

    print("\n===== 5. 현재 커밋 =====")
    git("log", "-1", "--format=%h %s")

print("\n✅ Git 상태 확인 완료!")

===== 1. 프로젝트 확인 =====
프로젝트 존재: True
Git 저장소 존재: True

===== 2. 현재 브랜치 =====

$ git branch --show-current
main

===== 3. GitHub 연결 주소 =====

$ git remote -v
origin	https://github.com/pinpoint-capstone/pinpoint-capstone.git (fetch)
origin	https://github.com/pinpoint-capstone/pinpoint-capstone.git (push)

===== 4. 변경·추가된 파일 =====

$ git status --short
?? ai/features/fusion_search.py
?? ai/features/fusion_search.py.bak
?? ai/features/run_moment_detr_inference_before_fusion.py
?? ai/preprocessing/transcribe_service.py
?? backend/services/

===== 5. 현재 커밋 =====

$ git log -1 --format=%h %s
14b3eb9 fix: include CLIP tokenizer vocabulary

✅ Git 상태 확인 완료!


In [251]:
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")
branch_name = "feature/korean-ai-fusion"

# 이미 같은 이름의 브랜치가 있는지 확인
check = subprocess.run(
    ["git", "branch", "--list", branch_name],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True
)

if check.stdout.strip():
    print("⚠️ 같은 이름의 브랜치가 이미 있어.")
    print("기존 브랜치는 변경하지 않았어.")
else:
    subprocess.run(
        ["git", "switch", "-c", branch_name],
        cwd=repo,
        check=True
    )

print("\n===== 현재 브랜치 =====")
subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=repo,
    check=True
)

print("\n===== 파일 변경 상태 =====")
subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True
)

print("\n✅ 브랜치 확인 완료!")


===== 현재 브랜치 =====

===== 파일 변경 상태 =====

✅ 브랜치 확인 완료!


In [252]:
import subprocess

repo = "/content/pinpoint-capstone"

print("===== 현재 브랜치 =====")
subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=repo,
    check=True
)

print("\n===== 전체 로컬 브랜치 =====")
subprocess.run(
    ["git", "branch", "--list"],
    cwd=repo,
    check=True
)

print("\n===== 변경된 파일 확인 =====")
subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True
)

===== 현재 브랜치 =====

===== 전체 로컬 브랜치 =====

===== 변경된 파일 확인 =====


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [253]:
import subprocess

repo = "/content/pinpoint-capstone"

def check_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True
    )

    print(f"\n$ git {' '.join(args)}")
    print("종료 코드:", result.returncode)
    print(result.stdout.strip() or "(출력 없음)")

    if result.stderr.strip():
        print("메시지:", result.stderr.strip())

print("===== 브랜치 및 파일 상태 확인 =====")

check_git("branch", "--show-current")
check_git("branch", "--list")
check_git("status", "--short")

===== 브랜치 및 파일 상태 확인 =====

$ git branch --show-current
종료 코드: 0
feature/korean-ai-fusion

$ git branch --list
종료 코드: 0
* feature/korean-ai-fusion
  main

$ git status --short
종료 코드: 0
?? ai/features/fusion_search.py
?? ai/features/fusion_search.py.bak
?? ai/features/run_moment_detr_inference_before_fusion.py
?? ai/preprocessing/transcribe_service.py
?? backend/services/


In [254]:
from pathlib import Path
import hashlib

repo_file = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

backup_file = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/fusion_search_20261010.py"
)

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("===== Fusion 코드 비교 =====")

print("현재 코드 존재:", repo_file.exists())
print("Drive 백업 존재:", backup_file.exists())

if repo_file.exists() and backup_file.exists():

    same = file_hash(repo_file) == file_hash(backup_file)

    print("두 파일이 완전히 동일한가?:", same)

    print(
        "현재 코드 줄 수:",
        len(repo_file.read_text(encoding="utf-8").splitlines())
    )

    print(
        "백업 코드 줄 수:",
        len(backup_file.read_text(encoding="utf-8").splitlines())
    )

    print("현재 코드 SHA256:", file_hash(repo_file))
    print("Drive 백업 SHA256:", file_hash(backup_file))

print("\n✅ Fusion 코드 비교 완료!")

===== Fusion 코드 비교 =====
현재 코드 존재: True
Drive 백업 존재: True
두 파일이 완전히 동일한가?: True
현재 코드 줄 수: 242
백업 코드 줄 수: 242
현재 코드 SHA256: 672cc1c808dab593a5e1cf77002d335d412a4e5a1350b94984fd805b8f5f27e6
Drive 백업 SHA256: 672cc1c808dab593a5e1cf77002d335d412a4e5a1350b94984fd805b8f5f27e6

✅ Fusion 코드 비교 완료!


In [255]:
from pathlib import Path
import ast

path = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

code = path.read_text(encoding="utf-8")
tree = ast.parse(code)

print("===== 5단계. Fusion 코드 의존성 확인 =====")

print("\n[1] 사용 라이브러리 및 모듈")

for node in tree.body:
    if isinstance(node, ast.Import):
        for item in node.names:
            print("import", item.name)

    elif isinstance(node, ast.ImportFrom):
        print(
            "from",
            node.module,
            "import",
            ", ".join(item.name for item in node.names)
        )

print("\n[2] 주요 함수 연결")

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        if node.name in [
            "get_semantic_model",
            "select_subtitle_query",
            "predict_moments_fusion"
        ]:
            print(f"\n--- {node.name} ---")
            print(ast.get_source_segment(code, node))

print("\n[3] Python 문법 검사")
compile(code, str(path), "exec")
print("✅ 문법 검사 통과")

print("\n✅ 5단계 확인 완료!")

===== 5단계. Fusion 코드 의존성 확인 =====

[1] 사용 라이브러리 및 모듈
from functools import lru_cache
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
from ai.features.run_moment_detr_inference import predict_moments_hybrid

[2] 주요 함수 연결

--- get_semantic_model ---
def get_semantic_model():
    return SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

--- select_subtitle_query ---
def select_subtitle_query(subtitles, korean_query, english_query):
    from langdetect import detect, DetectorFactory, LangDetectException

    DetectorFactory.seed = 0

    full_text = " ".join(
        str(item.get("text", ""))
        for item in subtitles
    ).strip()

    if not full_text:
        return "unknown", None

    try:
        language = detect(full_text)
    except LangDetectException:
        return "unknown", None

    if language == "ko":
        return "ko", korean_query

    if language == "en":
        return "en

In [256]:
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

files = [
    repo / "ai/preprocessing/transcribe_service.py",
    repo / "ai/preprocessing/transcribe.py"
]

print("===== 6단계. Whisper 코드 비교 =====")

for path in files:
    print("\n" + "=" * 60)
    print("파일:", path.relative_to(repo))
    print("=" * 60)

    if not path.exists():
        print("❌ 파일 없음")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines, 1):
        print(f"{i:3d}: {line}")

print("\n✅ Whisper 코드 비교 완료!")

===== 6단계. Whisper 코드 비교 =====

파일: ai/preprocessing/transcribe_service.py
  1: import json
  2: from pathlib import Path
  3: from functools import lru_cache
  4: 
  5: from faster_whisper import WhisperModel
  6: 
  7: 
  8: @lru_cache(maxsize=1)
  9: def get_whisper_model():
 10:     return WhisperModel(
 11:         "small",
 12:         device="cpu",
 13:         compute_type="int8"
 14:     )
 15: 
 16: 
 17: def transcribe_video(video_path, video_id):
 18:     output_path = (
 19:         Path("data/processed")
 20:         / video_id
 21:         / "subtitles.json"
 22:     )
 23:     output_path.parent.mkdir(
 24:         parents=True,
 25:         exist_ok=True
 26:     )
 27: 
 28:     segments, info = get_whisper_model().transcribe(
 29:         str(video_path),
 30:         beam_size=5
 31:     )
 32: 
 33:     subtitles = [
 34:         {
 35:             "start_time": round(s.start, 2),
 36:             "end_time": round(s.end, 2),
 37:             "text": s.text.strip()

# Fusion에 MiniLM 모델 선택 기능 추가하기

In [257]:
from pathlib import Path
from datetime import datetime
import shutil

repo = Path("/content/pinpoint-capstone")
path = repo / "ai/features/fusion_search.py"

backup_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup"
)
backup_dir.mkdir(parents=True, exist_ok=True)

backup_path = (
    backup_dir
    / f"fusion_search_before_model_selection_{datetime.now():%Y%m%d_%H%M%S}.py"
)

code = path.read_text(encoding="utf-8")

old_function = '''def get_semantic_model():
    return SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )'''

new_function = '''def get_semantic_model(model_type="base"):
    """
    base: 기존 다국어 MiniLM
    finetuned: AI-Hub 한국어 데이터로 추가 학습한 MiniLM
    """
    import os

    if model_type == "base":
        model_path = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    elif model_type == "finetuned":
        model_path = os.environ.get("PINPOINT_MINILM_MODEL_PATH")

        if not model_path:
            raise ValueError(
                "Fine-tuned MiniLM 경로가 설정되지 않았습니다. "
                "PINPOINT_MINILM_MODEL_PATH 환경변수를 확인하세요."
            )

        if not Path(model_path).is_dir():
            raise FileNotFoundError(
                f"Fine-tuned MiniLM 모델 폴더가 없습니다: {model_path}"
            )

    else:
        raise ValueError(
            "model_type은 'base' 또는 'finetuned'만 가능합니다."
        )

    return SentenceTransformer(model_path)'''

assert code.count(old_function) == 1, (
    "기존 get_semantic_model 함수가 예상과 다릅니다. "
    "파일을 수정하지 않았습니다."
)

# 기존 파일을 Drive에 백업
shutil.copy2(path, backup_path)

# 함수만 교체
updated = code.replace(old_function, new_function, 1)

# 저장 전에 문법 검사
compile(updated, str(path), "exec")
path.write_text(updated, encoding="utf-8")

print("✅ MiniLM 모델 선택 함수 추가 완료!")
print("✅ 백업 파일:", backup_path)
print("✅ 수정 파일:", path)

✅ MiniLM 모델 선택 함수 추가 완료!
✅ 백업 파일: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/fusion_search_before_model_selection_20261010_130203.py
✅ 수정 파일: /content/pinpoint-capstone/ai/features/fusion_search.py


In [258]:
from pathlib import Path
from datetime import datetime
import shutil

path = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

backup_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup"
)

code = path.read_text(encoding="utf-8")

# 1. 함수 입력값에 model_type 추가
old_signature = '''    video_duration, subtitle_json_path=None, top_k=5
):'''

new_signature = '''    video_duration, subtitle_json_path=None, top_k=5,
    model_type="base"
):'''

# 2. 선택한 MiniLM을 사용하도록 연결
old_call = '''            model=get_semantic_model(),
            top_k=top_k'''

new_call = '''            model=get_semantic_model(model_type=model_type),
            top_k=top_k'''

# 예상한 코드인지 먼저 검사
assert code.count(old_signature) == 1, (
    "함수 입력값 위치가 예상과 다릅니다. 수정하지 않았습니다."
)

assert code.count(old_call) == 1, (
    "MiniLM 호출 위치가 예상과 다릅니다. 수정하지 않았습니다."
)

# 백업
backup_dir.mkdir(parents=True, exist_ok=True)

backup_path = (
    backup_dir /
    f"fusion_search_before_model_connection_{datetime.now():%Y%m%d_%H%M%S}.py"
)

shutil.copy2(path, backup_path)

# 필요한 부분만 변경
updated = code.replace(old_signature, new_signature, 1)
updated = updated.replace(old_call, new_call, 1)

# Python 문법 검사
compile(updated, str(path), "exec")

# 저장
path.write_text(updated, encoding="utf-8")

print("✅ Fusion 검색 함수에 MiniLM 모델 선택 기능 연결 완료!")
print("✅ 기존 코드 백업:", backup_path)

# 실제 함수 입력값 확인
import ast

tree = ast.parse(updated)

for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        if node.name == "predict_moments_fusion":
            print(
                "✅ 함수 입력값:",
                [arg.arg for arg in node.args.args]
            )

✅ Fusion 검색 함수에 MiniLM 모델 선택 기능 연결 완료!
✅ 기존 코드 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/fusion_search_before_model_connection_20261010_130236.py
✅ 함수 입력값: ['video_id', 'feature_path', 'subtitles', 'korean_query', 'english_query', 'video_duration', 'subtitle_json_path', 'top_k', 'model_type']


In [260]:
from pathlib import Path
import ast

path = Path(
    "/content/pinpoint-capstone/ai/features/fusion_search.py"
)

code = path.read_text(encoding="utf-8")

# pathlib.Path import가 이미 있는지 확인
tree = ast.parse(code)

has_path_import = any(
    isinstance(node, ast.ImportFrom)
    and node.module == "pathlib"
    and any(alias.name == "Path" for alias in node.names)
    for node in tree.body
)

if has_path_import:
    print("✅ Path import가 이미 존재합니다. 수정하지 않았습니다.")

else:
    # 기존 코드의 첫 번째 import 앞에 추가
    updated = "from pathlib import Path\n" + code

    # 문법 검사 후 저장
    compile(updated, str(path), "exec")
    path.write_text(updated, encoding="utf-8")

    print("✅ from pathlib import Path 추가 완료!")

print("수정 파일:", path)
print("✅ 문법 검사 완료!")

✅ from pathlib import Path 추가 완료!
수정 파일: /content/pinpoint-capstone/ai/features/fusion_search.py
✅ 문법 검사 완료!


In [261]:
import os
import importlib
from pathlib import Path
from ai.features import fusion_search

# Google Drive에 저장된 파인튜닝 모델
model_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned/final_model"
)

assert model_path.is_dir(), "모델 폴더를 찾을 수 없습니다."

# 모델 경로 설정
os.environ["PINPOINT_MINILM_MODEL_PATH"] = str(model_path)

# 방금 수정한 Fusion 코드 다시 불러오기
importlib.reload(fusion_search)

print("===== Fine-tuned MiniLM 테스트 =====")

finetuned_model = fusion_search.get_semantic_model(
    model_type="finetuned"
)

embedding = finetuned_model.encode(
    "두부를 칼로 자르는 장면"
)

print("✅ Fine-tuned MiniLM 로드 성공!")
print("임베딩 차원:", embedding.shape)
print("모델 경로:", model_path)

print("\n✅ 9-2단계 테스트 완료!")

===== Fine-tuned MiniLM 테스트 =====


The tokenizer you are loading from '/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Fine-tuned MiniLM 로드 성공!
임베딩 차원: (384,)
모델 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model

✅ 9-2단계 테스트 완료!


In [262]:
import os
import importlib
import json
from pathlib import Path

from ai.features import fusion_search

repo = Path("/content/pinpoint-capstone")
drive = Path("/content/drive/MyDrive/pinpoint_capstone")

feature_path = repo / "data/features/kr_tofu_001_clip_features.pt"

# Drive에 보관된 Feature도 사용할 수 있도록 처리
if not feature_path.exists():
    feature_path = (
        drive / "02_Korean_Video_Test"
        / "clip_features/kr_tofu_001_clip_features.pt"
    )

subtitle_path = (
    drive / "02_Korean_Video_Test"
    / "kr_tofu_001_subtitles.json"
)

model_path = (
    drive / "03_Korean_Training_Data"
    / "minilm_finetuned/final_model"
)

assert feature_path.is_file(), "두부 CLIP Feature가 없습니다."
assert subtitle_path.is_file(), "두부 자막 파일이 없습니다."
assert model_path.is_dir(), "Fine-tuned MiniLM 모델이 없습니다."

os.environ["PINPOINT_MINILM_MODEL_PATH"] = str(model_path)

importlib.reload(fusion_search)

subtitles = json.loads(
    subtitle_path.read_text(encoding="utf-8")
)

print("===== 실제 Fusion 검색 테스트 =====")

for model_type in ["base", "finetuned"]:
    print(f"\n[{model_type} MiniLM]")

    predictions = fusion_search.predict_moments_fusion(
        video_id="kr_tofu_001",
        feature_path=str(feature_path),
        subtitles=subtitles,
        korean_query="두부를 칼로 자르는 장면",
        english_query="cutting tofu with a knife",
        video_duration=164.98,
        subtitle_json_path=None,
        top_k=5,
        model_type=model_type
    )

    print("검색 결과 개수:", len(predictions))

    for i, pred in enumerate(predictions, 1):
        print(
            f"{i}위: "
            f"{pred['start_time']:.2f}~"
            f"{pred['end_time']:.2f}초 "
            f"| 출처={pred.get('source')}"
        )

print("\n✅ 두 모델의 실제 Fusion 검색 테스트 완료!")

===== 실제 Fusion 검색 테스트 =====

[base MiniLM]
검색 결과 개수: 5
1위: 98.50~118.96초 | 출처=hybrid
2위: 140.86~143.58초 | 출처=subtitle
3위: 12.88~21.06초 | 출처=subtitle
4위: 84.50~102.78초 | 출처=hybrid
5위: 135.06~137.58초 | 출처=subtitle

[finetuned MiniLM]


The tokenizer you are loading from '/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_finetuned/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


검색 결과 개수: 5
1위: 98.50~118.96초 | 출처=hybrid
2위: 4.74~8.70초 | 출처=subtitle
3위: 12.88~16.32초 | 출처=subtitle
4위: 84.50~102.78초 | 출처=hybrid
5위: 23.72~26.46초 | 출처=subtitle

✅ 두 모델의 실제 Fusion 검색 테스트 완료!


In [263]:
from pathlib import Path

repo = Path("/content/pinpoint-capstone")
path = repo / "ai/features/consensus_rerank.py"

# 이미 파일이 있다면 덮어쓰지 않음
assert not path.exists(), (
    "consensus_rerank.py가 이미 존재합니다. "
    "기존 파일을 먼저 확인해야 합니다."
)

code = '''
"""
PinPoint Consensus Reranking

V1: 서로 다른 검색 방식의 시간 구간 겹침을 모두 가점
V2: IoU 0.5 이상인 겹침만 가점

실험용 재정렬 알고리즘이며,
정답 구간(Ground Truth)은 순위 결정에 사용하지 않는다.
"""

from ai.features.fusion_search import temporal_iou


def _rank_by_consensus(hybrid, ko, en, min_overlap=0.0):
    groups = {
        "hybrid": hybrid,
        "ko": ko,
        "en": en
    }

    candidates = []

    for source, group in groups.items():
        for rank, candidate in enumerate(group, 1):
            item = dict(candidate)

            item["rank_source"] = source
            item["original_rank"] = rank
            item["rank_score"] = 1.0 / rank

            candidates.append(item)

    for candidate in candidates:
        support = 0.0

        for source, group in groups.items():
            if source == candidate["rank_source"]:
                continue

            best_overlap = max(
                (
                    temporal_iou(candidate, other)
                    for other in group
                ),
                default=0.0
            )

            if best_overlap >= min_overlap:
                support += best_overlap

        candidate["consensus_score"] = (
            candidate["rank_score"] + support
        )

    return sorted(
        candidates,
        key=lambda x: x["consensus_score"],
        reverse=True
    )


def rank_by_consensus(hybrid, ko, en):
    """Consensus V1"""
    return _rank_by_consensus(
        hybrid, ko, en,
        min_overlap=0.0
    )


def rank_by_consensus_v2(hybrid, ko, en):
    """Consensus V2"""
    return _rank_by_consensus(
        hybrid, ko, en,
        min_overlap=0.5
    )
'''

compile(code, str(path), "exec")
path.write_text(code.lstrip(), encoding="utf-8")

print("✅ Consensus V1·V2 코드 파일 생성 완료!")
print("파일:", path)
print("파일 크기:", path.stat().st_size, "bytes")
print("기존 Fusion 코드는 수정하지 않았습니다.")

✅ Consensus V1·V2 코드 파일 생성 완료!
파일: /content/pinpoint-capstone/ai/features/consensus_rerank.py
파일 크기: 1745 bytes
기존 Fusion 코드는 수정하지 않았습니다.


In [264]:
import importlib

from ai.features import consensus_rerank
from ai.features import fusion_search

importlib.reload(consensus_rerank)

print("===== 12단계. Consensus 코드 일치 검증 =====")

# 기존에 실험했던 후보 데이터가 남아 있는지 확인
required = [
    "hybrid",
    "ko",
    "en",
    "rank_by_consensus",
    "rank_by_consensus_v2"
]

missing = [name for name in required if name not in globals()]

if missing:
    print("⚠️ 없는 변수:", missing)
    print("기존 실험 결과 또는 후보를 다시 불러와야 합니다.")

else:
    for name, old_func, new_func in [
        (
            "Consensus V1",
            rank_by_consensus,
            consensus_rerank.rank_by_consensus
        ),
        (
            "Consensus V2",
            rank_by_consensus_v2,
            consensus_rerank.rank_by_consensus_v2
        )
    ]:
        old_results = old_func(hybrid, ko, en)
        new_results = new_func(hybrid, ko, en)

        old_signature = [
            (
                r["start_time"],
                r["end_time"],
                r["rank_source"],
                round(r["consensus_score"], 8)
            )
            for r in old_results
        ]

        new_signature = [
            (
                r["start_time"],
                r["end_time"],
                r["rank_source"],
                round(r["consensus_score"], 8)
            )
            for r in new_results
        ]

        print(f"\n[{name}]")
        print("기존 후보 수:", len(old_signature))
        print("새 코드 후보 수:", len(new_signature))
        print("결과 완전히 일치:", old_signature == new_signature)

        assert old_signature == new_signature, (
            f"{name} 결과가 기존 실험과 다릅니다!"
        )

    print("\n✅ Consensus V1·V2 기존 실험과 일치!")

print("\n✅ 12단계 확인 완료!")

===== 12단계. Consensus 코드 일치 검증 =====

[Consensus V1]
기존 후보 수: 13
새 코드 후보 수: 13
결과 완전히 일치: True

[Consensus V2]
기존 후보 수: 13
새 코드 후보 수: 13
결과 완전히 일치: True

✅ Consensus V1·V2 기존 실험과 일치!

✅ 12단계 확인 완료!


In [265]:
from pathlib import Path
import shutil
import hashlib

source = Path(
    "/content/pinpoint-capstone/ai/features/consensus_rerank.py"
)

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/consensus_rerank.py"
)

assert source.is_file(), "Consensus 원본 코드가 없습니다."
assert not backup.exists(), (
    "Drive에 같은 이름의 백업이 이미 있습니다. "
    "덮어쓰지 않고 중단합니다."
)

backup.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, backup)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("===== Consensus 코드 백업 확인 =====")
print("Colab 원본:", source)
print("Drive 백업:", backup)
print("백업 파일 존재:", backup.is_file())
print("원본과 백업 동일:", sha256(source) == sha256(backup))

assert sha256(source) == sha256(backup)

print("\n✅ Consensus V1·V2 코드 백업 완료!")

===== Consensus 코드 백업 확인 =====
Colab 원본: /content/pinpoint-capstone/ai/features/consensus_rerank.py
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/consensus_rerank.py
백업 파일 존재: True
원본과 백업 동일: True

✅ Consensus V1·V2 코드 백업 완료!


In [266]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/PinPoint_Korean_Video_Test.ipynb"
)

print("===== 14단계. MiniLM 학습 코드 확인 =====")
print("노트북 존재:", notebook_path.is_file())

if notebook_path.is_file():

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    cells = notebook.get("cells", [])

    print("전체 셀 개수:", len(cells))

    keywords = [
        "SentenceTransformerTrainer",
        "MultipleNegativesRankingLoss",
        "TrainingArguments",
        "SentenceTransformerTrainingArguments",
        "minilm_train_pairs",
        "trainer.train",
        "model.save",
        "final_model"
    ]

    print("\n===== 학습 관련 코드 셀 =====")

    found = 0

    for i, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        matches = [
            word for word in keywords
            if word in source
        ]

        if matches:
            found += 1

            print(f"\n셀 번호: {i}")
            print("관련 키워드:", matches)

            # 학습 코드의 앞부분만 확인
            print("코드 미리보기:")
            print(source[:900])

    print("\n학습 관련 셀 개수:", found)

print("\n✅ MiniLM 학습 코드 확인 완료!")

===== 14단계. MiniLM 학습 코드 확인 =====
노트북 존재: True
전체 셀 개수: 438

===== 학습 관련 코드 셀 =====

셀 번호: 310
관련 키워드: ['minilm_train_pairs']
코드 미리보기:
import json
import io
import zipfile
from pathlib import Path

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

output_path = training_dir / "minilm_train_pairs.jsonl"

model = fusion_search.get_semantic_model()
tokenizer = model.tokenizer
limit = model.max_seq_length

count = 0

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z, \
     output_path.open("w", encoding="utf-8") as f:

    for name in z.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(z.read(name).decode("utf-8-sig"))

        video_id = data.get("metadata", {}).get("filename")
        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])
        transcript = " ".join(
            str(t.get("transcription") or "").strip()
            for t

In [267]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/PinPoint_Korean_Video_Test.ipynb"
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

cells = notebook["cells"]

# 최종 MiniLM 학습 과정이 포함된 구간
start_cell = 308
end_cell = 325

print("===== 15단계. MiniLM 학습 코드 전체 확인 =====")

for i in range(start_cell, end_cell + 1):
    cell = cells[i]

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 60}")
    print(f"셀 번호: {i}")
    print("=" * 60)
    print(source)

print("\n✅ MiniLM 학습 코드 확인 완료!")

===== 15단계. MiniLM 학습 코드 전체 확인 =====

셀 번호: 308
import json
import random
from pathlib import Path
from collections import defaultdict
from sentence_transformers import util

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

train_path = training_dir / "korean_subtitle_candidates.jsonl"

grouped = defaultdict(list)

with train_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        grouped[record["video_id"]].append(record)

# 재현 가능한 무작위 샘플
rng = random.Random(42)
sample_ids = rng.sample(sorted(grouped.keys()), 10)

model = fusion_search.get_semantic_model()

print("===== MiniLM 학습용 Positive 후보 10개 =====")

for video_id in sample_ids:
    candidates = grouped[video_id]
    summary = candidates[0]["summary"]
    texts = [record["text"] for record in candidates]

    embeddings = model.encode(
        [summary] + texts,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        embeddin

In [268]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/PinPoint_Korean_Video_Test.ipynb"
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

print("===== 16단계. 원본 데이터 연결 코드 확인 =====")

keywords = [
    "zip_bytes =",
    "val_zip_bytes =",
    "TL_12",
    "VL_12",
    "tarfile.open",
    "ZipFile("
]

for i, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matches = [
        keyword for keyword in keywords
        if keyword in source
    ]

    if matches:
        print(f"\n{'=' * 60}")
        print("셀 번호:", i)
        print("발견된 항목:", matches)
        print("코드:")
        print(source[:3500])

print("\n✅ 원본 데이터 연결 코드 확인 완료!")

===== 16단계. 원본 데이터 연결 코드 확인 =====

셀 번호: 283
발견된 항목: ['tarfile.open']
코드:
import tarfile
from pathlib import Path

# AI-Hub 한국어 학습 데이터 위치
training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data"
)

# 한글 파일명 인코딩 문제를 피하기 위해 자동 탐색
tar_files = list(training_dir.glob("*.tar"))

print("===== AI-Hub 데이터 확인 =====")
print("TAR 파일 개수:", len(tar_files))

for tar_path in tar_files:
    print("\n파일명:", tar_path.name)
    print("크기:", round(tar_path.stat().st_size / 1024**2, 2), "MB")

    with tarfile.open(tar_path, "r:*") as tar:
        members = [m for m in tar.getmembers() if m.isfile()]

        json_files = [
            m for m in members
            if m.name.lower().endswith(".json")
        ]

        print("전체 파일 개수:", len(members))
        print("JSON 파일 개수:", len(json_files))

        print("\n===== 처음 10개 파일 =====")
        for member in members[:10]:
            print(member.name)

셀 번호: 284
발견된 항목: ['tarfile.open']
코드:
import tarfile
from pathlib i

In [269]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

print("===== 17단계. GitHub 작업 환경 안전 점검 =====")

# 1. 현재 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

print("\n현재 브랜치:", branch)

assert branch == "feature/korean-ai-fusion", (
    "⚠️ 예상한 AI 전용 브랜치가 아닙니다. 작업을 중단합니다."
)

# 2. 기존 실험 폴더 확인
experiments_dir = repo / "ai/experiments"
target_dir = experiments_dir / "korean_minilm"

print("\n===== 기존 폴더 구조 =====")

for path in [experiments_dir, target_dir]:
    print(f"\n경로: {path}")
    print("존재 여부:", path.exists())

    if path.is_dir():
        print("하위 항목:")
        for item in sorted(path.iterdir()):
            print(" -", item.name)

# 3. 생성 예정 파일과 충돌 여부
print("\n===== 생성 예정 파일 충돌 검사 =====")

target_files = [
    "prepare_data.py",
    "train.py",
    "evaluate.py",
    "README.md"
]

for filename in target_files:
    path = target_dir / filename

    status = "⚠️ 이미 존재" if path.exists() else "✅ 생성 가능"
    print(f"{filename}: {status}")

# 4. Git 변경 사항 확인
print("\n===== 현재 Git 변경 사항 =====")

subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True
)

print("\n✅ 17단계 안전 점검 완료!")
print("파일 생성·수정·삭제·Git 업로드 없음")

===== 17단계. GitHub 작업 환경 안전 점검 =====

현재 브랜치: feature/korean-ai-fusion

===== 기존 폴더 구조 =====

경로: /content/pinpoint-capstone/ai/experiments
존재 여부: True
하위 항목:
 - .gitkeep

경로: /content/pinpoint-capstone/ai/experiments/korean_minilm
존재 여부: False

===== 생성 예정 파일 충돌 검사 =====
prepare_data.py: ✅ 생성 가능
train.py: ✅ 생성 가능
evaluate.py: ✅ 생성 가능
README.md: ✅ 생성 가능

===== 현재 Git 변경 사항 =====

✅ 17단계 안전 점검 완료!
파일 생성·수정·삭제·Git 업로드 없음


In [270]:
from pathlib import Path
import ast

repo = Path("/content/pinpoint-capstone")
target_dir = repo / "ai/experiments/korean_minilm"
target_file = target_dir / "prepare_data.py"

# 기존 파일 보호
assert not target_file.exists(), (
    "⚠️ prepare_data.py가 이미 존재합니다. "
    "기존 파일을 덮어쓰지 않습니다."
)

code = r'''
"""
PinPoint Korean MiniLM training data preparation.

Input:
    AI-Hub TL_12 and VL_12 TAR files containing ZIP archives of JSON labels.

Output:
    minilm_train_pairs.jsonl
    minilm_train_pairs_clean.jsonl
    minilm_val_pairs.jsonl

AI-Hub source data and generated text pairs must not be committed to GitHub.
"""

import argparse
import io
import json
import tarfile
import zipfile
from collections import Counter
from pathlib import Path

from sentence_transformers import SentenceTransformer


MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

MAX_TOKENS = 128
EXCLUDED_VAL_IDS = {"MYL_18127", "MYL_18128"}


def load_archive(tar_path):
    """Read the single ZIP archive stored inside an AI-Hub TAR file."""

    with tarfile.open(tar_path, "r:*") as tar:
        members = [m for m in tar.getmembers() if m.isfile()]

        if len(members) != 1:
            raise ValueError(
                f"Expected one ZIP in {tar_path}, found {len(members)}"
            )

        extracted = tar.extractfile(members[0])

        if extracted is None:
            raise ValueError(f"Cannot read TAR member: {tar_path}")

        zip_bytes = extracted.read()

    archive = zipfile.ZipFile(io.BytesIO(zip_bytes))

    if archive.testzip() is not None:
        archive.close()
        raise ValueError(f"Damaged ZIP archive: {tar_path}")

    return archive


def build_pairs(archive, tokenizer, split):
    """Create summary-to-transcript pairs with the original token filter."""

    pairs = []

    for name in archive.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(archive.read(name).decode("utf-8-sig"))

        video_id = data.get("metadata", {}).get("filename")

        if split == "val" and video_id in EXCLUDED_VAL_IDS:
            continue

        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])

        transcript = " ".join(
            str(term.get("transcription") or "").strip()
            for term in terms
            if isinstance(term, dict)
        ).strip()

        if not summary or not transcript:
            continue

        summary_tokens = len(
            tokenizer(
                summary,
                truncation=False,
                verbose=False
            )["input_ids"]
        )

        transcript_tokens = len(
            tokenizer(
                transcript,
                truncation=False,
                verbose=False
            )["input_ids"]
        )

        if (
            summary_tokens > MAX_TOKENS
            or transcript_tokens > MAX_TOKENS
        ):
            continue

        pairs.append({
            "video_id": video_id,
            "query": summary,
            "positive": transcript
        })

    return pairs


def save_jsonl(path, records):
    """Create a new JSONL file without overwriting existing data."""

    with path.open("x", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-tar", type=Path, required=True)
    parser.add_argument("--val-tar", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)

    args = parser.parse_args()

    train_tar = args.train_tar.resolve()
    val_tar = args.val_tar.resolve()

    if not train_tar.is_file() or not val_tar.is_file():
        raise FileNotFoundError("Training or Validation TAR not found.")

    if train_tar == val_tar:
        raise ValueError("Training and Validation TAR must differ.")

    if not train_tar.name.startswith("TL_12"):
        raise ValueError("Expected a TL_12 Training TAR file.")

    if not val_tar.name.startswith("VL_12"):
        raise ValueError("Expected a VL_12 Validation TAR file.")

    output_dir = args.output_dir.resolve()

    output_files = {
        "train": output_dir / "minilm_train_pairs.jsonl",
        "clean": output_dir / "minilm_train_pairs_clean.jsonl",
        "val": output_dir / "minilm_val_pairs.jsonl"
    }

    for path in output_files.values():
        if path.exists():
            raise FileExistsError(
                f"Refusing to overwrite existing file: {path}"
            )

    model = SentenceTransformer(MODEL_NAME)
    model.max_seq_length = MAX_TOKENS
    tokenizer = model.tokenizer

    with load_archive(train_tar) as train_archive:
        train_pairs = build_pairs(
            train_archive, tokenizer, split="train"
        )

    with load_archive(val_tar) as val_archive:
        val_pairs = build_pairs(
            val_archive, tokenizer, split="val"
        )

    train_ids = {p["video_id"] for p in train_pairs}
    val_ids = {p["video_id"] for p in val_pairs}

    overlap = train_ids & val_ids

    if overlap:
        raise ValueError(
            f"Training/Validation video ID overlap: {len(overlap)}"
        )

    query_counts = Counter(
        p["query"].strip() for p in train_pairs
    )

    clean_pairs = [
        p for p in train_pairs
        if query_counts[p["query"].strip()] == 1
    ]

    if not train_pairs or not clean_pairs or not val_pairs:
        raise ValueError("One or more datasets are empty.")

    output_dir.mkdir(parents=True, exist_ok=True)

    save_jsonl(output_files["train"], train_pairs)
    save_jsonl(output_files["clean"], clean_pairs)
    save_jsonl(output_files["val"], val_pairs)

    print("===== PinPoint MiniLM data preparation =====")
    print("Training pairs:", len(train_pairs))
    print("Clean training pairs:", len(clean_pairs))
    print("Validation pairs:", len(val_pairs))
    print("Training/Validation ID overlap:", len(overlap))
    print("Output:", output_dir)


if __name__ == "__main__":
    main()
'''

# Python 문법 검사
ast.parse(code)
compile(code, str(target_file), "exec")

# 신규 폴더와 파일만 생성
target_dir.mkdir(parents=True, exist_ok=True)

# 'x' 모드: 같은 이름의 파일이 있으면 덮어쓰지 않고 오류 발생
with target_file.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

print("===== 18단계. 전처리 코드 생성 =====")
print("✅ 신규 파일 생성 완료!")
print("파일:", target_file)
print("파일 크기:", target_file.stat().st_size, "bytes")
print("✅ Python 문법 검사 통과")
print("✅ 기존 파일 덮어쓰기 없음")
print("✅ 원본 AI-Hub 데이터 변경 없음")
print("✅ GitHub 업로드 없음")

===== 18단계. 전처리 코드 생성 =====
✅ 신규 파일 생성 완료!
파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/prepare_data.py
파일 크기: 5849 bytes
✅ Python 문법 검사 통과
✅ 기존 파일 덮어쓰기 없음
✅ 원본 AI-Hub 데이터 변경 없음
✅ GitHub 업로드 없음


In [271]:
from pathlib import Path
import json
import subprocess

repo = Path("/content/pinpoint-capstone")

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

test_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/prepare_data_verification_20261010"
)

print("===== 19단계. 전처리 실행 전 안전 점검 =====")

# 1. 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

print("\n현재 브랜치:", branch)
assert branch == "feature/korean-ai-fusion"

# 2. 원본 데이터 확인
train_files = list(training_dir.glob("TL_12*.tar"))
val_files = list(training_dir.glob("VL_12*.tar"))

print("\n===== AI-Hub 원본 파일 =====")
print("Training 파일 수:", len(train_files))
print("Validation 파일 수:", len(val_files))

for path in train_files + val_files:
    print("-", path.name)

assert len(train_files) == 1, "Training TAR 파일을 확인해야 합니다."
assert len(val_files) == 1, "Validation TAR 파일을 확인해야 합니다."

# 3. 기존 학습 데이터 개수 확인
print("\n===== 기존 학습 데이터 =====")

existing_files = [
    "minilm_train_pairs.jsonl",
    "minilm_train_pairs_clean.jsonl",
    "minilm_val_pairs.jsonl"
]

for filename in existing_files:
    path = training_dir / filename
    assert path.is_file(), f"기존 파일이 없습니다: {filename}"

    with path.open("r", encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())

    print(f"{filename}: {count:,}개")

# 4. 테스트 폴더 충돌 확인
print("\n===== 별도 검증 경로 =====")
print("검증 경로:", test_dir)
print("이미 존재:", test_dir.exists())

assert not test_dir.exists(), (
    "검증 폴더가 이미 존재합니다. "
    "덮어쓰지 않도록 작업을 중단합니다."
)

# 5. 전처리 코드 확인
script = repo / "ai/experiments/korean_minilm/prepare_data.py"

assert script.is_file(), "전처리 코드가 없습니다."
print("\n전처리 코드 존재: ✅")

print("\n✅ 19단계 안전 점검 완료!")
print("✅ 원본 데이터 변경 없음")
print("✅ 새 테스트 파일 생성 없음")
print("✅ 기존 팀원 코드 변경 없음")
print("✅ GitHub 업로드 없음")

===== 19단계. 전처리 실행 전 안전 점검 =====

현재 브랜치: feature/korean-ai-fusion

===== AI-Hub 원본 파일 =====
Training 파일 수: 1
Validation 파일 수: 1
- TL_12.음식요리.tar
- VL_12.음식요리.tar

===== 기존 학습 데이터 =====
minilm_train_pairs.jsonl: 15,823개
minilm_train_pairs_clean.jsonl: 15,502개
minilm_val_pairs.jsonl: 1,948개

===== 별도 검증 경로 =====
검증 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/prepare_data_verification_20261010
이미 존재: False

전처리 코드 존재: ✅

✅ 19단계 안전 점검 완료!
✅ 원본 데이터 변경 없음
✅ 새 테스트 파일 생성 없음
✅ 기존 팀원 코드 변경 없음
✅ GitHub 업로드 없음


In [272]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib

repo = Path("/content/pinpoint-capstone")

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

test_dir = (
    training_dir / "prepare_data_verification_20261010"
)

script = (
    repo / "ai/experiments/korean_minilm/prepare_data.py"
)

train_tar = next(training_dir.glob("TL_12*.tar"))
val_tar = next(training_dir.glob("VL_12*.tar"))

# 안전 확인
assert subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip() == "feature/korean-ai-fusion"

assert script.is_file()
assert train_tar.is_file()
assert val_tar.is_file()

assert not test_dir.exists(), (
    "⚠️ 테스트 폴더가 이미 존재합니다. "
    "기존 데이터를 보호하기 위해 중단합니다."
)

print("===== 20단계. 전처리 재현 검증 =====")
print("Training:", train_tar.name)
print("Validation:", val_tar.name)
print("테스트 출력 경로:", test_dir)

# 새 전처리 코드 실행
result = subprocess.run(
    [
        sys.executable,
        str(script),
        "--train-tar", str(train_tar),
        "--val-tar", str(val_tar),
        "--output-dir", str(test_dir)
    ],
    cwd=repo,
    text=True,
    capture_output=True
)

print("\n===== 전처리 실행 결과 =====")
print(result.stdout)

if result.returncode != 0:
    print("❌ 전처리 실행 중 오류 발생")
    print(result.stderr[-5000:])
    raise RuntimeError(
        "전처리가 완료되지 않았습니다. "
        "오류를 확인한 뒤 다음 단계를 진행해야 합니다."
    )

# 기존 데이터와 새 데이터 비교
filenames = [
    "minilm_train_pairs.jsonl",
    "minilm_train_pairs_clean.jsonl",
    "minilm_val_pairs.jsonl"
]

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

print("\n===== 기존 데이터와 재현 데이터 비교 =====")

all_matched = True

for filename in filenames:
    original = training_dir / filename
    reproduced = test_dir / filename

    old_data = load_jsonl(original)
    new_data = load_jsonl(reproduced)

    same_count = len(old_data) == len(new_data)
    same_content = old_data == new_data

    print(f"\n[{filename}]")
    print("기존 개수:", len(old_data))
    print("재현 개수:", len(new_data))
    print("개수 일치:", same_count)
    print("내용 및 순서 일치:", same_content)

    if not same_content:
        all_matched = False

print("\n===== 최종 검증 결과 =====")

if all_matched:
    print("✅ 기존 학습 데이터와 완전히 일치!")
    print("✅ MiniLM 전처리 코드 재현 검증 성공!")
else:
    print("⚠️ 기존 데이터와 다른 부분이 있습니다.")
    print("⚠️ 원인을 확인하기 전까지 코드를 확정하지 않습니다.")

print("\n기존 원본 데이터 및 팀원 코드 변경 없음")
print("GitHub 업로드 없음")

===== 20단계. 전처리 재현 검증 =====
Training: TL_12.음식요리.tar
Validation: VL_12.음식요리.tar
테스트 출력 경로: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/prepare_data_verification_20261010

===== 전처리 실행 결과 =====
===== PinPoint MiniLM data preparation =====
Training pairs: 15823
Clean training pairs: 15502
Validation pairs: 1948
Training/Validation ID overlap: 0
Output: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/prepare_data_verification_20261010


===== 기존 데이터와 재현 데이터 비교 =====

[minilm_train_pairs.jsonl]
기존 개수: 15823
재현 개수: 15823
개수 일치: True
내용 및 순서 일치: True

[minilm_train_pairs_clean.jsonl]
기존 개수: 15502
재현 개수: 15502
개수 일치: True
내용 및 순서 일치: True

[minilm_val_pairs.jsonl]
기존 개수: 1948
재현 개수: 1948
개수 일치: True
내용 및 순서 일치: True

===== 최종 검증 결과 =====
✅ 기존 학습 데이터와 완전히 일치!
✅ MiniLM 전처리 코드 재현 검증 성공!

기존 원본 데이터 및 팀원 코드 변경 없음
GitHub 업로드 없음


In [273]:
from pathlib import Path
import ast

repo = Path("/content/pinpoint-capstone")
target = repo / "ai/experiments/korean_minilm/train.py"

assert target.parent.is_dir(), "학습 코드 폴더가 없습니다."
assert not target.exists(), (
    "⚠️ train.py가 이미 존재합니다. 기존 파일을 덮어쓰지 않습니다."
)

code = r'''
"""
PinPoint Korean MiniLM fine-tuning.

Reproduces the original Korean MiniLM training configuration.

Example:
    python -m ai.experiments.korean_minilm.train \
        --train-jsonl /path/to/minilm_train_pairs_clean.jsonl \
        --output-dir /path/to/new_training_output

Do not upload licensed AI-Hub training data or model weights to GitHub.
"""

import argparse
import json
from pathlib import Path

import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    losses,
)
from sentence_transformers.training_args import (
    SentenceTransformerTrainingArguments,
    BatchSamplers,
)


MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


def load_training_pairs(path):
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            query = str(record.get("query") or "").strip()
            positive = str(record.get("positive") or "").strip()

            if not query or not positive:
                raise ValueError("Empty query or positive text found.")

            records.append({
                "anchor": query,
                "positive": positive,
            })

    if not records:
        raise ValueError("Training data is empty.")

    return Dataset.from_list(records)


def main():
    parser = argparse.ArgumentParser(
        description="Fine-tune PinPoint Korean MiniLM"
    )

    parser.add_argument(
        "--train-jsonl",
        type=Path,
        required=True
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        required=True
    )

    args = parser.parse_args()

    train_path = args.train_jsonl.resolve()
    output_dir = args.output_dir.resolve()

    if not train_path.is_file():
        raise FileNotFoundError(train_path)

    # 기존 모델 및 체크포인트를 보호한다.
    if output_dir.exists():
        raise FileExistsError(
            f"Output directory already exists: {output_dir}"
        )

    # 학습 시작 전에 폴더가 비어 있음을 확인한다.
    # 기존 모델 경로에 잘못 지정하는 사고를 막는다.
    if output_dir == train_path.parent:
        raise ValueError(
            "Output directory must differ from the data directory."
        )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Original training used CUDA with fp16. "
            "Please enable a GPU runtime."
        )

    train_dataset = load_training_pairs(train_path)

    model = SentenceTransformer(
        MODEL_NAME,
        device="cuda"
    )
    model.max_seq_length = 128

    train_loss = losses.MultipleNegativesRankingLoss(
        model=model
    )

    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=1,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        fp16=True,
        bf16=False,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        save_strategy="epoch",
        save_total_limit=1,
        eval_strategy="no",
        logging_steps=100,
        report_to="none",
        seed=42,
        data_seed=42,
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        loss=train_loss,
    )

    print("===== PinPoint MiniLM Fine-tuning =====")
    print("Training samples:", len(train_dataset))
    print("Model:", MODEL_NAME)
    print("Epochs:", training_args.num_train_epochs)
    print("Batch size:", training_args.per_device_train_batch_size)
    print("Learning rate:", training_args.learning_rate)
    print("Output:", output_dir)

    trainer.train()

    final_dir = output_dir / "final_model"

    if final_dir.exists():
        raise FileExistsError(
            f"Refusing to overwrite model: {final_dir}"
        )

    trainer.model.save_pretrained(str(final_dir))

    print("✅ MiniLM fine-tuning completed.")
    print("Saved model:", final_dir)


if __name__ == "__main__":
    main()
'''

ast.parse(code)
compile(code, str(target), "exec")

with target.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

print("===== 21단계. MiniLM 학습 코드 생성 =====")
print("✅ train.py 생성 완료!")
print("파일:", target)
print("파일 크기:", target.stat().st_size, "bytes")
print("✅ Python 문법 검사 통과")
print("✅ 학습 실행하지 않음")
print("✅ 기존 모델 및 체크포인트 변경 없음")
print("✅ 기존 팀원 파일 변경 없음")
print("✅ GitHub 업로드 없음")

===== 21단계. MiniLM 학습 코드 생성 =====
✅ train.py 생성 완료!
파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/train.py
파일 크기: 4246 bytes
✅ Python 문법 검사 통과
✅ 학습 실행하지 않음
✅ 기존 모델 및 체크포인트 변경 없음
✅ 기존 팀원 파일 변경 없음
✅ GitHub 업로드 없음


In [274]:
import ast
import importlib
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")
script = repo / "ai/experiments/korean_minilm/train.py"

data_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_train_pairs_clean.jsonl"
)

print("===== 22단계. MiniLM 학습 코드 안전 검증 =====")

# 1. 작업 브랜치 보호
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인:", branch)

# 2. 코드 문법 검사
assert script.is_file(), "train.py 파일이 없습니다."

source = script.read_text(encoding="utf-8")
tree = ast.parse(source)

compile(source, str(script), "exec")
print("✅ Python 문법 검사 통과")

# 3. 학습 설정 확인: 실제 학습은 실행하지 않음
expected_settings = {
    "num_train_epochs": 1,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "fp16": True,
    "bf16": False,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "eval_strategy": "no",
    "seed": 42,
    "data_seed": 42,
}

args_calls = [
    node for node in ast.walk(tree)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id == "SentenceTransformerTrainingArguments"
]

assert len(args_calls) == 1, "학습 설정 생성 부분을 확인해야 합니다."

actual_settings = {
    keyword.arg: ast.literal_eval(keyword.value)
    for keyword in args_calls[0].keywords
    if keyword.arg in expected_settings
}

for name, expected in expected_settings.items():
    actual = actual_settings.get(name)
    assert actual == expected, (
        f"학습 설정 불일치: {name} "
        f"(기대={expected}, 실제={actual})"
    )

print("✅ 기존 MiniLM 학습 하이퍼파라미터 일치")

# 4. 학습 실행 코드가 존재하는지만 확인
train_calls = [
    node for node in ast.walk(tree)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Attribute)
    and node.func.attr == "train"
]

assert train_calls, "학습 실행 코드가 없습니다."
print("✅ 학습 실행 코드 존재 확인 (실행하지 않음)")

# 5. 기존 데이터 로드 함수만 불러오기
module = importlib.import_module(
    "ai.experiments.korean_minilm.train"
)

assert data_path.is_file(), "기존 학습 데이터가 없습니다."

dataset = module.load_training_pairs(data_path)

assert len(dataset) == 15502, (
    f"학습 데이터 개수가 다릅니다: {len(dataset)}"
)

assert dataset.column_names == ["anchor", "positive"]

print("✅ 기존 학습 데이터 로드 성공")
print("학습 데이터:", f"{len(dataset):,}쌍")
print("데이터 열:", dataset.column_names)

print("\n✅ 22단계 검증 완료!")
print("✅ 모델 학습 실행 없음")
print("✅ 기존 데이터 및 모델 변경 없음")
print("✅ 팀원 코드 변경 없음")
print("✅ GitHub 업로드 없음")

===== 22단계. MiniLM 학습 코드 안전 검증 =====
✅ AI 전용 브랜치 확인: feature/korean-ai-fusion
✅ Python 문법 검사 통과
✅ 기존 MiniLM 학습 하이퍼파라미터 일치
✅ 학습 실행 코드 존재 확인 (실행하지 않음)
✅ 기존 학습 데이터 로드 성공
학습 데이터: 15,502쌍
데이터 열: ['anchor', 'positive']

✅ 22단계 검증 완료!
✅ 모델 학습 실행 없음
✅ 기존 데이터 및 모델 변경 없음
✅ 팀원 코드 변경 없음
✅ GitHub 업로드 없음


In [275]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/PinPoint_Korean_Video_Test.ipynb"
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

print("===== 23단계. MiniLM 평가 코드 확인 =====")

keywords = [
    "minilm_full_validation_comparison.json",
    "minilm_finetuning_comparison.json",
    "recall_at_1",
    "recall_at_5",
    "evaluate_minilm"
]

matched_cells = []

for i, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        matched_cells.append(i)

print("평가 관련 셀 번호:", matched_cells)

# 실제 학습 직후 평가 및 전체 Validation 평가 구간 확인
for i in matched_cells:
    if not (313 <= i <= 380):
        continue

    source = "".join(
        notebook["cells"][i].get("source", [])
    )

    print(f"\n{'=' * 60}")
    print(f"셀 번호: {i}")
    print("=" * 60)
    print(source[:10000])

print("\n✅ 23단계 평가 코드 확인 완료!")
print("✅ 노트북 수정 없음")
print("✅ 모델 평가 실행 없음")
print("✅ 팀원 코드 변경 없음")
print("✅ GitHub 업로드 없음")

===== 23단계. MiniLM 평가 코드 확인 =====
평가 관련 셀 번호: [313, 327, 329, 342, 357, 373, 380]

셀 번호: 313
import json
import random
import numpy as np
from sentence_transformers import util
from pathlib import Path

# 추가 학습 전 기존 MiniLM
model = fusion_search.get_semantic_model()

# 동일한 샘플을 재현할 수 있도록 고정
rng = random.Random(42)
eval_pairs = rng.sample(val_pairs, 100)

queries = [p["query"] for p in eval_pairs]
documents = [p["positive"] for p in eval_pairs]

# 기존 MiniLM 임베딩 생성
query_embeddings = model.encode(
    queries,
    convert_to_tensor=True,
    show_progress_bar=True
)

doc_embeddings = model.encode(
    documents,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 100개 요약문 × 100개 발화문 유사도
scores = util.cos_sim(query_embeddings, doc_embeddings)

ranks = []

for i in range(len(eval_pairs)):
    ranked_indices = scores[i].argsort(descending=True).tolist()

    # 같은 인덱스의 요약문-발화문을 정답 쌍으로 간주
    rank = ranked_indices.index(i) + 1
    ranks.append(rank)

recall_at_1 = np.mean([rank <= 1 for

In [276]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")
target = repo / "ai/experiments/korean_minilm/evaluate.py"

# 팀원 코드 보호: 지정된 브랜치에서만 작업
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion", (
    "예상한 AI 전용 브랜치가 아닙니다."
)

# 기존 파일 덮어쓰기 방지
assert target.parent.is_dir()
assert not target.exists(), (
    "evaluate.py가 이미 존재합니다. 덮어쓰지 않습니다."
)

code = r'''
"""
PinPoint Korean MiniLM full-validation retrieval evaluation.

Reproduces the original notebook's summary-to-transcript
retrieval metrics: Recall@1, Recall@5, and MRR.

The matching transcript at the same index is treated as
the single correct answer.

This does NOT measure temporal video moment retrieval.
"""

import argparse
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer, util


BASE_MODEL = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


def load_validation_pairs(path):
    pairs = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            query = record.get("query")
            positive = record.get("positive")

            if not query or not positive:
                raise ValueError("Empty evaluation text found.")

            pairs.append({
                "query": query,
                "positive": positive
            })

    if not pairs:
        raise ValueError("Validation dataset is empty.")

    return pairs


def evaluate_minilm(model, queries, documents):
    """Match the original notebook's retrieval evaluation."""

    query_embeddings = model.encode(
        queries,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    doc_embeddings = model.encode(
        documents,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        query_embeddings,
        doc_embeddings
    )

    ranks = []

    for i in range(len(queries)):
        ranked_indices = (
            scores[i].argsort(descending=True).tolist()
        )

        ranks.append(ranked_indices.index(i) + 1)

    return {
        "recall_at_1": float(
            np.mean([r == 1 for r in ranks])
        ),
        "recall_at_5": float(
            np.mean([r <= 5 for r in ranks])
        ),
        "mrr": float(
            np.mean([1 / r for r in ranks])
        )
    }


def main():
    parser = argparse.ArgumentParser(
        description="Evaluate Korean MiniLM retrieval"
    )

    parser.add_argument(
        "--val-jsonl",
        type=Path,
        required=True
    )

    parser.add_argument(
        "--finetuned-model",
        type=Path,
        required=True
    )

    parser.add_argument(
        "--output",
        type=Path,
        required=True
    )

    args = parser.parse_args()

    val_path = args.val_jsonl.resolve()
    model_path = args.finetuned_model.resolve()
    output_path = args.output.resolve()

    if not val_path.is_file():
        raise FileNotFoundError(val_path)

    if not model_path.is_dir():
        raise FileNotFoundError(model_path)

    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite: {output_path}"
        )

    pairs = load_validation_pairs(val_path)

    queries = [p["query"] for p in pairs]
    documents = [p["positive"] for p in pairs]

    print("===== PinPoint Korean MiniLM evaluation =====")
    print("Validation pairs:", len(pairs))

    print("\n[Original MiniLM]")
    original_model = SentenceTransformer(BASE_MODEL)

    original_result = evaluate_minilm(
        original_model, queries, documents
    )

    print(original_result)

    print("\n[Fine-tuned MiniLM]")
    trained_model = SentenceTransformer(
        str(model_path)
    )

    trained_result = evaluate_minilm(
        trained_model, queries, documents
    )

    print(trained_result)

    result = {
        "evaluation": (
            "AI-Hub full validation "
            "summary-to-transcript retrieval"
        ),
        "sample_size": len(pairs),
        "original": original_result,
        "finetuned": trained_result
    }

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with output_path.open("x", encoding="utf-8") as f:
        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n✅ Evaluation completed!")
    print("Saved:", output_path)


if __name__ == "__main__":
    main()
'''

# 코드 문법 검증 후 새 파일만 생성
ast.parse(code)
compile(code, str(target), "exec")

with target.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

print("===== 24단계. MiniLM 평가 코드 생성 =====")
print("✅ evaluate.py 생성 완료!")
print("파일:", target)
print("파일 크기:", target.stat().st_size, "bytes")
print("✅ Python 문법 검사 통과")
print("✅ 평가 실행하지 않음")
print("✅ 기존 평가 JSON 변경 없음")
print("✅ 기존 팀원 코드 변경 없음")
print("✅ GitHub 업로드 없음")

===== 24단계. MiniLM 평가 코드 생성 =====
✅ evaluate.py 생성 완료!
파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/evaluate.py
파일 크기: 4182 bytes
✅ Python 문법 검사 통과
✅ 평가 실행하지 않음
✅ 기존 평가 JSON 변경 없음
✅ 기존 팀원 코드 변경 없음
✅ GitHub 업로드 없음


In [277]:
import ast
import importlib
import subprocess
import torch
import numpy as np

from pathlib import Path
from sentence_transformers import util

repo = Path("/content/pinpoint-capstone")
script = repo / "ai/experiments/korean_minilm/evaluate.py"

print("===== 25단계. MiniLM 평가 함수 검증 =====")

# 1. 브랜치 및 코드 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert script.is_file()

ast.parse(script.read_text(encoding="utf-8"))

print("✅ AI 전용 브랜치 및 코드 확인")

# 2. 새 평가 함수 불러오기
module = importlib.import_module(
    "ai.experiments.korean_minilm.evaluate"
)

# 3. 임베딩을 미리 정해 놓은 가짜 모델
# 실제 MiniLM이나 학습 모델은 불러오지 않음
class MockModel:
    def __init__(self, embeddings):
        self.embeddings = embeddings

    def encode(self, texts, **kwargs):
        return torch.tensor(
            [self.embeddings[text] for text in texts],
            dtype=torch.float32
        )

embeddings = {
    "q1": [1.0, 0.0, 0.0],
    "q2": [0.0, 1.0, 0.0],
    "q3": [0.0, 0.0, 1.0],
    "d1": [1.0, 0.0, 0.0],
    "d2": [0.0, 0.0, 1.0],
    "d3": [0.0, 1.0, 0.0],
}

queries = ["q1", "q2", "q3"]
documents = ["d1", "d2", "d3"]

mock_model = MockModel(embeddings)

# 4. 기존 노트북의 평가 계산식
query_embeddings = mock_model.encode(queries)
doc_embeddings = mock_model.encode(documents)

scores = util.cos_sim(
    query_embeddings,
    doc_embeddings
)

ranks = []

for i in range(len(queries)):
    ranked_indices = scores[i].argsort(
        descending=True
    ).tolist()

    ranks.append(ranked_indices.index(i) + 1)

old_result = {
    "recall_at_1": float(
        np.mean([r == 1 for r in ranks])
    ),
    "recall_at_5": float(
        np.mean([r <= 5 for r in ranks])
    ),
    "mrr": float(
        np.mean([1 / r for r in ranks])
    )
}

# 5. 새 evaluate.py의 평가 함수
new_result = module.evaluate_minilm(
    mock_model,
    queries,
    documents
)

print("\n===== 평가 결과 비교 =====")
print("기존 계산식:", old_result)
print("새 평가 함수:", new_result)

assert old_result == new_result, (
    "⚠️ 기존 평가 방식과 새 평가 함수가 다릅니다."
)

assert old_result["recall_at_1"] == 1 / 3

print("\n✅ 기존 평가 계산식과 새 함수 결과 일치!")
print("✅ 실제 모델 로드 및 평가 실행 없음")
print("✅ 기존 데이터·모델·팀원 코드 수정 없음")
print("✅ GitHub 업로드 없음")

===== 25단계. MiniLM 평가 함수 검증 =====
✅ AI 전용 브랜치 및 코드 확인

===== 평가 결과 비교 =====
기존 계산식: {'recall_at_1': 0.3333333333333333, 'recall_at_5': 1.0, 'mrr': 0.5555555555555555}
새 평가 함수: {'recall_at_1': 0.3333333333333333, 'recall_at_5': 1.0, 'mrr': 0.5555555555555555}

✅ 기존 평가 계산식과 새 함수 결과 일치!
✅ 실제 모델 로드 및 평가 실행 없음
✅ 기존 데이터·모델·팀원 코드 수정 없음
✅ GitHub 업로드 없음


In [278]:
from pathlib import Path
import subprocess
import sys
import json

repo = Path("/content/pinpoint-capstone")

training_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data"
)

script = repo / "ai/experiments/korean_minilm/evaluate.py"

val_path = training_dir / "minilm_val_pairs.jsonl"
model_path = training_dir / "minilm_finetuned/final_model"

original_result_path = (
    training_dir / "minilm_full_validation_comparison.json"
)

new_result_path = (
    training_dir / "minilm_full_validation_verification_20261010.json"
)

print("===== 26단계. MiniLM 전체 평가 재현 검증 =====")

# 1. 기존 코드와 결과 보호
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

assert script.is_file()
assert val_path.is_file()
assert model_path.is_dir()
assert original_result_path.is_file()

assert not new_result_path.exists(), (
    "⚠️ 검증 결과 파일이 이미 존재합니다. "
    "덮어쓰지 않도록 중단합니다."
)

print("✅ AI 전용 브랜치 확인")
print("✅ 기존 Validation 데이터 확인")
print("✅ Fine-tuned MiniLM 모델 확인")
print("✅ 기존 평가 결과 파일 확인")
print("✅ 새 결과 파일 충돌 없음")

# 2. 기존 저장 결과 확인
old_result = json.loads(
    original_result_path.read_text(encoding="utf-8")
)

print("\n===== 기존 평가 결과 =====")
print(json.dumps(old_result, ensure_ascii=False, indent=2))

# 3. 새로운 평가 코드 실행
print("\n===== 전체 Validation 1,948쌍 평가 실행 =====")

result = subprocess.run(
    [
        sys.executable,
        str(script),
        "--val-jsonl", str(val_path),
        "--finetuned-model", str(model_path),
        "--output", str(new_result_path),
    ],
    cwd=repo,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print("❌ 평가 중 오류 발생")
    print(result.stderr[-5000:])
    raise RuntimeError(
        "평가가 실패했습니다. 기존 파일은 변경하지 않았습니다."
    )

# 4. 성능 수치 비교
new_result = json.loads(
    new_result_path.read_text(encoding="utf-8")
)

print("\n===== 기존 결과와 재현 결과 비교 =====")

all_matched = True

for model_name in ["original", "finetuned"]:
    print(f"\n[{model_name}]")

    for metric in ["recall_at_1", "recall_at_5", "mrr"]:
        old_value = old_result[model_name][metric]
        new_value = new_result[model_name][metric]

        difference = abs(old_value - new_value)
        matched = difference < 1e-6

        print(
            f"{metric}: "
            f"기존={old_value:.6f}, "
            f"재현={new_value:.6f}, "
            f"일치={matched}"
        )

        if not matched:
            all_matched = False

print("\n===== 최종 결과 =====")

if all_matched:
    print("✅ MiniLM 전체 Validation 성능 재현 성공!")
else:
    print("⚠️ 일부 지표가 기존 결과와 다릅니다.")
    print("⚠️ 원인을 분석하기 전에는 평가 코드를 확정하지 않습니다.")

print("\n✅ 기존 평가 JSON 및 모델 변경 없음")
print("✅ 팀원 코드 변경 없음")
print("✅ GitHub 업로드 없음")

===== 26단계. MiniLM 전체 평가 재현 검증 =====
✅ AI 전용 브랜치 확인
✅ 기존 Validation 데이터 확인
✅ Fine-tuned MiniLM 모델 확인
✅ 기존 평가 결과 파일 확인
✅ 새 결과 파일 충돌 없음

===== 기존 평가 결과 =====
{
  "evaluation": "AI-Hub full validation summary-to-transcript retrieval",
  "sample_size": 1948,
  "original": {
    "recall_at_1": 0.5102669404517454,
    "recall_at_5": 0.6555441478439425,
    "mrr": 0.577302482579806
  },
  "finetuned": {
    "recall_at_1": 0.7515400410677618,
    "recall_at_5": 0.858829568788501,
    "mrr": 0.7996197008399655
  }
}

===== 전체 Validation 1,948쌍 평가 실행 =====
===== PinPoint Korean MiniLM evaluation =====
Validation pairs: 1948

[Original MiniLM]
{'recall_at_1': 0.5102669404517454, 'recall_at_5': 0.6555441478439425, 'mrr': 0.577302482579806}

[Fine-tuned MiniLM]
{'recall_at_1': 0.7515400410677618, 'recall_at_5': 0.858829568788501, 'mrr': 0.7996197008399655}

✅ Evaluation completed!
Saved: /content/drive/MyDrive/pinpoint_capstone/03_Korean_Training_Data/minilm_full_validation_verification_20261010.js

In [279]:
from pathlib import Path
import json
import subprocess

repo = Path("/content/pinpoint-capstone")

source = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_full_validation_comparison.json"
)

target = (
    repo / "ai/experiments/korean_minilm/"
    "results/full_validation_metrics.json"
)

print("===== 27단계. 평가 요약 결과 저장 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert source.is_file(), "원본 평가 결과가 없습니다."
assert not target.exists(), "대상 파일이 이미 존재합니다. 덮어쓰지 않습니다."

# 2. 필요한 숫자만 추출
original = json.loads(source.read_text(encoding="utf-8"))

metrics = ("recall_at_1", "recall_at_5", "mrr")

assert original["sample_size"] == 1948

for model_name in ("original", "finetuned"):
    assert all(
        isinstance(original[model_name][metric], (int, float))
        for metric in metrics
    )

# 원문 텍스트나 학습 데이터는 포함하지 않음
summary = {
    "task": "Korean summary-to-transcript retrieval",
    "dataset": "AI-Hub food/cooking validation",
    "sample_size": 1948,
    "evaluation_note": (
        "The transcript at the same index is the single "
        "positive. These are text retrieval metrics, not "
        "video temporal localization metrics."
    ),
    "base_model": (
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
    "original": {
        metric: original["original"][metric]
        for metric in metrics
    },
    "finetuned": {
        metric: original["finetuned"][metric]
        for metric in metrics
    },
    "verification": (
        "Recomputed on 2026-10-10; all six metrics matched "
        "the previously saved evaluation within 1e-6."
    )
}

# 3. 새 파일만 생성
target.parent.mkdir(parents=True, exist_ok=True)

with target.open("x", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# 4. 저장 내용 검증
saved = json.loads(target.read_text(encoding="utf-8"))
assert saved == summary

print("✅ 평가 요약 JSON 생성 완료!")
print("경로:", target)
print("평가 데이터 수:", saved["sample_size"])
print("기본 MiniLM:", saved["original"])
print("Fine-tuned MiniLM:", saved["finetuned"])
print("✅ AI-Hub 원문 및 모델 가중치 포함 없음")
print("✅ 기존 결과 및 팀원 코드 변경 없음")
print("✅ GitHub 업로드 없음")

===== 27단계. 평가 요약 결과 저장 =====
✅ 평가 요약 JSON 생성 완료!
경로: /content/pinpoint-capstone/ai/experiments/korean_minilm/results/full_validation_metrics.json
평가 데이터 수: 1948
기본 MiniLM: {'recall_at_1': 0.5102669404517454, 'recall_at_5': 0.6555441478439425, 'mrr': 0.577302482579806}
Fine-tuned MiniLM: {'recall_at_1': 0.7515400410677618, 'recall_at_5': 0.858829568788501, 'mrr': 0.7996197008399655}
✅ AI-Hub 원문 및 모델 가중치 포함 없음
✅ 기존 결과 및 팀원 코드 변경 없음
✅ GitHub 업로드 없음


In [282]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")
target = repo / "ai/experiments/korean_minilm/README.md"

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion", (
    "AI 전용 브랜치가 아닙니다. 작업을 중단합니다."
)

# 2. 기존 README 보호
assert target.parent.is_dir(), "MiniLM 실험 폴더가 없습니다."
assert not target.exists(), (
    "README.md가 이미 존재합니다. 덮어쓰지 않습니다."
)

# 3. README 내용 작성
# 내부 코드 블록은 백틱 3개 대신 ~~~를 사용해
# Colab 복사 과정에서 코드가 끊기지 않도록 처리
readme = r"""# PinPoint Korean MiniLM Fine-tuning

## 1. 개요

PinPoint의 한국어 자막 의미 검색 성능을 개선하기 위해
다국어 Sentence-Transformer 모델을 추가 학습한 실험입니다.

- Base model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- Dataset: AI-Hub 음식·요리 분야 한국어 영상 라벨 데이터
- Learning task: 한국어 영상 요약문과 전체 음성 전사문의 의미 정렬
- Loss: `MultipleNegativesRankingLoss`

이 모델은 영상의 시간 구간을 직접 예측하는 모델이 아닙니다.
PinPoint의 자막 기반 검색에 사용할 텍스트 임베딩을 생성합니다.

## 2. 파일 구성

| 파일 | 역할 |
| --- | --- |
| `prepare_data.py` | AI-Hub 원본 라벨 데이터에서 학습 쌍 생성 |
| `train.py` | MiniLM 대조학습 |
| `evaluate.py` | 기본 모델과 Fine-tuned 모델의 검색 성능 비교 |
| `results/full_validation_metrics.json` | 전체 검증 데이터 평가 결과 |

## 3. 데이터 전처리

AI-Hub Training과 Validation 원본을 별도로 처리합니다.

- Training: `TL_12*.tar`
- Validation: `VL_12*.tar`
- TAR 내부 ZIP에 저장된 JSON 라벨 사용
- `summary`를 검색 문장(query)으로 사용
- `video.term[].transcription`을 결합해 positive 문장 생성
- 각각 128토큰을 초과하는 문장 쌍 제외
- Training에서 반복된 query를 가진 쌍 제거
- Validation에서 중복 의심 클립 `MYL_18127`, `MYL_18128` 제외
- Training과 Validation의 영상 ID 중복 검사

검증된 데이터 수:

| 구분 | 문장 쌍 수 |
| --- | ---: |
| Training (정제 전) | 15,823 |
| Training (최종) | 15,502 |
| Validation | 1,948 |

### 전처리 실행

저장소 루트에서 실행합니다.

~~~bash
python -m ai.experiments.korean_minilm.prepare_data \
  --train-tar /path/to/TL_12_data.tar \
  --val-tar /path/to/VL_12_data.tar \
  --output-dir /path/to/new_prepared_data
~~~

출력 파일이 이미 존재하면 덮어쓰지 않고 중단합니다.

## 4. 학습 방법

기존 실험에 사용한 설정:

| 항목 | 값 |
| --- | --- |
| Base model | paraphrase-multilingual-MiniLM-L12-v2 |
| Epoch | 1 |
| Batch size | 8 |
| Gradient accumulation | 2 |
| Learning rate | 2e-5 |
| Warmup ratio | 0.1 |
| Loss | MultipleNegativesRankingLoss |
| Maximum sequence length | 128 |
| Precision | FP16 |
| Seed / Data seed | 42 / 42 |

GPU가 필요합니다. 필요한 주요 패키지는
`torch`, `sentence-transformers`, `transformers`,
`datasets`, `accelerate`입니다.

### 학습 실행

~~~bash
python -m ai.experiments.korean_minilm.train \
  --train-jsonl /path/to/minilm_train_pairs_clean.jsonl \
  --output-dir /path/to/new_training_output
~~~

최종 모델은 지정한 출력 폴더의 `final_model/`에 저장됩니다.

**주의:** `train.py`는 기존 학습 설정 및 데이터 로드 검증을
통과했지만, 전체 학습을 다시 실행하여 가중치까지
재현했는지는 아직 확인하지 않았습니다.

## 5. 평가 방법

Validation의 요약문과 전사문 각각 1,948개를 임베딩으로
변환한 뒤 전체 코사인 유사도를 계산합니다.

각 요약문과 같은 인덱스의 전사문을 유일한 정답으로 간주하여
Recall@1, Recall@5, MRR을 산출합니다.

### 평가 실행

~~~bash
python -m ai.experiments.korean_minilm.evaluate \
  --val-jsonl /path/to/minilm_val_pairs.jsonl \
  --finetuned-model /path/to/final_model \
  --output /path/to/new_evaluation_result.json
~~~

기존 결과 파일을 덮어쓰지 않도록 새 출력 경로를 지정합니다.

### 전체 Validation 결과

| 지표 | Base MiniLM | Fine-tuned MiniLM |
| --- | ---: | ---: |
| Recall@1 | 0.5103 | 0.7515 |
| Recall@5 | 0.6555 | 0.8588 |
| MRR | 0.5773 | 0.7996 |

2026-10-10에 `evaluate.py`로 전체 Validation 1,948쌍을
다시 평가했으며, 기존에 저장한 여섯 개 지표와
모두 일치하는 것을 확인했습니다.

이 값은 **요약문–전사문 텍스트 검색 성능**입니다.
PinPoint의 실제 영상 구간 검색 성능으로 해석하면 안 됩니다.

동일한 의미를 가진 다른 전사문도 존재할 수 있지만
이 평가에서는 같은 인덱스의 문장 하나만 정답으로 처리합니다.

## 6. PinPoint 검색에 모델 연결

`ai/features/fusion_search.py`는 다음 모델 유형을 지원합니다.

- `model_type="base"`: 기존 사전학습 MiniLM
- `model_type="finetuned"`: 추가 학습된 MiniLM

Fine-tuned 모델을 사용하려면 모델 폴더 경로를 환경변수
`PINPOINT_MINILM_MODEL_PATH`로 지정해야 합니다.

예시:

~~~python
import os
from ai.features.fusion_search import get_semantic_model

os.environ["PINPOINT_MINILM_MODEL_PATH"] = "/path/to/final_model"

model = get_semantic_model(model_type="finetuned")
embedding = model.encode("두부를 칼로 자르는 장면")
~~~

실제 Fusion 검색에서도 두 모델의 로드와 후보 생성이
동작하는 것을 확인했습니다.

단, 기존 Fusion은 Hybrid 후보를 우선 배치하므로
MiniLM 모델 변경이 반드시 Top-1 성능 향상을
의미하는 것은 아닙니다.

## 7. 제한사항 및 관리 원칙

- AI-Hub 원본 TAR 및 생성한 문장 쌍 JSONL은 GitHub에 업로드하지 않습니다.
- Fine-tuned 모델 가중치와 체크포인트 역시 GitHub에 올리지 않습니다.
- 해당 데이터와 모델은 승인된 접근 권한 및 이용 조건에 따라 별도로 관리합니다.
- Fine-tuned 모델 로드 시 토크나이저 regex 경고가 관찰되었습니다. 모델은 로드되지만 해당 경고의 영향은 추가 점검이 필요합니다.
- 실제 한국어 영상 구간 검색 성능은 MiniLM 텍스트 검색 평가와 별도로 검증해야 합니다.
- 모델 파일이 저장소에 포함되지 않으므로, 팀원은 권한에 맞게 모델을 별도로 전달받아 설정해야 합니다.
"""

# 4. 새 README 파일만 생성 (기존 파일 덮어쓰기 금지)
with target.open("x", encoding="utf-8") as f:
    f.write(readme)

# 5. 저장 내용 검증
saved = target.read_text(encoding="utf-8")
assert saved == readme

print("===== 28단계. MiniLM README 작성 =====")
print("✅ README.md 생성 완료!")
print("경로:", target)
print("파일 크기:", target.stat().st_size, "bytes")
print("✅ 기존 팀원 코드 변경 없음")
print("✅ 원본 데이터 및 모델 변경 없음")
print("✅ GitHub 업로드 없음")

===== 28단계. MiniLM README 작성 =====
✅ README.md 생성 완료!
경로: /content/pinpoint-capstone/ai/experiments/korean_minilm/README.md
파일 크기: 5919 bytes
✅ 기존 팀원 코드 변경 없음
✅ 원본 데이터 및 모델 변경 없음
✅ GitHub 업로드 없음


In [283]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

files = [
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
    "ai/experiments/korean_minilm/results/full_validation_metrics.json",
]

print("===== 29단계. AI 코드 Git 안전 점검 =====")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

print("\n현재 브랜치:", branch)

assert branch == "feature/korean-ai-fusion", (
    "예상한 AI 전용 브랜치가 아닙니다."
)

print("\n===== 파일별 상태 =====")

for relative_path in files:
    path = repo / relative_path

    print(f"\n[{relative_path}]")

    if not path.is_file():
        print("❌ 파일 없음")
        continue

    print("✅ 파일 존재")

    tracked = subprocess.run(
        ["git", "ls-files", "--error-unmatch", "--", relative_path],
        cwd=repo,
        capture_output=True,
        text=True
    )

    if tracked.returncode == 0:
        print("Git 상태: 이미 추적 중")
    else:
        ignored = subprocess.run(
            ["git", "check-ignore", "-v", "--", relative_path],
            cwd=repo,
            capture_output=True,
            text=True
        )

        if ignored.returncode == 0:
            print("⚠️ Git에서 무시 중")
            print("무시 규칙:", ignored.stdout.strip())
        else:
            print("✅ 새 파일로 Git 추가 가능")

print("\n===== 현재 Git 변경 사항 =====")

subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n===== .gitignore 파일 존재 여부 =====")
gitignore = repo / ".gitignore"
print("존재:", gitignore.is_file())

print("\n✅ 29단계 점검 완료!")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 29단계. AI 코드 Git 안전 점검 =====

현재 브랜치: feature/korean-ai-fusion

===== 파일별 상태 =====

[ai/features/fusion_search.py]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/features/consensus_rerank.py]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/preprocessing/transcribe_service.py]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/experiments/korean_minilm/prepare_data.py]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/experiments/korean_minilm/train.py]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/experiments/korean_minilm/evaluate.py]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/experiments/korean_minilm/README.md]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

[ai/experiments/korean_minilm/results/full_validation_metrics.json]
✅ 파일 존재
✅ 새 파일로 Git 추가 가능

===== 현재 Git 변경 사항 =====

===== .gitignore 파일 존재 여부 =====
존재: True

✅ 29단계 점검 완료!
✅ 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ 팀원 main 브랜치 변경 없음


In [284]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

print("===== 30단계. Git 저장소 정밀 점검 =====")

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True
    )
    print(f"\n$ git {' '.join(args)}")
    print("종료 코드:", result.returncode)
    print(result.stdout.strip() or "(출력 없음)")
    if result.stderr.strip():
        print("오류:", result.stderr.strip())
    return result

# 1. 실제 Git 저장소 루트 확인
run_git("rev-parse", "--show-toplevel")

# 2. 현재 브랜치 및 HEAD 확인
branch = run_git("branch", "--show-current")
assert branch.stdout.strip() == "feature/korean-ai-fusion"

run_git("log", "-1", "--oneline")

# 3. Git의 전체 변경 상태 확인
run_git(
    "status",
    "--porcelain=v1",
    "--untracked-files=all"
)

# 4. untracked 파일 숨김 설정 확인
run_git("config", "--show-origin", "--get", "status.showUntrackedFiles")

# 5. 대표 파일이 실제 Git 인덱스에 존재하는지 확인
run_git(
    "ls-files",
    "--stage",
    "--",
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/experiments/korean_minilm/train.py"
)

# 6. 새 파일이 Git에서 무시되는지 강제로 확인
run_git(
    "check-ignore",
    "-v",
    "--no-index",
    "ai/features/fusion_search.py",
    "ai/experiments/korean_minilm/train.py"
)

print("\n✅ 30단계 정밀 점검 완료!")
print("✅ 파일 수정 및 삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 30단계. Git 저장소 정밀 점검 =====

$ git rev-parse --show-toplevel
종료 코드: 0
/content/pinpoint-capstone

$ git branch --show-current
종료 코드: 0
feature/korean-ai-fusion

$ git log -1 --oneline
종료 코드: 0
14b3eb9 fix: include CLIP tokenizer vocabulary

$ git status --porcelain=v1 --untracked-files=all
종료 코드: 0
?? ai/experiments/korean_minilm/README.md
?? ai/experiments/korean_minilm/evaluate.py
?? ai/experiments/korean_minilm/prepare_data.py
?? ai/experiments/korean_minilm/results/full_validation_metrics.json
?? ai/experiments/korean_minilm/train.py
?? ai/features/consensus_rerank.py
?? ai/features/fusion_search.py
?? ai/features/fusion_search.py.bak
?? ai/features/run_moment_detr_inference_before_fusion.py
?? ai/preprocessing/transcribe_service.py
?? backend/services/indexer.py
?? backend/services/search_api.py

$ git config --show-origin --get status.showUntrackedFiles
종료 코드: 1
(출력 없음)

$ git ls-files --stage -- ai/features/fusion_search.py ai/features/consensus_rerank.py ai/experiments/kore

In [285]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

drive_dirs = [
    Path(
        "/content/drive/MyDrive/pinpoint_capstone/"
        "02_Korean_Video_Test"
    ),
    Path(
        "/content/drive/MyDrive/pinpoint_capstone/"
        "03_Korean_Training_Data"
    )
]

print("===== 31단계. AI 코드 누락 여부 확인 =====")

# 현재 브랜치 보호
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

print("\n===== 1. 저장소 AI Python 파일 =====")

ai_dir = repo / "ai"

for path in sorted(ai_dir.rglob("*.py")):
    if ".venv" in path.parts or "__pycache__" in path.parts:
        continue

    print(path.relative_to(repo))

print("\n===== 2. Google Drive 실험 Python 파일 =====")

for folder in drive_dirs:
    print(f"\n[{folder.name}]")

    if not folder.is_dir():
        print("⚠️ 폴더 없음")
        continue

    python_files = sorted(folder.rglob("*.py"))

    for path in python_files:
        if "__pycache__" in path.parts:
            continue

        print(path.relative_to(folder))

print("\n===== 3. 저장소 실험 결과 파일 =====")

experiments_dir = repo / "ai/experiments"

for path in sorted(experiments_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(repo))

print("\n✅ 31단계 점검 완료!")
print("✅ 파일 생성·수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 코드 변경 없음")

===== 31단계. AI 코드 누락 여부 확인 =====

===== 1. 저장소 AI Python 파일 =====
ai/experiments/korean_minilm/evaluate.py
ai/experiments/korean_minilm/prepare_data.py
ai/experiments/korean_minilm/train.py
ai/features/build_moment_detr_inputs.py
ai/features/check_features.py
ai/features/compare_official_inference.py
ai/features/consensus_rerank.py
ai/features/extract_query_features.py
ai/features/fusion_search.py
ai/features/load_moment_detr_model.py
ai/features/moment_detr_features.py
ai/features/prepare_moment_detr_input.py
ai/features/run_moment_detr_inference.py
ai/features/run_moment_detr_inference_before_fusion.py
ai/preprocessing/align_transcript.py
ai/preprocessing/build_video_data.py
ai/preprocessing/extract_audio.py
ai/preprocessing/extract_frames.py
ai/preprocessing/parse_subtitle.py
ai/preprocessing/pipeline.py
ai/preprocessing/segment_video.py
ai/preprocessing/transcribe.py
ai/preprocessing/transcribe_service.py
ai/preprocessing/video_metadata.py
ai/vendor/moment_detr/moment_detr/__init__

In [286]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion", (
    "AI 전용 브랜치가 아닙니다. 점검을 중단합니다."
)

files = {
    "Fusion 검색": repo / "ai/features/fusion_search.py",
    "Consensus 재정렬": repo / "ai/features/consensus_rerank.py",
    "Whisper 서비스": repo / "ai/preprocessing/transcribe_service.py",
    "기존 Moment-DETR 추론": repo / "ai/features/run_moment_detr_inference.py",
    "기존 영상 전처리": repo / "ai/preprocessing/pipeline.py",
    "Drive 실험 코드": Path(
        "/content/drive/MyDrive/pinpoint_capstone/"
        "02_Korean_Video_Test/fusion_experiment.py"
    ),
}

print("===== 32단계. AI 코드 의존성 확인 =====")

for label, path in files.items():
    print("\n" + "=" * 60)
    print(label)
    print("파일:", path)

    if not path.is_file():
        print("⚠️ 파일이 없습니다.")
        continue

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    functions = []
    imports = []

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append(node.name)

        elif isinstance(node, ast.Import):
            imports.extend(alias.name for alias in node.names)

        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""
            names = ", ".join(alias.name for alias in node.names)
            imports.append(f"from {module} import {names}")

    print("함수 목록:")
    for name in functions:
        print(" -", name)

    print("상위 import 목록:")
    for item in imports:
        print(" -", item)

    print("파일 크기:", path.stat().st_size, "bytes")

print("\n===== Git 상태 확인 =====")
result = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout or "(변경 사항 없음)")

print("\n✅ 32단계 코드 의존성 확인 완료!")
print("✅ 파일 생성·수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 기존 팀원 코드 변경 없음")

===== 32단계. AI 코드 의존성 확인 =====

Fusion 검색
파일: /content/pinpoint-capstone/ai/features/fusion_search.py
함수 목록:
 - get_semantic_model
 - temporal_iou
 - get_semantic_subtitle_candidates
 - select_subtitle_query
 - get_multilingual_subtitle_candidates
 - deduplicate_multilingual_candidates
 - make_fusion_result
 - predict_moments_fusion
상위 import 목록:
 - from pathlib import Path
 - from functools import lru_cache
 - from sentence_transformers import SentenceTransformer
 - from sentence_transformers import util
 - from ai.features.run_moment_detr_inference import predict_moments_hybrid
파일 크기: 6859 bytes

Consensus 재정렬
파일: /content/pinpoint-capstone/ai/features/consensus_rerank.py
함수 목록:
 - _rank_by_consensus
 - rank_by_consensus
 - rank_by_consensus_v2
상위 import 목록:
 - from ai.features.fusion_search import temporal_iou
파일 크기: 1745 bytes

Whisper 서비스
파일: /content/pinpoint-capstone/ai/preprocessing/transcribe_service.py
함수 목록:
 - get_whisper_model
 - transcribe_video
상위 import 목록:
 - json
 - f

In [287]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion", (
    "AI 전용 브랜치가 아닙니다."
)

files = {
    "Whisper 서비스": repo / "ai/preprocessing/transcribe_service.py",
    "Fusion 검색": repo / "ai/features/fusion_search.py",
    "기존 자막 파싱": repo / "ai/preprocessing/parse_subtitle.py",
}

print("===== 33단계. Whisper-Fusion 입출력 확인 =====")

for label, path in files.items():

    print("\n" + "=" * 60)
    print(label)
    print("경로:", path)

    assert path.is_file(), f"파일이 없습니다: {path}"

    code = path.read_text(encoding="utf-8")
    tree = ast.parse(code)

    if label == "Whisper 서비스":
        function_names = {
            "get_whisper_model",
            "transcribe_video"
        }

    elif label == "Fusion 검색":
        function_names = {
            "get_semantic_subtitle_candidates",
            "get_multilingual_subtitle_candidates",
            "predict_moments_fusion"
        }

    else:
        function_names = {
            node.name
            for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        }

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name not in function_names:
                continue

            print(f"\n--- 함수: {node.name} ---")

            start = node.lineno - 1
            end = node.end_lineno

            lines = code.splitlines()
            print("\n".join(lines[start:end]))

    if label == "기존 자막 파싱" and not function_names:
        print("\n--- 파일 전체 ---")
        print(code[:5000])

print("\n✅ 33단계 확인 완료!")
print("✅ 파일 생성·수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 기존 팀원 코드 변경 없음")

===== 33단계. Whisper-Fusion 입출력 확인 =====

Whisper 서비스
경로: /content/pinpoint-capstone/ai/preprocessing/transcribe_service.py

--- 함수: get_whisper_model ---
def get_whisper_model():
    return WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )

--- 함수: transcribe_video ---
def transcribe_video(video_path, video_id):
    output_path = (
        Path("data/processed")
        / video_id
        / "subtitles.json"
    )
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    segments, info = get_whisper_model().transcribe(
        str(video_path),
        beam_size=5
    )

    subtitles = [
        {
            "start_time": round(s.start, 2),
            "end_time": round(s.end, 2),
            "text": s.text.strip()
        }
        for s in segments
    ]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            subtitles,
            f,
            ensure_ascii=False,
            indent=2
  

In [288]:
from pathlib import Path
import shutil
import hashlib
import subprocess

repo = Path("/content/pinpoint-capstone")

source = repo / "ai/preprocessing/transcribe_service.py"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "transcribe_service_before_path_fix_20261010.py"
)

print("===== 34단계. Whisper 수정 전 백업 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인")

# 2. 원본 파일 확인
assert source.is_file(), "Whisper 서비스 파일이 없습니다."

# 3. 기존 백업 덮어쓰기 금지
assert not backup.exists(), (
    "백업 파일이 이미 존재합니다. 덮어쓰지 않습니다."
)

# 4. Google Drive에 원본 백업
backup.parent.mkdir(parents=True, exist_ok=True)

with source.open("rb") as src, backup.open("xb") as dst:
    shutil.copyfileobj(src, dst)

# 5. 원본과 백업 동일 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(source) == sha256(backup), (
    "원본과 백업이 일치하지 않습니다."
)

print("\n✅ Whisper 원본 코드 백업 완료!")
print("원본:", source)
print("백업:", backup)
print("원본과 백업 동일: True")

print("\n✅ 원본 코드 수정 없음")
print("✅ 기존 팀원 코드 수정 없음")
print("✅ Git add/commit/push 없음")

===== 34단계. Whisper 수정 전 백업 =====
✅ AI 전용 브랜치 확인

✅ Whisper 원본 코드 백업 완료!
원본: /content/pinpoint-capstone/ai/preprocessing/transcribe_service.py
백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/transcribe_service_before_path_fix_20261010.py
원본과 백업 동일: True

✅ 원본 코드 수정 없음
✅ 기존 팀원 코드 수정 없음
✅ Git add/commit/push 없음


In [289]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")
target = repo / "ai/preprocessing/transcribe_service.py"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "transcribe_service_before_path_fix_20261010.py"
)

print("===== 35단계. Whisper 서비스 안전 수정 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion", (
    "AI 전용 브랜치가 아닙니다. 수정을 중단합니다."
)

# 2. 원본 및 백업 확인
assert target.is_file(), "Whisper 서비스 원본 파일이 없습니다."
assert backup.is_file(), "Whisper 백업 파일이 없습니다."

# 이전 단계에서 백업한 코드와 현재 코드가 같은지 확인
original = backup.read_text(encoding="utf-8")
current = target.read_text(encoding="utf-8")

assert current == original, (
    "⚠️ 백업 이후 Whisper 코드가 변경됐습니다. "
    "덮어쓰지 않고 중단합니다."
)

# 3. 개선된 Whisper 서비스 코드
updated = '''"""
PinPoint Whisper transcription service.

Converts uploaded video speech into timestamped subtitle JSON.
Existing subtitle files are reused without overwriting them.
"""

import json
from pathlib import Path
from functools import lru_cache

from faster_whisper import WhisperModel


# Always resolve paths relative to the PinPoint repository.
PROJECT_ROOT = Path(__file__).resolve().parents[2]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


@lru_cache(maxsize=1)
def get_whisper_model():
    return WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )


def transcribe_video(video_path, video_id):
    """
    Generate or reuse subtitles for a video.

    Args:
        video_path: Existing input video path.
        video_id: Video identifier used for the output directory.

    Returns:
        Absolute path to data/processed/{video_id}/subtitles.json.
    """

    video_path = Path(video_path).resolve()

    if not video_path.is_file():
        raise FileNotFoundError(
            f"Input video not found: {video_path}"
        )

    video_id = str(video_id)

    # Prevent paths such as ../ from escaping the processed directory.
    if (
        not video_id
        or video_id in {".", ".."}
        or Path(video_id).name != video_id
        or "/" in video_id
        or "\\\\" in video_id
    ):
        raise ValueError(
            f"Invalid video_id: {video_id!r}"
        )

    output_path = (
        PROCESSED_DIR
        / video_id
        / "subtitles.json"
    )

    # Reuse existing subtitle data without running Whisper again.
    if output_path.is_file():
        return str(output_path)

    segments, info = get_whisper_model().transcribe(
        str(video_path),
        beam_size=5
    )

    subtitles = [
        {
            "start_time": round(s.start, 2),
            "end_time": round(s.end, 2),
            "text": s.text.strip()
        }
        for s in segments
    ]

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Exclusive creation protects previously generated subtitles.
    with output_path.open("x", encoding="utf-8") as f:
        json.dump(
            subtitles,
            f,
            ensure_ascii=False,
            indent=2
        )

    return str(output_path)
'''

# 4. 수정 코드 문법 확인
ast.parse(updated)
compile(updated, str(target), "exec")

# 기존 백업과의 일치 확인 후 이 파일 하나만 수정
target.write_text(updated, encoding="utf-8")

# 5. 수정 결과 확인
saved = target.read_text(encoding="utf-8")
assert saved == updated

print("✅ Whisper 서비스 수정 완료!")
print("파일:", target)
print("✅ Python 문법 검사 통과")
print("✅ 저장 경로를 PinPoint 저장소 기준으로 고정")
print("✅ 기존 subtitles.json 덮어쓰기 방지")
print("✅ transcribe_video(video_path, video_id) 유지")
print("✅ 팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 35단계. Whisper 서비스 안전 수정 =====
✅ Whisper 서비스 수정 완료!
파일: /content/pinpoint-capstone/ai/preprocessing/transcribe_service.py
✅ Python 문법 검사 통과
✅ 저장 경로를 PinPoint 저장소 기준으로 고정
✅ 기존 subtitles.json 덮어쓰기 방지
✅ transcribe_video(video_path, video_id) 유지
✅ 팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [290]:
from pathlib import Path
from tempfile import TemporaryDirectory
from types import SimpleNamespace
from unittest.mock import patch
import importlib
import json
import subprocess

from ai.preprocessing import transcribe_service

repo = Path("/content/pinpoint-capstone")

print("===== 36단계. Whisper 서비스 안전 테스트 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인")

# 2. 수정된 코드 다시 불러오기
importlib.reload(transcribe_service)

original_processed_dir = transcribe_service.PROCESSED_DIR

print("실제 프로젝트 저장 경로:", original_processed_dir)

assert original_processed_dir == (
    repo / "data" / "processed"
)

# 3. 임시 폴더에서만 테스트
with TemporaryDirectory(prefix="pinpoint_whisper_test_") as tmp:
    test_root = Path(tmp)
    test_video = test_root / "test.mp4"

    # 실제 영상이 아니라 경로 확인용 임시 파일
    test_video.write_bytes(b"test")

    test_processed = test_root / "processed"

    fake_segments = [
        SimpleNamespace(
            start=1.25,
            end=3.75,
            text="두부를 자르는 장면입니다."
        )
    ]

    class FakeWhisperModel:
        def __init__(self):
            self.call_count = 0

        def transcribe(self, video_path, beam_size=5):
            self.call_count += 1
            return fake_segments, None

    fake_model = FakeWhisperModel()

    # 실제 Whisper와 프로젝트 data/processed 폴더는 사용하지 않음
    with (
        patch.object(
            transcribe_service,
            "PROCESSED_DIR",
            test_processed
        ),
        patch.object(
            transcribe_service,
            "get_whisper_model",
            return_value=fake_model
        )
    ):
        # 첫 번째 호출: 임시 자막 파일 생성
        first_path = Path(
            transcribe_service.transcribe_video(
                test_video,
                "test_video_001"
            )
        )

        assert first_path.is_file()

        subtitles = json.loads(
            first_path.read_text(encoding="utf-8")
        )

        assert subtitles == [
            {
                "start_time": 1.25,
                "end_time": 3.75,
                "text": "두부를 자르는 장면입니다."
            }
        ]

        assert fake_model.call_count == 1

        print("\n✅ 첫 번째 호출: 자막 JSON 생성 성공")
        print("자막:", subtitles)

        # 두 번째 호출: 기존 자막을 재사용해야 함
        second_path = Path(
            transcribe_service.transcribe_video(
                test_video,
                "test_video_001"
            )
        )

        assert second_path == first_path
        assert fake_model.call_count == 1

        print("✅ 두 번째 호출: 기존 자막 재사용 성공")
        print("✅ Whisper 중복 실행 없음")

        # 위험한 video_id 거부 여부 확인
        try:
            transcribe_service.transcribe_video(
                test_video,
                "../outside"
            )
        except ValueError:
            print("✅ 잘못된 video_id 차단 성공")
        else:
            raise AssertionError(
                "잘못된 video_id가 차단되지 않았습니다."
            )

print("\n✅ 36단계 안전 테스트 완료!")
print("✅ 실제 Whisper 모델 실행 없음")
print("✅ 실제 프로젝트 자막 변경 없음")
print("✅ 기존 팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 36단계. Whisper 서비스 안전 테스트 =====
✅ AI 전용 브랜치 확인
실제 프로젝트 저장 경로: /content/pinpoint-capstone/data/processed

✅ 첫 번째 호출: 자막 JSON 생성 성공
자막: [{'start_time': 1.25, 'end_time': 3.75, 'text': '두부를 자르는 장면입니다.'}]
✅ 두 번째 호출: 기존 자막 재사용 성공
✅ Whisper 중복 실행 없음
✅ 잘못된 video_id 차단 성공

✅ 36단계 안전 테스트 완료!
✅ 실제 Whisper 모델 실행 없음
✅ 실제 프로젝트 자막 변경 없음
✅ 기존 팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [291]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")

print("===== 37단계. Fusion-백엔드 연동 형식 확인 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인")

# 2. 확인할 파일
files = {
    "Fusion 검색": repo / "ai/features/fusion_search.py",
    "백엔드 검색 서비스": repo / "backend/services/search_api.py",
    "백엔드 영상 인덱싱": repo / "backend/services/indexer.py",
}

# 3. 함수 이름과 입력 인자 확인
for label, path in files.items():
    print("\n" + "=" * 60)
    print(label)
    print("파일:", path)

    if not path.is_file():
        print("⚠️ 파일 없음")
        continue

    code = path.read_text(encoding="utf-8")
    tree = ast.parse(code)

    print("\n함수 목록 및 입력값:")

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = [arg.arg for arg in node.args.args]

            print(f" - {node.name}({', '.join(args)})")

    # Fusion은 결과 구성 함수의 실제 코드 확인
    if label == "Fusion 검색":
        targets = {
            "make_fusion_result",
            "predict_moments_fusion"
        }
    else:
        # 백엔드는 전체 코드 대신 입출력 관련 부분만 확인
        targets = {
            node.name
            for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        }

    lines = code.splitlines()

    for node in tree.body:
        if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            continue

        if node.name not in targets:
            continue

        print(f"\n--- {node.name} ---")

        # 너무 긴 함수는 앞부분 최대 100줄만 출력
        start = node.lineno - 1
        end = min(node.end_lineno, start + 100)

        print("\n".join(lines[start:end]))

print("\n✅ 37단계 확인 완료!")
print("✅ 모든 파일 읽기 전용")
print("✅ 백엔드 코드 수정 없음")
print("✅ Git add/commit/push 없음")

===== 37단계. Fusion-백엔드 연동 형식 확인 =====
✅ AI 전용 브랜치 확인

Fusion 검색
파일: /content/pinpoint-capstone/ai/features/fusion_search.py

함수 목록 및 입력값:
 - get_semantic_model(model_type)
 - temporal_iou(a, b)
 - get_semantic_subtitle_candidates(query, subtitles, model, top_k)
 - select_subtitle_query(subtitles, korean_query, english_query)
 - get_multilingual_subtitle_candidates(korean_query, english_query, subtitles, model, top_k)
 - deduplicate_multilingual_candidates(candidates, iou_threshold, start_gap)
 - make_fusion_result(video_id, query, candidate)
 - predict_moments_fusion(video_id, feature_path, subtitles, korean_query, english_query, video_duration, subtitle_json_path, top_k, model_type)

--- make_fusion_result ---
def make_fusion_result(video_id, query, candidate):
    source = candidate["source"]
    is_hybrid = source in ("hybrid", "visual")

    return {
        "video_id": video_id,
        "query": query,
        "start_time": round(float(candidate["start_time"]), 4),
        "end_ti

In [292]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")

files = [
    repo / "backend/services/search_api.py",
    repo / "ai/features/fusion_search.py",
]

print("===== 38단계. AI-백엔드 응답 형식 점검 =====")

# 1. 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인")

# 2. 요청/응답 관련 코드 확인
keywords = [
    "class SearchRequest",
    "class MultiSearchRequest",
    "results =",
    '"score"',
    "def search_multi",
    "def make_fusion_result",
]

for path in files:
    assert path.is_file(), f"파일 없음: {path}"

    lines = path.read_text(encoding="utf-8").splitlines()

    print("\n" + "=" * 60)
    print("파일:", path.relative_to(repo))

    matched = set()

    for i, line in enumerate(lines):
        if any(keyword in line for keyword in keywords):
            matched.add(i)

    # 같은 코드가 여러 번 출력되지 않도록 인접 구간 병합
    selected = set()

    for i in matched:
        start = max(0, i - 5)
        end = min(len(lines), i + 12)
        selected.update(range(start, end))

    previous = -2

    for i in sorted(selected):
        if i > previous + 1:
            print("\n---")

        print(f"{i + 1:4d}: {lines[i]}")
        previous = i

print("\n✅ 38단계 확인 완료!")
print("✅ 파일 읽기만 수행")
print("✅ 기존 팀원 코드 수정 없음")
print("✅ Git add/commit/push 없음")

===== 38단계. AI-백엔드 응답 형식 점검 =====
✅ AI 전용 브랜치 확인

파일: backend/services/search_api.py

---
  14: 
  15: YOUTUBE_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")
  16: MAX_DURATION_SEC = 30 * 60  # 임시값. 팀 합의 후 조정
  17: 
  18: 
  19: class SearchRequest(BaseModel):
  20:     youtube_video_id: str
  21:     query: str
  22:     top_k: int = 5
  23: 
  24: class MultiSearchRequest(BaseModel):
  25:     query: str
  26:     youtube_video_ids: list[str] | None = None
  27:     top_k_per_video: int = 3
  28:     max_videos: int = 10
  29:     auto_analyze: bool = False      # true면 미분석 영상의 분석을 백그라운드로 시작
  30:     max_new_analyses: int = 3       # 한 번의 요청으로 새로 시작할 최대 영상 수 (최대 5)
  31: 
  32: def _err(status: int, code: str, message: str) -> JSONResponse:
  33:     return JSONResponse(status_code=status,
  34:                         content={"error": {"code": code, "message": message}})
  35: 

---
  49:                 video_id=video_id, video_feature_path=feature_path,
  50:                 query=q

In [293]:
from pathlib import Path
import ast
import subprocess

repo = Path("/content/pinpoint-capstone")

fusion_path = repo / "ai/features/fusion_search.py"
backend_path = repo / "backend/services/search_api.py"

print("===== 39단계. AI-백엔드 결과 형식 안전 검사 =====")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

assert fusion_path.is_file()
assert backend_path.is_file()

# 기존 파일은 읽기만 함
fusion_source = fusion_path.read_text(encoding="utf-8")
backend_source = backend_path.read_text(encoding="utf-8")

fusion_tree = ast.parse(fusion_source)
ast.parse(backend_source)

result_function = next(
    node for node in fusion_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "make_fusion_result"
)

print("\n===== Fusion 반환 형식 확인 =====")

lines = fusion_source.splitlines()

print("\n".join(
    lines[result_function.lineno - 1:result_function.end_lineno]
))

# 실제 Fusion 결과 생성 함수만 독립적으로 확인
from ai.features.fusion_search import make_fusion_result

hybrid_candidate = {
    "start_time": 10.0,
    "end_time": 20.0,
    "score": 0.8,
    "source": "hybrid"
}

subtitle_candidate = {
    "start_time": 30.0,
    "end_time": 35.0,
    "score": 0.9,
    "source": "subtitle",
    "language": "ko"
}

hybrid_result = make_fusion_result(
    "test_video", "두부를 자르는 장면", hybrid_candidate
)

subtitle_result = make_fusion_result(
    "test_video", "두부를 자르는 장면", subtitle_candidate
)

print("\n===== Hybrid 결과 =====")
print(hybrid_result)

print("\n===== 자막 결과 =====")
print(subtitle_result)

print("\n===== 백엔드 요구 필드 검사 =====")

required = ["start_time", "end_time", "score"]

for name, result in [
    ("Hybrid", hybrid_result),
    ("Subtitle", subtitle_result)
]:
    missing = [key for key in required if key not in result]

    numeric_score = (
        isinstance(result.get("score"), (int, float))
        and not isinstance(result.get("score"), bool)
    )

    print(f"\n[{name}]")
    print("필수 필드 누락:", missing)
    print("score 숫자 여부:", numeric_score)
    print("source:", result.get("source"))
    print("subtitle_score:", result.get("subtitle_score"))

print("\n===== 검사 결론 =====")

if subtitle_result["score"] is None:
    print("⚠️ 자막 후보의 score는 None입니다.")
    print("⚠️ 현재 백엔드 multi-search 점수 비교와 바로 호환되지 않습니다.")

print("\n✅ 39단계 검사 완료!")
print("✅ 모델 추론 및 학습 실행 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 백엔드 코드 변경 없음")

===== 39단계. AI-백엔드 결과 형식 안전 검사 =====

===== Fusion 반환 형식 확인 =====
def make_fusion_result(video_id, query, candidate):
    source = candidate["source"]
    is_hybrid = source in ("hybrid", "visual")

    return {
        "video_id": video_id,
        "query": query,
        "start_time": round(float(candidate["start_time"]), 4),
        "end_time": round(float(candidate["end_time"]), 4),
        "score": candidate.get("score") if is_hybrid else None,
        "source": source,
        "hybrid_source": candidate.get("hybrid_source"),
        "visual_score": candidate.get("score") if is_hybrid else None,
        "subtitle_score": (
            candidate.get("subtitle_score")
            if is_hybrid
            else candidate.get("score")
        ),
        "language": candidate.get("language")
    }

===== Hybrid 결과 =====
{'video_id': 'test_video', 'query': '두부를 자르는 장면', 'start_time': 10.0, 'end_time': 20.0, 'score': 0.8, 'source': 'hybrid', 'hybrid_source': None, 'visual_score': 0.8, 'su

In [294]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

print("===== 40단계. Fusion 연동 입력값 확인 =====")

files = {
    "백엔드 검색 API": repo / "backend/services/search_api.py",
    "백엔드 인덱싱": repo / "backend/services/indexer.py",
    "Fusion 검색": repo / "ai/features/fusion_search.py",
    "Moment-DETR 추론": repo / "ai/features/run_moment_detr_inference.py",
}

keywords = [
    "duration",
    "translate",
    "translation",
    "english_query",
    "korean_query",
    "query_en",
    "query_ko",
    "resolve_subtitle_path",
    "subtitle_path",
    "video_metadata",
]

for label, path in files.items():
    print("\n" + "=" * 60)
    print(label)
    print("경로:", path.relative_to(repo))

    assert path.is_file(), f"파일이 없습니다: {path}"

    lines = path.read_text(encoding="utf-8").splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    if not matches:
        print("관련 코드 없음")
        continue

    shown = set()

    for i in matches:
        for j in range(max(0, i - 2), min(len(lines), i + 3)):
            shown.add(j)

    for i in sorted(shown):
        print(f"{i + 1:4d}: {lines[i]}")

print("\n===== 관련 패키지 확인 =====")

requirements = repo / "requirements.txt"

if requirements.is_file():
    lines = requirements.read_text(encoding="utf-8").splitlines()

    for line in lines:
        if any(
            word in line.lower()
            for word in [
                "sentence-transformers",
                "faster-whisper",
                "langdetect",
                "translate",
                "transformers",
            ]
        ):
            print(line)

print("\n✅ 40단계 확인 완료!")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 백엔드 코드 변경 없음")

===== 40단계. Fusion 연동 입력값 확인 =====

백엔드 검색 API
경로: backend/services/search_api.py
  14: 
  15: YOUTUBE_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")
  16: MAX_DURATION_SEC = 30 * 60  # 임시값. 팀 합의 후 조정
  17: 
  18: 
  69:         if not info:
  70:             return "NOT_FOUND", False
  71:         if info["duration"] and info["duration"] > MAX_DURATION_SEC:
  72:             return "TOO_LONG", False
  73:         title = info["title"]
 140:             if not info:
 141:                 return _err(404, "VIDEO_NOT_FOUND", "유튜브에서 영상을 찾을 수 없습니다.")
 142:             if info["duration"] and info["duration"] > MAX_DURATION_SEC:
 143:                 return _err(422, "VIDEO_TOO_LONG",
 144:                             f"{MAX_DURATION_SEC // 60}분 이하 영상만 분석할 수 있습니다.")
 145:             title = info["title"]
 146:             meta = {"channel_name": info["channel_name"],

백엔드 인덱싱
경로: backend/services/indexer.py
  72: 
  73: 
  74: def resolve_subtitle_path(youtube_video_id: str):
  75:     """자막이 

In [5]:
from google.colab import drive
from pathlib import Path

# Google Drive 다시 연결
drive.mount("/content/drive")

# 기존 PinPoint 자료 확인
project_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone"
)

print("\n===== PinPoint Google Drive 복구 확인 =====")
print("프로젝트 폴더 존재:", project_dir.is_dir())

if project_dir.is_dir():
    for path in sorted(project_dir.iterdir()):
        print(" -", path.name)

model_dir = (
    project_dir
    / "03_Korean_Training_Data/minilm_finetuned/final_model"
)

backup_dir = (
    project_dir
    / "02_Korean_Video_Test/AI_Backup"
)

print("\nFine-tuned MiniLM 모델:", "✅ 존재" if model_dir.is_dir() else "❌ 없음")
print("AI 코드 백업:", "✅ 존재" if backup_dir.is_dir() else "❌ 없음")

print("\n✅ Drive 연결 및 파일 확인 완료!")

Mounted at /content/drive

===== PinPoint Google Drive 복구 확인 =====
프로젝트 폴더 존재: True
 - 01_AI_인수인계
 - 02_Korean_Video_Test
 - 03_Korean_Training_Data
 - data
 - notebooks

Fine-tuned MiniLM 모델: ✅ 존재
AI 코드 백업: ✅ 존재

✅ Drive 연결 및 파일 확인 완료!


In [6]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")
repo_url = "https://github.com/pinpoint-capstone/pinpoint-capstone.git"
branch_name = "feature/korean-ai-fusion"

print("===== 41-3단계. GitHub 프로젝트 복구 =====")

# 1. 기존 폴더 보호
assert not repo.exists(), (
    "프로젝트 폴더가 이미 존재합니다. "
    "덮어쓰지 않고 중단합니다."
)

# 2. GitHub에서 기존 코드 가져오기
subprocess.run(
    ["git", "clone", repo_url, str(repo)],
    check=True
)

print("\n✅ GitHub 프로젝트 Clone 완료")

# 3. 로컬 AI 작업 브랜치 생성
# 원격에 동일한 브랜치가 이미 있는 경우에는 중단
remote_branches = subprocess.check_output(
    ["git", "branch", "-r"],
    cwd=repo,
    text=True
)

assert f"origin/{branch_name}" not in remote_branches, (
    "원격에 같은 이름의 브랜치가 있습니다. "
    "기존 작업을 확인해야 합니다."
)

subprocess.run(
    ["git", "switch", "-c", branch_name],
    cwd=repo,
    check=True
)

# 4. 결과 확인
current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert current_branch == branch_name

print("\n===== 복구 결과 =====")
print("프로젝트 경로:", repo)
print("현재 브랜치:", current_branch)

subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True
)

print("\n✅ 41-3단계 완료!")
print("✅ 팀원 main 브랜치 수정 없음")
print("✅ GitHub Push 없음")
print("✅ Google Drive 데이터 및 모델 변경 없음")

===== 41-3단계. GitHub 프로젝트 복구 =====

✅ GitHub 프로젝트 Clone 완료

===== 복구 결과 =====
프로젝트 경로: /content/pinpoint-capstone
현재 브랜치: feature/korean-ai-fusion

✅ 41-3단계 완료!
✅ 팀원 main 브랜치 수정 없음
✅ GitHub Push 없음
✅ Google Drive 데이터 및 모델 변경 없음


In [7]:
from pathlib import Path
import subprocess
import ast
import hashlib

repo = Path("/content/pinpoint-capstone")

backup_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup"
)

print("===== 41-4단계. AI 코드 백업 상태 확인 =====")

# 1. 안전 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert backup_dir.is_dir()

print("✅ AI 전용 브랜치 확인:", branch)

# 2. Google Drive 백업 확인
print("\n===== Google Drive AI 백업 =====")

for path in sorted(backup_dir.glob("*.py")):
    source = path.read_text(encoding="utf-8")

    try:
        tree = ast.parse(source)
        functions = [
            node.name
            for node in tree.body
            if isinstance(node, ast.FunctionDef)
        ]
        syntax = "정상"
    except SyntaxError:
        functions = []
        syntax = "오류"

    print(f"\n파일: {path.name}")
    print("크기:", path.stat().st_size, "bytes")
    print("Python 문법:", syntax)
    print("함수:", functions)
    print(
        "MiniLM 모델 선택 기능:",
        "model_type" in source
    )

# 3. 복구된 GitHub 저장소 상태
print("\n===== 현재 저장소 AI 파일 =====")

targets = [
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
]

for relative in targets:
    path = repo / relative
    print(
        relative,
        "✅ 존재" if path.is_file() else "❌ 복구 필요"
    )

print("\n===== Git 변경 사항 =====")
subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n✅ 41-4단계 확인 완료!")
print("✅ 파일 복사·수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-4단계. AI 코드 백업 상태 확인 =====
✅ AI 전용 브랜치 확인: feature/korean-ai-fusion

===== Google Drive AI 백업 =====

파일: consensus_rerank.py
크기: 1745 bytes
Python 문법: 정상
함수: ['_rank_by_consensus', 'rank_by_consensus', 'rank_by_consensus_v2']
MiniLM 모델 선택 기능: False

파일: fusion_search_20261010.py
크기: 5965 bytes
Python 문법: 정상
함수: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_multilingual_subtitle_candidates', 'deduplicate_multilingual_candidates', 'make_fusion_result', 'predict_moments_fusion']
MiniLM 모델 선택 기능: False

파일: fusion_search_before_model_connection_20261010_130236.py
크기: 6790 bytes
Python 문법: 정상
함수: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_multilingual_subtitle_candidates', 'deduplicate_multilingual_candidates', 'make_fusion_result', 'predict_moments_fusion']
MiniLM 모델 선택 기능: True

파일: fusion_search_before_model_selection_20261010_130203.py
크기: 5965 bytes
Python 문법

In [8]:
from pathlib import Path
import subprocess
import shutil
import hashlib
import ast

repo = Path("/content/pinpoint-capstone")

source = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/consensus_rerank.py"
)

target = repo / "ai/features/consensus_rerank.py"

print("===== 41-5단계. Consensus 안전 복구 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion", (
    "예상한 브랜치가 아닙니다."
)

# 2. 백업 파일 존재 확인
assert source.is_file(), "Google Drive 백업 파일이 없습니다."

# 3. 기존 파일 보호
assert not target.exists(), (
    "대상 파일이 이미 존재합니다. 덮어쓰지 않습니다."
)

# 4. 백업 코드 문법 검사
code = source.read_text(encoding="utf-8")
ast.parse(code)
compile(code, str(target), "exec")

# 5. 새 파일로 복구
target.parent.mkdir(parents=True, exist_ok=True)

with source.open("rb") as src, target.open("xb") as dst:
    shutil.copyfileobj(src, dst)

# 6. 원본과 복구 파일 동일 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(source) == sha256(target)

print("✅ Consensus V1·V2 복구 완료!")
print("복구 경로:", target)
print("✅ Google Drive 백업과 내용 완전히 일치")
print("✅ Python 문법 검사 통과")
print("✅ 기존 파일 덮어쓰기 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-5단계. Consensus 안전 복구 =====
✅ Consensus V1·V2 복구 완료!
복구 경로: /content/pinpoint-capstone/ai/features/consensus_rerank.py
✅ Google Drive 백업과 내용 완전히 일치
✅ Python 문법 검사 통과
✅ 기존 파일 덮어쓰기 없음
✅ Git add/commit/push 없음
✅ 팀원 main 브랜치 변경 없음


In [9]:
from pathlib import Path
import subprocess
import ast
import hashlib
import re

repo = Path("/content/pinpoint-capstone")

source = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "fusion_search_before_model_connection_20261010_130236.py"
)

target = repo / "ai/features/fusion_search.py"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "fusion_search_FINAL_restored_20261010.py"
)

print("===== 41-6단계. Fusion 검색 코드 복구 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 기존 파일 보호
assert source.is_file(), "Fusion 백업 파일이 없습니다."
assert not target.exists(), "Fusion 대상 파일이 이미 존재합니다."
assert not backup.exists(), "최종 백업 파일이 이미 존재합니다."

# 3. 이전 백업 코드 확인
code = source.read_text(encoding="utf-8")
tree = ast.parse(code)

functions = {
    node.name: node
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

assert "get_semantic_model" in functions
assert "predict_moments_fusion" in functions

# 4. Path import 확인 및 추가
has_path_import = any(
    isinstance(node, ast.ImportFrom)
    and node.module == "pathlib"
    and any(alias.name == "Path" for alias in node.names)
    for node in tree.body
)

if not has_path_import:
    code = "from pathlib import Path\n" + code

# 5. Fusion 함수의 기존 시그니처를 AST로 확인
tree = ast.parse(code)

fusion_func = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "predict_moments_fusion"
)

args = [arg.arg for arg in fusion_func.args.args]

expected_prefix = [
    "video_id",
    "feature_path",
    "subtitles",
    "korean_query",
    "english_query",
    "video_duration",
    "subtitle_json_path",
    "top_k",
]

assert args == expected_prefix, (
    f"예상과 다른 Fusion 함수 인자입니다: {args}"
)

# 6. Fusion 함수에 model_type 인자 추가
signature = (
    "    video_duration, subtitle_json_path=None, top_k=5\n"
    "):"
)

replacement = (
    "    video_duration, subtitle_json_path=None, top_k=5,\n"
    '    model_type="base"\n'
    "):"
)

assert code.count(signature) == 1, (
    "Fusion 함수 시그니처 위치를 정확히 찾지 못했습니다."
)

code = code.replace(signature, replacement, 1)

# 7. Fusion 내부 모델 호출에 model_type 전달
old_call = "model=get_semantic_model(),"
new_call = "model=get_semantic_model(model_type=model_type),"

assert code.count(old_call) == 1, (
    "Fusion 모델 호출 위치를 정확히 찾지 못했습니다."
)

code = code.replace(old_call, new_call, 1)

# 8. 최종 코드 문법 및 함수 인자 검사
compile(code, str(target), "exec")

final_tree = ast.parse(code)

model_func = next(
    node for node in final_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "get_semantic_model"
)

fusion_func = next(
    node for node in final_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "predict_moments_fusion"
)

assert [a.arg for a in model_func.args.args] == ["model_type"]
assert [a.arg for a in fusion_func.args.args][-1] == "model_type"

# 9. 복구 파일과 Drive 백업을 새로 생성
target.parent.mkdir(parents=True, exist_ok=True)

with target.open("x", encoding="utf-8") as f:
    f.write(code)

with backup.open("x", encoding="utf-8") as f:
    f.write(code)

# 10. 두 파일 동일 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\n✅ 최신 Fusion 코드 복구 완료!")
print("복구 파일:", target)
print("Drive 백업:", backup)
print("✅ Path import 확인")
print("✅ MiniLM base/finetuned 모델 선택 기능 확인")
print("✅ Fusion 검색 함수에 model_type 연결")
print("✅ Python 문법 검사 통과")
print("✅ 복구 파일과 Drive 백업 완전히 일치")
print("✅ 기존 팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-6단계. Fusion 검색 코드 복구 =====

✅ 최신 Fusion 코드 복구 완료!
복구 파일: /content/pinpoint-capstone/ai/features/fusion_search.py
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/fusion_search_FINAL_restored_20261010.py
✅ Path import 확인
✅ MiniLM base/finetuned 모델 선택 기능 확인
✅ Fusion 검색 함수에 model_type 연결
✅ Python 문법 검사 통과
✅ 복구 파일과 Drive 백업 완전히 일치
✅ 기존 팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [10]:
import os
import sys
import inspect
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

model_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned/final_model"
)

print("===== 41-7단계. Fusion 모델 선택 기능 검증 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert model_path.is_dir(), "Fine-tuned MiniLM 모델 폴더가 없습니다."

# 2. 프로젝트 import 경로 설정
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# 3. 모델 경로 설정
os.environ["PINPOINT_MINILM_MODEL_PATH"] = str(model_path)

# 4. Fusion 코드 불러오기
from ai.features import fusion_search

print("\n===== 함수 입력값 확인 =====")
print("get_semantic_model:", inspect.signature(fusion_search.get_semantic_model))
print("predict_moments_fusion:", inspect.signature(fusion_search.predict_moments_fusion))

# 5. 기본 모델 확인
print("\n===== 기본 MiniLM =====")

base_model = fusion_search.get_semantic_model(
    model_type="base"
)

base_embedding = base_model.encode(
    "두부를 칼로 자르는 장면"
)

print("✅ 기본 MiniLM 로드 성공")
print("임베딩 차원:", base_embedding.shape)

# 6. Fine-tuned 모델 확인
print("\n===== Fine-tuned MiniLM =====")

finetuned_model = fusion_search.get_semantic_model(
    model_type="finetuned"
)

finetuned_embedding = finetuned_model.encode(
    "두부를 칼로 자르는 장면"
)

print("✅ Fine-tuned MiniLM 로드 성공")
print("임베딩 차원:", finetuned_embedding.shape)

# 7. 모델 선택 결과 확인
assert base_embedding.shape == (384,)
assert finetuned_embedding.shape == (384,)
assert base_model is not finetuned_model

print("\n✅ 두 모델이 서로 다른 객체임을 확인")
print("✅ 41-7단계 검증 완료!")
print("✅ 코드 및 모델 파일 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-7단계. Fusion 모델 선택 기능 검증 =====


ModuleNotFoundError: No module named 'ffmpeg'

In [11]:
import sys
import subprocess

print("===== 41-7-1단계. FFmpeg Python 패키지 설치 =====")

# Colab 런타임에만 패키지를 설치
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "ffmpeg-python"
    ],
    check=True
)

# 실제 import 가능 여부 확인
import ffmpeg

print("\n✅ ffmpeg-python 설치 완료!")
print("✅ import ffmpeg 성공!")
print("✅ 프로젝트 코드 및 모델 파일 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-7-1단계. FFmpeg Python 패키지 설치 =====

✅ ffmpeg-python 설치 완료!
✅ import ffmpeg 성공!
✅ 프로젝트 코드 및 모델 파일 변경 없음
✅ Git add/commit/push 없음


In [12]:
import os
import sys
import inspect
import importlib
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

model_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned/final_model"
)

print("===== 41-7-2단계. Fusion 모델 선택 기능 재검증 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert model_path.is_dir(), "Fine-tuned MiniLM 모델 폴더가 없습니다."

# 2. 프로젝트 경로 및 모델 경로 설정
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["PINPOINT_MINILM_MODEL_PATH"] = str(model_path)

# 3. 복구된 Fusion 코드 불러오기
from ai.features import fusion_search

importlib.reload(fusion_search)

print("\n===== 함수 입력값 확인 =====")
print("get_semantic_model:", inspect.signature(fusion_search.get_semantic_model))
print("predict_moments_fusion:", inspect.signature(fusion_search.predict_moments_fusion))

# 4. 기본 MiniLM 로드
print("\n===== 기본 MiniLM =====")

base_model = fusion_search.get_semantic_model(model_type="base")
base_embedding = base_model.encode("두부를 칼로 자르는 장면")

assert base_embedding.shape == (384,)

print("✅ 기본 MiniLM 로드 성공")
print("임베딩 차원:", base_embedding.shape)

# 5. Fine-tuned MiniLM 로드
print("\n===== Fine-tuned MiniLM =====")

finetuned_model = fusion_search.get_semantic_model(model_type="finetuned")
finetuned_embedding = finetuned_model.encode("두부를 칼로 자르는 장면")

assert finetuned_embedding.shape == (384,)
assert base_model is not finetuned_model

print("✅ Fine-tuned MiniLM 로드 성공")
print("임베딩 차원:", finetuned_embedding.shape)
print("✅ 두 모델이 서로 다른 객체임을 확인")

print("\n✅ 41-7-2단계 검증 완료!")
print("✅ 프로젝트 코드 및 모델 파일 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-7-2단계. Fusion 모델 선택 기능 재검증 =====


ModuleNotFoundError: No module named 'ftfy'

In [13]:
import sys
import subprocess
import importlib

print("===== 41-7-3단계. CLIP 필수 라이브러리 설치 =====")

# 기존 Moment-DETR CLIP 코드에서 사용하는 라이브러리
packages = [
    "ftfy",
    "regex",
    "tqdm"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        *packages
    ],
    check=True
)

print("\n===== 설치 확인 =====")

for module_name in ["ftfy", "regex", "tqdm"]:
    importlib.import_module(module_name)
    print(f"✅ {module_name} import 성공")

print("\n✅ 41-7-3단계 완료!")
print("✅ 프로젝트 코드 및 모델 파일 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-7-3단계. CLIP 필수 라이브러리 설치 =====

===== 설치 확인 =====
✅ ftfy import 성공
✅ regex import 성공
✅ tqdm import 성공

✅ 41-7-3단계 완료!
✅ 프로젝트 코드 및 모델 파일 변경 없음
✅ Git add/commit/push 없음


In [14]:
import os
import sys
import inspect
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

model_path = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/minilm_finetuned/final_model"
)

print("===== 41-7-4단계. Fusion 모델 선택 기능 검증 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert model_path.is_dir(), "Fine-tuned MiniLM 모델 폴더가 없습니다."

# 2. 프로젝트 경로 및 모델 경로 설정
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["PINPOINT_MINILM_MODEL_PATH"] = str(model_path)

# 3. 복구된 Fusion 코드 불러오기
from ai.features import fusion_search

print("\n===== 함수 입력값 확인 =====")
print(
    "get_semantic_model:",
    inspect.signature(fusion_search.get_semantic_model)
)
print(
    "predict_moments_fusion:",
    inspect.signature(fusion_search.predict_moments_fusion)
)

# 4. 기본 MiniLM 테스트
print("\n===== 기본 MiniLM =====")

base_model = fusion_search.get_semantic_model(
    model_type="base"
)

base_embedding = base_model.encode(
    "두부를 칼로 자르는 장면"
)

assert base_embedding.shape == (384,)

print("✅ 기본 MiniLM 로드 성공")
print("임베딩 차원:", base_embedding.shape)

# 5. Fine-tuned MiniLM 테스트
print("\n===== Fine-tuned MiniLM =====")

finetuned_model = fusion_search.get_semantic_model(
    model_type="finetuned"
)

finetuned_embedding = finetuned_model.encode(
    "두부를 칼로 자르는 장면"
)

assert finetuned_embedding.shape == (384,)
assert base_model is not finetuned_model

print("✅ Fine-tuned MiniLM 로드 성공")
print("임베딩 차원:", finetuned_embedding.shape)

print("\n✅ 두 모델 선택 기능 정상 작동!")
print("✅ 41-7-4단계 완료!")
print("✅ 기존 코드 및 모델 파일 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-7-4단계. Fusion 모델 선택 기능 검증 =====

===== 함수 입력값 확인 =====
get_semantic_model: (model_type='base')
predict_moments_fusion: (video_id, feature_path, subtitles, korean_query, english_query, video_duration, subtitle_json_path=None, top_k=5, model_type='base')

===== 기본 MiniLM =====


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 기본 MiniLM 로드 성공
임베딩 차원: (384,)

===== Fine-tuned MiniLM =====


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

✅ Fine-tuned MiniLM 로드 성공
임베딩 차원: (384,)

✅ 두 모델 선택 기능 정상 작동!
✅ 41-7-4단계 완료!
✅ 기존 코드 및 모델 파일 변경 없음
✅ Git add/commit/push 없음


In [15]:
from pathlib import Path
import subprocess
import hashlib
import ast

repo = Path("/content/pinpoint-capstone")

original_backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "transcribe_service_before_path_fix_20261010.py"
)

target = repo / "ai/preprocessing/transcribe_service.py"

final_backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "transcribe_service_FINAL_restored_20261010.py"
)

print("===== 41-8단계. Whisper 서비스 복구 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 기존 파일 보호
assert original_backup.is_file(), "Whisper 기존 백업이 없습니다."
assert not target.exists(), "Whisper 파일이 이미 존재합니다. 덮어쓰지 않습니다."
assert not final_backup.exists(), "최종 백업이 이미 존재합니다."

# 3. 기존 Whisper 함수 구조 확인
original = original_backup.read_text(encoding="utf-8")
tree = ast.parse(original)

original_functions = {
    node.name
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

assert {"get_whisper_model", "transcribe_video"} <= original_functions

# 4. 검증했던 개선 코드 복구
updated = r'''"""
PinPoint Whisper transcription service.

Converts uploaded video speech into timestamped subtitle JSON.
Existing subtitle files are reused without overwriting them.
"""

import json
from pathlib import Path
from functools import lru_cache

from faster_whisper import WhisperModel


PROJECT_ROOT = Path(__file__).resolve().parents[2]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


@lru_cache(maxsize=1)
def get_whisper_model():
    return WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )


def transcribe_video(video_path, video_id):
    """
    Generate or reuse subtitles for a video.

    Returns an absolute path to:
    data/processed/{video_id}/subtitles.json
    """

    video_path = Path(video_path).resolve()

    if not video_path.is_file():
        raise FileNotFoundError(
            f"Input video not found: {video_path}"
        )

    video_id = str(video_id)

    if (
        not video_id
        or video_id in {".", ".."}
        or Path(video_id).name != video_id
        or "/" in video_id
        or "\\" in video_id
    ):
        raise ValueError(
            f"Invalid video_id: {video_id!r}"
        )

    output_path = (
        PROCESSED_DIR
        / video_id
        / "subtitles.json"
    )

    if output_path.is_file():
        return str(output_path)

    segments, info = get_whisper_model().transcribe(
        str(video_path),
        beam_size=5
    )

    subtitles = [
        {
            "start_time": round(s.start, 2),
            "end_time": round(s.end, 2),
            "text": s.text.strip()
        }
        for s in segments
    ]

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with output_path.open("x", encoding="utf-8") as f:
        json.dump(
            subtitles,
            f,
            ensure_ascii=False,
            indent=2
        )

    return str(output_path)
'''

# 5. 문법과 함수 인터페이스 확인
ast.parse(updated)
compile(updated, str(target), "exec")

updated_tree = ast.parse(updated)
functions = {
    node.name: node
    for node in updated_tree.body
    if isinstance(node, ast.FunctionDef)
}

assert {"get_whisper_model", "transcribe_video"} <= set(functions)

args = [
    arg.arg
    for arg in functions["transcribe_video"].args.args
]

assert args == ["video_path", "video_id"]

# 6. 원래 위치에 새 파일 생성
with target.open("x", encoding="utf-8") as f:
    f.write(updated)

# 7. Drive에 최신 버전 백업
with final_backup.open("x", encoding="utf-8") as f:
    f.write(updated)

# 8. 파일 동일 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(final_backup)

print("\n✅ Whisper 서비스 코드 복구 완료!")
print("복구 파일:", target)
print("Drive 최신 백업:", final_backup)
print("✅ Python 문법 검사 통과")
print("✅ 기존 함수 인터페이스 유지")
print("✅ 저장소 기준 자막 경로 적용")
print("✅ 기존 자막 파일 덮어쓰기 방지")
print("✅ 복구 파일과 Drive 백업 완전히 일치")
print("✅ 팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-8단계. Whisper 서비스 복구 =====

✅ Whisper 서비스 코드 복구 완료!
복구 파일: /content/pinpoint-capstone/ai/preprocessing/transcribe_service.py
Drive 최신 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/transcribe_service_FINAL_restored_20261010.py
✅ Python 문법 검사 통과
✅ 기존 함수 인터페이스 유지
✅ 저장소 기준 자막 경로 적용
✅ 기존 자막 파일 덮어쓰기 방지
✅ 복구 파일과 Drive 백업 완전히 일치
✅ 팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [16]:
from pathlib import Path
import subprocess
import ast
import hashlib

repo = Path("/content/pinpoint-capstone")

target = (
    repo / "ai/experiments/korean_minilm/prepare_data.py"
)

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "minilm_prepare_data_restored_20261010.py"
)

print("===== 41-9단계. MiniLM 전처리 코드 복구 =====")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert not target.exists(), "대상 파일이 이미 있습니다. 덮어쓰지 않습니다."
assert not backup.exists(), "백업 파일이 이미 있습니다. 덮어쓰지 않습니다."

code = '''
"""
PinPoint Korean MiniLM training data preparation.

Input:
    AI-Hub TL_12 and VL_12 TAR files containing ZIP archives of JSON labels.

Output:
    minilm_train_pairs.jsonl
    minilm_train_pairs_clean.jsonl
    minilm_val_pairs.jsonl

AI-Hub source data and generated text pairs must not be committed to GitHub.
"""

import argparse
import io
import json
import tarfile
import zipfile
from collections import Counter
from pathlib import Path

from sentence_transformers import SentenceTransformer


MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

MAX_TOKENS = 128
EXCLUDED_VAL_IDS = {"MYL_18127", "MYL_18128"}


def load_archive(tar_path):
    """Read the single ZIP archive stored inside an AI-Hub TAR file."""

    with tarfile.open(tar_path, "r:*") as tar:
        members = [m for m in tar.getmembers() if m.isfile()]

        if len(members) != 1:
            raise ValueError(
                f"Expected one ZIP in {tar_path}, found {len(members)}"
            )

        extracted = tar.extractfile(members[0])

        if extracted is None:
            raise ValueError(f"Cannot read TAR member: {tar_path}")

        zip_bytes = extracted.read()

    archive = zipfile.ZipFile(io.BytesIO(zip_bytes))

    if archive.testzip() is not None:
        archive.close()
        raise ValueError(f"Damaged ZIP archive: {tar_path}")

    return archive


def build_pairs(archive, tokenizer, split):
    """Create summary-to-transcript pairs with the original token filter."""

    pairs = []

    for name in archive.namelist():
        if not name.lower().endswith(".json"):
            continue

        data = json.loads(archive.read(name).decode("utf-8-sig"))

        video_id = data.get("metadata", {}).get("filename")

        if split == "val" and video_id in EXCLUDED_VAL_IDS:
            continue

        summary = str(data.get("summary") or "").strip()

        terms = data.get("video", {}).get("term", [])

        transcript = " ".join(
            str(term.get("transcription") or "").strip()
            for term in terms
            if isinstance(term, dict)
        ).strip()

        if not summary or not transcript:
            continue

        summary_tokens = len(
            tokenizer(
                summary,
                truncation=False,
                verbose=False
            )["input_ids"]
        )

        transcript_tokens = len(
            tokenizer(
                transcript,
                truncation=False,
                verbose=False
            )["input_ids"]
        )

        if (
            summary_tokens > MAX_TOKENS
            or transcript_tokens > MAX_TOKENS
        ):
            continue

        pairs.append({
            "video_id": video_id,
            "query": summary,
            "positive": transcript
        })

    return pairs


def save_jsonl(path, records):
    """Create a new JSONL file without overwriting existing data."""

    with path.open("x", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\\n")


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-tar", type=Path, required=True)
    parser.add_argument("--val-tar", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)

    args = parser.parse_args()

    train_tar = args.train_tar.resolve()
    val_tar = args.val_tar.resolve()

    if not train_tar.is_file() or not val_tar.is_file():
        raise FileNotFoundError("Training or Validation TAR not found.")

    if train_tar == val_tar:
        raise ValueError("Training and Validation TAR must differ.")

    if not train_tar.name.startswith("TL_12"):
        raise ValueError("Expected a TL_12 Training TAR file.")

    if not val_tar.name.startswith("VL_12"):
        raise ValueError("Expected a VL_12 Validation TAR file.")

    output_dir = args.output_dir.resolve()

    output_files = {
        "train": output_dir / "minilm_train_pairs.jsonl",
        "clean": output_dir / "minilm_train_pairs_clean.jsonl",
        "val": output_dir / "minilm_val_pairs.jsonl"
    }

    for path in output_files.values():
        if path.exists():
            raise FileExistsError(
                f"Refusing to overwrite existing file: {path}"
            )

    model = SentenceTransformer(MODEL_NAME)
    model.max_seq_length = MAX_TOKENS
    tokenizer = model.tokenizer

    with load_archive(train_tar) as train_archive:
        train_pairs = build_pairs(
            train_archive, tokenizer, split="train"
        )

    with load_archive(val_tar) as val_archive:
        val_pairs = build_pairs(
            val_archive, tokenizer, split="val"
        )

    train_ids = {p["video_id"] for p in train_pairs}
    val_ids = {p["video_id"] for p in val_pairs}

    overlap = train_ids & val_ids

    if overlap:
        raise ValueError(
            f"Training/Validation video ID overlap: {len(overlap)}"
        )

    query_counts = Counter(
        p["query"].strip() for p in train_pairs
    )

    clean_pairs = [
        p for p in train_pairs
        if query_counts[p["query"].strip()] == 1
    ]

    if not train_pairs or not clean_pairs or not val_pairs:
        raise ValueError("One or more datasets are empty.")

    output_dir.mkdir(parents=True, exist_ok=True)

    save_jsonl(output_files["train"], train_pairs)
    save_jsonl(output_files["clean"], clean_pairs)
    save_jsonl(output_files["val"], val_pairs)

    print("===== PinPoint MiniLM data preparation =====")
    print("Training pairs:", len(train_pairs))
    print("Clean training pairs:", len(clean_pairs))
    print("Validation pairs:", len(val_pairs))
    print("Training/Validation ID overlap:", len(overlap))
    print("Output:", output_dir)


if __name__ == "__main__":
    main()
'''

# 문법 검사 및 기존 실험 조건 확인
ast.parse(code)
compile(code, str(target), "exec")

assert 'MAX_TOKENS = 128' in code
assert 'EXCLUDED_VAL_IDS = {"MYL_18127", "MYL_18128"}' in code

# 새 파일 생성
target.parent.mkdir(parents=True, exist_ok=True)

with target.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

# Drive에 별도 백업
backup.parent.mkdir(parents=True, exist_ok=True)

with backup.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

# 복구 파일과 백업 비교
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\\n✅ MiniLM 전처리 코드 복구 완료!")
print("복구 파일:", target)
print("Drive 백업:", backup)
print("✅ Python 문법 검사 통과")
print("✅ 기존 전처리 조건 유지")
print("✅ 복구 파일과 백업 완전히 일치")
print("✅ 실제 데이터 전처리 실행 없음")
print("✅ 기존 팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-9단계. MiniLM 전처리 코드 복구 =====
\n✅ MiniLM 전처리 코드 복구 완료!
복구 파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/prepare_data.py
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/minilm_prepare_data_restored_20261010.py
✅ Python 문법 검사 통과
✅ 기존 전처리 조건 유지
✅ 복구 파일과 백업 완전히 일치
✅ 실제 데이터 전처리 실행 없음
✅ 기존 팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [17]:
from pathlib import Path
import subprocess
import ast
import hashlib

repo = Path("/content/pinpoint-capstone")

target = repo / "ai/experiments/korean_minilm/train.py"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "minilm_train_restored_20261010.py"
)

print("===== 41-10단계. MiniLM 학습 코드 복구 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert target.parent.is_dir()
assert not target.exists(), "train.py가 이미 있습니다. 덮어쓰지 않습니다."
assert not backup.exists(), "백업이 이미 있습니다. 덮어쓰지 않습니다."

# 2. 이전에 검증했던 학습 코드 복원
code = r'''
"""
PinPoint Korean MiniLM fine-tuning.

Reproduces the original Korean MiniLM training configuration.

Example:
    python -m ai.experiments.korean_minilm.train \
        --train-jsonl /path/to/minilm_train_pairs_clean.jsonl \
        --output-dir /path/to/new_training_output

Do not upload licensed AI-Hub training data or model weights to GitHub.
"""

import argparse
import json
from pathlib import Path

import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    losses,
)
from sentence_transformers.training_args import (
    SentenceTransformerTrainingArguments,
    BatchSamplers,
)


MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


def load_training_pairs(path):
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            query = str(record.get("query") or "").strip()
            positive = str(record.get("positive") or "").strip()

            if not query or not positive:
                raise ValueError("Empty query or positive text found.")

            records.append({
                "anchor": query,
                "positive": positive,
            })

    if not records:
        raise ValueError("Training data is empty.")

    return Dataset.from_list(records)


def main():
    parser = argparse.ArgumentParser(
        description="Fine-tune PinPoint Korean MiniLM"
    )

    parser.add_argument(
        "--train-jsonl",
        type=Path,
        required=True
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        required=True
    )

    args = parser.parse_args()

    train_path = args.train_jsonl.resolve()
    output_dir = args.output_dir.resolve()

    if not train_path.is_file():
        raise FileNotFoundError(train_path)

    # 기존 모델과 체크포인트 보호
    if output_dir.exists():
        raise FileExistsError(
            f"Output directory already exists: {output_dir}"
        )

    if output_dir == train_path.parent:
        raise ValueError(
            "Output directory must differ from the data directory."
        )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Original training used CUDA with fp16. "
            "Please enable a GPU runtime."
        )

    train_dataset = load_training_pairs(train_path)

    model = SentenceTransformer(
        MODEL_NAME,
        device="cuda"
    )
    model.max_seq_length = 128

    train_loss = losses.MultipleNegativesRankingLoss(
        model=model
    )

    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=1,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        fp16=True,
        bf16=False,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        save_strategy="epoch",
        save_total_limit=1,
        eval_strategy="no",
        logging_steps=100,
        report_to="none",
        seed=42,
        data_seed=42,
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        loss=train_loss,
    )

    print("===== PinPoint MiniLM Fine-tuning =====")
    print("Training samples:", len(train_dataset))
    print("Model:", MODEL_NAME)
    print("Epochs:", training_args.num_train_epochs)
    print("Batch size:", training_args.per_device_train_batch_size)
    print("Learning rate:", training_args.learning_rate)
    print("Output:", output_dir)

    trainer.train()

    final_dir = output_dir / "final_model"

    if final_dir.exists():
        raise FileExistsError(
            f"Refusing to overwrite model: {final_dir}"
        )

    trainer.model.save_pretrained(str(final_dir))

    print("MiniLM fine-tuning completed.")
    print("Saved model:", final_dir)


if __name__ == "__main__":
    main()
'''

# 3. 코드 문법 및 설정 확인
ast.parse(code)
compile(code, str(target), "exec")

tree = ast.parse(code)

training_calls = [
    node for node in ast.walk(tree)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id == "SentenceTransformerTrainingArguments"
]

assert len(training_calls) == 1

expected = {
    "num_train_epochs": 1,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "fp16": True,
    "seed": 42,
    "data_seed": 42,
}

actual = {
    kw.arg: ast.literal_eval(kw.value)
    for kw in training_calls[0].keywords
    if kw.arg in expected
}

assert actual == expected, "기존 학습 설정과 다릅니다."

# 4. 저장소에 신규 파일 생성
with target.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

# 5. Google Drive에 즉시 백업
with backup.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

# 6. 두 파일이 완전히 같은지 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\n✅ MiniLM 학습 코드 복구 완료!")
print("복구 파일:", target)
print("Drive 백업:", backup)
print("✅ Python 문법 검사 통과")
print("✅ 기존 하이퍼파라미터 일치")
print("✅ 복구 파일과 백업 완전히 일치")
print("✅ 모델 학습 실행 없음")
print("✅ 기존 데이터·모델·팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-10단계. MiniLM 학습 코드 복구 =====

✅ MiniLM 학습 코드 복구 완료!
복구 파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/train.py
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/minilm_train_restored_20261010.py
✅ Python 문법 검사 통과
✅ 기존 하이퍼파라미터 일치
✅ 복구 파일과 백업 완전히 일치
✅ 모델 학습 실행 없음
✅ 기존 데이터·모델·팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [18]:
from pathlib import Path
import subprocess
import ast
import hashlib

repo = Path("/content/pinpoint-capstone")

target = repo / "ai/experiments/korean_minilm/evaluate.py"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "minilm_evaluate_restored_20261010.py"
)

print("===== 41-11단계. MiniLM 평가 코드 복구 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
assert target.parent.is_dir()
assert not target.exists(), "evaluate.py가 이미 존재합니다."
assert not backup.exists(), "Drive 백업이 이미 존재합니다."

# 2. 기존에 평가 수치를 재현했던 코드 복원
code = r'''
"""
PinPoint Korean MiniLM full-validation retrieval evaluation.

Reproduces the original notebook's summary-to-transcript
retrieval metrics: Recall@1, Recall@5, and MRR.

The transcript at the same index is treated as
the single correct answer.

This does NOT measure temporal video moment retrieval.
"""

import argparse
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer, util


BASE_MODEL = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


def load_validation_pairs(path):
    pairs = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            query = record.get("query")
            positive = record.get("positive")

            if not query or not positive:
                raise ValueError("Empty evaluation text found.")

            pairs.append({
                "query": query,
                "positive": positive
            })

    if not pairs:
        raise ValueError("Validation dataset is empty.")

    return pairs


def evaluate_minilm(model, queries, documents):
    """Match the original notebook's retrieval evaluation."""

    query_embeddings = model.encode(
        queries,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    doc_embeddings = model.encode(
        documents,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        query_embeddings,
        doc_embeddings
    )

    ranks = []

    for i in range(len(queries)):
        ranked_indices = (
            scores[i].argsort(descending=True).tolist()
        )

        ranks.append(ranked_indices.index(i) + 1)

    return {
        "recall_at_1": float(
            np.mean([r == 1 for r in ranks])
        ),
        "recall_at_5": float(
            np.mean([r <= 5 for r in ranks])
        ),
        "mrr": float(
            np.mean([1 / r for r in ranks])
        )
    }


def main():
    parser = argparse.ArgumentParser(
        description="Evaluate Korean MiniLM retrieval"
    )

    parser.add_argument(
        "--val-jsonl",
        type=Path,
        required=True
    )

    parser.add_argument(
        "--finetuned-model",
        type=Path,
        required=True
    )

    parser.add_argument(
        "--output",
        type=Path,
        required=True
    )

    args = parser.parse_args()

    val_path = args.val_jsonl.resolve()
    model_path = args.finetuned_model.resolve()
    output_path = args.output.resolve()

    if not val_path.is_file():
        raise FileNotFoundError(val_path)

    if not model_path.is_dir():
        raise FileNotFoundError(model_path)

    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite: {output_path}"
        )

    pairs = load_validation_pairs(val_path)

    queries = [p["query"] for p in pairs]
    documents = [p["positive"] for p in pairs]

    print("===== PinPoint Korean MiniLM evaluation =====")
    print("Validation pairs:", len(pairs))

    print("\n[Original MiniLM]")
    original_model = SentenceTransformer(BASE_MODEL)

    original_result = evaluate_minilm(
        original_model, queries, documents
    )

    print(original_result)

    print("\n[Fine-tuned MiniLM]")
    trained_model = SentenceTransformer(
        str(model_path)
    )

    trained_result = evaluate_minilm(
        trained_model, queries, documents
    )

    print(trained_result)

    result = {
        "evaluation": (
            "AI-Hub full validation "
            "summary-to-transcript retrieval"
        ),
        "sample_size": len(pairs),
        "original": original_result,
        "finetuned": trained_result
    }

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with output_path.open("x", encoding="utf-8") as f:
        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n✅ Evaluation completed!")
    print("Saved:", output_path)


if __name__ == "__main__":
    main()
'''

# 3. 문법 및 함수 확인
ast.parse(code)
compile(code, str(target), "exec")

tree = ast.parse(code)

functions = {
    node.name
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

assert {"load_validation_pairs", "evaluate_minilm", "main"} <= functions
assert '"recall_at_1"' in code
assert '"recall_at_5"' in code
assert '"mrr"' in code

# 4. 저장소에 새 파일 생성
with target.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

# 5. Drive에 즉시 백업
with backup.open("x", encoding="utf-8") as f:
    f.write(code.lstrip())

# 6. 백업 일치 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\n✅ MiniLM 평가 코드 복구 완료!")
print("복구 파일:", target)
print("Drive 백업:", backup)
print("✅ Python 문법 검사 통과")
print("✅ 기존 Recall@1, Recall@5, MRR 계산 방식 유지")
print("✅ 복구 파일과 Drive 백업 완전히 일치")
print("✅ 실제 모델 평가 실행 없음")
print("✅ 기존 데이터·모델·팀원 코드 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-11단계. MiniLM 평가 코드 복구 =====

✅ MiniLM 평가 코드 복구 완료!
복구 파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/evaluate.py
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/minilm_evaluate_restored_20261010.py
✅ Python 문법 검사 통과
✅ 기존 Recall@1, Recall@5, MRR 계산 방식 유지
✅ 복구 파일과 Drive 백업 완전히 일치
✅ 실제 모델 평가 실행 없음
✅ 기존 데이터·모델·팀원 코드 변경 없음
✅ Git add/commit/push 없음


In [19]:
from pathlib import Path
import subprocess
import json
import hashlib

repo = Path("/content/pinpoint-capstone")

source = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "03_Korean_Training_Data/"
    "minilm_full_validation_comparison.json"
)

target = (
    repo
    / "ai/experiments/korean_minilm/"
    "results/full_validation_metrics.json"
)

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "minilm_full_validation_metrics_restored_20261010.json"
)

print("===== 41-12단계. MiniLM 평가 결과 복구 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 원본 및 대상 파일 확인
assert source.is_file(), "Drive의 원본 평가 JSON이 없습니다."
assert not target.exists(), "대상 JSON이 이미 존재합니다."
assert not backup.exists(), "Drive 백업이 이미 존재합니다."

# 3. 실제 평가 결과 읽기
original = json.loads(source.read_text(encoding="utf-8"))

assert original["sample_size"] == 1948

required_metrics = {"recall_at_1", "recall_at_5", "mrr"}

assert required_metrics <= set(original["original"])
assert required_metrics <= set(original["finetuned"])

# 4. 수치가 이전 검증 결과와 일치하는지 확인
expected = {
    "original": {
        "recall_at_1": 0.5102669404517454,
        "recall_at_5": 0.6555441478439425,
        "mrr": 0.577302482579806,
    },
    "finetuned": {
        "recall_at_1": 0.7515400410677618,
        "recall_at_5": 0.858829568788501,
        "mrr": 0.7996197008399655,
    }
}

for model_name in ("original", "finetuned"):
    for metric_name, expected_value in expected[model_name].items():
        actual_value = float(original[model_name][metric_name])

        assert abs(actual_value - expected_value) < 1e-10, (
            f"평가 수치 불일치: {model_name}/{metric_name}"
        )

# 5. 민감한 원본 텍스트 없이 성능 결과만 정리
result = {
    "task": "Korean summary-to-transcript text retrieval",
    "dataset": "AI-Hub validation pairs",
    "sample_size": 1948,
    "evaluation_note": (
        "Each query is matched against validation transcripts. "
        "The transcript at the same index is treated as the "
        "single correct result. These are text retrieval metrics, "
        "not temporal video moment retrieval metrics."
    ),
    "base_model": (
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
    "original": {
        key: float(original["original"][key])
        for key in ("recall_at_1", "recall_at_5", "mrr")
    },
    "finetuned": {
        key: float(original["finetuned"][key])
        for key in ("recall_at_1", "recall_at_5", "mrr")
    },
    "verification": (
        "Metrics reproduced on the full 1,948-pair "
        "validation dataset in the earlier Colab experiment."
    )
}

content = json.dumps(
    result,
    ensure_ascii=False,
    indent=2
) + "\n"

# 6. 저장소에 새 JSON 생성
target.parent.mkdir(parents=True, exist_ok=True)

with target.open("x", encoding="utf-8") as f:
    f.write(content)

# 7. Drive에 즉시 백업
with backup.open("x", encoding="utf-8") as f:
    f.write(content)

# 8. 백업 검증
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\n===== 복구된 평가 결과 =====")

for model_name, label in [
    ("original", "기본 MiniLM"),
    ("finetuned", "Fine-tuned MiniLM")
]:
    metrics = result[model_name]

    print(f"\n{label}")
    print(f"Recall@1: {metrics['recall_at_1'] * 100:.2f}%")
    print(f"Recall@5: {metrics['recall_at_5'] * 100:.2f}%")
    print(f"MRR: {metrics['mrr']:.4f}")

print("\n✅ 41-12단계 성능 결과 JSON 복구 완료!")
print("복구 파일:", target)
print("Drive 백업:", backup)
print("✅ 실제 평가 JSON에서 수치 추출")
print("✅ 기존 검증 수치와 일치")
print("✅ AI-Hub 원본 텍스트 포함하지 않음")
print("✅ 복구 파일과 Drive 백업 완전히 일치")
print("✅ 모델 재학습 및 재평가 없음")
print("✅ Git add/commit/push 없음")

===== 41-12단계. MiniLM 평가 결과 복구 =====

===== 복구된 평가 결과 =====

기본 MiniLM
Recall@1: 51.03%
Recall@5: 65.55%
MRR: 0.5773

Fine-tuned MiniLM
Recall@1: 75.15%
Recall@5: 85.88%
MRR: 0.7996

✅ 41-12단계 성능 결과 JSON 복구 완료!
복구 파일: /content/pinpoint-capstone/ai/experiments/korean_minilm/results/full_validation_metrics.json
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/minilm_full_validation_metrics_restored_20261010.json
✅ 실제 평가 JSON에서 수치 추출
✅ 기존 검증 수치와 일치
✅ AI-Hub 원본 텍스트 포함하지 않음
✅ 복구 파일과 Drive 백업 완전히 일치
✅ 모델 재학습 및 재평가 없음
✅ Git add/commit/push 없음


In [20]:
from pathlib import Path
import subprocess
import hashlib

repo = Path("/content/pinpoint-capstone")

target = repo / "ai/experiments/korean_minilm/README.md"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "minilm_README_restored_20261010.md"
)

print("===== 41-13단계. MiniLM README 복구 =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 기존 파일 덮어쓰기 방지
assert not target.exists(), "README.md가 이미 존재합니다."
assert not backup.exists(), "Drive 백업이 이미 존재합니다."

# 3. README 내용 작성
readme = r"""# PinPoint Korean MiniLM Fine-tuning

## 1. 개요

PinPoint는 사용자가 자연어로 원하는 장면을 검색하면,
영상 내부의 관련 시간 구간을 찾아주는 서비스입니다.

한국어 영상에서도 검색 성능을 개선하기 위해
기존 Moment-DETR 기반 영상 검색에
한국어 자막 의미 검색을 결합하는 Fusion 방식을 실험했습니다.

자막 의미 검색에는 다음 모델을 사용합니다.

- 기본 모델: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- 추가 학습 모델: 위 모델을 한국어 데이터로 Fine-tuning한 MiniLM

이 디렉터리는 MiniLM의 데이터 준비, 학습, 평가 방법과
실험 결과를 정리합니다.

**주의:** MiniLM은 영상 장면을 직접 분석하는 모델이 아닙니다.
텍스트 간 의미적 유사도를 계산하며,
영상 장면 검색은 별도의 Moment-DETR 계열 모델과 결합합니다.

---

## 2. 폴더 구성

~~~text
ai/
├── features/
│   ├── fusion_search.py
│   └── consensus_rerank.py
├── preprocessing/
│   └── transcribe_service.py
└── experiments/
    └── korean_minilm/
        ├── prepare_data.py
        ├── train.py
        ├── evaluate.py
        ├── README.md
        └── results/
            └── full_validation_metrics.json
~~~

각 파일의 역할:

| 파일 | 역할 |
|---|---|
| `prepare_data.py` | AI-Hub 학습·검증 데이터에서 텍스트 쌍 생성 |
| `train.py` | 기본 MiniLM Fine-tuning |
| `evaluate.py` | 기본 모델과 Fine-tuned 모델의 텍스트 검색 성능 비교 |
| `results/full_validation_metrics.json` | 전체 Validation 평가 결과 |
| `ai/features/fusion_search.py` | Moment-DETR 결과와 자막 의미 검색 결과 결합 |
| `ai/features/consensus_rerank.py` | 후보 구간 간 중첩을 활용한 순위 조정 |
| `ai/preprocessing/transcribe_service.py` | Whisper 음성 전사를 통한 자막 JSON 생성 |

---

## 3. 사용 데이터 및 전처리

AI-Hub 한국어 데이터의 `TL_12` 학습 TAR,
`VL_12` 검증 TAR를 사용한 실험입니다.

TAR 내부 ZIP의 JSON 라벨에서 다음 정보를 추출합니다.

- `query`: JSON의 `summary`
- `positive`: `video.term`의 `transcription`을 순서대로 결합한 텍스트
- `video_id`: `metadata.filename`

한 쌍은 다음과 같은 형식입니다.

~~~json
{
  "video_id": "example_video_id",
  "query": "검색 질문에 해당하는 요약문",
  "positive": "관련 대화 내용"
}
~~~

위 JSON은 설명을 위해 만든 가상 예시이며,
실제 AI-Hub 데이터 문장을 포함하지 않습니다.

전처리 조건:

1. 요약문과 대화문이 모두 존재하는 샘플만 사용
2. 기본 MiniLM 토크나이저 기준 각 텍스트 최대 128토큰
3. Validation의 `MYL_18127`, `MYL_18128` 제외
4. 학습·검증 영상 ID 중복 검사
5. 학습 데이터에서 중복된 query를 가진 샘플 제거
6. 원본 학습 쌍, 정제 학습 쌍, 검증 쌍을 JSONL로 저장

기존 실험의 데이터 규모:

| 데이터 | 쌍 개수 |
|---|---:|
| 원본 학습 쌍 | 15,823 |
| 중복 query 제거 후 학습 쌍 | 15,502 |
| 검증 쌍 | 1,948 |

학습과 검증의 영상 ID 중복은 0개였습니다.

이 숫자는 기존 실험에서 확인한 결과이며,
복구한 코드로 새 런타임에서 전처리를 재실행한 결과는 아닙니다.

---

## 4. 데이터 전처리 실행

프로젝트 루트에서 실행합니다.

~~~bash
python -m ai.experiments.korean_minilm.prepare_data \
  --train-tar "/path/to/TL_12_training.tar" \
  --val-tar "/path/to/VL_12_validation.tar" \
  --output-dir "/path/to/new_prepared_data"
~~~

생성되는 파일:

~~~text
new_prepared_data/
├── minilm_train_pairs.jsonl
├── minilm_train_pairs_clean.jsonl
└── minilm_val_pairs.jsonl
~~~

입력 TAR 파일명은 각각 `TL_12`, `VL_12`로 시작해야 합니다.

기존 파일을 실수로 덮어쓰지 않도록
출력 파일이 이미 존재하면 실행을 중단합니다.

---

## 5. MiniLM Fine-tuning

### 기본 모델

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

### 학습 방식

요약문을 Anchor, 대응하는 대화문을 Positive로 사용하여
`MultipleNegativesRankingLoss`로 학습합니다.

학습에서는 같은 배치 내의 다른 텍스트 쌍을
Negative 후보로 활용합니다.

### 학습 설정

| 항목 | 설정 |
|---|---|
| Epoch | 1 |
| Batch size | 8 |
| Gradient accumulation | 2 |
| Learning rate | 2e-5 |
| Warmup ratio | 0.1 |
| 최대 입력 길이 | 128 tokens |
| Loss | MultipleNegativesRankingLoss |
| Batch sampler | NO_DUPLICATES |
| FP16 | 사용 |
| Seed | 42 |
| 장치 | CUDA GPU |

### 학습 실행

~~~bash
python -m ai.experiments.korean_minilm.train \
  --train-jsonl "/path/to/minilm_train_pairs_clean.jsonl" \
  --output-dir "/path/to/new_training_output"
~~~

최종 모델 저장 위치:

~~~text
new_training_output/
└── final_model/
~~~

**이미 학습된 모델이 Google Drive에 보존되어 있으므로,
기존 결과를 사용하려는 경우 재학습할 필요가 없습니다.**

`train.py`는 기존 출력 디렉터리가 있으면
새 학습 실행을 거부하도록 작성되어 있습니다.

---

## 6. Validation 전체 평가

검증에는 총 1,948개의 한국어 텍스트 쌍을 사용했습니다.

각 query에 대해 검증 데이터의 모든 positive 텍스트와
코사인 유사도를 계산하고,
같은 인덱스의 positive를 정답으로 간주합니다.

평가 지표:

- **Recall@1:** 정답 텍스트가 1위인 비율
- **Recall@5:** 정답 텍스트가 상위 5위 이내에 있는 비율
- **MRR:** 정답 순위의 역수 평균

### 평가 실행

~~~bash
python -m ai.experiments.korean_minilm.evaluate \
  --val-jsonl "/path/to/minilm_val_pairs.jsonl" \
  --finetuned-model "/path/to/final_model" \
  --output "/path/to/new_evaluation.json"
~~~

### 기존 평가 결과

| 지표 | 기본 MiniLM | Fine-tuned MiniLM |
|---|---:|---:|
| Recall@1 | 51.03% | 75.15% |
| Recall@5 | 65.55% | 85.88% |
| MRR | 0.5773 | 0.7996 |

원래 평가 결과는 기존 실험에서 재현 검증했으며,
정확한 수치는 `results/full_validation_metrics.json`에 기록했습니다.

**이 결과는 한국어 요약문과 대화문 사이의 텍스트 검색 성능입니다.**

영상에서 정답 시간 구간을 찾는 성능인
R1@0.5, mAP, 시간 구간 IoU와 동일한 지표가 아닙니다.

또한 요약문과 대화문이 동일한 원천 데이터에서 생성된
텍스트 쌍을 대상으로 한 평가이므로,
실제 서비스의 사용자 질의 성능을 그대로 보장하지 않습니다.

---

## 7. Fusion 검색 모델 연결

`ai/features/fusion_search.py`의
`predict_moments_fusion()`에 `model_type`을 전달합니다.

- `model_type="base"`: 공개 기본 MiniLM 사용
- `model_type="finetuned"`: 학습 완료된 MiniLM 사용

Fine-tuned 모델은 다음 환경변수로 경로를 지정합니다.

~~~python
import os

os.environ["PINPOINT_MINILM_MODEL_PATH"] = (
    "/absolute/path/to/minilm_finetuned/final_model"
)
~~~

Fusion 함수의 주요 입력은 다음과 같습니다.

~~~python
predict_moments_fusion(
    video_id=video_id,
    feature_path=feature_path,
    subtitles=subtitles,
    korean_query=korean_query,
    english_query=english_query,
    video_duration=video_duration,
    subtitle_json_path=subtitle_json_path,
    top_k=5,
    model_type="finetuned",
)
~~~

이는 함수 호출 형식을 보여주는 예시입니다.
위 변수들에는 실제 영상의 값이 필요합니다.

Fusion은 영상 특징을 사용하는 Moment-DETR 검색과
자막 텍스트 의미 검색 후보를 결합합니다.

다만 현재 팀의 실제 검색 API가
이 Fusion 함수를 호출하도록 완전히 연결되었는지는
별도 확인이 필요합니다.

---

## 8. 실행 환경 및 의존성

다음은 실험 코드에서 사용하는 주요 Python 패키지입니다.

~~~text
torch
numpy
sentence-transformers
transformers
datasets
accelerate
faster-whisper
ffmpeg-python
ftfy
regex
tqdm
~~~

정확한 호환 버전은 실행 환경에 맞추어 검증해야 합니다.

기존 Moment-DETR 추론에는
프로젝트 내 별도 모델 코드와 가중치, CLIP 관련 의존성이 필요합니다.

Whisper STT, 영상 feature 추출, MiniLM 실행은
각각 처리 시간과 메모리를 사용하므로
실제 서비스 배포 환경에서 추가 성능 검증이 필요합니다.

---

## 9. 데이터·모델 파일 관리 주의사항

다음 항목은 GitHub에 업로드하지 않습니다.

- AI-Hub에서 제공받은 원본 TAR·ZIP·JSON 데이터
- AI-Hub 원문이 포함된 학습·검증 JSONL
- 대용량 학습 완료 모델 가중치 및 체크포인트
- 라이선스와 공개 가능 여부를 확인하지 않은 영상·자막
- Google Drive 개인 백업 및 임시 파일

GitHub에는 데이터가 아닌
**전처리·학습·평가 코드, 실행 설명, 집계 성능 수치**
중 공유 가능한 항목만 포함합니다.

실제 AI-Hub 자료는 별도 이용 조건과
팀 내 접근 권한을 확인해야 합니다.

---

## 10. 현재 한계와 추후 확인 사항

1. 텍스트 검색 성능 향상이 영상 구간 검색 성능 향상으로
   자동 연결되는 것은 아닙니다.
2. 한국어 질문을 Moment-DETR용 영어 질문으로 변환하는
   과정은 검색 API와의 연결 시 별도 처리해야 합니다.
3. Whisper가 생성한 자막의 품질에 따라
   자막 의미 검색 결과가 달라질 수 있습니다.
4. Fusion과 Consensus 후보 순위 및 최종 시작 시점의
   정확도는 실제 영상 테스트로 추가 검증해야 합니다.
5. 현재 Fusion 함수의 결과 형식과 백엔드 API의
   응답 형식이 완전히 호환되는지 확인해야 합니다.
6. 모델 로드, 영상 feature 생성, Whisper 전사에 따른
   처리 지연 및 서비스 자원 사용량을 점검해야 합니다.

이 문서는 실험과 코드의 재현 방법을 정리한 것으로,
백엔드 통합 완료나 배포 성능 검증을 의미하지 않습니다.
"""

# 4. 필수 내용 확인
required_sections = [
    "## 1. 개요",
    "## 3. 사용 데이터 및 전처리",
    "## 5. MiniLM Fine-tuning",
    "## 6. Validation 전체 평가",
    "## 7. Fusion 검색 모델 연결",
    "## 9. 데이터·모델 파일 관리 주의사항",
    "## 10. 현재 한계와 추후 확인 사항",
]

for section in required_sections:
    assert section in readme, f"필수 항목 누락: {section}"

# 5. 파일 생성
target.parent.mkdir(parents=True, exist_ok=True)

with target.open("x", encoding="utf-8") as f:
    f.write(readme)

# 6. Drive에 백업
backup.parent.mkdir(parents=True, exist_ok=True)

with backup.open("x", encoding="utf-8") as f:
    f.write(readme)

# 7. 파일 일치 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\n✅ MiniLM README 복구 완료!")
print("README:", target)
print("Drive 백업:", backup)
print("✅ 데이터 전처리·학습·평가 절차 기록")
print("✅ 실제 검증 성능 기록")
print("✅ Fusion 연결 방법 기록")
print("✅ 한계 및 데이터 공유 주의사항 기록")
print("✅ README와 Drive 백업 완전히 일치")
print("✅ Git add/commit/push 없음")

===== 41-13단계. MiniLM README 복구 =====

✅ MiniLM README 복구 완료!
README: /content/pinpoint-capstone/ai/experiments/korean_minilm/README.md
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/minilm_README_restored_20261010.md
✅ 데이터 전처리·학습·평가 절차 기록
✅ 실제 검증 성능 기록
✅ Fusion 연결 방법 기록
✅ 한계 및 데이터 공유 주의사항 기록
✅ README와 Drive 백업 완전히 일치
✅ Git add/commit/push 없음


In [21]:
from pathlib import Path
import subprocess
import hashlib

repo = Path("/content/pinpoint-capstone")

target = repo / "ai/experiments/KOREAN_AI_INTEGRATION.md"

backup = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "KOREAN_AI_INTEGRATION_restored_20261010.md"
)

print("===== 41-14단계. AI–백엔드 연동 문서 생성 =====")

# 1. AI 전용 브랜치인지 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 기존 파일 덮어쓰기 방지
assert not target.exists(), "문서가 이미 있습니다. 덮어쓰지 않습니다."
assert not backup.exists(), "Drive 백업이 이미 있습니다."

# 3. 연동 설명 문서
document = r"""# PinPoint 한국어 AI 검색 및 백엔드 연동 가이드

## 1. 목적

한국어 영상 검색을 위해 개발·실험한 AI 기능과
팀 백엔드 검색 API를 연결할 때 필요한 사항을 정리합니다.

이 문서는 연동 가이드이며,
백엔드 통합이나 서비스 배포가 완료됐다는 의미는 아닙니다.

## 2. 구현된 AI 구성

- Moment-DETR 기반 영상 구간 후보 탐색
- Whisper 기반 음성 전사 및 자막 JSON 생성 코드
- MiniLM 기반 한국어·영어 자막 의미 검색
- 기본 MiniLM / Fine-tuned MiniLM 선택 기능
- 영상·자막 후보를 결합하는 Fusion 검색
- 후보 시간 구간 중첩을 활용하는 Consensus V1·V2

관련 코드:

~~~text
ai/features/run_moment_detr_inference.py
ai/features/fusion_search.py
ai/features/consensus_rerank.py
ai/preprocessing/transcribe_service.py
~~~

MiniLM 실험 코드:

~~~text
ai/experiments/korean_minilm/
├── prepare_data.py
├── train.py
├── evaluate.py
├── README.md
└── results/
    └── full_validation_metrics.json
~~~

## 3. 기존 검색 API와 Fusion 입력 차이

이전 백엔드 코드 점검 당시 검색 API는
Moment-DETR Hybrid 함수를 호출하는 구조였습니다.

기존 주요 입력:

- video_id
- 영상 feature 경로
- 자연어 query
- top_k

Fusion 함수는 다음 입력을 요구합니다.

~~~python
predict_moments_fusion(
    video_id=video_id,
    feature_path=feature_path,
    subtitles=subtitles,
    korean_query=korean_query,
    english_query=english_query,
    video_duration=video_duration,
    subtitle_json_path=subtitle_json_path,
    top_k=5,
    model_type="finetuned",
)
~~~

따라서 백엔드 연동 전 다음 값을 확보해야 합니다.

- 한국어 사용자 질의
- Moment-DETR에 전달할 영어 질의
- 검색할 영상의 feature 경로
- 영상 전체 길이
- 해당 영상의 자막 데이터와 자막 JSON 경로
- MiniLM 모델 선택 및 모델 경로

백엔드 코드는 팀원 작업에 따라 변경될 수 있으므로
연동 직전 최신 코드를 다시 확인해야 합니다.

## 4. 한국어 질의 처리

현재 Fusion 함수는 한국어 검색어와 영어 검색어를
별도 인자로 전달받습니다.

예시:

~~~text
korean_query: "두부를 칼로 자르는 장면"
english_query: "the scene where tofu is cut with a knife"
~~~

위 영어 문장은 설명을 위한 번역 예시입니다.

**현재 Fusion 함수에 자동 번역 기능이 내장된 것은 아닙니다.**

실제 서비스에서는 한국어→영어 변환 방식,
번역 실패 처리, 영어 입력 처리 정책을 정해야 합니다.

## 5. Whisper 자막

전사 함수:

~~~python
transcribe_video(
    video_path=video_path,
    video_id=video_id,
)
~~~

자막 저장 위치:

~~~text
<project_root>/data/processed/<video_id>/subtitles.json
~~~

생성 데이터의 예시:

~~~json
[
  {
    "start_time": 1.25,
    "end_time": 3.47,
    "text": "예시 자막입니다."
  }
]
~~~

현재 코드에는 다음 보호 기능이 있습니다.

- 프로젝트 루트 기준 저장 경로
- 기존 자막 JSON 재사용
- 잘못된 video_id 차단
- 기존 자막 파일 덮어쓰기 방지

실제 업로드 영상에서의 전사,
동시 요청, 전사 실패 및 백엔드 연결은 별도로 검증해야 합니다.

## 6. MiniLM 모델 선택

기본 모델:

~~~text
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
~~~

Fine-tuned 모델 경로 지정:

~~~python
import os

os.environ["PINPOINT_MINILM_MODEL_PATH"] = (
    "/absolute/path/to/final_model"
)
~~~

선택 옵션:

- `model_type="base"`: 기본 MiniLM
- `model_type="finetuned"`: 추가 학습한 MiniLM

Colab에서 두 모델의 로드 및
384차원 한국어 문장 임베딩 생성을 확인했습니다.

이는 모델 선택 기능 테스트 결과이며,
백엔드 통합이나 최종 영상 검색 성능 검증은 아닙니다.

## 7. Fusion 결과와 백엔드 응답 형식

이전 검색 API에서는 다음 필드를 사용했습니다.

~~~text
segment_id
start_time
end_time
score
~~~

Fusion의 자막 기반 결과에는
`score=None`인 후보가 존재할 수 있습니다.

자막 검색 유사도는
`subtitle_score`에 담길 수 있습니다.

따라서 백엔드에서 다음처럼 계산하면
오류가 발생할 수 있습니다.

~~~python
max(moment["score"] for moment in moments)
~~~

연동 시 주의사항:

1. Fusion 결과의 필드 구조 확인
2. score=None인 후보의 안전한 처리
3. Moment-DETR 점수와 MiniLM 유사도 구분
4. 최종 순위 및 Top-1 선정 규칙 결정
5. 기존 프론트엔드 응답 형식 유지

서로 다른 점수를 검증 없이
같은 확률이나 동일한 의미의 점수로 취급하면 안 됩니다.

## 8. Consensus V1·V2

구현 코드:

~~~text
ai/features/consensus_rerank.py
~~~

Moment-DETR Hybrid, 한국어 자막 검색,
영어 자막 검색 후보 간 시간 구간 중첩을 활용합니다.

- V1: 후보 간 IoU를 순위 점수에 추가
- V2: IoU 0.5 이상인 중첩을 점수에 추가

Consensus는 현재 실험 기능입니다.

최종 서비스에 사용할 방식은
실제 영상의 Top-1 검색 정확도를 비교해 결정해야 합니다.

## 9. MiniLM 평가 결과

AI-Hub Validation 텍스트 쌍 1,948개 평가:

| 지표 | 기본 MiniLM | Fine-tuned MiniLM |
|---|---:|---:|
| Recall@1 | 51.03% | 75.15% |
| Recall@5 | 65.55% | 85.88% |
| MRR | 0.5773 | 0.7996 |

세부 결과:

~~~text
ai/experiments/korean_minilm/results/full_validation_metrics.json
~~~

위 값은 **한국어 요약문→대화문 텍스트 검색 성능**입니다.

영상 구간 검색의 R1@0.5, mAP, IoU와는 구별해야 합니다.

## 10. 서비스 연동 전 확인할 항목

- [ ] 최신 백엔드 검색 API 확인
- [ ] Fusion 호출에 필요한 입력값 확보
- [ ] 한국어→영어 질의 처리
- [ ] 자막 입력 형식 검증
- [ ] feature 경로 및 영상 길이 확인
- [ ] score=None 처리
- [ ] 프론트엔드 검색 결과 스키마 유지
- [ ] Whisper 실제 영상 전사 테스트
- [ ] 기본/Fine-tuned MiniLM 영상 검색 비교
- [ ] 처리 속도 및 메모리 측정
- [ ] 배포 환경의 라이브러리·모델 경로 확인

## 11. 데이터 및 코드 공유 주의사항

GitHub에 포함하지 않을 항목:

- AI-Hub 원본 TAR·ZIP·JSON
- 실제 AI-Hub 문장이 포함된 학습·검증 JSONL
- 대용량 모델 가중치 및 체크포인트
- 공개 권한을 확인하지 않은 영상 및 자막
- 개인 Google Drive 백업

GitHub에는 공유 가능한 코드와 설명,
개별 원문이 없는 집계 평가 결과만 포함합니다.

## 12. 현재 완료 범위

완료 또는 확인한 내용:

- Fusion 및 Consensus 코드 복구
- 기본/Fine-tuned MiniLM 모델 로드 검증
- Whisper 서비스 코드 복구
- MiniLM 전처리·학습·평가 코드 복구
- 전체 Validation 평가 결과 복구
- 재현 절차와 연동 요구사항 문서화

아직 완료되지 않은 내용:

- 최신 검색 API와 Fusion의 실제 연결
- 실제 업로드 영상 전체 흐름 검증
- 배포 서버에서의 모델·Whisper 실행 검증
- Fine-tuned 모델의 영상 구간 검색 개선 입증
- 최종 Top-1 정확도 및 지연 시간 검증
"""

# 4. 필수 내용 확인
required = [
    "## 3. 기존 검색 API와 Fusion 입력 차이",
    "## 4. 한국어 질의 처리",
    "## 5. Whisper 자막",
    "## 7. Fusion 결과와 백엔드 응답 형식",
    "## 10. 서비스 연동 전 확인할 항목",
    "## 12. 현재 완료 범위",
]

for heading in required:
    assert heading in document, f"필수 항목 누락: {heading}"

# 5. 문서와 Drive 백업 생성
target.parent.mkdir(parents=True, exist_ok=True)
backup.parent.mkdir(parents=True, exist_ok=True)

with target.open("x", encoding="utf-8") as f:
    f.write(document)

with backup.open("x", encoding="utf-8") as f:
    f.write(document)

# 6. 두 파일 동일 여부 확인
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(target) == sha256(backup)

print("\n✅ AI–백엔드 연동 문서 생성 완료!")
print("문서:", target)
print("Drive 백업:", backup)
print("✅ 현재 구현 범위 및 미완료 사항 기록")
print("✅ Fusion 입력값과 score 호환성 설명")
print("✅ README와 별도로 통합 주의사항 정리")
print("✅ Drive 백업과 내용 완전히 일치")
print("✅ 기존 백엔드 파일 변경 없음")
print("✅ Git add/commit/push 없음")

===== 41-14단계. AI–백엔드 연동 문서 생성 =====

✅ AI–백엔드 연동 문서 생성 완료!
문서: /content/pinpoint-capstone/ai/experiments/KOREAN_AI_INTEGRATION.md
Drive 백업: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/KOREAN_AI_INTEGRATION_restored_20261010.md
✅ 현재 구현 범위 및 미완료 사항 기록
✅ Fusion 입력값과 score 호환성 설명
✅ README와 별도로 통합 주의사항 정리
✅ Drive 백업과 내용 완전히 일치
✅ 기존 백엔드 파일 변경 없음
✅ Git add/commit/push 없음


In [22]:
from pathlib import Path
import subprocess
import ast

repo = Path("/content/pinpoint-capstone")

print("===== 41-15단계. GitHub 업로드 전 파일 점검 =====")

# 1. 현재 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ 현재 브랜치:", branch)

# 2. 업로드 예정 파일 목록
expected_files = [
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
    "ai/experiments/korean_minilm/results/full_validation_metrics.json",
    "ai/experiments/KOREAN_AI_INTEGRATION.md",
]

print("\n===== 업로드 예정 AI 파일 9개 =====")

for relative in expected_files:
    path = repo / relative

    assert path.is_file(), f"파일이 없습니다: {relative}"

    size = path.stat().st_size

    if path.suffix == ".py":
        ast.parse(path.read_text(encoding="utf-8"))

    assert size < 100_000, (
        f"예상보다 파일이 큽니다: {relative} ({size} bytes)"
    )

    print(f"✅ {relative} ({size:,} bytes)")

# 3. 현재 Git 변경 사항 확인
print("\n===== 전체 Git 변경 사항 =====")

status = subprocess.check_output(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    text=True,
)

print(status if status.strip() else "(변경 사항 없음)")

# 4. 기존 추적 파일 변경 여부 확인
tracked_changes = subprocess.check_output(
    ["git", "diff", "--name-only", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("\n===== 기존 Git 추적 파일 변경 =====")
print(tracked_changes if tracked_changes else "✅ 기존 추적 파일 변경 없음")

# 5. 예상 목록과 전체 변경 목록 비교
changed_paths = set()

for line in status.splitlines():
    if len(line) >= 4:
        changed_paths.add(line[3:])

unexpected = changed_paths - set(expected_files)
missing_from_status = set(expected_files) - changed_paths

print("\n===== 변경 파일 목록 비교 =====")

if not unexpected:
    print("✅ 예상하지 못한 변경 파일 없음")
else:
    print("⚠️ 예상 외 변경 파일:")
    for name in sorted(unexpected):
        print(" -", name)

if not missing_from_status:
    print("✅ 업로드 예정 9개 파일이 변경 목록에 모두 표시됨")
else:
    print("⚠️ Git 변경 목록에 나타나지 않은 파일:")
    for name in sorted(missing_from_status):
        print(" -", name)

print("\n✅ 41-15단계 조회 완료!")
print("✅ Git add/commit/push 실행 없음")
print("✅ 파일 수정·삭제 없음")

===== 41-15단계. GitHub 업로드 전 파일 점검 =====
✅ 현재 브랜치: feature/korean-ai-fusion

===== 업로드 예정 AI 파일 9개 =====
✅ ai/features/fusion_search.py (6,859 bytes)
✅ ai/features/consensus_rerank.py (1,745 bytes)
✅ ai/preprocessing/transcribe_service.py (1,919 bytes)
✅ ai/experiments/korean_minilm/prepare_data.py (5,849 bytes)
✅ ai/experiments/korean_minilm/train.py (4,092 bytes)
✅ ai/experiments/korean_minilm/evaluate.py (4,173 bytes)
✅ ai/experiments/korean_minilm/README.md (10,248 bytes)
✅ ai/experiments/korean_minilm/results/full_validation_metrics.json (805 bytes)
✅ ai/experiments/KOREAN_AI_INTEGRATION.md (7,330 bytes)

===== 전체 Git 변경 사항 =====
?? ai/experiments/KOREAN_AI_INTEGRATION.md
?? ai/experiments/korean_minilm/README.md
?? ai/experiments/korean_minilm/evaluate.py
?? ai/experiments/korean_minilm/prepare_data.py
?? ai/experiments/korean_minilm/results/full_validation_metrics.json
?? ai/experiments/korean_minilm/train.py
?? ai/features/consensus_rerank.py
?? ai/features/fusion_search.py
?? a

In [23]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

expected_files = [
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
    "ai/experiments/korean_minilm/results/full_validation_metrics.json",
    "ai/experiments/KOREAN_AI_INTEGRATION.md",
]

print("===== 41-16단계. AI 파일 9개 Staging =====")

# 1. 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True,
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. Git에 이미 등록된 변경이 없는지 확인
already_staged = subprocess.check_output(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    text=True,
).strip()

assert not already_staged, (
    "이미 Staging된 파일이 있습니다. 먼저 확인해야 합니다."
)

# 3. 예상한 파일이 모두 존재하는지 확인
for relative in expected_files:
    assert (repo / relative).is_file(), (
        f"파일 없음: {relative}"
    )

# 4. 지정한 9개 파일만 Staging
subprocess.run(
    ["git", "add", "--", *expected_files],
    cwd=repo,
    check=True,
)

# 5. Staging된 파일 목록 확인
staged = subprocess.check_output(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    text=True,
).splitlines()

assert set(staged) == set(expected_files), (
    "Staging 목록이 예상한 9개 파일과 다릅니다."
)

assert len(staged) == 9

# 6. 기존 추적 파일 수정 여부 확인
change_types = subprocess.check_output(
    ["git", "diff", "--cached", "--name-status"],
    cwd=repo,
    text=True,
).splitlines()

assert all(line.startswith("A\t") for line in change_types), (
    "새 파일 추가 이외의 변경이 있습니다."
)

print("\n===== Staging된 파일 =====")
for line in change_types:
    print(line)

print("\n===== 변경 요약 =====")
subprocess.run(
    ["git", "diff", "--cached", "--stat"],
    cwd=repo,
    check=True,
)

print("\n✅ AI 파일 9개만 Staging 완료!")
print("✅ 기존 팀원 파일 수정 없음")
print("✅ Git Commit 실행 없음")
print("✅ Git Push 실행 없음")
print("✅ main 브랜치 변경 없음")

===== 41-16단계. AI 파일 9개 Staging =====

===== Staging된 파일 =====
A	ai/experiments/KOREAN_AI_INTEGRATION.md
A	ai/experiments/korean_minilm/README.md
A	ai/experiments/korean_minilm/evaluate.py
A	ai/experiments/korean_minilm/prepare_data.py
A	ai/experiments/korean_minilm/results/full_validation_metrics.json
A	ai/experiments/korean_minilm/train.py
A	ai/features/consensus_rerank.py
A	ai/features/fusion_search.py
A	ai/preprocessing/transcribe_service.py

===== 변경 요약 =====

✅ AI 파일 9개만 Staging 완료!
✅ 기존 팀원 파일 수정 없음
✅ Git Commit 실행 없음
✅ Git Push 실행 없음
✅ main 브랜치 변경 없음


In [24]:
from pathlib import Path
import subprocess
import ast
import json
import re

repo = Path("/content/pinpoint-capstone")

expected_files = {
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
    "ai/experiments/korean_minilm/results/full_validation_metrics.json",
    "ai/experiments/KOREAN_AI_INTEGRATION.md",
}

print("===== 41-17단계. Commit 전 최종 검사 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인")

# 2. 실제 Staging된 파일 검사
staged = subprocess.check_output(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    text=True
).splitlines()

assert set(staged) == expected_files
assert len(staged) == 9

print("✅ Staging 파일 정확히 9개")

# 3. 모두 신규 파일인지 확인
staged_status = subprocess.check_output(
    ["git", "diff", "--cached", "--name-status"],
    cwd=repo,
    text=True
).splitlines()

assert all(line.startswith("A\t") for line in staged_status)
print("✅ 기존 추적 파일 수정 없음")

# 4. Git에 저장될 실제 내용을 검사
total_bytes = 0

for relative in sorted(staged):
    raw = subprocess.check_output(
        ["git", "show", f":{relative}"],
        cwd=repo
    )

    total_bytes += len(raw)

    assert len(raw) < 100_000, (
        f"예상보다 큰 파일: {relative}"
    )

    content = raw.decode("utf-8")

    if relative.endswith(".py"):
        ast.parse(content)

    if relative.endswith(".json"):
        data = json.loads(content)

        assert "sample_size" in data
        assert data["sample_size"] == 1948

    # Git에 들어갈 실제 파일과 현재 파일이 같은지 확인
    assert raw == (repo / relative).read_bytes(), (
        f"Staging 이후 파일 내용이 변경됨: {relative}"
    )

    # 의심스러운 인증정보 패턴 검사
    secret_patterns = [
        r"(?i)(?:api_key|secret_key|access_token)\s*=\s*['\"][^'\"]{12,}['\"]",
        r"hf_[A-Za-z0-9]{20,}",
        r"sk-[A-Za-z0-9]{20,}",
        r"ghp_[A-Za-z0-9]{20,}",
    ]

    for pattern in secret_patterns:
        assert not re.search(pattern, content), (
            f"인증정보 의심 패턴 발견: {relative}"
        )

    print(f"✅ {relative}")

# 5. 전체 크기 확인
print(f"\n전체 파일 크기: {total_bytes:,} bytes")

assert total_bytes < 100_000, (
    "전체 파일 크기가 예상보다 큽니다."
)

# 6. Git 공백/충돌 검사
subprocess.run(
    ["git", "diff", "--cached", "--check"],
    cwd=repo,
    check=True
)

print("\n✅ 41-17단계 최종 검사 완료!")
print("✅ 파일 9개와 실제 Staging 내용 일치")
print("✅ Python 문법 및 JSON 형식 확인")
print("✅ 기본적인 인증정보 의심 패턴 검사 통과")
print("✅ Git diff 공백/충돌 검사 통과")
print("✅ Git Commit/Push 실행 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-17단계. Commit 전 최종 검사 =====
✅ AI 전용 브랜치 확인
✅ Staging 파일 정확히 9개
✅ 기존 추적 파일 수정 없음
✅ ai/experiments/KOREAN_AI_INTEGRATION.md
✅ ai/experiments/korean_minilm/README.md
✅ ai/experiments/korean_minilm/evaluate.py
✅ ai/experiments/korean_minilm/prepare_data.py
✅ ai/experiments/korean_minilm/results/full_validation_metrics.json
✅ ai/experiments/korean_minilm/train.py
✅ ai/features/consensus_rerank.py
✅ ai/features/fusion_search.py
✅ ai/preprocessing/transcribe_service.py

전체 파일 크기: 43,020 bytes

✅ 41-17단계 최종 검사 완료!
✅ 파일 9개와 실제 Staging 내용 일치
✅ Python 문법 및 JSON 형식 확인
✅ 기본적인 인증정보 의심 패턴 검사 통과
✅ Git diff 공백/충돌 검사 통과
✅ Git Commit/Push 실행 없음
✅ 팀원 main 브랜치 변경 없음


In [27]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

print("===== 41-18-2단계. Git 작성자 정보 설정 =====")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 현재 PinPoint 저장소에만 적용
subprocess.run(
    ["git", "config", "--local", "user.name", "seoyeon0905"],
    cwd=repo,
    check=True
)

subprocess.run(
    ["git", "config", "--local", "user.email", "ssseoyeon95@naver.com"],
    cwd=repo,
    check=True
)

for key in ["user.name", "user.email"]:
    value = subprocess.check_output(
        ["git", "config", "--local", "--get", key],
        cwd=repo,
        text=True
    ).strip()
    print(f"✅ {key}: {value}")

staged = subprocess.check_output(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    text=True
).splitlines()

assert len(staged) == 9

print("\n✅ Git 작성자 설정 완료!")
print("✅ AI 파일 9개 Staging 유지")
print("✅ 현재 저장소에만 설정 적용")
print("✅ Commit/Push 아직 실행하지 않음")
print("✅ main 브랜치 변경 없음")

===== 41-18-2단계. Git 작성자 정보 설정 =====
✅ user.name: seoyeon0905
✅ user.email: ssseoyeon95@naver.com

✅ Git 작성자 설정 완료!
✅ AI 파일 9개 Staging 유지
✅ 현재 저장소에만 설정 적용
✅ Commit/Push 아직 실행하지 않음
✅ main 브랜치 변경 없음


In [28]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")

expected_files = {
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
    "ai/experiments/korean_minilm/results/full_validation_metrics.json",
    "ai/experiments/KOREAN_AI_INTEGRATION.md",
}

print("===== 41-18-3단계. AI 파일 9개 Git Commit =====")

# 1. 현재 AI 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인:", branch)

# 2. Staging된 파일 재확인
staged = subprocess.check_output(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    text=True
).splitlines()

assert set(staged) == expected_files
assert len(staged) == 9

print("✅ Commit 대상 AI 파일 정확히 9개")

# 3. 모든 파일이 신규 추가인지 확인
statuses = subprocess.check_output(
    ["git", "diff", "--cached", "--name-status"],
    cwd=repo,
    text=True
).splitlines()

assert all(line.startswith("A\t") for line in statuses)

# 4. Git 작성자 정보 확인
name = subprocess.check_output(
    ["git", "config", "--get", "user.name"],
    cwd=repo,
    text=True
).strip()

email = subprocess.check_output(
    ["git", "config", "--get", "user.email"],
    cwd=repo,
    text=True
).strip()

assert name == "seoyeon0905"
assert email == "ssseoyeon95@naver.com"

print("✅ Git 작성자 정보 확인")

# 5. Staging된 내용 최종 검사
subprocess.run(
    ["git", "diff", "--cached", "--check"],
    cwd=repo,
    check=True
)

# 6. AI 전용 브랜치에 Commit
subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "feat(ai): add Korean fusion search and MiniLM experiments"
    ],
    cwd=repo,
    check=True
)

# 7. 실제 Commit된 파일 확인
committed = subprocess.check_output(
    [
        "git",
        "diff-tree",
        "--no-commit-id",
        "--name-only",
        "-r",
        "HEAD"
    ],
    cwd=repo,
    text=True
).splitlines()

assert set(committed) == expected_files
assert len(committed) == 9

print("\n===== 생성된 Commit =====")
subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    check=True
)

print("\n===== 현재 Git 상태 =====")
subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True
)

print("\n✅ 41-18-3단계 Git Commit 성공!")
print("✅ AI 파일 9개 Commit 완료")
print("✅ GitHub Push 아직 실행하지 않음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-18-3단계. AI 파일 9개 Git Commit =====
✅ AI 전용 브랜치 확인: feature/korean-ai-fusion
✅ Commit 대상 AI 파일 정확히 9개
✅ Git 작성자 정보 확인

===== 생성된 Commit =====

===== 현재 Git 상태 =====

✅ 41-18-3단계 Git Commit 성공!
✅ AI 파일 9개 Commit 완료
✅ GitHub Push 아직 실행하지 않음
✅ 팀원 main 브랜치 변경 없음


In [29]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")
branch_name = "feature/korean-ai-fusion"

expected_files = {
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/README.md",
    "ai/experiments/korean_minilm/results/full_validation_metrics.json",
    "ai/experiments/KOREAN_AI_INTEGRATION.md",
}

print("===== 41-19단계. AI 전용 브랜치 GitHub Push =====")

# 1. 현재 브랜치 확인
current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert current_branch == branch_name
print("✅ 현재 브랜치:", current_branch)

# 2. Commit된 파일 확인
committed = subprocess.check_output(
    [
        "git", "diff-tree",
        "--no-commit-id",
        "--name-only",
        "-r", "HEAD"
    ],
    cwd=repo,
    text=True
).splitlines()

assert set(committed) == expected_files
assert len(committed) == 9
print("✅ AI 파일 9개 Commit 확인")

# 3. 작업 폴더가 깨끗한지 확인
status = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=repo,
    text=True
).strip()

assert not status, f"예상하지 못한 변경 사항:\n{status}"
print("✅ 작업 폴더 변경 사항 없음")

# 4. GitHub 원격 저장소 확인
remote_url = subprocess.check_output(
    ["git", "remote", "get-url", "origin"],
    cwd=repo,
    text=True
).strip()

assert remote_url in {
    "https://github.com/pinpoint-capstone/pinpoint-capstone.git",
    "git@github.com:pinpoint-capstone/pinpoint-capstone.git"
}, f"예상하지 못한 원격 저장소: {remote_url}"

print("✅ GitHub 원격 저장소 확인:", remote_url)

# 5. 원격에 같은 이름의 브랜치가 이미 있는지 확인
remote_branches = subprocess.check_output(
    [
        "git", "ls-remote",
        "--heads", "origin", branch_name
    ],
    cwd=repo,
    text=True
).strip()

assert not remote_branches, (
    "GitHub에 같은 이름의 브랜치가 이미 있습니다. "
    "덮어쓰지 않고 먼저 확인해야 합니다."
)

print("✅ GitHub에 동일한 브랜치 없음")

# 6. AI 전용 브랜치만 GitHub로 Push
print("\n===== GitHub Push 실행 =====")

subprocess.run(
    [
        "git", "push",
        "-u", "origin",
        branch_name
    ],
    cwd=repo,
    check=True
)

# 7. GitHub에 올라간 Commit 확인
local_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo,
    text=True
).strip()

remote_commit = subprocess.check_output(
    [
        "git", "ls-remote",
        "origin",
        f"refs/heads/{branch_name}"
    ],
    cwd=repo,
    text=True
).split()[0]

assert local_commit == remote_commit

print("\n✅ 41-19단계 GitHub Push 성공!")
print("✅ AI 전용 브랜치:", branch_name)
print("✅ GitHub Commit과 로컬 Commit 일치")
print("✅ AI 파일 9개 GitHub 저장 완료")
print("✅ main 브랜치에 직접 Push하지 않음")

print("\nGitHub 브랜치 주소:")
print(
    "https://github.com/pinpoint-capstone/"
    "pinpoint-capstone/tree/feature/korean-ai-fusion"
)

===== 41-19단계. AI 전용 브랜치 GitHub Push =====
✅ 현재 브랜치: feature/korean-ai-fusion
✅ AI 파일 9개 Commit 확인
✅ 작업 폴더 변경 사항 없음
✅ GitHub 원격 저장소 확인: https://github.com/pinpoint-capstone/pinpoint-capstone.git
✅ GitHub에 동일한 브랜치 없음

===== GitHub Push 실행 =====


CalledProcessError: Command '['git', 'push', '-u', 'origin', 'feature/korean-ai-fusion']' returned non-zero exit status 128.

In [30]:
from pathlib import Path
import subprocess

repo = Path("/content/pinpoint-capstone")
branch_name = "feature/korean-ai-fusion"

print("===== 41-19-1단계. GitHub Push 실패 원인 점검 =====")

# 1. 현재 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == branch_name

print("✅ 현재 브랜치:", branch)

# 2. 로컬 Commit 확인
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True
).strip()

print("✅ 로컬 Commit:", commit)

# 3. GitHub 저장소 주소 확인
remote = subprocess.check_output(
    ["git", "remote", "get-url", "origin"],
    cwd=repo,
    text=True
).strip()

assert remote == (
    "https://github.com/pinpoint-capstone/pinpoint-capstone.git"
)

print("✅ GitHub 원격 저장소 주소 정상")

# 4. GitHub 읽기 연결 검사
# 인증 입력을 요청하지 않도록 설정
print("\n===== GitHub 읽기 연결 검사 =====")

result = subprocess.run(
    [
        "git",
        "ls-remote",
        "--heads",
        "origin",
        "main"
    ],
    cwd=repo,
    capture_output=True,
    text=True,
    timeout=30,
    env={
        **__import__("os").environ,
        "GIT_TERMINAL_PROMPT": "0",
        "GCM_INTERACTIVE": "never",
    }
)

if result.returncode == 0:
    print("✅ GitHub 읽기 연결 성공")
else:
    print("⚠️ GitHub 읽기 연결 실패")
    print("오류 유형:", result.returncode)

# 5. Git 인증 도우미 설정 여부
# 실제 토큰이나 비밀번호는 출력하지 않음
print("\n===== 인증 설정 확인 =====")

helpers = subprocess.run(
    ["git", "config", "--get-all", "credential.helper"],
    cwd=repo,
    capture_output=True,
    text=True
)

print(
    "Git 인증 도우미 설정:",
    "있음" if helpers.returncode == 0 and helpers.stdout.strip()
    else "없음"
)

# 6. 작업 파일 확인
status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=repo,
    text=True
).strip()

print(
    "작업 폴더:",
    "✅ 변경 없음" if not status else "⚠️ 변경 사항 있음"
)

print("\n✅ 41-19-1단계 점검 완료!")
print("✅ Commit 삭제·수정 없음")
print("✅ Git Push 재시도 없음")
print("✅ Git 설정 변경 없음")
print("✅ main 브랜치 변경 없음")

===== 41-19-1단계. GitHub Push 실패 원인 점검 =====
✅ 현재 브랜치: feature/korean-ai-fusion
✅ 로컬 Commit: 690a47b
✅ GitHub 원격 저장소 주소 정상

===== GitHub 읽기 연결 검사 =====
✅ GitHub 읽기 연결 성공

===== 인증 설정 확인 =====
Git 인증 도우미 설정: 없음
작업 폴더: ✅ 변경 없음

✅ 41-19-1단계 점검 완료!
✅ Commit 삭제·수정 없음
✅ Git Push 재시도 없음
✅ Git 설정 변경 없음
✅ main 브랜치 변경 없음


In [31]:
from pathlib import Path
import subprocess
import os
import re

repo = Path("/content/pinpoint-capstone")
branch_name = "feature/korean-ai-fusion"

print("===== 41-19-2단계. GitHub Push 오류 진단 =====")

# 1. 현재 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == branch_name

# 2. 기존 Commit 확인
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True
).strip()

assert commit == "690a47b"

print("✅ AI 전용 브랜치 확인")
print("✅ 기존 Commit 확인:", commit)

# 3. 실제 업로드 없이 Push 가능 여부만 검사
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
env["GCM_INTERACTIVE"] = "never"

result = subprocess.run(
    [
        "git",
        "push",
        "--dry-run",
        "origin",
        branch_name
    ],
    cwd=repo,
    capture_output=True,
    text=True,
    timeout=40,
    env=env
)

# 4. 오류 메시지에서 인증정보 제거
message = (result.stdout + "\n" + result.stderr)

message = re.sub(
    r"https?://[^\s/@]+:[^\s/@]+@",
    "https://[REDACTED]@",
    message
)

message = re.sub(
    r"(gh[pousr]_|github_pat_|sk-|hf_)[A-Za-z0-9_-]+",
    "[REDACTED_TOKEN]",
    message
)

print("\n===== Push 진단 결과 =====")
print("종료 코드:", result.returncode)
print("Git 메시지:")
print(message.strip() or "(메시지 없음)")

print("\n✅ 실제 GitHub 업로드 없음 (--dry-run)")
print("✅ Git Commit 변경 없음")
print("✅ Git 설정 변경 없음")
print("✅ main 브랜치 변경 없음")

===== 41-19-2단계. GitHub Push 오류 진단 =====
✅ AI 전용 브랜치 확인
✅ 기존 Commit 확인: 690a47b

===== Push 진단 결과 =====
종료 코드: 128
Git 메시지:
fatal: could not read Username for 'https://github.com': terminal prompts disabled

✅ 실제 GitHub 업로드 없음 (--dry-run)
✅ Git Commit 변경 없음
✅ Git 설정 변경 없음
✅ main 브랜치 변경 없음


In [32]:
from pathlib import Path
import subprocess
import getpass
import os
import tempfile

repo = Path("/content/pinpoint-capstone")
branch_name = "feature/korean-ai-fusion"

print("===== 41-19-4단계. GitHub 인증 및 AI 브랜치 Push =====")

# 1. 현재 브랜치 및 Commit 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True
).strip()

assert branch == branch_name
assert commit == "690a47b"

print("✅ AI 전용 브랜치 확인")
print("✅ 기존 Commit 확인:", commit)

# 2. GitHub PAT를 화면에 표시하지 않고 입력
token = getpass.getpass("GitHub 토큰 입력 (화면에 표시되지 않음): ")

assert token.strip(), "토큰이 비어 있습니다."
token = token.strip()

# 3. 인증 정보를 일회성으로 전달하는 askpass 스크립트
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / "git_askpass.py"

    askpass.write_text(
        'import os, sys\n'
        'prompt = sys.argv[1] if len(sys.argv) > 1 else ""\n'
        'if "Username" in prompt:\n'
        '    print("seoyeon0905")\n'
        'elif "Password" in prompt:\n'
        '    print(os.environ["PINPOINT_GITHUB_PAT"])\n'
        'else:\n'
        '    sys.exit(1)\n',
        encoding="utf-8"
    )

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["PINPOINT_GITHUB_PAT"] = token

    # 4. AI 전용 브랜치만 Push
    result = subprocess.run(
        ["git", "push", "-u", "origin", branch_name],
        cwd=repo,
        env=env,
        capture_output=True,
        text=True
    )

    # 메모리의 인증 참조를 가능한 범위에서 정리
    del env["PINPOINT_GITHUB_PAT"]

del token

# 5. 오류 메시지는 토큰 노출 위험을 피하기 위해 상세 출력하지 않음
if result.returncode != 0:
    print("❌ GitHub Push 실패")
    print("종료 코드:", result.returncode)
    print("인증 또는 저장소 권한 문제일 수 있습니다.")
    print("상세 메시지는 보안상 출력하지 않았습니다.")
else:
    # 6. 원격 브랜치의 Commit 일치 검증
    local_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=repo,
        text=True
    ).strip()

    remote_info = subprocess.check_output(
        [
            "git", "ls-remote",
            "origin",
            f"refs/heads/{branch_name}"
        ],
        cwd=repo,
        text=True
    ).strip()

    assert remote_info, "원격 브랜치를 확인하지 못했습니다."

    remote_commit = remote_info.split()[0]

    assert local_commit == remote_commit, (
        "로컬 Commit과 원격 Commit이 일치하지 않습니다."
    )

    print("\n✅ GitHub Push 성공!")
    print("✅ AI 작업 브랜치:", branch_name)
    print("✅ 원격 Commit과 로컬 Commit 일치")
    print("✅ main 브랜치에 직접 Push하지 않음")

    print("\nGitHub 브랜치 주소:")
    print(
        "https://github.com/pinpoint-capstone/"
        "pinpoint-capstone/tree/feature/korean-ai-fusion"
    )

===== 41-19-4단계. GitHub 인증 및 AI 브랜치 Push =====
✅ AI 전용 브랜치 확인
✅ 기존 Commit 확인: 690a47b
GitHub 토큰 입력 (화면에 표시되지 않음): ··········
❌ GitHub Push 실패
종료 코드: 128
인증 또는 저장소 권한 문제일 수 있습니다.
상세 메시지는 보안상 출력하지 않았습니다.


In [33]:
import subprocess
from pathlib import Path

repo = Path("/content/pinpoint-capstone")

print("===== 41-19-5단계. Push 실패 원인 분석 =====")

# 1. 기존 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 기존 Commit 확인
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True
).strip()

assert commit == "690a47b"

print("✅ 작업 브랜치 정상")
print("✅ Commit 정상:", commit)

# 3. 직전 셀에서 저장한 Push 실패 결과만 읽기
assert "result" in globals(), (
    "직전 Push 실행 결과가 메모리에 없습니다."
)

assert result.returncode != 0, (
    "직전 실행 결과가 실패 상태가 아닙니다."
)

error = (result.stderr or "").lower()

# 4. 실제 오류 문장을 출력하지 않고 종류만 분류
print("\n===== 오류 원인 =====")

if "cannot exec" in error or "permission denied" in error and "askpass" in error:
    print("⚠️ 인증용 임시 스크립트를 실행하지 못했습니다.")
    print("원인 후보: 스크립트 실행 권한 또는 실행 방식 문제")

elif "exec format error" in error:
    print("⚠️ 인증용 임시 스크립트 형식 오류")
    print("원인 후보: Python 실행 명령 지정 누락")

elif "could not read username" in error or "could not read password" in error:
    print("⚠️ Git이 사용자 이름 또는 토큰을 전달받지 못했습니다.")

elif "403" in error or "write access" in error:
    print("⚠️ GitHub 저장소 쓰기 권한 또는 조직 승인 문제")

elif "authentication failed" in error or "invalid username or password" in error:
    print("⚠️ GitHub 인증 실패")
    print("토큰 권한·승인 상태 확인이 필요할 수 있습니다.")

elif "repository not found" in error:
    print("⚠️ 저장소 접근 권한 또는 인증 문제")

elif "askpass" in error:
    print("⚠️ Git 인증 스크립트 관련 오류")

else:
    print("⚠️ 위 분류에 해당하지 않는 오류")
    print("오류 메시지를 안전하게 확인할 추가 점검이 필요합니다.")

print("\n✅ Git Push 재실행 없음")
print("✅ 인증 토큰 출력 없음")
print("✅ Commit 및 파일 변경 없음")
print("✅ main 브랜치 변경 없음")

===== 41-19-5단계. Push 실패 원인 분석 =====
✅ 작업 브랜치 정상
✅ Commit 정상: 690a47b

===== 오류 원인 =====
⚠️ 인증용 임시 스크립트를 실행하지 못했습니다.
원인 후보: 스크립트 실행 권한 또는 실행 방식 문제

✅ Git Push 재실행 없음
✅ 인증 토큰 출력 없음
✅ Commit 및 파일 변경 없음
✅ main 브랜치 변경 없음


In [34]:
from pathlib import Path
import subprocess
import getpass
import tempfile
import os
import sys
import re

repo = Path("/content/pinpoint-capstone")
branch_name = "feature/korean-ai-fusion"

print("===== 41-19-6단계. GitHub 인증 수정 및 Push =====")

# 1. 브랜치와 Commit 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True
).strip()

assert branch == branch_name
assert commit == "690a47b"

# 2. 기존 작업 파일이 변경되지 않았는지 확인
status = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=repo,
    text=True
).strip()

assert not status, "예상하지 못한 파일 변경이 있습니다."

# 3. 원격 저장소 확인
remote = subprocess.check_output(
    ["git", "remote", "get-url", "origin"],
    cwd=repo,
    text=True
).strip()

assert remote == (
    "https://github.com/pinpoint-capstone/pinpoint-capstone.git"
)

print("✅ AI 전용 브랜치 확인")
print("✅ Commit 확인:", commit)
print("✅ 작업 파일 변경 없음")
print("✅ GitHub 저장소 확인")

# 4. GitHub 토큰을 화면에 표시하지 않고 입력
token = getpass.getpass(
    "\nGitHub 토큰 입력 (화면에 표시되지 않음): "
).strip()

assert token, "토큰이 비어 있습니다."

# 5. 정상 실행 가능한 임시 인증 스크립트 생성
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / "git_askpass.py"

    askpass.write_text(
        f"#!{sys.executable}\n"
        "import os\n"
        "import sys\n"
        "prompt = sys.argv[1] if len(sys.argv) > 1 else ''\n"
        "if 'Username' in prompt:\n"
        "    print('seoyeon0905')\n"
        "elif 'Password' in prompt:\n"
        "    print(os.environ['PINPOINT_GITHUB_PAT'])\n"
        "else:\n"
        "    sys.exit(1)\n",
        encoding="utf-8"
    )

    # 실행 권한 부여
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["PINPOINT_GITHUB_PAT"] = token

    print("\n===== AI 전용 브랜치 Push 실행 =====")

    result = subprocess.run(
        ["git", "push", "-u", "origin", branch_name],
        cwd=repo,
        env=env,
        capture_output=True,
        text=True,
        timeout=120
    )

    # 오류 메시지에서 인증정보 제거
    error = result.stderr or ""

    error = error.replace(token, "[REDACTED_TOKEN]")

    error = re.sub(
        r"(github_pat_|gh[pousr]_)[A-Za-z0-9_-]+",
        "[REDACTED_TOKEN]",
        error
    )

    # 임시 환경변수 참조 제거
    env.pop("PINPOINT_GITHUB_PAT", None)

del token

if result.returncode != 0:
    print("\n❌ GitHub Push 실패")
    print("종료 코드:", result.returncode)

    # 민감정보를 출력하지 않고 오류 유형만 표시
    error_lower = error.lower()

    if "403" in error_lower or "write access" in error_lower:
        print("⚠️ GitHub 저장소 쓰기 권한 또는 조직 승인 문제")

    elif "authentication failed" in error_lower:
        print("⚠️ GitHub 토큰 인증 실패")

    elif "permission denied" in error_lower:
        print("⚠️ 인증 또는 실행 권한 문제")

    elif "repository not found" in error_lower:
        print("⚠️ 저장소 접근 권한 문제")

    elif "askpass" in error_lower:
        print("⚠️ 인증 스크립트 실행 문제")

    else:
        print("⚠️ 추가 오류 확인 필요")
        print("오류 메시지는 보안상 출력하지 않았습니다.")

else:
    # 6. 실제 GitHub 브랜치에 저장됐는지 확인
    local_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=repo,
        text=True
    ).strip()

    remote_info = subprocess.check_output(
        [
            "git", "ls-remote",
            "origin",
            f"refs/heads/{branch_name}"
        ],
        cwd=repo,
        text=True
    ).strip()

    assert remote_info, "GitHub 원격 브랜치를 찾지 못했습니다."

    remote_commit = remote_info.split()[0]

    assert local_commit == remote_commit, (
        "로컬과 GitHub Commit이 일치하지 않습니다."
    )

    print("\n✅ 41-19-6단계 GitHub Push 성공!")
    print("✅ AI 파일 9개 GitHub 저장 완료")
    print("✅ 원격 Commit과 로컬 Commit 일치")
    print("✅ main 브랜치에 직접 Push하지 않음")

    print("\nGitHub 브랜치 주소:")
    print(
        "https://github.com/pinpoint-capstone/"
        "pinpoint-capstone/tree/feature/korean-ai-fusion"
    )

===== 41-19-6단계. GitHub 인증 수정 및 Push =====
✅ AI 전용 브랜치 확인
✅ Commit 확인: 690a47b
✅ 작업 파일 변경 없음
✅ GitHub 저장소 확인

GitHub 토큰 입력 (화면에 표시되지 않음): ··········

===== AI 전용 브랜치 Push 실행 =====

✅ 41-19-6단계 GitHub Push 성공!
✅ AI 파일 9개 GitHub 저장 완료
✅ 원격 Commit과 로컬 Commit 일치
✅ main 브랜치에 직접 Push하지 않음

GitHub 브랜치 주소:
https://github.com/pinpoint-capstone/pinpoint-capstone/tree/feature/korean-ai-fusion


In [35]:
from pathlib import Path
from collections import Counter

# Google Drive 기본 경로
base_dir = Path("/content/drive/MyDrive/pinpoint_capstone")

folders = [
    "02_Korean_Video_Test",
    "03_Korean_Training_Data",
]

print("=" * 80)
print("📂 Google Drive 전체 파일 목록")
print("=" * 80)

total_files = 0
total_size = 0

for folder_name in folders:
    folder_path = base_dir / folder_name

    print(f"\n{'=' * 80}")
    print(f"📁 {folder_name}")
    print(f"{'=' * 80}")

    if not folder_path.is_dir():
        print("❌ 폴더를 찾을 수 없습니다.")
        continue

    # 모든 하위 폴더 및 파일 검색
    all_paths = sorted(
        folder_path.rglob("*"),
        key=lambda p: str(p).lower()
    )

    files = [p for p in all_paths if p.is_file()]
    directories = [p for p in all_paths if p.is_dir()]

    folder_size = 0

    for path in all_paths:
        relative = path.relative_to(folder_path)
        depth = len(relative.parts) - 1
        indent = "    " * depth

        if path.is_dir():
            print(f"{indent}📁 {path.name}/")

        elif path.is_file():
            size = path.stat().st_size
            folder_size += size

            if size >= 1024**3:
                size_str = f"{size / 1024**3:.2f} GB"
            elif size >= 1024**2:
                size_str = f"{size / 1024**2:.2f} MB"
            elif size >= 1024:
                size_str = f"{size / 1024:.2f} KB"
            else:
                size_str = f"{size} B"

            print(f"{indent}📄 {path.name} ({size_str})")

    print(f"\n📌 폴더 수: {len(directories)}개")
    print(f"📌 파일 수: {len(files)}개")
    print(f"📌 전체 크기: {folder_size / 1024**2:.2f} MB")

    total_files += len(files)
    total_size += folder_size

print("\n" + "=" * 80)
print("✅ 전체 확인 완료")
print(f"✅ 총 파일 수: {total_files}개")
print(f"✅ 총 파일 크기: {total_size / 1024**3:.3f} GB")
print("=" * 80)

📂 Google Drive 전체 파일 목록

📁 02_Korean_Video_Test
📁 __pycache__/
    📄 fusion_experiment.cpython-313.pyc (6.51 KB)
📁 AI_Backup/
    📄 consensus_rerank.py (1.70 KB)
    📄 fusion_search_20261010.py (5.83 KB)
    📄 fusion_search_before_model_connection_20261010_130236.py (6.63 KB)
    📄 fusion_search_before_model_selection_20261010_130203.py (5.83 KB)
    📄 fusion_search_FINAL_restored_20261010.py (6.70 KB)
    📄 KOREAN_AI_INTEGRATION_restored_20261010.md (7.16 KB)
    📄 minilm_evaluate_restored_20261010.py (4.08 KB)
    📄 minilm_full_validation_metrics_restored_20261010.json (805 B)
    📄 minilm_prepare_data_restored_20261010.py (5.71 KB)
    📄 minilm_README_restored_20261010.md (10.01 KB)
    📄 minilm_train_restored_20261010.py (4.00 KB)
    📄 scratch_clip_align_best_ORIGINAL.ckpt (48.88 MB)
    📄 transcribe_service_before_path_fix_20261010.py (989 B)
    📄 transcribe_service_FINAL_restored_20261010.py (1.87 KB)
📁 clip_features/
    📄 kr_daiso_001_clip_features.pt (33.73 KB)
    📄 kr_kitc

In [36]:
from pathlib import Path
import json
import ast
import subprocess
from collections import Counter

repo = Path("/content/pinpoint-capstone")
drive = Path("/content/drive/MyDrive/pinpoint_capstone")

video_dir = drive / "02_Korean_Video_Test"
train_dir = drive / "03_Korean_Training_Data"

print("===== 41-22단계. 추가 업로드 자료 전체 검사 =====")

# 1. GitHub AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치:", branch)

# 2. 검사할 실험 결과 JSON 16개
json_files = [
    video_dir / "pinpoint_ai_experiment_summary.json",
    video_dir / "fusion_baseline_evaluation.json",
    video_dir / "detergent_evaluation.json",
    video_dir / "rice_washing_evaluation.json",
    video_dir / "seasoning_evaluation.json",
    video_dir / "squat_ground_truth.json",

    train_dir / "fusion_new_3queries_comparison.json",
    train_dir / "fusion_start_time_comparison.json",
    train_dir / "consensus_6queries_comparison.json",
    train_dir / "consensus_v2_12queries_comparison.json",
    train_dir / "minilm_fusion_6queries_comparison.json",
    train_dir / "squat_4methods_comparison.json",

    train_dir / "minilm_baseline_evaluation.json",
    train_dir / "minilm_finetuning_comparison.json",
    train_dir / "minilm_full_validation_comparison.json",
    train_dir / "minilm_full_validation_verification_20261010.json",
]

def describe(value):
    if isinstance(value, dict):
        return f"dict, {len(value)}개 필드"
    if isinstance(value, list):
        return f"list, {len(value)}개 항목"
    return type(value).__name__

print("\n===== 실험 JSON 구조 및 필드 =====")

for path in json_files:
    print(f"\n📄 {path.name}")

    if not path.is_file():
        print("   ❌ 파일 없음")
        continue

    data = json.loads(path.read_text(encoding="utf-8"))

    print("   크기:", path.stat().st_size, "bytes")
    print("   최상위:", describe(data))

    if isinstance(data, dict):
        for key, value in data.items():
            print(f"   - {key}: {describe(value)}")

    elif isinstance(data, list) and data:
        first = data[0]
        if isinstance(first, dict):
            print("   첫 번째 항목 필드:", sorted(first.keys()))

# 3. 노트북 구성 확인
notebook_path = video_dir / "PinPoint_Korean_Video_Test.ipynb"

print("\n===== 원본 Colab 노트북 =====")

assert notebook_path.is_file(), "원본 실험 노트북이 없습니다."

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

cells = notebook.get("cells", [])
code_cells = [
    c for c in cells
    if c.get("cell_type") == "code"
]
markdown_cells = [
    c for c in cells
    if c.get("cell_type") == "markdown"
]
output_cells = [
    c for c in code_cells
    if c.get("outputs")
]

print("파일 크기:", f"{notebook_path.stat().st_size / 1024**2:.2f} MB")
print("전체 셀:", len(cells))
print("코드 셀:", len(code_cells))
print("설명 셀:", len(markdown_cells))
print("실행 출력이 포함된 셀:", len(output_cells))

# 노트북에 포함된 잠재적으로 민감한 요소를 개수로만 확인
notebook_text = notebook_path.read_text(encoding="utf-8")

checks = {
    "GitHub 토큰 문자열 형태": ["github_pat_", "ghp_"],
    "개인 Google Drive 경로": ["/content/drive/MyDrive"],
    "환경변수·인증 관련 코드": [
        "getpass",
        "GIT_ASKPASS",
        "HF_TOKEN",
        "userdata.get",
    ],
    "영상·자막 경로": [".mp4", "subtitles.json"],
}

for label, patterns in checks.items():
    count = sum(
        notebook_text.count(pattern)
        for pattern in patterns
    )
    print(f"{label}: {count}회 발견")

# 4. 이전 Fusion 실험 스크립트 검사
script_path = video_dir / "fusion_experiment.py"

print("\n===== 이전 Fusion 실험 스크립트 =====")

assert script_path.is_file(), "fusion_experiment.py가 없습니다."

script = script_path.read_text(encoding="utf-8")
tree = ast.parse(script)

functions = [
    node.name for node in tree.body
    if isinstance(node, ast.FunctionDef)
]

print("파일 크기:", script_path.stat().st_size, "bytes")
print("정의된 함수:", functions)
print("Python 문법: ✅ 정상")

# 5. Git 상태 확인
print("\n===== 현재 Git 상태 =====")
subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n✅ 41-22단계 검사 완료!")
print("✅ Google Drive 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-22단계. 추가 업로드 자료 전체 검사 =====
✅ AI 전용 브랜치: feature/korean-ai-fusion

===== 실험 JSON 구조 및 필드 =====

📄 pinpoint_ai_experiment_summary.json
   크기: 699 bytes
   최상위: dict, 7개 필드
   - project: str
   - evaluation_type: str
   - video_count: int
   - query_count: int
   - baseline: dict, 3개 필드
   - fusion: dict, 3개 필드
   - limitations: list, 5개 항목

📄 fusion_baseline_evaluation.json
   크기: 2999 bytes
   최상위: dict, 6개 필드
   - description: str
   - evaluation_note: str
   - deduplication: str
   - ranking: str
   - ground_truth: dict, 3개 필드
   - results: dict, 3개 필드

📄 detergent_evaluation.json
   크기: 1373 bytes
   최상위: dict, 3개 필드
   - ground_truth: dict, 5개 필드
   - hybrid_results: list, 3개 항목
   - fusion_results: list, 5개 항목

📄 rice_washing_evaluation.json
   크기: 1257 bytes
   최상위: dict, 3개 필드
   - ground_truth: dict, 5개 필드
   - hybrid_results: list, 2개 항목
   - fusion_results: list, 5개 항목

📄 seasoning_evaluation.json
   크기: 1619 bytes
   최상위: dict, 3개 필드
   - ground_truth: dict, 5개 필드
 

In [37]:
from pathlib import Path
import json
import subprocess
from collections import Counter

repo = Path("/content/pinpoint-capstone")
base = Path("/content/drive/MyDrive/pinpoint_capstone")

video_dir = base / "02_Korean_Video_Test"
train_dir = base / "03_Korean_Training_Data"

print("===== 41-23단계. 실험 JSON 내부 안전성 검사 =====")

# 1. AI 작업 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"
print("✅ AI 전용 브랜치 확인")

# 2. 검사 대상 JSON 16개
files = [
    video_dir / "pinpoint_ai_experiment_summary.json",
    video_dir / "fusion_baseline_evaluation.json",
    video_dir / "detergent_evaluation.json",
    video_dir / "rice_washing_evaluation.json",
    video_dir / "seasoning_evaluation.json",
    video_dir / "squat_ground_truth.json",

    train_dir / "fusion_new_3queries_comparison.json",
    train_dir / "fusion_start_time_comparison.json",
    train_dir / "consensus_6queries_comparison.json",
    train_dir / "consensus_v2_12queries_comparison.json",
    train_dir / "minilm_fusion_6queries_comparison.json",
    train_dir / "squat_4methods_comparison.json",

    train_dir / "minilm_baseline_evaluation.json",
    train_dir / "minilm_finetuning_comparison.json",
    train_dir / "minilm_full_validation_comparison.json",
    train_dir / "minilm_full_validation_verification_20261010.json",
]

# 원문 또는 개인 데이터일 가능성이 있는 필드명
sensitive_keywords = [
    "query",
    "transcript",
    "subtitle",
    "text",
    "sentence",
    "prompt",
    "video_id",
    "filename",
    "path",
    "token",
    "password",
    "secret",
    "authorization",
]

def inspect_structure(value, prefix="", depth=0, found=None):
    if found is None:
        found = {}

    if depth > 8:
        return found

    if isinstance(value, dict):
        for key, item in value.items():
            key_str = str(key)
            location = f"{prefix}.{key_str}" if prefix else key_str

            if any(word in key_str.lower() for word in sensitive_keywords):
                found[location] = type(item).__name__

            inspect_structure(
                item,
                location,
                depth + 1,
                found
            )

    elif isinstance(value, list):
        # 반복되는 항목은 첫 3개만 구조 검사
        for item in value[:3]:
            inspect_structure(
                item,
                f"{prefix}[]",
                depth + 1,
                found
            )

    return found

print("\n===== 파일별 민감 가능 필드 =====")

for path in files:
    assert path.is_file(), f"파일 없음: {path.name}"

    data = json.loads(path.read_text(encoding="utf-8"))
    found = inspect_structure(data)

    print(f"\n📄 {path.name}")

    if not found:
        print("   ✅ 원문·인증정보 관련 필드명 발견 안 됨")
    else:
        for field, value_type in sorted(found.items()):
            print(f"   ⚠️ {field} ({value_type})")

print("\n===== 현재 Git 상태 =====")

subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n✅ 41-23단계 검사 완료!")
print("✅ JSON 실제 텍스트 값 출력 없음")
print("✅ Google Drive 파일 변경 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-23단계. 실험 JSON 내부 안전성 검사 =====
✅ AI 전용 브랜치 확인

===== 파일별 민감 가능 필드 =====

📄 pinpoint_ai_experiment_summary.json
   ⚠️ query_count (int)

📄 fusion_baseline_evaluation.json
   ✅ 원문·인증정보 관련 필드명 발견 안 됨

📄 detergent_evaluation.json
   ⚠️ ground_truth.query_en (str)
   ⚠️ ground_truth.query_ko (str)
   ⚠️ ground_truth.video_id (str)

📄 rice_washing_evaluation.json
   ⚠️ ground_truth.query_en (str)
   ⚠️ ground_truth.query_ko (str)
   ⚠️ ground_truth.video_id (str)

📄 seasoning_evaluation.json
   ⚠️ ground_truth.query_en (str)
   ⚠️ ground_truth.query_ko (str)
   ⚠️ ground_truth.video_id (str)

📄 squat_ground_truth.json
   ⚠️ [].english_query (str)
   ⚠️ [].korean_query (str)
   ⚠️ [].video_id (str)

📄 fusion_new_3queries_comparison.json
   ✅ 원문·인증정보 관련 필드명 발견 안 됨

📄 fusion_start_time_comparison.json
   ✅ 원문·인증정보 관련 필드명 발견 안 됨

📄 consensus_6queries_comparison.json
   ⚠️ results[].predictions[].query (str)
   ⚠️ results[].predictions[].subtitle_score (NoneType)
   ⚠️ results[].predictio

In [38]:
from pathlib import Path
from collections import defaultdict
import json
import subprocess

repo = Path("/content/pinpoint-capstone")
base = Path("/content/drive/MyDrive/pinpoint_capstone")

video_dir = base / "02_Korean_Video_Test"
train_dir = base / "03_Korean_Training_Data"

print("===== 41-24단계. 실험 결과 필드 전체 검사 =====")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

files = [
    video_dir / "pinpoint_ai_experiment_summary.json",
    video_dir / "fusion_baseline_evaluation.json",
    video_dir / "detergent_evaluation.json",
    video_dir / "rice_washing_evaluation.json",
    video_dir / "seasoning_evaluation.json",
    video_dir / "squat_ground_truth.json",

    train_dir / "fusion_new_3queries_comparison.json",
    train_dir / "fusion_start_time_comparison.json",
    train_dir / "consensus_6queries_comparison.json",
    train_dir / "consensus_v2_12queries_comparison.json",
    train_dir / "minilm_fusion_6queries_comparison.json",
    train_dir / "squat_4methods_comparison.json",

    train_dir / "minilm_baseline_evaluation.json",
    train_dir / "minilm_finetuning_comparison.json",
    train_dir / "minilm_full_validation_comparison.json",
    train_dir / "minilm_full_validation_verification_20261010.json",
]

def collect_fields(value, prefix="", fields=None):
    if fields is None:
        fields = defaultdict(set)

    if isinstance(value, dict):
        for key, item in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            fields[path].add(type(item).__name__)
            collect_fields(item, path, fields)

    elif isinstance(value, list):
        for item in value:
            collect_fields(item, f"{prefix}[]", fields)

    return fields

for path in files:
    assert path.is_file(), f"파일 없음: {path.name}"

    data = json.loads(path.read_text(encoding="utf-8"))
    fields = collect_fields(data)

    print("\n" + "=" * 65)
    print("📄", path.name)
    print("필드 경로:", len(fields), "개")

    for field, types in sorted(fields.items()):
        print(f"  {field}: {', '.join(sorted(types))}")

print("\n===== Git 상태 =====")

subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n✅ 41-24단계 완료!")
print("✅ 실제 검색어·자막·영상 ID 값 출력 없음")
print("✅ 파일 생성·수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ main 브랜치 변경 없음")

===== 41-24단계. 실험 결과 필드 전체 검사 =====

📄 pinpoint_ai_experiment_summary.json
필드 경로: 13 개
  baseline: dict
  baseline.mean_best_iou_reference: float
  baseline.mean_top1_iou: float
  baseline.model: str
  evaluation_type: str
  fusion: dict
  fusion.mean_best_iou_top5: float
  fusion.mean_top1_iou: float
  fusion.model: str
  limitations: list
  project: str
  query_count: int
  video_count: int

📄 fusion_baseline_evaluation.json
필드 경로: 36 개
  deduplication: str
  description: str
  evaluation_note: str
  ground_truth: dict
  ground_truth.다이소: dict
  ground_truth.다이소.end_time: float
  ground_truth.다이소.start_time: float
  ground_truth.두부: dict
  ground_truth.두부.end_time: float
  ground_truth.두부.start_time: float
  ground_truth.주방: dict
  ground_truth.주방.end_time: float
  ground_truth.주방.start_time: float
  ranking: str
  results: dict
  results.다이소: list
  results.다이소[].end_time: float
  results.다이소[].iou: float
  results.다이소[].language: NoneType, str
  results.다이소[].rank: int
  results.다이

In [39]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import subprocess

repo = Path("/content/pinpoint-capstone")
base = Path("/content/drive/MyDrive/pinpoint_capstone")

video_dir = base / "02_Korean_Video_Test"
train_dir = base / "03_Korean_Training_Data"

print("===== 41-25단계. 실험 원본 보존 및 수치 검사 =====")

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

files = [
    video_dir / "pinpoint_ai_experiment_summary.json",
    video_dir / "fusion_baseline_evaluation.json",
    video_dir / "detergent_evaluation.json",
    video_dir / "rice_washing_evaluation.json",
    video_dir / "seasoning_evaluation.json",
    video_dir / "squat_ground_truth.json",

    train_dir / "fusion_new_3queries_comparison.json",
    train_dir / "fusion_start_time_comparison.json",
    train_dir / "consensus_6queries_comparison.json",
    train_dir / "consensus_v2_12queries_comparison.json",
    train_dir / "minilm_fusion_6queries_comparison.json",
    train_dir / "squat_4methods_comparison.json",

    train_dir / "minilm_baseline_evaluation.json",
    train_dir / "minilm_finetuning_comparison.json",
    train_dir / "minilm_full_validation_comparison.json",
    train_dir / "minilm_full_validation_verification_20261010.json",
]

def count_values(value, counts):
    if isinstance(value, dict):
        counts["dict"] += 1

        for item in value.values():
            count_values(item, counts)

    elif isinstance(value, list):
        counts["list"] += 1

        for item in value:
            count_values(item, counts)

    elif isinstance(value, bool):
        counts["bool"] += 1

    elif isinstance(value, (int, float)):
        counts["numeric"] += 1

    elif isinstance(value, str):
        counts["text"] += 1

    elif value is None:
        counts["null"] += 1

    else:
        counts["other"] += 1

total = Counter()
manifest = []

for path in files:
    assert path.is_file(), f"원본 파일 없음: {path.name}"

    raw = path.read_bytes()
    data = json.loads(raw.decode("utf-8"))

    counts = Counter()
    count_values(data, counts)
    total.update(counts)

    digest = hashlib.sha256(raw).hexdigest()

    manifest.append({
        "relative_source": str(path.relative_to(base)),
        "sha256": digest,
        "size_bytes": len(raw),
        "value_counts": dict(counts),
    })

    print(f"\n📄 {path.name}")
    print("  숫자 값:", counts["numeric"], "개")
    print("  문자열 값:", counts["text"], "개")
    print("  null 값:", counts["null"], "개")
    print("  파일 크기:", len(raw), "bytes")
    print("  SHA-256:", digest[:16] + "...")

assert len(manifest) == 16

print("\n===== 전체 합계 =====")
print("검사 파일:", len(manifest), "개")
print("숫자 값:", total["numeric"], "개")
print("문자열 값:", total["text"], "개")
print("null 값:", total["null"], "개")

print("\n===== Git 상태 =====")
subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n✅ 41-25단계 검사 완료!")
print("✅ 원본 JSON 16개 모두 확인")
print("✅ 원본 파일별 SHA-256 계산")
print("✅ 검색어·자막 실제 내용 출력 없음")
print("✅ 파일 생성·수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-25단계. 실험 원본 보존 및 수치 검사 =====

📄 pinpoint_ai_experiment_summary.json
  숫자 값: 6 개
  문자열 값: 9 개
  null 값: 0 개
  파일 크기: 699 bytes
  SHA-256: 2c990d10efb2c8be...

📄 fusion_baseline_evaluation.json
  숫자 값: 62 개
  문자열 값: 26 개
  null 값: 6 개
  파일 크기: 2999 bytes
  SHA-256: 1060d39d2935c919...

📄 detergent_evaluation.json
  숫자 값: 34 개
  문자열 값: 12 개
  null 값: 1 개
  파일 크기: 1373 bytes
  SHA-256: b6fd8ee6be561f2e...

📄 rice_washing_evaluation.json
  숫자 값: 30 개
  문자열 값: 11 개
  null 값: 2 개
  파일 크기: 1257 bytes
  SHA-256: 2c0cd666dfc4e682...

📄 seasoning_evaluation.json
  숫자 값: 42 개
  문자열 값: 11 개
  null 값: 2 개
  파일 크기: 1619 bytes
  SHA-256: cf73b4ea6350ddf3...

📄 squat_ground_truth.json
  숫자 값: 6 개
  문자열 값: 9 개
  null 값: 0 개
  파일 크기: 650 bytes
  SHA-256: 6b72438ccc38dd29...

📄 fusion_new_3queries_comparison.json
  숫자 값: 49 개
  문자열 값: 19 개
  null 값: 0 개
  파일 크기: 2572 bytes
  SHA-256: a03771138c8db0a1...

📄 fusion_start_time_comparison.json
  숫자 값: 36 개
  문자열 값: 18 개
  null 값: 0 개
  파일 크기: 1838 by

In [40]:
from pathlib import Path
from collections import Counter, defaultdict
import subprocess
import json
import re
import hashlib

repo = Path("/content/pinpoint-capstone")
base = Path("/content/drive/MyDrive/pinpoint_capstone")

video_dir = base / "02_Korean_Video_Test"
train_dir = base / "03_Korean_Training_Data"

target_dir = repo / "ai/experiments/korean_video/results"

backup_dir = (
    video_dir
    / "AI_Backup"
    / "public_experiment_results_20261010"
)

print("===== 41-26단계. 실험 결과 JSON 공개용 복사본 생성 =====")

# 1. AI 전용 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. 원본 16개 지정
sources = [
    video_dir / "pinpoint_ai_experiment_summary.json",
    video_dir / "fusion_baseline_evaluation.json",
    video_dir / "detergent_evaluation.json",
    video_dir / "rice_washing_evaluation.json",
    video_dir / "seasoning_evaluation.json",
    video_dir / "squat_ground_truth.json",

    train_dir / "fusion_new_3queries_comparison.json",
    train_dir / "fusion_start_time_comparison.json",
    train_dir / "consensus_6queries_comparison.json",
    train_dir / "consensus_v2_12queries_comparison.json",
    train_dir / "minilm_fusion_6queries_comparison.json",
    train_dir / "squat_4methods_comparison.json",

    train_dir / "minilm_baseline_evaluation.json",
    train_dir / "minilm_finetuning_comparison.json",
    train_dir / "minilm_full_validation_comparison.json",
    train_dir / "minilm_full_validation_verification_20261010.json",
]

assert len(sources) == 16
assert len({p.name for p in sources}) == 16

# 3. 기존 파일 덮어쓰기 방지
for source in sources:
    assert source.is_file(), f"원본 없음: {source.name}"

    assert not (target_dir / source.name).exists(), (
        f"GitHub용 파일이 이미 있습니다: {source.name}"
    )

    assert not (backup_dir / source.name).exists(), (
        f"Drive 백업이 이미 있습니다: {source.name}"
    )

# 4. 모델 및 후보 출처에 해당하는 필드만 제한적으로 보존
category_keys = {
    "model",
    "source",
    "rank_source",
    "language",
    "hybrid_source",
}

safe_category_pattern = re.compile(
    r"^[\w가-힣 .()+-]{1,60}$"
)

# 5. 자유 텍스트는 원문 대신 일관된 익명 ID로 변환
# 같은 종류의 동일 문자열에는 같은 ID를 사용
anonymous_maps = defaultdict(dict)

def anonymize(value, category):
    mapping = anonymous_maps[category]

    if value not in mapping:
        mapping[value] = (
            f"{category}_{len(mapping) + 1:03d}"
        )

    return mapping[value]

def sanitize(value, field_name=None):
    if isinstance(value, dict):
        return {
            key: sanitize(item, str(key))
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            sanitize(item, field_name)
            for item in value
        ]

    # 숫자, Boolean, null은 원본 그대로 보존
    if value is None or isinstance(value, (int, float, bool)):
        return value

    if isinstance(value, str):
        # 모델 종류 및 후보 출처처럼 분석에 필요한 범주값
        if field_name in category_keys:
            if safe_category_pattern.fullmatch(value):
                return value

            return anonymize(value, "category")

        if field_name in {"video_id", "filename"}:
            return anonymize(value, "video")

        if field_name in {"name"}:
            return anonymize(value, "case")

        if field_name in {
            "query",
            "query_ko",
            "query_en",
            "korean_query",
            "english_query",
        }:
            return anonymize(value, "query")

        # 자막, 설명문, 경로 및 기타 자유 텍스트 보호
        return "[REDACTED]"

    raise TypeError(f"지원하지 않는 데이터 형식: {type(value)}")

# 6. 숫자·null 값이 원본과 완전히 동일한지 재귀 검증
def verify_numeric_values(original, public):
    if isinstance(original, dict):
        assert isinstance(public, dict)
        assert original.keys() == public.keys()

        for key in original:
            verify_numeric_values(
                original[key],
                public[key]
            )

    elif isinstance(original, list):
        assert isinstance(public, list)
        assert len(original) == len(public)

        for a, b in zip(original, public):
            verify_numeric_values(a, b)

    elif original is None:
        assert public is None

    elif isinstance(original, bool):
        assert type(public) is bool
        assert original == public

    elif isinstance(original, (int, float)):
        assert type(public) is type(original)
        assert original == public

    else:
        assert isinstance(public, str)

# 7. 원본 전체를 먼저 메모리에서 검사
prepared = []

for source in sources:
    original = json.loads(
        source.read_text(encoding="utf-8")
    )

    public = sanitize(original)
    verify_numeric_values(original, public)

    content = (
        json.dumps(
            public,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        )
        + "\n"
    )

    prepared.append((source, content))

print("✅ 원본 JSON 16개 읽기 완료")
print("✅ 수치·순위·후보 구조 보존 검증 완료")

# 8. 공개용 폴더와 Drive 백업 폴더 생성
target_dir.mkdir(parents=True, exist_ok=True)
backup_dir.mkdir(parents=True, exist_ok=True)

# 9. 파일 각각 새로 생성 (기존 파일 덮어쓰기 없음)
for source, content in prepared:
    target = target_dir / source.name
    backup = backup_dir / source.name

    with target.open("x", encoding="utf-8") as f:
        f.write(content)

    with backup.open("x", encoding="utf-8") as f:
        f.write(content)

    # 원본 수치 재검증
    original = json.loads(source.read_text(encoding="utf-8"))
    restored = json.loads(target.read_text(encoding="utf-8"))

    verify_numeric_values(original, restored)

    # 복사본과 백업 파일 완전 일치 확인
    target_hash = hashlib.sha256(
        target.read_bytes()
    ).hexdigest()

    backup_hash = hashlib.sha256(
        backup.read_bytes()
    ).hexdigest()

    assert target_hash == backup_hash

    print(f"✅ {source.name}")

print("\n===== 생성 결과 =====")
print("GitHub 업로드 예정 폴더:", target_dir)
print("Google Drive 백업 폴더:", backup_dir)
print("생성된 파일:", len(prepared), "개")

print("\n===== Git 변경 사항 =====")
subprocess.run(
    [
        "git",
        "status",
        "--short",
        "--untracked-files=all"
    ],
    cwd=repo,
    check=True
)

print("\n✅ 41-26단계 완료!")
print("✅ 실험 결과 JSON 16개 복사본 생성")
print("✅ 숫자·점수·시간 구간·순위 보존")
print("✅ 검색어·자막·영상 ID 익명화")
print("✅ Google Drive 원본 변경 없음")
print("✅ Drive 백업과 동일성 검증")
print("✅ Git add/commit/push 없음")
print("✅ 팀원 main 브랜치 변경 없음")

===== 41-26단계. 실험 결과 JSON 공개용 복사본 생성 =====
✅ 원본 JSON 16개 읽기 완료
✅ 수치·순위·후보 구조 보존 검증 완료
✅ pinpoint_ai_experiment_summary.json
✅ fusion_baseline_evaluation.json
✅ detergent_evaluation.json
✅ rice_washing_evaluation.json
✅ seasoning_evaluation.json
✅ squat_ground_truth.json
✅ fusion_new_3queries_comparison.json
✅ fusion_start_time_comparison.json
✅ consensus_6queries_comparison.json
✅ consensus_v2_12queries_comparison.json
✅ minilm_fusion_6queries_comparison.json
✅ squat_4methods_comparison.json
✅ minilm_baseline_evaluation.json
✅ minilm_finetuning_comparison.json
✅ minilm_full_validation_comparison.json
✅ minilm_full_validation_verification_20261010.json

===== 생성 결과 =====
GitHub 업로드 예정 폴더: /content/pinpoint-capstone/ai/experiments/korean_video/results
Google Drive 백업 폴더: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/public_experiment_results_20261010
생성된 파일: 16 개

===== Git 변경 사항 =====

✅ 41-26단계 완료!
✅ 실험 결과 JSON 16개 복사본 생성
✅ 숫자·점수·시간 구간·순위 보존
✅ 검색어·자막·영상 ID 익명화

In [41]:
from pathlib import Path
from collections import Counter
import subprocess
import json
import hashlib
import re

repo = Path("/content/pinpoint-capstone")

public_dir = (
    repo / "ai/experiments/korean_video/results"
)

backup_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "public_experiment_results_20261010"
)

print("===== 41-27단계. 공개용 JSON 최종 구조 검사 =====")

# 1. 브랜치 확인
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo,
    text=True
).strip()

assert branch == "feature/korean-ai-fusion"

# 2. JSON 16개 확인
files = sorted(public_dir.glob("*.json"))
assert len(files) == 16, f"예상 파일 16개, 실제 {len(files)}개"

# 자유 텍스트가 들어 있을 가능성이 높은 필드
private_fields = {
    "query",
    "query_ko",
    "query_en",
    "korean_query",
    "english_query",
    "text",
    "transcript",
    "transcription",
    "subtitle",
    "video_id",
    "filename",
    "name",
}

issues = []

def inspect(value, location, filename):
    if isinstance(value, dict):
        for key, item in value.items():
            key_str = str(key)

            # 숫자나 순수 영문 필드명이 아닌 키 검사
            if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key_str):
                issues.append(
                    (filename, "특수·한글 키", location + "." + key_str)
                )

            inspect(item, location + "." + key_str, filename)

    elif isinstance(value, list):
        for item in value:
            inspect(item, location + "[]", filename)

    elif isinstance(value, str):
        field_name = location.split(".")[-1]

        if field_name in private_fields:
            if not (
                value == "[REDACTED]"
                or re.fullmatch(
                    r"(video|case|query)_[0-9]{3}",
                    value
                )
            ):
                issues.append(
                    (filename, "원문 가능 값", location)
                )

        # 인증 토큰처럼 보이는 패턴 검사
        if re.search(
            r"(github_pat_|ghp_|gho_|hf_)[A-Za-z0-9_-]{12,}",
            value
        ):
            issues.append(
                (filename, "인증정보 의심", location)
            )

# 3. 전체 검사
for path in files:
    data = json.loads(path.read_text(encoding="utf-8"))
    backup = backup_dir / path.name

    assert backup.is_file(), f"백업 없음: {path.name}"

    assert hashlib.sha256(path.read_bytes()).digest() == (
        hashlib.sha256(backup.read_bytes()).digest()
    ), f"Drive 백업 불일치: {path.name}"

    inspect(data, "", path.name)

    print(f"✅ {path.name}")

print("\n===== 추가 확인이 필요한 항목 =====")

if issues:
    for filename, kind, location in issues:
        print(f"⚠️ {filename} | {kind} | {location}")

    print(f"\n⚠️ 총 {len(issues)}개 항목 추가 검토 필요")
else:
    print("✅ 지정한 원문·식별자·인증정보 패턴 이상 없음")

print("\n===== Git 상태 =====")
subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=repo,
    check=True
)

print("\n✅ 41-27단계 검사 완료!")
print("✅ 실제 검색어·자막 내용 출력 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ main 브랜치 변경 없음")

===== 41-27단계. 공개용 JSON 최종 구조 검사 =====
✅ consensus_6queries_comparison.json
✅ consensus_v2_12queries_comparison.json
✅ detergent_evaluation.json
✅ fusion_baseline_evaluation.json
✅ fusion_new_3queries_comparison.json
✅ fusion_start_time_comparison.json
✅ minilm_baseline_evaluation.json
✅ minilm_finetuning_comparison.json
✅ minilm_full_validation_comparison.json
✅ minilm_full_validation_verification_20261010.json
✅ minilm_fusion_6queries_comparison.json
✅ pinpoint_ai_experiment_summary.json
✅ rice_washing_evaluation.json
✅ seasoning_evaluation.json
✅ squat_4methods_comparison.json
✅ squat_ground_truth.json

===== 추가 확인이 필요한 항목 =====
⚠️ fusion_baseline_evaluation.json | 특수·한글 키 | .ground_truth.두부
⚠️ fusion_baseline_evaluation.json | 특수·한글 키 | .ground_truth.주방
⚠️ fusion_baseline_evaluation.json | 특수·한글 키 | .ground_truth.다이소
⚠️ fusion_baseline_evaluation.json | 특수·한글 키 | .results.두부
⚠️ fusion_baseline_evaluation.json | 특수·한글 키 | .results.주방
⚠️ fusion_baseline_evaluation.json | 특수·한글 키 | .r

In [42]:
# ===== 41-28단계. 공개용 JSON 민감정보 최종 검사 =====

import json
import re
from pathlib import Path

results_dir = Path(
    "/content/pinpoint-capstone/ai/experiments/korean_video/results"
)

patterns = {
    "이메일 주소": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    "Windows 로컬 경로": r"[A-Za-z]:\\(?:Users|Documents|Desktop)\\",
    "Colab/Drive 절대 경로": r"/content/(?:drive|pinpoint-capstone)",
    "인증 토큰 의심": r"(?i)(?:api[_-]?key|access[_-]?token|secret[_-]?key|password)\s*[\"']?\s*[:=]",
    "개인키": r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----",
}

files = sorted(results_dir.glob("*.json"))

print("===== 41-28단계. 공개용 JSON 민감정보 검사 =====")
print(f"검사 대상: {len(files)}개\n")

total_warnings = 0

if len(files) != 16:
    print("⚠️ 예상 파일 수(16개)와 다릅니다. 확인 필요!")

for file in files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        content = json.dumps(data, ensure_ascii=False)

        detected = []

        for label, pattern in patterns.items():
            matches = re.findall(pattern, content)

            if matches:
                detected.append((label, len(matches)))

        if detected:
            print(f"⚠️ {file.name}")

            for label, count in detected:
                print(f"   - {label}: {count}건")
                total_warnings += count
        else:
            print(f"✅ {file.name}")

    except Exception as e:
        print(f"❌ {file.name}: 검사 실패 ({type(e).__name__})")

print("\n===== 최종 결과 =====")
print(f"탐지 건수: {total_warnings}건")

if total_warnings == 0 and len(files) == 16:
    print("✅ 지정한 민감정보 패턴에서 탐지된 항목 없음")
else:
    print("⚠️ 추가 검토 필요")

print("\n✅ 파일 수정·삭제 없음")
print("✅ 민감정보 원문 출력 없음")
print("✅ Git add/commit/push 없음")

===== 41-28단계. 공개용 JSON 민감정보 검사 =====
검사 대상: 16개

✅ consensus_6queries_comparison.json
✅ consensus_v2_12queries_comparison.json
✅ detergent_evaluation.json
✅ fusion_baseline_evaluation.json
✅ fusion_new_3queries_comparison.json
✅ fusion_start_time_comparison.json
✅ minilm_baseline_evaluation.json
✅ minilm_finetuning_comparison.json
✅ minilm_full_validation_comparison.json
✅ minilm_full_validation_verification_20261010.json
✅ minilm_fusion_6queries_comparison.json
✅ pinpoint_ai_experiment_summary.json
✅ rice_washing_evaluation.json
✅ seasoning_evaluation.json
✅ squat_4methods_comparison.json
✅ squat_ground_truth.json

===== 최종 결과 =====
탐지 건수: 0건
✅ 지정한 민감정보 패턴에서 탐지된 항목 없음

✅ 파일 수정·삭제 없음
✅ 민감정보 원문 출력 없음
✅ Git add/commit/push 없음


In [43]:
# ===== 41-29단계. 공개용 JSON Git 업로드 전 상태 확인 =====

import subprocess
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")
results_dir = repo_dir / "ai/experiments/korean_video/results"

print("===== 41-29단계. Git 업로드 전 상태 확인 =====")

# 1. 현재 브랜치 확인
branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

print("\n[1] 현재 브랜치")
print(branch.stdout.strip() or "(확인되지 않음)")

# 2. 공개용 JSON 파일 개수
files = sorted(results_dir.glob("*.json"))

print("\n[2] 공개용 JSON 파일")
print(f"파일 개수: {len(files)}개")

for file in files:
    print(f"✅ {file.name}")

# 3. Git 상태 확인
status = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

print("\n[3] Git 변경 상태")
print(status.stdout.strip() or "변경사항 없음")

# 4. Git 추적·무시 여부 확인
print("\n[4] 공개용 JSON Git 인식 상태")

for file in files:
    relative_path = file.relative_to(repo_dir).as_posix()

    tracked = subprocess.run(
        ["git", "ls-files", "--error-unmatch", "--", relative_path],
        cwd=repo_dir,
        capture_output=True,
        text=True
    )

    if tracked.returncode == 0:
        print(f"✅ 추적 중: {file.name}")
        continue

    ignored = subprocess.run(
        ["git", "check-ignore", "-q", "--", relative_path],
        cwd=repo_dir
    )

    if ignored.returncode == 0:
        print(f"⚠️ Git 무시됨: {file.name}")
    else:
        print(f"✅ 새 파일로 추가 가능: {file.name}")

print("\n===== 검사 완료 =====")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 브랜치 변경 없음")

===== 41-29단계. Git 업로드 전 상태 확인 =====

[1] 현재 브랜치
feature/korean-ai-fusion

[2] 공개용 JSON 파일
파일 개수: 16개
✅ consensus_6queries_comparison.json
✅ consensus_v2_12queries_comparison.json
✅ detergent_evaluation.json
✅ fusion_baseline_evaluation.json
✅ fusion_new_3queries_comparison.json
✅ fusion_start_time_comparison.json
✅ minilm_baseline_evaluation.json
✅ minilm_finetuning_comparison.json
✅ minilm_full_validation_comparison.json
✅ minilm_full_validation_verification_20261010.json
✅ minilm_fusion_6queries_comparison.json
✅ pinpoint_ai_experiment_summary.json
✅ rice_washing_evaluation.json
✅ seasoning_evaluation.json
✅ squat_4methods_comparison.json
✅ squat_ground_truth.json

[3] Git 변경 상태
?? ai/experiments/korean_video/

[4] 공개용 JSON Git 인식 상태
✅ 새 파일로 추가 가능: consensus_6queries_comparison.json
✅ 새 파일로 추가 가능: consensus_v2_12queries_comparison.json
✅ 새 파일로 추가 가능: detergent_evaluation.json
✅ 새 파일로 추가 가능: fusion_baseline_evaluation.json
✅ 새 파일로 추가 가능: fusion_new_3queries_comparison.json
✅ 새 파일로 추가

In [44]:
# ===== 41-30단계. Git 업로드 대상 파일 최종 확인 =====

from pathlib import Path
import subprocess

repo_dir = Path("/content/pinpoint-capstone")
target_dir = repo_dir / "ai/experiments/korean_video"
results_dir = target_dir / "results"

print("===== 41-30단계. 업로드 대상 파일 확인 =====")

# 1. 전체 폴더의 파일 목록 (경로와 크기만 출력)
all_files = sorted(
    p for p in target_dir.rglob("*") if p.is_file()
)

print(f"\n전체 파일 수: {len(all_files)}개")

# 2. 공개용 JSON과 그 외 파일 구분
public_json = []
other_files = []

for file in all_files:
    relative = file.relative_to(repo_dir).as_posix()

    if file.parent == results_dir and file.suffix == ".json":
        public_json.append(file)
    else:
        other_files.append(file)

print(f"공개용 JSON: {len(public_json)}개")
print(f"그 외 파일: {len(other_files)}개")

print("\n[1] 공개용 JSON")
for file in public_json:
    print(f"✅ {file.name} | {file.stat().st_size:,} bytes")

print("\n[2] 그 외 파일")
if other_files:
    for file in other_files:
        relative = file.relative_to(repo_dir).as_posix()
        print(f"⚠️ {relative} | {file.stat().st_size:,} bytes")
else:
    print("✅ 없음")

# 3. 업로드 예정 JSON 총 용량
total_size = sum(f.stat().st_size for f in public_json)

print("\n[3] JSON 총 용량")
print(f"{total_size:,} bytes ({total_size / 1024 / 1024:.2f} MB)")

print("\n===== 검사 완료 =====")
print("✅ 파일 내용 출력 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ 브랜치 변경 없음")

===== 41-30단계. 업로드 대상 파일 확인 =====

전체 파일 수: 16개
공개용 JSON: 16개
그 외 파일: 0개

[1] 공개용 JSON
✅ consensus_6queries_comparison.json | 10,866 bytes
✅ consensus_v2_12queries_comparison.json | 23,696 bytes
✅ detergent_evaluation.json | 1,304 bytes
✅ fusion_baseline_evaluation.json | 2,867 bytes
✅ fusion_new_3queries_comparison.json | 2,441 bytes
✅ fusion_start_time_comparison.json | 1,728 bytes
✅ minilm_baseline_evaluation.json | 203 bytes
✅ minilm_finetuning_comparison.json | 279 bytes
✅ minilm_full_validation_comparison.json | 313 bytes
✅ minilm_full_validation_verification_20261010.json | 313 bytes
✅ minilm_fusion_6queries_comparison.json | 22,625 bytes
✅ pinpoint_ai_experiment_summary.json | 451 bytes
✅ rice_washing_evaluation.json | 1,239 bytes
✅ seasoning_evaluation.json | 1,566 bytes
✅ squat_4methods_comparison.json | 21,935 bytes
✅ squat_ground_truth.json | 439 bytes

[2] 그 외 파일
✅ 없음

[3] JSON 총 용량
92,265 bytes (0.09 MB)

===== 검사 완료 =====
✅ 파일 내용 출력 없음
✅ 파일 수정·삭제 없음
✅ Git add/commit/push

In [45]:
# ===== 41-31단계. 공개용 JSON 16개 Git Staging =====

from pathlib import Path
import subprocess

repo_dir = Path("/content/pinpoint-capstone")
results_dir = repo_dir / "ai/experiments/korean_video/results"

files = sorted(results_dir.glob("*.json"))

print("===== 41-31단계. Git Staging =====")

# 1. 브랜치 및 파일 개수 확인
branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

if branch != "feature/korean-ai-fusion":
    raise RuntimeError(f"예상과 다른 브랜치: {branch}")

if len(files) != 16:
    raise RuntimeError(f"예상과 다른 JSON 파일 수: {len(files)}")

print(f"✅ 브랜치: {branch}")
print(f"✅ JSON 파일: {len(files)}개")

# 2. 공개용 JSON 16개만 Staging
relative_paths = [
    f.relative_to(repo_dir).as_posix()
    for f in files
]

subprocess.run(
    ["git", "add", "--", *relative_paths],
    cwd=repo_dir,
    check=True
)

# 3. Staging 결과 확인
result = subprocess.run(
    ["git", "diff", "--cached", "--name-status"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True
)

print("\n[Staging 완료 파일]")
print(result.stdout)

print("===== 41-31단계 완료 =====")
print("✅ Git Staging 실행")
print("✅ Git commit/push 없음")
print("✅ main 브랜치 변경 없음")
print("✅ 원본 파일 수정·삭제 없음")

===== 41-31단계. Git Staging =====
✅ 브랜치: feature/korean-ai-fusion
✅ JSON 파일: 16개

[Staging 완료 파일]
A	ai/experiments/korean_video/results/consensus_6queries_comparison.json
A	ai/experiments/korean_video/results/consensus_v2_12queries_comparison.json
A	ai/experiments/korean_video/results/detergent_evaluation.json
A	ai/experiments/korean_video/results/fusion_baseline_evaluation.json
A	ai/experiments/korean_video/results/fusion_new_3queries_comparison.json
A	ai/experiments/korean_video/results/fusion_start_time_comparison.json
A	ai/experiments/korean_video/results/minilm_baseline_evaluation.json
A	ai/experiments/korean_video/results/minilm_finetuning_comparison.json
A	ai/experiments/korean_video/results/minilm_full_validation_comparison.json
A	ai/experiments/korean_video/results/minilm_full_validation_verification_20261010.json
A	ai/experiments/korean_video/results/minilm_fusion_6queries_comparison.json
A	ai/experiments/korean_video/results/pinpoint_ai_experiment_summary.json
A	ai/experiment

In [46]:
# ===== 41-32단계. 공개용 JSON 16개 Git Commit =====

from pathlib import Path
import subprocess

repo_dir = Path("/content/pinpoint-capstone")
expected_dir = "ai/experiments/korean_video/results/"

print("===== 41-32단계. Git Commit =====")

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# 1. 현재 브랜치 확인
branch = git("branch", "--show-current")

if branch != "feature/korean-ai-fusion":
    raise RuntimeError(f"예상과 다른 브랜치: {branch}")

print(f"✅ 현재 브랜치: {branch}")

# 2. Staging된 파일 확인
staged = git("diff", "--cached", "--name-only").splitlines()

if len(staged) != 16:
    raise RuntimeError(
        f"Staging 파일 수가 16개가 아닙니다: {len(staged)}개"
    )

if not all(
    path.startswith(expected_dir) and path.endswith(".json")
    for path in staged
):
    raise RuntimeError("예상하지 못한 파일이 Staging되어 있습니다.")

# 새 파일 추가 상태인지 확인
statuses = git("diff", "--cached", "--name-status").splitlines()

if not all(line.startswith("A\t") for line in statuses):
    raise RuntimeError("새 파일 추가 외의 변경사항이 포함되어 있습니다.")

print("✅ 공개용 JSON 16개만 Staging된 상태 확인")

# 3. Git Commit
message = "docs: add Korean video AI experiment results"

commit = subprocess.run(
    ["git", "commit", "-m", message],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

if commit.returncode != 0:
    print(commit.stdout)
    print(commit.stderr)
    raise RuntimeError("Git Commit 실패")

print("\n[Commit 결과]")
print(commit.stdout)

# 4. 최종 확인
print("\n[최신 Commit]")
print(git("log", "-1", "--format=%h %s"))

print("\n[현재 Git 상태]")
print(git("status", "--short") or "변경사항 없음")

print("\n===== 41-32단계 완료 =====")
print("✅ 공개용 JSON 16개 Commit 완료")
print("✅ Git push 없음")
print("✅ main 브랜치 변경 없음")
print("✅ 원본 파일 수정·삭제 없음")

===== 41-32단계. Git Commit =====
✅ 현재 브랜치: feature/korean-ai-fusion
✅ 공개용 JSON 16개만 Staging된 상태 확인

[Commit 결과]
[feature/korean-ai-fusion 5b7df89] docs: add Korean video AI experiment results
 16 files changed, 3483 insertions(+)
 create mode 100644 ai/experiments/korean_video/results/consensus_6queries_comparison.json
 create mode 100644 ai/experiments/korean_video/results/consensus_v2_12queries_comparison.json
 create mode 100644 ai/experiments/korean_video/results/detergent_evaluation.json
 create mode 100644 ai/experiments/korean_video/results/fusion_baseline_evaluation.json
 create mode 100644 ai/experiments/korean_video/results/fusion_new_3queries_comparison.json
 create mode 100644 ai/experiments/korean_video/results/fusion_start_time_comparison.json
 create mode 100644 ai/experiments/korean_video/results/minilm_baseline_evaluation.json
 create mode 100644 ai/experiments/korean_video/results/minilm_finetuning_comparison.json
 create mode 100644 ai/experiments/korean_video/results

In [47]:
# ===== 41-33단계. GitHub 업로드 전 원격 저장소 최종 확인 =====

from pathlib import Path
import subprocess

repo_dir = Path("/content/pinpoint-capstone")

print("===== 41-33단계. 원격 저장소 최종 확인 =====")

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout.strip()

# 1. 브랜치 확인
branch = git("branch", "--show-current")

if branch != "feature/korean-ai-fusion":
    raise RuntimeError(f"예상과 다른 브랜치: {branch}")

print(f"\n[1] 현재 브랜치: {branch}")

# 2. Commit 확인
commit = git("rev-parse", "--short", "HEAD")

if commit != "5b7df89":
    raise RuntimeError(f"예상과 다른 Commit: {commit}")

print(f"\n[2] 현재 Commit: {commit}")

# 3. 원격 저장소 확인 (인증정보 보호)
remote = git("remote")

print("\n[3] 원격 저장소 이름")
print(remote or "원격 저장소 없음")

# 4. 원격 브랜치 정보
print("\n[4] 원격 브랜치 목록")
branches = git("branch", "-r")
print(branches or "원격 브랜치 정보 없음")

# 5. 현재 작업 상태
print("\n[5] Git 작업 상태")
status = git("status", "--short")
print(status or "변경사항 없음")

# 6. 최신 Commit의 파일 확인
print("\n[6] 최신 Commit에 포함된 파일")
changed = git(
    "diff-tree",
    "--no-commit-id",
    "--name-only",
    "-r",
    "HEAD"
).splitlines()

for path in changed:
    print(f"✅ {path}")

print(f"\n총 {len(changed)}개 파일")

print("\n===== 41-33단계 완료 =====")
print("✅ GitHub push 없음")
print("✅ Git add/commit 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ 브랜치 변경 없음")

===== 41-33단계. 원격 저장소 최종 확인 =====

[1] 현재 브랜치: feature/korean-ai-fusion

[2] 현재 Commit: 5b7df89

[3] 원격 저장소 이름
origin

[4] 원격 브랜치 목록
origin/HEAD -> origin/main
  origin/feature/backend
  origin/feature/frontend
  origin/feature/korean-ai-fusion
  origin/main

[5] Git 작업 상태
변경사항 없음

[6] 최신 Commit에 포함된 파일
✅ ai/experiments/korean_video/results/consensus_6queries_comparison.json
✅ ai/experiments/korean_video/results/consensus_v2_12queries_comparison.json
✅ ai/experiments/korean_video/results/detergent_evaluation.json
✅ ai/experiments/korean_video/results/fusion_baseline_evaluation.json
✅ ai/experiments/korean_video/results/fusion_new_3queries_comparison.json
✅ ai/experiments/korean_video/results/fusion_start_time_comparison.json
✅ ai/experiments/korean_video/results/minilm_baseline_evaluation.json
✅ ai/experiments/korean_video/results/minilm_finetuning_comparison.json
✅ ai/experiments/korean_video/results/minilm_full_validation_comparison.json
✅ ai/experiments/korean_video/results/minilm_f

In [48]:
# ===== 41-34단계. 공개용 JSON 16개 GitHub Push =====

from pathlib import Path
import subprocess

repo_dir = Path("/content/pinpoint-capstone")
expected_branch = "feature/korean-ai-fusion"
expected_commit = "5b7df89"

print("===== 41-34단계. GitHub Push =====")

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# 1. 현재 브랜치 및 Commit 확인
branch = git("branch", "--show-current")
commit = git("rev-parse", "--short", "HEAD")
status = git("status", "--porcelain")

if branch != expected_branch:
    raise RuntimeError(f"예상과 다른 브랜치: {branch}")

if commit != expected_commit:
    raise RuntimeError(f"예상과 다른 Commit: {commit}")

if status:
    raise RuntimeError("작업 폴더에 변경사항이 있습니다.")

print(f"✅ 브랜치: {branch}")
print(f"✅ Commit: {commit}")
print("✅ 작업 폴더 깨끗함")

# 2. 원격 브랜치 최신 정보 가져오기
subprocess.run(
    ["git", "fetch", "origin", expected_branch],
    cwd=repo_dir,
    check=True,
    capture_output=True,
    text=True
)

# 3. 원격 브랜치가 현재 Commit의 조상인지 확인
ancestor = subprocess.run(
    ["git", "merge-base", "--is-ancestor",
     f"origin/{expected_branch}", "HEAD"],
    cwd=repo_dir
)

if ancestor.returncode != 0:
    raise RuntimeError(
        "원격 브랜치와 로컬 브랜치 이력이 다릅니다. "
        "안전을 위해 Push를 중단합니다."
    )

print("✅ 원격 브랜치와 Commit 이력 확인")

# 4. GitHub Push (작업 브랜치만)
push = subprocess.run(
    ["git", "push", "origin",
     f"HEAD:refs/heads/{expected_branch}"],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

print("\n[Push 결과]")
print(push.stdout)
print(push.stderr)

if push.returncode != 0:
    raise RuntimeError("GitHub Push 실패")

# 5. 원격 Commit 확인
remote_commit = git(
    "ls-remote", "origin",
    f"refs/heads/{expected_branch}"
).split()[0]

local_commit = git("rev-parse", "HEAD")

if remote_commit != local_commit:
    raise RuntimeError("Push 후 원격 Commit이 일치하지 않습니다.")

print("\n===== 최종 확인 =====")
print("✅ GitHub Push 성공")
print("✅ 로컬·원격 Commit 일치")
print(f"✅ 브랜치: {expected_branch}")
print("✅ main 브랜치 Push 없음")
print("✅ 다른 팀원 브랜치 변경 없음")
print("✅ 원본 파일 수정·삭제 없음")

===== 41-34단계. GitHub Push =====
✅ 브랜치: feature/korean-ai-fusion
✅ Commit: 5b7df89
✅ 작업 폴더 깨끗함
✅ 원격 브랜치와 Commit 이력 확인

[Push 결과]

fatal: could not read Username for 'https://github.com': No such device or address



RuntimeError: GitHub Push 실패

In [49]:
# ===== 41-34-1단계. GitHub 인증 도구 확인 =====

import subprocess
import shutil

print("===== GitHub 인증 환경 확인 =====")

# GitHub CLI 설치 여부
gh_path = shutil.which("gh")

if gh_path:
    print("✅ GitHub CLI 설치됨")
else:
    print("⚠️ GitHub CLI 설치되지 않음")

# Git Credential Helper 설정 확인
result = subprocess.run(
    ["git", "config", "--get", "credential.helper"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Git Credential Helper 설정됨")
    print("설정:", result.stdout.strip())
else:
    print("⚠️ Git Credential Helper 설정 없음")

# 현재 GitHub CLI 인증 여부 (토큰 노출 방지)
if gh_path:
    auth = subprocess.run(
        ["gh", "auth", "status"],
        capture_output=True,
        text=True
    )

    if auth.returncode == 0:
        print("✅ GitHub CLI 인증 완료")
    else:
        print("⚠️ GitHub CLI 인증 필요")

print("\n✅ 인증 토큰·비밀번호 출력 없음")
print("✅ Git Push 없음")
print("✅ Git Commit 변경 없음")
print("✅ 파일 수정·삭제 없음")

===== GitHub 인증 환경 확인 =====
✅ GitHub CLI 설치됨
⚠️ Git Credential Helper 설정 없음
⚠️ GitHub CLI 인증 필요

✅ 인증 토큰·비밀번호 출력 없음
✅ Git Push 없음
✅ Git Commit 변경 없음
✅ 파일 수정·삭제 없음


In [50]:
# ===== 41-34-2단계. 기존 GitHub 인증 환경 확인 =====

import os

print("===== 기존 GitHub 인증 환경 확인 =====")

# 토큰 자체는 절대 출력하지 않음
token_names = [
    "GITHUB_TOKEN",
    "GH_TOKEN",
    "GITHUB_PAT"
]

for name in token_names:
    if os.environ.get(name):
        print(f"✅ {name}: 설정되어 있음")
    else:
        print(f"⚠️ {name}: 설정 없음")

# 기존 AskPass 환경 확인
if os.environ.get("GIT_ASKPASS"):
    print("✅ GIT_ASKPASS: 설정되어 있음")
else:
    print("⚠️ GIT_ASKPASS: 설정 없음")

print("\n✅ 토큰 실제 값 출력 없음")
print("✅ Git Push 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Commit 변경 없음")

===== 기존 GitHub 인증 환경 확인 =====
⚠️ GITHUB_TOKEN: 설정 없음
⚠️ GH_TOKEN: 설정 없음
⚠️ GITHUB_PAT: 설정 없음
⚠️ GIT_ASKPASS: 설정 없음

✅ 토큰 실제 값 출력 없음
✅ Git Push 없음
✅ 파일 수정·삭제 없음
✅ Commit 변경 없음


In [51]:
# ===== 41-34-3단계. 기존 GitHub 토큰 다시 연결 =====

import os
import getpass

print("===== GitHub 토큰 재연결 =====")

token = getpass.getpass(
    "아까 생성한 GitHub Personal Access Token 입력: "
).strip()

if not token:
    raise RuntimeError("토큰이 입력되지 않았습니다.")

os.environ["GITHUB_TOKEN"] = token

# 불필요한 변수 제거
del token

print("\n✅ GitHub 토큰 환경변수 등록 완료")
print("✅ 토큰 실제 값 출력 없음")
print("✅ Git Push 없음")
print("✅ Git Commit 변경 없음")
print("✅ 기존 JSON 파일 수정·삭제 없음")

===== GitHub 토큰 재연결 =====
아까 생성한 GitHub Personal Access Token 입력: ··········

✅ GitHub 토큰 환경변수 등록 완료
✅ 토큰 실제 값 출력 없음
✅ Git Push 없음
✅ Git Commit 변경 없음
✅ 기존 JSON 파일 수정·삭제 없음


In [52]:
# ===== 41-34-4단계. Python AskPass 방식 GitHub Push =====

import os
import sys
import subprocess
import tempfile
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")
branch = "feature/korean-ai-fusion"
expected_commit = "5b7df89"

print("===== 41-34-4단계. GitHub Push =====")

# 1. 기존 토큰 확인 (값 출력 금지)
if not os.environ.get("GITHUB_TOKEN"):
    raise RuntimeError("GITHUB_TOKEN이 없습니다.")

# 2. 현재 브랜치와 Commit 확인
def git(*args, env=None):
    return subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        env=env,
        check=True
    ).stdout.strip()

current_branch = git("branch", "--show-current")
current_commit = git("rev-parse", "--short=7", "HEAD")

if current_branch != branch:
    raise RuntimeError("예상과 다른 브랜치입니다.")

if current_commit != expected_commit:
    raise RuntimeError("예상과 다른 Commit입니다.")

if git("status", "--porcelain"):
    raise RuntimeError("작업 폴더에 변경사항이 있습니다.")

print(f"✅ 브랜치: {current_branch}")
print(f"✅ Commit: {current_commit}")
print("✅ 작업 폴더 깨끗함")

# 3. 임시 Python AskPass 스크립트 생성
# 토큰 자체는 파일에 저장하지 않음
with tempfile.TemporaryDirectory() as temp_dir:
    askpass_path = Path(temp_dir) / "askpass.py"

    askpass_path.write_text(
        "import os, sys\n"
        "prompt = sys.argv[1] if len(sys.argv) > 1 else ''\n"
        "if 'Username' in prompt:\n"
        "    print('x-access-token')\n"
        "elif 'Password' in prompt:\n"
        "    print(os.environ['GITHUB_TOKEN'])\n",
        encoding="utf-8"
    )

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass_path)
    env["GIT_ASKPASS_REQUIRE"] = "force"
    env["GIT_TERMINAL_PROMPT"] = "0"

    # Python 스크립트를 AskPass로 실행
    wrapper_path = Path(temp_dir) / "askpass.sh"

    wrapper_path.write_text(
        f'#!/bin/sh\nexec "{sys.executable}" "{askpass_path}" "$@"\n',
        encoding="utf-8"
    )
    wrapper_path.chmod(0o700)

    env["GIT_ASKPASS"] = str(wrapper_path)

    # 4. 원격 브랜치 최신 상태 확인
    subprocess.run(
        ["git", "fetch", "origin", branch],
        cwd=repo_dir,
        env=env,
        capture_output=True,
        text=True,
        check=True
    )

    ancestor = subprocess.run(
        [
            "git", "merge-base", "--is-ancestor",
            f"origin/{branch}", "HEAD"
        ],
        cwd=repo_dir,
        env=env
    )

    if ancestor.returncode != 0:
        raise RuntimeError(
            "원격 브랜치 이력이 다릅니다. Push 중단."
        )

    print("✅ 원격 브랜치 이력 확인")

    # 5. GitHub Push
    push = subprocess.run(
        [
            "git", "push", "origin",
            f"HEAD:refs/heads/{branch}"
        ],
        cwd=repo_dir,
        env=env,
        capture_output=True,
        text=True
    )

    if push.returncode != 0:
        # 오류 메시지에 인증정보가 포함될 위험이 있어 원문 출력하지 않음
        print(f"❌ GitHub Push 실패 (종료 코드: {push.returncode})")
        raise RuntimeError("Push 실패. 원격 저장소는 확인이 필요합니다.")

    print("✅ GitHub Push 명령 성공")

    # 6. 원격 Commit 검증
    remote_result = subprocess.run(
        ["git", "ls-remote", "origin", f"refs/heads/{branch}"],
        cwd=repo_dir,
        env=env,
        capture_output=True,
        text=True,
        check=True
    )

    remote_hash = remote_result.stdout.split()[0]
    local_hash = git("rev-parse", "HEAD")

    if remote_hash != local_hash:
        raise RuntimeError("로컬·원격 Commit 불일치")

    print("✅ 로컬·원격 Commit 일치")

print("\n===== 41-34-4단계 완료 =====")
print("✅ JSON 16개 포함된 Commit Push 완료")
print("✅ main 브랜치 Push 없음")
print("✅ 다른 팀원 브랜치 변경 없음")
print("✅ 기존 파일 수정·삭제 없음")
print("✅ 인증 토큰 출력 없음")

===== 41-34-4단계. GitHub Push =====
✅ 브랜치: feature/korean-ai-fusion
✅ Commit: 5b7df89
✅ 작업 폴더 깨끗함
✅ 원격 브랜치 이력 확인
✅ GitHub Push 명령 성공
✅ 로컬·원격 Commit 일치

===== 41-34-4단계 완료 =====
✅ JSON 16개 포함된 Commit Push 완료
✅ main 브랜치 Push 없음
✅ 다른 팀원 브랜치 변경 없음
✅ 기존 파일 수정·삭제 없음
✅ 인증 토큰 출력 없음


In [53]:
# ===== 41-35단계. GitHub 업로드 파일 최종 검증 =====

from pathlib import Path
import subprocess

repo_dir = Path("/content/pinpoint-capstone")
branch = "feature/korean-ai-fusion"
prefix = "ai/experiments/korean_video/results/"

print("===== 41-35단계. GitHub 업로드 최종 검증 =====")

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# 1. 현재 브랜치 확인
current_branch = git("branch", "--show-current")
if current_branch != branch:
    raise RuntimeError("예상과 다른 브랜치입니다.")

# 2. 로컬·원격 Commit 확인
local_hash = git("rev-parse", "HEAD")
remote_hash = git("rev-parse", f"refs/remotes/origin/{branch}")

if local_hash != remote_hash:
    raise RuntimeError("로컬·원격 Commit 불일치")

print("✅ 로컬·원격 Commit 일치")
print(f"✅ Commit: {local_hash[:7]}")

# 3. 원격 브랜치에 저장된 JSON 확인
remote_files = git(
    "ls-tree", "-r", "--name-only",
    f"origin/{branch}", "--", prefix
).splitlines()

json_files = sorted(
    path for path in remote_files
    if path.startswith(prefix) and path.endswith(".json")
)

# 4. 로컬 JSON 목록과 비교
local_files = sorted(
    p.relative_to(repo_dir).as_posix()
    for p in (repo_dir / prefix).glob("*.json")
)

if len(json_files) != 16:
    raise RuntimeError(
        f"원격 JSON 파일 수가 16개가 아닙니다: {len(json_files)}개"
    )

if json_files != local_files:
    raise RuntimeError("로컬·원격 JSON 파일 목록 불일치")

print("\n[GitHub 업로드 확인 파일]")
for path in json_files:
    print(f"✅ {Path(path).name}")

print(f"\n✅ 총 {len(json_files)}개 JSON 확인")

# 5. 파일 내용 해시 비교
for path in json_files:
    remote_blob = git(
        "rev-parse", f"origin/{branch}:{path}"
    )
    local_blob = git("hash-object", "--", path)

    if remote_blob != local_blob:
        raise RuntimeError(f"파일 내용 불일치: {path}")

print("✅ 로컬·원격 JSON 16개 내용 해시 일치")

# 6. Git 작업 상태
status = git("status", "--porcelain")

print("\n[Git 작업 상태]")
print(status or "변경사항 없음")

print("\n===== 41-35단계 완료 =====")
print("✅ GitHub 업로드 파일 16개 검증")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ main 브랜치 변경 없음")

===== 41-35단계. GitHub 업로드 최종 검증 =====
✅ 로컬·원격 Commit 일치
✅ Commit: 5b7df89

[GitHub 업로드 확인 파일]
✅ consensus_6queries_comparison.json
✅ consensus_v2_12queries_comparison.json
✅ detergent_evaluation.json
✅ fusion_baseline_evaluation.json
✅ fusion_new_3queries_comparison.json
✅ fusion_start_time_comparison.json
✅ minilm_baseline_evaluation.json
✅ minilm_finetuning_comparison.json
✅ minilm_full_validation_comparison.json
✅ minilm_full_validation_verification_20261010.json
✅ minilm_fusion_6queries_comparison.json
✅ pinpoint_ai_experiment_summary.json
✅ rice_washing_evaluation.json
✅ seasoning_evaluation.json
✅ squat_4methods_comparison.json
✅ squat_ground_truth.json

✅ 총 16개 JSON 확인
✅ 로컬·원격 JSON 16개 내용 해시 일치

[Git 작업 상태]
변경사항 없음

===== 41-35단계 완료 =====
✅ GitHub 업로드 파일 16개 검증
✅ 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ main 브랜치 변경 없음


In [54]:
# ===== 41-36단계. GitHub PR 상태 확인 =====

import os
import subprocess
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")
branch = "feature/korean-ai-fusion"

print("===== 41-36단계. GitHub PR 상태 확인 =====")

# 1. 현재 브랜치 확인
result = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True
)

current_branch = result.stdout.strip()

if current_branch != branch:
    raise RuntimeError("예상과 다른 브랜치입니다.")

print(f"✅ 현재 브랜치: {current_branch}")

# 2. GitHub 토큰 확인
token = os.environ.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN이 없습니다.")

print("✅ GitHub 토큰 확인")

# 3. GitHub CLI로 PR 조회
env = os.environ.copy()
env["GH_TOKEN"] = token

result = subprocess.run(
    [
        "gh", "pr", "list",
        "--head", branch,
        "--state", "all",
        "--json",
        "number,title,state,url,baseRefName,headRefName"
    ],
    cwd=repo_dir,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("⚠️ PR 조회 실패")
    print("종료 코드:", result.returncode)
else:
    import json

    prs = json.loads(result.stdout)

    if not prs:
        print("⚠️ 현재 브랜치와 연결된 PR 없음")
    else:
        for pr in prs:
            print("\n[PR 정보]")
            print("PR 번호:", pr["number"])
            print("제목:", pr["title"])
            print("상태:", pr["state"])
            print("대상 브랜치:", pr["baseRefName"])
            print("작업 브랜치:", pr["headRefName"])
            print("URL:", pr["url"])

print("\n===== 41-36단계 완료 =====")
print("✅ PR 생성·수정·병합 없음")
print("✅ Git add/commit/push 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ main 브랜치 변경 없음")

===== 41-36단계. GitHub PR 상태 확인 =====
✅ 현재 브랜치: feature/korean-ai-fusion
✅ GitHub 토큰 확인

[PR 정보]
PR 번호: 1
제목: feat(ai): add Korean fusion search and MiniLM experiments
상태: OPEN
대상 브랜치: main
작업 브랜치: feature/korean-ai-fusion
URL: https://github.com/pinpoint-capstone/pinpoint-capstone/pull/1

===== 41-36단계 완료 =====
✅ PR 생성·수정·병합 없음
✅ Git add/commit/push 없음
✅ 파일 수정·삭제 없음
✅ main 브랜치 변경 없음


In [55]:
# ===== 41-37단계. PR #1 JSON 반영 여부 확인 =====

import os
import json
import subprocess
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")
prefix = "ai/experiments/korean_video/results/"

print("===== 41-37단계. PR #1 변경 파일 확인 =====")

token = os.environ.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN이 없습니다.")

env = os.environ.copy()
env["GH_TOKEN"] = token

# 1. PR #1의 전체 변경 파일 조회
result = subprocess.run(
    [
        "gh", "api",
        "--paginate",
        "repos/pinpoint-capstone/pinpoint-capstone/pulls/1/files",
        "--jq", ".[].filename"
    ],
    cwd=repo_dir,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"PR 파일 조회 실패 (종료 코드: {result.returncode})"
    )

changed_files = result.stdout.splitlines()

# 2. 공개용 JSON만 분리
json_files = sorted(
    path for path in changed_files
    if path.startswith(prefix) and path.endswith(".json")
)

expected_files = sorted(
    p.relative_to(repo_dir).as_posix()
    for p in (repo_dir / prefix).glob("*.json")
)

print(f"\nPR 전체 변경 파일: {len(changed_files)}개")
print(f"공개용 JSON: {len(json_files)}개")

print("\n[PR에 반영된 JSON]")
for path in json_files:
    print(f"✅ {Path(path).name}")

# 3. 정확한 파일 목록 비교
if json_files != expected_files:
    missing = set(expected_files) - set(json_files)
    extra = set(json_files) - set(expected_files)

    print(f"\n⚠️ 누락 파일: {len(missing)}개")
    print(f"⚠️ 예상 외 JSON: {len(extra)}개")

    raise RuntimeError("PR의 JSON 목록이 일치하지 않습니다.")

if len(json_files) != 16:
    raise RuntimeError("PR의 JSON 파일 수가 16개가 아닙니다.")

print("\n✅ PR #1에 공개용 JSON 16개 모두 반영 확인")
print("✅ 파일 목록 일치")

print("\n===== 41-37단계 완료 =====")
print("✅ PR 생성·수정·병합 없음")
print("✅ Git add/commit/push 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ main 브랜치 변경 없음")

===== 41-37단계. PR #1 변경 파일 확인 =====

PR 전체 변경 파일: 25개
공개용 JSON: 16개

[PR에 반영된 JSON]
✅ consensus_6queries_comparison.json
✅ consensus_v2_12queries_comparison.json
✅ detergent_evaluation.json
✅ fusion_baseline_evaluation.json
✅ fusion_new_3queries_comparison.json
✅ fusion_start_time_comparison.json
✅ minilm_baseline_evaluation.json
✅ minilm_finetuning_comparison.json
✅ minilm_full_validation_comparison.json
✅ minilm_full_validation_verification_20261010.json
✅ minilm_fusion_6queries_comparison.json
✅ pinpoint_ai_experiment_summary.json
✅ rice_washing_evaluation.json
✅ seasoning_evaluation.json
✅ squat_4methods_comparison.json
✅ squat_ground_truth.json

✅ PR #1에 공개용 JSON 16개 모두 반영 확인
✅ 파일 목록 일치

===== 41-37단계 완료 =====
✅ PR 생성·수정·병합 없음
✅ Git add/commit/push 없음
✅ 파일 수정·삭제 없음
✅ main 브랜치 변경 없음


In [56]:
# ===== 41-38단계. PR #1 나머지 변경 파일 확인 =====

import os
import json
import subprocess
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")
prefix = "ai/experiments/korean_video/results/"

print("===== 41-38단계. PR 나머지 파일 확인 =====")

token = os.environ.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("GITHUB_TOKEN이 없습니다.")

env = os.environ.copy()
env["GH_TOKEN"] = token

result = subprocess.run(
    [
        "gh", "api",
        "--paginate",
        "repos/pinpoint-capstone/pinpoint-capstone/pulls/1/files"
    ],
    cwd=repo_dir,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("PR 변경 파일 조회 실패")

# 페이지별 JSON 배열 처리
import json as json_module

decoder = json_module.JSONDecoder()
content = result.stdout.strip()
all_files = []

while content:
    data, index = decoder.raw_decode(content)
    all_files.extend(data)
    content = content[index:].strip()

other_files = [
    f for f in all_files
    if not (
        f["filename"].startswith(prefix)
        and f["filename"].endswith(".json")
    )
]

print(f"\nPR 전체 변경 파일: {len(all_files)}개")
print(f"JSON 제외 파일: {len(other_files)}개")

print("\n[JSON 제외 변경 파일]")

for f in other_files:
    print(
        f"✅ {f['filename']} "
        f"| 상태: {f['status']}"
    )

if len(all_files) != 25 or len(other_files) != 9:
    print("\n⚠️ 예상 파일 개수와 다릅니다.")
else:
    print("\n✅ 기존 변경 파일 9개 확인")

print("\n===== 41-38단계 완료 =====")
print("✅ 파일 내용 출력 없음")
print("✅ PR 수정·병합 없음")
print("✅ Git add/commit/push 없음")
print("✅ 기존 파일 수정·삭제 없음")
print("✅ main 브랜치 변경 없음")

===== 41-38단계. PR 나머지 파일 확인 =====

PR 전체 변경 파일: 25개
JSON 제외 파일: 9개

[JSON 제외 변경 파일]
✅ ai/experiments/KOREAN_AI_INTEGRATION.md | 상태: added
✅ ai/experiments/korean_minilm/README.md | 상태: added
✅ ai/experiments/korean_minilm/evaluate.py | 상태: added
✅ ai/experiments/korean_minilm/prepare_data.py | 상태: added
✅ ai/experiments/korean_minilm/results/full_validation_metrics.json | 상태: added
✅ ai/experiments/korean_minilm/train.py | 상태: added
✅ ai/features/consensus_rerank.py | 상태: added
✅ ai/features/fusion_search.py | 상태: added
✅ ai/preprocessing/transcribe_service.py | 상태: added

✅ 기존 변경 파일 9개 확인

===== 41-38단계 완료 =====
✅ 파일 내용 출력 없음
✅ PR 수정·병합 없음
✅ Git add/commit/push 없음
✅ 기존 파일 수정·삭제 없음
✅ main 브랜치 변경 없음


In [57]:
# ===== 41-39단계. 한국어 AI Python 코드 문법 검사 =====

import ast
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")

python_files = [
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/features/consensus_rerank.py",
    "ai/features/fusion_search.py",
    "ai/preprocessing/transcribe_service.py",
]

print("===== 41-39단계. Python 문법 검사 =====")

success = 0
errors = 0

for relative_path in python_files:
    file_path = repo_dir / relative_path

    if not file_path.is_file():
        print(f"❌ 파일 없음: {relative_path}")
        errors += 1
        continue

    try:
        source = file_path.read_text(encoding="utf-8")
        ast.parse(source, filename=relative_path)

        print(f"✅ 문법 정상: {relative_path}")
        success += 1

    except SyntaxError as e:
        print(f"❌ 문법 오류: {relative_path}")
        print(f"   줄 번호: {e.lineno}")
        print(f"   오류 유형: {type(e).__name__}")
        errors += 1

    except Exception as e:
        print(f"❌ 검사 실패: {relative_path}")
        print(f"   오류 유형: {type(e).__name__}")
        errors += 1

print("\n===== 검사 결과 =====")
print(f"정상: {success}개")
print(f"오류: {errors}개")

if success == 6 and errors == 0:
    print("✅ Python 파일 6개 문법 검사 통과")
else:
    print("⚠️ 추가 확인 필요")

print("\n✅ 모델 실행·학습 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-39단계. Python 문법 검사 =====
✅ 문법 정상: ai/experiments/korean_minilm/evaluate.py
✅ 문법 정상: ai/experiments/korean_minilm/prepare_data.py
✅ 문법 정상: ai/experiments/korean_minilm/train.py
✅ 문법 정상: ai/features/consensus_rerank.py
✅ 문법 정상: ai/features/fusion_search.py
✅ 문법 정상: ai/preprocessing/transcribe_service.py

===== 검사 결과 =====
정상: 6개
오류: 0개
✅ Python 파일 6개 문법 검사 통과

✅ 모델 실행·학습 없음
✅ 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [58]:
# ===== 41-40단계. Python Import 구조 검사 =====

import ast
import sys
from pathlib import Path
from collections import defaultdict

repo_dir = Path("/content/pinpoint-capstone")

python_files = [
    "ai/experiments/korean_minilm/evaluate.py",
    "ai/experiments/korean_minilm/prepare_data.py",
    "ai/experiments/korean_minilm/train.py",
    "ai/features/consensus_rerank.py",
    "ai/features/fusion_search.py",
    "ai/preprocessing/transcribe_service.py",
]

print("===== 41-40단계. Python Import 구조 검사 =====")

stdlib = set(sys.stdlib_module_names)
all_imports = defaultdict(set)
relative_imports = []
parse_errors = []

for relative_path in python_files:
    file_path = repo_dir / relative_path

    try:
        tree = ast.parse(file_path.read_text(encoding="utf-8"))
    except Exception as e:
        parse_errors.append((relative_path, type(e).__name__))
        continue

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                all_imports[relative_path].add(alias.name)

        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""

            if node.level > 0:
                relative_imports.append(
                    (relative_path, node.level, module)
                )
            else:
                all_imports[relative_path].add(module)

# 프로젝트 내부 최상위 모듈 후보
project_modules = {
    "ai", "backend", "frontend",
    "moment_detr", "utils", "config"
}

third_party = set()
internal = set()
standard = set()

for modules in all_imports.values():
    for module in modules:
        root = module.split(".")[0]

        if root in stdlib:
            standard.add(module)
        elif root in project_modules:
            internal.add(module)
        else:
            third_party.add(module)

print("\n[1] Python 표준 라이브러리")
for module in sorted(standard):
    print(f"✅ {module}")

print("\n[2] 프로젝트 내부 모듈")
if internal:
    for module in sorted(internal):
        print(f"🔎 {module}")
else:
    print("없음")

print("\n[3] 외부 라이브러리")
if third_party:
    for module in sorted(third_party):
        print(f"📦 {module}")
else:
    print("없음")

print("\n[4] 상대 Import")
if relative_imports:
    for path, level, module in relative_imports:
        print(f"🔎 {path}: {'.' * level}{module}")
else:
    print("✅ 상대 Import 없음")

print("\n[5] 프로젝트 내부 경로 확인")
missing = []

for module in sorted(internal):
    module_path = Path(*module.split("."))

    candidates = [
        repo_dir / module_path.with_suffix(".py"),
        repo_dir / module_path / "__init__.py",
        repo_dir / module_path,
    ]

    if any(path.exists() for path in candidates):
        print(f"✅ 존재: {module}")
    else:
        print(f"⚠️ 확인 필요: {module}")
        missing.append(module)

print("\n===== 최종 결과 =====")
print(f"검사 파일: {len(python_files)}개")
print(f"외부 모듈 종류: {len(third_party)}개")
print(f"내부 모듈 경로 확인 필요: {len(missing)}개")
print(f"파싱 오류: {len(parse_errors)}개")

if missing or parse_errors:
    print("⚠️ 추가 검토 필요")
else:
    print("✅ 정적 Import 구조 1차 검사 완료")

print("\n✅ 실제 Import 실행 없음")
print("✅ 모델 실행·학습 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-40단계. Python Import 구조 검사 =====

[1] Python 표준 라이브러리
✅ argparse
✅ collections
✅ functools
✅ io
✅ json
✅ os
✅ pathlib
✅ tarfile
✅ zipfile

[2] 프로젝트 내부 모듈
🔎 ai.features.fusion_search
🔎 ai.features.run_moment_detr_inference

[3] 외부 라이브러리
📦 datasets
📦 faster_whisper
📦 langdetect
📦 numpy
📦 sentence_transformers
📦 sentence_transformers.training_args
📦 torch

[4] 상대 Import
✅ 상대 Import 없음

[5] 프로젝트 내부 경로 확인
✅ 존재: ai.features.fusion_search
✅ 존재: ai.features.run_moment_detr_inference

===== 최종 결과 =====
검사 파일: 6개
외부 모듈 종류: 7개
내부 모듈 경로 확인 필요: 0개
파싱 오류: 0개
✅ 정적 Import 구조 1차 검사 완료

✅ 실제 Import 실행 없음
✅ 모델 실행·학습 없음
✅ 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [59]:
# ===== 41-41단계. 외부 라이브러리 설치 여부 확인 =====

import importlib.util
from importlib.metadata import version, PackageNotFoundError

print("===== 41-41단계. 외부 라이브러리 설치 확인 =====")

packages = {
    "datasets": "datasets",
    "faster_whisper": "faster-whisper",
    "langdetect": "langdetect",
    "numpy": "numpy",
    "sentence_transformers": "sentence-transformers",
    "torch": "torch",
}

installed = 0
missing = []

for module_name, package_name in packages.items():
    spec = importlib.util.find_spec(module_name)

    if spec is None:
        print(f"⚠️ 미설치: {package_name}")
        missing.append(package_name)
    else:
        try:
            pkg_version = version(package_name)
        except PackageNotFoundError:
            pkg_version = "버전 확인 불가"

        print(f"✅ 설치됨: {package_name} | {pkg_version}")
        installed += 1

# sentence_transformers.training_args는 별도 패키지가 아닌
# sentence_transformers 내부 하위 모듈
print("\n[추가 확인]")
print("🔎 sentence_transformers.training_args는 별도 설치 대상이 아닙니다.")

print("\n===== 검사 결과 =====")
print(f"설치 확인: {installed}개")
print(f"미설치: {len(missing)}개")

if missing:
    print("⚠️ 미설치 라이브러리:")
    for package in missing:
        print(f" - {package}")
else:
    print("✅ 주요 외부 라이브러리 설치 확인 완료")

print("\n✅ 라이브러리 설치·업데이트 없음")
print("✅ 실제 모델 실행·학습 없음")
print("✅ 기존 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-41단계. 외부 라이브러리 설치 확인 =====
✅ 설치됨: datasets | 4.8.5
⚠️ 미설치: faster-whisper
⚠️ 미설치: langdetect
✅ 설치됨: numpy | 2.1.3
✅ 설치됨: sentence-transformers | 5.7.0
✅ 설치됨: torch | 2.11.0+cpu

[추가 확인]
🔎 sentence_transformers.training_args는 별도 설치 대상이 아닙니다.

===== 검사 결과 =====
설치 확인: 4개
미설치: 2개
⚠️ 미설치 라이브러리:
 - faster-whisper
 - langdetect

✅ 라이브러리 설치·업데이트 없음
✅ 실제 모델 실행·학습 없음
✅ 기존 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [60]:
# ===== 41-42단계. 누락 라이브러리 설치 =====

import subprocess
import sys
import importlib.util

print("===== 41-42단계. 누락 라이브러리 설치 =====")

packages = {
    "faster_whisper": "faster-whisper",
    "langdetect": "langdetect",
}

for module_name, package_name in packages.items():

    if importlib.util.find_spec(module_name) is not None:
        print(f"✅ 이미 설치됨: {package_name}")
        continue

    print(f"\n🔧 설치 시작: {package_name}")

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name,
            "--quiet"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"✅ 설치 완료: {package_name}")
    else:
        print(f"❌ 설치 실패: {package_name}")
        print(f"종료 코드: {result.returncode}")

print("\n===== 설치 후 확인 =====")

for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is not None:
        print(f"✅ 설치 확인: {package_name}")
    else:
        print(f"⚠️ 확인 필요: {package_name}")

print("\n✅ 모델 학습·추론 실행 없음")
print("✅ 프로젝트 파일 직접 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-42단계. 누락 라이브러리 설치 =====

🔧 설치 시작: faster-whisper
✅ 설치 완료: faster-whisper

🔧 설치 시작: langdetect
✅ 설치 완료: langdetect

===== 설치 후 확인 =====
✅ 설치 확인: faster-whisper
✅ 설치 확인: langdetect

✅ 모델 학습·추론 실행 없음
✅ 프로젝트 파일 직접 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [61]:
# ===== 41-43단계. 외부 라이브러리 실제 Import 검사 =====

import subprocess
import sys

print("===== 41-43단계. 외부 라이브러리 Import 검사 =====")

modules = [
    "datasets",
    "faster_whisper",
    "langdetect",
    "numpy",
    "sentence_transformers",
    "sentence_transformers.training_args",
    "torch",
]

success = 0
failed = []

for module in modules:
    # 독립적인 Python 프로세스로 Import 검사
    # 실제 모델 학습이나 추론은 실행하지 않음
    result = subprocess.run(
        [
            sys.executable,
            "-c",
            f"import importlib; importlib.import_module({module!r})"
        ],
        capture_output=True,
        text=True,
        timeout=120
    )

    if result.returncode == 0:
        print(f"✅ Import 성공: {module}")
        success += 1
    else:
        print(f"❌ Import 실패: {module}")
        print(f"   종료 코드: {result.returncode}")
        failed.append(module)

print("\n===== 최종 결과 =====")
print(f"Import 성공: {success}개")
print(f"Import 실패: {len(failed)}개")

if not failed:
    print("✅ 외부 라이브러리 Import 검사 통과")
else:
    print("⚠️ Import 오류 추가 확인 필요")

print("\n✅ 모델 학습·추론 없음")
print("✅ 프로젝트 소스 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-43단계. 외부 라이브러리 Import 검사 =====
✅ Import 성공: datasets
✅ Import 성공: faster_whisper
✅ Import 성공: langdetect
✅ Import 성공: numpy
✅ Import 성공: sentence_transformers
✅ Import 성공: sentence_transformers.training_args
✅ Import 성공: torch

===== 최종 결과 =====
Import 성공: 7개
Import 실패: 0개
✅ 외부 라이브러리 Import 검사 통과

✅ 모델 학습·추론 없음
✅ 프로젝트 소스 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [62]:
# ===== 41-44단계. 프로젝트 내부 모듈 Import 검사 =====

import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")

modules = [
    "ai.features.fusion_search",
    "ai.features.consensus_rerank",
]

print("===== 41-44단계. 내부 모듈 Import 검사 =====")

success = 0
failed = []

for module in modules:
    print(f"\n검사: {module}")

    try:
        result = subprocess.run(
            [
                sys.executable,
                "-B",
                "-c",
                (
                    "import importlib; "
                    f"importlib.import_module({module!r})"
                )
            ],
            cwd=repo_dir,
            capture_output=True,
            text=True,
            timeout=120
        )

        if result.returncode == 0:
            print(f"✅ Import 성공: {module}")
            success += 1
        else:
            print(f"❌ Import 실패: {module}")
            print(f"   종료 코드: {result.returncode}")
            print("   오류 마지막 부분:")
            print("\n".join(result.stderr.splitlines()[-8:]))
            failed.append(module)

    except subprocess.TimeoutExpired:
        print(f"⚠️ Import 시간 초과: {module}")
        failed.append(module)

print("\n===== 최종 결과 =====")
print(f"Import 성공: {success}개")
print(f"Import 실패: {len(failed)}개")

if not failed:
    print("✅ 프로젝트 내부 모듈 Import 검사 통과")
else:
    print("⚠️ 내부 모듈 추가 확인 필요")

print("\n✅ 별도 Python 프로세스 사용")
print("✅ 별도 모델 학습·추론 명령 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-44단계. 내부 모듈 Import 검사 =====

검사: ai.features.fusion_search
✅ Import 성공: ai.features.fusion_search

검사: ai.features.consensus_rerank
✅ Import 성공: ai.features.consensus_rerank

===== 최종 결과 =====
Import 성공: 2개
Import 실패: 0개
✅ 프로젝트 내부 모듈 Import 검사 통과

✅ 별도 Python 프로세스 사용
✅ 별도 모델 학습·추론 명령 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [63]:
# ===== 41-45단계. AI 검색 함수 인터페이스 확인 =====

import ast
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")

target_files = [
    "ai/features/fusion_search.py",
    "ai/features/consensus_rerank.py",
    "ai/preprocessing/transcribe_service.py",
]

print("===== 41-45단계. AI 함수 인터페이스 확인 =====")

total_functions = 0

for relative_path in target_files:
    file_path = repo_dir / relative_path

    print(f"\n[파일] {relative_path}")

    if not file_path.exists():
        print("❌ 파일 없음")
        continue

    tree = ast.parse(file_path.read_text(encoding="utf-8"))

    # 파일 최상위에 정의된 함수와 클래스만 확인
    for node in tree.body:

        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = node.args

            parameters = [
                arg.arg
                for arg in (
                    args.posonlyargs
                    + args.args
                    + args.kwonlyargs
                )
            ]

            if args.vararg:
                parameters.append("*" + args.vararg.arg)

            if args.kwarg:
                parameters.append("**" + args.kwarg.arg)

            print(f"✅ 함수: {node.name}")
            print(f"   파라미터: {', '.join(parameters) or '(없음)'}")

            total_functions += 1

        elif isinstance(node, ast.ClassDef):
            print(f"🔎 클래스: {node.name}")

            for item in node.body:
                if isinstance(
                    item,
                    (ast.FunctionDef, ast.AsyncFunctionDef)
                ):
                    print(f"   - 메서드: {item.name}")

print("\n===== 검사 결과 =====")
print(f"최상위 함수 수: {total_functions}개")

print("\n✅ 함수 본문·검색어·자막 내용 출력 없음")
print("✅ 모델 실행·학습 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-45단계. AI 함수 인터페이스 확인 =====

[파일] ai/features/fusion_search.py
✅ 함수: get_semantic_model
   파라미터: model_type
✅ 함수: temporal_iou
   파라미터: a, b
✅ 함수: get_semantic_subtitle_candidates
   파라미터: query, subtitles, model, top_k
✅ 함수: select_subtitle_query
   파라미터: subtitles, korean_query, english_query
✅ 함수: get_multilingual_subtitle_candidates
   파라미터: korean_query, english_query, subtitles, model, top_k
✅ 함수: deduplicate_multilingual_candidates
   파라미터: candidates, iou_threshold, start_gap
✅ 함수: make_fusion_result
   파라미터: video_id, query, candidate
✅ 함수: predict_moments_fusion
   파라미터: video_id, feature_path, subtitles, korean_query, english_query, video_duration, subtitle_json_path, top_k, model_type

[파일] ai/features/consensus_rerank.py
✅ 함수: _rank_by_consensus
   파라미터: hybrid, ko, en, min_overlap
✅ 함수: rank_by_consensus
   파라미터: hybrid, ko, en
✅ 함수: rank_by_consensus_v2
   파라미터: hybrid, ko, en

[파일] ai/preprocessing/transcribe_service.py
✅ 함수: get_whisper_model
   파라미터: (없음)
✅ 함수

In [64]:
# ===== 41-46단계. AI 핵심 함수 입력·출력 구조 검사 =====

import ast
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")

targets = {
    "ai/features/fusion_search.py": "predict_moments_fusion",
    "ai/preprocessing/transcribe_service.py": "transcribe_video",
}

print("===== 41-46단계. AI 핵심 함수 입출력 검사 =====")

for relative_path, function_name in targets.items():
    path = repo_dir / relative_path

    print(f"\n[파일] {relative_path}")
    print(f"[함수] {function_name}")

    if not path.is_file():
        print("❌ 파일 없음")
        continue

    tree = ast.parse(path.read_text(encoding="utf-8"))

    function = next(
        (
            node for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ),
        None
    )

    if function is None:
        print("❌ 함수 없음")
        continue

    # 1. 파라미터 및 기본값 확인
    args = function.args
    positional = args.posonlyargs + args.args
    defaults = [None] * (len(positional) - len(args.defaults))
    defaults += list(args.defaults)

    print("\n[1] 입력 파라미터")

    for arg, default in zip(positional, defaults):
        if default is None:
            print(f"✅ {arg.arg}: 필수")
        else:
            # 기본값 자체는 출력하지 않음
            print(f"✅ {arg.arg}: 기본값 있음")

    for arg, default in zip(args.kwonlyargs, args.kw_defaults):
        required = default is None
        print(
            f"✅ {arg.arg}: "
            f"{'필수' if required else '기본값 있음'}"
        )

    # 2. 반환문 형태만 확인
    returns = [
        node for node in ast.walk(function)
        if isinstance(node, ast.Return)
    ]

    print("\n[2] 반환문 구조")
    print(f"반환문 개수: {len(returns)}개")

    for i, ret in enumerate(returns, 1):
        value = ret.value

        if value is None:
            kind = "None"
        elif isinstance(value, ast.Dict):
            kind = "dict"
            keys = [
                key.value for key in value.keys
                if isinstance(key, ast.Constant)
                and isinstance(key.value, str)
            ]
            # 키 이름은 출력하지 않고 개수만 확인
            kind += f" (문자열 키 {len(keys)}개)"
        elif isinstance(value, ast.List):
            kind = "list"
        elif isinstance(value, ast.Tuple):
            kind = "tuple"
        elif isinstance(value, ast.Name):
            kind = "변수 반환"
        elif isinstance(value, ast.Call):
            kind = "함수 호출 결과"
        else:
            kind = type(value).__name__

        print(f"✅ 반환 {i}: {kind}")

    # 3. 비동기 함수 여부
    is_async = isinstance(function, ast.AsyncFunctionDef)
    print(f"\n[3] 비동기 함수: {'예' if is_async else '아니오'}")

print("\n===== 검사 완료 =====")
print("✅ 모델 학습·추론 없음")
print("✅ 함수 실행 없음")
print("✅ 검색어·자막 내용 출력 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-46단계. AI 핵심 함수 입출력 검사 =====

[파일] ai/features/fusion_search.py
[함수] predict_moments_fusion

[1] 입력 파라미터
✅ video_id: 필수
✅ feature_path: 필수
✅ subtitles: 필수
✅ korean_query: 필수
✅ english_query: 필수
✅ video_duration: 필수
✅ subtitle_json_path: 기본값 있음
✅ top_k: 기본값 있음
✅ model_type: 기본값 있음

[2] 반환문 구조
반환문 개수: 1개
✅ 반환 1: ListComp

[3] 비동기 함수: 아니오

[파일] ai/preprocessing/transcribe_service.py
[함수] transcribe_video

[1] 입력 파라미터
✅ video_path: 필수
✅ video_id: 필수

[2] 반환문 구조
반환문 개수: 2개
✅ 반환 1: 함수 호출 결과
✅ 반환 2: 함수 호출 결과

[3] 비동기 함수: 아니오

===== 검사 완료 =====
✅ 모델 학습·추론 없음
✅ 함수 실행 없음
✅ 검색어·자막 내용 출력 없음
✅ 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [66]:
# ===== 41-47단계. AI 검색 결과 반환 형식 검사 =====

import ast
from pathlib import Path

repo_dir = Path("/content/pinpoint-capstone")
file_path = repo_dir / "ai/features/fusion_search.py"

targets = [
    "make_fusion_result",
    "predict_moments_fusion",
]

print("===== 41-47단계. AI 검색 결과 구조 검사 =====")

tree = ast.parse(file_path.read_text(encoding="utf-8"))

for function_name in targets:

    function = next(
        (
            node for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ),
        None
    )

    print(f"\n[함수] {function_name}")

    if function is None:
        print("❌ 함수 없음")
        continue

    returns = [
        node for node in ast.walk(function)
        if isinstance(node, ast.Return)
    ]

    for i, ret in enumerate(returns, 1):
        value = ret.value

        print(f"\n반환문 {i}")

        if isinstance(value, ast.Dict):
            keys = [
                key.value
                for key in value.keys
                if isinstance(key, ast.Constant)
                and isinstance(key.value, str)
            ]

            print("✅ 반환 형태: dict")
            print("✅ 반환 필드:")
            for key in keys:
                print(f"   - {key}")

        elif isinstance(value, ast.ListComp):
            print("✅ 반환 형태: list comprehension")

            element = value.elt

            if isinstance(element, ast.Call):
                if isinstance(element.func, ast.Name):
                    print(
                        "✅ 리스트 생성 함수:",
                        element.func.id
                    )
                else:
                    print("🔎 함수 호출 형태")

            elif isinstance(element, ast.Dict):
                print("✅ 리스트 내부: dict")

                for key in element.keys:
                    if (
                        isinstance(key, ast.Constant)
                        and isinstance(key.value, str)
                    ):
                        print(f"   - {key.value}")

            else:
                print(
                    "🔎 리스트 내부 AST 유형:",
                    type(element).__name__
                )

        elif isinstance(value, ast.Call):
            if isinstance(value.func, ast.Name):
                print("✅ 반환 함수:", value.func.id)
            else:
                print("🔎 함수 호출 결과 반환")

        else:
            print(
                "🔎 반환 유형:",
                type(value).__name__ if value else "None"
            )

print("\n===== 검사 완료 =====")
print("✅ 검색어·자막 내용 출력 없음")
print("✅ 실제 모델 실행·학습 없음")
print("✅ 기존 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-47단계. AI 검색 결과 구조 검사 =====

[함수] make_fusion_result

반환문 1
✅ 반환 형태: dict
✅ 반환 필드:
   - video_id
   - query
   - start_time
   - end_time
   - score
   - source
   - hybrid_source
   - visual_score
   - subtitle_score
   - language

[함수] predict_moments_fusion

반환문 1
✅ 반환 형태: list comprehension
✅ 리스트 생성 함수: make_fusion_result

===== 검사 완료 =====
✅ 검색어·자막 내용 출력 없음
✅ 실제 모델 실행·학습 없음
✅ 기존 파일 수정·삭제 없음
✅ Git add/commit/push 없음
✅ PR 수정·병합 없음
✅ main 브랜치 변경 없음


In [67]:
# ===== 41-48단계. Drive 02·03 폴더 전체 파일 재점검 =====

from pathlib import Path
from collections import Counter

drive_root = Path("/content/drive/MyDrive/pinpoint_capstone")

print("===== 41-48단계. 02·03 폴더 전체 점검 =====")

# 실제 폴더명 자동 검색
target_dirs = sorted(
    p for p in drive_root.iterdir()
    if p.is_dir() and p.name.startswith(("02_", "03_"))
)

if len(target_dirs) != 2:
    print("⚠️ 예상 폴더 2개를 찾지 못했습니다.")
    print("발견 폴더:", [p.name for p in target_dirs])

# 전체 파일 목록 메모리에 보관
drive_file_manifest = []

important_extensions = {
    ".py", ".ipynb", ".json", ".jsonl",
    ".yaml", ".yml", ".toml", ".md",
    ".pt", ".pth", ".ckpt",
    ".safetensors", ".npz", ".pkl"
}

for folder in target_dirs:

    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file()
    )

    print(f"\n{'=' * 55}")
    print(f"📁 {folder.name}")
    print(f"{'=' * 55}")

    print(f"전체 파일: {len(files)}개")

    # 전체 목록 저장 (파일 수정 없음)
    for p in files:
        drive_file_manifest.append({
            "folder": folder.name,
            "path": p.relative_to(drive_root).as_posix(),
            "size": p.stat().st_size,
            "extension": p.suffix.lower()
        })

    # 확장자별 통계
    counts = Counter(p.suffix.lower() or "(확장자 없음)" for p in files)

    print("\n[확장자별 파일 개수]")
    for ext, count in sorted(counts.items()):
        print(f"{ext}: {count}개")

    # 바로 아래 폴더 구성
    print("\n[하위 폴더]")
    for p in sorted(folder.iterdir()):
        if p.is_dir():
            count = sum(1 for f in p.rglob("*") if f.is_file())
            print(f"📁 {p.name}: {count}개")

    # 코드, 실험 결과, 모델 등 주요 파일
    important = [
        p for p in files
        if p.suffix.lower() in important_extensions
    ]

    print(f"\n[주요 파일: {len(important)}개]")

    for p in important[:80]:
        relative = p.relative_to(folder).as_posix()
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"✅ {relative} | {size_mb:.2f} MB")

    if len(important) > 80:
        print(f"... 나머지 {len(important)-80}개는 메모리에 보관")

print("\n===== 전체 점검 결과 =====")
print(f"검사 폴더: {len(target_dirs)}개")
print(f"전체 파일: {len(drive_file_manifest)}개")

print("\n✅ 파일 내용 출력 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-48단계. 02·03 폴더 전체 점검 =====

📁 02_Korean_Video_Test
전체 파일: 53개

[확장자별 파일 개수]
.ckpt: 1개
.ipynb: 1개
.json: 27개
.md: 2개
.mp4: 4개
.pt: 3개
.py: 14개
.pyc: 1개

[하위 폴더]
📁 AI_Backup: 30개
📁 __pycache__: 1개
📁 clip_features: 3개

[주요 파일: 48개]
✅ AI_Backup/KOREAN_AI_INTEGRATION_restored_20261010.md | 0.01 MB
✅ AI_Backup/consensus_rerank.py | 0.00 MB
✅ AI_Backup/fusion_search_20261010.py | 0.01 MB
✅ AI_Backup/fusion_search_FINAL_restored_20261010.py | 0.01 MB
✅ AI_Backup/fusion_search_before_model_connection_20261010_130236.py | 0.01 MB
✅ AI_Backup/fusion_search_before_model_selection_20261010_130203.py | 0.01 MB
✅ AI_Backup/minilm_README_restored_20261010.md | 0.01 MB
✅ AI_Backup/minilm_evaluate_restored_20261010.py | 0.00 MB
✅ AI_Backup/minilm_full_validation_metrics_restored_20261010.json | 0.00 MB
✅ AI_Backup/minilm_prepare_data_restored_20261010.py | 0.01 MB
✅ AI_Backup/minilm_train_restored_20261010.py | 0.00 MB
✅ AI_Backup/public_experiment_results_20261010/consensus_6queries_comparison

In [68]:
# ===== 41-49단계. 중요 코드·노트북 GitHub 중복 검사 =====

from pathlib import Path
import hashlib
import subprocess

repo = Path("/content/pinpoint-capstone")
drive = Path("/content/drive/MyDrive/pinpoint_capstone")

folders = [
    drive / "02_Korean_Video_Test",
    drive / "03_Korean_Training_Data",
]

extensions = {".py", ".ipynb", ".md"}

print("===== 41-49단계. 핵심 파일 중복 여부 검사 =====")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# 현재 Git 브랜치에 커밋된 파일 목록
result = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only", "HEAD"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True
)

tracked_paths = [
    p for p in result.stdout.splitlines()
    if Path(p).suffix.lower() in extensions
]

# Git에 커밋된 내용 자체를 해시로 비교
git_hashes = {}

for relative in tracked_paths:
    blob = subprocess.run(
        ["git", "show", f"HEAD:{relative}"],
        cwd=repo,
        capture_output=True,
        check=True
    ).stdout

    h = hashlib.sha256(blob).hexdigest()
    git_hashes.setdefault(h, []).append(relative)

# Drive 파일 검사
target_files = []

for folder in folders:
    if folder.exists():
        target_files.extend(
            p for p in folder.rglob("*")
            if p.is_file() and p.suffix.lower() in extensions
        )

matched = []
unmatched = []

for file in sorted(target_files):
    relative = file.relative_to(drive).as_posix()
    digest = sha256(file)

    if digest in git_hashes:
        matched.append((relative, git_hashes[digest]))
    else:
        unmatched.append(relative)

print(f"\n검사 대상: {len(target_files)}개")
print(f"✅ Git Commit과 내용 일치: {len(matched)}개")
print(f"⚠️ 동일 내용 미확인: {len(unmatched)}개")

print("\n[1] Git Commit과 내용이 동일한 파일")
for source, matches in matched:
    print(f"✅ Drive: {source}")
    for dest in matches:
        print(f"   ↳ Git: {dest}")

print("\n[2] 동일 내용이 확인되지 않은 파일")
for path in unmatched:
    print(f"⚠️ {path}")

print("\n===== 검사 완료 =====")
print("✅ 파일 내용·토큰 출력 없음")
print("✅ 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-49단계. 핵심 파일 중복 여부 검사 =====

검사 대상: 19개
✅ Git Commit과 내용 일치: 9개
⚠️ 동일 내용 미확인: 10개

[1] Git Commit과 내용이 동일한 파일
✅ Drive: 02_Korean_Video_Test/AI_Backup/KOREAN_AI_INTEGRATION_restored_20261010.md
   ↳ Git: ai/experiments/KOREAN_AI_INTEGRATION.md
✅ Drive: 02_Korean_Video_Test/AI_Backup/consensus_rerank.py
   ↳ Git: ai/features/consensus_rerank.py
✅ Drive: 02_Korean_Video_Test/AI_Backup/fusion_search_FINAL_restored_20261010.py
   ↳ Git: ai/features/fusion_search.py
✅ Drive: 02_Korean_Video_Test/AI_Backup/minilm_README_restored_20261010.md
   ↳ Git: ai/experiments/korean_minilm/README.md
✅ Drive: 02_Korean_Video_Test/AI_Backup/minilm_evaluate_restored_20261010.py
   ↳ Git: ai/experiments/korean_minilm/evaluate.py
✅ Drive: 02_Korean_Video_Test/AI_Backup/minilm_prepare_data_restored_20261010.py
   ↳ Git: ai/experiments/korean_minilm/prepare_data.py
✅ Drive: 02_Korean_Video_Test/AI_Backup/minilm_train_restored_20261010.py
   ↳ Git: ai/experiments/korean_minilm/train.py
✅ Drive: 02_Korea

In [69]:
# ===== 41-50단계. 미등록 핵심 파일 역할 검사 =====

from pathlib import Path
import ast
import json

drive = Path("/content/drive/MyDrive/pinpoint_capstone")

targets = [
    "02_Korean_Video_Test/AI_Backup/fusion_search_20261010.py",
    "02_Korean_Video_Test/AI_Backup/fusion_search_before_model_connection_20261010_130236.py",
    "02_Korean_Video_Test/AI_Backup/fusion_search_before_model_selection_20261010_130203.py",
    "02_Korean_Video_Test/AI_Backup/transcribe_service_before_path_fix_20261010.py",
    "02_Korean_Video_Test/PinPoint_Korean_Video_Test.ipynb",
    "02_Korean_Video_Test/fusion_experiment.py",
    "02_Korean_Video_Test/indexer_before_subtitle_integration.py",
    "02_Korean_Video_Test/search_api_before_fusion.py",
    "03_Korean_Training_Data/minilm_finetuned/checkpoint-969/README.md",
    "03_Korean_Training_Data/minilm_finetuned/final_model/README.md",
]

print("===== 41-50단계. 미등록 파일 역할 검사 =====")

for relative in targets:
    path = drive / relative

    print(f"\n[파일] {relative}")

    if not path.is_file():
        print("❌ 파일 없음")
        continue

    print(f"크기: {path.stat().st_size:,} bytes")

    if path.suffix == ".py":
        try:
            tree = ast.parse(path.read_text(encoding="utf-8"))

            funcs = [
                node.name for node in tree.body
                if isinstance(
                    node, (ast.FunctionDef, ast.AsyncFunctionDef)
                )
            ]

            classes = [
                node.name for node in tree.body
                if isinstance(node, ast.ClassDef)
            ]

            print(f"✅ 함수 수: {len(funcs)}")
            print(f"✅ 클래스 수: {len(classes)}")
            print("함수 이름:", funcs[:30])

            main_guard = any(
                isinstance(node, ast.If)
                and isinstance(node.test, ast.Compare)
                and isinstance(node.test.left, ast.Name)
                and node.test.left.id == "__name__"
                for node in tree.body
            )

            print("실행 진입점:", "있음" if main_guard else "없음")

        except Exception as e:
            print("⚠️ 분석 실패:", type(e).__name__)

    elif path.suffix == ".ipynb":
        try:
            notebook = json.loads(
                path.read_text(encoding="utf-8")
            )

            cells = notebook.get("cells", [])
            code_cells = [
                c for c in cells
                if c.get("cell_type") == "code"
            ]

            markdown_cells = [
                c for c in cells
                if c.get("cell_type") == "markdown"
            ]

            print(f"✅ 전체 셀: {len(cells)}")
            print(f"✅ 코드 셀: {len(code_cells)}")
            print(f"✅ 설명 셀: {len(markdown_cells)}")

            imports = set()

            for cell in code_cells:
                source = "".join(cell.get("source", []))

                try:
                    tree = ast.parse(source)
                except SyntaxError:
                    continue

                for node in ast.walk(tree):
                    if isinstance(node, ast.Import):
                        for alias in node.names:
                            imports.add(alias.name.split(".")[0])

                    elif isinstance(node, ast.ImportFrom):
                        if node.module:
                            imports.add(node.module.split(".")[0])

            print("사용 모듈:", sorted(imports))

            # 저장된 실행 결과 존재 여부
            output_cells = sum(
                bool(c.get("outputs"))
                for c in code_cells
            )

            print(f"출력 저장된 코드 셀: {output_cells}개")

        except Exception as e:
            print("⚠️ 분석 실패:", type(e).__name__)

    elif path.suffix == ".md":
        print("✅ 모델 설명 문서")
        print("✅ 파일 내용 출력 생략")

print("\n===== 검사 완료 =====")
print("✅ 파일 내용·검색어·자막 출력 없음")
print("✅ 코드 실행·모델 학습 없음")
print("✅ 기존 파일 수정·삭제 없음")
print("✅ Git add/commit/push 없음")
print("✅ PR 수정·병합 없음")
print("✅ main 브랜치 변경 없음")

===== 41-50단계. 미등록 파일 역할 검사 =====

[파일] 02_Korean_Video_Test/AI_Backup/fusion_search_20261010.py
크기: 5,965 bytes
✅ 함수 수: 8
✅ 클래스 수: 0
함수 이름: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_multilingual_subtitle_candidates', 'deduplicate_multilingual_candidates', 'make_fusion_result', 'predict_moments_fusion']
실행 진입점: 없음

[파일] 02_Korean_Video_Test/AI_Backup/fusion_search_before_model_connection_20261010_130236.py
크기: 6,790 bytes
✅ 함수 수: 8
✅ 클래스 수: 0
함수 이름: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_multilingual_subtitle_candidates', 'deduplicate_multilingual_candidates', 'make_fusion_result', 'predict_moments_fusion']
실행 진입점: 없음

[파일] 02_Korean_Video_Test/AI_Backup/fusion_search_before_model_selection_20261010_130203.py
크기: 5,965 bytes
✅ 함수 수: 8
✅ 클래스 수: 0
함수 이름: ['get_semantic_model', 'temporal_iou', 'get_semantic_subtitle_candidates', 'select_subtitle_query', 'get_mul